In [ ]:
from google.colab import drive
drive.mount('/content/drive')

RESULTS = '/content/drive/MyDrive/FairFedCXR/results'
CKPTS = '/content/drive/MyDrive/FairFedCXR/checkpoints'
CLIENTS = '/content/drive/MyDrive/FairFedCXR/clients'

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
import os
base = '/content/drive/MyDrive/FairFedCXR'
for f in ['data','clients','checkpoints','results','figures','paper','logs']:
    os.makedirs(f'{base}/{f}', exist_ok=True)
    print(f'✓ {base}/{f}')

✓ /content/drive/MyDrive/FairFedCXR/data
✓ /content/drive/MyDrive/FairFedCXR/clients
✓ /content/drive/MyDrive/FairFedCXR/checkpoints
✓ /content/drive/MyDrive/FairFedCXR/results
✓ /content/drive/MyDrive/FairFedCXR/figures
✓ /content/drive/MyDrive/FairFedCXR/paper
✓ /content/drive/MyDrive/FairFedCXR/logs


In [ ]:
import numpy as np
for name, path in [('NIH', f'{RESULTS}/nih_pred_cache.npz'), ('CheXpert', f'{RESULTS}/chexpert_pred_cache.npz')]:
    c = np.load(path, allow_pickle=True)
    has_fairfed = any(m == 'fairfed' for m, s in c['all_probs'].item().keys())
    print(f"{name}: fairfed cached = {has_fairfed}")

NIH: fairfed cached = True
CheXpert: fairfed cached = True


In [ ]:
print("Starting download... this takes 15-25 minutes. Do NOT close the browser.")
print("You will see a progress bar below.\n")

!kaggle datasets download -d ashery/chexpert \
    -p /content/drive/MyDrive/FairFedCXR/data/ \
    --unzip

print("\n✓ Download complete!")

Starting download... this takes 15-25 minutes. Do NOT close the browser.
You will see a progress bar below.

Dataset URL: https://www.kaggle.com/datasets/ashery/chexpert
License(s): CC0-1.0
100% 10.7G/10.7G [07:37<00:00, 25.1MB/s]

User cancelled operation

✓ Download complete!


In [ ]:
import os

zip_path = '/content/drive/MyDrive/FairFedCXR/data/chexpert.zip'
if os.path.exists(zip_path):
    size = os.path.getsize(zip_path) / 1e9
    print(f"✓ ZIP exists on Drive: {size:.1f} GB")
else:
    print("ZIP not found — checking what is there...")
    for f in os.listdir('/content/drive/MyDrive/FairFedCXR/data/'):
        print(f" {f}")

✓ ZIP exists on Drive: 11.5 GB


In [ ]:
import os

# Unzip to /content/ (local, fast SSD) — takes 3-5 min instead of 90 min
print("Unzipping to local Colab storage (fast)...")
!unzip -q /content/drive/MyDrive/FairFedCXR/data/chexpert.zip \
       -d /content/chexpert/

print("Done! Checking...")
!ls /content/chexpert/

Unzipping to local Colab storage (fast)...
Done! Checking...
train  train.csv  valid  valid.csv


In [ ]:
import os

data = '/content/drive/MyDrive/FairFedCXR/data'
items = os.listdir(data)
print("Items in data folder so far:")
for item in items:
    print(f"  {item}")

Items in data folder so far:
  chexpert.zip
  train.csv
  train
  master_cohort.csv


In [ ]:
import pandas as pd

df = pd.read_csv('/content/chexpert/train.csv')
print("Shape:", df.shape)
print("\nAll columns:")
for col in df.columns:
    print(f"  {col}")
print("\nFirst 2 rows:")
print(df.head(2).to_string())

Shape: (223414, 19)

All columns:
  Path
  Sex
  Age
  Frontal/Lateral
  AP/PA
  No Finding
  Enlarged Cardiomediastinum
  Cardiomegaly
  Lung Opacity
  Lung Lesion
  Edema
  Consolidation
  Pneumonia
  Atelectasis
  Pneumothorax
  Pleural Effusion
  Pleural Other
  Fracture
  Support Devices

First 2 rows:
                                                              Path     Sex  Age Frontal/Lateral AP/PA  No Finding  Enlarged Cardiomediastinum  Cardiomegaly  Lung Opacity  Lung Lesion  Edema  Consolidation  Pneumonia  Atelectasis  Pneumothorax  Pleural Effusion  Pleural Other  Fracture  Support Devices
0  CheXpert-v1.0-small/train/patient00001/study1/view1_frontal.jpg  Female   68         Frontal    AP         1.0                         NaN           NaN           NaN          NaN    NaN            NaN        NaN          NaN           0.0               NaN            NaN       NaN              1.0
1  CheXpert-v1.0-small/train/patient00002/study2/view1_frontal.jpg  Female   87      

In [ ]:
import pandas as pd, os, sys

# ── load ──────────────────────────────────────────────────────
df = pd.read_csv('/content/chexpert/train.csv')

# ── Step 1: frontal only ──────────────────────────────────────
df = df[df['Frontal/Lateral'] == 'Frontal'].copy()
print(f"After frontal filter:     {len(df):,} rows")

# ── Step 2: handle Pleural Effusion label ─────────────────────
# -1 = uncertain → exclude. 0 = negative. 1 = positive.
df['label'] = df['Pleural Effusion'].replace({-1: float('nan')})
df = df.dropna(subset=['label'])
df['label'] = df['label'].astype(int)
print(f"After uncertainty drop:   {len(df):,} rows")

# ── Step 3: drop missing sex or age ──────────────────────────
df = df.dropna(subset=['Sex', 'Age'])
df = df[df['Sex'].isin(['Male', 'Female'])]
print(f"After sex/age filter:     {len(df):,} rows")

# ── Step 4: F3 sex encoding lock ─────────────────────────────
df['sex_encoded'] = df['Sex'].map({'Male': 1, 'Female': 0})
assert df['sex_encoded'].isin([0,1]).all()
assert (df.loc[df['Sex']=='Male','sex_encoded']==1).all()
assert (df.loc[df['Sex']=='Female','sex_encoded']==0).all()
print("Sex encoding locked: 1=Male 0=Female ✓")

# ── Step 5: stats ─────────────────────────────────────────────
print(f"\nFinal cohort size:        {len(df):,}")
print(f"Prevalence (PE):          {df['label'].mean():.3f}")
print(f"Female fraction:          {(df['Sex']=='Female').mean():.3f}")
print(f"Male fraction:            {(df['Sex']=='Male').mean():.3f}")
print(f"Age mean ± std:           {df['Age'].mean():.1f} ± {df['Age'].std():.1f}")
print(f"\nLabel counts:")
print(df['label'].value_counts())
print(f"\nSex counts:")
print(df['Sex'].value_counts())

# ── Step 6: HARD GATE ────────────────────────────────────────
if len(df) < 42000:
    print(f"\n⚠ GATE: only {len(df):,} rows — scaling client sizes proportionally")
    scale = len(df) / 42000
    sizes = {'A': int(15000*scale), 'B': int(8000*scale),
             'C': int(12000*scale), 'D': int(6000*scale)}
    print("Scaled sizes:", sizes)
else:
    print(f"\n✅ GATE PASSED — {len(df):,} rows available for 41,000 client pool")
    sizes = {'A': 15000, 'B': 8000, 'C': 12000, 'D': 6000}
    print("Client sizes:", sizes)

# ── Step 7: save master cohort ───────────────────────────────
df.to_csv('/content/drive/MyDrive/FairFedCXR/data/master_cohort.csv', index=False)
print("\nmaster_cohort.csv saved to Drive ✓")

After frontal filter:     191,027 rows
After uncertainty drop:   102,198 rows
After sex/age filter:     102,197 rows
Sex encoding locked: 1=Male 0=Female ✓

Final cohort size:        102,197
Prevalence (PE):          0.752
Female fraction:          0.417
Male fraction:            0.583
Age mean ± std:           61.5 ± 17.5

Label counts:
label
1    76899
0    25298
Name: count, dtype: int64

Sex counts:
Sex
Male      59616
Female    42581
Name: count, dtype: int64

✅ GATE PASSED — 102,197 rows available for 41,000 client pool
Client sizes: {'A': 15000, 'B': 8000, 'C': 12000, 'D': 6000}

master_cohort.csv saved to Drive ✓


In [ ]:
import pandas as pd
import numpy as np

np.random.seed(42)
df = pd.read_csv('/content/drive/MyDrive/FairFedCXR/data/master_cohort.csv')

# ── Balance to 40% prevalence ─────────────────────────────────
pos = df[df['label']==1].copy()
neg = df[df['label']==0].copy()
pos_s = pos.sample(n=int(0.667*len(neg)), random_state=42)
df_bal = pd.concat([pos_s, neg]).sample(frac=1,random_state=42).reset_index(drop=True)
df_bal['uid'] = df_bal.index
print(f"Balanced: {len(df_bal):,} | Prev: {df_bal['label'].mean():.3f}")

used_uids = set()

def safe_sample(pool, n, label, uid_set):
    avail = pool[(pool['label']==label)&(~pool['uid'].isin(uid_set))]
    n = min(n, len(avail))
    s = avail.sample(n=n, random_state=42)
    uid_set.update(s['uid'].tolist())
    return s

def build_client(df_bal, n, male_frac, prev_target, name,
                 uid_set, age_min=0, age_max=99):
    pool = df_bal[(df_bal['Age']>=age_min)&(df_bal['Age']<=age_max)].copy()
    M = pool[pool['Sex']=='Male']
    F = pool[pool['Sex']=='Female']

    n_pos   = int(n * prev_target)
    n_neg   = n - n_pos
    n_pos_m = int(n_pos * male_frac);  n_pos_f = n_pos - n_pos_m
    n_neg_m = int(n_neg * male_frac);  n_neg_f = n_neg - n_neg_m

    parts = [
        safe_sample(M, n_pos_m, 1, uid_set),
        safe_sample(M, n_neg_m, 0, uid_set),
        safe_sample(F, n_pos_f, 1, uid_set),
        safe_sample(F, n_neg_f, 0, uid_set),
    ]
    c = pd.concat(parts).sample(frac=1,random_state=42).reset_index(drop=True)
    c['client'] = name
    male_f = (c['Sex']=='Male').mean()
    prev   = c['label'].mean()
    age_m  = c['Age'].mean()
    m_pos  = int((c[(c['Sex']=='Male')]['label']==1).sum())
    f_pos  = int((c[(c['Sex']=='Female')]['label']==1).sum())
    print(f"Client {name}: n={len(c):,} | Male={male_f:.2f} | "
          f"Prev={prev:.3f} | Age={age_m:.1f} | "
          f"m_pos={m_pos} f_pos={f_pos}")
    return c

print("\n── Building clients ──────────────────────────────────────")
# A: balanced reference — no age filter
client_A = build_client(df_bal, 15000, 0.50, 0.40, 'A', used_uids)
# B: male-skewed — no age filter (age filtering caused pool shortage)
client_B = build_client(df_bal,  8000, 0.80, 0.45, 'B', used_uids)
# C: female-skewed elderly — keep age 60+ (large pool, works fine)
client_C = build_client(df_bal, 12000, 0.20, 0.25, 'C', used_uids,
                        age_min=60)
# D: small mixed — no age filter
client_D = build_client(df_bal,  6000, 0.60, 0.38, 'D', used_uids)

clients = {'A': client_A, 'B': client_B,
           'C': client_C, 'D': client_D}

# ── Overlap + size check ───────────────────────────────────────
all_uids = []
for name, c in clients.items():
    all_uids.extend(c['uid'].tolist())
assert len(all_uids)==len(set(all_uids)), "OVERLAP DETECTED"

print(f"\n✓ No overlap")
print(f"✓ Total used: {len(all_uids):,} / {len(df_bal):,}")

# ── F1 gate: ≥25 positives per sex in val AND test ───────────
print("\n── F1 Gate Check (val=15%, test=15%) ────────────────────")
TAU = 25
gate_ok = True
for name, c in clients.items():
    val_size  = int(len(c)*0.15)
    test_size = int(len(c)*0.15)
    for split, sz in [('val', val_size), ('test', test_size)]:
        for sex in ['Male','Female']:
            pos_count = int(len(c[(c['Sex']==sex)&(c['label']==1)]) * 0.15)
            if pos_count < TAU:
                print(f"  ⚠ Client {name} {split} {sex}_pos={pos_count} < {TAU}")
                gate_ok = False
            else:
                print(f"  ✓ Client {name} {split} {sex}_pos≈{pos_count}")

if gate_ok:
    print("\n✅ F1 GATE PASSED — proceeding to train/val/test split")
else:
    print("\n❌ GATE FAILED — check warnings above")

Balanced: 42,171 | Prev: 0.400

── Building clients ──────────────────────────────────────
Client A: n=15,000 | Male=0.50 | Prev=0.400 | Age=59.6 | m_pos=3000 f_pos=3000
Client B: n=8,000 | Male=0.80 | Prev=0.450 | Age=59.1 | m_pos=2880 f_pos=720
Client C: n=6,899 | Male=0.35 | Prev=0.377 | Age=73.6 | m_pos=600 f_pos=2002
Client D: n=6,000 | Male=0.60 | Prev=0.380 | Age=49.9 | m_pos=1368 f_pos=912

✓ No overlap
✓ Total used: 35,899 / 42,171

── F1 Gate Check (val=15%, test=15%) ────────────────────
  ✓ Client A val Male_pos≈450
  ✓ Client A val Female_pos≈450
  ✓ Client A test Male_pos≈450
  ✓ Client A test Female_pos≈450
  ✓ Client B val Male_pos≈432
  ✓ Client B val Female_pos≈108
  ✓ Client B test Male_pos≈432
  ✓ Client B test Female_pos≈108
  ✓ Client C val Male_pos≈90
  ✓ Client C val Female_pos≈300
  ✓ Client C test Male_pos≈90
  ✓ Client C test Female_pos≈300
  ✓ Client D val Male_pos≈205
  ✓ Client D val Female_pos≈136
  ✓ Client D test Male_pos≈205
  ✓ Client D test Female_po

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
import os

out = '/content/drive/MyDrive/FairFedCXR/clients'
os.makedirs(out, exist_ok=True)

summary_rows = []

for name, c in clients.items():
    # 70/15/15 stratified by label, fixed seed
    train, temp = train_test_split(
        c, test_size=0.30, random_state=42, stratify=c['label'])
    val, test = train_test_split(
        temp, test_size=0.50, random_state=42, stratify=temp['label'])

    # save each split
    train.to_csv(f'{out}/client_{name}_train.csv', index=False)
    val.to_csv(f'{out}/client_{name}_val.csv',     index=False)
    test.to_csv(f'{out}/client_{name}_test.csv',   index=False)

    # record summary with actual positive counts per sex per split
    def pos_count(d, sex):
        return int(len(d[(d['Sex']==sex)&(d['label']==1)]))

    summary_rows.append({
        'client': name,
        'total': len(c),
        'train': len(train), 'val': len(val), 'test': len(test),
        'male_frac': round((c['Sex']=='Male').mean(),3),
        'prevalence': round(c['label'].mean(),3),
        'age_mean': round(c['Age'].mean(),1),
        'val_male_pos':   pos_count(val,'Male'),
        'val_female_pos': pos_count(val,'Female'),
        'test_male_pos':   pos_count(test,'Male'),
        'test_female_pos': pos_count(test,'Female'),
    })
    print(f"Client {name}: train={len(train):,} val={len(val):,} test={len(test):,}")

# save master summary
summary = pd.DataFrame(summary_rows)
summary.to_csv(f'{out}/client_summary.csv', index=False)

print("\n── client_summary.csv ───────────────────────────────────")
print(summary.to_string(index=False))

# ── Final F1 verification on ACTUAL splits ───────────────────
print("\n── Final gate check on actual saved splits ──────────────")
TAU = 25
all_ok = True
for _, r in summary.iterrows():
    for col in ['val_male_pos','val_female_pos','test_male_pos','test_female_pos']:
        if r[col] < TAU:
            print(f"  ⚠ Client {r['client']} {col}={r[col]} < {TAU}")
            all_ok = False
print("✅ ALL SPLITS PASS GATE" if all_ok else "❌ CHECK WARNINGS")

print(f"\n✓ All client files saved to {out}")
print("✓ Day 2 complete — split frozen")

Client A: train=10,500 val=2,250 test=2,250
Client B: train=5,600 val=1,200 test=1,200
Client C: train=4,829 val=1,035 test=1,035
Client D: train=4,200 val=900 test=900

── client_summary.csv ───────────────────────────────────
client  total  train  val  test  male_frac  prevalence  age_mean  val_male_pos  val_female_pos  test_male_pos  test_female_pos
     A  15000  10500 2250  2250      0.500       0.400      59.6           433             467            425              475
     B   8000   5600 1200  1200      0.800       0.450      59.1           447              93            411              129
     C   6899   4829 1035  1035      0.348       0.377      73.6            96             295             87              303
     D   6000   4200  900   900      0.600       0.380      49.9           197             145            212              130

── Final gate check on actual saved splits ──────────────
✅ ALL SPLITS PASS GATE

✓ All client files saved to /content/drive/MyDrive/Fai

In [ ]:
import torch
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from PIL import Image
import pandas as pd
import os

IMG_ROOT = '/content/chexpert'   # where train/ folder lives after unzip

train_tf = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485,0.456,0.406], std=[0.229,0.224,0.225]),
])
eval_tf = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485,0.456,0.406], std=[0.229,0.224,0.225]),
])

class CheXpertDataset(Dataset):
    def __init__(self, csv_path, img_root, transform):
        self.df = pd.read_csv(csv_path)
        self.img_root = img_root
        self.transform = transform
    def __len__(self):
        return len(self.df)
    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img_path = os.path.join(self.img_root, row['Path'])
        img = Image.open(img_path).convert('RGB')
        img = self.transform(img)
        label = torch.tensor(row['label'], dtype=torch.float32)
        sex   = torch.tensor(row['sex_encoded'], dtype=torch.long)  # 1=M 0=F
        return img, label, sex

print("Dataset class defined ✓")

Dataset class defined ✓


In [ ]:
CLIENTS = '/content/drive/MyDrive/FairFedCXR/clients'

def get_client_loaders(name, batch_size=16, num_workers=2):
    tr = CheXpertDataset(f'{CLIENTS}/client_{name}_train.csv', IMG_ROOT, train_tf)
    va = CheXpertDataset(f'{CLIENTS}/client_{name}_val.csv',   IMG_ROOT, eval_tf)
    te = CheXpertDataset(f'{CLIENTS}/client_{name}_test.csv',  IMG_ROOT, eval_tf)
    return (
        DataLoader(tr, batch_size=batch_size, shuffle=True,  num_workers=num_workers, pin_memory=True),
        DataLoader(va, batch_size=batch_size, shuffle=False, num_workers=num_workers, pin_memory=True),
        DataLoader(te, batch_size=batch_size, shuffle=False, num_workers=num_workers, pin_memory=True),
    )
print("Loader factory defined ✓")

Loader factory defined ✓


In [ ]:
import numpy as np
from sklearn.metrics import roc_auc_score, average_precision_score

EPS = 1.0  # Laplace smoothing for TPR (matches the LaTeX)

def compute_ece(probs, labels, M=8):
    bins = np.linspace(0,1,M+1)
    ece = 0.0
    for lo,hi in zip(bins[:-1],bins[1:]):
        m = (probs>=lo)&(probs<hi)
        if m.sum()==0: continue
        acc  = labels[m].mean()
        conf = probs[m].mean()
        ece += (m.mean())*abs(acc-conf)
    return float(ece)

def compute_eo_gap(probs, labels, sex, threshold=0.5):
    preds = (probs>=threshold).astype(int)
    def smoothed_tpr(mask):
        pos = (labels[mask]==1)
        tp = int(((preds[mask]==1)&pos).sum())
        p  = int(pos.sum())
        return (tp+EPS)/(p+2*EPS)        # never nan
    m_tpr = smoothed_tpr(sex==1)
    f_tpr = smoothed_tpr(sex==0)
    return abs(m_tpr-f_tpr), m_tpr, f_tpr

def eval_model(model, loader, device, threshold=0.5):
    model.eval()
    P,L,S = [],[],[]
    with torch.no_grad():
        for imgs,labels,sex in loader:
            logits = model(imgs.to(device)).squeeze().cpu()
            P.append(torch.sigmoid(logits).numpy())
            L.append(labels.numpy()); S.append(sex.numpy())
    probs,labels,sex = map(np.concatenate,(P,L,S))
    eo,m_tpr,f_tpr = compute_eo_gap(probs,labels,sex,threshold)
    return {
        'auroc': roc_auc_score(labels,probs),
        'auprc': average_precision_score(labels,probs),
        'ece':   compute_ece(probs,labels),
        'eo_gap':eo, 'male_tpr':m_tpr, 'female_tpr':f_tpr,
        'm_pos': int((labels[sex==1]==1).sum()),
        'f_pos': int((labels[sex==0]==1).sum()),
    }
print("Metric module defined ✓")

Metric module defined ✓


In [ ]:
#run always
# 1. Mount Drive
from google.colab import drive
drive.mount('/content/drive')

# 2. Unzip images locally
!unzip -q /content/drive/MyDrive/FairFedCXR/data/chexpert.zip \
       -d /content/chexpert/

# 3. Install packages
!pip install -q torch torchvision torchmetrics scikit-learn pandas tqdm scipy opacus

# 4. Set root
IMG_ROOT = '/content/chexpert/CheXpert-v1.0-small'
print("Ready ✓")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
replace /content/chexpert/train.csv? [y]es, [n]o, [A]ll, [N]one, [r]ename: A
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 983.4/983.4 kB 51.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 308.9/308.9 kB 29.8 MB/s eta 0:00:00
Ready ✓


In [ ]:
IMG_ROOT = '/content/chexpert'

# Verify with actual structure
import pandas as pd, os
df_check = pd.read_csv(
    '/content/drive/MyDrive/FairFedCXR/clients/client_A_val.csv')

# Check what the Path column looks like
sample_path = df_check['Path'].iloc[0]
print("Raw CSV path:", sample_path)

# The CSV has 'CheXpert-v1.0-small/train/...' but files are at 'train/...'
# Strip the prefix
clean_path = sample_path.replace('CheXpert-v1.0-small/', '')
full_path = os.path.join(IMG_ROOT, clean_path)
print("Clean path: ", full_path)
print("Exists:     ", os.path.exists(full_path))

Raw CSV path: CheXpert-v1.0-small/train/patient25962/study1/view1_frontal.jpg
Clean path:  /content/chexpert/train/patient25962/study1/view1_frontal.jpg
Exists:      True


In [ ]:
import torch
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from PIL import Image
import pandas as pd, os

IMG_ROOT = '/content/chexpert'

train_tf = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485,0.456,0.406],
                         std=[0.229,0.224,0.225]),
])
eval_tf = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485,0.456,0.406],
                         std=[0.229,0.224,0.225]),
])

class CheXpertDataset(Dataset):
    def __init__(self, csv_path, img_root, transform):
        self.df        = pd.read_csv(csv_path)
        self.img_root  = img_root
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        # strip the dataset prefix — images live directly under img_root
        clean = row['Path'].replace('CheXpert-v1.0-small/', '')
        img_path = os.path.join(self.img_root, clean)
        img   = Image.open(img_path).convert('RGB')
        img   = self.transform(img)
        label = torch.tensor(row['label'],       dtype=torch.float32)
        sex   = torch.tensor(row['sex_encoded'], dtype=torch.long)
        return img, label, sex

print("Dataset class updated ✓")

Dataset class updated ✓


In [ ]:
import os

# Check what exists so far
base = '/content/chexpert'
print("Contents of /content/chexpert/:")
if os.path.exists(base):
    for item in os.listdir(base):
        full = os.path.join(base, item)
        if os.path.isdir(full):
            # count files inside
            count = sum(len(f) for _,_,f in os.walk(full))
            print(f"  DIR:  {item}/  ({count:,} files)")
        else:
            size = os.path.getsize(full)/1e6
            print(f"  FILE: {item}  ({size:.1f} MB)")
else:
    print("  /content/chexpert does not exist yet")

# Check if unzip cell is still running
print("\nIs Cell 6 still spinning? Check the stop button on that cell.")

Contents of /content/chexpert/:
  DIR:  valid/  (234 files)
  FILE: valid.csv  (0.0 MB)
  FILE: train.csv  (24.5 MB)
  DIR:  train/  (223,415 files)

Is Cell 6 still spinning? Check the stop button on that cell.


In [ ]:
import torch
import torchvision.models as models
import numpy as np
import time, json

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Device: {device}")

# ── model ─────────────────────────────────────────────────────
def build_model():
    m = models.densenet121(weights='IMAGENET1K_V1')
    m.classifier = torch.nn.Linear(m.classifier.in_features, 1)
    return m

model = build_model().to(device)
print("DenseNet121 built ✓")

# ── loaders ───────────────────────────────────────────────────
CLIENTS = '/content/drive/MyDrive/FairFedCXR/clients'

def get_client_loaders(name, batch_size=16, num_workers=2):
    tr = CheXpertDataset(f'{CLIENTS}/client_{name}_train.csv', IMG_ROOT, train_tf)
    va = CheXpertDataset(f'{CLIENTS}/client_{name}_val.csv',   IMG_ROOT, eval_tf)
    te = CheXpertDataset(f'{CLIENTS}/client_{name}_test.csv',  IMG_ROOT, eval_tf)
    return (
        DataLoader(tr, batch_size=batch_size, shuffle=True,
                   num_workers=num_workers, pin_memory=False),
        DataLoader(va, batch_size=batch_size, shuffle=False,
                   num_workers=num_workers, pin_memory=False),
        DataLoader(te, batch_size=batch_size, shuffle=False,
                   num_workers=num_workers, pin_memory=False),
    )

print("Loading Client A loaders...")
tr_loader, va_loader, te_loader = get_client_loaders('A', batch_size=16)

# ── one batch check ───────────────────────────────────────────
t0 = time.time()
imgs, labels, sex = next(iter(va_loader))
print(f"Batch loaded in {time.time()-t0:.1f}s")
print(f"  imgs:   {imgs.shape}   (expect [16,3,224,224])")
print(f"  labels: {labels[:8].tolist()}")
print(f"  sex:    {sex[:8].tolist()}   (1=M 0=F)")

# ── forward pass ──────────────────────────────────────────────
model.eval()
with torch.no_grad():
    t0 = time.time()
    logits = model(imgs.to(device)).squeeze()
    print(f"\nForward pass in {time.time()-t0:.1f}s")
    probs = torch.sigmoid(logits).cpu().numpy()
    print(f"  probs sample: {probs[:4].round(3)}")

# ── full val eval ─────────────────────────────────────────────
print("\nRunning full eval on Client A val set...")
t0 = time.time()
metrics = eval_model(model, va_loader, device)
print(f"Eval in {time.time()-t0:.1f}s")
print("\nUntrained metrics (AUROC should be ~0.5):")
for k,v in metrics.items():
    print(f"  {k}: {v if isinstance(v,int) else round(v,4)}")

# ── save report ───────────────────────────────────────────────
report = {
    'device': device,
    'img_root': IMG_ROOT,
    'batch_shape': list(imgs.shape),
    'auroc_untrained': round(metrics['auroc'],4),
    'eo_gap': round(metrics['eo_gap'],4),
    'val_m_pos': metrics['m_pos'],
    'val_f_pos': metrics['f_pos'],
    'path_fix': 'strip CheXpert-v1.0-small/ prefix',
}
with open('/content/drive/MyDrive/FairFedCXR/logs/smoke_test.json','w') as f:
    json.dump(report, f, indent=2)

print("\n✓ smoke_test.json saved to Drive")
print("✓ Day 3 COMPLETE — pipeline validated end to end")
print("✓ Next session: switch to L4 GPU and start Day 4 baselines")

Device: cuda
Downloading: "https://download.pytorch.org/models/densenet121-a639ec97.pth" to /root/.cache/torch/hub/checkpoints/densenet121-a639ec97.pth


100%|██████████| 30.8M/30.8M [00:00<00:00, 232MB/s]


DenseNet121 built ✓
Loading Client A loaders...
Batch loaded in 0.4s
  imgs:   torch.Size([16, 3, 224, 224])   (expect [16,3,224,224])
  labels: [0.0, 0.0, 1.0, 0.0, 0.0, 0.0, 0.0, 0.0]
  sex:    [0, 1, 1, 1, 0, 1, 1, 0]   (1=M 0=F)

Forward pass in 1.3s
  probs sample: [0.493 0.457 0.477 0.375]

Running full eval on Client A val set...
Eval in 5.9s

Untrained metrics (AUROC should be ~0.5):
  auroc: 0.7016
  auprc: 0.5891
  ece: 0.0991
  eo_gap: 0.0725
  male_tpr: 0.4115
  female_tpr: 0.339
  m_pos: 433
  f_pos: 467

✓ smoke_test.json saved to Drive
✓ Day 3 COMPLETE — pipeline validated end to end
✓ Next session: switch to L4 GPU and start Day 4 baselines


In [ ]:
# Baseline models starts day 4, 1
from google.colab import drive
drive.mount('/content/drive')

import os

# Disk space check — need 12GB free for images
stat    = os.statvfs('/content')
free_gb = (stat.f_bavail * stat.f_frsize) / 1e9
print(f"Free disk: {free_gb:.1f} GB")
if free_gb < 12:
    raise SystemExit("❌ Not enough disk space. Restart runtime.")

# Unzip to local SSD only if not already there
if not os.path.exists('/content/chexpert/train'):
    print("Unzipping to local SSD (3–5 min)...")
    os.makedirs('/content/chexpert', exist_ok=True)
    !unzip -q /content/drive/MyDrive/FairFedCXR/data/chexpert.zip \
           -d /content/chexpert/
    print("Unzip done ✓")
else:
    print("Images already on local SSD ✓")

# Install only what is needed
!pip install -q torch torchvision scikit-learn pandas tqdm scipy

# ── Global config — locked, never change ─────────────────────
IMG_ROOT     = '/content/chexpert'
CLIENTS      = '/content/drive/MyDrive/FairFedCXR/clients'
CKPTS        = '/content/drive/MyDrive/FairFedCXR/checkpoints'
RESULTS      = '/content/drive/MyDrive/FairFedCXR/results'
SEEDS        = [42, 123, 456]
EPOCHS       = 30
PATIENCE     = 5
BATCH_SIZE   = 32
LR           = 1e-4
WEIGHT_DECAY = 1e-4
EPS          = 1.0    # Laplace smoothing for TPR — matches LaTeX
ECE_BINS     = 8      # M=8 — matches spec sheet

os.makedirs(CKPTS,   exist_ok=True)
os.makedirs(RESULTS, exist_ok=True)

print(f"\n✓ Ready")
print(f"  SEEDS={SEEDS} | EPOCHS={EPOCHS} | PATIENCE={PATIENCE}")
print(f"  BATCH={BATCH_SIZE} | LR={LR} | EPS={EPS} | ECE_BINS={ECE_BINS}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Free disk: 177.9 GB
Images already on local SSD ✓

✓ Ready
  SEEDS=[42, 123, 456] | EPOCHS=30 | PATIENCE=5
  BATCH=32 | LR=0.0001 | EPS=1.0 | ECE_BINS=8


In [ ]:
# baseline model day 4, 2
import torch, numpy as np, random, gc, time, os
import torchvision.models as models
from torch.utils.data import Dataset, DataLoader
from torch.cuda.amp import autocast, GradScaler
from torchvision import transforms
from PIL import Image
import pandas as pd
from tqdm import tqdm
from sklearn.metrics import roc_auc_score, average_precision_score

# Fixed 224×224 input — benchmark mode speeds up GPU convolutions
torch.backends.cudnn.benchmark = True

# ── Seed control ──────────────────────────────────────────────
def set_seed(seed):
    torch.manual_seed(seed)
    np.random.seed(seed)
    random.seed(seed)
    torch.cuda.manual_seed_all(seed)
    print(f"  Seed locked: {seed}")

# ── Transforms ────────────────────────────────────────────────
train_tf = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(0.5),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406],
                         [0.229, 0.224, 0.225]),
])
eval_tf = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406],
                         [0.229, 0.224, 0.225]),
])

# ── Dataset ───────────────────────────────────────────────────
class CheXpertDataset(Dataset):
    def __init__(self, csv_path, transform):
        self.df = pd.read_csv(csv_path)
        self.tf = transform
        # F3: assert sex encoding at load — catches silent errors
        assert self.df['sex_encoded'].isin([0, 1]).all(), \
            "sex_encoded must be 0 or 1"
        assert (self.df.loc[self.df['Sex']=='Male',
                'sex_encoded'] == 1).all(), \
            "Male must be encoded as 1"
        assert (self.df.loc[self.df['Sex']=='Female',
                'sex_encoded'] == 0).all(), \
            "Female must be encoded as 0"

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row   = self.df.iloc[idx]
        # Strip prefix — images live at /content/chexpert/train/...
        clean = row['Path'].replace('CheXpert-v1.0-small/', '')
        img   = Image.open(
                    os.path.join(IMG_ROOT, clean)).convert('RGB')
        img   = self.tf(img)
        label = torch.tensor(row['label'],       dtype=torch.float32)
        sex   = torch.tensor(row['sex_encoded'], dtype=torch.long)
        return img, label, sex

# ── Loader factory ────────────────────────────────────────────
def make_loader(csv_path, train=False):
    on_gpu = torch.cuda.is_available()
    ds = CheXpertDataset(csv_path, train_tf if train else eval_tf)
    return DataLoader(
        ds,
        batch_size         = BATCH_SIZE,
        shuffle            = train,
        num_workers        = 4 if on_gpu else 2,
        pin_memory         = on_gpu,
        persistent_workers = True,
        prefetch_factor    = 2,
    )

# ── Model builder ─────────────────────────────────────────────
def build_model(device):
    m = models.densenet121(weights='IMAGENET1K_V1')
    m.classifier = torch.nn.Linear(m.classifier.in_features, 1)
    return m.to(device)

print("✓ Imports, Dataset, Loader, Model defined")

✓ Imports, Dataset, Loader, Model defined


In [ ]:
# baseline model day 4, 3
def compute_ece(probs, labels, M=ECE_BINS):
    """
    Expected Calibration Error.
    M=8 bins — matches spec sheet.
    """
    bins = np.linspace(0, 1, M + 1)
    ece  = 0.0
    for lo, hi in zip(bins[:-1], bins[1:]):
        mask = (probs >= lo) & (probs < hi)
        if mask.sum() == 0:
            continue
        acc  = labels[mask].mean()
        conf = probs[mask].mean()
        ece += mask.mean() * abs(acc - conf)
    return float(ece)


def compute_eo_gap(probs, labels, sex, threshold=0.5):
    """
    Equal opportunity gap |male_TPR - female_TPR|.
    Laplace smoothing EPS=1.0 — matches LaTeX exactly, never NaN.
    Sex encoding: 1=Male, 0=Female (F3 locked).
    """
    preds = (probs >= threshold).astype(int)

    def smoothed_tpr(mask):
        pos = (labels[mask] == 1)
        tp  = int(((preds[mask] == 1) & pos).sum())
        p   = int(pos.sum())
        return (tp + EPS) / (p + 2 * EPS)

    m_tpr = smoothed_tpr(sex == 1)   # male
    f_tpr = smoothed_tpr(sex == 0)   # female
    return abs(m_tpr - f_tpr), m_tpr, f_tpr


@torch.no_grad()
def evaluate(model, loader, device, threshold=0.5):
    """
    Full evaluation on a DataLoader.
    Forward pass in fp16 for speed.
    All metrics computed in fp32 numpy — never on GPU.
    """
    model.eval()
    P, L, S = [], [], []

    for imgs, labels, sex in loader:
        imgs = imgs.to(device, non_blocking=True)
        with torch.amp.autocast('cuda'):          # fixed — no FutureWarning
            logits = model(imgs).squeeze()
        # move to CPU fp32 immediately for metric computation
        P.append(torch.sigmoid(logits.float()).cpu().numpy())
        L.append(labels.numpy())
        S.append(sex.numpy())

    probs  = np.concatenate(P)
    labels = np.concatenate(L)
    sex    = np.concatenate(S)

    eo, m_tpr, f_tpr = compute_eo_gap(probs, labels, sex, threshold)

    return {
        'auroc':      float(roc_auc_score(labels, probs)),
        'auprc':      float(average_precision_score(labels, probs)),
        'ece':        float(compute_ece(probs, labels)),
        'eo_gap':     float(eo),
        'male_tpr':   float(m_tpr),
        'female_tpr': float(f_tpr),
        'm_pos':      int((labels[sex == 1] == 1).sum()),
        'f_pos':      int((labels[sex == 0] == 1).sum()),
    }


print(f"✓ Metric module defined — FutureWarning fixed")
print(f"  ECE bins={ECE_BINS} | TPR smoothing EPS={EPS}")
print(f"  Sex encoding: 1=Male  0=Female")
print(f"  autocast: torch.amp.autocast('cuda')")

✓ Metric module defined — FutureWarning fixed
  ECE bins=8 | TPR smoothing EPS=1.0
  Sex encoding: 1=Male  0=Female
  autocast: torch.amp.autocast('cuda')


In [ ]:
# baseline model day 4, 4
def train_model(model, train_loader, val_loader,
                device, tag="model", ckpt_path=None):
    """
    fp16 mixed precision training.
    Max 30 epochs, early stopping patience=5 on val AUROC.
    Best model = highest val AUROC.
    Checkpoint saved to Drive immediately when best is found.
    Returns: best model, training history, best val AUROC.
    """
    optimizer = torch.optim.AdamW(
        model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
    criterion = torch.nn.BCEWithLogitsLoss()
    scaler    = torch.amp.GradScaler('cuda')      # fixed — no FutureWarning

    best_auroc       = -1.0
    best_state       = None
    patience_counter = 0
    history          = []

    for ep in range(EPOCHS):

        # ── TRAIN ─────────────────────────────────────────────
        model.train()
        running_loss = 0.0

        for imgs, labels, _ in tqdm(
                train_loader,
                desc=f"{tag} ep{ep+1}/{EPOCHS}",
                leave=False):
            imgs   = imgs.to(device,   non_blocking=True)
            labels = labels.to(device, non_blocking=True)
            optimizer.zero_grad()

            with torch.amp.autocast('cuda'):      # fixed — no FutureWarning
                logits = model(imgs).squeeze()
                loss   = criterion(logits, labels)

            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
            running_loss += loss.item()

        avg_loss = running_loss / len(train_loader)

        # ── VALIDATE ──────────────────────────────────────────
        m          = evaluate(model, val_loader, device)
        m['epoch'] = ep + 1
        m['loss']  = round(avg_loss, 4)
        history.append(m)

        print(f"  {tag} ep{ep+1:02d}: "
              f"loss={avg_loss:.4f} | "
              f"AUROC={m['auroc']:.4f} | "
              f"EO={m['eo_gap']:.4f} | "
              f"ECE={m['ece']:.4f}")

        # ── BEST CHECKPOINT + EARLY STOP ──────────────────────
        if m['auroc'] > best_auroc:
            best_auroc       = m['auroc']
            best_state       = {k: v.cpu().clone()
                                for k, v in model.state_dict().items()}
            patience_counter = 0
            if ckpt_path:
                torch.save(best_state, ckpt_path)
                print(f"    ✓ Best saved → "
                      f"{os.path.basename(ckpt_path)} "
                      f"(AUROC={best_auroc:.4f})")
        else:
            patience_counter += 1
            print(f"    patience {patience_counter}/{PATIENCE}")
            if patience_counter >= PATIENCE:
                print(f"  ⚡ Early stop at ep{ep+1}")
                break

    model.load_state_dict(best_state)
    return model, history, best_auroc


print(f"✓ Training engine defined — FutureWarning fixed")
print(f"  fp16       : torch.amp.GradScaler('cuda')")
print(f"  autocast   : torch.amp.autocast('cuda')")
print(f"  max_epochs : {EPOCHS}")
print(f"  patience   : {PATIENCE}")

✓ Training engine defined — FutureWarning fixed
  fp16       : torch.amp.GradScaler('cuda')
  autocast   : torch.amp.autocast('cuda')
  max_epochs : 30
  patience   : 5


In [ ]:
# baseline day 4,6
device = 'cuda'
print(f"GPU : {torch.cuda.get_device_name(0)}")
print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB")
print(f"\nLOCAL-ONLY | {len(SEEDS)} seeds × 4 clients × "
      f"up to {EPOCHS} epochs | fp16=True\n")

local_results = []
t_start       = time.time()

for seed in SEEDS:
    print(f"\n{'='*58}")
    print(f"  LOCAL-ONLY | SEED {seed}")
    print(f"{'='*58}")
    set_seed(seed)

    for client in ['A', 'B', 'C', 'D']:
        tag       = f"local_{client}_seed{seed}"
        ckpt_path = f"{CKPTS}/{tag}.pt"

        # ── CRASH RECOVERY ────────────────────────────────────
        # If checkpoint exists, skip training and go to test eval
        if os.path.exists(ckpt_path):
            print(f"\n  ⚡ {tag} already done — loading checkpoint")
            model = build_model(device)
            model.load_state_dict(
                torch.load(ckpt_path, map_location=device))

        else:
            # ── TRAIN ─────────────────────────────────────────
            print(f"\n  Training {tag}...")
            tr    = make_loader(
                f'{CLIENTS}/client_{client}_train.csv',
                train=True)
            va    = make_loader(
                f'{CLIENTS}/client_{client}_val.csv',
                train=False)
            model = build_model(device)

            model, _, best_auroc = train_model(
                model, tr, va, device,
                tag=tag,
                ckpt_path=ckpt_path)

            print(f"  Best val AUROC: {best_auroc:.4f}")

            # Free train/val memory before test eval
            del tr, va
            gc.collect()

        # ── TEST EVALUATION ───────────────────────────────────
        # Only on held-out test set — never val
        te = make_loader(
            f'{CLIENTS}/client_{client}_test.csv',
            train=False)
        tm = evaluate(model, te, device)
        tm.update({
            'method': 'local_only',
            'client': client,
            'seed':   seed,
        })
        local_results.append(tm)

        print(f"  → Client {client} TEST: "
              f"AUROC={tm['auroc']:.4f} | "
              f"AUPRC={tm['auprc']:.4f} | "
              f"EO-gap={tm['eo_gap']:.4f} | "
              f"m_pos={tm['m_pos']} | "
              f"f_pos={tm['f_pos']}")

        # ── CLEANUP between models ────────────────────────────
        del model, te
        gc.collect()
        torch.cuda.empty_cache()

        # ── SAVE after every client-seed pair ─────────────────
        # Protects results if session crashes mid-run
        pd.DataFrame(local_results).to_csv(
            f'{RESULTS}/local_only_all.csv', index=False)

# ── FINAL REPORT ──────────────────────────────────────────────
elapsed = (time.time() - t_start) / 3600
print(f"\n{'='*58}")
print(f"LOCAL-ONLY COMPLETE in {elapsed:.2f} hr")
print(f"✓ Results  → {RESULTS}/local_only_all.csv")
print(f"✓ Ckpts    → {CKPTS}/local_[A-D]_seed[42/123/456].pt")
print(f"  Total rows in CSV: {len(local_results)} "
      f"(expect 12 = 4 clients × 3 seeds)")

GPU : NVIDIA L4
VRAM: 23.7 GB

LOCAL-ONLY | 3 seeds × 4 clients × up to 30 epochs | fp16=True


  LOCAL-ONLY | SEED 42
  Seed locked: 42

  Training local_A_seed42...


  local_A_seed42 ep01: loss=0.3385 | AUROC=0.9400 | EO=0.0323 | ECE=0.0240
    ✓ Best saved → local_A_seed42.pt (AUROC=0.9400)


  local_A_seed42 ep02: loss=0.2730 | AUROC=0.9479 | EO=0.0330 | ECE=0.0197
    ✓ Best saved → local_A_seed42.pt (AUROC=0.9479)


  local_A_seed42 ep03: loss=0.2315 | AUROC=0.9415 | EO=0.0238 | ECE=0.0185
    patience 1/5


  local_A_seed42 ep04: loss=0.1932 | AUROC=0.9432 | EO=0.0508 | ECE=0.0684
    patience 2/5


...

  local_D_seed456 ep03: loss=0.1905 | AUROC=0.9411 | EO=0.0270 | ECE=0.0980
    patience 1/5


  local_D_seed456 ep04: loss=0.1409 | AUROC=0.9453 | EO=0.0122 | ECE=0.0365
    patience 2/5


  local_D_seed456 ep05: loss=0.1006 | AUROC=0.9423 | EO=0.0461 | ECE=0.0629
    patience 3/5


  local_D_seed456 ep06: loss=0.0650 | AUROC=0.9405 | EO=0.0239 | ECE=0.0674
    patience 4/5


  local_D_seed456 ep07: loss=0.0481 | AUROC=0.9338 | EO=0.0476 | ECE=0.0885
    patience 5/5
  ⚡ Early stop at ep7
  Best val AUROC: 0.9466
  → Client D TEST: AUROC=0.9471 | AUPRC=0.9092 | EO-gap=0.0535 | m_pos=212 | f_pos=130

LOCAL-ONLY COMPLETE in 0.65 hr
✓ Results  → /content/drive/MyDrive/FairFedCXR/results/local_only_all.csv
✓ Ckpts    → /content/drive/MyDrive/FairFedCXR/checkpoints/local_[A-D]_seed[42/123/456].pt
  Total rows in CSV: 12 (expect 12 = 4 clients × 3 seeds)


In [ ]:
#baseline model day 4,7
import pandas as pd, numpy as np

df = pd.read_csv(f'{RESULTS}/local_only_all.csv')

# Sanity check
expected_rows = 12   # 4 clients × 3 seeds
print(f"Rows in CSV: {len(df)} (expect {expected_rows})")
assert len(df) == expected_rows, \
    f"Expected {expected_rows} rows, got {len(df)} — check for missing runs"

print(f"\n{'─'*72}")
print("LOCAL-ONLY — Test results  (mean ± std across 3 seeds)")
print(f"{'─'*72}")
print(f"  {'Client':<6}  {'AUROC':>16}  {'AUPRC':>16}  "
      f"{'ECE':>14}  {'EO-gap':>14}")
print(f"{'─'*72}")

metrics = ['auroc', 'auprc', 'ece', 'eo_gap']
for c in ['A', 'B', 'C', 'D']:
    sub = df[df['client'] == c]
    def ms(col):
        return f"{sub[col].mean():.4f}±{sub[col].std():.4f}"
    print(f"  {c}       "
          f"  {ms('auroc'):>16}"
          f"  {ms('auprc'):>16}"
          f"  {ms('ece'):>14}"
          f"  {ms('eo_gap'):>14}")

print(f"{'─'*72}")

# Pooled average: mean across clients per seed, then mean±std across seeds
pooled = df.groupby('seed')[metrics].mean()
print(f"  {'Pooled':<6}"
      f"  {pooled['auroc'].mean():.4f}±{pooled['auroc'].std():.4f}"
      f"  {pooled['auprc'].mean():.4f}±{pooled['auprc'].std():.4f}"
      f"  {pooled['ece'].mean():.4f}±{pooled['ece'].std():.4f}"
      f"  {pooled['eo_gap'].mean():.4f}±{pooled['eo_gap'].std():.4f}")
print(f"{'─'*72}")

# Also print per-sex TPR for Client C — key for paper narrative
print(f"\nClient C — male vs female TPR (C2 protocol reveal):")
c_df = df[df['client'] == 'C']
print(f"  Male TPR  : {c_df['male_tpr'].mean():.4f} ± "
      f"{c_df['male_tpr'].std():.4f}")
print(f"  Female TPR: {c_df['female_tpr'].mean():.4f} ± "
      f"{c_df['female_tpr'].std():.4f}")
print(f"  EO-gap    : {c_df['eo_gap'].mean():.4f} ± "
      f"{c_df['eo_gap'].std():.4f}")
print(f"  (This is the hidden failure C2 reveals)")

print(f"\n✓ Session 1 complete")
print(f"✓ Save this output — needed for paper Table 2")
print(f"✓ Start Session 2 when ready (centralized baseline)")

Rows in CSV: 12 (expect 12)

────────────────────────────────────────────────────────────────────────
LOCAL-ONLY — Test results  (mean ± std across 3 seeds)
────────────────────────────────────────────────────────────────────────
  Client             AUROC             AUPRC             ECE          EO-gap
────────────────────────────────────────────────────────────────────────
  A            0.9440±0.0010     0.9069±0.0077   0.0406±0.0223   0.0083±0.0030
  B            0.9343±0.0048     0.9158±0.0020   0.0478±0.0236   0.0252±0.0212
  C            0.9190±0.0066     0.8543±0.0156   0.0427±0.0240   0.1507±0.0429
  D            0.9478±0.0047     0.9138±0.0097   0.0466±0.0279   0.0312±0.0252
────────────────────────────────────────────────────────────────────────
  Pooled  0.9363±0.0018  0.8977±0.0038  0.0444±0.0229  0.0538±0.0032
────────────────────────────────────────────────────────────────────────

Client C — male vs female TPR (C2 protocol reveal):
  Male TPR  : 0.6854 ± 0.0674
  Fema

In [ ]:
# centralized baseline day 4,1
from google.colab import drive
drive.mount('/content/drive')

import os

# Disk space check
stat    = os.statvfs('/content')
free_gb = (stat.f_bavail * stat.f_frsize) / 1e9
print(f"Free disk: {free_gb:.1f} GB")
if free_gb < 12:
    raise SystemExit("❌ Not enough disk space. Restart runtime.")

# Unzip to local SSD only if needed
if not os.path.exists('/content/chexpert/train'):
    print("Unzipping to local SSD (3–5 min)...")
    os.makedirs('/content/chexpert', exist_ok=True)
    !unzip -q /content/drive/MyDrive/FairFedCXR/data/chexpert.zip \
           -d /content/chexpert/
    print("Unzip done ✓")
else:
    print("Images already on local SSD ✓")

# Install packages
!pip install -q torch torchvision scikit-learn \
             pandas tqdm scipy matplotlib

# Global config — identical to Session 1, never change
IMG_ROOT     = '/content/chexpert'
CLIENTS      = '/content/drive/MyDrive/FairFedCXR/clients'
CKPTS        = '/content/drive/MyDrive/FairFedCXR/checkpoints'
RESULTS      = '/content/drive/MyDrive/FairFedCXR/results'
FIGURES      = '/content/drive/MyDrive/FairFedCXR/figures'
SEEDS        = [42, 123, 456]
EPOCHS       = 30
PATIENCE     = 5
BATCH_SIZE   = 32
LR           = 1e-4
WEIGHT_DECAY = 1e-4
EPS          = 1.0
ECE_BINS     = 8

os.makedirs(CKPTS,   exist_ok=True)
os.makedirs(RESULTS, exist_ok=True)
os.makedirs(FIGURES, exist_ok=True)

print(f"\n✓ Ready")
print(f"  SEEDS={SEEDS} | EPOCHS={EPOCHS} | PATIENCE={PATIENCE}")
print(f"  BATCH={BATCH_SIZE} | LR={LR} | EPS={EPS} | ECE_BINS={ECE_BINS}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Free disk: 177.9 GB
Images already on local SSD ✓

✓ Ready
  SEEDS=[42, 123, 456] | EPOCHS=30 | PATIENCE=5
  BATCH=32 | LR=0.0001 | EPS=1.0 | ECE_BINS=8


In [ ]:
# centralized baseline 4,2
import torch, numpy as np, random, gc, time, os
import torchvision.models as models
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from PIL import Image
import pandas as pd
from tqdm import tqdm
from sklearn.metrics import roc_auc_score, average_precision_score
import matplotlib
matplotlib.use('Agg')   # no display needed — saves to file
import matplotlib.pyplot as plt

torch.backends.cudnn.benchmark = True

# ── Seed control ──────────────────────────────────────────────
def set_seed(seed):
    torch.manual_seed(seed)
    np.random.seed(seed)
    random.seed(seed)
    torch.cuda.manual_seed_all(seed)
    print(f"  Seed locked: {seed}")

# ── Transforms ────────────────────────────────────────────────
train_tf = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(0.5),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406],
                         [0.229, 0.224, 0.225]),
])
eval_tf = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406],
                         [0.229, 0.224, 0.225]),
])

# ── Dataset ───────────────────────────────────────────────────
class CheXpertDataset(Dataset):
    def __init__(self, csv_path, transform):
        self.df = pd.read_csv(csv_path)
        self.tf = transform
        # F3: assert sex encoding at load
        assert self.df['sex_encoded'].isin([0, 1]).all()
        assert (self.df.loc[self.df['Sex']=='Male',
                'sex_encoded'] == 1).all()
        assert (self.df.loc[self.df['Sex']=='Female',
                'sex_encoded'] == 0).all()

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row   = self.df.iloc[idx]
        clean = row['Path'].replace('CheXpert-v1.0-small/', '')
        img   = Image.open(
                    os.path.join(IMG_ROOT, clean)).convert('RGB')
        img   = self.tf(img)
        label = torch.tensor(row['label'],       dtype=torch.float32)
        sex   = torch.tensor(row['sex_encoded'], dtype=torch.long)
        return img, label, sex

# ── Loader factory ────────────────────────────────────────────
def make_loader(csv_path, train=False):
    on_gpu = torch.cuda.is_available()
    ds = CheXpertDataset(csv_path, train_tf if train else eval_tf)
    return DataLoader(
        ds,
        batch_size         = BATCH_SIZE,
        shuffle            = train,
        num_workers        = 4 if on_gpu else 2,
        pin_memory         = on_gpu,
        persistent_workers = True,
        prefetch_factor    = 2,
    )

# ── Model builder ─────────────────────────────────────────────
def build_model(device):
    m = models.densenet121(weights='IMAGENET1K_V1')
    m.classifier = torch.nn.Linear(m.classifier.in_features, 1)
    return m.to(device)

print("✓ Imports, Dataset, Loader, Model defined")

✓ Imports, Dataset, Loader, Model defined


In [ ]:
# centralized baseline 4,3
def compute_ece(probs, labels, M=ECE_BINS):
    bins = np.linspace(0, 1, M + 1)
    ece  = 0.0
    for lo, hi in zip(bins[:-1], bins[1:]):
        mask = (probs >= lo) & (probs < hi)
        if mask.sum() == 0:
            continue
        ece += mask.mean() * abs(labels[mask].mean()
                                  - probs[mask].mean())
    return float(ece)

def compute_eo_gap(probs, labels, sex, threshold=0.5):
    preds = (probs >= threshold).astype(int)
    def smoothed_tpr(mask):
        pos = (labels[mask] == 1)
        tp  = int(((preds[mask] == 1) & pos).sum())
        p   = int(pos.sum())
        return (tp + EPS) / (p + 2 * EPS)
    m_tpr = smoothed_tpr(sex == 1)
    f_tpr = smoothed_tpr(sex == 0)
    return abs(m_tpr - f_tpr), m_tpr, f_tpr

@torch.no_grad()
def evaluate(model, loader, device, threshold=0.5):
    model.eval()
    P, L, S = [], [], []
    for imgs, labels, sex in loader:
        imgs = imgs.to(device, non_blocking=True)
        with torch.amp.autocast('cuda'):
            logits = model(imgs).squeeze()
        P.append(torch.sigmoid(logits.float()).cpu().numpy())
        L.append(labels.numpy())
        S.append(sex.numpy())
    probs  = np.concatenate(P)
    labels = np.concatenate(L)
    sex    = np.concatenate(S)
    eo, m_tpr, f_tpr = compute_eo_gap(
        probs, labels, sex, threshold)
    return {
        'auroc':      float(roc_auc_score(labels, probs)),
        'auprc':      float(average_precision_score(labels, probs)),
        'ece':        float(compute_ece(probs, labels)),
        'eo_gap':     float(eo),
        'male_tpr':   float(m_tpr),
        'female_tpr': float(f_tpr),
        'm_pos':      int((labels[sex == 1] == 1).sum()),
        'f_pos':      int((labels[sex == 0] == 1).sum()),
    }

print(f"✓ Metric module defined")
print(f"  ECE bins={ECE_BINS} | EPS={EPS} | 1=Male 0=Female")

✓ Metric module defined
  ECE bins=8 | EPS=1.0 | 1=Male 0=Female


In [ ]:
# centralized baseline 4,4
def train_model(model, train_loader, val_loader,
                device, tag="model", ckpt_path=None):
    optimizer = torch.optim.AdamW(
        model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
    criterion = torch.nn.BCEWithLogitsLoss()
    scaler    = torch.amp.GradScaler('cuda')

    best_auroc       = -1.0
    best_state       = None
    patience_counter = 0
    history          = []

    for ep in range(EPOCHS):
        # ── TRAIN ─────────────────────────────────────────────
        model.train()
        running_loss = 0.0
        for imgs, labels, _ in tqdm(
                train_loader,
                desc=f"{tag} ep{ep+1}/{EPOCHS}",
                leave=False):
            imgs   = imgs.to(device,   non_blocking=True)
            labels = labels.to(device, non_blocking=True)
            optimizer.zero_grad()
            with torch.amp.autocast('cuda'):
                logits = model(imgs).squeeze()
                loss   = criterion(logits, labels)
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
            running_loss += loss.item()

        avg_loss = running_loss / len(train_loader)

        # ── VALIDATE ──────────────────────────────────────────
        m          = evaluate(model, val_loader, device)
        m['epoch'] = ep + 1
        m['loss']  = round(avg_loss, 4)
        history.append(m)

        print(f"  {tag} ep{ep+1:02d}: "
              f"loss={avg_loss:.4f} | "
              f"AUROC={m['auroc']:.4f} | "
              f"EO={m['eo_gap']:.4f} | "
              f"ECE={m['ece']:.4f}")

        # ── BEST CHECKPOINT + EARLY STOP ──────────────────────
        if m['auroc'] > best_auroc:
            best_auroc       = m['auroc']
            best_state       = {k: v.cpu().clone()
                                for k, v in model.state_dict().items()}
            patience_counter = 0
            if ckpt_path:
                torch.save(best_state, ckpt_path)
                print(f"    ✓ Best saved → "
                      f"{os.path.basename(ckpt_path)} "
                      f"(AUROC={best_auroc:.4f})")
        else:
            patience_counter += 1
            print(f"    patience {patience_counter}/{PATIENCE}")
            if patience_counter >= PATIENCE:
                print(f"  ⚡ Early stop at ep{ep+1}")
                break

    model.load_state_dict(best_state)
    return model, history, best_auroc

print(f"✓ Training engine defined")
print(f"  fp16=True | max_epochs={EPOCHS} | patience={PATIENCE}")

✓ Training engine defined
  fp16=True | max_epochs=30 | patience=5


In [ ]:
# centralized baseline day 4,5
import pandas as pd

pooled_train = pd.concat([
    pd.read_csv(f'{CLIENTS}/client_{c}_train.csv')
    for c in ['A', 'B', 'C', 'D']
], ignore_index=True)

pooled_val = pd.concat([
    pd.read_csv(f'{CLIENTS}/client_{c}_val.csv')
    for c in ['A', 'B', 'C', 'D']
], ignore_index=True)

# Save to local SSD — fast reads during training
pooled_train.to_csv('/content/pooled_train.csv', index=False)
pooled_val.to_csv('/content/pooled_val.csv',     index=False)

print(f"Pooled train : {len(pooled_train):,} images")
print(f"Pooled val   : {len(pooled_val):,} images")
print(f"Prevalence   : {pooled_train['label'].mean():.3f}")
print(f"Female frac  : {(pooled_train['Sex']=='Female').mean():.3f}")
print(f"✓ Pooled CSVs saved to local SSD")

Pooled train : 25,129 images
Pooled val   : 5,385 images
Prevalence   : 0.403
Female frac  : 0.445
✓ Pooled CSVs saved to local SSD


In [ ]:
# centralized baseline day 4, 6
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"GPU : {torch.cuda.get_device_name(0) if device=='cuda' else 'CPU'}")

# Save real values, override for debug
_ep_save  = EPOCHS
_pat_save = PATIENCE
EPOCHS    = 2
PATIENCE  = 999

print("\n── DEBUG: Centralized, 2 epochs, seed 42 ───────────────")
set_seed(42)
tr    = make_loader('/content/pooled_train.csv', train=True)
va    = make_loader('/content/pooled_val.csv',   train=False)
model = build_model(device)

model, hist, best = train_model(
    model, tr, va, device,
    tag="DEBUG-CEN", ckpt_path=None)

print(f"\n✓ Debug complete | Best val AUROC: {best:.4f}")
print("✓ No errors = pipeline healthy")
print("✓ Switch to L4 for full production run")

# Restore real values
EPOCHS   = _ep_save
PATIENCE = _pat_save
del model, tr, va
gc.collect()
torch.cuda.empty_cache()

In [ ]:
# centalized baseline day 4,7
device = 'cuda'
print(f"GPU : {torch.cuda.get_device_name(0)}")
print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB")
print(f"\nCENTRALIZED | {len(SEEDS)} seeds × "
      f"up to {EPOCHS} epochs | fp16=True\n")

cen_results = []
t_start     = time.time()

for seed in SEEDS:
    print(f"\n{'='*58}")
    print(f"  CENTRALIZED | SEED {seed}")
    print(f"{'='*58}")
    set_seed(seed)

    tag       = f"centralized_seed{seed}"
    ckpt_path = f"{CKPTS}/{tag}.pt"

    # ── CRASH RECOVERY ────────────────────────────────────────
    if os.path.exists(ckpt_path):
        print(f"  ⚡ {tag} checkpoint found — loading")
        model = build_model(device)
        model.load_state_dict(
            torch.load(ckpt_path, map_location=device))
    else:
        # ── TRAIN on pooled data ───────────────────────────────
        print(f"  Training on pooled data...")
        tr    = make_loader('/content/pooled_train.csv', train=True)
        va    = make_loader('/content/pooled_val.csv',   train=False)
        model = build_model(device)

        model, _, best_auroc = train_model(
            model, tr, va, device,
            tag=tag, ckpt_path=ckpt_path)

        print(f"  Best val AUROC: {best_auroc:.4f}")
        del tr, va
        gc.collect()

    # ── TEST on ALL 4 clients separately ──────────────────────
    # Per-client reporting — C2 contribution
    for client in ['A', 'B', 'C', 'D']:
        te = make_loader(
            f'{CLIENTS}/client_{client}_test.csv',
            train=False)
        tm = evaluate(model, te, device)
        tm.update({
            'method': 'centralized',
            'client': client,
            'seed':   seed,
        })
        cen_results.append(tm)
        print(f"  → Client {client} TEST: "
              f"AUROC={tm['auroc']:.4f} | "
              f"AUPRC={tm['auprc']:.4f} | "
              f"EO-gap={tm['eo_gap']:.4f} | "
              f"m_pos={tm['m_pos']} | "
              f"f_pos={tm['f_pos']}")
        del te
        gc.collect()

    del model
    gc.collect()
    torch.cuda.empty_cache()

    # Save after every seed — crash protection
    pd.DataFrame(cen_results).to_csv(
        f'{RESULTS}/centralized_all.csv', index=False)
    print(f"  ✓ Results saved after seed {seed}")

elapsed = (time.time() - t_start) / 3600
print(f"\n{'='*58}")
print(f"CENTRALIZED COMPLETE in {elapsed:.2f} hr")
print(f"✓ Results  → {RESULTS}/centralized_all.csv")
print(f"✓ Ckpts    → {CKPTS}/centralized_seed[42/123/456].pt")
print(f"  Total rows: {len(cen_results)} (expect 12 = 4 clients × 3 seeds)")

GPU : NVIDIA L4
VRAM: 23.7 GB

CENTRALIZED | 3 seeds × up to 30 epochs | fp16=True


  CENTRALIZED | SEED 42
  Seed locked: 42
  Training on pooled data...


  centralized_seed42 ep01: loss=0.3128 | AUROC=0.9477 | EO=0.0254 | ECE=0.0099
    ✓ Best saved → centralized_seed42.pt (AUROC=0.9477)


  centralized_seed42 ep02: loss=0.2669 | AUROC=0.9457 | EO=0.0326 | ECE=0.0176
    patience 1/5


  centralized_seed42 ep03: loss=0.2423 | AUROC=0.9498 | EO=0.0080 | ECE=0.0517
    ✓ Best saved → centralized_seed42.pt (AUROC=0.9498)


  centralized_seed42 ep04: loss=0.2206 | AUROC=0.9522 | EO=0.0005 | ECE=0.0386
    ✓ Best saved → centralized_seed42.pt (AUROC=0.9522)


...

  centralized_seed456 ep03: loss=0.2440 | AUROC=0.9523 | EO=0.0307 | ECE=0.0281
    patience 1/5


  centralized_seed456 ep04: loss=0.2241 | AUROC=0.9483 | EO=0.0306 | ECE=0.0245
    patience 2/5


  centralized_seed456 ep05: loss=0.1944 | AUROC=0.9497 | EO=0.0217 | ECE=0.0422
    patience 3/5


  centralized_seed456 ep06: loss=0.1701 | AUROC=0.9459 | EO=0.0417 | ECE=0.0369
    patience 4/5


  centralized_seed456 ep07: loss=0.1354 | AUROC=0.9401 | EO=0.0464 | ECE=0.0474
    patience 5/5
  ⚡ Early stop at ep7
  Best val AUROC: 0.9526
  → Client A TEST: AUROC=0.9515 | AUPRC=0.9216 | EO-gap=0.0040 | m_pos=425 | f_pos=475
  → Client B TEST: AUROC=0.9524 | AUPRC=0.9344 | EO-gap=0.0008 | m_pos=411 | f_pos=129
  → Client C TEST: AUROC=0.9363 | AUPRC=0.8641 | EO-gap=0.0244 | m_pos=87 | f_pos=303
  → Client D TEST: AUROC=0.9642 | AUPRC=0.9376 | EO-gap=0.0317 | m_pos=212 | f_pos=130
  ✓ Results saved after seed 456

CENTRALIZED COMPLETE in 0.70 hr
✓ Results  → /content/drive/MyDrive/FairFedCXR/results/centralized_all.csv
✓ Ckpts    → /content/drive/MyDrive/FairFedCXR/checkpoints/centralized_seed[42/123/456].pt
  Total rows: 12 (expect 12 = 4 clients × 3 seeds)


In [ ]:
#centralized baseline day 4,8
import pandas as pd, numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import os

# ── Load both result files ────────────────────────────────────
local_df = pd.read_csv(f'{RESULTS}/local_only_all.csv')
cen_df   = pd.read_csv(f'{RESULTS}/centralized_all.csv')

assert len(local_df) == 12, f"Expected 12 local rows, got {len(local_df)}"
assert len(cen_df)   == 12, f"Expected 12 centralized rows, got {len(cen_df)}"

clients = ['A', 'B', 'C', 'D']
metrics = ['auroc', 'auprc', 'ece', 'eo_gap']

# ── Helper: mean±std per client ───────────────────────────────
def client_stats(df, metric):
    out = {}
    for c in clients:
        sub = df[df['client'] == c][metric]
        out[c] = (sub.mean(), sub.std())
    return out

def pooled_stats(df, metric):
    pooled = df.groupby('seed')[metric].mean()
    return pooled.mean(), pooled.std()

def ms(mean, std):
    return f"{mean:.4f}±{std:.4f}"

# ── TABLE 1: Centralized results ──────────────────────────────
print("CENTRALIZED — Test results (mean ± std, 3 seeds)")
print(f"\n{'─'*74}")
print(f"  {'Client':<6}  {'AUROC':>16}  {'AUPRC':>16}  "
      f"{'ECE':>14}  {'EO-gap':>14}")
print(f"{'─'*74}")
cen_stats = {m: client_stats(cen_df, m) for m in metrics}
for c in clients:
    print(f"  {c}       "
          f"  {ms(*cen_stats['auroc'][c]):>16}"
          f"  {ms(*cen_stats['auprc'][c]):>16}"
          f"  {ms(*cen_stats['ece'][c]):>14}"
          f"  {ms(*cen_stats['eo_gap'][c]):>14}")
print(f"{'─'*74}")
cen_pool_a = pooled_stats(cen_df, 'auroc')
cen_pool_e = pooled_stats(cen_df, 'eo_gap')
print(f"  {'Pooled':<6}  {ms(*cen_pool_a):>16}"
      f"{'':>18}{ms(*cen_pool_e):>32}")
print(f"{'─'*74}")

# ── TABLE 2: Side-by-side comparison ─────────────────────────
print(f"\n\nDAY 4 COMPARISON — Local-only vs Centralized")
print(f"{'─'*80}")
print(f"  {'':6}  {'── AUROC ──':^28}  {'── EO-gap ──':^28}")
print(f"  {'Client':<6}  {'Local-only':>14}  {'Centralized':>14}"
      f"  {'Local-only':>14}  {'Centralized':>14}")
print(f"{'─'*80}")

loc_stats_a = client_stats(local_df, 'auroc')
loc_stats_e = client_stats(local_df, 'eo_gap')

for c in clients:
    la, ls = loc_stats_a[c]
    ca, cs = cen_stats['auroc'][c]
    le, lse= loc_stats_e[c]
    ce, cse= cen_stats['eo_gap'][c]
    # arrow shows direction of change
    auroc_dir = "↑" if ca > la else "↓"
    eo_dir    = "↓" if ce < le else "↑"
    print(f"  {c}       "
          f"  {ms(la,ls):>14}"
          f"  {ms(ca,cs):>14}{auroc_dir}"
          f"  {ms(le,lse):>14}"
          f"  {ms(ce,cse):>14}{eo_dir}")

print(f"{'─'*80}")
# pooled
loc_pa = pooled_stats(local_df,'auroc'); loc_pe = pooled_stats(local_df,'eo_gap')
cen_pa = pooled_stats(cen_df,'auroc');   cen_pe = pooled_stats(cen_df,'eo_gap')
print(f"  {'Pooled':<6}  {ms(*loc_pa):>14}  {ms(*cen_pa):>14}"
      f"  {ms(*loc_pe):>14}  {ms(*cen_pe):>14}")
print(f"{'─'*80}")

# ── TABLE 3: Client C sex disparity ──────────────────────────
print(f"\n\nCLIENT C — Sex disparity (C2 protocol reveal)")
print(f"{'─'*60}")
print(f"  {'Metric':<16}  {'Local-only':>16}  {'Centralized':>16}")
print(f"{'─'*60}")

for col, label in [('male_tpr','Male TPR'),
                   ('female_tpr','Female TPR'),
                   ('eo_gap','EO-gap')]:
    lc = local_df[local_df['client']=='C'][col]
    cc = cen_df[cen_df['client']=='C'][col]
    print(f"  {label:<16}  "
          f"{ms(lc.mean(),lc.std()):>16}  "
          f"{ms(cc.mean(),cc.std()):>16}")

print(f"{'─'*60}")
print(f"  Pooled EO-gap:   "
      f"{ms(*loc_pe):>16}  {ms(*cen_pe):>16}")
print(f"\n  Key finding: pooled EO-gap hides Client C disparity")
print(f"  Pooled EO-gap local  = {loc_pe[0]:.4f} (looks acceptable)")
print(f"  Client C EO-gap local = "
      f"{local_df[local_df['client']=='C']['eo_gap'].mean():.4f}"
      f" (hidden failure — {local_df[local_df['client']==' C']['eo_gap'].mean()/loc_pe[0]:.1f}× larger)"
      if loc_pe[0] > 0 else "")

# ── VISUALIZATION: EO-gap per client bar chart ────────────────
print(f"\n\nGenerating EO-gap comparison figure...")

fig, ax = plt.subplots(figsize=(9, 5))

x        = np.arange(len(clients))
width    = 0.35
colors   = ['#2E86C1', '#117A65']

# local bars
loc_means = [loc_stats_e[c][0] for c in clients]
loc_stds  = [loc_stats_e[c][1] for c in clients]
# centralized bars
cen_means = [cen_stats['eo_gap'][c][0] for c in clients]
cen_stds  = [cen_stats['eo_gap'][c][1] for c in clients]

bars1 = ax.bar(x - width/2, loc_means, width,
               yerr=loc_stds, capsize=4,
               color=colors[0], alpha=0.85,
               label='Local-only', error_kw={'linewidth':1.2})
bars2 = ax.bar(x + width/2, cen_means, width,
               yerr=cen_stds, capsize=4,
               color=colors[1], alpha=0.85,
               label='Centralized', error_kw={'linewidth':1.2})

# Add value labels on bars
for bar, mean in zip(bars1, loc_means):
    ax.text(bar.get_x() + bar.get_width()/2,
            bar.get_height() + 0.005,
            f'{mean:.3f}', ha='center', va='bottom',
            fontsize=9, fontweight='bold', color=colors[0])
for bar, mean in zip(bars2, cen_means):
    ax.text(bar.get_x() + bar.get_width()/2,
            bar.get_height() + 0.005,
            f'{mean:.3f}', ha='center', va='bottom',
            fontsize=9, fontweight='bold', color=colors[1])

# Annotate Client C
ax.annotate('Largest disparity\n(female-skewed site)',
            xy=(2 - width/2, loc_means[2]),
            xytext=(2.5, loc_means[2] + 0.04),
            fontsize=9, color='#922B21',
            arrowprops=dict(arrowstyle='->', color='#922B21',
                           lw=1.5))

ax.set_xticks(x)
ax.set_xticklabels([f'Client {c}' for c in clients], fontsize=11)
ax.set_ylabel('Equal Opportunity Gap (↓ better)', fontsize=11)
ax.set_title('EO-gap per Client — Local-only vs Centralized\n'
             '(mean ± std, 3 seeds)',
             fontsize=12, fontweight='bold')
ax.legend(fontsize=10, loc='upper right')
ax.set_ylim(0, max(max(loc_means), max(cen_means)) + 0.08)
ax.axhline(y=loc_pe[0], color=colors[0], linestyle='--',
           alpha=0.5, linewidth=1,
           label=f'Local pooled avg ({loc_pe[0]:.3f})')
ax.axhline(y=cen_pe[0], color=colors[1], linestyle='--',
           alpha=0.5, linewidth=1,
           label=f'Centralized pooled avg ({cen_pe[0]:.3f})')
ax.legend(fontsize=9, loc='upper right')
ax.grid(axis='y', alpha=0.3)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

plt.tight_layout()
fig_path = f'{FIGURES}/day4_eo_gap_comparison.pdf'
plt.savefig(fig_path, dpi=300, bbox_inches='tight')
plt.show()
print(f"✓ Figure saved → {fig_path}")

# ── FINAL SUMMARY ─────────────────────────────────────────────
print(f"\n\n{'='*70}")
print("DAY 4 COMPLETE ✓")
print(f"{'─'*70}")
print("Files produced:")
print(f"  {RESULTS}/local_only_all.csv      (12 rows)")
print(f"  {RESULTS}/centralized_all.csv     (12 rows)")
print(f"  {CKPTS}/local_[A-D]_seed*.pt      (12 checkpoints)")
print(f"  {CKPTS}/centralized_seed*.pt      (3 checkpoints)")
print(f"  {FIGURES}/day4_eo_gap_comparison.pdf")
print(f"{'─'*70}")
print("Key findings:")
print(f"  1. Centralized AUROC vs local-only → see comparison table")
print(f"  2. Client C EO-gap is highest across both methods")
print(f"  3. Pooled EO-gap ({loc_pe[0]:.4f}) hides Client C gap "
      f"({local_df[local_df['client']=='C']['eo_gap'].mean():.4f})")
print(f"     This is the C2 contribution in one number")
print(f"{'─'*70}")
print("Next: Day 5-6 → FedAvg + FedProx (3 seeds, 30 rounds)")
print(f"{'='*70}")

CENTRALIZED — Test results (mean ± std, 3 seeds)

──────────────────────────────────────────────────────────────────────────
  Client             AUROC             AUPRC             ECE          EO-gap
──────────────────────────────────────────────────────────────────────────
  A            0.9502±0.0021     0.9177±0.0058   0.0337±0.0142   0.0097±0.0125
  B            0.9502±0.0044     0.9330±0.0065   0.0300±0.0132   0.0077±0.0061
  C            0.9354±0.0008     0.8684±0.0095   0.0543±0.0198   0.0417±0.0195
  D            0.9628±0.0018     0.9340±0.0033   0.0303±0.0185   0.0473±0.0136
──────────────────────────────────────────────────────────────────────────
  Pooled     0.9496±0.0022                                     0.0266±0.0099
──────────────────────────────────────────────────────────────────────────


DAY 4 COMPARISON — Local-only vs Centralized
────────────────────────────────────────────────────────────────────────────────
                  ── AUROC ──                   ── E

In [ ]:
# feavg baseline day 5,1
from google.colab import drive
drive.mount('/content/drive')

import os
stat    = os.statvfs('/content')
free_gb = (stat.f_bavail * stat.f_frsize) / 1e9
print(f"Free disk: {free_gb:.1f} GB")
if free_gb < 12:
    raise SystemExit("❌ Not enough disk space. Restart runtime.")

if not os.path.exists('/content/chexpert/train'):
    print("Unzipping to local SSD (3–5 min)...")
    os.makedirs('/content/chexpert', exist_ok=True)
    !unzip -q /content/drive/MyDrive/FairFedCXR/data/chexpert.zip \
           -d /content/chexpert/
    print("Unzip done ✓")
else:
    print("Images already on local SSD ✓")

!pip install -q torch torchvision scikit-learn pandas tqdm scipy matplotlib

IMG_ROOT     = '/content/chexpert'
CLIENTS      = '/content/drive/MyDrive/FairFedCXR/clients'
CKPTS        = '/content/drive/MyDrive/FairFedCXR/checkpoints'
RESULTS      = '/content/drive/MyDrive/FairFedCXR/results'
FIGURES      = '/content/drive/MyDrive/FairFedCXR/figures'
SEEDS        = [42, 123, 456, 789, 1010]
ROUNDS       = 30      # federated communication rounds
LOCAL_EPOCHS = 2       # local epochs per round
BATCH_SIZE   = 32
LR           = 1e-4
WEIGHT_DECAY = 1e-4
EPS          = 1.0
ECE_BINS     = 8
CLIENT_LIST  = ['A', 'B', 'C', 'D']

os.makedirs(CKPTS,   exist_ok=True)
os.makedirs(RESULTS, exist_ok=True)
os.makedirs(FIGURES, exist_ok=True)

print(f"\n✓ Ready")
print(f"  SEEDS={SEEDS} | ROUNDS={ROUNDS} | LOCAL_EPOCHS={LOCAL_EPOCHS}")
print(f"  BATCH={BATCH_SIZE} | LR={LR} | EPS={EPS} | ECE_BINS={ECE_BINS}")

Mounted at /content/drive
Free disk: 70.3 GB
Unzipping to local SSD (3–5 min)...
Unzip done ✓

✓ Ready
  SEEDS=[42, 123, 456, 789, 1010] | ROUNDS=30 | LOCAL_EPOCHS=2
  BATCH=32 | LR=0.0001 | EPS=1.0 | ECE_BINS=8


In [ ]:
import pandas as pd, os
r = f'{RESULTS}/fedavg_all.csv'
print(os.path.exists(r))
d = pd.read_csv(r)
print(len(d), sorted(d.seed.unique()))   # expect 12, [42, 123, 456]
print([os.path.exists(f'{CKPTS}/fedavg_seed{s}.pt') for s in [42,123,456]])

True
12 [np.int64(42), np.int64(123), np.int64(456)]
[True, True, True]


In [ ]:
# feavg baseline day 5,2
import torch, numpy as np, random, gc, time, os, copy
import torchvision.models as models
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from PIL import Image
import pandas as pd
from tqdm import tqdm
from sklearn.metrics import roc_auc_score, average_precision_score
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

torch.backends.cudnn.benchmark = True

def set_seed(seed):
    torch.manual_seed(seed); np.random.seed(seed); random.seed(seed)
    torch.cuda.manual_seed_all(seed)
    print(f"  Seed locked: {seed}")

train_tf = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(0.5),
    transforms.ToTensor(),
    transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225]),
])
eval_tf = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225]),
])

class CheXpertDataset(Dataset):
    def __init__(self, csv_path, transform):
        self.df = pd.read_csv(csv_path)
        self.tf = transform
        assert self.df['sex_encoded'].isin([0,1]).all()
        assert (self.df.loc[self.df['Sex']=='Male','sex_encoded']==1).all()
        assert (self.df.loc[self.df['Sex']=='Female','sex_encoded']==0).all()
    def __len__(self):
        return len(self.df)
    def __getitem__(self, idx):
        row   = self.df.iloc[idx]
        clean = row['Path'].replace('CheXpert-v1.0-small/', '')
        img   = Image.open(os.path.join(IMG_ROOT, clean)).convert('RGB')
        img   = self.tf(img)
        label = torch.tensor(row['label'],       dtype=torch.float32)
        sex   = torch.tensor(row['sex_encoded'], dtype=torch.long)
        return img, label, sex

def make_loader(csv_path, train=False):
    on_gpu = torch.cuda.is_available()
    ds = CheXpertDataset(csv_path, train_tf if train else eval_tf)
    return DataLoader(ds, batch_size=BATCH_SIZE, shuffle=train,
                      num_workers=4 if on_gpu else 2,
                      pin_memory=on_gpu, persistent_workers=True,
                      prefetch_factor=2)

def build_model(device):
    m = models.densenet121(weights='IMAGENET1K_V1')
    m.classifier = torch.nn.Linear(m.classifier.in_features, 1)
    return m.to(device)

# ── Metrics ───────────────────────────────────────────────────
def compute_ece(probs, labels, M=ECE_BINS):
    bins = np.linspace(0,1,M+1); ece=0.0
    for lo,hi in zip(bins[:-1],bins[1:]):
        mask=(probs>=lo)&(probs<hi)
        if mask.sum()==0: continue
        ece += mask.mean()*abs(labels[mask].mean()-probs[mask].mean())
    return float(ece)

def compute_eo_gap(probs, labels, sex, threshold=0.5):
    preds=(probs>=threshold).astype(int)
    def stpr(mask):
        pos=(labels[mask]==1); tp=int(((preds[mask]==1)&pos).sum()); p=int(pos.sum())
        return (tp+EPS)/(p+2*EPS)
    m,f = stpr(sex==1), stpr(sex==0)
    return abs(m-f), m, f

@torch.no_grad()
def evaluate(model, loader, device, threshold=0.5):
    model.eval(); P,L,S=[],[],[]
    for imgs,labels,sex in loader:
        imgs=imgs.to(device,non_blocking=True)
        with torch.amp.autocast('cuda'):
            logits=model(imgs).squeeze()
        P.append(torch.sigmoid(logits.float()).cpu().numpy())
        L.append(labels.numpy()); S.append(sex.numpy())
    probs,labels,sex=map(np.concatenate,(P,L,S))
    eo,mt,ft=compute_eo_gap(probs,labels,sex,threshold)
    return {'auroc':float(roc_auc_score(labels,probs)),
            'auprc':float(average_precision_score(labels,probs)),
            'ece':float(compute_ece(probs,labels)),'eo_gap':float(eo),
            'male_tpr':float(mt),'female_tpr':float(ft),
            'm_pos':int((labels[sex==1]==1).sum()),
            'f_pos':int((labels[sex==0]==1).sum())}

print("✓ Imports, Dataset, Metrics defined")

✓ Imports, Dataset, Metrics defined


In [ ]:
# feavg baseline day 5,3
from torch.amp import autocast, GradScaler

def local_train(global_state, train_loader, device, local_epochs=LOCAL_EPOCHS):
    """
    Train a local copy of the global model for local_epochs.
    Returns the updated state_dict (on CPU to save GPU memory).
    """
    model = build_model(device)
    model.load_state_dict(global_state)
    model.train()
    optimizer = torch.optim.AdamW(model.parameters(),
                                   lr=LR, weight_decay=WEIGHT_DECAY)
    criterion = torch.nn.BCEWithLogitsLoss()
    scaler    = GradScaler('cuda')

    for ep in range(local_epochs):
        for imgs, labels, _ in train_loader:
            imgs   = imgs.to(device,   non_blocking=True)
            labels = labels.to(device, non_blocking=True)
            optimizer.zero_grad()
            with autocast('cuda'):
                logits = model(imgs).squeeze()
                loss   = criterion(logits, labels)
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()

    # extract to CPU, free GPU model
    state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
    del model, optimizer
    gc.collect()
    torch.cuda.empty_cache()
    return state

def fedavg_aggregate(client_states, client_weights):
    """
    Weighted average of client state_dicts.
    client_weights: list of a_i = n_i / N (sum to 1).
    Returns aggregated global state_dict.
    """
    global_state = copy.deepcopy(client_states[0])
    for key in global_state:
        # weighted sum across clients
        stacked = torch.stack([
            client_states[i][key].float() * client_weights[i]
            for i in range(len(client_states))
        ], dim=0)
        agg = stacked.sum(dim=0)
        # preserve original dtype (handles int buffers like num_batches_tracked)
        global_state[key] = agg.to(client_states[0][key].dtype)
    return global_state

print("✓ Federated engine defined")
print("  local_train: fp16, returns CPU state_dict")
print("  fedavg_aggregate: sample-size weighted average")

✓ Federated engine defined
  local_train: fp16, returns CPU state_dict
  fedavg_aggregate: sample-size weighted average


In [ ]:
# feavg baseline day 5,4
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"GPU : {torch.cuda.get_device_name(0) if device=='cuda' else 'CPU'}")

# Build client train loaders + sample sizes
print("\nBuilding client loaders...")
client_train_loaders = {
    c: make_loader(f'{CLIENTS}/client_{c}_train.csv', train=True)
    for c in CLIENT_LIST}
client_sizes = {c: len(client_train_loaders[c].dataset) for c in CLIENT_LIST}
N = sum(client_sizes.values())
client_weights = [client_sizes[c]/N for c in CLIENT_LIST]
print(f"Client sizes: {client_sizes}")
print(f"FedAvg weights: "
      f"{[f'{c}={w:.3f}' for c,w in zip(CLIENT_LIST, client_weights)]}")

# Pooled val loader for global eval
pooled_val = pd.concat([pd.read_csv(f'{CLIENTS}/client_{c}_val.csv')
                        for c in CLIENT_LIST], ignore_index=True)
pooled_val.to_csv('/content/pooled_val.csv', index=False)
pooled_val_loader = make_loader('/content/pooled_val.csv', train=False)

# ── DEBUG: 2 rounds ───────────────────────────────────────────
print("\n── DEBUG: 2 rounds, seed 42 ──")
set_seed(42)
global_model = build_model(device)
global_state = {k:v.cpu().clone() for k,v in global_model.state_dict().items()}
del global_model; torch.cuda.empty_cache()

for rnd in range(2):
    client_states = []
    for c in CLIENT_LIST:
        st = local_train(global_state, client_train_loaders[c], device)
        client_states.append(st)
    global_state = fedavg_aggregate(client_states, client_weights)
    # eval
    gm = build_model(device); gm.load_state_dict(global_state)
    m = evaluate(gm, pooled_val_loader, device)
    print(f"  Round {rnd+1}: pooled val AUROC={m['auroc']:.4f} "
          f"EO={m['eo_gap']:.4f}")
    del gm; gc.collect(); torch.cuda.empty_cache()

print("\n✓ Debug complete — aggregation works, AUROC > 0.5")
print("✓ Switch to L4 for full 30-round run")
del client_train_loaders
gc.collect(); torch.cuda.empty_cache()

GPU : NVIDIA L4

Building client loaders...
Client sizes: {'A': 10500, 'B': 5600, 'C': 4829, 'D': 4200}
FedAvg weights: ['A=0.418', 'B=0.223', 'C=0.192', 'D=0.167']

── DEBUG: 2 rounds, seed 42 ──
  Seed locked: 42
  Round 1: pooled val AUROC=0.9491 EO=0.0278
  Round 2: pooled val AUROC=0.9534 EO=0.0424

✓ Debug complete — aggregation works, AUROC > 0.5
✓ Switch to L4 for full 30-round run


In [ ]:
# feavg baseline day 5,5
device = 'cuda'
print(f"GPU : {torch.cuda.get_device_name(0)}")
print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB")

# ── SAFE loader — no deadlock ─────────────────────────────────
def make_loader_safe(csv_path, train=False):
    ds = CheXpertDataset(csv_path, train_tf if train else eval_tf)
    return DataLoader(ds, batch_size=BATCH_SIZE, shuffle=train,
                      num_workers=2, pin_memory=True,
                      persistent_workers=False)

# ── local train WITH progress bar ─────────────────────────────
def local_train_v(global_state, loader, device, tag=""):
    model = build_model(device)
    model.load_state_dict(global_state)
    model.train()
    opt    = torch.optim.AdamW(model.parameters(),
                                lr=LR, weight_decay=WEIGHT_DECAY)
    crit   = torch.nn.BCEWithLogitsLoss()
    scaler = GradScaler('cuda')
    for ep in range(LOCAL_EPOCHS):
        for imgs, labels, _ in tqdm(loader,
                                     desc=f"{tag} ep{ep+1}",
                                     leave=False):
            imgs   = imgs.to(device, non_blocking=True)
            labels = labels.to(device, non_blocking=True)
            opt.zero_grad()
            with autocast('cuda'):
                loss = crit(model(imgs).squeeze(), labels)
            scaler.scale(loss).backward()
            scaler.step(opt); scaler.update()
    state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
    del model, opt; gc.collect(); torch.cuda.empty_cache()
    return state

# Build pooled val once
pooled_val = pd.concat([pd.read_csv(f'{CLIENTS}/client_{c}_val.csv')
                        for c in CLIENT_LIST], ignore_index=True)
pooled_val.to_csv('/content/pooled_val.csv', index=False)

fedavg_results   = []
fedavg_round_log = []
t_start = time.time()

for seed in SEEDS:
    print(f"\n{'='*58}")
    print(f"  FEDAVG | SEED {seed}")
    print(f"{'='*58}")

    ckpt_path = f"{CKPTS}/fedavg_seed{seed}.pt"

    # Crash recovery
    if os.path.exists(ckpt_path):
        print(f"  ⚡ fedavg_seed{seed} already done — loading for test")
        global_state = torch.load(ckpt_path, map_location='cpu')
    else:
        set_seed(seed)

        # Build loaders once for this seed
        client_train_loaders = {
            c: make_loader_safe(f'{CLIENTS}/client_{c}_train.csv', train=True)
            for c in CLIENT_LIST}
        client_sizes   = {c: len(client_train_loaders[c].dataset)
                          for c in CLIENT_LIST}
        N              = sum(client_sizes.values())
        client_weights = [client_sizes[c]/N for c in CLIENT_LIST]
        pooled_val_loader = make_loader_safe('/content/pooled_val.csv',
                                             train=False)
        print(f"  Weights: "
              f"{[f'{c}={w:.3f}' for c,w in zip(CLIENT_LIST, client_weights)]}")

        gm = build_model(device)
        global_state = {k:v.cpu().clone()
                        for k,v in gm.state_dict().items()}
        del gm; torch.cuda.empty_cache()

        best_auroc        = -1.0
        best_global_state = None

        for rnd in range(ROUNDS):
            # local training each client (progress bar visible)
            client_states = []
            for c in CLIENT_LIST:
                st = local_train_v(global_state,
                                   client_train_loaders[c], device,
                                   tag=f"R{rnd+1}-{c}")
                client_states.append(st)

            # aggregate
            global_state = fedavg_aggregate(client_states, client_weights)
            del client_states; gc.collect()

            # eval global on pooled val
            gm = build_model(device)
            gm.load_state_dict(global_state)
            vm = evaluate(gm, pooled_val_loader, device)
            del gm; gc.collect(); torch.cuda.empty_cache()

            fedavg_round_log.append({
                'method':'fedavg','seed':seed,'round':rnd+1,
                'val_auroc':vm['auroc'],'val_eo_gap':vm['eo_gap'],
                'val_ece':vm['ece']})

            print(f"  Round {rnd+1:02d}/{ROUNDS}: "
                  f"val AUROC={vm['auroc']:.4f} | EO={vm['eo_gap']:.4f}")

            if vm['auroc'] > best_auroc:
                best_auroc        = vm['auroc']
                best_global_state = copy.deepcopy(global_state)

            pd.DataFrame(fedavg_round_log).to_csv(
                f'{RESULTS}/fedavg_round_log.csv', index=False)

        global_state = best_global_state
        torch.save(global_state, ckpt_path)
        print(f"  ✓ Best global saved (val AUROC={best_auroc:.4f})")

        del client_train_loaders, pooled_val_loader
        gc.collect(); torch.cuda.empty_cache()

    # ── TEST on each client ───────────────────────────────────
    gm = build_model(device)
    gm.load_state_dict(global_state)
    for c in CLIENT_LIST:
        te = make_loader_safe(f'{CLIENTS}/client_{c}_test.csv', train=False)
        tm = evaluate(gm, te, device)
        tm.update({'method':'fedavg','client':c,'seed':seed})
        fedavg_results.append(tm)
        print(f"  → Client {c} TEST: "
              f"AUROC={tm['auroc']:.4f} | EO={tm['eo_gap']:.4f}")
        del te; gc.collect()
    del gm; gc.collect(); torch.cuda.empty_cache()

    pd.DataFrame(fedavg_results).to_csv(
        f'{RESULTS}/fedavg_all.csv', index=False)
    print(f"  ✓ Test results saved after seed {seed}")

elapsed = (time.time()-t_start)/3600
print(f"\n{'='*58}")
print(f"FEDAVG COMPLETE in {elapsed:.2f} hr")
print(f"✓ Test → {RESULTS}/fedavg_all.csv")
print(f"✓ Log  → {RESULTS}/fedavg_round_log.csv")
print(f"✓ Ckpts → {CKPTS}/fedavg_seed*.pt")
print(f"  Test rows: {len(fedavg_results)} (expect 12)")

GPU : NVIDIA L4
VRAM: 23.7 GB

  FEDAVG | SEED 42
  Seed locked: 42
  Weights: ['A=0.418', 'B=0.223', 'C=0.192', 'D=0.167']


  Round 01/30: val AUROC=0.9476 | EO=0.0408


  Round 02/30: val AUROC=0.9541 | EO=0.0332


  Round 03/30: val AUROC=0.9538 | EO=0.0409


  Round 04/30: val AUROC=0.9530 | EO=0.0407


...

  Round 26/30: val AUROC=0.9495 | EO=0.0243


  Round 27/30: val AUROC=0.9481 | EO=0.0345


  Round 28/30: val AUROC=0.9481 | EO=0.0258


  Round 29/30: val AUROC=0.9474 | EO=0.0263


  Round 30/30: val AUROC=0.9480 | EO=0.0180
  ✓ Best global saved (val AUROC=0.9529)
  → Client A TEST: AUROC=0.9519 | EO=0.0199
  → Client B TEST: AUROC=0.9502 | EO=0.0050
  → Client C TEST: AUROC=0.9398 | EO=0.0004
  → Client D TEST: AUROC=0.9641 | EO=0.0365
  ✓ Test results saved after seed 456

FEDAVG COMPLETE in 5.02 hr
✓ Test → /content/drive/MyDrive/FairFedCXR/results/fedavg_all.csv
✓ Log  → /content/drive/MyDrive/FairFedCXR/results/fedavg_round_log.csv
✓ Ckpts → /content/drive/MyDrive/FairFedCXR/checkpoints/fedavg_seed*.pt
  Test rows: 12 (expect 12)


In [ ]:
# feavg baseline day 5,6
import pandas as pd, numpy as np
import matplotlib.pyplot as plt

clients = ['A','B','C','D']
metrics = ['auroc','auprc','ece','eo_gap']

def cstat(df, m):
    return {c:(df[df['client']==c][m].mean(),
               df[df['client']==c][m].std()) for c in clients}
def pstat(df, m):
    p=df.groupby('seed')[m].mean(); return p.mean(), p.std()
def ms(a,s): return f"{a:.4f}±{s:.4f}"

# Load all three methods
fed   = pd.read_csv(f'{RESULTS}/fedavg_all.csv')
local = pd.read_csv(f'{RESULTS}/local_only_all.csv')
cen   = pd.read_csv(f'{RESULTS}/centralized_all.csv')

assert len(fed)==12, f"Expected 12 fedavg rows, got {len(fed)}"

# ── TABLE: FedAvg per-client ──────────────────────────────────
print("FEDAVG — Test results (mean ± std, 3 seeds)")
print(f"\n{'─'*74}")
print(f"  {'Client':<6} {'AUROC':>16} {'AUPRC':>16} {'ECE':>14} {'EO-gap':>14}")
print(f"{'─'*74}")
fa = {m:cstat(fed,m) for m in metrics}
for c in clients:
    print(f"  {c}     "
          f"  {ms(*fa['auroc'][c]):>16}"
          f"  {ms(*fa['auprc'][c]):>16}"
          f"  {ms(*fa['ece'][c]):>14}"
          f"  {ms(*fa['eo_gap'][c]):>14}")
print(f"{'─'*74}")
print(f"  {'Pooled':<6}  {ms(*pstat(fed,'auroc')):>16}"
      f"{'':>18}{ms(*pstat(fed,'eo_gap')):>30}")

# ── TABLE: 3-method comparison (AUROC + EO-gap) ──────────────
print(f"\n\n3-METHOD COMPARISON")
print(f"{'─'*88}")
print(f"  {'':6} {'──────── AUROC ────────':^40} {'──────── EO-gap ────────':^40}")
print(f"  {'Clnt':<5} {'Local':>12} {'Central':>12} {'FedAvg':>12}"
      f"  {'Local':>12} {'Central':>12} {'FedAvg':>12}")
print(f"{'─'*88}")
for c in clients:
    la=local[local['client']==c]['auroc']; ca=cen[cen['client']==c]['auroc']; faa=fed[fed['client']==c]['auroc']
    le=local[local['client']==c]['eo_gap']; ce=cen[cen['client']==c]['eo_gap']; fee=fed[fed['client']==c]['eo_gap']
    print(f"  {c:<5} {la.mean():>12.4f} {ca.mean():>12.4f} {faa.mean():>12.4f}"
          f"  {le.mean():>12.4f} {ce.mean():>12.4f} {fee.mean():>12.4f}")
print(f"{'─'*88}")
lp=local.groupby('seed')['auroc'].mean(); cp=cen.groupby('seed')['auroc'].mean(); fp=fed.groupby('seed')['auroc'].mean()
lpe=local.groupby('seed')['eo_gap'].mean(); cpe=cen.groupby('seed')['eo_gap'].mean(); fpe=fed.groupby('seed')['eo_gap'].mean()
print(f"  {'Pool':<5} {lp.mean():>12.4f} {cp.mean():>12.4f} {fp.mean():>12.4f}"
      f"  {lpe.mean():>12.4f} {cpe.mean():>12.4f} {fpe.mean():>12.4f}")
print(f"{'─'*88}")

# ── Client C focus ────────────────────────────────────────────
print(f"\n\nCLIENT C — EO-gap across methods (the key site)")
print(f"  Local-only : {local[local['client']=='C']['eo_gap'].mean():.4f}")
print(f"  Centralized: {cen[cen['client']=='C']['eo_gap'].mean():.4f}")
print(f"  FedAvg     : {fed[fed['client']=='C']['eo_gap'].mean():.4f}")
print(f"  → FedAvg gives Client C only 19.2% aggregation weight")
print(f"  → DWFA will address this (Day 9-12)")

# ── FIGURE: Convergence curves ────────────────────────────────
print(f"\nGenerating convergence figure...")
rl = pd.read_csv(f'{RESULTS}/fedavg_round_log.csv')

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5))

# AUROC convergence — mean ± std band across seeds
auroc_pivot = rl.pivot_table(index='round', columns='seed',
                              values='val_auroc')
eo_pivot    = rl.pivot_table(index='round', columns='seed',
                             values='val_eo_gap')
rounds = auroc_pivot.index

a_mean = auroc_pivot.mean(axis=1); a_std = auroc_pivot.std(axis=1)
e_mean = eo_pivot.mean(axis=1);    e_std = eo_pivot.std(axis=1)

ax1.plot(rounds, a_mean, color='#1B4F72', lw=2, label='FedAvg')
ax1.fill_between(rounds, a_mean-a_std, a_mean+a_std,
                 color='#1B4F72', alpha=0.2)
ax1.set_xlabel('Communication Round', fontsize=11)
ax1.set_ylabel('Pooled Validation AUROC', fontsize=11)
ax1.set_title('FedAvg Convergence — AUROC', fontsize=12, fontweight='bold')
ax1.grid(alpha=0.3); ax1.legend(fontsize=10)
ax1.spines['top'].set_visible(False); ax1.spines['right'].set_visible(False)

ax2.plot(rounds, e_mean, color='#922B21', lw=2, label='FedAvg')
ax2.fill_between(rounds, e_mean-e_std, e_mean+e_std,
                 color='#922B21', alpha=0.2)
ax2.set_xlabel('Communication Round', fontsize=11)
ax2.set_ylabel('Pooled Validation EO-gap', fontsize=11)
ax2.set_title('FedAvg Convergence — EO-gap', fontsize=12, fontweight='bold')
ax2.grid(alpha=0.3); ax2.legend(fontsize=10)
ax2.spines['top'].set_visible(False); ax2.spines['right'].set_visible(False)

plt.tight_layout()
fig_path = f'{FIGURES}/fedavg_convergence.pdf'
plt.savefig(fig_path, dpi=300, bbox_inches='tight')
plt.show()
print(f"✓ Figure saved → {fig_path}")

print(f"\n{'='*70}")
print("DAY 5 FEDAVG COMPLETE ✓")
print(f"{'─'*70}")
print("Files: fedavg_all.csv, fedavg_round_log.csv, fedavg_convergence.pdf")
print("       3 checkpoints fedavg_seed*.pt")
print(f"{'─'*70}")
print("Next: Day 6 → FedProx (μ=0.01, same 30 rounds × 3 seeds)")
print(f"{'='*70}")

FEDAVG — Test results (mean ± std, 3 seeds)

──────────────────────────────────────────────────────────────────────────
  Client            AUROC            AUPRC            ECE         EO-gap
──────────────────────────────────────────────────────────────────────────
  A          0.9520±0.0011     0.9147±0.0017   0.0434±0.0100   0.0138±0.0076
  B          0.9477±0.0039     0.9247±0.0121   0.0508±0.0094   0.0065±0.0015
  C          0.9379±0.0025     0.8670±0.0043   0.0573±0.0161   0.0160±0.0152
  D          0.9643±0.0003     0.9387±0.0008   0.0473±0.0093   0.0398±0.0100
──────────────────────────────────────────────────────────────────────────
  Pooled     0.9505±0.0017                                   0.0190±0.0032


3-METHOD COMPARISON
────────────────────────────────────────────────────────────────────────────────────────
                 ──────── AUROC ────────                  ──────── EO-gap ────────        
  Clnt         Local      Central       FedAvg         Local      Centra

In [ ]:
# ============================================================================
# FEDAVG SEED EXTENSION — SEEDS 789 AND 1010, PER-ROUND CRASH RECOVERY
#
# PURPOSE
#   Bring FedAvg to the same five common seeds as LPR, CADR, and DWFA so it can
#   enter the primary inferential family. Protocol is byte-identical to the
#   original three-seed run: same client CSVs, DenseNet-121, AdamW 1e-4,
#   weight decay 1e-4, batch 32, two local epochs, 30 rounds, sample-size
#   aggregation, and best-pooled-validation-AUROC checkpoint selection.
#
# RUN THESE OLD CELLS FIRST, IN THIS ORDER, IN THE CURRENT SESSION
#   1. "# feavg baseline day 5,1"   mounts Drive, unzips images, sets IMG_ROOT,
#                                   CLIENTS, CKPTS, RESULTS, ROUNDS,
#                                   LOCAL_EPOCHS, BATCH_SIZE, LR, WEIGHT_DECAY,
#                                   EPS, ECE_BINS, CLIENT_LIST
#   2. "# feavg baseline day 5,2"   imports, set_seed, CheXpertDataset,
#                                   build_model, compute_eo_gap, evaluate
#   3. "# feavg baseline day 5,3"   local_train, fedavg_aggregate
#
# DO NOT RUN
#   "# feavg baseline day 5,4"  debug cell, wastes GPU on two throwaway rounds
#   "# feavg baseline day 5,5"  the original driver. It starts fedavg_results
#                               and fedavg_round_log as EMPTY lists and then
#                               writes fedavg_all.csv and fedavg_round_log.csv,
#                               so running it would erase seeds 42, 123, 456.
#                               This cell appends instead.
#
# CRASH BEHAVIOUR
#   A progress checkpoint is written after EVERY round to
#   {CKPTS}/fedavg_seed{seed}_progress.pt via a temp file plus atomic rename,
#   keeping one .bak generation. A disconnect loses at most the round in
#   flight. Re-run this same cell to resume. Finished seeds are skipped.
#   Python, NumPy, torch, and CUDA RNG states are saved and restored, so a
#   resumed run follows the same stochastic path as an uninterrupted one.
#
# AFTER THIS CELL
#   Training alone does not update Table 5 or Table 6. You still need NIH and
#   CheXpert predictions cached for ('fedavg', 789) and ('fedavg', 1010).
#   Run your existing "#CELL 1: GPU INFERENCE-ONLY REVISION CELL" afterward.
# ============================================================================

import os, gc, copy, time, random
import numpy as np
import pandas as pd
import torch
from tqdm import tqdm
from torch.amp import autocast, GradScaler
from torch.utils.data import DataLoader

NEW_SEEDS = [789, 1010]
EXPECTED_PRIOR_SEEDS = [42, 123, 456]

for _n in ['CLIENTS', 'CKPTS', 'RESULTS', 'IMG_ROOT', 'ROUNDS', 'LOCAL_EPOCHS',
           'BATCH_SIZE', 'LR', 'WEIGHT_DECAY', 'CLIENT_LIST']:
    if _n not in globals():
        raise NameError(f'{_n} is not defined. Run "# feavg baseline day 5,1" first.')
for _f in ['build_model', 'evaluate', 'set_seed', 'CheXpertDataset',
           'fedavg_aggregate', 'train_tf', 'eval_tf']:
    if _f not in globals():
        raise NameError(f'{_f} is not defined. Run "# feavg baseline day 5,2" '
                        f'and "# feavg baseline day 5,3" first.')

device = 'cuda' if torch.cuda.is_available() else 'cpu'
if device != 'cuda':
    raise SystemExit('No GPU visible. Switch runtime to GPU before running.')
print(f'GPU  : {torch.cuda.get_device_name(0)}')
print(f'VRAM : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

RES_CSV = f'{RESULTS}/fedavg_all.csv'
LOG_CSV = f'{RESULTS}/fedavg_round_log.csv'


# ---------------------------------------------------------------------------
# CRASH-SAFE IO
# ---------------------------------------------------------------------------

def atomic_save(obj, path):
    """Write to a temp file then rename, keeping one .bak generation, so a
    disconnect during the write cannot leave a truncated checkpoint."""
    tmp = f'{path}.tmp'
    torch.save(obj, tmp)
    if os.path.exists(path):
        try:
            os.replace(path, f'{path}.bak')
        except Exception:
            pass
    os.replace(tmp, path)


def robust_load(path):
    for p in [path, f'{path}.bak']:
        if os.path.exists(p):
            try:
                return torch.load(p, map_location='cpu', weights_only=False)
            except Exception as e:
                print(f'  Checkpoint unreadable, trying older generation: {p} ({e})')
    return None


def save_csv_atomic(df, path):
    tmp = f'{path}.tmp'
    df.to_csv(tmp, index=False)
    os.replace(tmp, path)


def grab_rng():
    return {'torch': torch.get_rng_state(),
            'cuda': torch.cuda.get_rng_state_all() if torch.cuda.is_available() else None,
            'numpy': np.random.get_state(),
            'python': random.getstate()}


def put_rng(s):
    if not s:
        return
    torch.set_rng_state(s['torch'])
    if s.get('cuda') is not None and torch.cuda.is_available():
        torch.cuda.set_rng_state_all(s['cuda'])
    np.random.set_state(s['numpy'])
    random.setstate(s['python'])


# ---------------------------------------------------------------------------
# LOADERS AND LOCAL TRAINING — identical protocol to the original run
# ---------------------------------------------------------------------------

def make_loader_safe(csv_path, train=False):
    ds = CheXpertDataset(csv_path, train_tf if train else eval_tf)
    return DataLoader(ds, batch_size=BATCH_SIZE, shuffle=train,
                      num_workers=2, pin_memory=True,
                      persistent_workers=False)


def local_train_v(global_state, loader, device, tag=''):
    model = build_model(device)
    model.load_state_dict(global_state)
    model.train()
    opt = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
    crit = torch.nn.BCEWithLogitsLoss()
    scaler = GradScaler('cuda')
    for ep in range(LOCAL_EPOCHS):
        for imgs, labels, _ in tqdm(loader, desc=f'{tag} ep{ep + 1}', leave=False):
            imgs = imgs.to(device, non_blocking=True)
            labels = labels.to(device, non_blocking=True)
            opt.zero_grad()
            with autocast('cuda'):
                loss = crit(model(imgs).squeeze(), labels)
            scaler.scale(loss).backward()
            scaler.step(opt)
            scaler.update()
    state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
    del model, opt
    gc.collect()
    torch.cuda.empty_cache()
    return state


# ---------------------------------------------------------------------------
# LOAD EXISTING RESULTS — APPEND, NEVER OVERWRITE
# ---------------------------------------------------------------------------

results = pd.read_csv(RES_CSV).to_dict('records') if os.path.exists(RES_CSV) else []
round_log = pd.read_csv(LOG_CSV).to_dict('records') if os.path.exists(LOG_CSV) else []

done_seeds = {int(r['seed']) for r in results}
print(f'\nExisting fedavg_all.csv rows : {len(results)}')
print(f'Seeds already complete       : {sorted(done_seeds)}')

if not set(EXPECTED_PRIOR_SEEDS).issubset(done_seeds):
    missing = sorted(set(EXPECTED_PRIOR_SEEDS) - done_seeds)
    print(f'\nWARNING: seeds {missing} are NOT in fedavg_all.csv. If you expected '
          f'them there, stop and check the file before continuing, because this '
          f'cell will only add {NEW_SEEDS}.')

pooled_val = pd.concat([pd.read_csv(f'{CLIENTS}/client_{c}_val.csv')
                        for c in CLIENT_LIST], ignore_index=True)
pooled_val.to_csv('/content/pooled_val.csv', index=False)

t_start = time.time()

for seed in NEW_SEEDS:
    print(f'\n{"=" * 62}')
    print(f'  FEDAVG | SEED {seed}')
    print(f'{"=" * 62}')

    if seed in done_seeds:
        print(f'  Seed {seed} already has test rows in fedavg_all.csv — skipping.')
        continue

    final_ckpt = f'{CKPTS}/fedavg_seed{seed}.pt'
    prog_ckpt = f'{CKPTS}/fedavg_seed{seed}_progress.pt'

    if os.path.exists(final_ckpt):
        print('  Final checkpoint found — loading best global state for test only.')
        global_state = torch.load(final_ckpt, map_location='cpu', weights_only=False)
    else:
        client_train_loaders = {
            c: make_loader_safe(f'{CLIENTS}/client_{c}_train.csv', train=True)
            for c in CLIENT_LIST}
        client_sizes = {c: len(client_train_loaders[c].dataset) for c in CLIENT_LIST}
        N = sum(client_sizes.values())
        client_weights = [client_sizes[c] / N for c in CLIENT_LIST]
        pooled_val_loader = make_loader_safe('/content/pooled_val.csv', train=False)
        print(f'  Client sizes  : {client_sizes}')
        print(f'  FedAvg weights: '
              f'{[f"{c}={w:.3f}" for c, w in zip(CLIENT_LIST, client_weights)]}')

        ck = robust_load(prog_ckpt)
        if ck is not None:
            global_state = ck['global_state']
            best_global_state = ck['best_global_state']
            best_auroc = ck['best_auroc']
            start_round = ck['next_round']
            put_rng(ck.get('rng'))
            round_log = [r for r in round_log
                         if not (int(r.get('seed', -1)) == seed
                                 and int(r.get('round', 0)) > start_round)]
            print(f'  Resuming from round {start_round + 1} of {ROUNDS} '
                  f'(best val AUROC so far {best_auroc:.4f})')
        else:
            set_seed(seed)
            gm = build_model(device)
            global_state = {k: v.cpu().clone() for k, v in gm.state_dict().items()}
            del gm
            torch.cuda.empty_cache()
            best_global_state, best_auroc, start_round = None, -1.0, 0
            round_log = [r for r in round_log if int(r.get('seed', -1)) != seed]
            print(f'  Fresh start at round 1 of {ROUNDS}')

        for rnd in range(start_round, ROUNDS):
            client_states = []
            for c in CLIENT_LIST:
                st = local_train_v(global_state, client_train_loaders[c], device,
                                   tag=f'R{rnd + 1}-{c}')
                client_states.append(st)

            global_state = fedavg_aggregate(client_states, client_weights)
            del client_states
            gc.collect()

            gm = build_model(device)
            gm.load_state_dict(global_state)
            vm = evaluate(gm, pooled_val_loader, device)
            del gm
            gc.collect()
            torch.cuda.empty_cache()

            round_log.append({'method': 'fedavg', 'seed': seed, 'round': rnd + 1,
                              'val_auroc': vm['auroc'], 'val_eo_gap': vm['eo_gap'],
                              'val_ece': vm['ece']})

            if vm['auroc'] > best_auroc:
                best_auroc = vm['auroc']
                best_global_state = copy.deepcopy(global_state)
                marker = '  <- new best'
            else:
                marker = ''

            print(f'  Round {rnd + 1:02d}/{ROUNDS}: val AUROC={vm["auroc"]:.4f} | '
                  f'EO={vm["eo_gap"]:.4f}{marker}')

            save_csv_atomic(
                pd.DataFrame(round_log).drop_duplicates(['method', 'seed', 'round'],
                                                        keep='last'),
                LOG_CSV)
            atomic_save({'global_state': global_state,
                         'best_global_state': best_global_state,
                         'best_auroc': best_auroc,
                         'next_round': rnd + 1,
                         'seed': seed,
                         'rng': grab_rng()}, prog_ckpt)

        global_state = best_global_state
        atomic_save(global_state, final_ckpt)
        print(f'  Best global saved (val AUROC={best_auroc:.4f}) -> {final_ckpt}')

        for p in [prog_ckpt, f'{prog_ckpt}.bak']:
            if os.path.exists(p):
                os.remove(p)

        del client_train_loaders, pooled_val_loader
        gc.collect()
        torch.cuda.empty_cache()

    gm = build_model(device)
    gm.load_state_dict(global_state)
    for c in CLIENT_LIST:
        te = make_loader_safe(f'{CLIENTS}/client_{c}_test.csv', train=False)
        tm = evaluate(gm, te, device)
        tm.update({'method': 'fedavg', 'client': c, 'seed': seed})
        results.append(tm)
        print(f'  Client {c} TEST: AUROC={tm["auroc"]:.4f} | EO={tm["eo_gap"]:.4f}')
        del te
        gc.collect()
    del gm
    gc.collect()
    torch.cuda.empty_cache()

    save_csv_atomic(
        pd.DataFrame(results).drop_duplicates(['method', 'seed', 'client'],
                                              keep='last'),
        RES_CSV)
    done_seeds.add(seed)
    print(f'  Test rows appended for seed {seed}.')

# ---------------------------------------------------------------------------
# VERIFY
# ---------------------------------------------------------------------------

final_res = pd.read_csv(RES_CSV)
final_log = pd.read_csv(LOG_CSV)
seeds_present = sorted(final_res['seed'].unique())

print(f'\n{"=" * 62}')
print(f'Elapsed this session: {(time.time() - t_start) / 3600:.2f} hr')
print(f'fedavg_all.csv       : {len(final_res)} rows, seeds {seeds_present}')
print(f'fedavg_round_log.csv : {len(final_log)} rows, '
      f'seeds {sorted(final_log["seed"].unique())}')

expected = sorted(set(EXPECTED_PRIOR_SEEDS) | set(NEW_SEEDS))
if seeds_present == expected and len(final_res) == 4 * len(expected):
    print(f'COMPLETE: FedAvg now has {len(expected)} seeds and '
          f'{len(final_res)} test rows.')
else:
    print(f'INCOMPLETE: expected seeds {expected} and {4 * len(expected)} rows. '
          f'Re-run this cell to continue.')

print('\nPer-seed pooled test summary:')
print(final_res.groupby('seed')[['auroc', 'eo_gap']].mean().round(4).to_string())
print(f'{"=" * 62}')
print('NEXT: cache NIH and CheXpert predictions for the two new checkpoints by '
      'running "#CELL 1: GPU INFERENCE-ONLY REVISION CELL".')

GPU  : Tesla T4
VRAM : 15.6 GB

Existing fedavg_all.csv rows : 16
Seeds already complete       : [42, 123, 456, 789]

  FEDAVG | SEED 789
  Seed 789 already has test rows in fedavg_all.csv — skipping.

  FEDAVG | SEED 1010
  Client sizes  : {'A': 10500, 'B': 5600, 'C': 4829, 'D': 4200}
  FedAvg weights: ['A=0.418', 'B=0.223', 'C=0.192', 'D=0.167']
  Resuming from round 23 of 30 (best val AUROC so far 0.9544)
Downloading: "https://download.pytorch.org/models/densenet121-a639ec97.pth" to /root/.cache/torch/hub/checkpoints/densenet121-a639ec97.pth


100%|██████████| 30.8M/30.8M [00:00<00:00, 126MB/s]


  Round 23/30: val AUROC=0.9492 | EO=0.0420


  Round 24/30: val AUROC=0.9511 | EO=0.0381


  Round 25/30: val AUROC=0.9492 | EO=0.0332


  Round 26/30: val AUROC=0.9496 | EO=0.0346


  Round 27/30: val AUROC=0.9488 | EO=0.0312


  Round 28/30: val AUROC=0.9499 | EO=0.0422


  Round 29/30: val AUROC=0.9492 | EO=0.0251


  Round 30/30: val AUROC=0.9499 | EO=0.0254
  Best global saved (val AUROC=0.9544) -> /content/drive/MyDrive/FairFedCXR/checkpoints/fedavg_seed1010.pt
  Client A TEST: AUROC=0.9513 | EO=0.0285
  Client B TEST: AUROC=0.9484 | EO=0.0199
  Client C TEST: AUROC=0.9377 | EO=0.0374
  Client D TEST: AUROC=0.9641 | EO=0.0336
  Test rows appended for seed 1010.

Elapsed this session: 0.81 hr
fedavg_all.csv       : 20 rows, seeds [np.int64(42), np.int64(123), np.int64(456), np.int64(789), np.int64(1010)]
fedavg_round_log.csv : 150 rows, seeds [np.int64(42), np.int64(123), np.int64(456), np.int64(789), np.int64(1010)]
COMPLETE: FedAvg now has 5 seeds and 20 test rows.

Per-seed pooled test summary:
       auroc  eo_gap
seed                
42    0.9514  0.0217
123   0.9485  0.0199
456   0.9515  0.0155
789   0.9486  0.0142
1010  0.9504  0.0299
NEXT: cache NIH and CheXpert predictions for the two new checkpoints by running "#CELL 1: GPU INFERENCE-ONLY REVISION CELL".


In [ ]:
# baseline fadeprox day 6,1
from google.colab import drive
drive.mount('/content/drive')

import os
stat    = os.statvfs('/content')
free_gb = (stat.f_bavail * stat.f_frsize) / 1e9
print(f"Free disk: {free_gb:.1f} GB")
if free_gb < 12:
    raise SystemExit("❌ Not enough disk space. Restart runtime.")

if not os.path.exists('/content/chexpert/train'):
    print("Unzipping to local SSD (3–5 min)...")
    os.makedirs('/content/chexpert', exist_ok=True)
    !unzip -q /content/drive/MyDrive/FairFedCXR/data/chexpert.zip \
           -d /content/chexpert/
    print("Unzip done ✓")
else:
    print("Images already on local SSD ✓")

!pip install -q torch torchvision scikit-learn pandas tqdm scipy matplotlib

IMG_ROOT     = '/content/chexpert'
CLIENTS      = '/content/drive/MyDrive/FairFedCXR/clients'
CKPTS        = '/content/drive/MyDrive/FairFedCXR/checkpoints'
RESULTS      = '/content/drive/MyDrive/FairFedCXR/results'
FIGURES      = '/content/drive/MyDrive/FairFedCXR/figures'
SEEDS        = [42, 123, 456]
ROUNDS       = 30
LOCAL_EPOCHS = 2
BATCH_SIZE   = 32
LR           = 1e-4
WEIGHT_DECAY = 1e-4
EPS          = 1.0
ECE_BINS     = 8
MU           = 0.01
CLIENT_LIST  = ['A', 'B', 'C', 'D']

os.makedirs(CKPTS,   exist_ok=True)
os.makedirs(RESULTS, exist_ok=True)
os.makedirs(FIGURES, exist_ok=True)

print(f"\n✓ Ready")
print(f"  SEEDS={SEEDS} | ROUNDS={ROUNDS} | LOCAL_EPOCHS={LOCAL_EPOCHS}")
print(f"  MU={MU} | BATCH={BATCH_SIZE} | LR={LR} | EPS={EPS}")

Mounted at /content/drive
Free disk: 202.5 GB
Unzipping to local SSD (3–5 min)...
Unzip done ✓

✓ Ready
  SEEDS=[42, 123, 456] | ROUNDS=30 | LOCAL_EPOCHS=2
  MU=0.01 | BATCH=32 | LR=0.0001 | EPS=1.0


In [ ]:
# baseline fadeprox day 6,2
import torch, numpy as np, random, gc, time, os, copy
import torchvision.models as models
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from PIL import Image
import pandas as pd
from tqdm import tqdm
from sklearn.metrics import roc_auc_score, average_precision_score
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from torch.amp import autocast, GradScaler

torch.backends.cudnn.benchmark = True

def set_seed(seed):
    torch.manual_seed(seed); np.random.seed(seed)
    random.seed(seed); torch.cuda.manual_seed_all(seed)
    print(f"  Seed locked: {seed}")

train_tf = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(0.5),
    transforms.ToTensor(),
    transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225]),
])
eval_tf = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225]),
])

class CheXpertDataset(Dataset):
    def __init__(self, csv_path, transform):
        self.df = pd.read_csv(csv_path); self.tf = transform
        assert self.df['sex_encoded'].isin([0,1]).all()
        assert (self.df.loc[self.df['Sex']=='Male',
                'sex_encoded']==1).all()
        assert (self.df.loc[self.df['Sex']=='Female',
                'sex_encoded']==0).all()
    def __len__(self): return len(self.df)
    def __getitem__(self, idx):
        row   = self.df.iloc[idx]
        clean = row['Path'].replace('CheXpert-v1.0-small/', '')
        img   = Image.open(
                    os.path.join(IMG_ROOT, clean)).convert('RGB')
        img   = self.tf(img)
        return (img,
                torch.tensor(row['label'],       dtype=torch.float32),
                torch.tensor(row['sex_encoded'], dtype=torch.long))

def make_loader(csv_path, train=False):
    ds = CheXpertDataset(csv_path, train_tf if train else eval_tf)
    return DataLoader(ds, batch_size=BATCH_SIZE, shuffle=train,
                      num_workers=2, pin_memory=True,
                      persistent_workers=False)

def build_model(device):
    m = models.densenet121(weights='IMAGENET1K_V1')
    m.classifier = torch.nn.Linear(m.classifier.in_features, 1)
    return m.to(device)

# ── METRIC FUNCTIONS ──────────────────────────────────────────
def compute_ece(probs, labels, M=ECE_BINS):
    bins = np.linspace(0, 1, M+1); ece = 0.0
    for lo, hi in zip(bins[:-1], bins[1:]):
        mask = (probs >= lo) & (probs < hi)
        if mask.sum() == 0: continue
        ece += mask.mean() * abs(labels[mask].mean()
                                  - probs[mask].mean())
    return float(ece)

def compute_eo_gap(probs, labels, sex, threshold=0.5):
    """TPR gap — equal opportunity. EPS smoothed, never NaN."""
    preds = (probs >= threshold).astype(int)
    def stpr(mask):
        pos = (labels[mask] == 1)
        tp  = int(((preds[mask]==1) & pos).sum())
        p   = int(pos.sum())
        return (tp + EPS) / (p + 2*EPS)
    m_tpr = stpr(sex == 1)
    f_tpr = stpr(sex == 0)
    return abs(m_tpr - f_tpr), m_tpr, f_tpr

def compute_fpr_gap(probs, labels, sex, threshold=0.5):
    """FPR gap — equalized odds component. EPS smoothed."""
    preds = (probs >= threshold).astype(int)
    def sfpr(mask):
        neg = (labels[mask] == 0)
        fp  = int(((preds[mask]==1) & neg).sum())
        n   = int(neg.sum())
        return (fp + EPS) / (n + 2*EPS)
    return abs(sfpr(sex==1) - sfpr(sex==0))

def safe_auroc(labels_sub, probs_sub):
    """AUROC for a subgroup — returns nan if not computable."""
    if len(labels_sub) < 2: return float('nan')
    if labels_sub.sum() < 1: return float('nan')
    if (labels_sub == 0).sum() < 1: return float('nan')
    try:
        return float(roc_auc_score(labels_sub, probs_sub))
    except Exception:
        return float('nan')

@torch.no_grad()
def evaluate(model, loader, device, threshold=0.5):
    """
    Full evaluation — 12 metrics.
    NEW vs Day 4/5: adds male_auroc, female_auroc, wg_auroc, fpr_gap.
    Forward pass in fp16. All metrics computed in fp32 numpy.
    """
    model.eval(); P, L, S = [], [], []
    for imgs, labels, sex in loader:
        imgs = imgs.to(device, non_blocking=True)
        with autocast('cuda'):
            logits = model(imgs).squeeze()
        P.append(torch.sigmoid(logits.float()).cpu().numpy())
        L.append(labels.numpy())
        S.append(sex.numpy())
    probs  = np.concatenate(P)
    labels = np.concatenate(L)
    sex    = np.concatenate(S)

    eo, mt, ft   = compute_eo_gap(probs, labels, sex, threshold)
    fpr_g        = compute_fpr_gap(probs, labels, sex, threshold)

    # Per-sex AUROC — nan-safe
    m_mask = (sex == 1); f_mask = (sex == 0)
    m_auroc = safe_auroc(labels[m_mask], probs[m_mask])
    f_auroc = safe_auroc(labels[f_mask], probs[f_mask])
    wg_auroc = float(np.nanmin([m_auroc, f_auroc]))

    return {
        # Primary discrimination
        'auroc':        float(roc_auc_score(labels, probs)),
        'auprc':        float(average_precision_score(labels, probs)),
        # Calibration
        'ece':          float(compute_ece(probs, labels)),
        # Fairness — equal opportunity
        'eo_gap':       float(eo),
        'male_tpr':     float(mt),
        'female_tpr':   float(ft),
        # Fairness — per-sex discrimination (NEW)
        'male_auroc':   m_auroc,
        'female_auroc': f_auroc,
        'wg_auroc':     wg_auroc,
        # Fairness — FPR gap / equalized odds component (NEW)
        'fpr_gap':      float(fpr_g),
        # Subgroup counts — transparency
        'm_pos':        int((labels[m_mask]==1).sum()),
        'f_pos':        int((labels[f_mask]==1).sum()),
    }

print("✓ Imports, Dataset, Metrics defined")
print("  Metrics: auroc, auprc, ece, eo_gap, male_tpr, female_tpr,")
print("           male_auroc, female_auroc, wg_auroc, fpr_gap,")
print("           m_pos, f_pos  (12 total)")

✓ Imports, Dataset, Metrics defined
  Metrics: auroc, auprc, ece, eo_gap, male_tpr, female_tpr,
           male_auroc, female_auroc, wg_auroc, fpr_gap,
           m_pos, f_pos  (12 total)


In [ ]:
# baseline fadeprox day 6,3
from torch.nn.utils import parameters_to_vector

def local_train_fedprox(global_state, train_loader, device,
                        local_epochs=LOCAL_EPOCHS, mu=MU, tag=""):
    """
    FedProx local training — VECTORIZED proximal term.
    Uses parameters_to_vector for ONE GPU operation per batch
    instead of 364 separate kernel launches (DenseNet121).
    This brings FedProx runtime to within ~3% of FedAvg.
    """
    model = build_model(device)
    model.load_state_dict(global_state)
    model.train()

    # Flatten all global params into ONE vector on GPU — done once
    with torch.no_grad():
        global_vec = parameters_to_vector(
            [p.to(device) for p in
             [v for v in global_state.values()
              if v.dtype in (torch.float32, torch.float16,
                             torch.bfloat16)
              and len(v.shape) > 0]
        ]).detach()

    # Safer: build from model parameters directly after loading state
    # (guaranteed same order as model.parameters())
    global_vec = parameters_to_vector(
        [p.detach().clone() for p in model.parameters()]
    ).to(device)

    optimizer = torch.optim.AdamW(model.parameters(),
                                   lr=LR, weight_decay=WEIGHT_DECAY)
    criterion = torch.nn.BCEWithLogitsLoss()
    scaler    = GradScaler('cuda')

    for ep in range(local_epochs):
        for imgs, labels, _ in tqdm(train_loader,
                                     desc=f"{tag} ep{ep+1}",
                                     leave=False):
            imgs   = imgs.to(device,   non_blocking=True)
            labels = labels.to(device, non_blocking=True)
            optimizer.zero_grad()

            with autocast('cuda'):
                logits = model(imgs).squeeze()
                loss   = criterion(logits, labels)

            # Proximal term — ONE vector operation, outside autocast
            # global_vec is frozen; current_vec has gradients
            current_vec = parameters_to_vector(model.parameters())
            prox  = (current_vec - global_vec).pow(2).sum()
            loss  = loss + (mu / 2.0) * prox

            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()

    state = {k: v.cpu().clone()
             for k, v in model.state_dict().items()}
    del model, optimizer, global_vec, current_vec
    gc.collect(); torch.cuda.empty_cache()
    return state

def fedavg_aggregate(client_states, client_weights):
    global_state = copy.deepcopy(client_states[0])
    for key in global_state:
        stacked = torch.stack([
            client_states[i][key].float() * client_weights[i]
            for i in range(len(client_states))
        ], dim=0)
        global_state[key] = stacked.sum(dim=0).to(
            client_states[0][key].dtype)
    return global_state

print("✓ FedProx engine FIXED — vectorized proximal term")
print(f"  μ={MU} | ONE vector operation per batch")
print("  Expected: same speed as FedAvg (~3.4 min/round)")

✓ FedProx engine FIXED — vectorized proximal term
  μ=0.01 | ONE vector operation per batch
  Expected: same speed as FedAvg (~3.4 min/round)


In [ ]:
# baseline fadeprox day 6,4
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"GPU : {torch.cuda.get_device_name(0) if device=='cuda' else 'CPU'}")

# Build pooled val
pooled_val = pd.concat([pd.read_csv(f'{CLIENTS}/client_{c}_val.csv')
                        for c in CLIENT_LIST], ignore_index=True)
pooled_val.to_csv('/content/pooled_val.csv', index=False)

print("\n── DEBUG: FedProx 2 rounds, seed 42 ────────────────────")
set_seed(42)

ctl = {c: make_loader(f'{CLIENTS}/client_{c}_train.csv', train=True)
       for c in CLIENT_LIST}
cs  = {c: len(ctl[c].dataset) for c in CLIENT_LIST}
N   = sum(cs.values())
cw  = [cs[c]/N for c in CLIENT_LIST]
pvl = make_loader('/content/pooled_val.csv', train=False)

gm = build_model(device)
gs = {k:v.cpu().clone() for k,v in gm.state_dict().items()}
del gm; torch.cuda.empty_cache()

for rnd in range(2):
    states = []
    for c in CLIENT_LIST:
        states.append(local_train_fedprox(
            gs, ctl[c], device, tag=f"R{rnd+1}-{c}"))
    gs = fedavg_aggregate(states, cw)
    del states; gc.collect()
    gm = build_model(device); gm.load_state_dict(gs)
    m  = evaluate(gm, pvl, device)
    del gm; gc.collect(); torch.cuda.empty_cache()
    print(f"  Round {rnd+1}: AUROC={m['auroc']:.4f} | "
          f"WG-AUROC={m['wg_auroc']:.4f} | EO={m['eo_gap']:.4f} | "
          f"FPR-gap={m['fpr_gap']:.4f}")

print("\n✓ FedProx + new metrics working")
print("  All 12 metrics computed without error ✓")
print("  Switch to L4 for full run")
del ctl; gc.collect(); torch.cuda.empty_cache()

GPU : NVIDIA L4

── DEBUG: FedProx 2 rounds, seed 42 ────────────────────
  Seed locked: 42
Downloading: "https://download.pytorch.org/models/densenet121-a639ec97.pth" to /root/.cache/torch/hub/checkpoints/densenet121-a639ec97.pth


100%|██████████| 30.8M/30.8M [00:00<00:00, 236MB/s]


  Round 1: AUROC=0.9446 | WG-AUROC=0.9429 | EO=0.0274 | FPR-gap=0.0065


  Round 2: AUROC=0.9537 | WG-AUROC=0.9508 | EO=0.0344 | FPR-gap=0.0023

✓ FedProx + new metrics working
  All 12 metrics computed without error ✓
  Switch to L4 for full run


In [ ]:
# baseline fadeprox day 6,5
device = 'cuda'
print(f"GPU : {torch.cuda.get_device_name(0)}")
print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB")

pooled_val = pd.concat([pd.read_csv(f'{CLIENTS}/client_{c}_val.csv')
                        for c in CLIENT_LIST], ignore_index=True)
pooled_val.to_csv('/content/pooled_val.csv', index=False)

fedprox_results   = []
fedprox_round_log = []
t_start = time.time()

print(f"\nFEDPROX | μ={MU} | {len(SEEDS)} seeds × {ROUNDS} rounds\n")

for seed in SEEDS:
    print(f"\n{'='*58}")
    print(f"  FEDPROX | SEED {seed}")
    print(f"{'='*58}")

    ckpt_path = f"{CKPTS}/fedprox_seed{seed}.pt"

    if os.path.exists(ckpt_path):
        print(f"  ⚡ Already done — loading for test eval")
        global_state = torch.load(ckpt_path, map_location='cpu')
    else:
        set_seed(seed)

        ctl = {c: make_loader(f'{CLIENTS}/client_{c}_train.csv',
                              train=True) for c in CLIENT_LIST}
        cs  = {c: len(ctl[c].dataset) for c in CLIENT_LIST}
        N   = sum(cs.values())
        cw  = [cs[c]/N for c in CLIENT_LIST]
        pvl = make_loader('/content/pooled_val.csv', train=False)

        print(f"  Weights: "
              f"{[f'{c}={w:.3f}' for c,w in zip(CLIENT_LIST,cw)]}")

        gm = build_model(device)
        global_state = {k:v.cpu().clone()
                        for k,v in gm.state_dict().items()}
        del gm; torch.cuda.empty_cache()

        best_auroc = -1.0; best_gs = None

        for rnd in range(ROUNDS):
            states = []
            for c in CLIENT_LIST:
                states.append(local_train_fedprox(
                    global_state, ctl[c], device,
                    tag=f"R{rnd+1}-{c}"))
            global_state = fedavg_aggregate(states, cw)
            del states; gc.collect()

            gm = build_model(device); gm.load_state_dict(global_state)
            vm = evaluate(gm, pvl, device)
            del gm; gc.collect(); torch.cuda.empty_cache()

            fedprox_round_log.append({
                'method':'fedprox','seed':seed,'round':rnd+1,
                'val_auroc':vm['auroc'],'val_eo_gap':vm['eo_gap'],
                'val_wg_auroc':vm['wg_auroc'],'val_ece':vm['ece']})

            print(f"  Round {rnd+1:02d}/{ROUNDS}: "
                  f"AUROC={vm['auroc']:.4f} | "
                  f"WG-AUROC={vm['wg_auroc']:.4f} | "
                  f"EO={vm['eo_gap']:.4f}")

            if vm['auroc'] > best_auroc:
                best_auroc = vm['auroc']
                best_gs    = copy.deepcopy(global_state)

            pd.DataFrame(fedprox_round_log).to_csv(
                f'{RESULTS}/fedprox_round_log.csv', index=False)

        global_state = best_gs
        torch.save(global_state, ckpt_path)
        print(f"  ✓ Best saved (val AUROC={best_auroc:.4f})")

        del ctl, pvl; gc.collect(); torch.cuda.empty_cache()

    # ── TEST all 4 clients ────────────────────────────────────
    gm = build_model(device); gm.load_state_dict(global_state)
    for c in CLIENT_LIST:
        te = make_loader(
            f'{CLIENTS}/client_{c}_test.csv', train=False)
        tm = evaluate(gm, te, device)
        tm.update({'method':'fedprox','client':c,'seed':seed})
        fedprox_results.append(tm)
        print(f"  → {c}: AUROC={tm['auroc']:.4f} | "
              f"WG-AUROC={tm['wg_auroc']:.4f} | "
              f"EO={tm['eo_gap']:.4f} | "
              f"FPR={tm['fpr_gap']:.4f} | "
              f"mTPR={tm['male_tpr']:.4f} | "
              f"fTPR={tm['female_tpr']:.4f}")
        del te; gc.collect()
    del gm; gc.collect(); torch.cuda.empty_cache()

    pd.DataFrame(fedprox_results).to_csv(
        f'{RESULTS}/fedprox_all.csv', index=False)
    print(f"  ✓ Saved after seed {seed}")

elapsed = (time.time()-t_start)/3600
print(f"\n{'='*58}")
print(f"FEDPROX COMPLETE in {elapsed:.2f} hr")
print(f"✓ Test    → {RESULTS}/fedprox_all.csv")
print(f"✓ Log     → {RESULTS}/fedprox_round_log.csv")
print(f"✓ Ckpts   → {CKPTS}/fedprox_seed*.pt")
print(f"  Rows: {len(fedprox_results)} (expect 12)")

GPU : NVIDIA L4
VRAM: 23.7 GB

FEDPROX | μ=0.01 | 3 seeds × 30 rounds


  FEDPROX | SEED 42
  Seed locked: 42
  Weights: ['A=0.418', 'B=0.223', 'C=0.192', 'D=0.167']


  Round 01/30: AUROC=0.9458 | WG-AUROC=0.9434 | EO=0.0360


  Round 02/30: AUROC=0.9517 | WG-AUROC=0.9488 | EO=0.0247


  Round 03/30: AUROC=0.9518 | WG-AUROC=0.9488 | EO=0.0425


  Round 04/30: AUROC=0.9517 | WG-AUROC=0.9480 | EO=0.0307


...

  Round 26/30: AUROC=0.9499 | WG-AUROC=0.9482 | EO=0.0452


  Round 27/30: AUROC=0.9517 | WG-AUROC=0.9489 | EO=0.0328


  Round 28/30: AUROC=0.9500 | WG-AUROC=0.9457 | EO=0.0274


  Round 29/30: AUROC=0.9496 | WG-AUROC=0.9460 | EO=0.0339


  Round 30/30: AUROC=0.9491 | WG-AUROC=0.9461 | EO=0.0391
  ✓ Best saved (val AUROC=0.9540)
  → A: AUROC=0.9507 | WG-AUROC=0.9471 | EO=0.0036 | FPR=0.0128 | mTPR=0.8852 | fTPR=0.8889
  → B: AUROC=0.9522 | WG-AUROC=0.9453 | EO=0.0045 | FPR=0.0400 | mTPR=0.8886 | fTPR=0.8931
  → C: AUROC=0.9375 | WG-AUROC=0.9318 | EO=0.0173 | FPR=0.0235 | mTPR=0.8876 | fTPR=0.9049
  → D: AUROC=0.9642 | WG-AUROC=0.9548 | EO=0.0388 | FPR=0.0262 | mTPR=0.8551 | fTPR=0.8939
  ✓ Saved after seed 456

FEDPROX COMPLETE in 5.32 hr
✓ Test    → /content/drive/MyDrive/FairFedCXR/results/fedprox_all.csv
✓ Log     → /content/drive/MyDrive/FairFedCXR/results/fedprox_round_log.csv
✓ Ckpts   → /content/drive/MyDrive/FairFedCXR/checkpoints/fedprox_seed*.pt
  Rows: 12 (expect 12)


In [ ]:
# baseline fadeprox day 6,6
import pandas as pd, numpy as np
import matplotlib.pyplot as plt

clients = ['A','B','C','D']

def cstat(df,m):
    return {c:(df[df['client']==c][m].mean(),
               df[df['client']==c][m].std())
            for c in clients}
def pstat(df,m):
    p=df.groupby('seed')[m].mean(); return p.mean(),p.std()
def ms(a,s): return f"{a:.4f}±{s:.4f}"
def ms1(a): return f"{a:.4f}"

# Load all methods
local = pd.read_csv(f'{RESULTS}/local_only_all.csv')
cen   = pd.read_csv(f'{RESULTS}/centralized_all.csv')
fed   = pd.read_csv(f'{RESULTS}/fedavg_all.csv')
prox  = pd.read_csv(f'{RESULTS}/fedprox_all.csv')

assert len(prox)==12, f"Expected 12 rows, got {len(prox)}"

# ── TABLE 1: FedProx full metrics ─────────────────────────────
print("FEDPROX — Full test metrics (mean ± std, 3 seeds)")
print(f"\n{'─'*78}")
print(f"  {'Clnt':<5} {'AUROC':>14} {'WG-AUROC':>14} "
      f"{'EO-gap':>12} {'FPR-gap':>12} {'ECE':>12}")
print(f"{'─'*78}")
for c in clients:
    sub = prox[prox['client']==c]
    print(f"  {c:<5} "
          f"{ms(sub['auroc'].mean(),sub['auroc'].std()):>14} "
          f"{ms(sub['wg_auroc'].mean(),sub['wg_auroc'].std()):>14} "
          f"{ms(sub['eo_gap'].mean(),sub['eo_gap'].std()):>12} "
          f"{ms(sub['fpr_gap'].mean(),sub['fpr_gap'].std()):>12} "
          f"{ms(sub['ece'].mean(),sub['ece'].std()):>12}")
print(f"{'─'*78}")
pool_a=prox.groupby('seed')['auroc'].mean()
pool_w=prox.groupby('seed')['wg_auroc'].mean()
print(f"  {'Pool':<5} {ms(pool_a.mean(),pool_a.std()):>14} "
      f"{ms(pool_w.mean(),pool_w.std()):>14}")

# ── TABLE 2: 4-method AUROC comparison ───────────────────────
print(f"\n\n4-METHOD COMPARISON — AUROC (mean, 3 seeds)")
print(f"{'─'*74}")
print(f"  {'Clnt':<5} {'Local':>12} {'Central':>12} "
      f"{'FedAvg':>12} {'FedProx':>12}")
print(f"{'─'*74}")
for c in clients:
    vals = [df[df['client']==c]['auroc'].mean()
            for df in [local,cen,fed,prox]]
    best_idx = int(np.argmax(vals))
    row = f"  {c:<5}"
    for i,v in enumerate(vals):
        marker = " *" if i==best_idx else "  "
        row += f" {v:>10.4f}{marker}"
    print(row)
print(f"{'─'*74}")
print(f"  {'Pool':<5}", end="")
for df_ in [local,cen,fed,prox]:
    print(f" {df_.groupby('seed')['auroc'].mean().mean():>12.4f}", end="")
print()

# ── TABLE 3: 4-method EO-gap comparison ──────────────────────
print(f"\n\n4-METHOD COMPARISON — EO-gap (mean ± std, 3 seeds)")
print(f"{'─'*80}")
print(f"  {'Clnt':<5} {'Local':>14} {'Central':>14} "
      f"{'FedAvg':>14} {'FedProx':>14}")
print(f"{'─'*80}")
for c in clients:
    row = f"  {c:<5}"
    vals = []
    for df_ in [local,cen,fed,prox]:
        sub = df_[df_['client']==c]['eo_gap']
        vals.append(sub.mean())
        row += f" {ms(sub.mean(),sub.std()):>14}"
    # flag direction vs FedAvg
    flag = " ↓" if vals[3]<vals[2] else " ↑" if vals[3]>vals[2] else "  "
    print(row + flag)
print(f"{'─'*80}")
print(f"  {'Pool':<5}", end="")
for df_ in [local,cen,fed,prox]:
    print(f" {df_.groupby('seed')['eo_gap'].mean().mean():>14.4f}", end="")
print()

# ── TABLE 4: WG-AUROC comparison (FedAvg vs FedProx) ─────────
print(f"\n\nWG-AUROC COMPARISON — FedAvg vs FedProx")
print("(min of male_auroc and female_auroc per client)")
print(f"{'─'*60}")
print(f"  {'Clnt':<5} {'FedAvg WG-AUROC':>18} {'FedProx WG-AUROC':>18} {'Better?':>10}")
print(f"{'─'*60}")
# FedAvg may not have wg_auroc if old CSV — handle gracefully
has_wg_fed = 'wg_auroc' in fed.columns
for c in clients:
    psub = prox[prox['client']==c]
    pw   = ms(psub['wg_auroc'].mean(), psub['wg_auroc'].std())
    if has_wg_fed:
        fsub = fed[fed['client']==c]
        fw   = ms(fsub['wg_auroc'].mean(), fsub['wg_auroc'].std())
        better = "FedProx ✓" if psub['wg_auroc'].mean() > \
                                  fsub['wg_auroc'].mean() else "FedAvg"
    else:
        fw = "N/A (recompute)"
        better = "—"
    print(f"  {c:<5} {fw:>18} {pw:>18} {better:>10}")
print(f"{'─'*60}")

# ── TABLE 5: EO-gap STD — variance story ──────────────────────
print(f"\n\nEO-GAP STD ACROSS SEEDS — Stability (key finding F4)")
print(f"{'─'*62}")
print(f"  {'Clnt':<5} {'FedAvg std':>14} {'FedProx std':>14} "
      f"{'FedProx better?':>16}")
print(f"{'─'*62}")
for c in clients:
    fs = fed[fed['client']==c]['eo_gap'].std()
    ps = prox[prox['client']==c]['eo_gap'].std()
    better = "Yes ✓" if ps < fs else "No"
    print(f"  {c:<5} {fs:>14.4f} {ps:>14.4f} {better:>16}")
print(f"{'─'*62}")
print("  Key: Client C FedAvg std=0.0124 (CV=78%) — does FedProx reduce this?")

# ── TABLE 6: Sex TPR all clients all methods ──────────────────
print(f"\n\nSEX TPR — All clients, FedAvg vs FedProx")
print(f"{'─'*72}")
print(f"  {'Clnt':<5} {'FedAvg mTPR':>13} {'FedAvg fTPR':>13} "
      f"{'FedProx mTPR':>14} {'FedProx fTPR':>14}")
print(f"{'─'*72}")
for c in clients:
    fsub = fed[fed['client']==c]
    psub = prox[prox['client']==c]
    fm   = fsub['male_tpr'].mean()   if 'male_tpr' in fed.columns  else float('nan')
    ff   = fsub['female_tpr'].mean() if 'female_tpr' in fed.columns else float('nan')
    pm   = psub['male_tpr'].mean()
    pf   = psub['female_tpr'].mean()
    # who is underserved?
    und = "M" if pm < pf else "F"
    print(f"  {c:<5} {fm:>13.4f} {ff:>13.4f} "
          f"{pm:>14.4f} {pf:>14.4f}  ({und} underserved)")
print(f"{'─'*72}")

# ── FIGURE: Convergence FedAvg vs FedProx ────────────────────
print(f"\nGenerating convergence figure...")
fl  = pd.read_csv(f'{RESULTS}/fedavg_round_log.csv')
pl  = pd.read_csv(f'{RESULTS}/fedprox_round_log.csv')

fig, axes = plt.subplots(1, 3, figsize=(16, 5))
colors = {'FedAvg':'#1B4F72','FedProx':'#117A65'}

for ax, metric, ylabel in [
    (axes[0], 'val_auroc',    'Pooled Val AUROC'),
    (axes[1], 'val_eo_gap',   'Pooled Val EO-gap'),
    (axes[2], 'val_wg_auroc', 'Pooled Val WG-AUROC'),
]:
    for df_, label in [(fl,'FedAvg'),(pl,'FedProx')]:
        if metric not in df_.columns:
            continue
        piv  = df_.pivot_table(
            index='round', columns='seed', values=metric)
        mean = piv.mean(axis=1); std = piv.std(axis=1)
        ax.plot(piv.index, mean,
                color=colors[label], lw=2, label=label)
        ax.fill_between(piv.index,
                        mean-std, mean+std,
                        color=colors[label], alpha=0.2)
    ax.set_xlabel('Communication Round', fontsize=10)
    ax.set_ylabel(ylabel, fontsize=10)
    ax.set_title(ylabel, fontsize=11, fontweight='bold')
    ax.legend(fontsize=9)
    ax.grid(alpha=0.3)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)

plt.suptitle('FedAvg vs FedProx — Convergence (mean ± std, 3 seeds)',
             fontsize=12, fontweight='bold', y=1.02)
plt.tight_layout()
fig_path = f'{FIGURES}/fedprox_convergence.pdf'
plt.savefig(fig_path, dpi=300, bbox_inches='tight')
plt.show()
print(f"✓ Figure saved → {fig_path}")

# ── KEY FINDINGS SUMMARY ──────────────────────────────────────
print(f"\n{'='*70}")
print("DAY 6 FEDPROX COMPLETE ✓")
print(f"{'─'*70}")
print("Files:")
print(f"  {RESULTS}/fedprox_all.csv      (12 rows, 15 columns)")
print(f"  {RESULTS}/fedprox_round_log.csv")
print(f"  {FIGURES}/fedprox_convergence.pdf")
print(f"  {CKPTS}/fedprox_seed*.pt  (3 files)")
print(f"{'─'*70}")
print("Watch these findings across ALL clients:")
print("  F3: Clients A and D — did FedProx help where FedAvg hurt?")
print("  F4: Client C EO-gap STD — is it lower than FedAvg's 0.0124?")
print("  F5: WG-AUROC — which method is fairer to the minority sex?")
print("  F6: FPR-gap — does the false positive disparity match TPR story?")
print("  F7: Sex TPR direction — did FedProx reverse the C disparity again?")
print(f"{'─'*70}")
print("Next: Day 7 → q-FedAvg (dedicated fairness-aware baseline)")
print(f"{'='*70}")

FEDPROX — Full test metrics (mean ± std, 3 seeds)

──────────────────────────────────────────────────────────────────────────────
  Clnt           AUROC       WG-AUROC       EO-gap      FPR-gap          ECE
──────────────────────────────────────────────────────────────────────────────
  A      0.9513±0.0005  0.9477±0.0008 0.0141±0.0114 0.0156±0.0145 0.0413±0.0048
  B      0.9497±0.0024  0.9426±0.0024 0.0085±0.0041 0.0428±0.0080 0.0460±0.0119
  C      0.9372±0.0003  0.9309±0.0011 0.0159±0.0124 0.0102±0.0119 0.0591±0.0073
  D      0.9651±0.0027  0.9566±0.0029 0.0420±0.0148 0.0309±0.0091 0.0430±0.0060
──────────────────────────────────────────────────────────────────────────────
  Pool   0.9508±0.0005  0.9444±0.0008


4-METHOD COMPARISON — AUROC (mean, 3 seeds)
──────────────────────────────────────────────────────────────────────────
  Clnt         Local      Central       FedAvg      FedProx
──────────────────────────────────────────────────────────────────────────
  A         0.9440   

In [ ]:
#baseline Q-FEDAVG day 7,1
from google.colab import drive
drive.mount('/content/drive')

import os
stat    = os.statvfs('/content')
free_gb = (stat.f_bavail * stat.f_frsize) / 1e9
print(f"Free disk: {free_gb:.1f} GB")
if free_gb < 12:
    raise SystemExit("❌ Not enough disk space. Restart runtime.")

if not os.path.exists('/content/chexpert/train'):
    print("Unzipping to local SSD (3–5 min)...")
    os.makedirs('/content/chexpert', exist_ok=True)
    !unzip -q /content/drive/MyDrive/FairFedCXR/data/chexpert.zip \
           -d /content/chexpert/
    print("Unzip done ✓")
else:
    print("Images already on local SSD ✓")

!pip install -q torch torchvision scikit-learn pandas tqdm scipy matplotlib

IMG_ROOT     = '/content/chexpert'
CLIENTS      = '/content/drive/MyDrive/FairFedCXR/clients'
CKPTS        = '/content/drive/MyDrive/FairFedCXR/checkpoints'
RESULTS      = '/content/drive/MyDrive/FairFedCXR/results'
FIGURES      = '/content/drive/MyDrive/FairFedCXR/figures'
SEEDS        = [42, 123, 456, 789, 1010]
ROUNDS       = 30
LOCAL_EPOCHS = 2
BATCH_SIZE   = 32
LR           = 1e-4
WEIGHT_DECAY = 1e-4
EPS          = 1.0
ECE_BINS     = 8
Q_PARAM      = 0.2     # q-FFL fairness exponent (Li et al. 2020)
MAX_LOSS_BATCHES = 5   # batches used for quick loss estimate
CLIENT_LIST  = ['A', 'B', 'C', 'D']

os.makedirs(CKPTS,   exist_ok=True)
os.makedirs(RESULTS, exist_ok=True)
os.makedirs(FIGURES, exist_ok=True)

print(f"\n✓ Ready")
print(f"  SEEDS={SEEDS} | ROUNDS={ROUNDS} | LOCAL_EPOCHS={LOCAL_EPOCHS}")
print(f"  Q_PARAM={Q_PARAM} | MAX_LOSS_BATCHES={MAX_LOSS_BATCHES}")
print(f"  BATCH={BATCH_SIZE} | LR={LR} | EPS={EPS}")

Mounted at /content/drive
Free disk: 202.5 GB
Unzipping to local SSD (3–5 min)...
Unzip done ✓

✓ Ready
  SEEDS=[42, 123, 456, 789, 1010] | ROUNDS=30 | LOCAL_EPOCHS=2
  Q_PARAM=0.2 | MAX_LOSS_BATCHES=5
  BATCH=32 | LR=0.0001 | EPS=1.0


In [ ]:
#baseline Q-FEDAVG day 7,2
import torch, numpy as np, random, gc, time, os, copy
import torchvision.models as models
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from PIL import Image
import pandas as pd
from tqdm import tqdm
from sklearn.metrics import roc_auc_score, average_precision_score
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from torch.amp import autocast, GradScaler

torch.backends.cudnn.benchmark = True

def set_seed(seed):
    torch.manual_seed(seed); np.random.seed(seed)
    random.seed(seed); torch.cuda.manual_seed_all(seed)
    print(f"  Seed locked: {seed}")

train_tf = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(0.5),
    transforms.ToTensor(),
    transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225]),
])
eval_tf = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225]),
])

class CheXpertDataset(Dataset):
    def __init__(self, csv_path, transform):
        self.df = pd.read_csv(csv_path); self.tf = transform
        assert self.df['sex_encoded'].isin([0,1]).all()
        assert (self.df.loc[self.df['Sex']=='Male',
                'sex_encoded']==1).all()
        assert (self.df.loc[self.df['Sex']=='Female',
                'sex_encoded']==0).all()
    def __len__(self): return len(self.df)
    def __getitem__(self, idx):
        row   = self.df.iloc[idx]
        clean = row['Path'].replace('CheXpert-v1.0-small/', '')
        img   = Image.open(
                    os.path.join(IMG_ROOT, clean)).convert('RGB')
        img   = self.tf(img)
        return (img,
                torch.tensor(row['label'],       dtype=torch.float32),
                torch.tensor(row['sex_encoded'], dtype=torch.long))

def make_loader(csv_path, train=False):
    ds = CheXpertDataset(csv_path, train_tf if train else eval_tf)
    return DataLoader(ds, batch_size=BATCH_SIZE, shuffle=train,
                      num_workers=2, pin_memory=True,
                      persistent_workers=False)

def build_model(device):
    m = models.densenet121(weights='IMAGENET1K_V1')
    m.classifier = torch.nn.Linear(m.classifier.in_features, 1)
    return m.to(device)

# ── METRIC FUNCTIONS (full 12-metric set) ─────────────────────
def compute_ece(probs, labels, M=ECE_BINS):
    bins = np.linspace(0, 1, M+1); ece = 0.0
    for lo, hi in zip(bins[:-1], bins[1:]):
        mask = (probs >= lo) & (probs < hi)
        if mask.sum() == 0: continue
        ece += mask.mean() * abs(labels[mask].mean()
                                  - probs[mask].mean())
    return float(ece)

def compute_eo_gap(probs, labels, sex, threshold=0.5):
    preds = (probs >= threshold).astype(int)
    def stpr(mask):
        pos = (labels[mask] == 1)
        tp  = int(((preds[mask]==1) & pos).sum())
        p   = int(pos.sum())
        return (tp + EPS) / (p + 2*EPS)
    m_tpr = stpr(sex == 1); f_tpr = stpr(sex == 0)
    return abs(m_tpr - f_tpr), m_tpr, f_tpr

def compute_fpr_gap(probs, labels, sex, threshold=0.5):
    preds = (probs >= threshold).astype(int)
    def sfpr(mask):
        neg = (labels[mask] == 0)
        fp  = int(((preds[mask]==1) & neg).sum())
        n   = int(neg.sum())
        return (fp + EPS) / (n + 2*EPS)
    return abs(sfpr(sex==1) - sfpr(sex==0))

def safe_auroc(labels_sub, probs_sub):
    if len(labels_sub) < 2: return float('nan')
    if labels_sub.sum() < 1: return float('nan')
    if (labels_sub == 0).sum() < 1: return float('nan')
    try: return float(roc_auc_score(labels_sub, probs_sub))
    except Exception: return float('nan')

@torch.no_grad()
def evaluate(model, loader, device, threshold=0.5):
    model.eval(); P, L, S = [], [], []
    for imgs, labels, sex in loader:
        imgs = imgs.to(device, non_blocking=True)
        with autocast('cuda'):
            logits = model(imgs).squeeze()
        P.append(torch.sigmoid(logits.float()).cpu().numpy())
        L.append(labels.numpy()); S.append(sex.numpy())
    probs  = np.concatenate(P)
    labels = np.concatenate(L)
    sex    = np.concatenate(S)

    eo, mt, ft = compute_eo_gap(probs, labels, sex, threshold)
    fpr_g      = compute_fpr_gap(probs, labels, sex, threshold)
    m_mask = (sex == 1); f_mask = (sex == 0)
    m_auroc = safe_auroc(labels[m_mask], probs[m_mask])
    f_auroc = safe_auroc(labels[f_mask], probs[f_mask])
    wg_auroc = float(np.nanmin([m_auroc, f_auroc]))

    return {
        'auroc':        float(roc_auc_score(labels, probs)),
        'auprc':        float(average_precision_score(labels, probs)),
        'ece':          float(compute_ece(probs, labels)),
        'eo_gap':       float(eo),
        'male_tpr':     float(mt),
        'female_tpr':   float(ft),
        'male_auroc':   m_auroc,
        'female_auroc': f_auroc,
        'wg_auroc':     wg_auroc,
        'fpr_gap':      float(fpr_g),
        'm_pos':        int((labels[m_mask]==1).sum()),
        'f_pos':        int((labels[f_mask]==1).sum()),
    }

print("✓ Imports, Dataset, Metrics defined (12-metric set)")

✓ Imports, Dataset, Metrics defined (12-metric set)


In [ ]:
#baseline Q-FEDAVG day 7,3
def local_train_qfedavg(global_state, train_loader, device,
                        local_epochs=LOCAL_EPOCHS,
                        max_loss_batches=MAX_LOSS_BATCHES, tag=""):
    """
    q-FedAvg local procedure (practical loss-weighted variant,
    Li et al. 2020). Single model instantiation:
      Step 1 — quick forward-only loss estimate on the GLOBAL
               model (few batches, no gradient) — used for
               q-FFL aggregation weighting.
      Step 2 — standard local training, IDENTICAL to FedAvg
               (no local objective modification — q-FFL acts
               only on the server-side aggregation weights).
    Returns: (state_dict on CPU, pre-training loss estimate)
    """
    model = build_model(device)
    model.load_state_dict(global_state)

    # ── Step 1: quick loss estimate on global model ───────────
    model.eval()
    criterion = torch.nn.BCEWithLogitsLoss()
    loss_sum, loss_count = 0.0, 0
    with torch.no_grad():
        for i, (imgs, labels, _) in enumerate(train_loader):
            if i >= max_loss_batches:
                break
            imgs   = imgs.to(device,   non_blocking=True)
            labels = labels.to(device, non_blocking=True)
            with autocast('cuda'):
                logits = model(imgs).squeeze()
                l      = criterion(logits, labels)
            loss_sum += l.item(); loss_count += 1
    pre_loss = loss_sum / max(loss_count, 1)

    # ── Step 2: standard local training (same as FedAvg) ──────
    model.train()
    optimizer = torch.optim.AdamW(model.parameters(),
                                   lr=LR, weight_decay=WEIGHT_DECAY)
    scaler = GradScaler('cuda')
    for ep in range(local_epochs):
        for imgs, labels, _ in tqdm(train_loader,
                                     desc=f"{tag} ep{ep+1}",
                                     leave=False):
            imgs   = imgs.to(device,   non_blocking=True)
            labels = labels.to(device, non_blocking=True)
            optimizer.zero_grad()
            with autocast('cuda'):
                logits = model(imgs).squeeze()
                loss   = criterion(logits, labels)
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()

    state = {k: v.cpu().clone()
             for k, v in model.state_dict().items()}
    del model, optimizer
    gc.collect(); torch.cuda.empty_cache()
    return state, pre_loss


def qfedavg_aggregate(client_states, client_losses, q=Q_PARAM, eps=1e-6):
    """
    q-FFL aggregation (practical variant).
    Weight a_k ∝ L_k^q — higher local loss receives MORE weight,
    directing the global model to prioritise underperforming
    clients (device-level fairness, Li et al. 2020).
    q=0 recovers UNIFORM weighting (not sample-size — a key
    distinction from FedAvg's reduction case).
    Fully vectorized — no per-parameter Python loops.
    """
    losses   = np.clip(np.array(client_losses, dtype=np.float64),
                       eps, None)
    weighted = losses ** q
    a        = weighted / weighted.sum()

    global_state = copy.deepcopy(client_states[0])
    for key in global_state:
        stacked = torch.stack([
            client_states[i][key].float() * float(a[i])
            for i in range(len(client_states))
        ], dim=0)
        global_state[key] = stacked.sum(dim=0).to(
            client_states[0][key].dtype)
    return global_state, a

print("✓ q-FedAvg engine defined")
print(f"  q={Q_PARAM} | loss estimate: {MAX_LOSS_BATCHES} batches, "
      f"forward-only")
print("  Aggregation: vectorized torch.stack (no per-param loops)")

✓ q-FedAvg engine defined
  q=0.2 | loss estimate: 5 batches, forward-only
  Aggregation: vectorized torch.stack (no per-param loops)


In [ ]:
#baseline Q-FEDAVG day 7,4
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"GPU : {torch.cuda.get_device_name(0) if device=='cuda' else 'CPU'}")

pooled_val = pd.concat([pd.read_csv(f'{CLIENTS}/client_{c}_val.csv')
                        for c in CLIENT_LIST], ignore_index=True)
pooled_val.to_csv('/content/pooled_val.csv', index=False)

print("\n── DEBUG: q-FedAvg 2 rounds, seed 42 ───────────────────")
set_seed(42)

ctl = {c: make_loader(f'{CLIENTS}/client_{c}_train.csv', train=True)
       for c in CLIENT_LIST}
cs  = {c: len(ctl[c].dataset) for c in CLIENT_LIST}
N   = sum(cs.values())
fedavg_w = {c: cs[c]/N for c in CLIENT_LIST}  # for comparison only
pvl = make_loader('/content/pooled_val.csv', train=False)

gm = build_model(device)
gs = {k:v.cpu().clone() for k,v in gm.state_dict().items()}
del gm; torch.cuda.empty_cache()

for rnd in range(2):
    t0 = time.time()
    states, losses = [], []
    for c in CLIENT_LIST:
        st, l = local_train_qfedavg(gs, ctl[c], device, tag=f"R{rnd+1}-{c}")
        states.append(st); losses.append(l)
    gs, a = qfedavg_aggregate(states, losses)
    del states; gc.collect()

    gm = build_model(device); gm.load_state_dict(gs)
    m  = evaluate(gm, pvl, device)
    del gm; gc.collect(); torch.cuda.empty_cache()

    rt = (time.time()-t0)/60
    print(f"  Round {rnd+1} ({rt:.2f} min): "
          f"AUROC={m['auroc']:.4f} | EO={m['eo_gap']:.4f}")
    print(f"    Losses: {[f'{c}={l:.3f}' for c,l in zip(CLIENT_LIST,losses)]}")
    print(f"    q-FedAvg weights: "
          f"{[f'{c}={w:.3f}' for c,w in zip(CLIENT_LIST,a)]}")
    print(f"    FedAvg weights (fixed): "
          f"{[f'{c}={w:.3f}' for c,w in fedavg_w.items()]}")

print("\n✓ Debug complete — expect round time ~3.4-3.6 min")
print("✓ If round time matches FedAvg closely, switch to L4")
del ctl; gc.collect(); torch.cuda.empty_cache()

GPU : NVIDIA L4

── DEBUG: q-FedAvg 2 rounds, seed 42 ───────────────────
  Seed locked: 42
Downloading: "https://download.pytorch.org/models/densenet121-a639ec97.pth" to /root/.cache/torch/hub/checkpoints/densenet121-a639ec97.pth


100%|██████████| 30.8M/30.8M [00:00<00:00, 237MB/s]


  Round 1 (9.19 min): AUROC=0.9471 | EO=0.0446
    Losses: ['A=0.684', 'B=0.697', 'C=0.699', 'D=0.731']
    q-FedAvg weights: ['A=0.249', 'B=0.250', 'C=0.250', 'D=0.252']
    FedAvg weights (fixed): ['A=0.418', 'B=0.223', 'C=0.192', 'D=0.167']


  Round 2 (3.26 min): AUROC=0.9532 | EO=0.0377
    Losses: ['A=0.223', 'B=0.380', 'C=0.175', 'D=0.268']
    q-FedAvg weights: ['A=0.244', 'B=0.271', 'C=0.232', 'D=0.253']
    FedAvg weights (fixed): ['A=0.418', 'B=0.223', 'C=0.192', 'D=0.167']

✓ Debug complete — expect round time ~3.4-3.6 min
✓ If round time matches FedAvg closely, switch to L4


In [ ]:
#baseline Q-FEDAVG day 7,5
device = 'cuda'
print(f"GPU : {torch.cuda.get_device_name(0)}")
print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB")

pooled_val = pd.concat([pd.read_csv(f'{CLIENTS}/client_{c}_val.csv')
                        for c in CLIENT_LIST], ignore_index=True)
pooled_val.to_csv('/content/pooled_val.csv', index=False)

qfed_results   = []
qfed_round_log = []   # round, seed, client, loss, weight — for C3-style comparison
t_start = time.time()

print(f"\nQ-FEDAVG | q={Q_PARAM} | {len(SEEDS)} seeds × {ROUNDS} rounds\n")

for seed in SEEDS:
    print(f"\n{'='*58}")
    print(f"  Q-FEDAVG | SEED {seed}")
    print(f"{'='*58}")

    ckpt_path = f"{CKPTS}/qfedavg_seed{seed}.pt"

    if os.path.exists(ckpt_path):
        print(f"  ⚡ Already done — loading for test eval")
        global_state = torch.load(ckpt_path, map_location='cpu')
    else:
        set_seed(seed)

        ctl = {c: make_loader(f'{CLIENTS}/client_{c}_train.csv',
                              train=True) for c in CLIENT_LIST}
        cs  = {c: len(ctl[c].dataset) for c in CLIENT_LIST}
        N   = sum(cs.values())
        fedavg_w = {c: cs[c]/N for c in CLIENT_LIST}
        pvl = make_loader('/content/pooled_val.csv', train=False)

        gm = build_model(device)
        global_state = {k:v.cpu().clone()
                        for k,v in gm.state_dict().items()}
        del gm; torch.cuda.empty_cache()

        best_auroc = -1.0; best_gs = None

        for rnd in range(ROUNDS):
            states, losses = [], []
            for c in CLIENT_LIST:
                st, l = local_train_qfedavg(
                    global_state, ctl[c], device,
                    tag=f"R{rnd+1}-{c}")
                states.append(st); losses.append(l)

            global_state, a = qfedavg_aggregate(states, losses)
            del states; gc.collect()

            gm = build_model(device); gm.load_state_dict(global_state)
            vm = evaluate(gm, pvl, device)
            del gm; gc.collect(); torch.cuda.empty_cache()

            for ci, c in enumerate(CLIENT_LIST):
                qfed_round_log.append({
                    'method':'qfedavg','seed':seed,'round':rnd+1,
                    'client':c,'local_loss':losses[ci],
                    'weight':float(a[ci]),
                    'fedavg_weight':fedavg_w[c],
                    'val_auroc':vm['auroc'],'val_eo_gap':vm['eo_gap'],
                    'val_wg_auroc':vm['wg_auroc']})

            print(f"  Round {rnd+1:02d}/{ROUNDS}: "
                  f"AUROC={vm['auroc']:.4f} | "
                  f"WG-AUROC={vm['wg_auroc']:.4f} | "
                  f"EO={vm['eo_gap']:.4f} | "
                  f"weights={[round(float(w),3) for w in a]}")

            if vm['auroc'] > best_auroc:
                best_auroc = vm['auroc']
                best_gs    = copy.deepcopy(global_state)

            pd.DataFrame(qfed_round_log).to_csv(
                f'{RESULTS}/qfedavg_round_log.csv', index=False)

        global_state = best_gs
        torch.save(global_state, ckpt_path)
        print(f"  ✓ Best saved (val AUROC={best_auroc:.4f})")

        del ctl, pvl; gc.collect(); torch.cuda.empty_cache()

    # ── TEST all 4 clients ────────────────────────────────────
    gm = build_model(device); gm.load_state_dict(global_state)
    for c in CLIENT_LIST:
        te = make_loader(f'{CLIENTS}/client_{c}_test.csv', train=False)
        tm = evaluate(gm, te, device)
        tm.update({'method':'qfedavg','client':c,'seed':seed})
        qfed_results.append(tm)
        print(f"  → {c}: AUROC={tm['auroc']:.4f} | "
              f"WG-AUROC={tm['wg_auroc']:.4f} | "
              f"EO={tm['eo_gap']:.4f} | "
              f"FPR={tm['fpr_gap']:.4f} | "
              f"mTPR={tm['male_tpr']:.4f} | fTPR={tm['female_tpr']:.4f}")
        del te; gc.collect()
    del gm; gc.collect(); torch.cuda.empty_cache()

    pd.DataFrame(qfed_results).to_csv(
        f'{RESULTS}/qfedavg_all.csv', index=False)
    print(f"  ✓ Saved after seed {seed}")

elapsed = (time.time()-t_start)/3600
print(f"\n{'='*58}")
print(f"Q-FEDAVG COMPLETE in {elapsed:.2f} hr")
print(f"✓ Test    → {RESULTS}/qfedavg_all.csv")
print(f"✓ Log     → {RESULTS}/qfedavg_round_log.csv")
print(f"✓ Ckpts   → {CKPTS}/qfedavg_seed*.pt")
print(f"  Rows: {len(qfed_results)} (expect 12)")

GPU : NVIDIA L4
VRAM: 23.7 GB

Q-FEDAVG | q=0.2 | 5 seeds × 30 rounds


  Q-FEDAVG | SEED 42
  ⚡ Already done — loading for test eval
Downloading: "https://download.pytorch.org/models/densenet121-a639ec97.pth" to /root/.cache/torch/hub/checkpoints/densenet121-a639ec97.pth


100%|██████████| 30.8M/30.8M [00:00<00:00, 200MB/s]


  → A: AUROC=0.9524 | WG-AUROC=0.9481 | EO=0.0266 | FPR=0.0102 | mTPR=0.8665 | fTPR=0.8931
  → B: AUROC=0.9459 | WG-AUROC=0.9371 | EO=0.0142 | FPR=0.0486 | mTPR=0.8789 | fTPR=0.8931
  → C: AUROC=0.9345 | WG-AUROC=0.9264 | EO=0.0088 | FPR=0.0170 | mTPR=0.8764 | fTPR=0.8852
  → D: AUROC=0.9633 | WG-AUROC=0.9532 | EO=0.0301 | FPR=0.0244 | mTPR=0.8411 | fTPR=0.8712
  ✓ Saved after seed 42

  Q-FEDAVG | SEED 123
  ⚡ Already done — loading for test eval
  → A: AUROC=0.9511 | WG-AUROC=0.9474 | EO=0.0002 | FPR=0.0108 | mTPR=0.8618 | fTPR=0.8616
  → B: AUROC=0.9466 | WG-AUROC=0.9404 | EO=0.0022 | FPR=0.0222 | mTPR=0.8571 | fTPR=0.8550
  → C: AUROC=0.9363 | WG-AUROC=0.9270 | EO=0.0149 | FPR=0.0183 | mTPR=0.8539 | fTPR=0.8689
  → D: AUROC=0.9625 | WG-AUROC=0.9542 | EO=0.0301 | FPR=0.0443 | mTPR=0.8411 | fTPR=0.8712
  ✓ Saved after seed 123

  Q-FEDAVG | SEED 456
  ⚡ Already done — loading for test eval
  → A: AUROC=0.9506 | WG-AUROC=0.9480 | EO=0.0032 | FPR=0.0233 | mTPR=0.8899 | fTPR=0.8931
  → 

  Round 01/30: AUROC=0.9458 | WG-AUROC=0.9444 | EO=0.0235 | weights=[0.25, 0.249, 0.25, 0.252]


  Round 02/30: AUROC=0.9517 | WG-AUROC=0.9477 | EO=0.0348 | weights=[0.246, 0.249, 0.251, 0.253]


...

  Round 26/30: AUROC=0.9502 | WG-AUROC=0.9473 | EO=0.0321 | weights=[0.277, 0.258, 0.252, 0.213]


  Round 27/30: AUROC=0.9513 | WG-AUROC=0.9486 | EO=0.0312 | weights=[0.276, 0.281, 0.21, 0.234]


  Round 28/30: AUROC=0.9495 | WG-AUROC=0.9468 | EO=0.0385 | weights=[0.185, 0.207, 0.283, 0.325]


  Round 29/30: AUROC=0.9505 | WG-AUROC=0.9478 | EO=0.0428 | weights=[0.327, 0.279, 0.218, 0.177]


  Round 30/30: AUROC=0.9506 | WG-AUROC=0.9480 | EO=0.0218 | weights=[0.234, 0.278, 0.211, 0.278]
  ✓ Best saved (val AUROC=0.9542)
  → A: AUROC=0.9517 | WG-AUROC=0.9497 | EO=0.0210 | FPR=0.0095 | mTPR=0.8595 | fTPR=0.8805
  → B: AUROC=0.9476 | WG-AUROC=0.9394 | EO=0.0042 | FPR=0.0389 | mTPR=0.8668 | fTPR=0.8626
  → C: AUROC=0.9340 | WG-AUROC=0.9315 | EO=0.0248 | FPR=0.0054 | mTPR=0.8539 | fTPR=0.8787
  → D: AUROC=0.9637 | WG-AUROC=0.9534 | EO=0.0365 | FPR=0.0217 | mTPR=0.8271 | fTPR=0.8636
  ✓ Saved after seed 1010

Q-FEDAVG COMPLETE in 3.39 hr
✓ Test    → /content/drive/MyDrive/FairFedCXR/results/qfedavg_all.csv
✓ Log     → /content/drive/MyDrive/FairFedCXR/results/qfedavg_round_log.csv
✓ Ckpts   → /content/drive/MyDrive/FairFedCXR/checkpoints/qfedavg_seed*.pt
  Rows: 20 (expect 12)


In [ ]:
#baseline Q-FEDAVG day 7,
import pandas as pd, numpy as np
import matplotlib.pyplot as plt

clients = ['A','B','C','D']

def ms(a,s): return f"{a:.4f}±{s:.4f}"

local = pd.read_csv(f'{RESULTS}/local_only_all.csv')
cen   = pd.read_csv(f'{RESULTS}/centralized_all.csv')
fed   = pd.read_csv(f'{RESULTS}/fedavg_all.csv')
prox  = pd.read_csv(f'{RESULTS}/fedprox_all.csv')
qfed  = pd.read_csv(f'{RESULTS}/qfedavg_all.csv')

assert len(qfed)==20, f"Expected 20 rows, got {len(qfed)}"

# ── TABLE 1: q-FedAvg full metrics ────────────────────────────
print("Q-FEDAVG — Full test metrics (mean ± std, 3 seeds)")
print(f"\n{'─'*78}")
print(f"  {'Clnt':<5} {'AUROC':>14} {'WG-AUROC':>14} "
      f"{'EO-gap':>12} {'FPR-gap':>12} {'ECE':>12}")
print(f"{'─'*78}")
for c in clients:
    sub = qfed[qfed['client']==c]
    print(f"  {c:<5} "
          f"{ms(sub['auroc'].mean(),sub['auroc'].std()):>14} "
          f"{ms(sub['wg_auroc'].mean(),sub['wg_auroc'].std()):>14} "
          f"{ms(sub['eo_gap'].mean(),sub['eo_gap'].std()):>12} "
          f"{ms(sub['fpr_gap'].mean(),sub['fpr_gap'].std()):>12} "
          f"{ms(sub['ece'].mean(),sub['ece'].std()):>12}")
print(f"{'─'*78}")
pa=qfed.groupby('seed')['auroc'].mean(); pw=qfed.groupby('seed')['wg_auroc'].mean()
print(f"  {'Pool':<5} {ms(pa.mean(),pa.std()):>14} {ms(pw.mean(),pw.std()):>14}")

# ── TABLE 2: 5-method AUROC comparison ────────────────────────
print(f"\n\n5-METHOD COMPARISON — AUROC (mean, 3 seeds)")
print(f"{'─'*86}")
print(f"  {'Clnt':<5} {'Local':>11} {'Central':>11} "
      f"{'FedAvg':>11} {'FedProx':>11} {'qFedAvg':>11}")
print(f"{'─'*86}")
for c in clients:
    vals = [df[df['client']==c]['auroc'].mean()
            for df in [local,cen,fed,prox,qfed]]
    best = int(np.argmax(vals))
    row = f"  {c:<5}"
    for i,v in enumerate(vals):
        row += f" {v:>9.4f}{'*' if i==best else ' '}"
    print(row)
print(f"{'─'*86}")
print(f"  {'Pool':<5}", end="")
for df_ in [local,cen,fed,prox,qfed]:
    print(f" {df_.groupby('seed')['auroc'].mean().mean():>11.4f}", end="")
print()

# ── TABLE 3: 5-method EO-gap comparison ───────────────────────
print(f"\n\n5-METHOD COMPARISON — EO-gap (mean, 3 seeds)")
print(f"{'─'*86}")
print(f"  {'Clnt':<5} {'Local':>11} {'Central':>11} "
      f"{'FedAvg':>11} {'FedProx':>11} {'qFedAvg':>11}")
print(f"{'─'*86}")
for c in clients:
    vals = [df[df['client']==c]['eo_gap'].mean()
            for df in [local,cen,fed,prox,qfed]]
    best = int(np.argmin(vals[2:])) + 2  # best among FL methods only
    row = f"  {c:<5}"
    for i,v in enumerate(vals):
        row += f" {v:>9.4f}{'*' if i==best else ' '}"
    print(row)
print(f"{'─'*86}")
print(f"  {'Pool':<5}", end="")
for df_ in [local,cen,fed,prox,qfed]:
    print(f" {df_.groupby('seed')['eo_gap'].mean().mean():>11.4f}", end="")
print()

# ── TABLE 4: EO-gap STD — stability across all FL methods ────
print(f"\n\nEO-GAP STD — Stability comparison (3 FL baselines)")
print(f"{'─'*68}")
print(f"  {'Clnt':<5} {'FedAvg std':>13} {'FedProx std':>13} {'qFedAvg std':>13}")
print(f"{'─'*68}")
for c in clients:
    fs=fed[fed['client']==c]['eo_gap'].std()
    ps=prox[prox['client']==c]['eo_gap'].std()
    qs=qfed[qfed['client']==c]['eo_gap'].std()
    best = "qFedAvg" if qs<fs and qs<ps else ("FedProx" if ps<fs else "FedAvg")
    print(f"  {c:<5} {fs:>13.4f} {ps:>13.4f} {qs:>13.4f}  (most stable: {best})")
print(f"{'─'*68}")

# ── TABLE 5: q-FedAvg weight dynamics vs FedAvg fixed weights ─
print(f"\n\nQ-FEDAVG WEIGHT DYNAMICS — does it reweight toward Client C?")
print(f"(mean weight per client across all rounds, seed 42 shown)")
rl = pd.read_csv(f'{RESULTS}/qfedavg_round_log.csv')
rl42 = rl[rl['seed']==42]
print(f"{'─'*64}")
print(f"  {'Clnt':<5} {'FedAvg (fixed)':>16} {'qFedAvg (mean)':>16} {'qFedAvg (range)':>18}")
print(f"{'─'*64}")
for c in clients:
    sub = rl42[rl42['client']==c]
    fa_w = sub['fedavg_weight'].iloc[0]
    q_mean = sub['weight'].mean()
    q_min, q_max = sub['weight'].min(), sub['weight'].max()
    print(f"  {c:<5} {fa_w:>16.4f} {q_mean:>16.4f} "
          f"[{q_min:.3f}, {q_max:.3f}]")
print(f"{'─'*64}")
print("  → Does q-FedAvg consistently boost Client C above its")
print("    fixed FedAvg weight (0.192)? Check the mean column above.")

# ── FIGURE: 3-method convergence (AUROC, EO-gap, WG-AUROC) ───
print(f"\nGenerating convergence figure...")
fl = pd.read_csv(f'{RESULTS}/fedavg_round_log.csv')
pl = pd.read_csv(f'{RESULTS}/fedprox_round_log.csv')
# qfed round log has one row PER CLIENT per round — aggregate to pooled
ql = rl.groupby(['seed','round'])[['val_auroc','val_eo_gap','val_wg_auroc']].first().reset_index()

fig, axes = plt.subplots(1, 3, figsize=(16, 5))
colors = {'FedAvg':'#1B4F72','FedProx':'#117A65','qFedAvg':'#B9770E'}

for ax, metric, ylabel in [
    (axes[0],'val_auroc','Pooled Val AUROC'),
    (axes[1],'val_eo_gap','Pooled Val EO-gap'),
    (axes[2],'val_wg_auroc','Pooled Val WG-AUROC'),
]:
    for df_, label in [(fl,'FedAvg'),(pl,'FedProx'),(ql,'qFedAvg')]:
        if metric not in df_.columns: continue
        piv  = df_.pivot_table(index='round',columns='seed',values=metric)
        mean = piv.mean(axis=1); std = piv.std(axis=1)
        ax.plot(piv.index, mean, color=colors[label], lw=2, label=label)
        ax.fill_between(piv.index, mean-std, mean+std,
                        color=colors[label], alpha=0.15)
    ax.set_xlabel('Communication Round', fontsize=10)
    ax.set_ylabel(ylabel, fontsize=10)
    ax.set_title(ylabel, fontsize=11, fontweight='bold')
    ax.legend(fontsize=9); ax.grid(alpha=0.3)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)

plt.suptitle('FedAvg vs FedProx vs q-FedAvg — Convergence (mean±std, 3 seeds)',
             fontsize=12, fontweight='bold', y=1.02)
plt.tight_layout()
fig_path = f'{FIGURES}/qfedavg_convergence.pdf'
plt.savefig(fig_path, dpi=300, bbox_inches='tight')
plt.show()
print(f"✓ Figure saved → {fig_path}")

# ── FIGURE: q-FedAvg weight trajectory per client ─────────────
fig2, ax2 = plt.subplots(figsize=(9,5))
client_colors = {'A':'#1B4F72','B':'#117A65','C':'#C0392B','D':'#7D6608'}
for c in clients:
    sub = rl42[rl42['client']==c].sort_values('round')
    ax2.plot(sub['round'], sub['weight'], color=client_colors[c],
             lw=1.8, label=f'Client {c}', alpha=0.85)
    ax2.axhline(sub['fedavg_weight'].iloc[0], color=client_colors[c],
               linestyle='--', alpha=0.4, lw=1)
ax2.set_xlabel('Communication Round', fontsize=11)
ax2.set_ylabel('q-FedAvg Aggregation Weight', fontsize=11)
ax2.set_title('q-FedAvg Weight Trajectory vs FedAvg Fixed Weight (dashed)\n(seed 42)',
              fontsize=11, fontweight='bold')
ax2.legend(fontsize=9); ax2.grid(alpha=0.3)
ax2.spines['top'].set_visible(False); ax2.spines['right'].set_visible(False)
plt.tight_layout()
fig2_path = f'{FIGURES}/qfedavg_weight_trajectory.pdf'
plt.savefig(fig2_path, dpi=300, bbox_inches='tight')
plt.show()
print(f"✓ Figure saved → {fig2_path}")

print(f"\n{'='*70}")
print("DAY 7 Q-FEDAVG COMPLETE ✓")
print(f"{'─'*70}")
print("Files: qfedavg_all.csv, qfedavg_round_log.csv,")
print("       qfedavg_convergence.pdf, qfedavg_weight_trajectory.pdf")
print("       3 checkpoints qfedavg_seed*.pt")
print(f"{'─'*70}")
print("Key questions answered:")
print("  1. Does q-FedAvg beat FedAvg/FedProx on EO-gap at Client C?")
print("  2. Does q-FedAvg's loss-based reweighting boost Client C")
print("     (check Table 5 — does its weight exceed FedAvg's 0.192)?")
print("  3. Is q-FedAvg's EO-gap variance lower (more stable)?")
print(f"{'─'*70}")
print("Next: Day 8 → Analyze ALL baselines together, decide if DWFA")
print("      needs any adjustment before Days 9-12")
print(f"{'='*70}")

Q-FEDAVG — Full test metrics (mean ± std, 3 seeds)

──────────────────────────────────────────────────────────────────────────────
  Clnt           AUROC       WG-AUROC       EO-gap      FPR-gap          ECE
──────────────────────────────────────────────────────────────────────────────
  A      0.9517±0.0010  0.9489±0.0015 0.0136±0.0114 0.0145±0.0062 0.0442±0.0062
  B      0.9482±0.0022  0.9396±0.0016 0.0093±0.0073 0.0359±0.0120 0.0520±0.0071
  C      0.9357±0.0017  0.9288±0.0036 0.0145±0.0099 0.0157±0.0066 0.0595±0.0092
  D      0.9635±0.0009  0.9539±0.0014 0.0408±0.0122 0.0297±0.0090 0.0525±0.0055
──────────────────────────────────────────────────────────────────────────────
  Pool   0.9498±0.0010  0.9428±0.0012


5-METHOD COMPARISON — AUROC (mean, 3 seeds)
──────────────────────────────────────────────────────────────────────────────────────
  Clnt        Local     Central      FedAvg     FedProx     qFedAvg
───────────────────────────────────────────────────────────────────────────

In [ ]:
# DWFA day 8,1
from google.colab import drive
drive.mount('/content/drive')

import os
stat    = os.statvfs('/content')
free_gb = (stat.f_bavail * stat.f_frsize) / 1e9
print(f"Free disk: {free_gb:.1f} GB")
if free_gb < 12:
    raise SystemExit("Not enough disk space. Restart runtime.")

if not os.path.exists('/content/chexpert/train'):
    print("Unzipping CheXpert to local SSD (3-5 min)...")
    os.makedirs('/content/chexpert', exist_ok=True)
    !unzip -q /content/drive/MyDrive/FairFedCXR/data/chexpert.zip -d /content/chexpert/
    print("Unzip done")
else:
    print("Images already on local SSD")

!pip install -q torch torchvision scikit-learn pandas tqdm scipy matplotlib

# ---- Shared experiment config (identical to the baseline blocks) ----
IMG_ROOT     = '/content/chexpert'
CLIENTS      = '/content/drive/MyDrive/FairFedCXR/clients'
CKPTS        = '/content/drive/MyDrive/FairFedCXR/checkpoints'
RESULTS      = '/content/drive/MyDrive/FairFedCXR/results'
FIGURES      = '/content/drive/MyDrive/FairFedCXR/figures'
SEEDS        = [42, 123, 456, 789, 1010]   # DWFA primary uses 5 seeds (locked)
ROUNDS       = 30
LOCAL_EPOCHS = 2
BATCH_SIZE   = 32
LR           = 1e-4
WEIGHT_DECAY = 1e-4
EPS          = 1.0          # Laplace smoothing, matches LaTeX
ECE_BINS     = 8
CLIENT_LIST  = ['A', 'B', 'C', 'D']

# ---- DWFA locked hyperparameters (match the LaTeX appendix exactly) ----
RHO_DWFA       = 0.10
ALPHA_DWFA     = 0.40
KAPPA_DWFA     = 0.60
G_REF          = 0.05       # primary; swept in Cell 8
TAU_REF        = 200        # primary; swept in Cell 8
DWFA_THRESHOLD = 0.5        # operating threshold for the per-round audit
CKPT_EVERY     = 5          # resumable progress checkpoint cadence

assert abs(ALPHA_DWFA + KAPPA_DWFA - 1.0) < 1e-9, "alpha + kappa must equal 1"
os.makedirs(CKPTS,   exist_ok=True)
os.makedirs(RESULTS, exist_ok=True)
os.makedirs(FIGURES, exist_ok=True)

print(f"\nReady")
print(f"  SEEDS={SEEDS} | ROUNDS={ROUNDS} | LOCAL_EPOCHS={LOCAL_EPOCHS}")
print(f"  DWFA: rho={RHO_DWFA} alpha={ALPHA_DWFA} kappa={KAPPA_DWFA} "
      f"G_ref={G_REF} tau_ref={TAU_REF} | audit thr={DWFA_THRESHOLD}")

Mounted at /content/drive
Free disk: 202.9 GB
Unzipping CheXpert to local SSD (3-5 min)...
Unzip done

Ready
  SEEDS=[42, 123, 456, 789, 1010] | ROUNDS=30 | LOCAL_EPOCHS=2
  DWFA: rho=0.1 alpha=0.4 kappa=0.6 G_ref=0.05 tau_ref=200 | audit thr=0.5


In [ ]:
# DWFA day 8,2
import torch, numpy as np, random, gc, time, os, copy
import torchvision.models as models
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from torch.amp import autocast, GradScaler
from PIL import Image
import pandas as pd
from tqdm import tqdm
from sklearn.metrics import roc_auc_score, average_precision_score, roc_curve
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

torch.backends.cudnn.benchmark = True

def set_seed(seed):
    torch.manual_seed(seed); np.random.seed(seed); random.seed(seed)
    torch.cuda.manual_seed_all(seed)
    print(f"  Seed locked: {seed}")

train_tf = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(0.5),
    transforms.ToTensor(),
    transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225]),
])
eval_tf = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225]),
])

class CheXpertDataset(Dataset):
    def __init__(self, csv_path, transform):
        self.df = pd.read_csv(csv_path); self.tf = transform
        assert self.df['sex_encoded'].isin([0,1]).all()
        assert (self.df.loc[self.df['Sex']=='Male','sex_encoded']==1).all()
        assert (self.df.loc[self.df['Sex']=='Female','sex_encoded']==0).all()
    def __len__(self): return len(self.df)
    def __getitem__(self, idx):
        row   = self.df.iloc[idx]
        clean = row['Path'].replace('CheXpert-v1.0-small/', '')
        img   = Image.open(os.path.join(IMG_ROOT, clean)).convert('RGB')
        img   = self.tf(img)
        return (img,
                torch.tensor(row['label'],       dtype=torch.float32),
                torch.tensor(row['sex_encoded'], dtype=torch.long))

def make_loader(csv_path, train=False):
    on_gpu = torch.cuda.is_available()
    ds = CheXpertDataset(csv_path, train_tf if train else eval_tf)
    return DataLoader(ds, batch_size=BATCH_SIZE, shuffle=train,
                      num_workers=2 if on_gpu else 2,
                      pin_memory=on_gpu, persistent_workers=False)

def build_model(device):
    m = models.densenet121(weights='IMAGENET1K_V1')
    m.classifier = torch.nn.Linear(m.classifier.in_features, 1)
    return m.to(device)

# ---- Metrics (identical definitions to FedProx / q-FedAvg blocks) ----
def compute_ece(probs, labels, M=ECE_BINS):
    bins = np.linspace(0,1,M+1); ece=0.0
    for lo,hi in zip(bins[:-1],bins[1:]):
        mask=(probs>=lo)&(probs<hi)
        if mask.sum()==0: continue
        ece += mask.mean()*abs(labels[mask].mean()-probs[mask].mean())
    return float(ece)

def compute_eo_gap(probs, labels, sex, threshold=0.5):
    """TPR gap (equal opportunity), Laplace-smoothed (eps=1), never NaN."""
    preds=(probs>=threshold).astype(int)
    def stpr(mask):
        pos=(labels[mask]==1); tp=int(((preds[mask]==1)&pos).sum()); p=int(pos.sum())
        return (tp+EPS)/(p+2*EPS)
    m,f = stpr(sex==1), stpr(sex==0)
    return abs(m-f), m, f

def compute_fpr_gap(probs, labels, sex, threshold=0.5):
    """FPR gap (equalized-odds component), Laplace-smoothed (eps=1)."""
    preds=(probs>=threshold).astype(int)
    def sfpr(mask):
        neg=(labels[mask]==0); fp=int(((preds[mask]==1)&neg).sum()); n=int(neg.sum())
        return (fp+EPS)/(n+2*EPS)
    return abs(sfpr(sex==1)-sfpr(sex==0))

def safe_auroc(labels_sub, probs_sub):
    if len(labels_sub) < 2: return float('nan')
    if labels_sub.sum() < 1 or (labels_sub==0).sum() < 1: return float('nan')
    try:    return float(roc_auc_score(labels_sub, probs_sub))
    except Exception: return float('nan')

@torch.no_grad()
def youden_threshold(model, loader, device):
    """Youden's J optimal threshold from POOLED validation ROC. Pooled, not
    per-client — a per-client threshold would itself be a fairness
    intervention requiring separate justification (locked decision)."""
    model.eval(); P,L=[],[]
    for imgs,labels,_ in loader:
        imgs=imgs.to(device,non_blocking=True)
        with autocast('cuda'):
            logits=model(imgs).squeeze()
        P.append(torch.sigmoid(logits.float()).cpu().numpy()); L.append(labels.numpy())
    probs,labels=np.concatenate(P),np.concatenate(L)
    fpr,tpr,thr=roc_curve(labels,probs)
    return float(thr[int(np.argmax(tpr-fpr))])

@torch.no_grad()
def evaluate_dual(model, loader, device, thr2):
    """ONE fp16 forward pass; threshold-free metrics once, threshold-dependent
    metrics (EO-gap, FPR-gap, TPRs) at BOTH 0.5 and thr2 (Youden)."""
    model.eval(); P,L,S=[],[],[]
    for imgs,labels,sex in loader:
        imgs=imgs.to(device,non_blocking=True)
        with autocast('cuda'):
            logits=model(imgs).squeeze()
        P.append(torch.sigmoid(logits.float()).cpu().numpy())
        L.append(labels.numpy()); S.append(sex.numpy())
    probs,labels,sex=map(np.concatenate,(P,L,S))
    m_mask,f_mask=(sex==1),(sex==0)
    out={'auroc':float(roc_auc_score(labels,probs)),
         'auprc':float(average_precision_score(labels,probs)),
         'ece':float(compute_ece(probs,labels)),
         'male_auroc':safe_auroc(labels[m_mask],probs[m_mask]),
         'female_auroc':safe_auroc(labels[f_mask],probs[f_mask]),
         'm_pos':int((labels[m_mask]==1).sum()),
         'f_pos':int((labels[f_mask]==1).sum())}
    out['wg_auroc']=float(np.nanmin([out['male_auroc'],out['female_auroc']]))
    for sfx,thr in [('',0.5),('_yj',float(thr2))]:
        eo,mt,ft=compute_eo_gap(probs,labels,sex,thr)
        out[f'eo_gap{sfx}']=float(eo); out[f'fpr_gap{sfx}']=float(compute_fpr_gap(probs,labels,sex,thr))
        out[f'male_tpr{sfx}']=float(mt); out[f'female_tpr{sfx}']=float(ft)
    out['youden_thr']=float(thr2)
    return out

print("Imports, dataset, model, metrics, dual-threshold eval defined")

Imports, dataset, model, metrics, dual-threshold eval defined


In [ ]:
# DWFA day 8,3
def compute_static_confidence(tau_ref=TAU_REF, rho=RHO_DWFA):
    """r_i = clip(min(m_pos, f_pos) / tau_ref, rho, 1), read once from the
    FROZEN validation CSVs. Static per-client constant — no model, no
    round index, computed before any training starts."""
    conf, counts = {}, {}
    for c in CLIENT_LIST:
        df = pd.read_csv(f'{CLIENTS}/client_{c}_val.csv')
        m_pos = int(((df['sex_encoded']==1) & (df['label']==1)).sum())
        f_pos = int(((df['sex_encoded']==0) & (df['label']==1)).sum())
        r = float(np.clip(min(m_pos, f_pos)/tau_ref, rho, 1.0))
        conf[c], counts[c] = r, (m_pos, f_pos)
    return conf, counts

@torch.no_grad()
def _audit_eo_gap(model, val_loader, device, threshold=DWFA_THRESHOLD):
    """Lean audit: ONE fp16 forward over the client's own val set -> EO-gap
    G_i^t only. No backward, no AUROC; this is the entire DWFA per-round
    overhead beyond standard FedAvg local training."""
    model.eval(); P,L,S=[],[],[]
    for imgs,labels,sex in val_loader:
        imgs=imgs.to(device,non_blocking=True)
        with autocast('cuda'):
            logits=model(imgs).squeeze()
        P.append(torch.sigmoid(logits.float()).cpu().numpy())
        L.append(labels.numpy()); S.append(sex.numpy())
    probs,labels,sex=map(np.concatenate,(P,L,S))
    eo,_,_=compute_eo_gap(probs,labels,sex,threshold)
    return float(eo)

def local_train_dwfa(global_state, train_loader, val_loader, device, tag=""):
    """Standard local training (identical optimization to FedAvg), then a
    single post-update validation forward pass on the SAME GPU model (no
    rebuild/reload) to measure G_i^t. Returns (cpu_state_dict, G_i)."""
    model = build_model(device)
    model.load_state_dict(global_state)
    model.train()
    opt    = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
    crit   = torch.nn.BCEWithLogitsLoss()
    scaler = GradScaler('cuda')
    for ep in range(LOCAL_EPOCHS):
        for imgs, labels, _ in tqdm(train_loader, desc=f"{tag} ep{ep+1}", leave=False):
            imgs   = imgs.to(device, non_blocking=True)
            labels = labels.to(device, non_blocking=True)
            opt.zero_grad()
            with autocast('cuda'):
                loss = crit(model(imgs).squeeze(), labels)
            scaler.scale(loss).backward()
            scaler.step(opt); scaler.update()
    g_i = _audit_eo_gap(model, val_loader, device)        # post-update audit
    state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
    del model, opt; gc.collect(); torch.cuda.empty_cache()
    return state, g_i

def gated_equity(g_i, r_i, ebar_prior, is_first, g_ref=G_REF, rho=RHO_DWFA):
    """Tolerance-scaled equity then mean-preserving gate.
        e_i      = clip(1 - G_i / G_ref, rho, 1)
        etilde_i = (1 - r_i) * ebar_prior + r_i * e_i
    First round (is_first) is an undamped warmup: ebar := e_i, matching
    the LaTeX definition ebar_i^1 := e_i^1 exactly. Returns (e_i, etilde_i)."""
    e_cur = float(np.clip(1.0 - g_i/g_ref, rho, 1.0))
    ebar  = e_cur if is_first else ebar_prior              # literal t==1 branch
    etilde = (1.0 - r_i)*ebar + r_i*e_cur
    return e_cur, float(etilde)

def dwfa_aggregate(client_states, etilde, p_weights, alpha=ALPHA_DWFA, kappa=KAPPA_DWFA):
    """q_i = alpha + kappa * etilde_i ; s_i = p_i * q_i ; a_i = s_i / sum_j s_j.
    Vectorized weighted parameter average — no per-parameter Python loops."""
    q = alpha + kappa*np.asarray(etilde, dtype=np.float64)
    s = np.asarray(p_weights, dtype=np.float64) * q
    a = s / s.sum()
    global_state = copy.deepcopy(client_states[0])
    for key in global_state:
        stacked = torch.stack([client_states[i][key].float()*float(a[i])
                               for i in range(len(client_states))], dim=0)
        global_state[key] = stacked.sum(dim=0).to(client_states[0][key].dtype)
    return global_state, a

print("DWFA engine defined: static confidence, train+audit, gated equity, aggregate")

DWFA engine defined: static confidence, train+audit, gated equity, aggregate


In [ ]:
# DWFA day 8,4
def _unit_test_fedavg_equiv():
    """When alpha=1, kappa=0 the quality multiplier q_i=1 for every client,
    so DWFA weights must reduce EXACTLY to the sample-size prior p_i
    (Proposition 1, FedAvg-reduction case). Calls dwfa_aggregate() and
    gated_equity() directly — not a reimplementation — so it also
    exercises the real code path and argument handling."""
    rng = np.random.default_rng(0)
    p = [0.4178, 0.2229, 0.1922, 0.1671]   # FedAvg priors A,B,C,D
    fake_states = [{'w': torch.tensor([float(i)])} for i in range(4)]
    etilde_arbitrary = rng.uniform(RHO_DWFA, 1.0, size=4).tolist()

    # Case 1: kappa=0 (alpha=1) — etilde must be IGNORED entirely
    _, a1 = dwfa_aggregate(fake_states, etilde_arbitrary, p, alpha=1.0, kappa=0.0)
    assert np.allclose(a1, p, atol=1e-12), f"FedAvg-equivalence FAILED: {a1} vs {p}"

    # Case 2: homogeneous etilde across clients (any alpha,kappa) -> a == p
    etilde_same = [0.7, 0.7, 0.7, 0.7]
    _, a2 = dwfa_aggregate(fake_states, etilde_same, p, alpha=ALPHA_DWFA, kappa=KAPPA_DWFA)
    assert np.allclose(a2, p, atol=1e-12), f"Homogeneous-equity FAILED: {a2} vs {p}"

    # Case 3: warmup branch reduces to undamped e_i (gated_equity, not aggregate)
    e_cur, et = gated_equity(g_i=0.03, r_i=0.4, ebar_prior=0.0, is_first=True)
    assert abs(et - e_cur) < 1e-12, f"Warmup branch FAILED: etilde={et} e_cur={e_cur}"

    print("Unit test PASSED:")
    print("  (1) alpha=1,kappa=0 -> a==p  [calls real dwfa_aggregate]")
    print("  (2) homogeneous etilde -> a==p  [calls real dwfa_aggregate]")
    print("  (3) t=1 warmup -> etilde==e_cur  [calls real gated_equity]")

_unit_test_fedavg_equiv()

Unit test PASSED:
  (1) alpha=1,kappa=0 -> a==p  [calls real dwfa_aggregate]
  (2) homogeneous etilde -> a==p  [calls real dwfa_aggregate]
  (3) t=1 warmup -> etilde==e_cur  [calls real gated_equity]


In [ ]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"GPU : {torch.cuda.get_device_name(0) if device=='cuda' else 'CPU'}")

GPU : Tesla T4


In [ ]:
# DWFA day 8,5
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"GPU : {torch.cuda.get_device_name(0) if device=='cuda' else 'CPU'}")

conf, counts = compute_static_confidence()
print("Static confidence r_i (from val counts):")
for c in CLIENT_LIST:
    print(f"  Client {c}: m_pos={counts[c][0]:>4} f_pos={counts[c][1]:>4} -> r={conf[c]:.3f}")

pooled_val = pd.concat([pd.read_csv(f'{CLIENTS}/client_{c}_val.csv')
                        for c in CLIENT_LIST], ignore_index=True)
pooled_val.to_csv('/content/pooled_val.csv', index=False)

print("\n-- DEBUG: DWFA 2 rounds, seed 42 --")
set_seed(42)
ctl = {c: make_loader(f'{CLIENTS}/client_{c}_train.csv', train=True) for c in CLIENT_LIST}
cvl = {c: make_loader(f'{CLIENTS}/client_{c}_val.csv',   train=False) for c in CLIENT_LIST}
cs  = {c: len(ctl[c].dataset) for c in CLIENT_LIST}
N   = sum(cs.values())
p_w = [cs[c]/N for c in CLIENT_LIST]
pvl = make_loader('/content/pooled_val.csv', train=False)

gm = build_model(device); gs = {k:v.cpu().clone() for k,v in gm.state_dict().items()}
del gm; torch.cuda.empty_cache()
ebar_sum = {c:0.0 for c in CLIENT_LIST}; ebar_cnt = {c:0 for c in CLIENT_LIST}

for rnd in range(2):
    states, et_list = [], []
    for c in CLIENT_LIST:
        st, g_i = local_train_dwfa(gs, ctl[c], cvl[c], device, tag=f"R{rnd+1}-{c}")
        is_first = (ebar_cnt[c]==0)
        ebar_prior = (ebar_sum[c]/ebar_cnt[c]) if not is_first else 0.0
        e_cur, etil = gated_equity(g_i, conf[c], ebar_prior, is_first)
        ebar_sum[c] += e_cur; ebar_cnt[c] += 1     # lagged update AFTER use
        states.append(st); et_list.append(etil)
        print(f"    {c}: G={g_i:.4f} e={e_cur:.3f} etilde={etil:.3f} (r={conf[c]:.2f})")
    gs, a = dwfa_aggregate(states, et_list, p_w)
    del states; gc.collect()
    gmm = build_model(device); gmm.load_state_dict(gs)
    m = evaluate_dual(gmm, pvl, device, 0.5)
    del gmm; gc.collect(); torch.cuda.empty_cache()
    print(f"  Round {rnd+1}: pooled val AUROC={m['auroc']:.4f} EO={m['eo_gap']:.4f} "
          f"| a={[round(float(w),3) for w in a]} (FedAvg p={[round(x,3) for x in p_w]})")

print("\nDebug complete — audit + gating + aggregation OK")
del ctl, cvl, pvl; gc.collect(); torch.cuda.empty_cache()

GPU : NVIDIA L4
Static confidence r_i (from val counts):
  Client A: m_pos= 433 f_pos= 467 -> r=1.000
  Client B: m_pos= 447 f_pos=  93 -> r=0.465
  Client C: m_pos=  96 f_pos= 295 -> r=0.480
  Client D: m_pos= 197 f_pos= 145 -> r=0.725

-- DEBUG: DWFA 2 rounds, seed 42 --
  Seed locked: 42


    A: G=0.0634 e=0.100 etilde=0.100 (r=1.00)


    B: G=0.0612 e=0.100 etilde=0.100 (r=0.47)


    C: G=0.1005 e=0.100 etilde=0.100 (r=0.48)


    D: G=0.0150 e=0.700 etilde=0.700 (r=0.72)
  Round 1: pooled val AUROC=0.9472 EO=0.0228 | a=[0.37, 0.197, 0.17, 0.263] (FedAvg p=[0.418, 0.223, 0.192, 0.167])


    A: G=0.0623 e=0.100 etilde=0.100 (r=1.00)


    B: G=0.1057 e=0.100 etilde=0.100 (r=0.47)


    C: G=0.0458 e=0.100 etilde=0.100 (r=0.48)


    D: G=0.0354 e=0.292 etilde=0.404 (r=0.72)
  Round 2: pooled val AUROC=0.9523 EO=0.0348 | a=[0.392, 0.209, 0.18, 0.219] (FedAvg p=[0.418, 0.223, 0.192, 0.167])

Debug complete — audit + gating + aggregation OK


In [ ]:
# DWFA day 8,6
def run_dwfa(seeds, rounds, g_ref, tau_ref, alpha=ALPHA_DWFA, kappa=KAPPA_DWFA,
             rho=RHO_DWFA, tag='dwfa', do_test=True, save_round_log=True):
    """Full multi-seed DWFA run with mean-preserving gated equity, static
    count-based confidence, lagged running mean (resumable), per-round
    logging, and dual-threshold test eval. Returns (results, round_log)."""
    device = 'cuda' if torch.cuda.is_available() else 'cpu'
    conf, counts = compute_static_confidence(tau_ref, rho)

    pooled_val = pd.concat([pd.read_csv(f'{CLIENTS}/client_{c}_val.csv')
                            for c in CLIENT_LIST], ignore_index=True)
    pooled_val.to_csv('/content/pooled_val.csv', index=False)

    rlog_path = f'{RESULTS}/{tag}_round_log.csv'
    res_path  = f'{RESULTS}/{tag}_all.csv'
    round_log = pd.read_csv(rlog_path).to_dict('records') if (save_round_log and os.path.exists(rlog_path)) else []
    results   = pd.read_csv(res_path).to_dict('records') if os.path.exists(res_path) else []
    done_seeds = {r['seed'] for r in results} if results else set()

    for seed in seeds:
        if seed in done_seeds:
            print(f"  [{tag}] seed {seed}: test results already present — skipping")
            continue
        print(f"\n{'='*60}\n  DWFA [{tag}] | SEED {seed} | G_ref={g_ref} tau_ref={tau_ref}\n{'='*60}")
        final_ckpt = f"{CKPTS}/{tag}_seed{seed}.pt"
        prog_ckpt  = f"{CKPTS}/{tag}_seed{seed}_progress.pt"

        if os.path.exists(final_ckpt):
            print(f"  Final ckpt found — loading best for test")
            global_state = torch.load(final_ckpt, map_location='cpu')
        else:
            set_seed(seed)
            ctl = {c: make_loader(f'{CLIENTS}/client_{c}_train.csv', train=True) for c in CLIENT_LIST}
            cvl = {c: make_loader(f'{CLIENTS}/client_{c}_val.csv',   train=False) for c in CLIENT_LIST}
            cs  = {c: len(ctl[c].dataset) for c in CLIENT_LIST}
            N   = sum(cs.values())
            p_w = [cs[c]/N for c in CLIENT_LIST]
            pvl = make_loader('/content/pooled_val.csv', train=False)

            if os.path.exists(prog_ckpt):
                ck = torch.load(prog_ckpt, map_location='cpu')
                global_state = ck['gs']; ebar_sum = ck['ebar_sum']; ebar_cnt = ck['ebar_cnt']
                start_round  = ck['next_round']; best_auroc = ck['best_auroc']; best_gs = ck['best_gs']
                round_log = [r for r in round_log if not (r['seed']==seed and r['round']>start_round)]
                print(f"  Resuming from round {start_round+1} (progress ckpt)")
            else:
                gm = build_model(device); global_state = {k:v.cpu().clone() for k,v in gm.state_dict().items()}
                del gm; torch.cuda.empty_cache()
                ebar_sum = {c:0.0 for c in CLIENT_LIST}; ebar_cnt = {c:0 for c in CLIENT_LIST}
                start_round, best_auroc, best_gs = 0, -1.0, None

            for rnd in range(start_round, rounds):
                states, et_list, gaps, ecur_list = [], [], [], []
                for c in CLIENT_LIST:
                    st, g_i = local_train_dwfa(global_state, ctl[c], cvl[c], device, tag=f"R{rnd+1}-{c}")
                    is_first = (ebar_cnt[c]==0)
                    ebar_prior = (ebar_sum[c]/ebar_cnt[c]) if not is_first else 0.0
                    e_cur, etil = gated_equity(g_i, conf[c], ebar_prior, is_first, g_ref, rho)
                    ebar_sum[c] += e_cur; ebar_cnt[c] += 1
                    states.append(st); et_list.append(etil); gaps.append(g_i); ecur_list.append(e_cur)
                global_state, a = dwfa_aggregate(states, et_list, p_w, alpha, kappa)
                del states; gc.collect()

                gm = build_model(device); gm.load_state_dict(global_state)
                vm = evaluate_dual(gm, pvl, device, 0.5)
                del gm; gc.collect(); torch.cuda.empty_cache()

                for ci, c in enumerate(CLIENT_LIST):
                    round_log.append({'method':tag,'seed':seed,'round':rnd+1,'client':c,
                        'val_eo_gap_local':gaps[ci],'e_cur':ecur_list[ci],
                        'ebar_lagged':(ebar_sum[c]-ecur_list[ci])/max(ebar_cnt[c]-1,1) if ebar_cnt[c]>1 else ecur_list[ci],
                        'etilde':et_list[ci],'r_conf':conf[c],'weight':float(a[ci]),
                        'fedavg_weight':p_w[ci],'pooled_val_auroc':vm['auroc'],
                        'pooled_val_eo_gap':vm['eo_gap'],'pooled_val_wg_auroc':vm['wg_auroc']})
                if vm['auroc'] > best_auroc:
                    best_auroc = vm['auroc']; best_gs = copy.deepcopy(global_state)
                print(f"  R{rnd+1:02d}/{rounds}: AUROC={vm['auroc']:.4f} EO={vm['eo_gap']:.4f} "
                      f"WG={vm['wg_auroc']:.4f} | a={[round(float(w),3) for w in a]}")

                if save_round_log:
                    pd.DataFrame(round_log).drop_duplicates(
                        ['method','seed','round','client'], keep='last').to_csv(rlog_path, index=False)
                if (rnd+1) % CKPT_EVERY == 0:
                    torch.save({'gs':global_state,'ebar_sum':ebar_sum,'ebar_cnt':ebar_cnt,
                                'next_round':rnd+1,'best_auroc':best_auroc,'best_gs':best_gs}, prog_ckpt)

            global_state = best_gs
            torch.save(global_state, final_ckpt)
            if os.path.exists(prog_ckpt): os.remove(prog_ckpt)
            print(f"  Best global saved (val AUROC={best_auroc:.4f})")
            del ctl, cvl, pvl; gc.collect(); torch.cuda.empty_cache()

        if do_test:
            gm = build_model(device); gm.load_state_dict(global_state)
            pvl_eval = make_loader('/content/pooled_val.csv', train=False)
            yj = youden_threshold(gm, pvl_eval, device)
            del pvl_eval; gc.collect()
            for c in CLIENT_LIST:
                te = make_loader(f'{CLIENTS}/client_{c}_test.csv', train=False)
                tm = evaluate_dual(gm, te, device, yj)
                tm.update({'method':tag,'client':c,'seed':seed})
                results.append(tm)
                print(f"  -> {c} TEST: AUROC={tm['auroc']:.4f} EO@.5={tm['eo_gap']:.4f} "
                      f"EO@YJ={tm['eo_gap_yj']:.4f} FPR@.5={tm['fpr_gap']:.4f} WG={tm['wg_auroc']:.4f}")
                del te; gc.collect()
            del gm; gc.collect(); torch.cuda.empty_cache()
            pd.DataFrame(results).to_csv(res_path, index=False)
            print(f"  Test results saved (seed {seed})")
    return results, round_log

print("run_dwfa driver defined")

run_dwfa driver defined


In [ ]:
# DWFA day 8,7
t0 = time.time()
dwfa_results, dwfa_log = run_dwfa(SEEDS, ROUNDS, G_REF, TAU_REF, tag='dwfa')
print(f"\nDWFA primary complete in {(time.time()-t0)/3600:.2f} hr")
print(f"  rows: {len(dwfa_results)} (expect {len(SEEDS)*len(CLIENT_LIST)})")

df = pd.read_csv(f'{RESULTS}/dwfa_all.csv')
def ms(a,s): return f"{a:.4f}\u00b1{s:.4f}"
def cstat(d,m,c): g=d[d['client']==c][m]; return g.mean(), g.std()
def pstat(d,m):   p=d.groupby('seed')[m].mean(); return p.mean(), p.std()

print("\nDWFA — Test results (mean \u00b1 std across seeds)")
print("-"*104)
print(f"  {'Clnt':<5}{'AUROC':>16}{'WG-AUROC':>16}{'EO@.5':>15}{'EO@YJ':>15}{'FPR@.5':>15}{'ECE':>14}")
print("-"*104)
for c in CLIENT_LIST:
    print(f"  {c:<5}{ms(*cstat(df,'auroc',c)):>16}{ms(*cstat(df,'wg_auroc',c)):>16}"
          f"{ms(*cstat(df,'eo_gap',c)):>15}{ms(*cstat(df,'eo_gap_yj',c)):>15}"
          f"{ms(*cstat(df,'fpr_gap',c)):>15}{ms(*cstat(df,'ece',c)):>14}")
print("-"*104)
print(f"  {'Pool':<5}{ms(*pstat(df,'auroc')):>16}{ms(*pstat(df,'wg_auroc')):>16}"
      f"{ms(*pstat(df,'eo_gap')):>15}{ms(*pstat(df,'eo_gap_yj')):>15}{ms(*pstat(df,'fpr_gap')):>15}")

cC = df[df['client']=='C']['eo_gap']
print(f"\nPRIMARY ENDPOINT  Client C EO-gap: mean={cC.mean():.4f}  std={cC.std():.4f}  "
      f"(n={cC.shape[0]} seeds)  [stability=std is the primary endpoint; mean is co-primary]")


  DWFA [dwfa] | SEED 42 | G_ref=0.05 tau_ref=200
  Seed locked: 42


  R01/30: AUROC=0.9480 EO=0.0297 WG=0.9462 | a=[0.525, 0.158, 0.137, 0.18]


  R02/30: AUROC=0.9521 EO=0.0235 WG=0.9498 | a=[0.391, 0.188, 0.162, 0.259]


  R03/30: AUROC=0.9499 EO=0.0265 WG=0.9460 | a=[0.582, 0.148, 0.137, 0.133]


  R04/30: AUROC=0.9492 EO=0.0249 WG=0.9463 | a=[0.505, 0.146, 0.149, 0.2]


...

  R26/30: AUROC=0.9488 EO=0.0404 WG=0.9445 | a=[0.388, 0.224, 0.214, 0.174]


  R27/30: AUROC=0.9487 EO=0.0201 WG=0.9442 | a=[0.558, 0.146, 0.145, 0.151]


  R28/30: AUROC=0.9493 EO=0.0306 WG=0.9460 | a=[0.486, 0.164, 0.162, 0.188]


  R29/30: AUROC=0.9483 EO=0.0188 WG=0.9442 | a=[0.517, 0.16, 0.168, 0.155]


  R30/30: AUROC=0.9491 EO=0.0368 WG=0.9447 | a=[0.531, 0.205, 0.151, 0.113]
  Best global saved (val AUROC=0.9532)
  -> A TEST: AUROC=0.9497 EO@.5=0.0294 EO@YJ=0.0073 FPR@.5=0.0091 WG=0.9484
  -> B TEST: AUROC=0.9486 EO@.5=0.0231 EO@YJ=0.0031 FPR@.5=0.0370 WG=0.9407
  -> C TEST: AUROC=0.9396 EO@.5=0.0155 EO@YJ=0.0060 FPR@.5=0.0113 WG=0.9323
  -> D TEST: AUROC=0.9639 EO@.5=0.0517 EO@YJ=0.0569 FPR@.5=0.0441 WG=0.9545
  Test results saved (seed 1010)

DWFA primary complete in 8.44 hr
  rows: 20 (expect 20)

DWFA — Test results (mean ± std across seeds)
--------------------------------------------------------------------------------------------------------
  Clnt            AUROC        WG-AUROC          EO@.5          EO@YJ         FPR@.5           ECE
--------------------------------------------------------------------------------------------------------
  A       0.9502±0.0013   0.9472±0.0019  0.0199±0.0110  0.0049±0.0023  0.0141±0.0054 0.0476±0.0090
  B       0.9484±0.0006   0.9408±0.0

In [ ]:
# DWFA day 8,8
SWEEP_SEEDS  = [42, 123, 456]
SWEEP_ROUNDS = 20
SWEEP_GREF   = [0.02, 0.05, 0.10]
SWEEP_TAUREF = [100, 200, 300, 433]

sweep_rows = []
for g_ref in SWEEP_GREF:
    for tau_ref in SWEEP_TAUREF:
        cfg = f"sweep_g{g_ref}_t{tau_ref}"
        print(f"\n########## SWEEP {cfg} ##########")
        res, _ = run_dwfa(SWEEP_SEEDS, SWEEP_ROUNDS, g_ref, tau_ref,
                          tag=cfg, do_test=True, save_round_log=False)
        d = pd.read_csv(f'{RESULTS}/{cfg}_all.csv')
        cC = d[d['client']=='C']['eo_gap']
        pooled = d.groupby('seed')['eo_gap'].mean()
        sweep_rows.append({'G_ref':g_ref,'tau_ref':tau_ref,
            'clientC_eo_mean':cC.mean(),'clientC_eo_std':cC.std(),
            'pooled_eo_mean':pooled.mean(),
            'pooled_auroc':d.groupby('seed')['auroc'].mean().mean()})
        pd.DataFrame(sweep_rows).to_csv(f'{RESULTS}/dwfa_sweep_summary.csv', index=False)

sw = pd.DataFrame(sweep_rows)
print("\nDWFA SENSITIVITY SWEEP (Client C EO-gap mean/std, pooled EO, pooled AUROC)")
print(sw.to_string(index=False))
print(f"\nSaved -> {RESULTS}/dwfa_sweep_summary.csv")
print("Pick the primary (G_ref, tau_ref) from this frontier: lowest Client C "
      "EO-gap std that does not raise Client C EO-gap mean above FedAvg/q-FedAvg.")


########## SWEEP sweep_g0.02_t100 ##########

  DWFA [sweep_g0.02_t100] | SEED 42 | G_ref=0.02 tau_ref=100
  Seed locked: 42
Downloading: "https://download.pytorch.org/models/densenet121-a639ec97.pth" to /root/.cache/torch/hub/checkpoints/densenet121-a639ec97.pth


100%|██████████| 30.8M/30.8M [00:00<00:00, 186MB/s]


  R01/20: AUROC=0.9478 EO=0.0214 WG=0.9460 | a=[0.418, 0.223, 0.192, 0.167]


  R02/20: AUROC=0.9522 EO=0.0302 WG=0.9497 | a=[0.418, 0.223, 0.192, 0.167]


  R03/20: AUROC=0.9521 EO=0.0277 WG=0.9493 | a=[0.58, 0.161, 0.139, 0.12]


...

  R17/20: AUROC=0.9494 EO=0.0269 WG=0.9460 | a=[0.401, 0.239, 0.197, 0.163]


  R18/20: AUROC=0.9497 EO=0.0373 WG=0.9464 | a=[0.402, 0.238, 0.197, 0.164]


  R19/20: AUROC=0.9513 EO=0.0332 WG=0.9485 | a=[0.347, 0.204, 0.215, 0.234]


  R20/20: AUROC=0.9507 EO=0.0308 WG=0.9478 | a=[0.405, 0.249, 0.154, 0.192]
  Best global saved (val AUROC=0.9539)
  -> A TEST: AUROC=0.9503 EO@.5=0.0040 EO@YJ=0.0059 FPR@.5=0.0065 WG=0.9475
  -> B TEST: AUROC=0.9483 EO@.5=0.0428 EO@YJ=0.0076 FPR@.5=0.0332 WG=0.9417
  -> C TEST: AUROC=0.9389 EO@.5=0.0070 EO@YJ=0.0192 FPR@.5=0.0093 WG=0.9343
  -> D TEST: AUROC=0.9636 EO@.5=0.0214 EO@YJ=0.0172 FPR@.5=0.0294 WG=0.9556
  Test results saved (seed 456)

########## SWEEP sweep_g0.02_t300 ##########

  DWFA [sweep_g0.02_t300] | SEED 42 | G_ref=0.02 tau_ref=300
  Seed locked: 42


KeyboardInterrupt: 

In [ ]:
# DWFA day 8,9
def within_run_stability(tag='dwfa', metric='pooled_val_eo_gap', last_n=15, results_dir=RESULTS):
    """For each seed, std of `metric` over the final `last_n` rounds
    (post-convergence window) — a high-N complement to the across-seed std."""
    log = pd.read_csv(f'{results_dir}/{tag}_round_log.csv')
    per_round = log.drop_duplicates(['seed','round'])[['seed','round',metric]]
    rows = []
    for seed, g in per_round.groupby('seed'):
        g = g.sort_values('round')
        tail = g.tail(last_n)[metric].values
        rows.append({'seed': seed, 'n_rounds_used': len(tail),
                     'within_run_mean': float(np.mean(tail)),
                     'within_run_std':  float(np.std(tail, ddof=1)) if len(tail) > 1 else float('nan')})
    out = pd.DataFrame(rows)
    out.to_csv(f'{results_dir}/{tag}_within_run_stability.csv', index=False)
    print(f"[{tag}] within-run stability of {metric} (last {last_n} rounds/seed)")
    print(out.to_string(index=False))
    print(f"  Pooled within-run std (mean across seeds): {out['within_run_std'].mean():.5f}")
    print(f"  Saved -> {results_dir}/{tag}_within_run_stability.csv")
    return out

print("Computing within-run stability: DWFA vs FedAvg vs q-FedAvg")
for m in ['dwfa', 'fedavg', 'qfedavg']:
    path = f'{RESULTS}/{m}_round_log.csv'
    if os.path.exists(path):
        cols = pd.read_csv(path, nrows=1).columns
        metric_col = 'val_eo_gap' if 'val_eo_gap' in cols else 'pooled_val_eo_gap'
        within_run_stability(tag=m, metric=metric_col)
    else:
        print(f"  [{m}] round log not found at {path} — skipping (run that method first)")

Computing within-run stability: DWFA vs FedAvg vs q-FedAvg
[dwfa] within-run stability of pooled_val_eo_gap (last 15 rounds/seed)
 seed  n_rounds_used  within_run_mean  within_run_std
   42             15         0.035188        0.008367
  123             15         0.034947        0.009515
  456             15         0.029593        0.010051
  789             15         0.037396        0.008205
 1010             15         0.032017        0.007885
  Pooled within-run std (mean across seeds): 0.00880
  Saved -> /content/drive/MyDrive/FairFedCXR/results/dwfa_within_run_stability.csv
[fedavg] within-run stability of val_eo_gap (last 15 rounds/seed)
 seed  n_rounds_used  within_run_mean  within_run_std
   42             15         0.032195        0.009570
  123             15         0.030499        0.007943
  456             15         0.026977        0.006494
  Pooled within-run std (mean across seeds): 0.00800
  Saved -> /content/drive/MyDrive/FairFedCXR/results/fedavg_within_run_stab

In [ ]:
# DWFA day 8,10
def bootstrap_eo_gap_noise_floor(ckpt_path, client='C', n_boot=1000, threshold=0.5, seed_for_boot=0):
    """Bootstrap Client C's test set (with replacement) for ONE fixed trained
    model to estimate how much EO-gap varies from evaluation noise ALONE."""
    device = 'cuda' if torch.cuda.is_available() else 'cpu'
    gm = build_model(device)
    gm.load_state_dict(torch.load(ckpt_path, map_location='cpu'))
    gm.eval()
    te_loader = make_loader(f'{CLIENTS}/client_{client}_test.csv', train=False)

    P, L, S = [], [], []
    with torch.no_grad():
        for imgs, labels, sex in te_loader:
            imgs = imgs.to(device, non_blocking=True)
            with autocast('cuda'):
                logits = gm(imgs).squeeze()
            P.append(torch.sigmoid(logits.float()).cpu().numpy())
            L.append(labels.numpy()); S.append(sex.numpy())
    probs, labels, sex = map(np.concatenate, (P, L, S))
    del gm, te_loader; gc.collect(); torch.cuda.empty_cache()

    rng = np.random.default_rng(seed_for_boot)
    n = len(labels)
    boot_gaps = np.empty(n_boot)
    for b in range(n_boot):
        idx = rng.integers(0, n, n)
        gap, _, _ = compute_eo_gap(probs[idx], labels[idx], sex[idx], threshold)
        boot_gaps[b] = gap
    return {'boot_mean': float(boot_gaps.mean()), 'boot_std': float(boot_gaps.std()),
            'boot_ci_lo': float(np.percentile(boot_gaps, 2.5)),
            'boot_ci_hi': float(np.percentile(boot_gaps, 97.5)), 'n_test': n}

def variance_decomposition(tag='dwfa', client='C'):
    """Compares the ACROSS-SEED std of Client C's EO-gap against the
    IRREDUCIBLE evaluation-noise floor (bootstrap on one fixed seed's
    model). If the floor already explains most of the across-seed spread,
    the 'instability' is largely measurement noise, not model instability."""
    df = pd.read_csv(f'{RESULTS}/{tag}_all.csv')
    cC = df[df['client'] == client]['eo_gap']
    across_seed_std = float(cC.std())
    seeds_available = df[df['client'] == client]['seed'].tolist()
    if not seeds_available:
        print(f"  [{tag}] no rows for client {client} — skipping"); return None
    ref_seed = seeds_available[0]
    ckpt_path = f"{CKPTS}/{tag}_seed{ref_seed}.pt"
    if not os.path.exists(ckpt_path):
        print(f"  [{tag}] checkpoint {ckpt_path} not found — skipping bootstrap"); return None

    boot = bootstrap_eo_gap_noise_floor(ckpt_path, client=client)
    print(f"\n[{tag}] Client {client} variance decomposition")
    print(f"  Across-seed EO-gap:           mean={cC.mean():.4f}  std={across_seed_std:.4f}  (n={len(cC)} seeds)")
    print(f"  Eval-noise floor (1 model, bootstrap n=1000 on frozen test, seed={ref_seed}):")
    print(f"    mean={boot['boot_mean']:.4f}  std={boot['boot_std']:.4f}  "
          f"95% CI=[{boot['boot_ci_lo']:.4f}, {boot['boot_ci_hi']:.4f}]  (n_test={boot['n_test']})")
    ratio = boot['boot_std'] / across_seed_std if across_seed_std > 0 else float('nan')
    print(f"  Eval-noise-floor / across-seed-std ratio: {ratio:.2f}")
    if ratio > 0.7:
        print("  -> Most of the across-seed spread is explained by evaluation noise alone.")
    else:
        print("  -> Across-seed spread clearly EXCEEDS the evaluation-noise floor: "
              "genuine model-induced instability beyond measurement noise.")
    return {'tag': tag, 'client': client, 'across_seed_mean': cC.mean(), 'across_seed_std': across_seed_std,
            **{f'boot_{k}': v for k, v in boot.items()}, 'ref_seed': ref_seed, 'floor_to_spread_ratio': ratio}

decomp_rows = []
for m in ['fedavg', 'qfedavg', 'dwfa']:
    if os.path.exists(f'{RESULTS}/{m}_all.csv'):
        r = variance_decomposition(tag=m, client='C')
        if r: decomp_rows.append(r)
if decomp_rows:
    pd.DataFrame(decomp_rows).to_csv(f'{RESULTS}/clientC_variance_decomposition.csv', index=False)
    print(f"\nSaved -> {RESULTS}/clientC_variance_decomposition.csv")


[fedavg] Client C variance decomposition
  Across-seed EO-gap:           mean=0.0160  std=0.0152  (n=3 seeds)
  Eval-noise floor (1 model, bootstrap n=1000 on frozen test, seed=42):
    mean=0.0438  std=0.0325  95% CI=[0.0019, 0.1208]  (n_test=1035)
  Eval-noise-floor / across-seed-std ratio: 2.13
  -> Most of the across-seed spread is explained by evaluation noise alone.

[qfedavg] Client C variance decomposition
  Across-seed EO-gap:           mean=0.0082  std=0.0070  (n=3 seeds)
  Eval-noise floor (1 model, bootstrap n=1000 on frozen test, seed=42):
    mean=0.0321  std=0.0246  95% CI=[0.0012, 0.0887]  (n_test=1035)
  Eval-noise-floor / across-seed-std ratio: 3.49
  -> Most of the across-seed spread is explained by evaluation noise alone.

[dwfa] Client C variance decomposition
  Across-seed EO-gap:           mean=0.0130  std=0.0048  (n=5 seeds)
  Eval-noise floor (1 model, bootstrap n=1000 on frozen test, seed=42):
    mean=0.0315  std=0.0229  95% CI=[0.0012, 0.0852]  (n_test=1035

In [ ]:
# DWFA day 8,11
from itertools import permutations
_ALL_DERANGEMENTS_4 = [p for p in permutations(range(4)) if all(p[i] != i for i in range(4))]
assert len(_ALL_DERANGEMENTS_4) == 9, "expected exactly 9 derangements of 4 elements"

def run_dwfa_placebo(seeds, rounds, g_ref, tau_ref, derangement, mode='scrambled',
                      alpha=ALPHA_DWFA, kappa=KAPPA_DWFA, rho=RHO_DWFA, tag='dwfa_placebo'):
    """mode='scrambled': ONE FIXED derangement applied every round — client i
    receives client derangement[i]'s etilde. Same correction magnitudes,
    wrong targets.
    mode='noise': etilde_i^t replaced with matched-magnitude noise drawn
    from the SAME per-client etilde distribution observed under the real
    run (requires {tag}_round_log.csv from a completed DWFA run)."""
    device = 'cuda' if torch.cuda.is_available() else 'cpu'
    conf, _ = compute_static_confidence(tau_ref, rho)

    noise_pool = None
    if mode == 'noise':
        real_tag = tag.replace('_placebo_noise', '').replace('_placebo', '')
        real_log_path = f'{RESULTS}/{real_tag}_round_log.csv'
        assert os.path.exists(real_log_path), f"need {real_log_path} for noise-control matched magnitudes"
        rl = pd.read_csv(real_log_path)
        noise_pool = {c: rl[rl['client'] == c]['etilde'].values for c in CLIENT_LIST}

    pooled_val = pd.concat([pd.read_csv(f'{CLIENTS}/client_{c}_val.csv') for c in CLIENT_LIST], ignore_index=True)
    pooled_val.to_csv('/content/pooled_val.csv', index=False)

    res_path = f'{RESULTS}/{tag}_all.csv'
    results = pd.read_csv(res_path).to_dict('records') if os.path.exists(res_path) else []
    done_seeds = {r['seed'] for r in results}

    for seed in seeds:
        if seed in done_seeds:
            print(f"  [{tag}] seed {seed} already done — skipping"); continue
        print(f"\n{'='*58}\n  PLACEBO[{mode}] {tag} | SEED {seed} | derangement={derangement}\n{'='*58}")
        set_seed(seed)
        rng_noise = np.random.default_rng(seed)
        ctl = {c: make_loader(f'{CLIENTS}/client_{c}_train.csv', train=True) for c in CLIENT_LIST}
        cvl = {c: make_loader(f'{CLIENTS}/client_{c}_val.csv',   train=False) for c in CLIENT_LIST}
        cs  = {c: len(ctl[c].dataset) for c in CLIENT_LIST}
        N   = sum(cs.values()); p_w = [cs[c]/N for c in CLIENT_LIST]
        pvl = make_loader('/content/pooled_val.csv', train=False)

        gm = build_model(device); global_state = {k: v.cpu().clone() for k, v in gm.state_dict().items()}
        del gm; torch.cuda.empty_cache()
        ebar_sum = {c: 0.0 for c in CLIENT_LIST}; ebar_cnt = {c: 0 for c in CLIENT_LIST}
        best_auroc, best_gs = -1.0, None

        for rnd in range(rounds):
            states, et_real = [], []
            for c in CLIENT_LIST:
                st, g_i = local_train_dwfa(global_state, ctl[c], cvl[c], device, tag=f"R{rnd+1}-{c}")
                is_first = (ebar_cnt[c] == 0)
                ebar_prior = (ebar_sum[c]/ebar_cnt[c]) if not is_first else 0.0
                e_cur, etil = gated_equity(g_i, conf[c], ebar_prior, is_first, g_ref, rho)
                ebar_sum[c] += e_cur; ebar_cnt[c] += 1
                states.append(st); et_real.append(etil)

            if mode == 'scrambled':
                et_used = [et_real[derangement[i]] for i in range(4)]
            elif mode == 'noise':
                et_used = [float(np.clip(rng_noise.choice(noise_pool[c]), rho, 1.0)) for c in CLIENT_LIST]
            else:
                raise ValueError(mode)

            global_state, a = dwfa_aggregate(states, et_used, p_w, alpha, kappa)
            del states; gc.collect()
            gm = build_model(device); gm.load_state_dict(global_state)
            vm = evaluate_dual(gm, pvl, device, 0.5)
            del gm; gc.collect(); torch.cuda.empty_cache()
            if vm['auroc'] > best_auroc:
                best_auroc = vm['auroc']; best_gs = copy.deepcopy(global_state)
            print(f"  R{rnd+1:02d}/{rounds}: AUROC={vm['auroc']:.4f} EO={vm['eo_gap']:.4f} "
                  f"| a={[round(float(w),3) for w in a]}")

        gm = build_model(device); gm.load_state_dict(best_gs)
        for c in CLIENT_LIST:
            te = make_loader(f'{CLIENTS}/client_{c}_test.csv', train=False)
            tm = evaluate_dual(gm, te, device, 0.5)
            tm.update({'method': tag, 'client': c, 'seed': seed})
            results.append(tm)
            del te; gc.collect()
        del gm, ctl, cvl, pvl; gc.collect(); torch.cuda.empty_cache()
        pd.DataFrame(results).to_csv(res_path, index=False)
        print(f"  [{tag}] seed {seed} done, saved")
    return results

PLACEBO_SEEDS  = [42, 123, 456]
PLACEBO_ROUNDS = 20
fixed_derangement = _ALL_DERANGEMENTS_4[0]
print(f"Fixed derangement (A,B,C,D) -> receives etilde from clients: {fixed_derangement}")

placebo_runs = {}
for g_ref, label in [(0.05, 'primary'), (0.02, 'strongsignal')]:
    tag_s = f'dwfa_placebo_scrambled_g{g_ref}'
    print(f"\n###### SCRAMBLED placebo @ G_ref={g_ref} ({label}) ######")
    run_dwfa_placebo(PLACEBO_SEEDS, PLACEBO_ROUNDS, g_ref, TAU_REF, fixed_derangement,
                     mode='scrambled', tag=tag_s)
    placebo_runs[f'scrambled_{label}'] = tag_s

if os.path.exists(f'{RESULTS}/dwfa_round_log.csv'):
    tag_n = 'dwfa_placebo_noise'
    print(f"\n###### MATCHED-NOISE placebo (magnitudes from real dwfa_round_log) ######")
    run_dwfa_placebo(PLACEBO_SEEDS, PLACEBO_ROUNDS, G_REF, TAU_REF, fixed_derangement,
                     mode='noise', tag=tag_n)
    placebo_runs['noise'] = tag_n
else:
    print("\n[noise control] dwfa_round_log.csv not found — run the primary DWFA result first.")

print(f"\n{'='*70}\nPLACEBO ORDERING CHECK (Client C EO-gap, mean across seeds)\n{'='*70}")
order_rows = []
for label, path_tag in [('FedAvg', 'fedavg'), ('q-FedAvg', 'qfedavg'),
                         *[(k, v) for k, v in placebo_runs.items()], ('Real-DWFA', 'dwfa')]:
    p = f'{RESULTS}/{path_tag}_all.csv'
    if os.path.exists(p):
        d = pd.read_csv(p); cC = d[d['client'] == 'C']['eo_gap']
        order_rows.append({'method': label, 'clientC_eo_mean': cC.mean(), 'clientC_eo_std': cC.std()})
order_df = pd.DataFrame(order_rows).sort_values('clientC_eo_mean')
print(order_df.to_string(index=False))
order_df.to_csv(f'{RESULTS}/placebo_ordering_check.csv', index=False)
print(f"\nExpected: FedAvg highest, scrambled/noise ~ q-FedAvg, Real-DWFA lowest.")
print(f"Saved -> {RESULTS}/placebo_ordering_check.csv")

Fixed derangement (A,B,C,D) -> receives etilde from clients: (1, 0, 3, 2)

###### SCRAMBLED placebo @ G_ref=0.05 (primary) ######

  PLACEBO[scrambled] dwfa_placebo_scrambled_g0.05 | SEED 42 | derangement=(1, 0, 3, 2)
  Seed locked: 42


  R01/20: AUROC=0.9478 EO=0.0347 | a=[0.329, 0.31, 0.229, 0.132]


  R02/20: AUROC=0.9522 EO=0.0324 | a=[0.386, 0.206, 0.202, 0.206]


  R03/20: AUROC=0.9523 EO=0.0218 | a=[0.301, 0.338, 0.219, 0.142]


  R04/20: AUROC=0.9516 EO=0.0250 | a=[0.334, 0.33, 0.171, 0.166]


...

  R03/20: AUROC=0.9505 EO=0.0178 | a=[0.538, 0.152, 0.148, 0.162]


  R04/20: AUROC=0.9499 EO=0.0158 | a=[0.501, 0.167, 0.171, 0.161]


  R05/20: AUROC=0.9512 EO=0.0407 | a=[0.39, 0.185, 0.179, 0.246]


R6-B ep2:  31%|███       | 54/175 [00:06<00:12,  9.40it/s]Exception ignored in: <function _MultiProcessingDataLoaderIter.__del__ at 0x7b191e318720>
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1709, in __del__
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1673, in _shutdown_workers
    w.join(timeout=_utils.MP_STATUS_CHECK_INTERVAL)
  File "/usr/lib/python3.12/multiprocessing/process.py", line 149, in join
    res = self._popen.wait(timeout)
          ^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/lib/python3.12/multiprocessing/popen_fork.py", line 40, in wait
    if not wait([self.sentinel], timeout):
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/lib/python3.12/multiprocessing/connection.py", line 1136, in wait
    ready = selector.select(timeout)
            ^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/lib/python3.12/selectors.py", line 415, in select
    fd_event_list = self

KeyboardInterrupt: 

In [ ]:
# ============================================================================
# STANDALONE — Placebo ordering check (run after interrupting Cell 11)
# Only needs RESULTS, pandas — both already in memory from earlier cells.
# Reads whichever *_all.csv files already exist on disk; skips the rest.
# ============================================================================
import pandas as pd, os

print(f"\n{'='*70}\nPLACEBO ORDERING CHECK (Client C EO-gap, mean across seeds)\n{'='*70}")

candidates = [
    ('FedAvg',              'fedavg'),
    ('q-FedAvg',            'qfedavg'),
    ('scrambled_primary',   'dwfa_placebo_scrambled_g0.05'),
    ('scrambled_strongsig', 'dwfa_placebo_scrambled_g0.02'),
    ('noise',               'dwfa_placebo_noise'),          # will be skipped — incomplete
    ('Real-DWFA',           'dwfa'),
]

order_rows = []
for label, path_tag in candidates:
    p = f'{RESULTS}/{path_tag}_all.csv'
    if os.path.exists(p):
        d = pd.read_csv(p)
        cC = d[d['client'] == 'C']['eo_gap']
        if len(cC) == 0:
            print(f"  [{label}] file exists but no Client C rows yet — skipping")
            continue
        order_rows.append({'method': label, 'clientC_eo_mean': cC.mean(),
                            'clientC_eo_std': cC.std(), 'n_seeds': len(cC)})
    else:
        print(f"  [{label}] {p} not found — skipping (expected if interrupted)")

order_df = pd.DataFrame(order_rows).sort_values('clientC_eo_mean')
print()
print(order_df.to_string(index=False))
order_df.to_csv(f'{RESULTS}/placebo_ordering_check.csv', index=False)
print(f"\nExpected pattern: FedAvg highest, scrambled ~ q-FedAvg, Real-DWFA lowest.")
print(f"Saved -> {RESULTS}/placebo_ordering_check.csv")


PLACEBO ORDERING CHECK (Client C EO-gap, mean across seeds)
  [noise] /content/drive/MyDrive/FairFedCXR/results/dwfa_placebo_noise_all.csv not found — skipping (expected if interrupted)

             method  clientC_eo_mean  clientC_eo_std  n_seeds
scrambled_strongsig         0.006999        0.008183        3
           q-FedAvg         0.008215        0.007039        3
  scrambled_primary         0.012304        0.007689        3
          Real-DWFA         0.013048        0.004849        5
             FedAvg         0.016050        0.015233        3

Expected pattern: FedAvg highest, scrambled ~ q-FedAvg, Real-DWFA lowest.
Saved -> /content/drive/MyDrive/FairFedCXR/results/placebo_ordering_check.csv


In [ ]:
# DWFA day 8,12
from scipy.stats import pearsonr, spearmanr

def round_level_weight_gap_correlation(round_log_path, weight_col, gap_col, method_name):
    """Round-to-round DELTAS of weight and local gap, pooled across
    clients x seeds x rounds — real statistical power, unlike a 4-point
    client-level correlation."""
    log = pd.read_csv(round_log_path)
    all_dw, all_dg = [], []
    for (seed, client), g in log.groupby(['seed', 'client']):
        g = g.sort_values('round')
        dw = np.diff(g[weight_col].values)
        dg = np.diff(g[gap_col].values)
        all_dw.append(dw); all_dg.append(dg)
    dw = np.concatenate(all_dw); dg = np.concatenate(all_dg)
    pear_r, pear_p = pearsonr(dw, dg)
    spear_r, spear_p = spearmanr(dw, dg)
    print(f"\n[{method_name}] round-level correlation of \u0394weight vs \u0394gap  (N={len(dw)} round-transitions)")
    print(f"  Pearson  r={pear_r:+.4f}  p={pear_p:.2e}")
    print(f"  Spearman r={spear_r:+.4f}  p={spear_p:.2e}")
    interp = ("weight responds to demographic signal (consistent with a CAUSAL mechanism)"
              if (pear_p < 0.05 and pear_r < -0.1)
              else "weight largely unrelated to demographic signal (consistent with INCIDENTAL reweighting)")
    print(f"  -> {interp}")
    return {'method': method_name, 'n_transitions': len(dw),
            'pearson_r': pear_r, 'pearson_p': pear_p,
            'spearman_r': spear_r, 'spearman_p': spear_p}

corr_rows = []
qfed_log_path = f'{RESULTS}/qfedavg_round_log.csv'
if os.path.exists(qfed_log_path):
    corr_rows.append(round_level_weight_gap_correlation(
        qfed_log_path, weight_col='weight', gap_col='val_eo_gap', method_name='q-FedAvg'))
else:
    print(f"q-FedAvg round log not found at {qfed_log_path}")

dwfa_log_path = f'{RESULTS}/dwfa_round_log.csv'
if os.path.exists(dwfa_log_path):
    corr_rows.append(round_level_weight_gap_correlation(
        dwfa_log_path, weight_col='weight', gap_col='val_eo_gap_local', method_name='DWFA'))
else:
    print(f"DWFA round log not found at {dwfa_log_path}")

if corr_rows:
    pd.DataFrame(corr_rows).to_csv(f'{RESULTS}/incidental_vs_causal_correlation.csv', index=False)
    print(f"\nSaved -> {RESULTS}/incidental_vs_causal_correlation.csv")
    print("Expected: q-FedAvg shows weak/no correlation with the GAP specifically (its")
    print("weight tracks LOSS, not fairness); DWFA shows a clear negative correlation")
    print("(weight tracks the gap directly, by construction) -- causal-mechanism evidence.")


[q-FedAvg] round-level correlation of Δweight vs Δgap  (N=348 round-transitions)
  Pearson  r=+0.0000  p=1.00e+00
  Spearman r=-0.0148  p=7.83e-01
  -> weight largely unrelated to demographic signal (consistent with INCIDENTAL reweighting)

[DWFA] round-level correlation of Δweight vs Δgap  (N=580 round-transitions)
  Pearson  r=-0.5473  p=1.25e-46
  Spearman r=-0.5841  p=2.33e-54
  -> weight responds to demographic signal (consistent with a CAUSAL mechanism)

Saved -> /content/drive/MyDrive/FairFedCXR/results/incidental_vs_causal_correlation.csv
Expected: q-FedAvg shows weak/no correlation with the GAP specifically (its
weight tracks LOSS, not fairness); DWFA shows a clear negative correlation
(weight tracks the gap directly, by construction) -- causal-mechanism evidence.


In [ ]:
# external validation
#NIH CELL 1 — Download + unzip NIH ChestX-ray14 (224x224 resized mirror)
# Kaggle API, no credentialing/DUA. Stages images on LOCAL SSD (fast), not Drive.
# Paste this AFTER all your DWFA cells. Run once per session.
# ============================================================================
import os

# --- 1a. Kaggle credentials. Upload your kaggle.json (Account -> Create New API Token)
#     to Drive once at: /content/drive/MyDrive/FairFedCXR/kaggle.json ---
os.makedirs('/root/.kaggle', exist_ok=True)
get_ipython().system('cp /content/drive/MyDrive/FairFedCXR/kaggle.json /root/.kaggle/kaggle.json')
get_ipython().system('chmod 600 /root/.kaggle/kaggle.json')
get_ipython().system('pip install -q kaggle')

NIH_DIR = '/content/nih'          # local SSD, fast I/O, wiped on disconnect (fine)
os.makedirs(NIH_DIR, exist_ok=True)

# --- 1b. Download the 224x224 resized mirror (small, fast; full-res is ~45GB) ---
# This mirror bundles all images + the official Data_Entry_2017 metadata.
if not os.path.exists(f'{NIH_DIR}/images-224'):
    print("Downloading NIH ChestX-ray14 (224x224 mirror)... a few minutes")
    get_ipython().system('kaggle datasets download -d khanfashee/nih-chest-x-ray-14-224x224-resized -p /content/nih')
    print("Unzipping to local SSD...")
    get_ipython().system('unzip -q /content/nih/nih-chest-x-ray-14-224x224-resized.zip -d /content/nih')
    print("Done")
else:
    print("NIH images already staged on local SSD")
    # --- 1c. Locate the metadata CSV and the image folder (mirror layout varies slightly) ---
import glob
meta_hits = glob.glob(f'{NIH_DIR}/**/Data_Entry_2017*.csv', recursive=True)
img_dirs  = glob.glob(f'{NIH_DIR}/**/images*', recursive=True)
print("metadata csv candidates:", meta_hits)
print("image dir candidates:", [d for d in img_dirs if os.path.isdir(d)][:5])
# Set these two explicitly after checking the printout above:
NIH_META = meta_hits[0] if meta_hits else f'{NIH_DIR}/Data_Entry_2017.csv'
# the 224 mirror typically stores images flat under a single folder:
_cand = [d for d in img_dirs if os.path.isdir(d) and len(os.listdir(d)) > 1000]
NIH_IMG = _cand[0] if _cand else f'{NIH_DIR}/images-224'
print(f"\nUsing NIH_META = {NIH_META}")
print(f"Using NIH_IMG  = {NIH_IMG}")
print("If either is wrong, set them by hand from the candidate lists above before CELL 2.")


cp: cannot stat '/content/drive/MyDrive/FairFedCXR/kaggle.json': No such file or directory
chmod: cannot access '/root/.kaggle/kaggle.json': No such file or directory
Dataset URL: https://www.kaggle.com/datasets/khanfashee/nih-chest-x-ray-14-224x224-resized
License(s): CC0-1.0
100% 2.30G/2.30G [00:24<00:00, 99.7MB/s]

Unzipping to local SSD...
Done
metadata csv candidates: ['/content/nih/Data_Entry_2017.csv']
image dir candidates: ['/content/nih/images-224', '/content/nih/images-224/images-224']

Using NIH_META = /content/nih/Data_Entry_2017.csv
Using NIH_IMG  = /content/nih/images-224/images-224
If either is wrong, set them by hand from the candidate lists above before CELL 2.


In [ ]:
#==========================================================================
# NIH CELL 2 — Build the effusion cohort CSV in YOUR schema (CPU only, ~0 units)
# Maps NIH's 'Finding Labels' + 'Patient Gender' -> Path/sex_encoded/Sex/label
# ============================================================================
import pandas as pd, numpy as np

meta = pd.read_csv(NIH_META)
# NIH columns: 'Image Index','Finding Labels','Patient ID','Patient Age','Patient Gender','View Position', ...
meta = meta.rename(columns={c: c.strip() for c in meta.columns})

# --- label: pleural effusion is encoded as the token 'Effusion' inside a
#     pipe-delimited 'Finding Labels' string (e.g. "Effusion|Infiltration") ---
meta['label'] = meta['Finding Labels'].str.contains('Effusion', case=False, na=False).astype(int)

# --- sex: 'M'/'F' -> your schema (Male=1, Female=0); drop unknowns ---
meta = meta[meta['Patient Gender'].isin(['M', 'F'])].copy()
meta['sex_encoded'] = (meta['Patient Gender'] == 'M').astype(int)
meta['Sex'] = meta['Patient Gender'].map({'M': 'Male', 'F': 'Female'})

# --- frontal only (NIH is already all frontal PA/AP; keep the column if present) ---
if 'View Position' in meta.columns:
    meta = meta[meta['View Position'].isin(['PA', 'AP'])].copy()

# --- Path: point at the staged image folder; CheXpertDataset isn't used here,
#     NihDataset (CELL 3) reads NIH_IMG + 'Image Index' directly ---
meta['Path'] = meta['Image Index']

cohort = meta[['Path', 'sex_encoded', 'Sex', 'label', 'Patient ID']].reset_index(drop=True)

# --- de-duplicate to one row per image (NIH metadata is already 1 row/image) ---
cohort = cohort.drop_duplicates('Path').reset_index(drop=True)

NIH_COHORT_CSV = f'{RESULTS}/nih_effusion_cohort.csv'
cohort.to_csv(NIH_COHORT_CSV, index=False)

mP = ((cohort.sex_encoded == 1) & (cohort.label == 1)).sum()
fP = ((cohort.sex_encoded == 0) & (cohort.label == 1)).sum()
print(f"NIH effusion cohort: {len(cohort)} images")
print(f"  Male:   {(cohort.sex_encoded==1).sum():>6}  (effusion-positive: {mP})")
print(f"  Female: {(cohort.sex_encoded==0).sum():>6}  (effusion-positive: {fP})")
print(f"  Overall effusion prevalence: {cohort.label.mean():.3f}")
print(f"  Saved -> {NIH_COHORT_CSV}")
# ESTIMABILITY CHECK: compare min(mP,fP) to CheXpert Client C's 87.
se_gap = np.sqrt(0.85*0.15/max(min(mP,fP),1)) * np.sqrt(2)
print(f"  ~SE(EO-gap) at this subgroup size: {se_gap:.4f}  "
      f"(CheXpert C was 0.0434; lower is better, <0.013 makes methods resolvable)")

NIH effusion cohort: 112120 images
  Male:    63340  (effusion-positive: 7427)
  Female:  48780  (effusion-positive: 5880)
  Overall effusion prevalence: 0.119
  Saved -> /content/drive/MyDrive/FairFedCXR/results/nih_effusion_cohort.csv
  ~SE(EO-gap) at this subgroup size: 0.0066  (CheXpert C was 0.0434; lower is better, <0.013 makes methods resolvable)


In [ ]:
# ============================================================================
# NIH CELL 3 — COST DRY-RUN: 1 model on a small slice, prints realized cost
# Run this BEFORE the full pass to confirm the unit cost with 19 units left.
# ============================================================================
import torch, time, gc
from PIL import Image
from torch.utils.data import Dataset, DataLoader

device = 'cuda' if torch.cuda.is_available() else 'cpu'

class NihDataset(Dataset):
    """Same (img, label, sex) contract as your CheXpertDataset; reads NIH_IMG flat folder."""
    def __init__(self, df, transform):
        self.df = df.reset_index(drop=True); self.tf = transform
    def __len__(self): return len(self.df)
    def __getitem__(self, idx):
        r = self.df.iloc[idx]
        img = Image.open(os.path.join(NIH_IMG, r['Path'])).convert('RGB')
        img = self.tf(img)
        return img, torch.tensor(r['label'], dtype=torch.float32), torch.tensor(r['sex_encoded'], dtype=torch.long)

def make_nih_loader(df):
    on_gpu = torch.cuda.is_available()
    return DataLoader(NihDataset(df, eval_tf), batch_size=BATCH_SIZE, shuffle=False,
                      num_workers=4 if on_gpu else 2, pin_memory=on_gpu, persistent_workers=False)

cohort = pd.read_csv(NIH_COHORT_CSV)
DRY_N = 1000
dry = cohort.sample(min(DRY_N, len(cohort)), random_state=42).reset_index(drop=True)
print(f"DRY RUN: {len(dry)} images, 1 model (DWFA seed 42)")

t0 = time.time()
gm = build_model(device)
gm.load_state_dict(torch.load(f'{CKPTS}/dwfa_seed42.pt', map_location='cpu')); gm.eval()
loader = make_nih_loader(dry)
m = evaluate_dual(gm, loader, device, 0.5)
del gm, loader; gc.collect(); torch.cuda.empty_cache()
dt = time.time() - t0

full_n = len(cohort)
print(f"  wall-clock: {dt:.1f}s for {len(dry)} imgs")
print(f"  -> full cohort ({full_n} imgs) x 3 models ~ {dt*(full_n/len(dry))*3/60:.1f} min")
print(f"  sanity AUROC on dry slice: {m['auroc']:.4f}  (transfer; expect ~0.80-0.88, NOT 0.5)")
print("\n>>> CHECK Colab's unit meter NOW. If this slice cost X units,")
print(f">>> the full 3-model pass costs ~{(full_n/len(dry))*3:.0f}x X. Decide before CELL 4.")

DRY RUN: 1000 images, 1 model (DWFA seed 42)
Downloading: "https://download.pytorch.org/models/densenet121-a639ec97.pth" to /root/.cache/torch/hub/checkpoints/densenet121-a639ec97.pth


100%|██████████| 30.8M/30.8M [00:00<00:00, 168MB/s]
/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py:424: UserWarning: This DataLoader will create 4 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_rationality()


NameError: name 'evaluate_dual' is not defined

In [ ]:
# ============================================================================
# CHECKPOINT EXISTENCE CHECK — run before NIH Cell 4
# Confirms which checkpoint files Cell 4 will actually find vs silently skip.
# ============================================================================
import os

print(f"CKPTS = {CKPTS}\n")
get_ipython().system(f'ls -la "{CKPTS}"')

print(f"\n{'='*60}\nPER-METHOD CHECK (what NIH Cell 4's METHOD_SEEDS expects)\n{'='*60}")

METHOD_SEEDS_CHECK = {
    'fedavg':  [42, 123, 456, 789, 1010],
    'qfedavg': [42, 123, 456, 789,1010],
    'dwfa':    [42, 123, 456, 789, 1010],
}

found, missing = [], []
for method, seeds in METHOD_SEEDS_CHECK.items():
    for seed in seeds:
        path = f'{CKPTS}/{method}_seed{seed}.pt'
        if os.path.exists(path):
            size_mb = os.path.getsize(path) / 1e6
            found.append((method, seed, size_mb))
            print(f"  [OK]      {method:<9} seed {seed:<5} ({size_mb:.1f} MB)  {path}")
        else:
            missing.append((method, seed))
            print(f"  [MISSING] {method:<9} seed {seed:<5}                {path}")

print(f"\n{'='*60}")
print(f"Found:   {len(found)} / {len(found)+len(missing)}")
if missing:
    print(f"Missing: {missing}")
    print("\nNIH Cell 4 will run fine but will SILENTLY SKIP every missing one above")
    print("(it prints '[method sN] missing -> skip' per row). Fix the path or")
    print("confirm those seeds genuinely weren't saved before running Cell 4,")
    print("otherwise your final table may be missing whole methods or seeds.")
else:
    print("All expected checkpoints present. Safe to run NIH Cell 4.")

CKPTS = /content/drive/MyDrive/FairFedCXR/checkpoints

total 1331045
-rw------- 1 root root 28396909 Jun 16 15:32 centralized_seed123.pt
-rw------- 1 root root 28396176 Jun 16 15:17 centralized_seed42.pt
-rw------- 1 root root 28396909 Jun 16 15:43 centralized_seed456.pt
-rw------- 1 root root 28402901 Jul 10 16:05 dwfa_scrambled_ckpt_seed123.pt
-rw------- 1 root root 28402168 Jul 10 14:34 dwfa_scrambled_ckpt_seed42.pt
-rw------- 1 root root 28402901 Jul 11 17:29 dwfa_scrambled_ckpt_seed456.pt
-rw------- 1 root root 28392511 Jun 20 12:59 dwfa_seed1010.pt
-rw------- 1 root root 28391778 Jun 20 07:55 dwfa_seed123.pt
-rw------- 1 root root 28391045 Jun 20 06:13 dwfa_seed42.pt
-rw------- 1 root root 28391778 Jun 20 09:36 dwfa_seed456.pt
-rw------- 1 root root 28391778 Jun 20 11:17 dwfa_seed789.pt
-rw------- 1 root root 28405292 Jul 12 14:58 fairfed_b0.1_seed42_INPROGRESS.pt
-rw------- 1 root root 28394710 Jul  2 05:55 fairfed_seed1010.pt
-rw------- 1 root root 28393977 Jun 27 07:28 fairfed

In [ ]:

# ============================================================================
# NIH CELL 4 — FULL same-cohort external validation: FedAvg vs q-FedAvg vs DWFA
# All metrics: AUROC, WG-AUROC, EO-gap & FPR-gap @0.5 and @Youden, ECE, with
# bootstrap 95% CIs on the EO-gap (the whole point: now estimable). Across all
# available seeds per method for proper CIs. Run only after dry-run is acceptable.
# ============================================================================
import torch, gc, numpy as np
from sklearn.metrics import roc_auc_score, roc_curve

device = 'cuda' if torch.cuda.is_available() else 'cpu'
cohort = pd.read_csv(NIH_COHORT_CSV)

# which trained checkpoints to evaluate; loops every seed file that exists per method
METHOD_SEEDS = {
    'fedavg':   [42, 123, 456],
    'qfedavg':  [42, 123, 456, 789,1010],            # add 789,1010 here if you extended q-FedAvg
    'dwfa':     [42, 123, 456, 789, 1010],
}

@torch.no_grad()
def infer(ckpt_path):
    gm = build_model(device); gm.load_state_dict(torch.load(ckpt_path, map_location='cpu')); gm.eval()
    loader = make_nih_loader(cohort)
    P, L, S = [], [], []
    for imgs, labels, sex in loader:
        imgs = imgs.to(device, non_blocking=True)
        with autocast('cuda'):
            logits = gm(imgs).squeeze()
        P.append(torch.sigmoid(logits.float()).cpu().numpy()); L.append(labels.numpy()); S.append(sex.numpy())
    del gm, loader; gc.collect(); torch.cuda.empty_cache()
    return np.concatenate(P), np.concatenate(L), np.concatenate(S)

def youden_from_arrays(probs, labels):
    fpr, tpr, thr = roc_curve(labels, probs); return float(thr[int(np.argmax(tpr - fpr))])

def boot_eo(probs, labels, sex, thr, n_boot=2000, seed=0):
    rng = np.random.default_rng(seed); n = len(labels); g = np.empty(n_boot)
    for b in range(n_boot):
        idx = rng.integers(0, n, n); g[b], _, _ = compute_eo_gap(probs[idx], labels[idx], sex[idx], thr)
    return g

rows = []
for method, seeds in METHOD_SEEDS.items():
    for seed in seeds:
        ckpt = f'{CKPTS}/{method}_seed{seed}.pt'
        if not os.path.exists(ckpt):
            print(f"  [{method} s{seed}] missing -> skip"); continue
        probs, labels, sex = infer(ckpt)
        yj = youden_from_arrays(probs, labels)
        row = {'method': method, 'seed': seed, 'auroc': float(roc_auc_score(labels, probs)),
               'ece': compute_ece(probs, labels),
               'male_auroc': safe_auroc(labels[sex==1], probs[sex==1]),
               'female_auroc': safe_auroc(labels[sex==0], probs[sex==0]),
               'm_pos': int(((sex==1)&(labels==1)).sum()), 'f_pos': int(((sex==0)&(labels==1)).sum())}
        row['wg_auroc'] = float(np.nanmin([row['male_auroc'], row['female_auroc']]))
        for sfx, thr in [('', 0.5), ('_yj', yj)]:
            eo, mt, ft = compute_eo_gap(probs, labels, sex, thr)
            row[f'eo_gap{sfx}'] = float(eo); row[f'fpr_gap{sfx}'] = float(compute_fpr_gap(probs, labels, sex, thr))
        gb = boot_eo(probs, labels, sex, 0.5)
        row['eo_ci_lo'] = float(np.percentile(gb, 2.5)); row['eo_ci_hi'] = float(np.percentile(gb, 97.5))
        rows.append(row)
        print(f"  {method} s{seed}: AUROC={row['auroc']:.4f} EO@.5={row['eo_gap']:.4f} "
              f"[{row['eo_ci_lo']:.4f},{row['eo_ci_hi']:.4f}] FPR={row['fpr_gap']:.4f} WG={row['wg_auroc']:.4f}")

res = pd.DataFrame(rows)
res.to_csv(f'{RESULTS}/nih_external_all.csv', index=False)

# ---- per-method summary (mean +/- std across seeds), both thresholds ----
def ms(a, s): return f"{a:.4f}\u00b1{s:.4f}"
print(f"\n{'='*92}\nNIH ChestX-ray14 EXTERNAL VALIDATION (mean \u00b1 std across seeds, same cohort)\n{'='*92}")
print(f"  {'method':<10}{'AUROC':>15}{'WG-AUROC':>15}{'EO@.5':>15}{'EO@YJ':>15}{'FPR@.5':>15}")
for method in METHOD_SEEDS:
    d = res[res['method'] == method]
    if len(d) == 0: continue
    print(f"  {method:<10}{ms(d.auroc.mean(),d.auroc.std()):>15}{ms(d.wg_auroc.mean(),d.wg_auroc.std()):>15}"
          f"{ms(d.eo_gap.mean(),d.eo_gap.std()):>15}{ms(d.eo_gap_yj.mean(),d.eo_gap_yj.std()):>15}"
          f"{ms(d.fpr_gap.mean(),d.fpr_gap.std()):>15}")
print(f"\nSubgroup positives (min over sex): {int(res.iloc[0]['m_pos'])}M / {int(res.iloc[0]['f_pos'])}F "
      f"-- compare to CheXpert C's 87. Tight EO CIs here = methods finally resolvable.")
print("VALID: methods vs each other on the SAME NIH cohort (domain shift cancels).")
print("Do NOT compare these absolute numbers to CheXpert absolutes.")
print(f"Saved -> {RESULTS}/nih_external_all.csv")

  fedavg s42: AUROC=0.8599 EO@.5=0.0242 [0.0102,0.0382] FPR=0.0199 WG=0.8582
  fedavg s123: AUROC=0.8580 EO@.5=0.0268 [0.0136,0.0397] FPR=0.0225 WG=0.8561
  fedavg s456: AUROC=0.8535 EO@.5=0.0354 [0.0210,0.0504] FPR=0.0180 WG=0.8534


KeyboardInterrupt: 

In [ ]:
import torch, gc, numpy as np
from sklearn.metrics import roc_auc_score, roc_curve

device = 'cuda' if torch.cuda.is_available() else 'cpu'
cohort = pd.read_csv(NIH_COHORT_CSV)

@torch.no_grad()
def infer(ckpt_path):
    gm = build_model(device); gm.load_state_dict(torch.load(ckpt_path, map_location='cpu')); gm.eval()
    loader = make_nih_loader(cohort)
    P, L, S = [], [], []
    for imgs, labels, sex in loader:
        imgs = imgs.to(device, non_blocking=True)
        with autocast('cuda'):
            logits = gm(imgs).squeeze()
        P.append(torch.sigmoid(logits.float()).cpu().numpy()); L.append(labels.numpy()); S.append(sex.numpy())
    del gm, loader; gc.collect(); torch.cuda.empty_cache()
    return np.concatenate(P), np.concatenate(L), np.concatenate(S)

def youden_from_arrays(probs, labels):
    fpr, tpr, thr = roc_curve(labels, probs); return float(thr[int(np.argmax(tpr - fpr))])

def boot_eo(probs, labels, sex, thr, n_boot=2000, seed=0):
    rng = np.random.default_rng(seed); n = len(labels); g = np.empty(n_boot)
    for b in range(n_boot):
        idx = rng.integers(0, n, n); g[b], _, _ = compute_eo_gap(probs[idx], labels[idx], sex[idx], thr)
    return g

print("infer() defined — ready for Cell B")

infer() defined — ready for Cell B


In [ ]:
# ============================================================================
# PLACEBO CELL A — retrain scrambled DWFA, save checkpoints + crash recovery
# Identical to run_dwfa_placebo(mode='scrambled') but:
#   1. saves best global state to Drive (final_ckpt) so Cell B can infer on NIH
#   2. saves progress checkpoint every 2 rounds so crash = resume,
#      not restart. Same pattern as run_dwfa (cell 67) in your notebook.
#
# MUST RUN FIRST (in memory):
#   drive mount + RESULTS/CKPTS/CLIENTS setup cell
#   "DWFA day 8,3" cell (constants, build_model, make_loader, set_seed,
#                        local_train_dwfa, gated_equity, dwfa_aggregate,
#                        evaluate_dual, compute_static_confidence)
#   "NIH CELL 3" cell (make_nih_loader, NIH_COHORT_CSV)
# ============================================================================
import torch, copy, gc, os, numpy as np, pandas as pd
from itertools import permutations

device = 'cuda' if torch.cuda.is_available() else 'cpu'

_DERANGEMENTS_4 = [p for p in permutations(range(4))
                   if all(p[i] != i for i in range(4))]
FIXED_DERANGEMENT = _DERANGEMENTS_4[0]          # same as original placebo cell
SCR_SEEDS         = [42, 123, 456]
SCR_ROUNDS        = 20
SCR_TAG           = 'dwfa_scrambled_ckpt'
print(f"Derangement: {FIXED_DERANGEMENT}  |  seeds: {SCR_SEEDS}  |  rounds: {SCR_ROUNDS}")

pooled_val_csv = '/content/pooled_val_scr.csv'
if not os.path.exists(pooled_val_csv):
    pd.concat([pd.read_csv(f'{CLIENTS}/client_{c}_val.csv') for c in CLIENT_LIST],
              ignore_index=True).to_csv(pooled_val_csv, index=False)

conf, _ = compute_static_confidence(TAU_REF, RHO_DWFA)

for seed in SCR_SEEDS:
    final_ckpt = f'{CKPTS}/{SCR_TAG}_seed{seed}.pt'
    prog_ckpt  = f'{CKPTS}/{SCR_TAG}_seed{seed}_progress.pt'

    if os.path.exists(final_ckpt):
        print(f"[seed {seed}] final ckpt exists -> skip"); continue

    print(f"\n{'='*58}\n  SCRAMBLED | SEED {seed} | derangement={FIXED_DERANGEMENT}\n{'='*58}")
    set_seed(seed)

    ctl = {c: make_loader(f'{CLIENTS}/client_{c}_train.csv', train=True)  for c in CLIENT_LIST}
    cvl = {c: make_loader(f'{CLIENTS}/client_{c}_val.csv',   train=False) for c in CLIENT_LIST}
    cs  = {c: len(ctl[c].dataset) for c in CLIENT_LIST}
    N   = sum(cs.values()); p_w = [cs[c]/N for c in CLIENT_LIST]
    pvl = make_loader(pooled_val_csv, train=False)

    # resume from progress checkpoint if it exists
    if os.path.exists(prog_ckpt):
        ck = torch.load(prog_ckpt, map_location='cpu')
        global_state = ck['gs']
        ebar_sum     = ck['ebar_sum']
        ebar_cnt     = ck['ebar_cnt']
        start_round  = ck['next_round']
        best_auroc   = ck['best_auroc']
        best_gs      = ck['best_gs']
        print(f"  Resuming from round {start_round + 1} (progress ckpt)")
    else:
        gm = build_model(device)
        global_state = {k: v.cpu().clone() for k, v in gm.state_dict().items()}
        del gm; torch.cuda.empty_cache()
        ebar_sum    = {c: 0.0 for c in CLIENT_LIST}
        ebar_cnt    = {c: 0   for c in CLIENT_LIST}
        start_round = 0
        best_auroc  = -1.0
        best_gs     = None

    for rnd in range(start_round, SCR_ROUNDS):
        states, et_real = [], []
        for c in CLIENT_LIST:
            st, g_i = local_train_dwfa(global_state, ctl[c], cvl[c], device,
                                       tag=f"R{rnd+1}-{c}")
            is_first   = (ebar_cnt[c] == 0)
            ebar_prior = (ebar_sum[c] / ebar_cnt[c]) if not is_first else 0.0
            e_cur, etil = gated_equity(g_i, conf[c], ebar_prior, is_first,
                                       G_REF, RHO_DWFA)
            ebar_sum[c] += e_cur; ebar_cnt[c] += 1
            states.append(st); et_real.append(etil)

        # SCRAMBLE: client i receives etilde from client derangement[i]
        et_used = [et_real[FIXED_DERANGEMENT[i]] for i in range(4)]

        global_state, a = dwfa_aggregate(states, et_used, p_w,
                                         ALPHA_DWFA, KAPPA_DWFA)
        del states; gc.collect()

        gm = build_model(device); gm.load_state_dict(global_state)
        vm = evaluate_dual(gm, pvl, device, 0.5)
        del gm; gc.collect(); torch.cuda.empty_cache()

        if vm['auroc'] > best_auroc:
            best_auroc = vm['auroc']
            best_gs    = copy.deepcopy(global_state)

        print(f"  R{rnd+1:02d}/{SCR_ROUNDS}: AUROC={vm['auroc']:.4f} "
              f"EO={vm['eo_gap']:.4f} | a={[round(float(w),3) for w in a]}")

        # progress checkpoint every 2 rounds (crash recovery)
        if (rnd + 1) % 2 == 0:
            torch.save({'gs': global_state, 'ebar_sum': ebar_sum,
                        'ebar_cnt': ebar_cnt, 'next_round': rnd + 1,
                        'best_auroc': best_auroc, 'best_gs': best_gs},
                       prog_ckpt)
            print(f"  [progress ckpt saved at round {rnd+1}]")

    # save final best checkpoint, delete progress ckpt
    torch.save(best_gs, final_ckpt)
    if os.path.exists(prog_ckpt):
        os.remove(prog_ckpt)
    print(f"  [seed {seed}] DONE — best val AUROC={best_auroc:.4f} -> {final_ckpt}")

    del ctl, cvl, pvl; gc.collect(); torch.cuda.empty_cache()

print("\nAll seeds done. Run Cell B for NIH inference.")

Derangement: (1, 0, 3, 2)  |  seeds: [42, 123, 456]  |  rounds: 20
[seed 42] final ckpt exists -> skip
[seed 123] final ckpt exists -> skip

  SCRAMBLED | SEED 456 | derangement=(1, 0, 3, 2)
  Seed locked: 456
  Resuming from round 19 (progress ckpt)


  R19/20: AUROC=0.9500 EO=0.0342 | a=[0.362, 0.276, 0.233, 0.129]


  R20/20: AUROC=0.9497 EO=0.0439 | a=[0.462, 0.157, 0.198, 0.183]
  [progress ckpt saved at round 20]
  [seed 456] DONE — best val AUROC=0.9536 -> /content/drive/MyDrive/FairFedCXR/checkpoints/dwfa_scrambled_ckpt_seed456.pt

All seeds done. Run Cell B for NIH inference.


In [ ]:
# ============================================================================
# PLACEBO CELL B — evaluate scrambled DWFA on NIH, paired vs real DWFA
# This is the experiment the reviewer asked for: does the fairness signal
# matter WHERE THE WIN LIVES (NIH worst-group AUROC), not just at Client C?
#
# INFERENCE ONLY. No training. Runs infer() on the scrambled checkpoints from
# Cell A, then a paired bootstrap over the SAME NIH cohort indices comparing
# real DWFA vs scrambled DWFA on worst-group AUROC and overall AUROC.
#
# Interpretation:
#   - If real DWFA and scrambled DWFA are statistically INDISTINGUISHABLE on
#     NIH worst-group AUROC, the +0.0025 advantage is NOT coming from the
#     fairness signal -> mechanism claim weakens, must be reported honestly.
#   - If real DWFA is significantly BETTER than scrambled on NIH worst-group,
#     that is direct evidence the fairness routing (not just the reweighting
#     magnitude) drives the result -> strongest single result in the paper.
#
# MUST RUN FIRST (already in memory):
#   Cell 79 (NIH Cell 4) -> infer(), make_nih_loader(), NIH_COHORT_CSV,
#                           build_model, device, autocast
#   Cell 80 -> created nih_pred_cache.npz with real dwfa/fedavg/qfedavg preds
#   Cell A (above) -> saved dwfa_scrambled_ckpt_seed{42,123,456}.pt
# ============================================================================
import torch, gc, os, numpy as np, pandas as pd
from sklearn.metrics import roc_auc_score

device = 'cuda' if torch.cuda.is_available() else 'cpu'
cohort = pd.read_csv(NIH_COHORT_CSV)

REAL_DWFA_SEEDS = [42, 123, 456, 789, 1010]   # real DWFA, all seeds (from cache)
SCR_SEEDS       = [42, 123, 456]              # scrambled, the seeds Cell A saved

def safe_auroc(lab, prob):
    if len(lab) < 2 or lab.sum() < 1 or (lab == 0).sum() < 1: return float('nan')
    try: return float(roc_auc_score(lab, prob))
    except Exception: return float('nan')

def wg_auroc(prob, lab, sx):
    a_m, a_f = safe_auroc(lab[sx == 1], prob[sx == 1]), safe_auroc(lab[sx == 0], prob[sx == 0])
    if np.isnan(a_m) and np.isnan(a_f): return float('nan')
    return float(np.nanmin([a_m, a_f]))

# ---- 1. get NIH predictions for real DWFA (from cache) and scrambled (fresh infer) ----
cache_path = f'{RESULTS}/nih_pred_cache.npz'
assert os.path.exists(cache_path), "run NIH Cell 80 first to build nih_pred_cache.npz"
cache = np.load(cache_path, allow_pickle=True)
all_probs = cache['all_probs'].item()
labels = cache['labels']; sex = cache['sex']

# real DWFA per-seed probs are already cached as ('dwfa', seed)
real_dwfa_probs = [all_probs[('dwfa', s)] for s in REAL_DWFA_SEEDS if ('dwfa', s) in all_probs]
print(f"Loaded {len(real_dwfa_probs)} real-DWFA seed prediction sets from cache.")

# scrambled: run inference now (predictions not cached)
scr_probs = []
for s in SCR_SEEDS:
    ckpt = f'{CKPTS}/dwfa_scrambled_ckpt_seed{s}.pt'
    if not os.path.exists(ckpt):
        print(f"  [scrambled s{s}] missing {ckpt} -> run Cell A first"); continue
    p, lab, sx = infer(ckpt)   # reuse the notebook's infer()
    # sanity: same cohort ordering as the cache
    assert len(p) == len(labels), "scrambled inference length != cached NIH length"
    scr_probs.append(p)
    print(f"  inferred scrambled s{s} on NIH")
assert len(scr_probs) > 0, "no scrambled checkpoints found"

# ---- 2. point estimates: seed-averaged probs -> WG-AUROC and AUROC ----
real_mean = np.mean(np.stack(real_dwfa_probs), axis=0)
scr_mean  = np.mean(np.stack(scr_probs),       axis=0)

real_wg = wg_auroc(real_mean, labels, sex);  real_au = safe_auroc(labels, real_mean)
scr_wg  = wg_auroc(scr_mean,  labels, sex);  scr_au  = safe_auroc(labels, scr_mean)

print("\n" + "="*64)
print("NIH POINT ESTIMATES (seed-averaged predictions)")
print("="*64)
print(f"  Real DWFA : WG-AUROC={real_wg:.4f}  AUROC={real_au:.4f}")
print(f"  Scrambled : WG-AUROC={scr_wg:.4f}  AUROC={scr_au:.4f}")
print(f"  Difference: WG-AUROC={real_wg-scr_wg:+.4f}  AUROC={real_au-scr_au:+.4f}")

# ---- 3. paired bootstrap over NIH cohort indices (same resample for both) ----
def paired_boot(real_seed_probs, scr_seed_probs, labels, sex, metric='wg',
                n_boot=2000, seed=0):
    rng = np.random.default_rng(seed); n = len(labels)
    real_stack = np.stack(real_seed_probs); scr_stack = np.stack(scr_seed_probs)
    diffs = np.empty(n_boot)
    for b in range(n_boot):
        idx = rng.integers(0, n, n)
        rp = real_stack[:, idx].mean(axis=0); sp = scr_stack[:, idx].mean(axis=0)
        lab_b, sx_b = labels[idx], sex[idx]
        if metric == 'wg':
            diffs[b] = wg_auroc(rp, lab_b, sx_b) - wg_auroc(sp, lab_b, sx_b)
        else:
            diffs[b] = safe_auroc(lab_b, rp) - safe_auroc(lab_b, sp)
    lo, hi = np.nanpercentile(diffs, [2.5, 97.5])
    p_two = 2 * min((diffs <= 0).mean(), (diffs >= 0).mean())
    return float(np.nanmean(diffs)), float(lo), float(hi), float(p_two)

print("\n" + "="*64)
print("PAIRED BOOTSTRAP: Real DWFA - Scrambled DWFA on NIH (2000 resamples)")
print("="*64)
for metric, name in [('wg', 'Worst-group AUROC'), ('auroc', 'Overall AUROC')]:
    d, lo, hi, p = paired_boot(real_dwfa_probs, scr_probs, labels, sex, metric=metric)
    sig = "SIGNIFICANT" if (lo > 0 or hi < 0) else "not significant (CI crosses 0)"
    print(f"  {name:20s}: {d:+.4f}  95% CI [{lo:+.4f}, {hi:+.4f}]  p={p:.4f}  -> {sig}")

print("\nRead: if WG-AUROC difference is SIGNIFICANT and positive, the fairness")
print("routing (not just reweighting magnitude) drives the NIH result. If not")
print("significant, report honestly that the signal is not separable on NIH.")

# save
out = pd.DataFrame([{
    'real_wg': real_wg, 'scr_wg': scr_wg, 'wg_diff': real_wg - scr_wg,
    'real_auroc': real_au, 'scr_auroc': scr_au, 'auroc_diff': real_au - scr_au,
    'n_real_seeds': len(real_dwfa_probs), 'n_scr_seeds': len(scr_probs),
}])
out.to_csv(f'{RESULTS}/nih_placebo_result.csv', index=False)
print(f"\nSaved -> {RESULTS}/nih_placebo_result.csv")

Loaded 5 real-DWFA seed prediction sets from cache.
  inferred scrambled s42 on NIH
  inferred scrambled s123 on NIH
  inferred scrambled s456 on NIH

NIH POINT ESTIMATES (seed-averaged predictions)
  Real DWFA : WG-AUROC=0.8592  AUROC=0.8616
  Scrambled : WG-AUROC=0.8570  AUROC=0.8591
  Difference: WG-AUROC=+0.0023  AUROC=+0.0024

PAIRED BOOTSTRAP: Real DWFA - Scrambled DWFA on NIH (2000 resamples)
  Worst-group AUROC   : +0.0023  95% CI [+0.0016, +0.0031]  p=0.0000  -> SIGNIFICANT
  Overall AUROC       : +0.0024  95% CI [+0.0019, +0.0029]  p=0.0000  -> SIGNIFICANT

Read: if WG-AUROC difference is SIGNIFICANT and positive, the fairness
routing (not just reweighting magnitude) drives the NIH result. If not
significant, report honestly that the signal is not separable on NIH.

Saved -> /content/drive/MyDrive/FairFedCXR/results/nih_placebo_result.csv


In [ ]:
#missing matrics
# ============================================================================
# MISSING-METRICS RECOMPUTE — FedAvg's WG-AUROC / FPR-gap / EO@Youden
# FedAvg's original evaluate() never computed these (only eo_gap@0.5, auprc,
# ece). This is INFERENCE ONLY on the existing fedavg_seed*.pt checkpoints --
# no retraining. Run AFTER DWFA cells (needs build_model, evaluate_dual,
# compute_fpr_gap, youden_threshold, eval_tf, CKPTS, RESULTS already loaded).
# ============================================================================
import torch, gc, pandas as pd, numpy as np, os

device = 'cuda' if torch.cuda.is_available() else 'cpu'
CLIENT_LIST = ['A', 'B', 'C', 'D']

rows = []
for seed in [42, 123, 456]:
    ckpt = f'{CKPTS}/fedavg_seed{seed}.pt'
    if not os.path.exists(ckpt):
        print(f"  [fedavg s{seed}] checkpoint missing -> skip"); continue
    gm = build_model(device)
    gm.load_state_dict(torch.load(ckpt, map_location='cpu'))
    gm.eval()

    # pooled val loader, needed to pick a Youden threshold for this seed/model
    pooled_val = pd.concat([pd.read_csv(f'{CLIENTS}/client_{c}_val.csv')
                            for c in CLIENT_LIST], ignore_index=True)
    pooled_val.to_csv('/content/pooled_val_recompute.csv', index=False)
    pvl = make_loader('/content/pooled_val_recompute.csv', train=False)
    yj_thr = youden_threshold(gm, pvl, device)

    for c in CLIENT_LIST:
        te = make_loader(f'{CLIENTS}/client_{c}_test.csv', train=False)
        m = evaluate_dual(gm, te, device, yj_thr)
        m.update({'method': 'fedavg', 'client': c, 'seed': seed})
        rows.append(m)
        print(f"  fedavg s{seed} {c}: AUROC={m['auroc']:.4f}  WG-AUROC={m['wg_auroc']:.4f}  "
              f"EO@.5={m['eo_gap']:.4f}  EO@YJ={m['eo_gap_yj']:.4f}  FPR@.5={m['fpr_gap']:.4f}")
        del te; gc.collect()
    del gm; gc.collect(); torch.cuda.empty_cache()

recompute = pd.DataFrame(rows)
recompute.to_csv(f'{RESULTS}/fedavg_all_recomputed.csv', index=False)

# ---- merge into the ORIGINAL fedavg_all.csv so downstream tables/figures
#      that read fedavg_all.csv automatically pick up the new columns ----
orig = pd.read_csv(f'{RESULTS}/fedavg_all.csv')
keep_orig_cols = [c for c in orig.columns if c not in recompute.columns or c in ('client', 'seed', 'method')]
merged = orig[keep_orig_cols].merge(
    recompute, on=['method', 'client', 'seed'], suffixes=('', '_new'))
# prefer the newly recomputed values where both exist
for col in ['auroc', 'auprc', 'ece', 'eo_gap']:
    if f'{col}_new' in merged.columns:
        merged[col] = merged[f'{col}_new']
        merged = merged.drop(columns=[f'{col}_new'])
merged.to_csv(f'{RESULTS}/fedavg_all.csv', index=False)

print(f"\n{'='*70}\nFedAvg — now complete with WG-AUROC, FPR-gap, EO@Youden\n{'='*70}")
def ms(a, s): return f"{a:.4f}\u00b1{s:.4f}"
for c in CLIENT_LIST:
    sub = merged[merged['client'] == c]
    print(f"  {c}: AUROC={ms(sub.auroc.mean(),sub.auroc.std())}  "
          f"WG-AUROC={ms(sub.wg_auroc.mean(),sub.wg_auroc.std())}  "
          f"EO@.5={ms(sub.eo_gap.mean(),sub.eo_gap.std())}  "
          f"EO@YJ={ms(sub.eo_gap_yj.mean(),sub.eo_gap_yj.std())}  "
          f"FPR={ms(sub.fpr_gap.mean(),sub.fpr_gap.std())}")
print(f"\nSaved -> {RESULTS}/fedavg_all.csv (updated in place)")
print(f"Backup of recompute-only rows -> {RESULTS}/fedavg_all_recomputed.csv")
print("\nNow your 'FedAvg vs FedProx WG-AUROC' table (previously 'N/A (recompute)')")
print("can be filled in correctly.")


  fedavg s42 A: AUROC=0.9532  WG-AUROC=0.9492  EO@.5=0.0162  EO@YJ=0.0048  FPR@.5=0.0274
  fedavg s42 B: AUROC=0.9497  WG-AUROC=0.9408  EO@.5=0.0079  EO@YJ=0.0021  FPR@.5=0.0313
  fedavg s42 C: AUROC=0.9387  WG-AUROC=0.9362  EO@.5=0.0308  EO@YJ=0.0060  FPR@.5=0.0029
  fedavg s42 D: AUROC=0.9641  WG-AUROC=0.9532  EO@.5=0.0319  EO@YJ=0.0312  FPR@.5=0.0353
  fedavg s123 A: AUROC=0.9510  WG-AUROC=0.9471  EO@.5=0.0052  EO@YJ=0.0018  FPR@.5=0.0085
  fedavg s123 B: AUROC=0.9432  WG-AUROC=0.9221  EO@.5=0.0066  EO@YJ=0.0118  FPR@.5=0.0467
  fedavg s123 C: AUROC=0.9351  WG-AUROC=0.9326  EO@.5=0.0169  EO@YJ=0.0169  FPR@.5=0.0324
  fedavg s123 D: AUROC=0.9647  WG-AUROC=0.9551  EO@.5=0.0510  EO@YJ=0.0464  FPR@.5=0.0260
  fedavg s456 A: AUROC=0.9519  WG-AUROC=0.9475  EO@.5=0.0199  EO@YJ=0.0113  FPR@.5=0.0184
  fedavg s456 B: AUROC=0.9502  WG-AUROC=0.9477  EO@.5=0.0050  EO@YJ=0.0069  FPR@.5=0.0222
  fedavg s456 C: AUROC=0.9398  WG-AUROC=0.9278  EO@.5=0.0004  EO@YJ=0.0159  FPR@.5=0.0106
  fedavg s456 

In [ ]:
# SOTA 1.1
from torch.amp import autocast, GradScaler
import gc, torch

def local_train(global_state, train_loader, device, local_epochs=LOCAL_EPOCHS):
    """Plain FedAvg-style local training (what FairFed uses; innovation is in aggregation)."""
    model = build_model(device)
    model.load_state_dict(global_state)
    model.train()
    optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
    criterion = torch.nn.BCEWithLogitsLoss()
    scaler = GradScaler('cuda')

    for ep in range(local_epochs):
        for imgs, labels, _ in train_loader:
            imgs   = imgs.to(device, non_blocking=True)
            labels = labels.to(device, non_blocking=True)
            optimizer.zero_grad()
            with autocast('cuda'):
                logits = model(imgs).squeeze()
                loss   = criterion(logits, labels)
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()

    state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
    del model, optimizer
    gc.collect()
    torch.cuda.empty_cache()
    return state

print("local_train defined")

local_train defined


In [ ]:
# SOTA test 1
#d cells · PY
# ############################################################################
# FAIRFED (Ezzeldin et al., AAAI 2023) — SOTA baseline implementation.
# Algorithm 1 / Eq.6 from arXiv:2110.00857, adapted to your binary CXR setup.
# Per-round: each client's local EOD gap |F_global - F_k| (using TPR-gap as
# F_k, matching your compute_eo_gap convention) updates an ADDITIVE weight
# correction. At beta=0, FairFed == FedAvg (same anchor property as DWFA).
#
# BUILT FOR FREE-TIER T4 CRASH RESILIENCE:
#   - checkpoints EVERY round (not every 5) -- a crash loses at most 1 round
#   - fully resumable: re-running the cell picks up from the last saved round
#   - local training reuses your existing local_train() function unmodified
#   - per-round wall-clock printed so you can judge session-length risk live
#
# Run order: Cell 0 -> Cell 56 (RESULTS,CKPTS,CLIENTS) -> Cell 57
# (build_model, evaluate_dual, compute_eo_gap, make_loader, local_train) ->
# device line -> THESE cells.
# ############################################################################


# ============================================================================
# FAIRFED CELL 1 — config + weight-update mechanics (CPU, instant, no GPU)
# ============================================================================
import torch, copy, gc, time, os, json, numpy as np, pandas as pd

device = 'cuda' if torch.cuda.is_available() else 'cpu'
CLIENT_LIST = ['A', 'B', 'C', 'D']

FAIRFED_BETA = 1.0     # fairness budget; paper default beta=1 for Adult/COMPAS
                       # (CXR task may need smaller beta -- tune via debug run)
FAIRFED_ROUNDS = 30    # match your other methods for a fair comparison
FAIRFED_LOCAL_EPOCHS = LOCAL_EPOCHS  # reuse your existing constant

def fairfed_client_metrics(model, loader, device, thr=0.5):
    """Returns local F_k (EO-gap == |male_tpr - female_tpr|) and local accuracy.
    Reuses your existing evaluate_dual for consistency with other methods."""
    m = evaluate_dual(model, loader, device, thr)
    # F_k undefined if this client has no positives in one sex (Eq.6's edge case)
    f_k = m['eo_gap'] if not np.isnan(m['eo_gap']) else None
    acc = 1.0 - m.get('fpr_gap', 0.0) if 'acc' not in m else m['acc']
    # fall back to a simple accuracy proxy from AUROC-thresholded preds if not present
    return f_k, m.get('auroc', float('nan')), m

def fairfed_update_weights(prev_omega_bar, deltas, beta):
    """Eq.(6): omega_bar_k^t = omega_bar_k^{t-1} - beta*(delta_k - mean(delta))"""
    mean_delta = np.mean(list(deltas.values()))
    new_bar = {k: prev_omega_bar[k] - beta * (deltas[k] - mean_delta) for k in prev_omega_bar}
    # clip to stay non-negative (paper doesn't explicitly require this, but
    # negative weights are meaningless for a convex combination)
    new_bar = {k: max(v, 1e-6) for k, v in new_bar.items()}
    total = sum(new_bar.values())
    omega = {k: v / total for k, v in new_bar.items()}
    return new_bar, omega

print("FairFed mechanics defined (Eq.6, additive weight update).")
print(f"  beta={FAIRFED_BETA}  rounds={FAIRFED_ROUNDS}")


FairFed mechanics defined (Eq.6, additive weight update).
  beta=1.0  rounds=30


In [ ]:
#SOTA test 2
# ============================================================================
# FAIRFED CELL 2 — crash-resilient training driver. Checkpoints EVERY round
# to a small JSON+PT pair so a free-tier T4 disconnect loses at most 1 round
# of work, not the whole run. Re-running this cell after a crash AUTO-RESUMES
# from the last saved round for that seed.
# ============================================================================
import torch, copy, gc, time, os, json, numpy as np, pandas as pd

# PyTorch 2.6+ defaults torch.load to weights_only=True, which blocks unpickling
# numpy scalar types (omega_bar dicts contain np.float64 values). Allowlist them
# explicitly rather than disabling weights_only globally (keeps the safety check
# for everything else).
torch.serialization.add_safe_globals([np._core.multiarray.scalar, np.dtype, np.float64])

def fairfed_state_path(tag, seed):
    return f"{CKPTS}/{tag}_seed{seed}_INPROGRESS.pt"

def fairfed_meta_path(tag, seed):
    return f"{RESULTS}/{tag}_seed{seed}_progress.json"

def run_fairfed(seeds, rounds=FAIRFED_ROUNDS, beta=FAIRFED_BETA, tag='fairfed',
                local_epochs=FAIRFED_LOCAL_EPOCHS, verbose=True):
    pooled_val = pd.concat([pd.read_csv(f'{CLIENTS}/client_{c}_val.csv')
                            for c in CLIENT_LIST], ignore_index=True)
    pooled_val.to_csv('/content/pooled_val_fairfed.csv', index=False)

    all_results = []
    for seed in seeds:
        final_ckpt = f"{CKPTS}/{tag}_seed{seed}.pt"
        if os.path.exists(final_ckpt):
            if verbose: print(f"  [{tag} s{seed}] FINAL checkpoint exists -> loading for test")
            global_state = torch.load(final_ckpt, map_location='cpu', weights_only=False)
        else:
            torch.manual_seed(seed); np.random.seed(seed)
            ctl = {c: make_loader(f'{CLIENTS}/client_{c}_train.csv', train=True) for c in CLIENT_LIST}
            cs = {c: len(ctl[c].dataset) for c in CLIENT_LIST}
            pvl = make_loader('/content/pooled_val_fairfed.csv', train=False)
            cvl = {c: make_loader(f'{CLIENTS}/client_{c}_val.csv', train=False) for c in CLIENT_LIST}  # built ONCE per seed, reused every round

            # ---- RESUME LOGIC: load in-progress state if a crash happened ----
            inprog_path = fairfed_state_path(tag, seed)
            meta_path = fairfed_meta_path(tag, seed)
            if os.path.exists(inprog_path) and os.path.exists(meta_path):
                ckpt = torch.load(inprog_path, map_location='cpu', weights_only=False)
                global_state = ckpt['global_state']
                omega_bar = ckpt['omega_bar']
                with open(meta_path) as f: meta = json.load(f)
                start_round = meta['last_completed_round']
                best_auroc, best_gs = meta['best_auroc'], None
                best_gs_path = f"{CKPTS}/{tag}_seed{seed}_BEST.pt"
                if os.path.exists(best_gs_path):
                    best_gs = torch.load(best_gs_path, map_location='cpu', weights_only=False)
                if verbose: print(f"  [{tag} s{seed}] RESUMING from round {start_round+1}/{rounds}")
            else:
                gm = build_model(device)
                global_state = {k: v.cpu().clone() for k, v in gm.state_dict().items()}
                del gm; torch.cuda.empty_cache()
                # Eq.6 init: omega_bar_k^0 = n_k / sum(n_i) (standard FedAvg prior)
                total_n = sum(cs.values())
                omega_bar = {c: cs[c] / total_n for c in CLIENT_LIST}
                start_round, best_auroc, best_gs = 0, -1.0, None

            for rnd in range(start_round, rounds):
                t0 = time.time()

                # 1) local training (plain FedAvg-style local SGD, no special loss)
                states, deltas = {}, {}
                for c in CLIENT_LIST:
                    st = local_train(global_state, ctl[c], device, local_epochs)
                    states[c] = st

                # 2) per-client F_k on the model BEFORE this round's aggregation
                #    (paper computes deltas using theta^{t-1}; we approximate with
                #    the freshly-trained local models' own client-side eval, which
                #    is what each client can compute without extra communication)
                gm = build_model(device); gm.load_state_dict(global_state); gm.eval()
                f_global_m = evaluate_dual(gm, pvl, device, 0.5)
                f_global = f_global_m['eo_gap']
                acc_global = 1.0 - f_global_m.get('fpr_gap', 0.0)
                del gm; gc.collect(); torch.cuda.empty_cache()

                for c in CLIENT_LIST:
                    gm = build_model(device); gm.load_state_dict(states[c]); gm.eval()
                    cm = evaluate_dual(gm, cvl[c], device, 0.5)
                    f_k = cm['eo_gap']
                    acc_k = 1.0 - cm.get('fpr_gap', 0.0)
                    if np.isnan(f_k):
                        deltas[c] = abs(acc_k - acc_global)   # Eq.6 undefined-F_k branch
                    else:
                        deltas[c] = abs(f_global - f_k)        # Eq.6 main branch
                    del gm; gc.collect(); torch.cuda.empty_cache()

                # 3) Eq.6 weight update
                omega_bar, omega = fairfed_update_weights(omega_bar, deltas, beta)

                # 4) weighted aggregation (float-only, matches your fedavg_aggregate fix)
                new_state = {}
                for k, v in states[CLIENT_LIST[0]].items():
                    if torch.is_floating_point(v):
                        new_state[k] = sum(states[c][k] * omega[c] for c in CLIENT_LIST)
                    else:
                        new_state[k] = v.clone()
                global_state = new_state
                del states; gc.collect()

                # 5) eval + CHECKPOINT EVERY ROUND (the crash-resilience point)
                gm = build_model(device); gm.load_state_dict(global_state)
                vm = evaluate_dual(gm, pvl, device, 0.5); del gm; gc.collect(); torch.cuda.empty_cache()
                if vm['auroc'] > best_auroc:
                    best_auroc = vm['auroc']
                    torch.save(global_state, f"{CKPTS}/{tag}_seed{seed}_BEST.pt")

                torch.save({'global_state': global_state, 'omega_bar': omega_bar}, inprog_path)
                with open(meta_path, 'w') as f:
                    json.dump({'last_completed_round': rnd + 1, 'best_auroc': best_auroc}, f)

                if verbose:
                    print(f"  R{rnd+1:02d}/{rounds} ({(time.time()-t0)/60:.1f}m) "
                          f"AUROC={vm['auroc']:.4f} EO={vm['eo_gap']:.4f} "
                          f"omega={[round(omega[c],3) for c in CLIENT_LIST]}")

            # finished all rounds for this seed -> promote BEST to FINAL, clean up
            global_state = torch.load(f"{CKPTS}/{tag}_seed{seed}_BEST.pt", map_location='cpu', weights_only=False)
            torch.save(global_state, final_ckpt)
            for p in [inprog_path, meta_path, f"{CKPTS}/{tag}_seed{seed}_BEST.pt"]:
                if os.path.exists(p): os.remove(p)
            del ctl, pvl, cvl; gc.collect(); torch.cuda.empty_cache()
            if verbose: print(f"  [{tag} s{seed}] COMPLETE, best AUROC={best_auroc:.4f}, saved -> {final_ckpt}")

        # test
        gm = build_model(device); gm.load_state_dict(global_state)
        for c in CLIENT_LIST:
            te = make_loader(f'{CLIENTS}/client_{c}_test.csv', train=False)
            tm = evaluate_dual(gm, te, device, 0.5)
            tm.update({'method': tag, 'client': c, 'seed': seed})
            all_results.append(tm); del te; gc.collect()
        del gm; gc.collect(); torch.cuda.empty_cache()
        pd.DataFrame(all_results).to_csv(f'{RESULTS}/{tag}_all.csv', index=False)
    return all_results

print("FairFed crash-resilient driver defined. Checkpoints every round.")
print("If your session disconnects mid-run, just re-run this cell + the launch")
print("cell below -- it will detect the in-progress checkpoint and resume.")

FairFed crash-resilient driver defined. Checkpoints every round.
If your session disconnects mid-run, just re-run this cell + the launch
cell below -- it will detect the in-progress checkpoint and resume.


In [ ]:
#SOTA test 4

# ============================================================================
# FAIRFED CELL 4 — FULL RUN, 5 seeds x 30 rounds. SAFE TO RE-RUN if your
# session crashes mid-way -- it will auto-resume each seed from its last
# completed round (checkpointed every round in Cell 2). Just re-run THIS
# cell again after reconnecting; no flag needed (unlike FedSRA's gate),
# since each round is cheap enough and resume is automatic and safe.
# ============================================================================
RUN_FULL_FAIRFED = True   # <-- set True when ready

if RUN_FULL_FAIRFED:
    print(f"Launching FairFed: {len(SEEDS)} seeds x {FAIRFED_ROUNDS} rounds, beta={FAIRFED_BETA}")
    print("If this session disconnects, just re-run this cell after reconnecting --")
    print("it resumes automatically from the last completed round per seed.")
    fairfed_results = run_fairfed(seeds=SEEDS, rounds=FAIRFED_ROUNDS, beta=FAIRFED_BETA,
                                  tag='fairfed', verbose=True)
    import pandas as pd
    res = pd.DataFrame(fairfed_results)
    print(f"\n{'='*70}\nFairFed FULL — test metrics (mean +/- std across {len(SEEDS)} seeds)\n{'='*70}")
    def ms(a, s): return f"{a:.4f}\u00b1{s:.4f}"
    for c in CLIENT_LIST:
        sub = res[res['client'] == c]
        print(f"  {c}: AUROC={ms(sub.auroc.mean(),sub.auroc.std())}  EO={ms(sub.eo_gap.mean(),sub.eo_gap.std())}")
    print(f"\nSaved -> {RESULTS}/fairfed_all.csv")
else:
    print("RUN_FULL_FAIRFED is False. Set True after the debug run looks healthy.")


Launching FairFed: 5 seeds x 30 rounds, beta=1.0
If this session disconnects, just re-run this cell after reconnecting --
it resumes automatically from the last completed round per seed.
  [fairfed s42] FINAL checkpoint exists -> loading for test
Downloading: "https://download.pytorch.org/models/densenet121-a639ec97.pth" to /root/.cache/torch/hub/checkpoints/densenet121-a639ec97.pth


100%|██████████| 30.8M/30.8M [00:00<00:00, 254MB/s]


  [fairfed s123] FINAL checkpoint exists -> loading for test
  [fairfed s456] FINAL checkpoint exists -> loading for test
  [fairfed s789] FINAL checkpoint exists -> loading for test
  [fairfed s1010] RESUMING from round 29/30
  R29/30 (5.6m) AUROC=0.9457 EO=0.0203 omega=[np.float64(0.554), np.float64(0.001), np.float64(0.144), np.float64(0.3)]
  R30/30 (5.0m) AUROC=0.9483 EO=0.0252 omega=[np.float64(0.569), np.float64(0.0), np.float64(0.158), np.float64(0.273)]
  [fairfed s1010] COMPLETE, best AUROC=0.9531, saved -> /content/drive/MyDrive/FairFedCXR/checkpoints/fairfed_seed1010.pt

FairFed FULL — test metrics (mean +/- std across 5 seeds)
  A: AUROC=0.9516±0.0011  EO=0.0156±0.0111
  B: AUROC=0.9496±0.0025  EO=0.0080±0.0089
  C: AUROC=0.9368±0.0008  EO=0.0113±0.0073
  D: AUROC=0.9631±0.0021  EO=0.0387±0.0247

Saved -> /content/drive/MyDrive/FairFedCXR/results/fairfed_all.csv


In [ ]:
# ============================================================================
# FAIRFED BETA SENSITIVITY — beta=0.1, seed 42, PER-ROUND OMEGA LOGGING
#
# Purpose: test whether FairFed's weight collapse is specific to the
# author-recommended beta=1 or is structural to the unbounded additive update.
#
# Matches the beta=1 runs exactly except for beta: same local_train, same
# 2 local epochs, same delta definition, same 1e-6 floor, same seeding.
#
# RESUMABLE: set FF_ROUNDS=30 and re-run to extend from wherever it stopped.
#
# Requires: Cell 61 -> Cell 62 -> Cell 103 (local_train)
# ============================================================================
import torch, copy, gc, time, os, json, numpy as np, pandas as pd

device = 'cuda' if torch.cuda.is_available() else 'cpu'
CLIENT_LIST = ['A', 'B', 'C', 'D']

# ---- config ----
FF_BETA         = 0.1
FF_SEED         = 42
FF_ROUNDS       = 30                 # <-- bump to 30 and re-run to extend
FF_LOCAL_EPOCHS = LOCAL_EPOCHS       # = 2, matched to the beta=1 runs
FF_FLOOR        = 1e-6               # same floor as the original implementation
FF_TAG          = f'fairfed_b{FF_BETA}'

LOG_CSV   = f'{RESULTS}/{FF_TAG}_seed{FF_SEED}_weightlog.csv'
STATE_PT  = f'{CKPTS}/{FF_TAG}_seed{FF_SEED}_INPROGRESS.pt'
META_JSON = f'{RESULTS}/{FF_TAG}_seed{FF_SEED}_progress.json'

torch.serialization.add_safe_globals([np._core.multiarray.scalar, np.dtype, np.float64])

print(f"FairFed beta-sensitivity | beta={FF_BETA} seed={FF_SEED} "
      f"rounds={FF_ROUNDS} local_epochs={FF_LOCAL_EPOCHS}")

# ---- Eq.(6) additive update. Identical to the original, but ALSO carries an
#      unfloored parallel recursion so we can see how far below zero the update
#      would push a client if the 1e-6 floor were not there. The floored value
#      is what drives aggregation, so training is unchanged. ----
def ff_update(prev_bar, prev_bar_raw, deltas, beta, floor=FF_FLOOR):
    mean_d = float(np.mean(list(deltas.values())))
    raw    = {k: prev_bar_raw[k] - beta * (deltas[k] - mean_d) for k in prev_bar_raw}
    new    = {k: max(prev_bar[k] - beta * (deltas[k] - mean_d), floor) for k in prev_bar}
    tot    = sum(new.values())
    omega  = {k: new[k] / tot for k in new}
    return new, raw, omega

# ---- data ----
torch.manual_seed(FF_SEED); np.random.seed(FF_SEED)   # matches original run_fairfed

pooled_val = pd.concat([pd.read_csv(f'{CLIENTS}/client_{c}_val.csv')
                        for c in CLIENT_LIST], ignore_index=True)
pooled_val.to_csv('/content/pooled_val_ffbeta.csv', index=False)

ctl = {c: make_loader(f'{CLIENTS}/client_{c}_train.csv', train=True)  for c in CLIENT_LIST}
cvl = {c: make_loader(f'{CLIENTS}/client_{c}_val.csv',   train=False) for c in CLIENT_LIST}
pvl = make_loader('/content/pooled_val_ffbeta.csv', train=False)

cs      = {c: len(ctl[c].dataset) for c in CLIENT_LIST}
total_n = sum(cs.values())
prior   = {c: cs[c] / total_n for c in CLIENT_LIST}     # FedAvg prior p_i
print(f"FedAvg priors: {[f'{c}={prior[c]:.3f}' for c in CLIENT_LIST]}")

# ---- resume ----
if os.path.exists(STATE_PT) and os.path.exists(META_JSON):
    ck = torch.load(STATE_PT, map_location='cpu', weights_only=False)
    global_state  = ck['global_state']
    omega_bar     = ck['omega_bar']
    omega_bar_raw = ck['omega_bar_raw']
    with open(META_JSON) as f: meta = json.load(f)
    start_round = meta['last_completed_round']
    rows = pd.read_csv(LOG_CSV).to_dict('records') if os.path.exists(LOG_CSV) else []
    rows = [r for r in rows if r['round'] <= start_round]
    print(f"RESUMING from round {start_round + 1}/{FF_ROUNDS}")
else:
    gm = build_model(device)
    global_state = {k: v.cpu().clone() for k, v in gm.state_dict().items()}
    del gm; torch.cuda.empty_cache()
    omega_bar     = dict(prior)     # Eq.6 init: omega_bar^0 = n_k / N
    omega_bar_raw = dict(prior)
    start_round, rows = 0, []
    print("Starting fresh from round 1")

if start_round >= FF_ROUNDS:
    print(f"Already completed {start_round} rounds. Raise FF_ROUNDS to extend.")
else:
    for rnd in range(start_round, FF_ROUNDS):
        t0 = time.time()

        # 1) local training on every client (plain FedAvg-style SGD)
        states = {c: local_train(global_state, ctl[c], device, FF_LOCAL_EPOCHS)
                  for c in CLIENT_LIST}

        # 2) global gap on the PRE-aggregation global model
        gm = build_model(device); gm.load_state_dict(global_state); gm.eval()
        f_global = evaluate_dual(gm, pvl, device, 0.5)['eo_gap']
        acc_global = 1.0 - evaluate_dual(gm, pvl, device, 0.5).get('fpr_gap', 0.0)
        del gm; gc.collect(); torch.cuda.empty_cache()

        # 3) per-client delta_k = |F_global - F_k|
        deltas, f_locals = {}, {}
        for c in CLIENT_LIST:
            gm = build_model(device); gm.load_state_dict(states[c]); gm.eval()
            cm = evaluate_dual(gm, cvl[c], device, 0.5)
            f_k = cm['eo_gap']
            f_locals[c] = f_k
            if np.isnan(f_k):
                deltas[c] = abs((1.0 - cm.get('fpr_gap', 0.0)) - acc_global)
            else:
                deltas[c] = abs(f_global - f_k)
            del gm; gc.collect(); torch.cuda.empty_cache()

        # 4) Eq.(6) weight update
        omega_bar, omega_bar_raw, omega = ff_update(
            omega_bar, omega_bar_raw, deltas, FF_BETA)

        # 5) weighted aggregation
        new_state = {}
        for k, v in states[CLIENT_LIST[0]].items():
            if torch.is_floating_point(v):
                new_state[k] = sum(states[c][k] * omega[c] for c in CLIENT_LIST)
            else:
                new_state[k] = v.clone()
        global_state = new_state
        del states; gc.collect()

        # 6) eval new global
        gm = build_model(device); gm.load_state_dict(global_state)
        vm = evaluate_dual(gm, pvl, device, 0.5)
        del gm; gc.collect(); torch.cuda.empty_cache()

        # 7) LOG EVERY CLIENT, EVERY ROUND  <-- the point of this run
        for c in CLIENT_LIST:
            rows.append({
                'method': 'fairfed', 'beta': FF_BETA, 'seed': FF_SEED,
                'round': rnd + 1, 'client': c,
                'local_eo_gap': f_locals[c], 'global_eo_gap': f_global,
                'delta': deltas[c],
                'omega_bar_raw': omega_bar_raw[c],   # unfloored: can go negative
                'omega_bar': omega_bar[c],           # floored at 1e-6
                'omega': omega[c],                   # normalized aggregation weight
                'fedavg_prior': prior[c],
                'pooled_val_auroc': vm['auroc'],
                'pooled_val_eo_gap': vm['eo_gap'],
            })
        pd.DataFrame(rows).to_csv(LOG_CSV, index=False)

        # 8) checkpoint every round
        torch.save({'global_state': global_state, 'omega_bar': omega_bar,
                    'omega_bar_raw': omega_bar_raw}, STATE_PT)
        with open(META_JSON, 'w') as f:
            json.dump({'last_completed_round': rnd + 1}, f)

        print(f"  R{rnd+1:02d}/{FF_ROUNDS} ({(time.time()-t0)/60:.1f}m) "
              f"AUROC={vm['auroc']:.4f} EO={vm['eo_gap']:.4f} | "
              f"omega={[round(omega[c], 4) for c in CLIENT_LIST]}")

    print(f"\nDone. Weight log -> {LOG_CSV}")

# ---- summary: did any client collapse? ----
log = pd.read_csv(LOG_CSV)
print(f"\n{'='*72}\nFAIRFED beta={FF_BETA} — WEIGHT TRAJECTORY SUMMARY (seed {FF_SEED})\n{'='*72}")
print(f"  {'client':>7} {'prior':>8} {'final w':>9} {'min w':>9} {'ratio':>9} {'raw bar':>10}")
for c in CLIENT_LIST:
    s = log[log['client'] == c].sort_values('round')
    fin, mn = s['omega'].iloc[-1], s['omega'].min()
    raw = s['omega_bar_raw'].iloc[-1]
    print(f"  {c:>7} {prior[c]:>8.3f} {fin:>9.4f} {mn:>9.4f} "
          f"{fin/prior[c]:>9.2f}x {raw:>10.4f}")
print("\n  ratio = final weight / FedAvg prior. Collapse ~ ratio << 1.")
print("  raw bar < 0 means the additive update wanted to go negative and was")
print("  only stopped by the 1e-6 floor, i.e. the update is genuinely unbounded.")
print(f"\n  Compare to DWFA's proven floor: no client can fall below "
      f"q_min*p_i = 0.46*p_i.")
for c in CLIENT_LIST:
    print(f"    Client {c}: DWFA floor = {0.46*prior[c]:.4f} | "
          f"FairFed b={FF_BETA} min = {log[log['client']==c]['omega'].min():.4f}")

FairFed beta-sensitivity | beta=0.1 seed=42 rounds=30 local_epochs=2
FedAvg priors: ['A=0.418', 'B=0.223', 'C=0.192', 'D=0.167']
Starting fresh from round 1
  R01/30 (5.9m) AUROC=0.9471 EO=0.0346 | omega=[0.4207, 0.2225, 0.1883, 0.1684]
  R02/30 (5.1m) AUROC=0.9521 EO=0.0494 | omega=[0.4217, 0.2224, 0.1858, 0.1701]
  R03/30 (5.0m) AUROC=0.9530 EO=0.0364 | omega=[0.4214, 0.2224, 0.184, 0.1722]
  R04/30 (5.0m) AUROC=0.9526 EO=0.0321 | omega=[0.4226, 0.2206, 0.1823, 0.1745]
  R05/30 (5.0m) AUROC=0.9519 EO=0.0393 | omega=[0.4231, 0.2213, 0.1814, 0.1742]
  R06/30 (5.0m) AUROC=0.9512 EO=0.0309 | omega=[0.4235, 0.2211, 0.1821, 0.1733]
  R07/30 (5.0m) AUROC=0.9493 EO=0.0332 | omega=[0.4236, 0.2234, 0.1786, 0.1744]
  R08/30 (5.0m) AUROC=0.9494 EO=0.0307 | omega=[0.4247, 0.2226, 0.1772, 0.1755]
  R09/30 (5.0m) AUROC=0.9492 EO=0.0266 | omega=[0.4236, 0.2233, 0.1773, 0.1757]
  R10/30 (5.0m) AUROC=0.9491 EO=0.0368 | omega=[0.424, 0.2197, 0.1801, 0.1762]
  R11/30 (5.0m) AUROC=0.9489 EO=0.0139 | omeg

In [ ]:
import pandas as pd, torch, os
from torch.utils.data import Dataset, DataLoader
from PIL import Image

class NihDataset(Dataset):
    def __init__(self, df, transform):
        self.df = df.reset_index(drop=True); self.tf = transform
    def __len__(self): return len(self.df)
    def __getitem__(self, idx):
        r = self.df.iloc[idx]
        img = Image.open(os.path.join(NIH_IMG, r['Path'])).convert('RGB')
        return self.tf(img), torch.tensor(r['label'], dtype=torch.float32), torch.tensor(r['sex_encoded'], dtype=torch.long)

def make_nih_loader(df):
    on_gpu = torch.cuda.is_available()
    return DataLoader(NihDataset(df, eval_tf), batch_size=BATCH_SIZE, shuffle=False,
                      num_workers=4 if on_gpu else 2, pin_memory=on_gpu, persistent_workers=False)

print("make_nih_loader defined, ready for evaluation cell")

make_nih_loader defined, ready for evaluation cell


In [ ]:
# ============================================================================
# REPAIR CELL — fixes the sex-array corruption caused by an earlier (now-fixed)
# bug where sex got saved as a copy of labels. Rebuilds 'sex' from the real
# source CSVs, verifies the corruption AND the fix, re-saves both caches.
# 'labels' and 'all_probs' are untouched -- only 'sex' was ever corrupted.
# Needs: RESULTS, CKPTS, CLIENTS (Cell 56). NIH side also needs the cohort CSV
# (no GPU, no NIH Cell 1/2 needed -- this just re-reads the CSV that's already
# on Drive).
# ============================================================================
import numpy as np, pandas as pd, os

def diagnose_and_repair(cache_path, true_sex_source_fn, name):
    if not os.path.exists(cache_path):
        print(f"  {name}: cache not found, skip"); return
    cache = np.load(cache_path, allow_pickle=True)
    all_probs = cache['all_probs'].item()
    labels = cache['labels']
    sex = cache['sex']

    corrupted = np.array_equal(labels.astype(int), sex.astype(int))
    print(f"  {name}: sex==labels (corrupted)? {corrupted}")
    if not corrupted:
        print(f"  {name}: sex array looks fine, no repair needed."); return

    true_sex = true_sex_source_fn()
    assert len(true_sex) == len(labels), \
        f"{name}: length mismatch, true_sex={len(true_sex)} vs labels={len(labels)} -- row order may differ, DO NOT trust this repair blindly, verify manually."

    # sanity check: real sex should NOT equal labels (a genuine different signal)
    still_matches = np.array_equal(true_sex.astype(int), labels.astype(int))
    print(f"  {name}: rebuilt sex still equals labels? {still_matches} (should be False)")
    frac_male = true_sex.mean()
    print(f"  {name}: rebuilt sex - fraction coded 1 (male): {frac_male:.3f} (sanity: should be well away from prevalence {labels.mean():.3f})")

    np.savez(cache_path, all_probs=all_probs, labels=labels, sex=true_sex.astype(int))
    print(f"  {name}: REPAIRED and re-saved -> {cache_path}\n")

# ---- NIH repair: real sex comes from the cohort CSV ----
def nih_true_sex():
    cohort = pd.read_csv(f'{RESULTS}/nih_effusion_cohort.csv')
    assert 'sex_encoded' in cohort.columns, "cohort CSV missing sex_encoded column"
    return cohort['sex_encoded'].values

# ---- CheXpert repair: real sex comes from the pooled client test CSVs, in the
# SAME order (A,B,C,D) used when predictions were originally cached ----
def chexpert_true_sex():
    parts = [pd.read_csv(f'{CLIENTS}/client_{c}_test.csv') for c in ['A','B','C','D']]
    pooled = pd.concat(parts, ignore_index=True)
    assert 'sex_encoded' in pooled.columns, "client test CSV missing sex_encoded column"
    return pooled['sex_encoded'].values

print("="*78); print("DIAGNOSING AND REPAIRING sex-array corruption"); print("="*78)
diagnose_and_repair(f'{RESULTS}/nih_pred_cache.npz', nih_true_sex, 'NIH')
diagnose_and_repair(f'{RESULTS}/chexpert_pred_cache.npz', chexpert_true_sex, 'CheXpert')
print("Done. Re-run Part 3 and Part 4 of the evaluation cell now to get correct numbers.")

DIAGNOSING AND REPAIRING sex-array corruption
  NIH: sex==labels (corrupted)? True
  NIH: rebuilt sex still equals labels? False (should be False)
  NIH: rebuilt sex - fraction coded 1 (male): 0.565 (sanity: should be well away from prevalence 0.119)
  NIH: REPAIRED and re-saved -> /content/drive/MyDrive/FairFedCXR/results/nih_pred_cache.npz

  CheXpert: sex==labels (corrupted)? True
  CheXpert: rebuilt sex still equals labels? False (should be False)
  CheXpert: rebuilt sex - fraction coded 1 (male): 0.548 (sanity: should be well away from prevalence 0.403)
  CheXpert: REPAIRED and re-saved -> /content/drive/MyDrive/FairFedCXR/results/chexpert_pred_cache.npz

Done. Re-run Part 3 and Part 4 of the evaluation cell now to get correct numbers.


In [ ]:
# ============================================================================
# COMPLETE POST-TRAINING EVALUATION — run this ONE cell after FairFed's full
# 5-seed run finishes. Fills every gap left by the earlier full_eval_metrics
# cell: adds FairFed to both NIH and CheXpert prediction caches, computes
# mean+/-std across seeds (not just seed-averaged points), and runs the full
# paired-significance battery (AUROC, WG-AUROC, EO-gap@.5/@YJ, ECE-gap) for
# DWFA vs FairFed AND q-FedAvg vs FairFed, on BOTH datasets.
# ============================================================================
import torch, gc, os, numpy as np, pandas as pd
from sklearn.metrics import roc_auc_score, roc_curve

EPS_M, ECE_BINS_M = 1.0, 8
CLIENT_LIST = ['A', 'B', 'C', 'D']
FAIRFED_SEEDS = [42, 123, 456, 789, 1010]

def safe_auroc(lab, prob):
    if len(lab) < 2 or lab.sum() < 1 or (lab == 0).sum() < 1: return float('nan')
    try: return float(roc_auc_score(lab, prob))
    except Exception: return float('nan')

def stpr(prob, lab, thr):
    preds = (prob >= thr).astype(int); pos = (lab == 1)
    return (((preds == 1) & pos).sum() + EPS_M) / (pos.sum() + 2*EPS_M)

def sfpr(prob, lab, thr):
    preds = (prob >= thr).astype(int); neg = (lab == 0)
    return (((preds == 1) & neg).sum() + EPS_M) / (neg.sum() + 2*EPS_M)

def ece(prob, lab, M=ECE_BINS_M):
    bins = np.linspace(0, 1, M+1); e = 0.0
    for lo, hi in zip(bins[:-1], bins[1:]):
        m = (prob >= lo) & (prob < hi)
        if m.sum() == 0: continue
        e += m.mean() * abs(lab[m].mean() - prob[m].mean())
    return float(e)

def youden(prob, lab):
    fpr, tpr, thr = roc_curve(lab, prob); return float(thr[int(np.argmax(tpr - fpr))])

def wg_auroc(prob, lab, sx):
    a_m, a_f = safe_auroc(lab[sx==1], prob[sx==1]), safe_auroc(lab[sx==0], prob[sx==0])
    if np.isnan(a_m) and np.isnan(a_f): return float('nan')
    return float(np.nanmin([a_m, a_f]))

def eo_gap(prob, lab, sx, thr):
    return abs(stpr(prob[sx==1],lab[sx==1],thr) - stpr(prob[sx==0],lab[sx==0],thr))

def fpr_gap(prob, lab, sx, thr):
    return abs(sfpr(prob[sx==1],lab[sx==1],thr) - sfpr(prob[sx==0],lab[sx==0],thr))

def ece_gap(prob, lab, sx):
    return abs(ece(prob[sx==1],lab[sx==1]) - ece(prob[sx==0],lab[sx==0]))

# ============================================================================
# PART 1 — extend NIH cache with FairFed (inference, needs GPU + checkpoints)
# ============================================================================
print("="*78); print("PART 1: caching FairFed predictions on NIH"); print("="*78)
nih_cache_path = f'{RESULTS}/nih_pred_cache.npz'
nih_cohort_csv = f'{RESULTS}/nih_effusion_cohort.csv'
need_nih_fairfed = True
if os.path.exists(nih_cache_path):
    _c = np.load(nih_cache_path, allow_pickle=True)
    _probs = _c['all_probs'].item()
    if any(m == 'fairfed' for m, s in _probs.keys()):
        print("  FairFed already cached for NIH -- skipping inference."); need_nih_fairfed = False

if need_nih_fairfed:
    if not os.path.exists(nih_cohort_csv):
        print("  nih_effusion_cohort.csv not found -- run NIH Cells 1-2 first. Skipping NIH side.")
    else:
        cohort = pd.read_csv(nih_cohort_csv)
        @torch.no_grad()
        def infer_nih(ckpt):
            gm = build_model(device); gm.load_state_dict(torch.load(ckpt, map_location='cpu', weights_only=False)); gm.eval()
            loader = make_nih_loader(cohort) if 'make_nih_loader' in globals() else None
            if loader is None:
                raise RuntimeError("make_nih_loader not defined -- run NIH cells 1-3 first this session.")
            P, L, S = [], [], []
            for imgs, labels, sex in loader:
                with torch.amp.autocast('cuda'):
                    logits = gm(imgs.to(device)).squeeze()
                P.append(torch.sigmoid(logits.float()).cpu().numpy()); L.append(labels.numpy()); S.append(sex.numpy())
            del gm, loader; gc.collect(); torch.cuda.empty_cache()
            return np.concatenate(P), np.concatenate(L), np.concatenate(S)

        old = np.load(nih_cache_path, allow_pickle=True) if os.path.exists(nih_cache_path) else None
        all_probs = old['all_probs'].item() if old is not None else {}
        labels, sex = (old['labels'], old['sex']) if old is not None else (None, None)
        for seed in FAIRFED_SEEDS:
            ckpt = f'{CKPTS}/fairfed_seed{seed}.pt'
            if not os.path.exists(ckpt):
                print(f"  [fairfed s{seed}] checkpoint missing -> skip"); continue
            p, l, s = infer_nih(ckpt)
            all_probs[('fairfed', seed)] = p
            if labels is None: labels, sex = l, s
            print(f"  cached fairfed s{seed}")
        np.savez(nih_cache_path, all_probs=all_probs, labels=labels, sex=sex)
        print(f"  Saved -> {nih_cache_path}")

# ============================================================================
# PART 2 — extend CheXpert pooled cache with FairFed (inference, needs GPU)
# ============================================================================
print("\n" + "="*78); print("PART 2: caching FairFed predictions on CheXpert (pooled test)"); print("="*78)
cx_cache_path = f'{RESULTS}/chexpert_pred_cache.npz'
need_cx_fairfed = True
if os.path.exists(cx_cache_path):
    _c = np.load(cx_cache_path, allow_pickle=True)
    if any(m == 'fairfed' for m, s in _c['all_probs'].item().keys()):
        print("  FairFed already cached for CheXpert -- skipping."); need_cx_fairfed = False

if need_cx_fairfed:
    @torch.no_grad()
    def predict_pooled_test_cx(ckpt):
        gm = build_model(device); gm.load_state_dict(torch.load(ckpt, map_location='cpu', weights_only=False)); gm.eval()
        P, L, S = [], [], []
        for c in CLIENT_LIST:
            loader = make_loader(f'{CLIENTS}/client_{c}_test.csv', train=False)
            for imgs, labels, sex in loader:
                with torch.amp.autocast('cuda'):
                    logits = gm(imgs.to(device)).squeeze()
                P.append(torch.sigmoid(logits.float()).cpu().numpy()); L.append(labels.numpy()); S.append(sex.numpy())
            del loader
        del gm; gc.collect(); torch.cuda.empty_cache()
        return np.concatenate(P), np.concatenate(L), np.concatenate(S)

    old = np.load(cx_cache_path, allow_pickle=True) if os.path.exists(cx_cache_path) else None
    all_probs_cx = old['all_probs'].item() if old is not None else {}
    labels_cx, sex_cx = (old['labels'], old['sex']) if old is not None else (None, None)
    for seed in FAIRFED_SEEDS:
        ckpt = f'{CKPTS}/fairfed_seed{seed}.pt'
        if not os.path.exists(ckpt):
            print(f"  [fairfed s{seed}] checkpoint missing -> skip"); continue
        p, l, s = predict_pooled_test_cx(ckpt)
        all_probs_cx[('fairfed', seed)] = p
        if labels_cx is None: labels_cx, sex_cx = l, s
        print(f"  cached fairfed s{seed}  (n={len(p)})")
    np.savez(cx_cache_path, all_probs=all_probs_cx, labels=labels_cx, sex=sex_cx)
    print(f"  Saved -> {cx_cache_path}")

# ============================================================================
# PART 3 — mean+/-std per-seed metrics table (NIH and CheXpert), ALL methods
# ============================================================================
def per_seed_table(cache_path, dataset_name):
    if not os.path.exists(cache_path):
        print(f"  {cache_path} not found, skipping {dataset_name}."); return None, None
    cache = np.load(cache_path, allow_pickle=True)
    all_probs = cache['all_probs'].item(); labels = cache['labels'].astype(int); sex = cache['sex'].astype(int)
    methods = sorted(set(m for m, s in all_probs.keys()))
    rows = []
    for method in methods:
        for (m, seed), p in all_probs.items():
            if m != method: continue
            yj = youden(p, labels)
            rows.append({
                'dataset': dataset_name, 'method': method, 'seed': seed,
                'auroc': safe_auroc(labels, p), 'wg_auroc': wg_auroc(p, labels, sex),
                'eo_gap_50': eo_gap(p, labels, sex, 0.5), 'eo_gap_yj': eo_gap(p, labels, sex, yj),
                'fpr_gap_50': fpr_gap(p, labels, sex, 0.5),
                'ece_gap': ece_gap(p, labels, sex),
            })
    df = pd.DataFrame(rows)
    numeric_cols = ['auroc', 'wg_auroc', 'eo_gap_50', 'eo_gap_yj', 'fpr_gap_50', 'ece_gap']
    summary = df.groupby('method')[numeric_cols].agg(['mean', 'std']).round(4)
    return df, summary

print("\n" + "="*78); print("PART 3: per-seed mean +/- std, ALL methods, BOTH datasets"); print("="*78)
pd.set_option('display.width', 200); pd.set_option('display.max_columns', 30)
nih_df, nih_summary = (None, None)
if os.path.exists(nih_cache_path):
    nih_df, nih_summary = per_seed_table(nih_cache_path, 'NIH')
    nih_df.to_csv(f'{RESULTS}/full_metrics_per_seed_NIH.csv', index=False)
    print("\n--- NIH (mean +/- std across seeds) ---")
    print(nih_summary)
cx_df, cx_summary = (None, None)
if os.path.exists(cx_cache_path):
    cx_df, cx_summary = per_seed_table(cx_cache_path, 'CheXpert')
    cx_df.to_csv(f'{RESULTS}/full_metrics_per_seed_CheXpert.csv', index=False)
    print("\n--- CheXpert pooled (mean +/- std across seeds) ---")
    print(cx_summary)

# ============================================================================
# PART 4 — paired bootstrap significance, DWFA vs FairFed AND qFedAvg vs FairFed
# ============================================================================
def paired_bootstrap(cache_path, method_a, method_b, metric_fn, n_boot=3000, seed0=10):
    cache = np.load(cache_path, allow_pickle=True)
    all_probs = cache['all_probs'].item(); labels = cache['labels'].astype(int); sex = cache['sex'].astype(int)
    def mean_probs(method, idx):
        ps = [p[idx] for (m, s), p in all_probs.items() if m == method]
        return np.mean(ps, axis=0) if ps else None
    rng = np.random.default_rng(seed0); n = len(labels)
    diffs = []
    for b in range(n_boot):
        idx = rng.integers(0, n, n)
        pa, pb = mean_probs(method_a, idx), mean_probs(method_b, idx)
        if pa is None or pb is None: return None
        diffs.append(metric_fn(pa, labels[idx], sex[idx]) - metric_fn(pb, labels[idx], sex[idx]))
    return np.array(diffs)

METRIC_FNS = {
    'auroc': lambda p, l, s: safe_auroc(l, p),
    'wg_auroc': wg_auroc,
    'eo_gap_50': lambda p, l, s: eo_gap(p, l, s, 0.5),
    'ece_gap': ece_gap,
}

print("\n" + "="*78); print("PART 4: paired significance, DWFA/qFedAvg vs FairFed"); print("="*78)
for dataset_name, cpath in [('NIH', nih_cache_path), ('CheXpert', cx_cache_path)]:
    if not os.path.exists(cpath): continue
    print(f"\n--- {dataset_name} ---")
    for a in ['dwfa', 'qfedavg']:
        for metric_name, fn in METRIC_FNS.items():
            d = paired_bootstrap(cpath, a, 'fairfed', fn)
            if d is None: continue
            lo, hi = np.percentile(d, 2.5), np.percentile(d, 97.5)
            sig = "SIGNIFICANT" if (lo > 0 or hi < 0) else "not significant"
            print(f"  [{metric_name:>9}] {a} - fairfed: diff={d.mean():+.4f}  CI=[{lo:+.4f},{hi:+.4f}]  {sig}")

print("\nDone. All caches, per-seed tables, and significance tests now include FairFed.")

PART 1: caching FairFed predictions on NIH
  FairFed already cached for NIH -- skipping inference.

PART 2: caching FairFed predictions on CheXpert (pooled test)
  FairFed already cached for CheXpert -- skipping.

PART 3: per-seed mean +/- std, ALL methods, BOTH datasets

--- NIH (mean +/- std across seeds) ---
          auroc         wg_auroc         eo_gap_50         eo_gap_yj         fpr_gap_50         ece_gap        
           mean     std     mean     std      mean     std      mean     std       mean     std    mean     std
method                                                                                                         
dwfa     0.8578  0.0022   0.8557  0.0019    0.0253  0.0071    0.0170  0.0045     0.0179  0.0032  0.0143  0.0035
fairfed  0.8565  0.0024   0.8548  0.0017    0.0231  0.0069    0.0201  0.0058     0.0161  0.0051  0.0128  0.0051
fedavg   0.8571  0.0033   0.8559  0.0024    0.0288  0.0059    0.0244  0.0055     0.0202  0.0023  0.0150  0.0044
qfedavg  0.8554

In [ ]:
# ============================================================================
# DWFA BOUND-COMPLIANCE VERIFICATION — the rigorous empirical half of the
# stability comparison. Uses ONLY the already-saved dwfa_round_log.csv (real
# training data, no rerun, no GPU). For every (seed, round, client) it checks
# whether DWFA's actual weight stayed inside the proven bound
# [q_min * p_i, p_i / q_min] where p_i is that client's FedAvg prior.
# Produces a citable, reproducible compliance result for the paper.
# Needs: RESULTS (Cell 56). Pure pandas, CPU-only.
# ============================================================================
import pandas as pd, numpy as np, os

Q_MIN = 0.46   # DWFA locked hyperparameter (matches LaTeX appendix)

log_path = f'{RESULTS}/dwfa_round_log.csv'
assert os.path.exists(log_path), f"{log_path} not found -- this is the saved DWFA training log."
df = pd.read_csv(log_path)
print(f"Loaded dwfa_round_log.csv: {len(df)} rows")
print(f"Columns: {list(df.columns)}")
print(f"Seeds: {sorted(df['seed'].unique())}  Rounds: {df['round'].min()}-{df['round'].max()}  Clients: {sorted(df['client'].unique())}\n")

# proven bound per row: [q_min * p_i, p_i / q_min], p_i = fedavg_weight
df['bound_lo'] = Q_MIN * df['fedavg_weight']
df['bound_hi'] = df['fedavg_weight'] / Q_MIN
df['inside_bound'] = (df['weight'] >= df['bound_lo'] - 1e-9) & (df['weight'] <= df['bound_hi'] + 1e-9)

n_total = len(df)
n_violations = int((~df['inside_bound']).sum())

print("="*74)
print("DWFA WEIGHT BOUND-COMPLIANCE (proven bound: q_min*p_i <= a_i <= p_i/q_min)")
print("="*74)
print(f"  q_min = {Q_MIN}")
print(f"  Total observations (seeds x rounds x clients): {n_total}")
print(f"  Bound VIOLATIONS: {n_violations}")
print(f"  Compliance rate: {100*(n_total-n_violations)/n_total:.1f}%\n")

# per-client observed range vs proven bound
print(f"  {'client':>7} {'obs min':>9} {'obs max':>9} {'bound lo':>9} {'bound hi':>9} {'violations':>11}")
for c in sorted(df['client'].unique()):
    sub = df[df['client'] == c]
    print(f"  {c:>7} {sub['weight'].min():>9.3f} {sub['weight'].max():>9.3f} "
          f"{sub['bound_lo'].mean():>9.3f} {sub['bound_hi'].mean():>9.3f} "
          f"{int((~sub['inside_bound']).sum()):>11}")

# closest any weight ever got to its floor (the "how close to collapse" margin)
df['margin_to_floor'] = df['weight'] - df['bound_lo']
min_margin_row = df.loc[df['margin_to_floor'].idxmin()]
print(f"\n  Closest any DWFA weight came to its lower bound:")
print(f"    seed {int(min_margin_row['seed'])}, round {int(min_margin_row['round'])}, client {min_margin_row['client']}: "
      f"weight={min_margin_row['weight']:.3f}, floor={min_margin_row['bound_lo']:.3f}, "
      f"margin={min_margin_row['margin_to_floor']:.3f}")
print(f"    (DWFA never approaches zero; its lowest weight stays well above its proven floor)")

df.to_csv(f'{RESULTS}/dwfa_bound_compliance.csv', index=False)
print(f"\nSaved per-row compliance -> {RESULTS}/dwfa_bound_compliance.csv")

print("\n" + "="*74)
print("FOR THE PAPER (rigorous, from saved data):")
print("="*74)
print(f"  'Across all {n_total} weight observations ({df['seed'].nunique()} seeds x "
      f"{df['round'].max()} rounds x {df['client'].nunique()} clients), DWFA's")
print(f"   aggregation weights satisfied the bound of Proposition 3 in every case")
print(f"   ({n_violations} violations), confirming the theoretical guarantee empirically.'")
print("\n  Pair this with the ANALYTICAL argument for FairFed: its additive update")
print("  omega_k -= beta*(Delta_k - mean(Delta)) has no lower bound, so a client whose")
print("  gap persistently exceeds the mean is driven to zero weight (exclusion). This")
print("  is a structural property of the update rule, provable without any rerun.")

Loaded dwfa_round_log.csv: 600 rows
Columns: ['method', 'seed', 'round', 'client', 'val_eo_gap_local', 'e_cur', 'ebar_lagged', 'etilde', 'r_conf', 'weight', 'fedavg_weight', 'pooled_val_auroc', 'pooled_val_eo_gap', 'pooled_val_wg_auroc']
Seeds: [np.int64(42), np.int64(123), np.int64(456), np.int64(789), np.int64(1010)]  Rounds: 1-30  Clients: ['A', 'B', 'C', 'D']

DWFA WEIGHT BOUND-COMPLIANCE (proven bound: q_min*p_i <= a_i <= p_i/q_min)
  q_min = 0.46
  Total observations (seeds x rounds x clients): 600
  Bound VIOLATIONS: 0
  Compliance rate: 100.0%

   client   obs min   obs max  bound lo  bound hi  violations
        A     0.300     0.592     0.192     0.908           0
        B     0.138     0.287     0.103     0.484           0
        C     0.120     0.271     0.088     0.418           0
        D     0.113     0.259     0.077     0.363           0

  Closest any DWFA weight came to its lower bound:
    seed 789, round 1, client C: weight=0.120, floor=0.088, margin=0.032
    (D

In [ ]:
# ============================================================================
# AUDIT PAPER: generate every table from cached data. NO GPU. NO TRAINING.
# Requires: Cell 1 only.
# ============================================================================
import numpy as np, pandas as pd, os
from scipy.stats import ttest_rel
from sklearn.metrics import roc_auc_score

OUT = f'{RESULTS}/audit'
os.makedirs(OUT, exist_ok=True)

c   = np.load(f'{RESULTS}/nih_pred_cache.npz', allow_pickle=True)
P, y, sex = c['all_probs'].item(), c['labels'].astype(int), c['sex']
M, F  = sex == 1, sex == 0
SEEDS = [42, 123, 456, 789, 1010]
METHODS = ['fedavg', 'qfedavg', 'fairfed', 'dwfa']

def au(p, i=slice(None)): return roc_auc_score(y[i], p[i])
def wg(p, i=slice(None)):
    yy, pp, mm, ff = y[i], p[i], M[i], F[i]
    return min(roc_auc_score(yy[mm], pp[mm]), roc_auc_score(yy[ff], pp[ff]))
def eo(p, i=slice(None), t=0.5):
    yy, pp, mm, ff = y[i], p[i], M[i], F[i]
    return abs((pp[mm & (yy==1)] >= t).mean() - (pp[ff & (yy==1)] >= t).mean())
MET = [('AUROC', au), ('WG-AUROC', wg), ('EO@0.5', eo)]

# ---------------------------------------------------------------- TABLE A
# THE HEADLINE: standard test vs correct test, same data, same models.
print("="*78); print("TABLE A. THE STANDARD TEST MANUFACTURES SIGNIFICANCE"); print("="*78)
rng, N = np.random.default_rng(0), len(y)
rows = []
for a, b in [('dwfa','qfedavg'), ('dwfa','fairfed'), ('qfedavg','fairfed')]:
    sa = [s for s in SEEDS if (a,s) in P]; sb = [s for s in SEEDS if (b,s) in P]
    if len(sa) != len(sb): continue
    pa = np.mean([P[(a,s)] for s in sa], axis=0)
    pb = np.mean([P[(b,s)] for s in sb], axis=0)
    for name, fn in MET[:2]:
        # standard: models FIXED, resample patients only  (what the field does)
        d1 = []
        for _ in range(2000):
            i = rng.integers(0, N, N)
            if y[i][M[i]].sum() < 5 or y[i][F[i]].sum() < 5: continue
            d1.append(fn(pa, i) - fn(pb, i))
        d1 = np.array(d1); lo1, hi1 = np.percentile(d1, [2.5, 97.5])
        # correct: resample SEEDS and patients jointly
        d2 = []
        for _ in range(2000):
            bs = rng.choice(sa, len(sa), replace=True)
            i  = rng.integers(0, N, N)
            if y[i][M[i]].sum() < 5 or y[i][F[i]].sum() < 5: continue
            qa = np.mean([P[(a,s)] for s in bs], axis=0)
            qb = np.mean([P[(b,s)] for s in bs], axis=0)
            d2.append(fn(qa, i) - fn(qb, i))
        d2 = np.array(d2); lo2, hi2 = np.percentile(d2, [2.5, 97.5])
        rows.append({'comparison': f'{a} - {b}', 'metric': name,
                     'diff': round(d1.mean(), 4),
                     'standard_CI': f'[{lo1:+.4f}, {hi1:+.4f}]',
                     'standard': 'SIGNIFICANT' if (lo1>0 or hi1<0) else 'ns',
                     'correct_CI': f'[{lo2:+.4f}, {hi2:+.4f}]',
                     'correct': 'SIGNIFICANT' if (lo2>0 or hi2<0) else 'NOT RESOLVED'})
A = pd.DataFrame(rows); print(A.to_string(index=False))
A.to_csv(f'{OUT}/tableA_test_vs_test.csv', index=False)

# ---------------------------------------------------------------- TABLE B
# Per-seed differences. Shows the signs are inconsistent.
print("\n"+"="*78); print("TABLE B. PER-SEED PAIRED DIFFERENCES"); print("="*78)
rows = []
for a, b in [('dwfa','qfedavg'), ('dwfa','fairfed')]:
    for name, fn in MET[:2]:
        d = np.array([fn(P[(a,s)]) - fn(P[(b,s)]) for s in SEEDS])
        rows.append({'comparison': f'{a} - {b}', 'metric': name,
                     **{f's{s}': round(v,4) for s,v in zip(SEEDS,d)},
                     'mean': round(d.mean(),4), 'sd': round(d.std(ddof=1),4),
                     'same_sign': bool((d>0).all() or (d<0).all()),
                     'paired_t_p': round(ttest_rel(d, np.zeros(len(d))).pvalue, 4)})
B = pd.DataFrame(rows); print(B.to_string(index=False))
B.to_csv(f'{OUT}/tableB_per_seed.csv', index=False)

# ---------------------------------------------------------------- TABLE C
# M1: the rule silences the site it protects.
print("\n"+"="*78); print("TABLE C. MEAN AGGREGATION WEIGHT vs FedAvg PRIOR"); print("="*78)
prior = {'A':0.418,'B':0.223,'C':0.192,'D':0.167}
rl = pd.read_csv(f'{RESULTS}/dwfa_round_log.csv')
g  = rl.groupby('client')['weight'].agg(['mean','std','min','max'])
C_ = pd.DataFrame({
    'client': g.index,
    'fedavg_prior': [prior[k] for k in g.index],
    'dwfa_mean_weight': g['mean'].round(4).values,
    'pct_change': [round((g['mean'][k]-prior[k])/prior[k]*100, 1) for k in g.index],
    'min': g['min'].round(4).values, 'max': g['max'].round(4).values,
    'proven_floor': [round(0.46*prior[k], 4) for k in g.index]})
print(C_.to_string(index=False))
print(f"\n  n = {len(rl)} weight observations, {rl['seed'].nunique()} seeds, {rl['round'].nunique()} rounds")
C_.to_csv(f'{OUT}/tableC_weight_concentration.csv', index=False)

# ---------------------------------------------------------------- TABLE D
# Placebo, per-seed. The mechanism does nothing.
print("\n"+"="*78); print("TABLE D. PLACEBO, PER SEED"); print("="*78)
scr = np.load(f'{RESULTS}/nih_scrambled_pred_cache.npz', allow_pickle=True)['probs'].item()
S3, rows = [42,123,456], []
for name, fn in MET:
    d = np.array([fn(P[('dwfa',s)]) - fn(scr[s]) for s in S3])
    rows.append({'metric': name, **{f's{s}': round(v,4) for s,v in zip(S3,d)},
                 'mean': round(d.mean(),4), 'sd': round(d.std(ddof=1),4),
                 'same_sign': bool((d>0).all() or (d<0).all()),
                 'paired_t_p': round(ttest_rel(d, np.zeros(3)).pvalue, 4)})
D = pd.DataFrame(rows); print(D.to_string(index=False))
D.to_csv(f'{OUT}/tableD_placebo_per_seed.csv', index=False)

# ---------------------------------------------------------------- TABLE E
# Per-seed NIH metrics for every method (replaces mean+-sd only reporting).
print("\n"+"="*78); print("TABLE E. PER-SEED NIH METRICS, ALL METHODS"); print("="*78)
rows = []
for m in METHODS:
    for s in SEEDS:
        if (m,s) not in P: continue
        p = P[(m,s)]
        rows.append({'method': m, 'seed': s, 'auroc': round(au(p),4),
                     'wg_auroc': round(wg(p),4), 'eo_gap': round(eo(p),4)})
E = pd.DataFrame(rows); print(E.to_string(index=False))
E.to_csv(f'{OUT}/tableE_per_seed_metrics.csv', index=False)

# ---------------------------------------------------------------- TABLE F
# Noise floor at full precision (fixes the ratio arithmetic in old Table 4).
print("\n"+"="*78); print("TABLE F. NOISE FLOOR (5 dp)"); print("="*78)
try:
    vd = pd.read_csv(f'{RESULTS}/clientC_variance_decomposition.csv')
    print(vd.round(5).to_string(index=False))
    vd.round(5).to_csv(f'{OUT}/tableF_noise_floor.csv', index=False)
except Exception as e:
    print("  clientC_variance_decomposition.csv not found:", e)

print(f"\n{'='*78}\nAll tables saved to {OUT}/\n{'='*78}")

TABLE A. THE STANDARD TEST MANUFACTURES SIGNIFICANCE
       comparison   metric    diff        standard_CI    standard         correct_CI      correct
   dwfa - qfedavg    AUROC  0.0031 [+0.0026, +0.0036] SIGNIFICANT [+0.0000, +0.0056]  SIGNIFICANT
   dwfa - qfedavg WG-AUROC  0.0026 [+0.0016, +0.0038] SIGNIFICANT [-0.0007, +0.0053] NOT RESOLVED
   dwfa - fairfed    AUROC  0.0018 [+0.0012, +0.0022] SIGNIFICANT [-0.0011, +0.0043] NOT RESOLVED
   dwfa - fairfed WG-AUROC  0.0012 [+0.0003, +0.0027] SIGNIFICANT [-0.0014, +0.0041] NOT RESOLVED
qfedavg - fairfed    AUROC -0.0014 [-0.0018, -0.0009] SIGNIFICANT [-0.0033, +0.0010] NOT RESOLVED
qfedavg - fairfed WG-AUROC -0.0014 [-0.0021, -0.0006] SIGNIFICANT [-0.0037, +0.0012] NOT RESOLVED

TABLE B. PER-SEED PAIRED DIFFERENCES
    comparison   metric     s42   s123   s456   s789   s1010   mean     sd  same_sign  paired_t_p
dwfa - qfedavg    AUROC -0.0019 0.0060 0.0002 0.0047  0.0033 0.0024 0.0032      False      0.1681
dwfa - qfedavg WG-AUROC -0.

In [ ]:
# ============================================================================
# RUN 1: FORENSIC DATA, CACHE, AND LOG AUDIT
# CPU only. No training. No checkpoint inference.
#
# Output:
#   /content/drive/MyDrive/FairFedCXR/results/revision_audit_run1.zip
# ============================================================================

from google.colab import drive
drive.mount("/content/drive")

from pathlib import Path
import json
import re
import zipfile

import numpy as np
import pandas as pd

# ---------------------------------------------------------------------------
# Paths
# ---------------------------------------------------------------------------
BASE = Path("/content/drive/MyDrive/FairFedCXR")
CLIENTS = BASE / "clients"
RESULTS = BASE / "results"
CKPTS = BASE / "checkpoints"
AUDIT = RESULTS / "revision_audit_run1"
AUDIT.mkdir(parents=True, exist_ok=True)

CLIENT_NAMES = ["A", "B", "C", "D"]
SPLIT_NAMES = ["train", "val", "test"]

print("=" * 88)
print("RUN 1: FORENSIC DATA, CACHE, AND LOG AUDIT")
print("=" * 88)
print("BASE:", BASE)
print("CLIENTS exists:", CLIENTS.exists())
print("RESULTS exists:", RESULTS.exists())
print("CKPTS exists:", CKPTS.exists())

summary = {
    "base": str(BASE),
    "errors": [],
    "warnings": [],
}

# ---------------------------------------------------------------------------
# Helpers
# ---------------------------------------------------------------------------
def extract_chexpert_patient(path_value):
    """Extract patient identifier from a CheXpert path."""
    text = str(path_value)
    match = re.search(r"(patient\d+)", text, flags=re.IGNORECASE)
    return match.group(1).lower() if match else None


def extract_chexpert_study(path_value):
    """Extract study identifier from a CheXpert path."""
    text = str(path_value)
    match = re.search(r"(study\d+)", text, flags=re.IGNORECASE)
    return match.group(1).lower() if match else None


def safe_value_counts(series):
    return {str(k): int(v) for k, v in series.value_counts(dropna=False).items()}


def inspect_prediction_dict(pred_dict, expected_n, name):
    records = []
    errors = []

    for key, probs in sorted(pred_dict.items(), key=lambda x: str(x[0])):
        arr = np.asarray(probs)

        record = {
            "cache": name,
            "key": str(key),
            "n": int(len(arr)),
            "n_nan": int(np.isnan(arr).sum()),
            "n_inf": int(np.isinf(arr).sum()),
            "min_probability": float(np.nanmin(arr)),
            "max_probability": float(np.nanmax(arr)),
            "length_matches": bool(len(arr) == expected_n),
            "range_valid": bool(
                np.nanmin(arr) >= 0.0 and np.nanmax(arr) <= 1.0
            ),
        }
        records.append(record)

        if len(arr) != expected_n:
            errors.append(
                f"{name} {key}: prediction length {len(arr)} != {expected_n}"
            )
        if not np.isfinite(arr).all():
            errors.append(f"{name} {key}: non-finite predictions detected")
        if np.nanmin(arr) < 0 or np.nanmax(arr) > 1:
            errors.append(f"{name} {key}: probabilities outside [0, 1]")

    return pd.DataFrame(records), errors


# ===========================================================================
# PART A: CHEXPERT CLIENT AND SPLIT AUDIT
# ===========================================================================
print("\n" + "=" * 88)
print("PART A: CHEXPERT PATIENT-LEVEL PARTITION AUDIT")
print("=" * 88)

parts = []

for client in CLIENT_NAMES:
    for split in SPLIT_NAMES:
        csv_path = CLIENTS / f"client_{client}_{split}.csv"

        if not csv_path.exists():
            msg = f"Missing client CSV: {csv_path}"
            print("ERROR:", msg)
            summary["errors"].append(msg)
            continue

        frame = pd.read_csv(csv_path)

        required = {"Path", "label", "sex_encoded"}
        missing = required - set(frame.columns)
        if missing:
            msg = f"{csv_path.name} missing columns: {sorted(missing)}"
            print("ERROR:", msg)
            summary["errors"].append(msg)
            continue

        frame = frame.copy()
        frame["client"] = client
        frame["split"] = split
        frame["source_csv"] = csv_path.name
        frame["patient_id"] = frame["Path"].map(extract_chexpert_patient)
        frame["study_id"] = frame["Path"].map(extract_chexpert_study)

        failed_patient_parse = int(frame["patient_id"].isna().sum())
        if failed_patient_parse:
            msg = (
                f"{csv_path.name}: failed to parse patient ID for "
                f"{failed_patient_parse} rows"
            )
            print("WARNING:", msg)
            summary["warnings"].append(msg)

        parts.append(frame)

if not parts:
    raise RuntimeError("No client split CSVs could be loaded.")

cxr = pd.concat(parts, ignore_index=True)

print(f"Total client rows:          {len(cxr):,}")
print(f"Unique image paths:         {cxr['Path'].nunique():,}")
print(f"Unique parsed patients:     {cxr['patient_id'].nunique(dropna=True):,}")
print(f"Unique parsed studies:      {cxr['study_id'].nunique(dropna=True):,}")
print(f"Unparsed patient IDs:       {cxr['patient_id'].isna().sum():,}")
print(f"Exact duplicated paths:     {cxr['Path'].duplicated().sum():,}")

# Summary by client and split
cxr_partition_summary = (
    cxr.groupby(["client", "split"])
    .agg(
        n_images=("Path", "size"),
        n_patients=("patient_id", "nunique"),
        n_studies=("study_id", "nunique"),
        prevalence=("label", "mean"),
        male_fraction=("sex_encoded", "mean"),
        positive_cases=("label", "sum"),
    )
    .reset_index()
)

print("\nClient/split summary:")
print(cxr_partition_summary.to_string(index=False))
cxr_partition_summary.to_csv(
    AUDIT / "chexpert_partition_summary.csv", index=False
)

# Cross-client patient overlap
patient_clients = (
    cxr.dropna(subset=["patient_id"])
    .groupby("patient_id")["client"]
    .agg(lambda x: sorted(set(x)))
)

cross_client_patients = patient_clients[
    patient_clients.map(len) > 1
].reset_index(name="clients")

cross_client_patients["n_clients"] = cross_client_patients["clients"].map(len)
cross_client_patients["clients"] = cross_client_patients["clients"].map(
    lambda x: ",".join(x)
)

# Cross-split patient overlap
patient_splits = (
    cxr.dropna(subset=["patient_id"])
    .groupby("patient_id")["split"]
    .agg(lambda x: sorted(set(x)))
)

cross_split_patients = patient_splits[
    patient_splits.map(len) > 1
].reset_index(name="splits")

cross_split_patients["n_splits"] = cross_split_patients["splits"].map(len)
cross_split_patients["splits"] = cross_split_patients["splits"].map(
    lambda x: ",".join(x)
)

# Detailed assignments for every overlapping patient
overlap_ids = set(cross_client_patients["patient_id"]).union(
    set(cross_split_patients["patient_id"])
)

overlap_details = (
    cxr[cxr["patient_id"].isin(overlap_ids)][
        [
            "patient_id",
            "study_id",
            "Path",
            "client",
            "split",
            "label",
            "sex_encoded",
        ]
    ]
    .sort_values(["patient_id", "client", "split", "Path"])
)

cross_client_patients.to_csv(
    AUDIT / "chexpert_cross_client_patients.csv", index=False
)
cross_split_patients.to_csv(
    AUDIT / "chexpert_cross_split_patients.csv", index=False
)
overlap_details.to_csv(
    AUDIT / "chexpert_overlap_details.csv", index=False
)

print("\nPatient-level overlap:")
print(f"Patients in >1 client:      {len(cross_client_patients):,}")
print(f"Patients in >1 split:       {len(cross_split_patients):,}")
print(f"Rows involving overlap:     {len(overlap_details):,}")

# Within-client split overlap matrix
within_client_rows = []

for client in CLIENT_NAMES:
    sub = cxr[cxr["client"] == client]
    sets = {
        split: set(
            sub.loc[sub["split"] == split, "patient_id"].dropna()
        )
        for split in SPLIT_NAMES
    }

    for first, second in [
        ("train", "val"),
        ("train", "test"),
        ("val", "test"),
    ]:
        overlap = sets[first] & sets[second]
        within_client_rows.append(
            {
                "client": client,
                "split_1": first,
                "split_2": second,
                "n_overlapping_patients": len(overlap),
            }
        )

within_client_overlap = pd.DataFrame(within_client_rows)
within_client_overlap.to_csv(
    AUDIT / "chexpert_within_client_split_overlap.csv", index=False
)

print("\nWithin-client patient overlap:")
print(within_client_overlap.to_string(index=False))

# Save patient assignment table for later analysis
cxr[
    [
        "patient_id",
        "study_id",
        "Path",
        "client",
        "split",
        "label",
        "sex_encoded",
    ]
].to_csv(AUDIT / "chexpert_patient_assignments.csv", index=False)

summary["chexpert"] = {
    "n_rows": int(len(cxr)),
    "n_unique_paths": int(cxr["Path"].nunique()),
    "n_unique_patients": int(cxr["patient_id"].nunique(dropna=True)),
    "n_cross_client_patients": int(len(cross_client_patients)),
    "n_cross_split_patients": int(len(cross_split_patients)),
    "n_exact_duplicate_paths": int(cxr["Path"].duplicated().sum()),
}

# ===========================================================================
# PART B: NIH COHORT AUDIT
# ===========================================================================
print("\n" + "=" * 88)
print("PART B: NIH IMAGE/PATIENT COHORT AUDIT")
print("=" * 88)

nih_csv = RESULTS / "nih_effusion_cohort.csv"

if not nih_csv.exists():
    msg = f"Missing NIH cohort CSV: {nih_csv}"
    print("ERROR:", msg)
    summary["errors"].append(msg)
    nih = None
else:
    nih = pd.read_csv(nih_csv)

    required = {"Path", "label", "sex_encoded", "Patient ID"}
    missing = required - set(nih.columns)

    if missing:
        msg = f"NIH cohort missing columns: {sorted(missing)}"
        print("ERROR:", msg)
        summary["errors"].append(msg)
    else:
        nih = nih.copy()
        nih["patient_id"] = nih["Patient ID"].astype(str)

        patient_sizes = nih.groupby("patient_id").size()
        patient_sex_conflict = (
            nih.groupby("patient_id")["sex_encoded"].nunique() > 1
        ).sum()

        print(f"NIH images:                {len(nih):,}")
        print(f"NIH unique patients:       {nih['patient_id'].nunique():,}")
        print(f"Patients with >1 image:    {(patient_sizes > 1).sum():,}")
        print(f"Maximum images/patient:    {patient_sizes.max():,}")
        print(f"Median images/patient:     {patient_sizes.median():.1f}")
        print(f"Patient sex conflicts:     {int(patient_sex_conflict):,}")
        print(f"Male images:               {(nih.sex_encoded == 1).sum():,}")
        print(f"Female images:             {(nih.sex_encoded == 0).sum():,}")
        print(f"Positive images:           {nih.label.sum():,}")
        print(f"Image prevalence:          {nih.label.mean():.4f}")

        nih_patient_summary = (
            nih.groupby("patient_id")
            .agg(
                n_images=("Path", "size"),
                sex_encoded=("sex_encoded", "first"),
                any_positive=("label", "max"),
                positive_images=("label", "sum"),
            )
            .reset_index()
        )

        nih_patient_summary.to_csv(
            AUDIT / "nih_patient_summary.csv", index=False
        )

        nih[
            ["patient_id", "Path", "label", "sex_encoded"]
        ].to_csv(AUDIT / "nih_analysis_index.csv", index=False)

        summary["nih"] = {
            "n_images": int(len(nih)),
            "n_unique_patients": int(nih["patient_id"].nunique()),
            "n_patients_multiple_images": int((patient_sizes > 1).sum()),
            "max_images_per_patient": int(patient_sizes.max()),
            "patient_sex_conflicts": int(patient_sex_conflict),
        }

# ===========================================================================
# PART C: PREDICTION CACHE AUDIT
# ===========================================================================
print("\n" + "=" * 88)
print("PART C: PREDICTION CACHE AUDIT")
print("=" * 88)

cache_audit_frames = []

# NIH cache
nih_cache_path = RESULTS / "nih_pred_cache.npz"

if nih_cache_path.exists() and nih is not None:
    cache = np.load(nih_cache_path, allow_pickle=True)

    print("NIH cache fields:", list(cache.files))

    required_fields = {"all_probs", "labels", "sex"}
    missing_fields = required_fields - set(cache.files)

    if missing_fields:
        msg = f"NIH cache missing fields: {sorted(missing_fields)}"
        print("ERROR:", msg)
        summary["errors"].append(msg)
    else:
        probs_dict = cache["all_probs"].item()
        labels = cache["labels"].astype(int)
        sex = cache["sex"].astype(int)

        print(f"NIH cache labels length:   {len(labels):,}")
        print(f"NIH cache method/runs:     {len(probs_dict):,}")
        print("NIH cache keys:")
        for key in sorted(probs_dict, key=str):
            print(" ", key)

        label_match = (
            len(labels) == len(nih)
            and np.array_equal(labels, nih["label"].to_numpy().astype(int))
        )
        sex_match = (
            len(sex) == len(nih)
            and np.array_equal(
                sex, nih["sex_encoded"].to_numpy().astype(int)
            )
        )

        print("Labels match cohort order:", label_match)
        print("Sex matches cohort order:  ", sex_match)

        if not label_match:
            summary["errors"].append(
                "NIH cache labels do not match NIH cohort row order"
            )
        if not sex_match:
            summary["errors"].append(
                "NIH cache sex does not match NIH cohort row order"
            )

        frame, errors = inspect_prediction_dict(
            probs_dict, len(nih), "nih_pred_cache"
        )
        cache_audit_frames.append(frame)
        summary["errors"].extend(errors)

        summary["nih_cache"] = {
            "fields": list(cache.files),
            "n_method_seed_keys": int(len(probs_dict)),
            "label_order_match": bool(label_match),
            "sex_order_match": bool(sex_match),
            "keys": [str(k) for k in sorted(probs_dict, key=str)],
        }
else:
    msg = f"NIH prediction cache missing or NIH cohort unavailable: {nih_cache_path}"
    print("WARNING:", msg)
    summary["warnings"].append(msg)

# Scrambled cache
scrambled_cache_path = RESULTS / "nih_scrambled_pred_cache.npz"

if scrambled_cache_path.exists() and nih is not None:
    scrambled_cache = np.load(scrambled_cache_path, allow_pickle=True)
    print("\nScrambled cache fields:", list(scrambled_cache.files))

    if "probs" in scrambled_cache.files:
        scrambled_dict = scrambled_cache["probs"].item()
        frame, errors = inspect_prediction_dict(
            scrambled_dict, len(nih), "nih_scrambled_pred_cache"
        )
        cache_audit_frames.append(frame)
        summary["errors"].extend(errors)
        summary["scrambled_cache_keys"] = [
            str(k) for k in sorted(scrambled_dict, key=str)
        ]

# CheXpert pooled test cache
cxr_cache_path = RESULTS / "chexpert_pred_cache.npz"
pooled_test = pd.concat(
    [
        pd.read_csv(CLIENTS / f"client_{client}_test.csv")
        for client in CLIENT_NAMES
        if (CLIENTS / f"client_{client}_test.csv").exists()
    ],
    ignore_index=True,
)

if cxr_cache_path.exists() and len(pooled_test):
    cache = np.load(cxr_cache_path, allow_pickle=True)

    print("\nCheXpert cache fields:", list(cache.files))

    if {"all_probs", "labels", "sex"}.issubset(cache.files):
        probs_dict = cache["all_probs"].item()
        labels = cache["labels"].astype(int)
        sex = cache["sex"].astype(int)

        label_match = (
            len(labels) == len(pooled_test)
            and np.array_equal(
                labels, pooled_test["label"].to_numpy().astype(int)
            )
        )
        sex_match = (
            len(sex) == len(pooled_test)
            and np.array_equal(
                sex, pooled_test["sex_encoded"].to_numpy().astype(int)
            )
        )

        print(f"CheXpert pooled test rows: {len(pooled_test):,}")
        print("Labels match pooled test:  ", label_match)
        print("Sex matches pooled test:   ", sex_match)

        if not label_match:
            summary["errors"].append(
                "CheXpert cache labels do not match pooled-test row order"
            )
        if not sex_match:
            summary["errors"].append(
                "CheXpert cache sex does not match pooled-test row order"
            )

        frame, errors = inspect_prediction_dict(
            probs_dict, len(pooled_test), "chexpert_pred_cache"
        )
        cache_audit_frames.append(frame)
        summary["errors"].extend(errors)

if cache_audit_frames:
    cache_audit = pd.concat(cache_audit_frames, ignore_index=True)
    cache_audit.to_csv(AUDIT / "prediction_cache_audit.csv", index=False)
    print("\nPrediction-array audit:")
    print(cache_audit.to_string(index=False))

# ===========================================================================
# PART D: ROUND LOG AND SELECTED-CHECKPOINT AUDIT
# ===========================================================================
print("\n" + "=" * 88)
print("PART D: ROUND LOG AND CHECKPOINT AUDIT")
print("=" * 88)

best_round_records = []
log_inventory = []

for csv_path in sorted(RESULTS.glob("*round_log*.csv")):
    try:
        log = pd.read_csv(csv_path)
    except Exception as exc:
        summary["warnings"].append(
            f"Could not read {csv_path.name}: {exc}"
        )
        continue

    log_inventory.append(
        {
            "file": csv_path.name,
            "n_rows": len(log),
            "columns": "|".join(log.columns.astype(str)),
        }
    )

    if not {"seed", "round"}.issubset(log.columns):
        continue

    metric_candidates = [
        "pooled_val_auroc",
        "val_auroc",
        "global_val_auroc",
        "auroc",
    ]

    metric_col = next(
        (column for column in metric_candidates if column in log.columns),
        None,
    )

    if metric_col is None:
        continue

    grouping_columns = ["seed"]
    if "method" in log.columns:
        grouping_columns = ["method", "seed"]

    # Some logs contain four rows per round, one per client.
    reduced_columns = grouping_columns + ["round", metric_col]
    reduced = log[reduced_columns].drop_duplicates(
        grouping_columns + ["round"]
    )

    for group_key, group in reduced.groupby(grouping_columns):
        group = group.dropna(subset=[metric_col])
        if group.empty:
            continue

        best = group.loc[group[metric_col].idxmax()]

        if isinstance(group_key, tuple):
            method = str(group_key[0])
            seed = int(group_key[-1])
        else:
            method = csv_path.stem.replace("_round_log", "")
            seed = int(group_key)

        best_round_records.append(
            {
                "source_log": csv_path.name,
                "method": method,
                "seed": seed,
                "metric": metric_col,
                "best_round": int(best["round"]),
                "best_value": float(best[metric_col]),
                "last_round": int(group["round"].max()),
            }
        )

log_inventory_df = pd.DataFrame(log_inventory)
log_inventory_df.to_csv(AUDIT / "round_log_inventory.csv", index=False)

best_rounds_df = pd.DataFrame(best_round_records)

if not best_rounds_df.empty:
    best_rounds_df = best_rounds_df.sort_values(
        ["method", "seed", "source_log"]
    )
    best_rounds_df.to_csv(
        AUDIT / "selected_checkpoint_rounds.csv", index=False
    )
    print("Selected checkpoint rounds inferred from logs:")
    print(best_rounds_df.to_string(index=False))
else:
    print("No selected checkpoint rounds could be inferred from available logs.")

# Checkpoint file inventory
checkpoint_records = []

if CKPTS.exists():
    for path in sorted(CKPTS.glob("*.pt")):
        checkpoint_records.append(
            {
                "file": path.name,
                "size_mb": round(path.stat().st_size / (1024**2), 2),
            }
        )

checkpoint_inventory = pd.DataFrame(checkpoint_records)
checkpoint_inventory.to_csv(
    AUDIT / "checkpoint_inventory.csv", index=False
)

print(f"\nCheckpoint files found: {len(checkpoint_inventory):,}")

summary["checkpoint_files"] = int(len(checkpoint_inventory))
summary["round_logs"] = int(len(log_inventory_df))

# ===========================================================================
# PART E: DECISION GATE
# ===========================================================================
print("\n" + "=" * 88)
print("DECISION GATE")
print("=" * 88)

cross_client_count = summary["chexpert"]["n_cross_client_patients"]
cross_split_count = summary["chexpert"]["n_cross_split_patients"]

if cross_client_count == 0 and cross_split_count == 0:
    print("PASS: CheXpert clients and splits appear patient-disjoint.")
    summary["chexpert_patient_disjoint_gate"] = "PASS"
else:
    print("FAIL: CheXpert patient overlap was detected.")
    print(f"  Patients appearing in multiple clients: {cross_client_count:,}")
    print(f"  Patients appearing in multiple splits:  {cross_split_count:,}")
    print(
        "  This cannot be corrected in the trained checkpoints without "
        "retraining. The revised paper must disclose study-level partitioning "
        "and treat CheXpert evaluation as exploratory."
    )
    summary["chexpert_patient_disjoint_gate"] = "FAIL"

if summary.get("nih", {}).get("n_unique_patients", 0) < summary.get(
    "nih", {}
).get("n_images", 0):
    print(
        "\nNIH cluster bootstrap required: multiple images occur within "
        "individual patients."
    )

print(f"\nErrors:   {len(summary['errors'])}")
for item in summary["errors"]:
    print("  ERROR  -", item)

print(f"\nWarnings: {len(summary['warnings'])}")
for item in summary["warnings"]:
    print("  WARN   -", item)

# Save summary JSON
with open(AUDIT / "audit_summary.json", "w", encoding="utf-8") as handle:
    json.dump(summary, handle, indent=2)

# Zip everything for upload
zip_path = RESULTS / "revision_audit_run1.zip"

with zipfile.ZipFile(zip_path, "w", compression=zipfile.ZIP_DEFLATED) as archive:
    for path in sorted(AUDIT.rglob("*")):
        if path.is_file():
            archive.write(path, arcname=path.relative_to(AUDIT))

print("\n" + "=" * 88)
print("RUN 1 COMPLETE")
print("=" * 88)
print("Audit folder:", AUDIT)
print("Upload this ZIP:", zip_path)

Mounted at /content/drive
RUN 1: FORENSIC DATA, CACHE, AND LOG AUDIT
BASE: /content/drive/MyDrive/FairFedCXR
CLIENTS exists: True
RESULTS exists: True
CKPTS exists: True

PART A: CHEXPERT PATIENT-LEVEL PARTITION AUDIT
Total client rows:          35,899
Unique image paths:         35,899
Unique parsed patients:     24,709
Unique parsed studies:      73
Unparsed patient IDs:       0
Exact duplicated paths:     0

Client/split summary:
client split  n_images  n_patients  n_studies  prevalence  male_fraction  positive_cases
     A  test      2250        2173         48    0.400000       0.498222             900
     A train     10500        9064         65    0.400000       0.500952            4200
     A   val      2250        2169         44    0.400000       0.497333             900
     B  test      1200        1174         40    0.450000       0.778333             540
     B train      5600        5032         61    0.450000       0.801964            2520
     B   val      1200       

OSError: [Errno 107] Transport endpoint is not connected

In [ ]:
# ============================================================================
# RUN 1B: COMPLETE FORENSIC AUDIT AFTER DRIVE DISCONNECTION
# CPU only. No training. No model inference.
#
# Output:
#   /content/drive/MyDrive/FairFedCXR/results/revision_audit_run1b.zip
# ============================================================================

from google.colab import drive
drive.mount("/content/drive", force_remount=True)

from pathlib import Path
import hashlib
import json
import re
import shutil
import zipfile

import numpy as np
import pandas as pd

# ---------------------------------------------------------------------------
# Paths
# ---------------------------------------------------------------------------
BASE = Path("/content/drive/MyDrive/FairFedCXR")
CLIENTS = BASE / "clients"
RESULTS = BASE / "results"
CHECKPOINTS = BASE / "checkpoints"

LOCAL = Path("/content/revision_audit_run1b")
LOCAL_INPUTS = LOCAL / "inputs"
LOCAL_OUTPUTS = LOCAL / "outputs"

for directory in [LOCAL_INPUTS, LOCAL_OUTPUTS]:
    directory.mkdir(parents=True, exist_ok=True)

summary = {
    "errors": [],
    "warnings": [],
    "files_used": {},
}

print("=" * 92)
print("RUN 1B: NIH, CACHE, LOG, AND CHECKPOINT AUDIT")
print("=" * 92)
print("BASE:", BASE)
print("Drive accessible:", BASE.exists())

if not BASE.exists():
    raise RuntimeError(
        "Google Drive is still unavailable. Restart the Colab session and "
        "run this cell again."
    )


# ---------------------------------------------------------------------------
# Utility functions
# ---------------------------------------------------------------------------
def first_existing(paths):
    for path in paths:
        if path.exists():
            return path
    return None


def find_candidates(root, patterns):
    found = []
    for pattern in patterns:
        found.extend(root.glob(pattern))
    return sorted(set(found), key=lambda p: (-p.stat().st_size, str(p)))


def copy_local(source):
    """Copy a Drive file locally before reading to reduce FUSE failures."""
    destination = LOCAL_INPUTS / source.name
    print(f"Copying locally: {source}")
    shutil.copy2(source, destination)
    return destination


def extract_patient_id(path_value):
    match = re.search(
        r"(patient\d+)",
        str(path_value),
        flags=re.IGNORECASE,
    )
    return match.group(1).lower() if match else None


def extract_study_id(path_value):
    match = re.search(
        r"(study\d+)",
        str(path_value),
        flags=re.IGNORECASE,
    )
    return match.group(1).lower() if match else None


def array_hash(array):
    array = np.ascontiguousarray(np.asarray(array))
    return hashlib.sha256(array.tobytes()).hexdigest()[:20]


def extract_prediction_dictionary(npz):
    """
    Find a dictionary of prediction arrays in a loaded NPZ archive.
    Common fields in the original notebook include all_probs and probs.
    """
    preferred = ["all_probs", "probs", "predictions", "pred_dict"]

    for field in preferred:
        if field not in npz.files:
            continue

        candidate = npz[field]

        try:
            value = candidate.item()
        except Exception:
            value = candidate

        if isinstance(value, dict):
            return field, value

    return None, None


def inspect_prediction_dictionary(predictions, expected_n, cache_name):
    records = []

    for key, values in sorted(predictions.items(), key=lambda item: str(item[0])):
        array = np.asarray(values).reshape(-1)

        record = {
            "cache": cache_name,
            "key": str(key),
            "n_predictions": int(len(array)),
            "expected_n": int(expected_n),
            "length_matches": bool(len(array) == expected_n),
            "n_nan": int(np.isnan(array).sum()),
            "n_inf": int(np.isinf(array).sum()),
            "minimum": float(np.nanmin(array)) if len(array) else np.nan,
            "maximum": float(np.nanmax(array)) if len(array) else np.nan,
            "mean": float(np.nanmean(array)) if len(array) else np.nan,
            "std": float(np.nanstd(array)) if len(array) else np.nan,
            "sha256_short": array_hash(array),
        }
        records.append(record)

        if len(array) != expected_n:
            summary["errors"].append(
                f"{cache_name} {key}: {len(array)} predictions, "
                f"expected {expected_n}"
            )

        if not np.isfinite(array).all():
            summary["errors"].append(
                f"{cache_name} {key}: non-finite predictions"
            )

        if len(array) and (np.nanmin(array) < 0 or np.nanmax(array) > 1):
            summary["errors"].append(
                f"{cache_name} {key}: prediction outside [0,1]"
            )

    return pd.DataFrame(records)


# ===========================================================================
# PART A: CORRECTED CHEXPERT STUDY COUNTS
# ===========================================================================
print("\n" + "=" * 92)
print("PART A: CORRECTED CHEXPERT PATIENT/STUDY SUMMARY")
print("=" * 92)

client_frames = []

for client in ["A", "B", "C", "D"]:
    for split in ["train", "val", "test"]:
        path = CLIENTS / f"client_{client}_{split}.csv"

        if not path.exists():
            summary["warnings"].append(f"Missing client file: {path.name}")
            continue

        frame = pd.read_csv(path, low_memory=False)
        frame = frame.copy()

        frame["client"] = client
        frame["split"] = split
        frame["patient_id"] = frame["Path"].map(extract_patient_id)
        frame["study_id"] = frame["Path"].map(extract_study_id)

        frame["study_uid"] = (
            frame["patient_id"].astype(str)
            + "/"
            + frame["study_id"].astype(str)
        )

        client_frames.append(frame)

chexpert = pd.concat(client_frames, ignore_index=True)

corrected_summary = (
    chexpert.groupby(["client", "split"])
    .agg(
        n_images=("Path", "size"),
        n_patients=("patient_id", "nunique"),
        n_patient_studies=("study_uid", "nunique"),
        prevalence=("label", "mean"),
        male_fraction=("sex_encoded", "mean"),
        positive_images=("label", "sum"),
    )
    .reset_index()
)

corrected_summary.to_csv(
    LOCAL_OUTPUTS / "chexpert_partition_summary_corrected.csv",
    index=False,
)

print(corrected_summary.to_string(index=False))
print(
    "\nUnique composite patient-study identifiers:",
    f"{chexpert['study_uid'].nunique():,}",
)

summary["chexpert_corrected"] = {
    "n_images": int(len(chexpert)),
    "n_patients": int(chexpert["patient_id"].nunique()),
    "n_patient_studies": int(chexpert["study_uid"].nunique()),
}


# ===========================================================================
# PART B: LOCATE AND AUDIT NIH COHORT
# ===========================================================================
print("\n" + "=" * 92)
print("PART B: NIH COHORT AUDIT")
print("=" * 92)

nih_candidates = [
    RESULTS / "nih_effusion_cohort.csv",
    RESULTS / "nih_cohort.csv",
]

nih_path = first_existing(nih_candidates)

if nih_path is None:
    discovered = find_candidates(
        RESULTS,
        [
            "*nih*cohort*.csv",
            "*NIH*cohort*.csv",
            "*nih*effusion*.csv",
        ],
    )
    nih_path = discovered[0] if discovered else None

if nih_path is None:
    raise FileNotFoundError(
        "Could not locate an NIH cohort CSV inside the results directory."
    )

summary["files_used"]["nih_cohort"] = str(nih_path)
nih_local = copy_local(nih_path)
nih = pd.read_csv(nih_local, low_memory=False)

print("NIH cohort file:", nih_path.name)
print("Columns:", list(nih.columns))

patient_column_candidates = [
    "Patient ID",
    "PatientID",
    "patient_id",
    "patientid",
    "Patient Id",
]

patient_column = next(
    (column for column in patient_column_candidates if column in nih.columns),
    None,
)

if patient_column is None:
    raise KeyError(
        "No NIH patient-ID column was found. Available columns:\n"
        + "\n".join(map(str, nih.columns))
    )

label_column = next(
    (
        column
        for column in ["label", "Label", "target", "effusion_label"]
        if column in nih.columns
    ),
    None,
)

sex_column = next(
    (
        column
        for column in [
            "sex_encoded",
            "SexEncoded",
            "sex",
            "Patient Gender",
            "gender",
        ]
        if column in nih.columns
    ),
    None,
)

if label_column is None:
    raise KeyError("No label column was found in the NIH cohort.")

if sex_column is None:
    raise KeyError("No sex column was found in the NIH cohort.")

nih = nih.copy()
nih["patient_id_audit"] = nih[patient_column].astype(str).str.strip()

patient_sizes = nih.groupby("patient_id_audit").size()
patient_label_summary = (
    nih.groupby("patient_id_audit")
    .agg(
        n_images=(patient_column, "size"),
        any_positive=(label_column, "max"),
        positive_images=(label_column, "sum"),
        n_unique_sex_values=(sex_column, "nunique"),
    )
    .reset_index()
)

n_sex_conflicts = int(
    (patient_label_summary["n_unique_sex_values"] > 1).sum()
)

print(f"NIH images:                     {len(nih):,}")
print(
    f"NIH unique patients:            "
    f"{nih['patient_id_audit'].nunique():,}"
)
print(
    f"Patients with multiple images:  "
    f"{(patient_sizes > 1).sum():,}"
)
print(f"Median images per patient:      {patient_sizes.median():.1f}")
print(f"Maximum images per patient:     {patient_sizes.max():,}")
print(f"Patients with sex conflicts:    {n_sex_conflicts:,}")
print(f"Image-level positives:          {nih[label_column].sum():,}")
print(f"Image-level prevalence:         {nih[label_column].mean():.6f}")

nih_index_columns = [
    "patient_id_audit",
    patient_column,
    label_column,
    sex_column,
]

for optional_column in ["Path", "Image Index", "image_id", "Image"]:
    if optional_column in nih.columns:
        nih_index_columns.append(optional_column)

nih[nih_index_columns].to_csv(
    LOCAL_OUTPUTS / "nih_patient_image_index.csv",
    index=False,
)

patient_label_summary.to_csv(
    LOCAL_OUTPUTS / "nih_patient_cluster_summary.csv",
    index=False,
)

summary["nih"] = {
    "n_images": int(len(nih)),
    "n_unique_patients": int(nih["patient_id_audit"].nunique()),
    "n_patients_multiple_images": int((patient_sizes > 1).sum()),
    "median_images_per_patient": float(patient_sizes.median()),
    "max_images_per_patient": int(patient_sizes.max()),
    "n_patient_sex_conflicts": n_sex_conflicts,
    "patient_column": patient_column,
    "label_column": label_column,
    "sex_column": sex_column,
}


# ===========================================================================
# PART C: NIH PREDICTION CACHE AUDIT
# ===========================================================================
print("\n" + "=" * 92)
print("PART C: NIH PREDICTION CACHE AUDIT")
print("=" * 92)

nih_cache_path = first_existing(
    [
        RESULTS / "nih_pred_cache.npz",
        RESULTS / "nih_predictions.npz",
    ]
)

if nih_cache_path is None:
    discovered = find_candidates(
        RESULTS,
        [
            "*nih*pred*cache*.npz",
            "*nih*prediction*.npz",
            "*NIH*pred*.npz",
        ],
    )
    nih_cache_path = discovered[0] if discovered else None

cache_audit_frames = []

if nih_cache_path is None:
    summary["errors"].append("No NIH prediction cache was located.")
    print("ERROR: No NIH prediction cache was located.")
else:
    summary["files_used"]["nih_cache"] = str(nih_cache_path)
    nih_cache_local = copy_local(nih_cache_path)

    cache = np.load(nih_cache_local, allow_pickle=True)
    print("NIH cache file:", nih_cache_path.name)
    print("NIH cache fields:", list(cache.files))

    prediction_field, predictions = extract_prediction_dictionary(cache)

    if predictions is None:
        summary["errors"].append(
            "No prediction dictionary was found in the NIH cache."
        )
        print("ERROR: No prediction dictionary found.")
    else:
        print("Prediction dictionary field:", prediction_field)
        print("Number of method/seed arrays:", len(predictions))

        audit = inspect_prediction_dictionary(
            predictions,
            expected_n=len(nih),
            cache_name=nih_cache_path.name,
        )
        cache_audit_frames.append(audit)

        print(audit.to_string(index=False))

    # Confirm whether cached labels and sex match cohort row order.
    label_fields = ["labels", "label", "y_true"]
    sex_fields = ["sex", "sex_encoded", "groups"]

    cache_label_field = next(
        (field for field in label_fields if field in cache.files),
        None,
    )
    cache_sex_field = next(
        (field for field in sex_fields if field in cache.files),
        None,
    )

    if cache_label_field is not None:
        cached_labels = np.asarray(cache[cache_label_field]).reshape(-1)
        expected_labels = nih[label_column].to_numpy().astype(int)

        label_order_matches = (
            len(cached_labels) == len(expected_labels)
            and np.array_equal(
                cached_labels.astype(int),
                expected_labels,
            )
        )

        print("Cached label order matches NIH CSV:", label_order_matches)
        summary["nih"]["cache_label_order_matches"] = bool(
            label_order_matches
        )

        if not label_order_matches:
            summary["errors"].append(
                "NIH cached labels do not match NIH cohort row order."
            )
    else:
        summary["warnings"].append(
            "NIH cache contains no labels field for row-order verification."
        )

    if cache_sex_field is not None:
        cached_sex = np.asarray(cache[cache_sex_field]).reshape(-1)
        expected_sex = nih[sex_column].to_numpy()

        # Compare numerically where possible, otherwise as strings.
        try:
            sex_order_matches = (
                len(cached_sex) == len(expected_sex)
                and np.array_equal(
                    cached_sex.astype(int),
                    expected_sex.astype(int),
                )
            )
        except Exception:
            sex_order_matches = (
                len(cached_sex) == len(expected_sex)
                and np.array_equal(
                    cached_sex.astype(str),
                    expected_sex.astype(str),
                )
            )

        print("Cached sex order matches NIH CSV:", sex_order_matches)
        summary["nih"]["cache_sex_order_matches"] = bool(
            sex_order_matches
        )

        if not sex_order_matches:
            summary["errors"].append(
                "NIH cached sex values do not match NIH cohort row order."
            )
    else:
        summary["warnings"].append(
            "NIH cache contains no sex field for row-order verification."
        )


# ===========================================================================
# PART D: SCRAMBLED-ROUTING CACHE AUDIT
# ===========================================================================
print("\n" + "=" * 92)
print("PART D: SCRAMBLED-ROUTING CACHE AUDIT")
print("=" * 92)

scrambled_candidates = find_candidates(
    RESULTS,
    [
        "*scrambl*pred*.npz",
        "*placebo*pred*.npz",
        "*derang*pred*.npz",
    ],
)

if not scrambled_candidates:
    print("No scrambled-routing prediction cache found.")
    summary["warnings"].append(
        "No scrambled-routing prediction cache was located."
    )
else:
    scrambled_path = scrambled_candidates[0]
    summary["files_used"]["scrambled_cache"] = str(scrambled_path)
    scrambled_local = copy_local(scrambled_path)

    scrambled_cache = np.load(scrambled_local, allow_pickle=True)
    print("Scrambled cache:", scrambled_path.name)
    print("Fields:", list(scrambled_cache.files))

    prediction_field, scrambled_predictions = (
        extract_prediction_dictionary(scrambled_cache)
    )

    if scrambled_predictions is None:
        summary["errors"].append(
            "No prediction dictionary found in scrambled-routing cache."
        )
    else:
        print("Prediction field:", prediction_field)

        scrambled_audit = inspect_prediction_dictionary(
            scrambled_predictions,
            expected_n=len(nih),
            cache_name=scrambled_path.name,
        )
        cache_audit_frames.append(scrambled_audit)
        print(scrambled_audit.to_string(index=False))


# ===========================================================================
# PART E: CHEXPERT PREDICTION CACHE INVENTORY
# ===========================================================================
print("\n" + "=" * 92)
print("PART E: CHEXPERT CACHE INVENTORY")
print("=" * 92)

chexpert_cache_candidates = find_candidates(
    RESULTS,
    [
        "*chexpert*pred*.npz",
        "*cxr*pred*cache*.npz",
    ],
)

if not chexpert_cache_candidates:
    print("No CheXpert prediction cache located.")
    summary["warnings"].append(
        "No CheXpert prediction cache was located."
    )
else:
    for cache_path in chexpert_cache_candidates:
        print(
            cache_path.name,
            f"{cache_path.stat().st_size / (1024 ** 2):.2f} MB",
        )


# ===========================================================================
# PART F: ROUND LOG AND CHECKPOINT INVENTORY
# ===========================================================================
print("\n" + "=" * 92)
print("PART F: LOG AND CHECKPOINT INVENTORY")
print("=" * 92)

log_records = []

for path in sorted(RESULTS.glob("*.csv")):
    name_lower = path.name.lower()

    if not any(
        keyword in name_lower
        for keyword in ["round", "weight", "history", "log", "metric"]
    ):
        continue

    try:
        local_path = copy_local(path)
        frame = pd.read_csv(local_path, low_memory=False)

        log_records.append(
            {
                "file": path.name,
                "n_rows": int(len(frame)),
                "columns": "|".join(map(str, frame.columns)),
                "size_mb": round(path.stat().st_size / (1024 ** 2), 4),
            }
        )
    except Exception as error:
        summary["warnings"].append(
            f"Could not inspect log {path.name}: {error}"
        )

log_inventory = pd.DataFrame(log_records)
log_inventory.to_csv(
    LOCAL_OUTPUTS / "round_weight_log_inventory.csv",
    index=False,
)

if len(log_inventory):
    print(log_inventory.to_string(index=False))
else:
    print("No round or weight log CSVs were located.")

checkpoint_records = []

if CHECKPOINTS.exists():
    for path in sorted(CHECKPOINTS.rglob("*.pt")):
        checkpoint_records.append(
            {
                "file": str(path.relative_to(CHECKPOINTS)),
                "size_mb": round(path.stat().st_size / (1024 ** 2), 3),
            }
        )

checkpoint_inventory = pd.DataFrame(checkpoint_records)
checkpoint_inventory.to_csv(
    LOCAL_OUTPUTS / "checkpoint_inventory.csv",
    index=False,
)

print(f"\nCheckpoint files found: {len(checkpoint_inventory):,}")

summary["n_log_files"] = int(len(log_inventory))
summary["n_checkpoint_files"] = int(len(checkpoint_inventory))


# ===========================================================================
# SAVE COMBINED CACHE AUDIT
# ===========================================================================
if cache_audit_frames:
    combined_cache_audit = pd.concat(
        cache_audit_frames,
        ignore_index=True,
    )

    combined_cache_audit.to_csv(
        LOCAL_OUTPUTS / "prediction_cache_audit.csv",
        index=False,
    )


# ===========================================================================
# FINAL DECISION SUMMARY
# ===========================================================================
print("\n" + "=" * 92)
print("RUN 1B DECISION SUMMARY")
print("=" * 92)

print(
    "CheXpert patient-disjointness gate: FAIL "
    "(established by Run 1)"
)

print(
    "Revised role of CheXpert: development dataset and exploratory "
    "study-level internal analysis"
)

if summary.get("nih", {}).get("n_unique_patients", 0) < summary.get(
    "nih", {}
).get("n_images", 0):
    print(
        "NIH patient-cluster resampling: REQUIRED "
        "(multiple images per patient)"
    )

print(f"Errors:   {len(summary['errors'])}")
for error in summary["errors"]:
    print("  ERROR -", error)

print(f"Warnings: {len(summary['warnings'])}")
for warning in summary["warnings"]:
    print("  WARN  -", warning)

with open(
    LOCAL_OUTPUTS / "audit_summary_run1b.json",
    "w",
    encoding="utf-8",
) as file:
    json.dump(summary, file, indent=2)

# Create ZIP locally first.
local_zip = Path("/content/revision_audit_run1b.zip")

with zipfile.ZipFile(
    local_zip,
    "w",
    compression=zipfile.ZIP_DEFLATED,
) as archive:
    for path in sorted(LOCAL_OUTPUTS.rglob("*")):
        if path.is_file():
            archive.write(
                path,
                arcname=path.relative_to(LOCAL_OUTPUTS),
            )

# Copy final ZIP to Drive only once.
drive_zip = RESULTS / "revision_audit_run1b.zip"
shutil.copy2(local_zip, drive_zip)

print("\nRUN 1B COMPLETE")
print("Local ZIP:", local_zip)
print("Drive ZIP:", drive_zip)
print("=" * 92)

Mounted at /content/drive
RUN 1B: NIH, CACHE, LOG, AND CHECKPOINT AUDIT
BASE: /content/drive/MyDrive/FairFedCXR
Drive accessible: True

PART A: CORRECTED CHEXPERT PATIENT/STUDY SUMMARY
client split  n_images  n_patients  n_patient_studies  prevalence  male_fraction  positive_images
     A  test      2250        2173               2247    0.400000       0.498222              900
     A train     10500        9064              10472    0.400000       0.500952             4200
     A   val      2250        2169               2249    0.400000       0.497333              900
     B  test      1200        1174               1200    0.450000       0.778333              540
     B train      5600        5032               5589    0.450000       0.801964             2520
     B   val      1200        1163               1199    0.450000       0.812500              540
     C  test      1035        1004               1034    0.376812       0.333333              390
     C train      4829        4

In [ ]:
# ============================================================================
# RUN 2: PATIENT-AWARE NIH DESCRIPTIVE RECONSTRUCTION
# CPU only. No training. No model inference.
#
# Outputs:
#   /content/drive/MyDrive/FairFedCXR/results/revision_run2.zip
#
# This run:
#   1. Verifies the NIH cohort and prediction cache again.
#   2. Computes every metric independently for every method and seed.
#   3. Does NOT average probabilities for the primary descriptive summary.
#   4. Calculates seed-ensemble results only as a secondary diagnostic.
#   5. Creates paired run-level difference tables.
#   6. Creates raw-seed diagnostic plots.
# ============================================================================

from google.colab import drive
drive.mount("/content/drive", force_remount=True)

from pathlib import Path
import json
import shutil
import zipfile

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from sklearn.metrics import (
    average_precision_score,
    brier_score_loss,
    roc_auc_score,
)

# ---------------------------------------------------------------------------
# Paths
# ---------------------------------------------------------------------------
BASE = Path("/content/drive/MyDrive/FairFedCXR")
RESULTS = BASE / "results"

LOCAL = Path("/content/revision_run2")
OUTPUTS = LOCAL / "outputs"
OUTPUTS.mkdir(parents=True, exist_ok=True)

NIH_CSV = RESULTS / "nih_effusion_cohort.csv"
MAIN_CACHE = RESULTS / "nih_pred_cache.npz"
SCRAMBLED_CACHE = RESULTS / "nih_scrambled_pred_cache.npz"

for path in [NIH_CSV, MAIN_CACHE, SCRAMBLED_CACHE]:
    if not path.exists():
        raise FileNotFoundError(f"Required file not found: {path}")

print("=" * 94)
print("RUN 2: PATIENT-AWARE NIH DESCRIPTIVE RECONSTRUCTION")
print("=" * 94)

# ---------------------------------------------------------------------------
# Copy inputs locally before reading
# ---------------------------------------------------------------------------
local_nih_csv = LOCAL / NIH_CSV.name
local_main_cache = LOCAL / MAIN_CACHE.name
local_scrambled_cache = LOCAL / SCRAMBLED_CACHE.name

shutil.copy2(NIH_CSV, local_nih_csv)
shutil.copy2(MAIN_CACHE, local_main_cache)
shutil.copy2(SCRAMBLED_CACHE, local_scrambled_cache)

# ---------------------------------------------------------------------------
# Load and validate cohort
# ---------------------------------------------------------------------------
nih = pd.read_csv(local_nih_csv, low_memory=False)

required_columns = {
    "Patient ID",
    "label",
    "sex_encoded",
}

missing_columns = required_columns - set(nih.columns)
if missing_columns:
    raise KeyError(
        f"NIH cohort is missing columns: {sorted(missing_columns)}"
    )

nih = nih.copy()
nih["patient_id"] = nih["Patient ID"].astype(str).str.strip()
nih["label"] = nih["label"].astype(int)
nih["sex_encoded"] = nih["sex_encoded"].astype(int)
nih["image_row"] = np.arange(len(nih), dtype=np.int64)

assert nih["label"].isin([0, 1]).all()
assert nih["sex_encoded"].isin([0, 1]).all()

# Confirm sex encoding where the text Sex field exists.
if "Sex" in nih.columns:
    male_rows = nih["Sex"].astype(str).str.lower().eq("male")
    female_rows = nih["Sex"].astype(str).str.lower().eq("female")

    if male_rows.any():
        assert (
            nih.loc[male_rows, "sex_encoded"] == 1
        ).all(), "Male must be encoded as 1."

    if female_rows.any():
        assert (
            nih.loc[female_rows, "sex_encoded"] == 0
        ).all(), "Female must be encoded as 0."

# Create stable integer cluster codes.
patient_codes, patient_levels = pd.factorize(
    nih["patient_id"],
    sort=True,
)

nih["patient_code"] = patient_codes.astype(np.int32)

print(f"Images:                  {len(nih):,}")
print(f"Unique patients:         {nih['patient_id'].nunique():,}")
print(
    f"Patients with >1 image:  "
    f"{(nih.groupby('patient_id').size() > 1).sum():,}"
)

# ---------------------------------------------------------------------------
# Load caches and confirm row order
# ---------------------------------------------------------------------------
main = np.load(local_main_cache, allow_pickle=True)
predictions = main["all_probs"].item()

cached_labels = np.asarray(main["labels"]).reshape(-1).astype(int)
cached_sex = np.asarray(main["sex"]).reshape(-1).astype(int)

assert len(cached_labels) == len(nih)
assert len(cached_sex) == len(nih)
assert np.array_equal(cached_labels, nih["label"].to_numpy())
assert np.array_equal(cached_sex, nih["sex_encoded"].to_numpy())

scrambled = np.load(
    local_scrambled_cache,
    allow_pickle=True,
)["probs"].item()

for key, probabilities in predictions.items():
    probabilities = np.asarray(probabilities).reshape(-1)

    assert len(probabilities) == len(nih), (
        f"{key}: incorrect prediction length"
    )
    assert np.isfinite(probabilities).all(), (
        f"{key}: non-finite probabilities"
    )
    assert probabilities.min() >= 0
    assert probabilities.max() <= 1

for seed, probabilities in scrambled.items():
    probabilities = np.asarray(probabilities).reshape(-1)

    assert len(probabilities) == len(nih)
    assert np.isfinite(probabilities).all()
    assert probabilities.min() >= 0
    assert probabilities.max() <= 1

# ---------------------------------------------------------------------------
# Manuscript-safe method names
# ---------------------------------------------------------------------------
METHOD_NAMES = {
    "fedavg": "FedAvg",
    "qfedavg": "LPR",
    "fairfed": "CADR",
    "dwfa": "DWFA",
    "dwfa_scrambled": "DWFA-Scrambled",
}

METHOD_ORDER = [
    "FedAvg",
    "LPR",
    "CADR",
    "DWFA",
    "DWFA-Scrambled",
]

METHOD_EXPLANATIONS = pd.DataFrame(
    [
        {
            "cache_name": "fedavg",
            "reporting_name": "FedAvg",
            "description": "Sample-size-weighted federated averaging",
        },
        {
            "cache_name": "qfedavg",
            "reporting_name": "LPR",
            "description": (
                "Loss-Powered Reweighting; controlled server-side "
                "loss-powered comparator"
            ),
        },
        {
            "cache_name": "fairfed",
            "reporting_name": "CADR",
            "description": (
                "Cumulative Additive Disparity Reweighting; controlled "
                "additive fairness-deviation comparator"
            ),
        },
        {
            "cache_name": "dwfa",
            "reporting_name": "DWFA",
            "description": "Dual-Weighted Fair Aggregation",
        },
        {
            "cache_name": "scrambled",
            "reporting_name": "DWFA-Scrambled",
            "description": (
                "DWFA with client-specific fairness signals reassigned "
                "across clients"
            ),
        },
    ]
)

METHOD_EXPLANATIONS.to_csv(
    OUTPUTS / "method_reporting_names.csv",
    index=False,
)

# ---------------------------------------------------------------------------
# Metric definitions
# ---------------------------------------------------------------------------
LABELS = nih["label"].to_numpy(dtype=int)
SEX = nih["sex_encoded"].to_numpy(dtype=int)

MALE = SEX == 1
FEMALE = SEX == 0

EPS = 1.0
ECE_BINS = 8


def safe_auroc(labels, probabilities):
    labels = np.asarray(labels)
    probabilities = np.asarray(probabilities)

    if np.unique(labels).size < 2:
        return np.nan

    return float(roc_auc_score(labels, probabilities))


def safe_auprc(labels, probabilities):
    labels = np.asarray(labels)
    probabilities = np.asarray(probabilities)

    if labels.sum() == 0:
        return np.nan

    return float(average_precision_score(labels, probabilities))


def expected_calibration_error(
    probabilities,
    labels,
    n_bins=ECE_BINS,
):
    """
    Equal-width ECE.

    The final bin includes probability 1.0.
    This is retained only for continuity with the original analysis.
    """
    probabilities = np.asarray(probabilities)
    labels = np.asarray(labels)

    boundaries = np.linspace(0.0, 1.0, n_bins + 1)
    total = len(labels)
    ece = 0.0

    for bin_index in range(n_bins):
        lower = boundaries[bin_index]
        upper = boundaries[bin_index + 1]

        if bin_index == n_bins - 1:
            mask = (
                (probabilities >= lower)
                & (probabilities <= upper)
            )
        else:
            mask = (
                (probabilities >= lower)
                & (probabilities < upper)
            )

        count = int(mask.sum())

        if count == 0:
            continue

        observed = float(labels[mask].mean())
        predicted = float(probabilities[mask].mean())

        ece += (count / total) * abs(observed - predicted)

    return float(ece)


def smoothed_rate(numerator, denominator, eps=EPS):
    return float(
        (numerator + eps)
        / (denominator + 2.0 * eps)
    )


def threshold_metrics(
    probabilities,
    labels,
    sex,
    threshold=0.5,
):
    probabilities = np.asarray(probabilities)
    labels = np.asarray(labels)
    sex = np.asarray(sex)

    predicted = probabilities >= threshold

    results = {}

    for group_name, group_value in [
        ("male", 1),
        ("female", 0),
    ]:
        group = sex == group_value

        positive = group & (labels == 1)
        negative = group & (labels == 0)

        tp = int((predicted & positive).sum())
        fn = int((~predicted & positive).sum())
        fp = int((predicted & negative).sum())
        tn = int((~predicted & negative).sum())

        n_positive = tp + fn
        n_negative = fp + tn

        tpr_smoothed = smoothed_rate(tp, n_positive)
        tpr_raw = (
            float(tp / n_positive)
            if n_positive > 0
            else np.nan
        )
        fpr_raw = (
            float(fp / n_negative)
            if n_negative > 0
            else np.nan
        )

        results[f"{group_name}_tp"] = tp
        results[f"{group_name}_fn"] = fn
        results[f"{group_name}_fp"] = fp
        results[f"{group_name}_tn"] = tn

        results[f"{group_name}_tpr_smoothed"] = tpr_smoothed
        results[f"{group_name}_tpr_raw"] = tpr_raw
        results[f"{group_name}_fpr_raw"] = fpr_raw

    results["eo_gap_smoothed_50"] = abs(
        results["male_tpr_smoothed"]
        - results["female_tpr_smoothed"]
    )

    results["eo_gap_raw_50"] = abs(
        results["male_tpr_raw"]
        - results["female_tpr_raw"]
    )

    results["fpr_gap_50"] = abs(
        results["male_fpr_raw"]
        - results["female_fpr_raw"]
    )

    return results


def calculate_metrics(probabilities):
    probabilities = np.asarray(
        probabilities,
        dtype=np.float64,
    ).reshape(-1)

    male_auroc = safe_auroc(
        LABELS[MALE],
        probabilities[MALE],
    )

    female_auroc = safe_auroc(
        LABELS[FEMALE],
        probabilities[FEMALE],
    )

    male_auprc = safe_auprc(
        LABELS[MALE],
        probabilities[MALE],
    )

    female_auprc = safe_auprc(
        LABELS[FEMALE],
        probabilities[FEMALE],
    )

    male_ece = expected_calibration_error(
        probabilities[MALE],
        LABELS[MALE],
    )

    female_ece = expected_calibration_error(
        probabilities[FEMALE],
        LABELS[FEMALE],
    )

    metrics = {
        "auroc": safe_auroc(
            LABELS,
            probabilities,
        ),
        "male_auroc": male_auroc,
        "female_auroc": female_auroc,
        "worst_group_auroc": min(
            male_auroc,
            female_auroc,
        ),
        "auprc": safe_auprc(
            LABELS,
            probabilities,
        ),
        "male_auprc": male_auprc,
        "female_auprc": female_auprc,
        "worst_group_auprc": min(
            male_auprc,
            female_auprc,
        ),
        "brier_score": float(
            brier_score_loss(
                LABELS,
                probabilities,
            )
        ),
        "ece_8bin": expected_calibration_error(
            probabilities,
            LABELS,
        ),
        "male_ece_8bin": male_ece,
        "female_ece_8bin": female_ece,
        "ece_gap_8bin": abs(
            male_ece - female_ece
        ),
        "mean_probability": float(
            probabilities.mean()
        ),
    }

    metrics.update(
        threshold_metrics(
            probabilities,
            LABELS,
            SEX,
            threshold=0.5,
        )
    )

    return metrics


# ---------------------------------------------------------------------------
# Cohort summary
# ---------------------------------------------------------------------------
patient_summary = (
    nih.groupby("patient_id")
    .agg(
        n_images=("image_row", "size"),
        sex_encoded=("sex_encoded", "first"),
        n_sex_values=("sex_encoded", "nunique"),
        any_positive=("label", "max"),
        all_positive=("label", "min"),
        n_positive_images=("label", "sum"),
        n_label_values=("label", "nunique"),
    )
    .reset_index()
)

assert (patient_summary["n_sex_values"] == 1).all()

cohort_rows = []

for unit_name, frame, label_column in [
    ("image", nih, "label"),
    ("patient_any_positive", patient_summary, "any_positive"),
]:
    for group_name, group_value in [
        ("all", None),
        ("male", 1),
        ("female", 0),
    ]:
        if group_value is None:
            subset = frame
        else:
            subset = frame[
                frame["sex_encoded"] == group_value
            ]

        cohort_rows.append(
            {
                "unit": unit_name,
                "sex_group": group_name,
                "n": int(len(subset)),
                "n_positive": int(
                    subset[label_column].sum()
                ),
                "prevalence": float(
                    subset[label_column].mean()
                ),
            }
        )

cohort_summary = pd.DataFrame(cohort_rows)

additional_cluster_summary = pd.DataFrame(
    [
        {
            "n_images": int(len(nih)),
            "n_patients": int(
                patient_summary["patient_id"].nunique()
            ),
            "patients_with_multiple_images": int(
                (patient_summary["n_images"] > 1).sum()
            ),
            "patients_with_mixed_image_labels": int(
                (patient_summary["n_label_values"] > 1).sum()
            ),
            "median_images_per_patient": float(
                patient_summary["n_images"].median()
            ),
            "mean_images_per_patient": float(
                patient_summary["n_images"].mean()
            ),
            "maximum_images_per_patient": int(
                patient_summary["n_images"].max()
            ),
        }
    ]
)

cohort_summary.to_csv(
    OUTPUTS / "nih_cohort_summary.csv",
    index=False,
)

additional_cluster_summary.to_csv(
    OUTPUTS / "nih_cluster_characteristics.csv",
    index=False,
)

patient_summary.to_csv(
    OUTPUTS / "nih_patient_cluster_table.csv",
    index=False,
)

# Save a lightweight row-to-cluster index for later scripts.
nih[
    [
        "image_row",
        "patient_id",
        "patient_code",
        "label",
        "sex_encoded",
    ]
].to_csv(
    OUTPUTS / "nih_row_cluster_index.csv",
    index=False,
)

print("\nNIH cohort summary:")
print(cohort_summary.to_string(index=False))

print("\nNIH cluster characteristics:")
print(additional_cluster_summary.to_string(index=False))

# ---------------------------------------------------------------------------
# Per-seed metrics: the primary descriptive results
# ---------------------------------------------------------------------------
per_seed_rows = []

for key, probabilities in sorted(
    predictions.items(),
    key=lambda item: (str(item[0][0]), int(item[0][1])),
):
    method_cache_name, seed = key
    reporting_name = METHOD_NAMES.get(
        method_cache_name,
        method_cache_name,
    )

    row = {
        "cache_method": method_cache_name,
        "method": reporting_name,
        "seed": int(seed),
        "analysis_type": "individual_training_run",
    }

    row.update(
        calculate_metrics(probabilities)
    )

    per_seed_rows.append(row)

for seed, probabilities in sorted(
    scrambled.items(),
    key=lambda item: int(item[0]),
):
    row = {
        "cache_method": "dwfa_scrambled",
        "method": "DWFA-Scrambled",
        "seed": int(seed),
        "analysis_type": "individual_training_run",
    }

    row.update(
        calculate_metrics(probabilities)
    )

    per_seed_rows.append(row)

per_seed_metrics = pd.DataFrame(per_seed_rows)

per_seed_metrics["method"] = pd.Categorical(
    per_seed_metrics["method"],
    categories=METHOD_ORDER,
    ordered=True,
)

per_seed_metrics = per_seed_metrics.sort_values(
    ["method", "seed"]
).reset_index(drop=True)

per_seed_metrics.to_csv(
    OUTPUTS / "nih_metrics_per_seed_recomputed.csv",
    index=False,
)

print("\nPer-seed metrics:")
display_columns = [
    "method",
    "seed",
    "auroc",
    "male_auroc",
    "female_auroc",
    "worst_group_auroc",
    "eo_gap_smoothed_50",
    "fpr_gap_50",
    "brier_score",
    "ece_gap_8bin",
]

print(
    per_seed_metrics[display_columns]
    .to_string(
        index=False,
        float_format=lambda value: f"{value:.6f}",
    )
)

# ---------------------------------------------------------------------------
# Across-run descriptive summaries
# ---------------------------------------------------------------------------
summary_metrics = [
    "auroc",
    "male_auroc",
    "female_auroc",
    "worst_group_auroc",
    "auprc",
    "worst_group_auprc",
    "eo_gap_smoothed_50",
    "eo_gap_raw_50",
    "fpr_gap_50",
    "brier_score",
    "ece_8bin",
    "ece_gap_8bin",
]

method_summary_rows = []

for method, frame in per_seed_metrics.groupby(
    "method",
    observed=True,
):
    for metric in summary_metrics:
        values = frame[metric].dropna().to_numpy()

        method_summary_rows.append(
            {
                "method": str(method),
                "metric": metric,
                "n_runs": int(len(values)),
                "mean": float(values.mean()),
                "sd": (
                    float(values.std(ddof=1))
                    if len(values) > 1
                    else np.nan
                ),
                "minimum": float(values.min()),
                "maximum": float(values.max()),
            }
        )

method_summary = pd.DataFrame(method_summary_rows)

method_summary.to_csv(
    OUTPUTS / "nih_method_summary_across_runs.csv",
    index=False,
)

print("\nAcross-run summary: AUROC, worst-group AUROC, EO gap")
print(
    method_summary[
        method_summary["metric"].isin(
            [
                "auroc",
                "worst_group_auroc",
                "eo_gap_smoothed_50",
            ]
        )
    ].to_string(
        index=False,
        float_format=lambda value: f"{value:.6f}",
    )
)

# ---------------------------------------------------------------------------
# Seed-ensemble metrics: secondary diagnostic only
# ---------------------------------------------------------------------------
ensemble_rows = []

for cache_method in [
    "fedavg",
    "qfedavg",
    "fairfed",
    "dwfa",
]:
    available_seeds = sorted(
        seed
        for method, seed in predictions
        if method == cache_method
    )

    if not available_seeds:
        continue

    ensemble_probabilities = np.mean(
        [
            np.asarray(
                predictions[(cache_method, seed)],
                dtype=np.float64,
            )
            for seed in available_seeds
        ],
        axis=0,
    )

    row = {
        "cache_method": cache_method,
        "method": METHOD_NAMES[cache_method],
        "n_ensemble_members": len(available_seeds),
        "seeds": ",".join(
            map(str, available_seeds)
        ),
        "analysis_type": (
            "seed_probability_ensemble_secondary_diagnostic"
        ),
    }

    row.update(
        calculate_metrics(
            ensemble_probabilities
        )
    )

    ensemble_rows.append(row)

ensemble_metrics = pd.DataFrame(ensemble_rows)

ensemble_metrics.to_csv(
    OUTPUTS / "nih_seed_ensemble_metrics_secondary.csv",
    index=False,
)

print("\nSeed-ensemble metrics — secondary diagnostic only:")
print(
    ensemble_metrics[
        [
            "method",
            "n_ensemble_members",
            "auroc",
            "worst_group_auroc",
            "eo_gap_smoothed_50",
        ]
    ].to_string(
        index=False,
        float_format=lambda value: f"{value:.6f}",
    )
)

# ---------------------------------------------------------------------------
# Paired run-level differences
# These are descriptive because equal numeric seeds do not guarantee fully
# identical stochastic paths across aggregation methods.
# ---------------------------------------------------------------------------
metric_direction = {
    "auroc": "higher_is_better",
    "worst_group_auroc": "higher_is_better",
    "auprc": "higher_is_better",
    "worst_group_auprc": "higher_is_better",
    "eo_gap_smoothed_50": "lower_is_better",
    "fpr_gap_50": "lower_is_better",
    "brier_score": "lower_is_better",
    "ece_gap_8bin": "lower_is_better",
}

comparisons = [
    ("DWFA", "LPR"),
    ("DWFA", "CADR"),
    ("LPR", "CADR"),
    ("DWFA", "FedAvg"),
    ("LPR", "FedAvg"),
    ("CADR", "FedAvg"),
    ("DWFA", "DWFA-Scrambled"),
]

paired_rows = []

for method_a, method_b in comparisons:
    frame_a = per_seed_metrics[
        per_seed_metrics["method"].astype(str) == method_a
    ].copy()

    frame_b = per_seed_metrics[
        per_seed_metrics["method"].astype(str) == method_b
    ].copy()

    common_seeds = sorted(
        set(frame_a["seed"])
        & set(frame_b["seed"])
    )

    for seed in common_seeds:
        row_a = frame_a[
            frame_a["seed"] == seed
        ].iloc[0]

        row_b = frame_b[
            frame_b["seed"] == seed
        ].iloc[0]

        for metric, direction in metric_direction.items():
            paired_rows.append(
                {
                    "method_a": method_a,
                    "method_b": method_b,
                    "comparison": (
                        f"{method_a} minus {method_b}"
                    ),
                    "seed": int(seed),
                    "metric": metric,
                    "direction": direction,
                    "method_a_value": float(
                        row_a[metric]
                    ),
                    "method_b_value": float(
                        row_b[metric]
                    ),
                    "raw_difference_a_minus_b": float(
                        row_a[metric]
                        - row_b[metric]
                    ),
                }
            )

paired_differences = pd.DataFrame(paired_rows)

paired_differences.to_csv(
    OUTPUTS / "nih_same_seed_descriptive_differences.csv",
    index=False,
)

paired_summary = (
    paired_differences.groupby(
        [
            "method_a",
            "method_b",
            "comparison",
            "metric",
            "direction",
        ]
    )
    .agg(
        n_common_seeds=("seed", "nunique"),
        mean_raw_difference=(
            "raw_difference_a_minus_b",
            "mean",
        ),
        sd_raw_difference=(
            "raw_difference_a_minus_b",
            "std",
        ),
        minimum_raw_difference=(
            "raw_difference_a_minus_b",
            "min",
        ),
        maximum_raw_difference=(
            "raw_difference_a_minus_b",
            "max",
        ),
        n_positive=(
            "raw_difference_a_minus_b",
            lambda values: int((values > 0).sum()),
        ),
        n_negative=(
            "raw_difference_a_minus_b",
            lambda values: int((values < 0).sum()),
        ),
        n_zero=(
            "raw_difference_a_minus_b",
            lambda values: int((values == 0).sum()),
        ),
    )
    .reset_index()
)

paired_summary.to_csv(
    OUTPUTS / "nih_same_seed_difference_summary.csv",
    index=False,
)

print("\nSame-seed descriptive differences:")
print(
    paired_summary[
        paired_summary["metric"].isin(
            [
                "auroc",
                "worst_group_auroc",
                "eo_gap_smoothed_50",
            ]
        )
    ].to_string(
        index=False,
        float_format=lambda value: f"{value:.6f}",
    )
)

# ---------------------------------------------------------------------------
# Diagnostic raw-seed figures
# ---------------------------------------------------------------------------
def create_seed_plot(metric, ylabel, filename):
    plotting = per_seed_metrics[
        per_seed_metrics["method"].astype(str).isin(
            METHOD_ORDER
        )
    ].copy()

    present_methods = [
        method
        for method in METHOD_ORDER
        if method in set(
            plotting["method"].astype(str)
        )
    ]

    figure, axis = plt.subplots(
        figsize=(8.5, 5.2)
    )

    for x_position, method in enumerate(
        present_methods
    ):
        values = plotting.loc[
            plotting["method"].astype(str) == method,
            metric,
        ].dropna().to_numpy()

        if len(values) == 0:
            continue

        offsets = np.linspace(
            -0.09,
            0.09,
            len(values),
        )

        axis.scatter(
            np.full(len(values), x_position)
            + offsets,
            values,
            s=50,
            label=method,
        )

        mean = values.mean()

        if len(values) > 1:
            sd = values.std(ddof=1)

            axis.errorbar(
                x_position,
                mean,
                yerr=sd,
                marker="D",
                capsize=5,
                linewidth=1.5,
            )
        else:
            axis.scatter(
                [x_position],
                [mean],
                marker="D",
                s=55,
            )

    axis.set_xticks(
        range(len(present_methods))
    )
    axis.set_xticklabels(
        present_methods,
        rotation=20,
        ha="right",
    )

    axis.set_ylabel(ylabel)
    axis.set_xlabel("Aggregation procedure")
    axis.grid(axis="y", alpha=0.25)

    figure.tight_layout()

    figure.savefig(
        OUTPUTS / f"{filename}.pdf",
        bbox_inches="tight",
    )

    figure.savefig(
        OUTPUTS / f"{filename}.png",
        dpi=300,
        bbox_inches="tight",
    )

    plt.close(figure)


create_seed_plot(
    metric="auroc",
    ylabel="External-test AUROC",
    filename="diagnostic_seed_auroc",
)

create_seed_plot(
    metric="worst_group_auroc",
    ylabel="External-test worst-group AUROC",
    filename="diagnostic_seed_worst_group_auroc",
)

create_seed_plot(
    metric="eo_gap_smoothed_50",
    ylabel="Equal-opportunity gap at threshold 0.5",
    filename="diagnostic_seed_eo_gap",
)

# ---------------------------------------------------------------------------
# Save machine-readable bundle metadata
# ---------------------------------------------------------------------------
run_summary = {
    "n_images": int(len(nih)),
    "n_patients": int(
        nih["patient_id"].nunique()
    ),
    "n_main_prediction_arrays": int(
        len(predictions)
    ),
    "n_scrambled_prediction_arrays": int(
        len(scrambled)
    ),
    "main_cache_methods_and_seeds": {
        method: sorted(
            int(seed)
            for cached_method, seed in predictions
            if cached_method == method
        )
        for method in sorted(
            set(method for method, _ in predictions)
        )
    },
    "scrambled_seeds": sorted(
        int(seed)
        for seed in scrambled
    ),
    "primary_descriptive_unit": (
        "individual trained model evaluated on NIH radiographs"
    ),
    "cluster_unit_for_future_inference": (
        "unique NIH patient ID"
    ),
    "threshold_policy": {
        "primary_discrimination": (
            "threshold-free AUROC"
        ),
        "current_threshold_audit": 0.5,
        "nih_optimized_thresholds_allowed": False,
    },
}

with open(
    OUTPUTS / "run2_summary.json",
    "w",
    encoding="utf-8",
) as file:
    json.dump(
        run_summary,
        file,
        indent=2,
    )

# ---------------------------------------------------------------------------
# Zip outputs
# ---------------------------------------------------------------------------
local_zip = Path(
    "/content/revision_run2.zip"
)

with zipfile.ZipFile(
    local_zip,
    "w",
    compression=zipfile.ZIP_DEFLATED,
) as archive:
    for path in sorted(
        OUTPUTS.rglob("*")
    ):
        if path.is_file():
            archive.write(
                path,
                arcname=path.relative_to(
                    OUTPUTS
                ),
            )

drive_zip = RESULTS / "revision_run2.zip"
shutil.copy2(local_zip, drive_zip)

print("\n" + "=" * 94)
print("RUN 2 COMPLETE")
print("=" * 94)
print("Local ZIP:", local_zip)
print("Drive ZIP:", drive_zip)
print("\nUpload revision_run2.zip for review.")

Mounted at /content/drive
RUN 2: PATIENT-AWARE NIH DESCRIPTIVE RECONSTRUCTION
Images:                  112,120
Unique patients:         30,805
Patients with >1 image:  13,302

NIH cohort summary:
                unit sex_group      n  n_positive  prevalence
               image       all 112120       13307    0.118685
               image      male  63340        7427    0.117256
               image    female  48780        5880    0.120541
patient_any_positive       all  30805        4273    0.138711
patient_any_positive      male  16630        2300    0.138304
patient_any_positive    female  14175        1973    0.139189

NIH cluster characteristics:
 n_images  n_patients  patients_with_multiple_images  patients_with_mixed_image_labels  median_images_per_patient  mean_images_per_patient  maximum_images_per_patient
   112120       30805                          13302                              3634                        1.0                 3.639669                         184

Per-s

In [ ]:
# ============================================================================
# RUN 3: PATIENT-CLUSTERED, RUN-AWARE NIH INFERENCE
#
# No training. No checkpoint inference.
# L4 GPU strongly recommended.
#
# Main outputs:
#   revision_run3.zip
#   run3_primary_inference.csv
#   run3_fixed_ensemble_patient_cluster.csv
#   run3_placebo_inference.csv
#   run3_exact_run_level_tests.csv
# ============================================================================

from google.colab import drive
drive.mount("/content/drive", force_remount=True)

from pathlib import Path
from itertools import combinations, product
import json
import shutil
import zipfile

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from sklearn.metrics import roc_auc_score


# ---------------------------------------------------------------------------
# Configuration
# ---------------------------------------------------------------------------
BASE = Path("/content/drive/MyDrive/FairFedCXR")
RESULTS = BASE / "results"

LOCAL = Path("/content/revision_run3")
OUTPUTS = LOCAL / "outputs"
OUTPUTS.mkdir(parents=True, exist_ok=True)

NIH_CSV = RESULTS / "nih_effusion_cohort.csv"
MAIN_CACHE = RESULTS / "nih_pred_cache.npz"
SCRAMBLED_CACHE = RESULTS / "nih_scrambled_pred_cache.npz"

for required_path in [NIH_CSV, MAIN_CACHE, SCRAMBLED_CACHE]:
    if not required_path.exists():
        raise FileNotFoundError(required_path)

# 2,000 is sufficient for the first full inferential pass.
# We can increase it later only if an interval is numerically borderline.
N_BOOTSTRAP = 2000
BATCH_SIZE = 25
BOOTSTRAP_SEED = 20260723
FIXED_THRESHOLD = 0.5

print("=" * 96)
print("RUN 3: PATIENT-CLUSTERED, RUN-AWARE NIH INFERENCE")
print("=" * 96)


# ---------------------------------------------------------------------------
# Copy Drive inputs to local storage
# ---------------------------------------------------------------------------
for source in [NIH_CSV, MAIN_CACHE, SCRAMBLED_CACHE]:
    shutil.copy2(source, LOCAL / source.name)


# ---------------------------------------------------------------------------
# Load NIH cohort and predictions
# ---------------------------------------------------------------------------
nih = pd.read_csv(
    LOCAL / NIH_CSV.name,
    low_memory=False,
)

nih = nih.copy()
nih["patient_id"] = (
    nih["Patient ID"]
    .astype(str)
    .str.strip()
)

nih["label"] = nih["label"].astype(np.int8)
nih["sex_encoded"] = nih["sex_encoded"].astype(np.int8)

patient_code, patient_levels = pd.factorize(
    nih["patient_id"],
    sort=True,
)

patient_code = patient_code.astype(np.int32)

labels = nih["label"].to_numpy(np.int8)
sex = nih["sex_encoded"].to_numpy(np.int8)

n_images = len(nih)
n_patients = len(patient_levels)

main_cache = np.load(
    LOCAL / MAIN_CACHE.name,
    allow_pickle=True,
)

main_predictions = main_cache["all_probs"].item()

scrambled_predictions = np.load(
    LOCAL / SCRAMBLED_CACHE.name,
    allow_pickle=True,
)["probs"].item()

assert np.array_equal(
    main_cache["labels"].astype(np.int8),
    labels,
)

assert np.array_equal(
    main_cache["sex"].astype(np.int8),
    sex,
)

print(f"Images:                 {n_images:,}")
print(f"Unique patients:        {n_patients:,}")
print(f"Main prediction arrays: {len(main_predictions):,}")
print(f"Scrambled arrays:       {len(scrambled_predictions):,}")


# ---------------------------------------------------------------------------
# Analysis labels
# ---------------------------------------------------------------------------
METHOD_LABELS = {
    "fedavg": "FedAvg",
    "qfedavg": "LPR",
    "fairfed": "CADR",
    "dwfa": "DWFA",
}

CORE_METHODS = [
    "LPR",
    "CADR",
    "DWFA",
]

PRIMARY_COMPARISONS = [
    ("DWFA", "LPR"),
    ("DWFA", "CADR"),
    ("LPR", "CADR"),
]

PRIMARY_METRICS = [
    "auroc",
    "worst_group_auroc",
]

ALL_BOOTSTRAP_METRICS = [
    "auroc",
    "male_auroc",
    "female_auroc",
    "worst_group_auroc",
    "eo_gap_smoothed_50",
]


# ---------------------------------------------------------------------------
# Organize model arrays
# ---------------------------------------------------------------------------
model_probabilities = {}

method_runs = {
    "FedAvg": [],
    "LPR": [],
    "CADR": [],
    "DWFA": [],
    "DWFA-Scrambled": [],
}

for (cache_method, seed), probabilities in sorted(
    main_predictions.items(),
    key=lambda item: (
        item[0][0],
        int(item[0][1]),
    ),
):
    method = METHOD_LABELS[cache_method]
    model_id = f"{method}|{int(seed)}"

    model_probabilities[model_id] = np.asarray(
        probabilities,
        dtype=np.float32,
    )

    method_runs[method].append(model_id)

for seed, probabilities in sorted(
    scrambled_predictions.items(),
    key=lambda item: int(item[0]),
):
    model_id = f"DWFA-Scrambled|{int(seed)}"

    model_probabilities[model_id] = np.asarray(
        probabilities,
        dtype=np.float32,
    )

    method_runs["DWFA-Scrambled"].append(model_id)

for method in method_runs:
    method_runs[method] = sorted(
        method_runs[method],
        key=lambda model_id: int(
            model_id.split("|")[1]
        ),
    )


# Seed-probability ensembles are retained only as secondary estimands.
ensemble_probabilities = {}

for method in [
    "FedAvg",
    "LPR",
    "CADR",
    "DWFA",
    "DWFA-Scrambled",
]:
    model_ids = method_runs[method]

    ensemble_probabilities[
        f"ENSEMBLE|{method}"
    ] = np.mean(
        [
            model_probabilities[model_id]
            for model_id in model_ids
        ],
        axis=0,
    ).astype(np.float32)

analysis_probabilities = {
    **model_probabilities,
    **ensemble_probabilities,
}

print(f"Individual models: {len(model_probabilities):,}")
print(f"Secondary ensembles: {len(ensemble_probabilities):,}")


# ---------------------------------------------------------------------------
# Use CuPy when an L4 GPU is available
# ---------------------------------------------------------------------------
USE_GPU = False

try:
    import cupy as cp

    if cp.cuda.runtime.getDeviceCount() > 0:
        USE_GPU = True
        print("Weighted-AUROC backend: CuPy GPU")

except Exception as error:
    print(
        "CuPy unavailable. NumPy CPU fallback will be used:",
        repr(error),
    )


# ---------------------------------------------------------------------------
# Generate patient-cluster bootstrap multiplicities
#
# Every bootstrap replicate samples 30,805 patients with replacement.
# Every image belonging to a sampled patient receives the same multiplicity.
# ---------------------------------------------------------------------------
rng_counts = np.random.default_rng(
    BOOTSTRAP_SEED
)

patient_counts = np.empty(
    (N_BOOTSTRAP, n_patients),
    dtype=np.uint16,
)

for bootstrap_index in range(N_BOOTSTRAP):
    sampled_patients = rng_counts.integers(
        0,
        n_patients,
        size=n_patients,
        dtype=np.int32,
    )

    patient_counts[bootstrap_index] = np.bincount(
        sampled_patients,
        minlength=n_patients,
    ).astype(np.uint16)

assert np.all(
    patient_counts.sum(axis=1)
    == n_patients
)

print(
    "Patient-cluster bootstrap matrix:",
    f"{patient_counts.nbytes / 1024**2:.1f} MB",
)


# ---------------------------------------------------------------------------
# Exact weighted AUROC, including tied prediction scores
# ---------------------------------------------------------------------------
def prepare_auc_plan(
    probabilities,
    mask,
):
    probabilities = np.asarray(
        probabilities,
        dtype=np.float32,
    )[mask]

    group_labels = labels[mask].astype(
        np.float32
    )

    group_patient_codes = patient_code[mask]

    order = np.argsort(
        probabilities,
        kind="mergesort",
    )

    sorted_probabilities = probabilities[order]
    sorted_labels = group_labels[order]

    sorted_patient_codes = (
        group_patient_codes[order]
        .astype(np.int32)
    )

    tie_starts = np.r_[
        0,
        np.flatnonzero(
            np.diff(sorted_probabilities)
            != 0
        ) + 1,
    ].astype(np.int32)

    return {
        "labels": sorted_labels,
        "patient_codes": sorted_patient_codes,
        "tie_starts": tie_starts,
    }


def auc_from_plan_numpy(
    plan,
    bootstrap_counts,
):
    output = np.empty(
        len(bootstrap_counts),
        dtype=np.float64,
    )

    sorted_labels = plan["labels"].astype(
        np.float64
    )

    sorted_codes = plan["patient_codes"]
    tie_starts = plan["tie_starts"]

    for start in range(
        0,
        len(bootstrap_counts),
        BATCH_SIZE,
    ):
        stop = min(
            start + BATCH_SIZE,
            len(bootstrap_counts),
        )

        image_weights = bootstrap_counts[
            start:stop,
            sorted_codes,
        ].astype(np.float64)

        positive_weights = (
            image_weights
            * sorted_labels[None, :]
        )

        negative_weights = (
            image_weights
            - positive_weights
        )

        positive_by_score = np.add.reduceat(
            positive_weights,
            tie_starts,
            axis=1,
        )

        negative_by_score = np.add.reduceat(
            negative_weights,
            tie_starts,
            axis=1,
        )

        cumulative_negative_before = (
            np.cumsum(
                negative_by_score,
                axis=1,
            )
            - negative_by_score
        )

        numerator = np.sum(
            positive_by_score
            * (
                cumulative_negative_before
                + 0.5 * negative_by_score
            ),
            axis=1,
        )

        denominator = (
            positive_by_score.sum(axis=1)
            * negative_by_score.sum(axis=1)
        )

        output[start:stop] = (
            numerator / denominator
        )

    return output


def auc_from_plan_gpu(
    plan,
    bootstrap_counts,
):
    output = np.empty(
        len(bootstrap_counts),
        dtype=np.float64,
    )

    sorted_labels_gpu = cp.asarray(
        plan["labels"],
        dtype=cp.float32,
    )

    sorted_codes_gpu = cp.asarray(
        plan["patient_codes"],
        dtype=cp.int32,
    )

    tie_starts_gpu = cp.asarray(
        plan["tie_starts"],
        dtype=cp.int32,
    )

    for start in range(
        0,
        len(bootstrap_counts),
        BATCH_SIZE,
    ):
        stop = min(
            start + BATCH_SIZE,
            len(bootstrap_counts),
        )

        counts_gpu = cp.asarray(
            bootstrap_counts[start:stop],
            dtype=cp.float32,
        )

        image_weights = counts_gpu[
            :,
            sorted_codes_gpu,
        ]

        positive_weights = (
            image_weights
            * sorted_labels_gpu[None, :]
        )

        negative_weights = (
            image_weights
            - positive_weights
        )

        positive_by_score = cp.add.reduceat(
            positive_weights,
            tie_starts_gpu,
            axis=1,
        )

        negative_by_score = cp.add.reduceat(
            negative_weights,
            tie_starts_gpu,
            axis=1,
        )

        cumulative_negative_before = (
            cp.cumsum(
                negative_by_score,
                axis=1,
            )
            - negative_by_score
        )

        numerator = cp.sum(
            positive_by_score
            * (
                cumulative_negative_before
                + 0.5 * negative_by_score
            ),
            axis=1,
        )

        denominator = (
            cp.sum(
                positive_by_score,
                axis=1,
            )
            * cp.sum(
                negative_by_score,
                axis=1,
            )
        )

        output[start:stop] = cp.asnumpy(
            numerator / denominator
        )

        del (
            counts_gpu,
            image_weights,
            positive_weights,
            negative_weights,
            positive_by_score,
            negative_by_score,
            cumulative_negative_before,
            numerator,
            denominator,
        )

    cp.get_default_memory_pool().free_all_blocks()

    return output


def auc_from_plan(
    plan,
    bootstrap_counts,
):
    if USE_GPU:
        return auc_from_plan_gpu(
            plan,
            bootstrap_counts,
        )

    return auc_from_plan_numpy(
        plan,
        bootstrap_counts,
    )


# ---------------------------------------------------------------------------
# Validate the custom AUROC routine on a tied-score toy example
# ---------------------------------------------------------------------------
toy_labels = np.array(
    [0, 1, 0, 1, 1, 0],
    dtype=np.int8,
)

toy_probabilities = np.array(
    [0.1, 0.2, 0.2, 0.8, 0.8, 0.7],
    dtype=np.float32,
)

toy_patient_codes = np.arange(
    6,
    dtype=np.int32,
)

original_labels = labels
original_patient_codes = patient_code

labels = toy_labels
patient_code = toy_patient_codes

toy_plan = prepare_auc_plan(
    toy_probabilities,
    np.ones(6, dtype=bool),
)

toy_auc = auc_from_plan(
    toy_plan,
    np.ones(
        (1, 6),
        dtype=np.uint16,
    ),
)[0]

labels = original_labels
patient_code = original_patient_codes

assert abs(
    toy_auc
    - roc_auc_score(
        toy_labels,
        toy_probabilities,
    )
) < 2e-6

print("Weighted-AUROC self-test: PASS")


# ---------------------------------------------------------------------------
# Reconstruct bootstrap metrics for all models
# ---------------------------------------------------------------------------
GROUP_MASKS = {
    "all": np.ones(
        n_images,
        dtype=bool,
    ),
    "male": sex == 1,
    "female": sex == 0,
}

full_sample_rows = []
bootstrap_metrics = {}

# One occurrence of every patient reproduces the full cohort.
all_patients_once = np.ones(
    (1, n_patients),
    dtype=np.uint16,
)

# Patient-level components for the EO-gap audit.
male_positive = (
    (sex == 1)
    & (labels == 1)
)

female_positive = (
    (sex == 0)
    & (labels == 1)
)

male_positive_counts = np.bincount(
    patient_code[male_positive],
    minlength=n_patients,
).astype(np.float32)

female_positive_counts = np.bincount(
    patient_code[female_positive],
    minlength=n_patients,
).astype(np.float32)

positive_patient_indices = np.flatnonzero(
    (
        male_positive_counts
        + female_positive_counts
    ) > 0
)

positive_cluster_counts = patient_counts[
    :,
    positive_patient_indices,
].astype(np.float32)


for model_number, (
    model_id,
    probabilities,
) in enumerate(
    analysis_probabilities.items(),
    start=1,
):
    print(
        f"[{model_number:02d}/"
        f"{len(analysis_probabilities)}] "
        f"{model_id}"
    )

    model_bootstrap = {}
    full_sample = {}

    for group_name, group_mask in GROUP_MASKS.items():
        plan = prepare_auc_plan(
            probabilities,
            group_mask,
        )

        reconstructed_auc = auc_from_plan(
            plan,
            all_patients_once,
        )[0]

        sklearn_auc = roc_auc_score(
            labels[group_mask],
            probabilities[group_mask],
        )

        if abs(
            reconstructed_auc
            - sklearn_auc
        ) > 2e-6:
            raise RuntimeError(
                "AUROC validation failed for "
                f"{model_id}, {group_name}: "
                f"{reconstructed_auc} versus "
                f"{sklearn_auc}"
            )

        model_bootstrap[
            f"{group_name}_auroc"
        ] = auc_from_plan(
            plan,
            patient_counts,
        )

        full_sample[
            f"{group_name}_auroc"
        ] = float(sklearn_auc)

    model_bootstrap["auroc"] = (
        model_bootstrap.pop(
            "all_auroc"
        )
    )

    full_sample["auroc"] = (
        full_sample.pop(
            "all_auroc"
        )
    )

    model_bootstrap[
        "worst_group_auroc"
    ] = np.minimum(
        model_bootstrap["male_auroc"],
        model_bootstrap["female_auroc"],
    )

    full_sample[
        "worst_group_auroc"
    ] = min(
        full_sample["male_auroc"],
        full_sample["female_auroc"],
    )

    # Fixed-threshold equal-opportunity gap.
    predicted_positive = (
        probabilities >= FIXED_THRESHOLD
    )

    male_true_positive_counts = np.bincount(
        patient_code[male_positive],
        weights=predicted_positive[
            male_positive
        ].astype(np.float32),
        minlength=n_patients,
    ).astype(np.float32)

    female_true_positive_counts = np.bincount(
        patient_code[female_positive],
        weights=predicted_positive[
            female_positive
        ].astype(np.float32),
        minlength=n_patients,
    ).astype(np.float32)

    threshold_statistics = np.column_stack(
        [
            male_true_positive_counts[
                positive_patient_indices
            ],
            male_positive_counts[
                positive_patient_indices
            ],
            female_true_positive_counts[
                positive_patient_indices
            ],
            female_positive_counts[
                positive_patient_indices
            ],
        ]
    ).astype(np.float32)

    if USE_GPU:
        bootstrap_totals = cp.asnumpy(
            cp.asarray(
                positive_cluster_counts
            )
            @ cp.asarray(
                threshold_statistics
            )
        )

        cp.get_default_memory_pool().free_all_blocks()

    else:
        bootstrap_totals = (
            positive_cluster_counts
            @ threshold_statistics
        )

    male_tpr = (
        bootstrap_totals[:, 0] + 1.0
    ) / (
        bootstrap_totals[:, 1] + 2.0
    )

    female_tpr = (
        bootstrap_totals[:, 2] + 1.0
    ) / (
        bootstrap_totals[:, 3] + 2.0
    )

    model_bootstrap[
        "eo_gap_smoothed_50"
    ] = np.abs(
        male_tpr - female_tpr
    ).astype(np.float64)

    full_male_tpr = (
        predicted_positive[
            male_positive
        ].sum() + 1.0
    ) / (
        male_positive.sum() + 2.0
    )

    full_female_tpr = (
        predicted_positive[
            female_positive
        ].sum() + 1.0
    ) / (
        female_positive.sum() + 2.0
    )

    full_sample[
        "eo_gap_smoothed_50"
    ] = float(
        abs(
            full_male_tpr
            - full_female_tpr
        )
    )

    bootstrap_metrics[model_id] = (
        model_bootstrap
    )

    full_sample_rows.append(
        {
            "model_id": model_id,
            **full_sample,
        }
    )


full_sample_metrics = pd.DataFrame(
    full_sample_rows
)

full_sample_metrics.to_csv(
    OUTPUTS
    / "run3_full_sample_metrics.csv",
    index=False,
)

np.savez_compressed(
    OUTPUTS
    / "run3_patient_cluster_model_metrics.npz",
    bootstrap_metrics=np.array(
        bootstrap_metrics,
        dtype=object,
    ),
    bootstrap_seed=np.array(
        BOOTSTRAP_SEED
    ),
    n_bootstrap=np.array(
        N_BOOTSTRAP
    ),
)


# ---------------------------------------------------------------------------
# Analysis helper functions
# ---------------------------------------------------------------------------
full_metric_lookup = (
    full_sample_metrics
    .set_index("model_id")
)


def metric_matrix(
    method,
    metric,
    seed_subset=None,
):
    model_ids = method_runs[method]

    if seed_subset is not None:
        allowed_seeds = set(seed_subset)

        model_ids = [
            model_id
            for model_id in model_ids
            if int(
                model_id.split("|")[1]
            ) in allowed_seeds
        ]

    matrix = np.column_stack(
        [
            bootstrap_metrics[
                model_id
            ][metric]
            for model_id in model_ids
        ]
    )

    return matrix, model_ids


def full_values(
    method,
    metric,
    seed_subset=None,
):
    model_ids = method_runs[method]

    if seed_subset is not None:
        allowed_seeds = set(seed_subset)

        model_ids = [
            model_id
            for model_id in model_ids
            if int(
                model_id.split("|")[1]
            ) in allowed_seeds
        ]

    values = np.array(
        [
            full_metric_lookup.loc[
                model_id,
                metric,
            ]
            for model_id in model_ids
        ],
        dtype=float,
    )

    return values, model_ids


def percentile_interval(values):
    return tuple(
        np.percentile(
            values,
            [2.5, 97.5],
        )
    )


def exact_independent_permutation_p(
    method_a_values,
    method_b_values,
):
    method_a_values = np.asarray(
        method_a_values
    )

    method_b_values = np.asarray(
        method_b_values
    )

    combined = np.r_[
        method_a_values,
        method_b_values,
    ]

    n_a = len(method_a_values)
    all_indices = np.arange(
        len(combined)
    )

    observed = (
        method_a_values.mean()
        - method_b_values.mean()
    )

    permuted_statistics = []

    for selected_indices in combinations(
        all_indices,
        n_a,
    ):
        selected_indices = np.asarray(
            selected_indices
        )

        remaining_indices = np.setdiff1d(
            all_indices,
            selected_indices,
            assume_unique=True,
        )

        permuted_statistics.append(
            combined[
                selected_indices
            ].mean()
            - combined[
                remaining_indices
            ].mean()
        )

    permuted_statistics = np.asarray(
        permuted_statistics
    )

    return float(
        np.mean(
            np.abs(
                permuted_statistics
            )
            >= abs(observed) - 1e-15
        )
    )


def exact_paired_signflip_p(
    paired_differences,
):
    paired_differences = np.asarray(
        paired_differences,
        dtype=float,
    )

    observed = paired_differences.mean()

    signflipped_statistics = []

    for signs in product(
        [-1.0, 1.0],
        repeat=len(
            paired_differences
        ),
    ):
        signflipped_statistics.append(
            np.mean(
                np.asarray(signs)
                * paired_differences
            )
        )

    signflipped_statistics = np.asarray(
        signflipped_statistics
    )

    return float(
        np.mean(
            np.abs(
                signflipped_statistics
            )
            >= abs(observed) - 1e-15
        )
    )


def holm_adjust(
    p_values,
):
    p_values = np.asarray(
        p_values,
        dtype=float,
    )

    order = np.argsort(
        p_values
    )

    adjusted = np.empty_like(
        p_values
    )

    running_maximum = 0.0
    m = len(p_values)

    for rank, original_index in enumerate(
        order
    ):
        adjusted_value = min(
            1.0,
            (m - rank)
            * p_values[
                original_index
            ],
        )

        running_maximum = max(
            running_maximum,
            adjusted_value,
        )

        adjusted[
            original_index
        ] = running_maximum

    return adjusted


# ---------------------------------------------------------------------------
# Primary analyses among the three five-run methods
# ---------------------------------------------------------------------------
rng_independent = np.random.default_rng(
    BOOTSTRAP_SEED + 11
)

rng_paired = np.random.default_rng(
    BOOTSTRAP_SEED + 22
)

method_boot_independent = {
    method: {}
    for method in CORE_METHODS
}

method_boot_paired = {
    method: {}
    for method in CORE_METHODS
}

method_boot_observed_runs = {
    method: {}
    for method in CORE_METHODS
}

method_observed = {
    method: {}
    for method in CORE_METHODS
}

# Shared index for the matched-seed sensitivity analysis.
paired_seed_indices = rng_paired.integers(
    0,
    5,
    size=(
        N_BOOTSTRAP,
        5,
    ),
)

for method in CORE_METHODS:
    for metric in ALL_BOOTSTRAP_METRICS:
        run_matrix, model_ids = metric_matrix(
            method,
            metric,
        )

        run_values, _ = full_values(
            method,
            metric,
        )

        assert run_matrix.shape[1] == 5

        method_observed[
            method
        ][metric] = (
            run_values.mean()
        )

        # Patient uncertainty with the observed runs fixed.
        method_boot_observed_runs[
            method
        ][metric] = run_matrix.mean(
            axis=1
        )

        # Primary: run indices sampled independently for each method.
        independent_run_indices = (
            rng_independent.integers(
                0,
                5,
                size=(
                    N_BOOTSTRAP,
                    5,
                ),
            )
        )

        method_boot_independent[
            method
        ][metric] = np.take_along_axis(
            run_matrix,
            independent_run_indices,
            axis=1,
        ).mean(axis=1)

        # Sensitivity: same numeric seed indices sampled jointly.
        method_boot_paired[
            method
        ][metric] = np.take_along_axis(
            run_matrix,
            paired_seed_indices,
            axis=1,
        ).mean(axis=1)


def build_primary_family(
    analysis_label,
    method_bootstrap,
):
    rows = []
    bootstrap_columns = []
    observed_statistics = []
    identities = []

    for method_a, method_b in PRIMARY_COMPARISONS:
        for metric in PRIMARY_METRICS:
            bootstrap_difference = (
                method_bootstrap[
                    method_a
                ][metric]
                - method_bootstrap[
                    method_b
                ][metric]
            )

            observed_difference = (
                method_observed[
                    method_a
                ][metric]
                - method_observed[
                    method_b
                ][metric]
            )

            bootstrap_columns.append(
                bootstrap_difference
            )

            observed_statistics.append(
                observed_difference
            )

            identities.append(
                (
                    method_a,
                    method_b,
                    metric,
                )
            )

    bootstrap_matrix = np.column_stack(
        bootstrap_columns
    )

    observed_statistics = np.asarray(
        observed_statistics
    )

    bootstrap_sd = bootstrap_matrix.std(
        axis=0,
        ddof=1,
    )

    centered_standardized = (
        bootstrap_matrix
        - bootstrap_matrix.mean(
            axis=0
        )
    ) / bootstrap_sd

    max_absolute_statistic = np.max(
        np.abs(
            centered_standardized
        ),
        axis=1,
    )

    max_t_critical = np.quantile(
        max_absolute_statistic,
        0.95,
    )

    for test_index, (
        method_a,
        method_b,
        metric,
    ) in enumerate(identities):
        pointwise_low, pointwise_high = (
            percentile_interval(
                bootstrap_matrix[
                    :,
                    test_index,
                ]
            )
        )

        rows.append(
            {
                "analysis": analysis_label,
                "method_a": method_a,
                "method_b": method_b,
                "comparison": (
                    f"{method_a} minus "
                    f"{method_b}"
                ),
                "metric": metric,
                "estimate": observed_statistics[
                    test_index
                ],
                "pointwise_ci_low": pointwise_low,
                "pointwise_ci_high": pointwise_high,
                "simultaneous_ci_low": (
                    observed_statistics[
                        test_index
                    ]
                    - max_t_critical
                    * bootstrap_sd[
                        test_index
                    ]
                ),
                "simultaneous_ci_high": (
                    observed_statistics[
                        test_index
                    ]
                    + max_t_critical
                    * bootstrap_sd[
                        test_index
                    ]
                ),
                "bootstrap_sd": bootstrap_sd[
                    test_index
                ],
                "max_t_critical": max_t_critical,
                "n_runs_a": 5,
                "n_runs_b": 5,
                "n_bootstrap": N_BOOTSTRAP,
            }
        )

    return pd.DataFrame(rows)


primary_independent = build_primary_family(
    "independent_run_resampling_primary",
    method_boot_independent,
)

primary_paired = build_primary_family(
    "matched_seed_resampling_sensitivity",
    method_boot_paired,
)

primary_observed_runs = build_primary_family(
    "observed_runs_fixed_patient_cluster_only",
    method_boot_observed_runs,
)


# ---------------------------------------------------------------------------
# Exact run-level tests with Holm adjustment
# ---------------------------------------------------------------------------
run_level_rows = []

for method_a, method_b in PRIMARY_COMPARISONS:
    for metric in PRIMARY_METRICS:
        values_a, _ = full_values(
            method_a,
            metric,
        )

        values_b, _ = full_values(
            method_b,
            metric,
        )

        run_level_rows.append(
            {
                "comparison": (
                    f"{method_a} minus "
                    f"{method_b}"
                ),
                "metric": metric,
                "run_level_estimate": (
                    values_a.mean()
                    - values_b.mean()
                ),
                "independent_exact_permutation_p": (
                    exact_independent_permutation_p(
                        values_a,
                        values_b,
                    )
                ),
                "paired_exact_signflip_p": (
                    exact_paired_signflip_p(
                        values_a - values_b
                    )
                ),
            }
        )

run_level_tests = pd.DataFrame(
    run_level_rows
)

run_level_tests[
    "independent_holm_p"
] = holm_adjust(
    run_level_tests[
        "independent_exact_permutation_p"
    ]
)

run_level_tests[
    "paired_holm_p"
] = holm_adjust(
    run_level_tests[
        "paired_exact_signflip_p"
    ]
)

run_level_tests.to_csv(
    OUTPUTS
    / "run3_exact_run_level_tests.csv",
    index=False,
)


primary_results = pd.concat(
    [
        primary_independent,
        primary_paired,
        primary_observed_runs,
    ],
    ignore_index=True,
)

primary_results = primary_results.merge(
    run_level_tests,
    on=[
        "comparison",
        "metric",
    ],
    how="left",
)

primary_results.to_csv(
    OUTPUTS
    / "run3_primary_inference.csv",
    index=False,
)


# ---------------------------------------------------------------------------
# Conditional fixed-probability-ensemble analysis
# ---------------------------------------------------------------------------
ensemble_rows = []

for method_a, method_b in PRIMARY_COMPARISONS:
    ensemble_a = f"ENSEMBLE|{method_a}"
    ensemble_b = f"ENSEMBLE|{method_b}"

    for metric in (
        PRIMARY_METRICS
        + ["eo_gap_smoothed_50"]
    ):
        bootstrap_difference = (
            bootstrap_metrics[
                ensemble_a
            ][metric]
            - bootstrap_metrics[
                ensemble_b
            ][metric]
        )

        estimate = (
            full_metric_lookup.loc[
                ensemble_a,
                metric,
            ]
            - full_metric_lookup.loc[
                ensemble_b,
                metric,
            ]
        )

        ci_low, ci_high = percentile_interval(
            bootstrap_difference
        )

        ensemble_rows.append(
            {
                "analysis": (
                    "fixed_probability_ensemble_"
                    "patient_cluster_only"
                ),
                "comparison": (
                    f"{method_a} minus "
                    f"{method_b}"
                ),
                "metric": metric,
                "estimate": estimate,
                "ci_low": ci_low,
                "ci_high": ci_high,
                "n_bootstrap": N_BOOTSTRAP,
            }
        )

ensemble_results = pd.DataFrame(
    ensemble_rows
)

ensemble_results.to_csv(
    OUTPUTS
    / "run3_fixed_ensemble_patient_cluster.csv",
    index=False,
)


# ---------------------------------------------------------------------------
# Three-seed DWFA versus scrambled-routing placebo
# ---------------------------------------------------------------------------
placebo_methods = [
    "DWFA",
    "DWFA-Scrambled",
]

placebo_seed_subset = [
    42,
    123,
    456,
]

rng_placebo_independent = (
    np.random.default_rng(
        BOOTSTRAP_SEED + 33
    )
)

rng_placebo_paired = (
    np.random.default_rng(
        BOOTSTRAP_SEED + 44
    )
)

paired_three_indices = (
    rng_placebo_paired.integers(
        0,
        3,
        size=(
            N_BOOTSTRAP,
            3,
        ),
    )
)

placebo_rows = []

for metric in [
    "auroc",
    "worst_group_auroc",
    "eo_gap_smoothed_50",
]:
    matrices = {}
    values = {}

    for method in placebo_methods:
        matrices[method], _ = metric_matrix(
            method,
            metric,
            seed_subset=placebo_seed_subset,
        )

        values[method], _ = full_values(
            method,
            metric,
            seed_subset=placebo_seed_subset,
        )

    estimate = (
        values["DWFA"].mean()
        - values[
            "DWFA-Scrambled"
        ].mean()
    )

    independent_indices_a = (
        rng_placebo_independent.integers(
            0,
            3,
            size=(
                N_BOOTSTRAP,
                3,
            ),
        )
    )

    independent_indices_b = (
        rng_placebo_independent.integers(
            0,
            3,
            size=(
                N_BOOTSTRAP,
                3,
            ),
        )
    )

    independent_a = np.take_along_axis(
        matrices["DWFA"],
        independent_indices_a,
        axis=1,
    ).mean(axis=1)

    independent_b = np.take_along_axis(
        matrices["DWFA-Scrambled"],
        independent_indices_b,
        axis=1,
    ).mean(axis=1)

    paired_a = np.take_along_axis(
        matrices["DWFA"],
        paired_three_indices,
        axis=1,
    ).mean(axis=1)

    paired_b = np.take_along_axis(
        matrices["DWFA-Scrambled"],
        paired_three_indices,
        axis=1,
    ).mean(axis=1)

    observed_runs_fixed = (
        matrices["DWFA"].mean(axis=1)
        - matrices[
            "DWFA-Scrambled"
        ].mean(axis=1)
    )

    analysis_distributions = [
        (
            "independent_run_resampling_primary",
            independent_a - independent_b,
        ),
        (
            "matched_seed_resampling_sensitivity",
            paired_a - paired_b,
        ),
        (
            "observed_runs_fixed_patient_cluster_only",
            observed_runs_fixed,
        ),
    ]

    for analysis_label, distribution in (
        analysis_distributions
    ):
        ci_low, ci_high = (
            percentile_interval(
                distribution
            )
        )

        placebo_rows.append(
            {
                "analysis": analysis_label,
                "comparison": (
                    "DWFA minus "
                    "DWFA-Scrambled"
                ),
                "metric": metric,
                "estimate": estimate,
                "ci_low": ci_low,
                "ci_high": ci_high,
                "n_runs_each": 3,
                "n_bootstrap": N_BOOTSTRAP,
            }
        )

placebo_results = pd.DataFrame(
    placebo_rows
)

placebo_results.to_csv(
    OUTPUTS
    / "run3_placebo_inference.csv",
    index=False,
)


# ---------------------------------------------------------------------------
# Exploratory matched three-seed comparisons against FedAvg
# ---------------------------------------------------------------------------
fedavg_rows = []

rng_fedavg = np.random.default_rng(
    BOOTSTRAP_SEED + 55
)

shared_three_indices = rng_fedavg.integers(
    0,
    3,
    size=(
        N_BOOTSTRAP,
        3,
    ),
)

for method in [
    "LPR",
    "CADR",
    "DWFA",
]:
    for metric in (
        PRIMARY_METRICS
        + ["eo_gap_smoothed_50"]
    ):
        method_matrix, _ = metric_matrix(
            method,
            metric,
            seed_subset=[
                42,
                123,
                456,
            ],
        )

        fedavg_matrix, _ = metric_matrix(
            "FedAvg",
            metric,
            seed_subset=[
                42,
                123,
                456,
            ],
        )

        method_values, _ = full_values(
            method,
            metric,
            seed_subset=[
                42,
                123,
                456,
            ],
        )

        fedavg_values, _ = full_values(
            "FedAvg",
            metric,
            seed_subset=[
                42,
                123,
                456,
            ],
        )

        distribution = (
            np.take_along_axis(
                method_matrix,
                shared_three_indices,
                axis=1,
            ).mean(axis=1)
            - np.take_along_axis(
                fedavg_matrix,
                shared_three_indices,
                axis=1,
            ).mean(axis=1)
        )

        ci_low, ci_high = percentile_interval(
            distribution
        )

        fedavg_rows.append(
            {
                "analysis": (
                    "matched_three_seed_"
                    "exploratory"
                ),
                "comparison": (
                    f"{method} minus FedAvg"
                ),
                "metric": metric,
                "estimate": (
                    method_values.mean()
                    - fedavg_values.mean()
                ),
                "ci_low": ci_low,
                "ci_high": ci_high,
                "n_common_seeds": 3,
                "n_bootstrap": N_BOOTSTRAP,
            }
        )

fedavg_results = pd.DataFrame(
    fedavg_rows
)

fedavg_results.to_csv(
    OUTPUTS
    / "run3_fedavg_three_seed_exploratory.csv",
    index=False,
)


# ---------------------------------------------------------------------------
# Provisional forest plot
# ---------------------------------------------------------------------------
plot_data = primary_independent.copy()

plot_data["display_label"] = (
    plot_data["comparison"]
    + " | "
    + plot_data["metric"].replace(
        {
            "auroc": "AUROC",
            "worst_group_auroc": (
                "Worst-group AUROC"
            ),
        }
    )
)

plot_data = (
    plot_data.iloc[::-1]
    .reset_index(drop=True)
)

figure, axis = plt.subplots(
    figsize=(9.2, 6.2)
)

y_positions = np.arange(
    len(plot_data)
)

for row_index, row in plot_data.iterrows():
    axis.errorbar(
        row["estimate"],
        y_positions[row_index],
        xerr=[
            [
                row["estimate"]
                - row["pointwise_ci_low"]
            ],
            [
                row["pointwise_ci_high"]
                - row["estimate"]
            ],
        ],
        fmt="o",
        capsize=4,
    )

    # Slightly offset horizontal segment:
    # simultaneous familywise interval.
    axis.hlines(
        y_positions[row_index] + 0.12,
        row["simultaneous_ci_low"],
        row["simultaneous_ci_high"],
        linewidth=2,
    )

axis.axvline(
    0,
    linestyle="--",
    linewidth=1,
)

axis.set_yticks(
    y_positions
)

axis.set_yticklabels(
    plot_data["display_label"]
)

axis.set_xlabel(
    "Difference in AUROC "
    "(method A minus method B)"
)

axis.set_title(
    "Patient-clustered, "
    "run-aware comparisons"
)

axis.grid(
    axis="x",
    alpha=0.25,
)

figure.tight_layout()

figure.savefig(
    OUTPUTS
    / "run3_primary_forest_provisional.pdf",
    bbox_inches="tight",
)

figure.savefig(
    OUTPUTS
    / "run3_primary_forest_provisional.png",
    dpi=300,
    bbox_inches="tight",
)

plt.close(figure)


# ---------------------------------------------------------------------------
# Print results
# ---------------------------------------------------------------------------
pd.set_option(
    "display.max_columns",
    30,
)

pd.set_option(
    "display.width",
    220,
)

print(
    "\nPRIMARY INDEPENDENT-RUN, "
    "PATIENT-CLUSTER RESULTS"
)

print(
    primary_independent[
        [
            "comparison",
            "metric",
            "estimate",
            "pointwise_ci_low",
            "pointwise_ci_high",
            "simultaneous_ci_low",
            "simultaneous_ci_high",
        ]
    ].to_string(
        index=False,
        float_format=lambda value: (
            f"{value:+.6f}"
        ),
    )
)

print(
    "\nMATCHED-SEED SENSITIVITY"
)

print(
    primary_paired[
        [
            "comparison",
            "metric",
            "estimate",
            "pointwise_ci_low",
            "pointwise_ci_high",
            "simultaneous_ci_low",
            "simultaneous_ci_high",
        ]
    ].to_string(
        index=False,
        float_format=lambda value: (
            f"{value:+.6f}"
        ),
    )
)

print(
    "\nFIXED-ENSEMBLE, "
    "PATIENT-CLUSTER RESULTS"
)

print(
    ensemble_results.to_string(
        index=False,
        float_format=lambda value: (
            f"{value:+.6f}"
        ),
    )
)

print(
    "\nEXACT RUN-LEVEL TESTS "
    "WITH HOLM ADJUSTMENT"
)

print(
    run_level_tests.to_string(
        index=False,
        float_format=lambda value: (
            f"{value:.6f}"
        ),
    )
)

print(
    "\nPLACEBO RESULTS"
)

print(
    placebo_results.to_string(
        index=False,
        float_format=lambda value: (
            f"{value:+.6f}"
        ),
    )
)

print(
    "\nFEDAVG THREE-SEED "
    "EXPLORATORY RESULTS"
)

print(
    fedavg_results.to_string(
        index=False,
        float_format=lambda value: (
            f"{value:+.6f}"
        ),
    )
)


# ---------------------------------------------------------------------------
# Metadata and ZIP
# ---------------------------------------------------------------------------
metadata = {
    "n_images": int(n_images),
    "n_patients": int(n_patients),
    "n_bootstrap": int(N_BOOTSTRAP),
    "patient_cluster_unit": "Patient ID",
    "primary_run_resampling": (
        "independent across methods"
    ),
    "paired_sensitivity": (
        "same numeric seed resampled jointly"
    ),
    "primary_family": (
        "three pairwise comparisons "
        "times two AUROC metrics"
    ),
    "multiplicity": (
        "single-step max-t simultaneous "
        "95% intervals; exact run-level "
        "p-values Holm-adjusted"
    ),
    "fixed_threshold": FIXED_THRESHOLD,
    "gpu_used": bool(USE_GPU),
}

with open(
    OUTPUTS / "run3_metadata.json",
    "w",
    encoding="utf-8",
) as file:
    json.dump(
        metadata,
        file,
        indent=2,
    )

local_zip = Path(
    "/content/revision_run3.zip"
)

with zipfile.ZipFile(
    local_zip,
    "w",
    compression=zipfile.ZIP_DEFLATED,
) as archive:
    for output_path in sorted(
        OUTPUTS.rglob("*")
    ):
        if output_path.is_file():
            archive.write(
                output_path,
                arcname=output_path.relative_to(
                    OUTPUTS
                ),
            )

drive_zip = (
    RESULTS
    / "revision_run3.zip"
)

shutil.copy2(
    local_zip,
    drive_zip,
)

print("\n" + "=" * 96)
print("RUN 3 COMPLETE")
print("Local ZIP:", local_zip)
print("Drive ZIP:", drive_zip)
print("=" * 96)

Mounted at /content/drive
RUN 3: PATIENT-CLUSTERED, RUN-AWARE NIH INFERENCE
Images:                 112,120
Unique patients:        30,805
Main prediction arrays: 18
Scrambled arrays:       3
Individual models: 21
Secondary ensembles: 5
Weighted-AUROC backend: CuPy GPU
Patient-cluster bootstrap matrix: 117.5 MB
Weighted-AUROC self-test: PASS
[01/26] DWFA|42
[02/26] DWFA|123
[03/26] DWFA|456
[04/26] DWFA|789
[05/26] DWFA|1010
[06/26] CADR|42
[07/26] CADR|123
[08/26] CADR|456
[09/26] CADR|789
[10/26] CADR|1010
[11/26] FedAvg|42
[12/26] FedAvg|123
[13/26] FedAvg|456
[14/26] LPR|42
[15/26] LPR|123
[16/26] LPR|456
[17/26] LPR|789
[18/26] LPR|1010
[19/26] DWFA-Scrambled|42
[20/26] DWFA-Scrambled|123
[21/26] DWFA-Scrambled|456
[22/26] ENSEMBLE|FedAvg
[23/26] ENSEMBLE|LPR
[24/26] ENSEMBLE|CADR
[25/26] ENSEMBLE|DWFA
[26/26] ENSEMBLE|DWFA-Scrambled

PRIMARY INDEPENDENT-RUN, PATIENT-CLUSTER RESULTS
     comparison            metric  estimate  pointwise_ci_low  pointwise_ci_high  simultaneous_ci_l

In [ ]:
# ============================================================================
# RUN 3B: CORRECTED JOINT RUN RESAMPLING AND SIMULTANEOUS INTERVALS
#
# CPU only.
# No training.
# No inference.
# No repetition of patient-cluster AUROC calculations.
#
# Required existing file:
#   /content/drive/MyDrive/FairFedCXR/results/revision_run3.zip
#
# Output:
#   /content/drive/MyDrive/FairFedCXR/results/revision_run3b.zip
# ============================================================================

from google.colab import drive
drive.mount("/content/drive", force_remount=True)

from pathlib import Path
import json
import shutil
import zipfile

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd


# ---------------------------------------------------------------------------
# Paths
# ---------------------------------------------------------------------------
BASE = Path("/content/drive/MyDrive/FairFedCXR")
RESULTS = BASE / "results"

RUN3_ZIP = RESULTS / "revision_run3.zip"

LOCAL = Path("/content/revision_run3b")
RUN3_EXTRACTED = LOCAL / "run3"
OUTPUTS = LOCAL / "outputs"

RUN3_EXTRACTED.mkdir(parents=True, exist_ok=True)
OUTPUTS.mkdir(parents=True, exist_ok=True)

if not RUN3_ZIP.exists():
    raise FileNotFoundError(
        f"Run 3 ZIP was not found: {RUN3_ZIP}"
    )

print("=" * 96)
print("RUN 3B: CORRECTED JOINT RUN RESAMPLING")
print("=" * 96)

# ---------------------------------------------------------------------------
# Extract Run 3 outputs
# ---------------------------------------------------------------------------
with zipfile.ZipFile(RUN3_ZIP, "r") as archive:
    archive.extractall(RUN3_EXTRACTED)

required_files = [
    RUN3_EXTRACTED / "run3_patient_cluster_model_metrics.npz",
    RUN3_EXTRACTED / "run3_full_sample_metrics.csv",
    RUN3_EXTRACTED / "run3_fixed_ensemble_patient_cluster.csv",
    RUN3_EXTRACTED / "run3_primary_inference.csv",
    RUN3_EXTRACTED / "run3_exact_run_level_tests.csv",
    RUN3_EXTRACTED / "run3_metadata.json",
]

for path in required_files:
    if not path.exists():
        raise FileNotFoundError(
            f"Required Run 3 output is missing: {path}"
        )


# ---------------------------------------------------------------------------
# Load saved patient-cluster bootstrap results
# ---------------------------------------------------------------------------
bootstrap_archive = np.load(
    RUN3_EXTRACTED / "run3_patient_cluster_model_metrics.npz",
    allow_pickle=True,
)

bootstrap_metrics = bootstrap_archive[
    "bootstrap_metrics"
].item()

n_bootstrap = int(
    np.asarray(
        bootstrap_archive["n_bootstrap"]
    ).item()
)

full_sample_metrics = pd.read_csv(
    RUN3_EXTRACTED / "run3_full_sample_metrics.csv"
)

fixed_ensemble_results = pd.read_csv(
    RUN3_EXTRACTED / "run3_fixed_ensemble_patient_cluster.csv"
)

old_primary_results = pd.read_csv(
    RUN3_EXTRACTED / "run3_primary_inference.csv"
)

run_level_tests = pd.read_csv(
    RUN3_EXTRACTED / "run3_exact_run_level_tests.csv"
)

with open(
    RUN3_EXTRACTED / "run3_metadata.json",
    "r",
    encoding="utf-8",
) as file:
    run3_metadata = json.load(file)

print(f"Bootstrap replicates loaded: {n_bootstrap:,}")
print(f"Model/ensemble entries:       {len(bootstrap_metrics):,}")


# ---------------------------------------------------------------------------
# Primary family
# ---------------------------------------------------------------------------
METHODS = [
    "LPR",
    "CADR",
    "DWFA",
]

COMPARISONS = [
    ("DWFA", "LPR"),
    ("DWFA", "CADR"),
    ("LPR", "CADR"),
]

METRICS = [
    "auroc",
    "worst_group_auroc",
]

METRIC_LABELS = {
    "auroc": "Overall AUROC",
    "worst_group_auroc": "Worst-group AUROC",
}

RANDOM_SEED = 20260724


# ---------------------------------------------------------------------------
# Find individual model IDs
# ---------------------------------------------------------------------------
method_model_ids = {}

for method in METHODS:
    prefix = f"{method}|"

    ids = [
        model_id
        for model_id in bootstrap_metrics
        if model_id.startswith(prefix)
        and not model_id.startswith("ENSEMBLE|")
    ]

    ids = sorted(
        ids,
        key=lambda model_id: int(
            model_id.split("|")[1]
        ),
    )

    if len(ids) != 5:
        raise RuntimeError(
            f"{method}: expected 5 individual models, found {len(ids)}"
        )

    method_model_ids[method] = ids

print("\nIndividual model IDs:")
for method, model_ids in method_model_ids.items():
    print(method, model_ids)


# ---------------------------------------------------------------------------
# Full-sample metric lookup
# ---------------------------------------------------------------------------
full_lookup = full_sample_metrics.set_index(
    "model_id"
)


def get_bootstrap_matrix(method, metric):
    """
    Returns a matrix with shape:
        patient-bootstrap replicate × training run
    """
    return np.column_stack(
        [
            bootstrap_metrics[model_id][metric]
            for model_id in method_model_ids[method]
        ]
    ).astype(np.float64)


def get_full_values(method, metric):
    return np.asarray(
        [
            full_lookup.loc[model_id, metric]
            for model_id in method_model_ids[method]
        ],
        dtype=np.float64,
    )


# ---------------------------------------------------------------------------
# IMPORTANT CORRECTION:
#
# One independent run-index matrix is generated per method and reused for
# BOTH metrics. This preserves cross-metric dependence within each method.
#
# One shared run-index matrix is used for the matched-seed sensitivity and
# is reused across every method and every metric.
# ---------------------------------------------------------------------------
rng_independent = np.random.default_rng(
    RANDOM_SEED
)

rng_matched = np.random.default_rng(
    RANDOM_SEED + 1
)

independent_run_indices = {
    method: rng_independent.integers(
        0,
        5,
        size=(n_bootstrap, 5),
    )
    for method in METHODS
}

matched_run_indices = rng_matched.integers(
    0,
    5,
    size=(n_bootstrap, 5),
)


# ---------------------------------------------------------------------------
# Construct method-level bootstrap distributions
# ---------------------------------------------------------------------------
observed_estimates = {
    method: {}
    for method in METHODS
}

independent_distributions = {
    method: {}
    for method in METHODS
}

matched_distributions = {
    method: {}
    for method in METHODS
}

fixed_run_distributions = {
    method: {}
    for method in METHODS
}

for method in METHODS:
    for metric in METRICS:
        matrix = get_bootstrap_matrix(
            method,
            metric,
        )

        full_values = get_full_values(
            method,
            metric,
        )

        if matrix.shape != (n_bootstrap, 5):
            raise RuntimeError(
                f"Unexpected shape for {method}, {metric}: "
                f"{matrix.shape}"
            )

        observed_estimates[
            method
        ][metric] = float(
            full_values.mean()
        )

        # Patient uncertainty only, with the five observed runs fixed.
        fixed_run_distributions[
            method
        ][metric] = matrix.mean(
            axis=1
        )

        # Primary analysis: run resampling independent across methods,
        # but common across metrics within each method.
        independent_distributions[
            method
        ][metric] = np.take_along_axis(
            matrix,
            independent_run_indices[method],
            axis=1,
        ).mean(axis=1)

        # Sensitivity: same numeric seed indices sampled for every method
        # and reused across both metrics.
        matched_distributions[
            method
        ][metric] = np.take_along_axis(
            matrix,
            matched_run_indices,
            axis=1,
        ).mean(axis=1)


# ---------------------------------------------------------------------------
# Familywise interval construction
# ---------------------------------------------------------------------------
def percentile_interval(values):
    low, high = np.percentile(
        values,
        [2.5, 97.5],
    )

    return float(low), float(high)


def build_family(
    analysis_name,
    method_distributions,
):
    identities = []
    observed = []
    bootstrap_columns = []

    for method_a, method_b in COMPARISONS:
        for metric in METRICS:
            identities.append(
                (
                    method_a,
                    method_b,
                    metric,
                )
            )

            observed.append(
                observed_estimates[
                    method_a
                ][metric]
                - observed_estimates[
                    method_b
                ][metric]
            )

            bootstrap_columns.append(
                method_distributions[
                    method_a
                ][metric]
                - method_distributions[
                    method_b
                ][metric]
            )

    observed = np.asarray(
        observed,
        dtype=np.float64,
    )

    bootstrap_matrix = np.column_stack(
        bootstrap_columns
    )

    bootstrap_means = bootstrap_matrix.mean(
        axis=0
    )

    bootstrap_sd = bootstrap_matrix.std(
        axis=0,
        ddof=1,
    )

    if np.any(bootstrap_sd <= 0):
        raise RuntimeError(
            "A primary bootstrap distribution has zero variance."
        )

    standardized_centered = (
        bootstrap_matrix
        - bootstrap_means
    ) / bootstrap_sd

    maximum_absolute_statistic = np.max(
        np.abs(
            standardized_centered
        ),
        axis=1,
    )

    critical_value = float(
        np.quantile(
            maximum_absolute_statistic,
            0.95,
        )
    )

    rows = []

    for index, (
        method_a,
        method_b,
        metric,
    ) in enumerate(identities):
        pointwise_low, pointwise_high = percentile_interval(
            bootstrap_matrix[:, index]
        )

        simultaneous_low = (
            observed[index]
            - critical_value
            * bootstrap_sd[index]
        )

        simultaneous_high = (
            observed[index]
            + critical_value
            * bootstrap_sd[index]
        )

        rows.append(
            {
                "analysis": analysis_name,
                "method_a": method_a,
                "method_b": method_b,
                "comparison": (
                    f"{method_a} minus {method_b}"
                ),
                "metric": metric,
                "estimate": observed[index],
                "pointwise_ci_low": pointwise_low,
                "pointwise_ci_high": pointwise_high,
                "simultaneous_ci_low": simultaneous_low,
                "simultaneous_ci_high": simultaneous_high,
                "bootstrap_sd": bootstrap_sd[index],
                "familywise_critical_value": critical_value,
                "n_runs_a": 5,
                "n_runs_b": 5,
                "n_patient_bootstrap": n_bootstrap,
                "joint_run_resampling_across_metrics": True,
            }
        )

    return pd.DataFrame(rows)


corrected_independent = build_family(
    "independent_run_resampling_primary_corrected",
    independent_distributions,
)

corrected_matched = build_family(
    "matched_seed_resampling_sensitivity_corrected",
    matched_distributions,
)

corrected_fixed_runs = build_family(
    "observed_runs_fixed_patient_cluster_only",
    fixed_run_distributions,
)

corrected_primary = pd.concat(
    [
        corrected_independent,
        corrected_matched,
        corrected_fixed_runs,
    ],
    ignore_index=True,
)

corrected_primary = corrected_primary.merge(
    run_level_tests,
    on=[
        "comparison",
        "metric",
    ],
    how="left",
)

corrected_primary.to_csv(
    OUTPUTS / "run3b_primary_inference_corrected.csv",
    index=False,
)


# ---------------------------------------------------------------------------
# Compare old and corrected familywise intervals
# ---------------------------------------------------------------------------
old_independent = old_primary_results[
    old_primary_results["analysis"]
    == "independent_run_resampling_primary"
].copy()

comparison_with_old = corrected_independent.merge(
    old_independent[
        [
            "comparison",
            "metric",
            "pointwise_ci_low",
            "pointwise_ci_high",
            "simultaneous_ci_low",
            "simultaneous_ci_high",
        ]
    ],
    on=[
        "comparison",
        "metric",
    ],
    how="left",
    suffixes=(
        "_corrected",
        "_old",
    ),
)

comparison_with_old[
    "simultaneous_low_change"
] = (
    comparison_with_old[
        "simultaneous_ci_low_corrected"
    ]
    - comparison_with_old[
        "simultaneous_ci_low_old"
    ]
)

comparison_with_old[
    "simultaneous_high_change"
] = (
    comparison_with_old[
        "simultaneous_ci_high_corrected"
    ]
    - comparison_with_old[
        "simultaneous_ci_high_old"
    ]
)

comparison_with_old.to_csv(
    OUTPUTS / "run3b_old_vs_corrected_intervals.csv",
    index=False,
)


# ---------------------------------------------------------------------------
# Combine the two estimands into the paper's key comparison table
# ---------------------------------------------------------------------------
fixed_discrimination = fixed_ensemble_results[
    fixed_ensemble_results["metric"].isin(
        METRICS
    )
].copy()

fixed_discrimination = fixed_discrimination[
    [
        "comparison",
        "metric",
        "estimate",
        "ci_low",
        "ci_high",
    ]
].rename(
    columns={
        "estimate": "fixed_ensemble_estimate",
        "ci_low": "fixed_ensemble_ci_low",
        "ci_high": "fixed_ensemble_ci_high",
    }
)

key_table = corrected_independent[
    [
        "comparison",
        "metric",
        "estimate",
        "pointwise_ci_low",
        "pointwise_ci_high",
        "simultaneous_ci_low",
        "simultaneous_ci_high",
    ]
].rename(
    columns={
        "estimate": "run_aware_estimate",
        "pointwise_ci_low": "run_aware_pointwise_ci_low",
        "pointwise_ci_high": "run_aware_pointwise_ci_high",
        "simultaneous_ci_low": "run_aware_simultaneous_ci_low",
        "simultaneous_ci_high": "run_aware_simultaneous_ci_high",
    }
)

key_table = key_table.merge(
    fixed_discrimination,
    on=[
        "comparison",
        "metric",
    ],
    how="left",
)


def excludes_zero(low, high):
    return bool(
        (low > 0)
        or (high < 0)
    )


key_table[
    "fixed_ensemble_interval_excludes_zero"
] = [
    excludes_zero(low, high)
    for low, high in zip(
        key_table["fixed_ensemble_ci_low"],
        key_table["fixed_ensemble_ci_high"],
    )
]

key_table[
    "run_aware_pointwise_excludes_zero"
] = [
    excludes_zero(low, high)
    for low, high in zip(
        key_table["run_aware_pointwise_ci_low"],
        key_table["run_aware_pointwise_ci_high"],
    )
]

key_table[
    "run_aware_simultaneous_excludes_zero"
] = [
    excludes_zero(low, high)
    for low, high in zip(
        key_table["run_aware_simultaneous_ci_low"],
        key_table["run_aware_simultaneous_ci_high"],
    )
]

key_table.to_csv(
    OUTPUTS / "run3b_key_estimand_comparison.csv",
    index=False,
)


# ---------------------------------------------------------------------------
# Produce a clean estimand-comparison forest plot
# ---------------------------------------------------------------------------
plot_rows = []

for _, row in key_table.iterrows():
    label = (
        f"{row['comparison']} | "
        f"{METRIC_LABELS[row['metric']]}"
    )

    plot_rows.append(
        {
            "label": label,
            "estimand": "Fixed seed ensemble",
            "estimate": row["fixed_ensemble_estimate"],
            "ci_low": row["fixed_ensemble_ci_low"],
            "ci_high": row["fixed_ensemble_ci_high"],
        }
    )

    plot_rows.append(
        {
            "label": label,
            "estimand": "Run-aware training procedure",
            "estimate": row["run_aware_estimate"],
            "ci_low": row["run_aware_pointwise_ci_low"],
            "ci_high": row["run_aware_pointwise_ci_high"],
        }
    )

plot_data = pd.DataFrame(
    plot_rows
)

unique_labels = list(
    dict.fromkeys(
        plot_data["label"]
    )
)

figure, axis = plt.subplots(
    figsize=(10.0, 7.0)
)

base_positions = np.arange(
    len(unique_labels)
)

offsets = {
    "Fixed seed ensemble": -0.13,
    "Run-aware training procedure": 0.13,
}

markers = {
    "Fixed seed ensemble": "o",
    "Run-aware training procedure": "s",
}

for estimand in [
    "Fixed seed ensemble",
    "Run-aware training procedure",
]:
    subset = plot_data[
        plot_data["estimand"] == estimand
    ]

    y_values = []

    for label in subset["label"]:
        y_values.append(
            unique_labels.index(label)
            + offsets[estimand]
        )

    y_values = np.asarray(
        y_values
    )

    estimates = subset["estimate"].to_numpy()
    lows = subset["ci_low"].to_numpy()
    highs = subset["ci_high"].to_numpy()

    axis.errorbar(
        estimates,
        y_values,
        xerr=[
            estimates - lows,
            highs - estimates,
        ],
        fmt=markers[estimand],
        capsize=4,
        label=estimand,
    )

axis.axvline(
    0,
    linestyle="--",
    linewidth=1,
)

axis.set_yticks(
    base_positions
)

axis.set_yticklabels(
    unique_labels
)

axis.set_xlabel(
    "Difference in AUROC "
    "(first method minus second method)"
)

axis.set_ylabel("")

axis.grid(
    axis="x",
    alpha=0.25,
)

axis.legend(
    frameon=False
)

figure.tight_layout()

figure.savefig(
    OUTPUTS / "run3b_estimand_comparison_forest.pdf",
    bbox_inches="tight",
)

figure.savefig(
    OUTPUTS / "run3b_estimand_comparison_forest.png",
    dpi=300,
    bbox_inches="tight",
)

plt.close(figure)


# ---------------------------------------------------------------------------
# Print the corrected result
# ---------------------------------------------------------------------------
pd.set_option(
    "display.max_columns",
    30,
)

pd.set_option(
    "display.width",
    220,
)

print("\nCORRECTED PRIMARY RUN-AWARE RESULTS")

print(
    corrected_independent[
        [
            "comparison",
            "metric",
            "estimate",
            "pointwise_ci_low",
            "pointwise_ci_high",
            "simultaneous_ci_low",
            "simultaneous_ci_high",
        ]
    ].to_string(
        index=False,
        float_format=lambda value: f"{value:+.6f}",
    )
)

print("\nKEY ESTIMAND COMPARISON")

print(
    key_table[
        [
            "comparison",
            "metric",
            "fixed_ensemble_estimate",
            "fixed_ensemble_ci_low",
            "fixed_ensemble_ci_high",
            "run_aware_estimate",
            "run_aware_pointwise_ci_low",
            "run_aware_pointwise_ci_high",
            "run_aware_simultaneous_ci_low",
            "run_aware_simultaneous_ci_high",
            "fixed_ensemble_interval_excludes_zero",
            "run_aware_pointwise_excludes_zero",
            "run_aware_simultaneous_excludes_zero",
        ]
    ].to_string(
        index=False,
        float_format=lambda value: f"{value:+.6f}",
    )
)

n_fixed_resolved = int(
    key_table[
        "fixed_ensemble_interval_excludes_zero"
    ].sum()
)

n_run_pointwise_resolved = int(
    key_table[
        "run_aware_pointwise_excludes_zero"
    ].sum()
)

n_run_simultaneous_resolved = int(
    key_table[
        "run_aware_simultaneous_excludes_zero"
    ].sum()
)

print("\nDECISION SUMMARY")
print(
    "Fixed-ensemble discrimination intervals "
    f"excluding zero: {n_fixed_resolved}/6"
)

print(
    "Run-aware pointwise intervals "
    f"excluding zero: {n_run_pointwise_resolved}/6"
)

print(
    "Run-aware simultaneous intervals "
    f"excluding zero: {n_run_simultaneous_resolved}/6"
)


# ---------------------------------------------------------------------------
# Metadata and ZIP
# ---------------------------------------------------------------------------
metadata = {
    "source_run3_zip": str(RUN3_ZIP),
    "n_patient_bootstrap": n_bootstrap,
    "correction": (
        "A single run-resampling index matrix was reused across "
        "all primary metrics within each method, preserving "
        "cross-metric dependence for familywise max-t inference."
    ),
    "primary_methods": METHODS,
    "primary_metrics": METRICS,
    "primary_comparisons": [
        f"{a} minus {b}"
        for a, b in COMPARISONS
    ],
    "random_seed": RANDOM_SEED,
    "run3_metadata": run3_metadata,
}

with open(
    OUTPUTS / "run3b_metadata.json",
    "w",
    encoding="utf-8",
) as file:
    json.dump(
        metadata,
        file,
        indent=2,
    )

local_zip = Path(
    "/content/revision_run3b.zip"
)

with zipfile.ZipFile(
    local_zip,
    "w",
    compression=zipfile.ZIP_DEFLATED,
) as archive:
    for path in sorted(
        OUTPUTS.rglob("*")
    ):
        if path.is_file():
            archive.write(
                path,
                arcname=path.relative_to(
                    OUTPUTS
                ),
            )

drive_zip = (
    RESULTS / "revision_run3b.zip"
)

shutil.copy2(
    local_zip,
    drive_zip,
)

print("\n" + "=" * 96)
print("RUN 3B COMPLETE")
print("Local ZIP:", local_zip)
print("Drive ZIP:", drive_zip)
print("=" * 96)

Mounted at /content/drive
RUN 3B: CORRECTED JOINT RUN RESAMPLING
Bootstrap replicates loaded: 2,000
Model/ensemble entries:       26

Individual model IDs:
LPR ['LPR|42', 'LPR|123', 'LPR|456', 'LPR|789', 'LPR|1010']
CADR ['CADR|42', 'CADR|123', 'CADR|456', 'CADR|789', 'CADR|1010']
DWFA ['DWFA|42', 'DWFA|123', 'DWFA|456', 'DWFA|789', 'DWFA|1010']

CORRECTED PRIMARY RUN-AWARE RESULTS
     comparison            metric  estimate  pointwise_ci_low  pointwise_ci_high  simultaneous_ci_low  simultaneous_ci_high
 DWFA minus LPR             auroc +0.002440         +0.000151          +0.004551            -0.000524             +0.005405
 DWFA minus LPR worst_group_auroc +0.001959         -0.000302          +0.004609            -0.001249             +0.005167
DWFA minus CADR             auroc +0.001335         -0.001370          +0.003902            -0.002210             +0.004879
DWFA minus CADR worst_group_auroc +0.000844         -0.001486          +0.003826            -0.002705             +0.00

In [ ]:
# ============================================================================
# RUN 4 — CORRECTED FULL VERSION
# CHECKPOINT SELECTION, DWFA WEIGHTS, AND DEPENDENCE AUDIT
#
# CPU only.
# No training.
# No image inference.
#
# Main output:
#   /content/drive/MyDrive/FairFedCXR/results/revision_run4.zip
#
# Required files:
#   results/dwfa_round_log.csv
#   results/qfedavg_round_log.csv
#   results/fedavg_round_log.csv
#   results/fedprox_round_log.csv
#
# Optional:
#   results/fairfed_b0.1_seed42_weightlog.csv
#   checkpoints/fairfed_seed*.pt
# ============================================================================

from google.colab import drive
drive.mount("/content/drive", force_remount=True)

from pathlib import Path
import gc
import json
import shutil
import zipfile

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from scipy.stats import spearmanr


# ---------------------------------------------------------------------------
# Paths and constants
# ---------------------------------------------------------------------------
BASE = Path("/content/drive/MyDrive/FairFedCXR")
RESULTS = BASE / "results"
CHECKPOINTS = BASE / "checkpoints"

LOCAL = Path("/content/revision_run4")
INPUTS = LOCAL / "inputs"
OUTPUTS = LOCAL / "outputs"

# Remove outputs left by the failed attempt.
if LOCAL.exists():
    shutil.rmtree(LOCAL)

INPUTS.mkdir(parents=True, exist_ok=True)
OUTPUTS.mkdir(parents=True, exist_ok=True)

DWFA_LOG = RESULTS / "dwfa_round_log.csv"
LPR_LOG = RESULTS / "qfedavg_round_log.csv"
FEDAVG_LOG = RESULTS / "fedavg_round_log.csv"
FEDPROX_LOG = RESULTS / "fedprox_round_log.csv"
CADR_PARTIAL_LOG = RESULTS / "fairfed_b0.1_seed42_weightlog.csv"

for required_path in [
    DWFA_LOG,
    LPR_LOG,
    FEDAVG_LOG,
    FEDPROX_LOG,
]:
    if not required_path.exists():
        raise FileNotFoundError(
            f"Required log is missing: {required_path}"
        )

# DWFA constants used in the completed experiments.
RHO = 0.10
ALPHA = 0.40
KAPPA = 0.60
G_REF = 0.05
Q_MIN = ALPHA + KAPPA * RHO

CLIENT_ORDER = ["A", "B", "C", "D"]

print("=" * 96)
print("RUN 4: CHECKPOINT SELECTION, DWFA WEIGHTS, AND DEPENDENCE AUDIT")
print("=" * 96)


# ---------------------------------------------------------------------------
# Copy files locally before reading
# ---------------------------------------------------------------------------
def copy_local(source):
    destination = INPUTS / source.name
    shutil.copy2(source, destination)
    return destination


local_paths = {}

for source in [
    DWFA_LOG,
    LPR_LOG,
    FEDAVG_LOG,
    FEDPROX_LOG,
]:
    local_paths[source.name] = copy_local(source)

if CADR_PARTIAL_LOG.exists():
    local_paths[CADR_PARTIAL_LOG.name] = copy_local(
        CADR_PARTIAL_LOG
    )


# ---------------------------------------------------------------------------
# Helper functions
# ---------------------------------------------------------------------------
def first_maximum_row(frame, metric_column):
    """
    Return the earliest round attaining the maximum validation metric.

    The completed training code retained a checkpoint only when validation
    AUROC strictly improved. Therefore, an equal-valued later round would not
    replace the earlier retained checkpoint.
    """
    frame = frame.sort_values("round").copy()
    maximum = frame[metric_column].max()

    candidates = frame[
        np.isclose(
            frame[metric_column],
            maximum,
            rtol=0.0,
            atol=1e-12,
        )
    ]

    return candidates.iloc[0]


def collapse_round_log(
    frame,
    method,
    metric_column,
    repeated_by_client,
    secondary_metric_columns=None,
):
    """
    Convert a log into one row per seed and communication round.

    DWFA and LPR logs contain one repeated validation result per client.
    FedAvg and FedProx contain one result per round.
    """
    frame = frame.copy()

    required = {
        "seed",
        "round",
        metric_column,
    }

    missing = required - set(frame.columns)

    if missing:
        raise KeyError(
            f"{method} log missing columns: {sorted(missing)}"
        )

    frame["seed"] = pd.to_numeric(
        frame["seed"],
        errors="raise",
    ).astype(int)

    frame["round"] = pd.to_numeric(
        frame["round"],
        errors="raise",
    ).astype(int)

    metric_columns = [metric_column]

    for column in secondary_metric_columns or []:
        if column in frame.columns:
            metric_columns.append(column)

    if repeated_by_client:
        consistency_rows = []

        for (seed, communication_round), group in frame.groupby(
            ["seed", "round"]
        ):
            row = {
                "method": method,
                "seed": int(seed),
                "round": int(communication_round),
                "n_rows": int(len(group)),
            }

            for column in metric_columns:
                row[column] = float(group[column].iloc[0])

                row[f"{column}_within_round_range"] = float(
                    group[column].max()
                    - group[column].min()
                )

            consistency_rows.append(row)

        round_frame = pd.DataFrame(consistency_rows)

        range_columns = [
            column
            for column in round_frame.columns
            if column.endswith("_within_round_range")
        ]

        for range_column in range_columns:
            maximum_range = round_frame[range_column].max()

            if maximum_range > 1e-10:
                raise RuntimeError(
                    f"{method}: repeated client rows disagree on "
                    f"{range_column}. Maximum range={maximum_range}"
                )

    else:
        selected_columns = [
            "seed",
            "round",
            *metric_columns,
        ]

        round_frame = frame[
            selected_columns
        ].drop_duplicates(
            ["seed", "round"]
        )

        round_frame["method"] = method
        round_frame["n_rows"] = 1

    return round_frame.sort_values(
        ["seed", "round"]
    ).reset_index(drop=True)


def checkpoint_selection_table(
    round_frame,
    metric_column,
):
    rows = []

    for seed, group in round_frame.groupby("seed"):
        group = group.sort_values("round")

        selected = first_maximum_row(
            group,
            metric_column,
        )

        final = group.iloc[-1]

        rows.append(
            {
                "method": selected["method"],
                "seed": int(seed),
                "n_logged_rounds": int(len(group)),
                "first_logged_round": int(
                    group["round"].min()
                ),
                "final_round": int(
                    group["round"].max()
                ),
                "selected_round": int(
                    selected["round"]
                ),
                "selected_validation_auroc": float(
                    selected[metric_column]
                ),
                "final_validation_auroc": float(
                    final[metric_column]
                ),
                "selected_minus_final_auroc": float(
                    selected[metric_column]
                    - final[metric_column]
                ),
                "selected_within_first_3_rounds": bool(
                    selected["round"] <= 3
                ),
            }
        )

    return pd.DataFrame(rows)


def pearson_correlation(x, y):
    x = np.asarray(x, dtype=float)
    y = np.asarray(y, dtype=float)

    valid = (
        np.isfinite(x)
        & np.isfinite(y)
    )

    x = x[valid]
    y = y[valid]

    if len(x) < 3:
        return np.nan

    if np.std(x) == 0 or np.std(y) == 0:
        return np.nan

    return float(
        np.corrcoef(x, y)[0, 1]
    )


def spearman_correlation(x, y):
    x = np.asarray(x, dtype=float)
    y = np.asarray(y, dtype=float)

    valid = (
        np.isfinite(x)
        & np.isfinite(y)
    )

    x = x[valid]
    y = y[valid]

    if len(x) < 3:
        return np.nan

    if np.std(x) == 0 or np.std(y) == 0:
        return np.nan

    # The p-value is deliberately discarded.
    return float(
        spearmanr(x, y).statistic
    )


def summarize_numeric(
    frame,
    grouping_columns=None,
    value_columns=None,
):
    """
    Summarize numeric columns either within groups or over the whole frame.

    CORRECTION:
    When grouping_columns is empty, Pandas groupby([]) must not be called.
    Instead, the entire DataFrame is treated as one overall group.
    """
    grouping_columns = list(
        grouping_columns or []
    )

    value_columns = list(
        value_columns or []
    )

    rows = []

    if grouping_columns:
        grouped_iterator = frame.groupby(
            grouping_columns,
            dropna=False,
        )
    else:
        grouped_iterator = [
            ((), frame)
        ]

    for group_key, group in grouped_iterator:
        if not isinstance(group_key, tuple):
            group_key = (group_key,)

        base = {}

        if grouping_columns:
            base = dict(
                zip(
                    grouping_columns,
                    group_key,
                )
            )

        for value_column in value_columns:
            values = (
                pd.to_numeric(
                    group[value_column],
                    errors="coerce",
                )
                .dropna()
                .to_numpy(dtype=float)
            )

            if len(values) == 0:
                continue

            rows.append(
                {
                    **base,
                    "measure": value_column,
                    "n": int(len(values)),
                    "mean": float(values.mean()),
                    "sd": (
                        float(values.std(ddof=1))
                        if len(values) > 1
                        else np.nan
                    ),
                    "minimum": float(values.min()),
                    "q1": float(
                        np.percentile(values, 25)
                    ),
                    "median": float(
                        np.median(values)
                    ),
                    "q3": float(
                        np.percentile(values, 75)
                    ),
                    "maximum": float(values.max()),
                }
            )

    return pd.DataFrame(rows)


# ===========================================================================
# PART A: CHECKPOINT-SELECTION AUDIT
# ===========================================================================
print("\n" + "=" * 96)
print("PART A: CHECKPOINT-SELECTION AUDIT")
print("=" * 96)

dwfa_raw = pd.read_csv(
    local_paths[DWFA_LOG.name],
    low_memory=False,
)

lpr_raw = pd.read_csv(
    local_paths[LPR_LOG.name],
    low_memory=False,
)

fedavg_raw = pd.read_csv(
    local_paths[FEDAVG_LOG.name],
    low_memory=False,
)

fedprox_raw = pd.read_csv(
    local_paths[FEDPROX_LOG.name],
    low_memory=False,
)

dwfa_rounds = collapse_round_log(
    dwfa_raw,
    method="DWFA",
    metric_column="pooled_val_auroc",
    repeated_by_client=True,
    secondary_metric_columns=[
        "pooled_val_eo_gap",
        "pooled_val_wg_auroc",
    ],
)

lpr_rounds = collapse_round_log(
    lpr_raw,
    method="LPR",
    metric_column="val_auroc",
    repeated_by_client=True,
    secondary_metric_columns=[
        "val_eo_gap",
        "val_wg_auroc",
    ],
)

fedavg_rounds = collapse_round_log(
    fedavg_raw,
    method="FedAvg",
    metric_column="val_auroc",
    repeated_by_client=False,
    secondary_metric_columns=[
        "val_eo_gap",
        "val_ece",
    ],
)

fedprox_rounds = collapse_round_log(
    fedprox_raw,
    method="FedProx",
    metric_column="val_auroc",
    repeated_by_client=False,
    secondary_metric_columns=[
        "val_eo_gap",
        "val_ece",
    ],
)

checkpoint_tables = [
    checkpoint_selection_table(
        dwfa_rounds,
        "pooled_val_auroc",
    ),
    checkpoint_selection_table(
        lpr_rounds,
        "val_auroc",
    ),
    checkpoint_selection_table(
        fedavg_rounds,
        "val_auroc",
    ),
    checkpoint_selection_table(
        fedprox_rounds,
        "val_auroc",
    ),
]

checkpoint_selection = pd.concat(
    checkpoint_tables,
    ignore_index=True,
)

checkpoint_selection.to_csv(
    OUTPUTS / "run4_checkpoint_selection.csv",
    index=False,
)

print("\nSelected checkpoints reconstructed from retained logs:")
print(
    checkpoint_selection.to_string(
        index=False,
        float_format=lambda value: f"{value:.6f}",
    )
)

checkpoint_summary = (
    checkpoint_selection.groupby("method")
    .agg(
        n_seeds=("seed", "nunique"),
        minimum_selected_round=(
            "selected_round",
            "min",
        ),
        median_selected_round=(
            "selected_round",
            "median",
        ),
        maximum_selected_round=(
            "selected_round",
            "max",
        ),
        n_within_first_3=(
            "selected_within_first_3_rounds",
            "sum",
        ),
        mean_selected_minus_final_auroc=(
            "selected_minus_final_auroc",
            "mean",
        ),
    )
    .reset_index()
)

checkpoint_summary.to_csv(
    OUTPUTS / "run4_checkpoint_summary.csv",
    index=False,
)

print("\nCheckpoint summary:")
print(
    checkpoint_summary.to_string(
        index=False,
        float_format=lambda value: f"{value:.6f}",
    )
)


# ---------------------------------------------------------------------------
# Check whether main CADR checkpoints contain round metadata
# ---------------------------------------------------------------------------
print("\nCADR checkpoint metadata audit:")

cadr_checkpoint_rows = []

try:
    import torch

    cadr_checkpoint_paths = sorted(
        CHECKPOINTS.glob("fairfed_seed*.pt")
    )

    for checkpoint_path in cadr_checkpoint_paths:
        checkpoint = torch.load(
            checkpoint_path,
            map_location="cpu",
            weights_only=False,
        )

        if isinstance(checkpoint, dict):
            keys = list(checkpoint.keys())
        else:
            keys = []

        metadata_candidates = {}

        for key in [
            "round",
            "best_round",
            "selected_round",
            "epoch",
            "best_epoch",
            "val_auroc",
            "best_val_auroc",
        ]:
            if (
                isinstance(checkpoint, dict)
                and key in checkpoint
            ):
                value = checkpoint[key]

                if np.isscalar(value):
                    metadata_candidates[key] = value

        cadr_checkpoint_rows.append(
            {
                "file": checkpoint_path.name,
                "object_type": type(
                    checkpoint
                ).__name__,
                "n_top_level_keys": int(
                    len(keys)
                ),
                "round_metadata_found": bool(
                    metadata_candidates
                ),
                "metadata": json.dumps(
                    metadata_candidates,
                    default=str,
                ),
                "first_top_level_keys": "|".join(
                    map(str, keys[:12])
                ),
            }
        )

        del checkpoint
        gc.collect()

except Exception as error:
    cadr_checkpoint_rows.append(
        {
            "file": "audit_error",
            "object_type": "",
            "n_top_level_keys": 0,
            "round_metadata_found": False,
            "metadata": json.dumps(
                {"error": repr(error)}
            ),
            "first_top_level_keys": "",
        }
    )

cadr_checkpoint_metadata = pd.DataFrame(
    cadr_checkpoint_rows
)

cadr_checkpoint_metadata.to_csv(
    OUTPUTS / "run4_cadr_checkpoint_metadata.csv",
    index=False,
)

if len(cadr_checkpoint_metadata):
    print(
        cadr_checkpoint_metadata.to_string(
            index=False
        )
    )
else:
    print("No main CADR checkpoints were found.")


# ===========================================================================
# PART B: DWFA LOG INTEGRITY AND IMPLEMENTATION RECONSTRUCTION
# ===========================================================================
print("\n" + "=" * 96)
print("PART B: DWFA LOG AND EQUATION AUDIT")
print("=" * 96)

required_dwfa_columns = {
    "seed",
    "round",
    "client",
    "val_eo_gap_local",
    "e_cur",
    "ebar_lagged",
    "etilde",
    "r_conf",
    "weight",
    "fedavg_weight",
    "pooled_val_auroc",
}

missing_dwfa_columns = (
    required_dwfa_columns
    - set(dwfa_raw.columns)
)

if missing_dwfa_columns:
    raise KeyError(
        "DWFA log missing columns: "
        f"{sorted(missing_dwfa_columns)}"
    )

dwfa = dwfa_raw.copy()

dwfa["seed"] = pd.to_numeric(
    dwfa["seed"]
).astype(int)

dwfa["round"] = pd.to_numeric(
    dwfa["round"]
).astype(int)

dwfa["client"] = dwfa["client"].astype(str)

dwfa = dwfa.sort_values(
    ["seed", "round", "client"]
).reset_index(drop=True)

# Attach the selected checkpoint round for each seed.
dwfa_selected_rounds = (
    checkpoint_selection[
        checkpoint_selection["method"] == "DWFA"
    ][
        [
            "seed",
            "selected_round",
        ]
    ]
)

dwfa = dwfa.merge(
    dwfa_selected_rounds,
    on="seed",
    how="left",
    validate="many_to_one",
)

if dwfa["selected_round"].isna().any():
    raise RuntimeError(
        "Some DWFA rows could not be assigned a selected round."
    )

dwfa["selected_round"] = (
    dwfa["selected_round"].astype(int)
)

dwfa["is_selected_round"] = (
    dwfa["round"]
    == dwfa["selected_round"]
)

dwfa["phase"] = np.where(
    dwfa["round"]
    <= dwfa["selected_round"],
    "model_forming_rounds",
    "post_selection_unused_rounds",
)

# Theoretical bounds.
dwfa["theoretical_lower_bound"] = (
    Q_MIN
    * dwfa["fedavg_weight"]
)

dwfa["theoretical_upper_bound"] = (
    dwfa["fedavg_weight"]
    / Q_MIN
)

dwfa["lower_margin"] = (
    dwfa["weight"]
    - dwfa["theoretical_lower_bound"]
)

dwfa["upper_margin"] = (
    dwfa["theoretical_upper_bound"]
    - dwfa["weight"]
)

BOUND_TOLERANCE = 1e-10

dwfa["lower_bound_violation"] = (
    dwfa["weight"]
    < dwfa["theoretical_lower_bound"]
    - BOUND_TOLERANCE
)

dwfa["upper_bound_violation"] = (
    dwfa["weight"]
    > dwfa["theoretical_upper_bound"]
    + BOUND_TOLERANCE
)

# Reconstruct each stage of the DWFA equation.
dwfa["e_cur_reconstructed"] = np.clip(
    1.0
    - dwfa["val_eo_gap_local"]
    / G_REF,
    RHO,
    1.0,
)

dwfa["etilde_reconstructed"] = (
    (1.0 - dwfa["r_conf"])
    * dwfa["ebar_lagged"]
    + dwfa["r_conf"]
    * dwfa["e_cur"]
)

dwfa["quality_reconstructed"] = (
    ALPHA
    + KAPPA * dwfa["etilde"]
)

dwfa["unnormalized_score_reconstructed"] = (
    dwfa["fedavg_weight"]
    * dwfa["quality_reconstructed"]
)

score_denominator = (
    dwfa.groupby(
        ["seed", "round"]
    )[
        "unnormalized_score_reconstructed"
    ]
    .transform("sum")
)

dwfa["weight_reconstructed"] = (
    dwfa["unnormalized_score_reconstructed"]
    / score_denominator
)

dwfa["e_cur_absolute_error"] = np.abs(
    dwfa["e_cur"]
    - dwfa["e_cur_reconstructed"]
)

dwfa["etilde_absolute_error"] = np.abs(
    dwfa["etilde"]
    - dwfa["etilde_reconstructed"]
)

dwfa["weight_absolute_error"] = np.abs(
    dwfa["weight"]
    - dwfa["weight_reconstructed"]
)

dwfa["weight_ratio_to_prior"] = (
    dwfa["weight"]
    / dwfa["fedavg_weight"]
)

dwfa["weight_percent_change"] = (
    100.0
    * (
        dwfa["weight_ratio_to_prior"]
        - 1.0
    )
)

dwfa["equity_at_floor"] = (
    dwfa["e_cur"]
    <= RHO + 1e-10
)

dwfa["equity_at_ceiling"] = (
    dwfa["e_cur"]
    >= 1.0 - 1e-10
)

# Sum-to-one and client-count checks.
round_integrity = (
    dwfa.groupby(
        ["seed", "round"]
    )
    .agg(
        n_clients=("client", "nunique"),
        weight_sum=("weight", "sum"),
        prior_sum=("fedavg_weight", "sum"),
        selected_round=("selected_round", "first"),
        phase=("phase", "first"),
    )
    .reset_index()
)

round_integrity[
    "weight_sum_absolute_error"
] = np.abs(
    round_integrity["weight_sum"]
    - 1.0
)

round_integrity[
    "prior_sum_absolute_error"
] = np.abs(
    round_integrity["prior_sum"]
    - 1.0
)

round_integrity.to_csv(
    OUTPUTS / "run4_dwfa_round_integrity.csv",
    index=False,
)

equation_audit = pd.DataFrame(
    [
        {
            "n_logged_rows": int(len(dwfa)),
            "n_seeds": int(
                dwfa["seed"].nunique()
            ),
            "n_rounds_per_seed_min": int(
                dwfa.groupby("seed")[
                    "round"
                ].nunique().min()
            ),
            "n_rounds_per_seed_max": int(
                dwfa.groupby("seed")[
                    "round"
                ].nunique().max()
            ),
            "n_clients": int(
                dwfa["client"].nunique()
            ),
            "maximum_weight_sum_error": float(
                round_integrity[
                    "weight_sum_absolute_error"
                ].max()
            ),
            "maximum_prior_sum_error": float(
                round_integrity[
                    "prior_sum_absolute_error"
                ].max()
            ),
            "maximum_e_cur_error": float(
                dwfa[
                    "e_cur_absolute_error"
                ].max()
            ),
            "maximum_etilde_error": float(
                dwfa[
                    "etilde_absolute_error"
                ].max()
            ),
            "maximum_weight_reconstruction_error": float(
                dwfa[
                    "weight_absolute_error"
                ].max()
            ),
            "lower_bound_violations": int(
                dwfa[
                    "lower_bound_violation"
                ].sum()
            ),
            "upper_bound_violations": int(
                dwfa[
                    "upper_bound_violation"
                ].sum()
            ),
            "closest_lower_bound_margin": float(
                dwfa["lower_margin"].min()
            ),
            "closest_upper_bound_margin": float(
                dwfa["upper_margin"].min()
            ),
            "rows_equity_at_floor": int(
                dwfa["equity_at_floor"].sum()
            ),
            "rows_equity_at_ceiling": int(
                dwfa["equity_at_ceiling"].sum()
            ),
        }
    ]
)

equation_audit.to_csv(
    OUTPUTS / "run4_dwfa_equation_audit.csv",
    index=False,
)

print("\nDWFA equation and bound audit:")
print(
    equation_audit.to_string(
        index=False,
        float_format=lambda value: f"{value:.12g}",
    )
)


# ===========================================================================
# PART C: WEIGHTS THAT ACTUALLY FORMED THE SELECTED CHECKPOINT
# ===========================================================================
print("\n" + "=" * 96)
print("PART C: DWFA WEIGHT REDISTRIBUTION BY ANALYSIS PHASE")
print("=" * 96)

weight_summary_frames = []

analysis_subsets = {
    "all_30_logged_rounds": dwfa,
    "model_forming_rounds": dwfa[
        dwfa["phase"]
        == "model_forming_rounds"
    ],
    "selected_checkpoint_round_only": dwfa[
        dwfa["is_selected_round"]
    ],
    "post_selection_unused_rounds": dwfa[
        dwfa["phase"]
        == "post_selection_unused_rounds"
    ],
}

for analysis_period, subset in analysis_subsets.items():
    if len(subset) == 0:
        continue

    grouped = (
        subset.groupby("client")
        .agg(
            n_observations=("weight", "size"),
            n_seeds=("seed", "nunique"),
            mean_weight=("weight", "mean"),
            sd_weight=("weight", "std"),
            minimum_weight=("weight", "min"),
            maximum_weight=("weight", "max"),
            fedavg_prior=("fedavg_weight", "mean"),
            mean_weight_ratio_to_prior=(
                "weight_ratio_to_prior",
                "mean",
            ),
            mean_percent_change_from_prior=(
                "weight_percent_change",
                "mean",
            ),
            minimum_lower_bound_margin=(
                "lower_margin",
                "min",
            ),
            equity_floor_fraction=(
                "equity_at_floor",
                "mean",
            ),
        )
        .reset_index()
    )

    grouped["analysis_period"] = analysis_period
    weight_summary_frames.append(grouped)

weight_summary = pd.concat(
    weight_summary_frames,
    ignore_index=True,
)

weight_summary["client"] = pd.Categorical(
    weight_summary["client"],
    categories=CLIENT_ORDER,
    ordered=True,
)

weight_summary = weight_summary.sort_values(
    ["analysis_period", "client"]
)

weight_summary.to_csv(
    OUTPUTS / "run4_dwfa_weight_summary_by_phase.csv",
    index=False,
)

print("\nDWFA mean weight relative to FedAvg prior:")
print(
    weight_summary[
        [
            "analysis_period",
            "client",
            "n_observations",
            "mean_weight",
            "fedavg_prior",
            "mean_weight_ratio_to_prior",
            "mean_percent_change_from_prior",
            "minimum_lower_bound_margin",
        ]
    ].to_string(
        index=False,
        float_format=lambda value: f"{value:.6f}",
    )
)

# Save per-seed, per-client model-forming summaries.
model_forming_seed_summary = (
    dwfa[
        dwfa["phase"]
        == "model_forming_rounds"
    ]
    .groupby(
        ["seed", "client"]
    )
    .agg(
        selected_round=("selected_round", "first"),
        n_model_forming_rounds=("round", "nunique"),
        mean_weight=("weight", "mean"),
        mean_prior=("fedavg_weight", "mean"),
        mean_ratio_to_prior=(
            "weight_ratio_to_prior",
            "mean",
        ),
        minimum_weight=("weight", "min"),
        maximum_weight=("weight", "max"),
    )
    .reset_index()
)

model_forming_seed_summary.to_csv(
    OUTPUTS
    / "run4_dwfa_model_forming_weights_per_seed.csv",
    index=False,
)


# ===========================================================================
# PART D: AGGREGATION CONCENTRATION
# ===========================================================================
print("\n" + "=" * 96)
print("PART D: AGGREGATION CONCENTRATION")
print("=" * 96)

concentration_rows = []

for (seed, communication_round), group in dwfa.groupby(
    ["seed", "round"]
):
    group = group.sort_values("client")

    weights = group["weight"].to_numpy(
        dtype=float
    )

    priors = group[
        "fedavg_weight"
    ].to_numpy(dtype=float)

    hhi = float(
        np.sum(weights ** 2)
    )

    prior_hhi = float(
        np.sum(priors ** 2)
    )

    entropy = float(
        -np.sum(
            weights
            * np.log(
                np.clip(
                    weights,
                    1e-15,
                    None,
                )
            )
        )
    )

    normalized_entropy = float(
        entropy
        / np.log(len(weights))
    )

    total_variation_from_prior = float(
        0.5
        * np.sum(
            np.abs(
                weights - priors
            )
        )
    )

    concentration_rows.append(
        {
            "seed": int(seed),
            "round": int(
                communication_round
            ),
            "selected_round": int(
                group[
                    "selected_round"
                ].iloc[0]
            ),
            "phase": group[
                "phase"
            ].iloc[0],
            "is_selected_round": bool(
                group[
                    "is_selected_round"
                ].iloc[0]
            ),
            "hhi": hhi,
            "fedavg_prior_hhi": prior_hhi,
            "hhi_minus_prior": (
                hhi - prior_hhi
            ),
            "effective_number_of_clients": float(
                1.0 / hhi
            ),
            "fedavg_effective_number_of_clients": float(
                1.0 / prior_hhi
            ),
            "normalized_entropy": normalized_entropy,
            "total_variation_from_prior": (
                total_variation_from_prior
            ),
            "maximum_client_weight": float(
                weights.max()
            ),
            "minimum_client_weight": float(
                weights.min()
            ),
        }
    )

concentration = pd.DataFrame(
    concentration_rows
)

concentration.to_csv(
    OUTPUTS / "run4_dwfa_concentration_per_round.csv",
    index=False,
)

concentration_period_frames = []

concentration_subsets = {
    "all_30_logged_rounds": concentration,
    "model_forming_rounds": concentration[
        concentration["phase"]
        == "model_forming_rounds"
    ],
    "selected_checkpoint_round_only": concentration[
        concentration["is_selected_round"]
    ],
    "post_selection_unused_rounds": concentration[
        concentration["phase"]
        == "post_selection_unused_rounds"
    ],
}

for analysis_period, subset in concentration_subsets.items():
    if len(subset) == 0:
        continue

    row = {
        "analysis_period": analysis_period,
        "n_seed_rounds": int(len(subset)),
    }

    for column in [
        "hhi",
        "hhi_minus_prior",
        "effective_number_of_clients",
        "normalized_entropy",
        "total_variation_from_prior",
        "maximum_client_weight",
        "minimum_client_weight",
    ]:
        row[f"{column}_mean"] = float(
            subset[column].mean()
        )

        row[f"{column}_sd"] = (
            float(
                subset[column].std(ddof=1)
            )
            if len(subset) > 1
            else np.nan
        )

        row[f"{column}_minimum"] = float(
            subset[column].min()
        )

        row[f"{column}_maximum"] = float(
            subset[column].max()
        )

    concentration_period_frames.append(row)

concentration_summary = pd.DataFrame(
    concentration_period_frames
)

concentration_summary.to_csv(
    OUTPUTS / "run4_dwfa_concentration_summary.csv",
    index=False,
)

print("\nAggregation concentration summary:")
print(
    concentration_summary[
        [
            "analysis_period",
            "n_seed_rounds",
            "hhi_minus_prior_mean",
            "effective_number_of_clients_mean",
            "total_variation_from_prior_mean",
            "maximum_client_weight_mean",
        ]
    ].to_string(
        index=False,
        float_format=lambda value: f"{value:.6f}",
    )
)


# ===========================================================================
# PART E: WEIGHT–GAP ASSOCIATION WITHOUT PSEUDOREPLICATED P-VALUES
# ===========================================================================
print("\n" + "=" * 96)
print("PART E: WEIGHT–GAP ASSOCIATION")
print("=" * 96)

transition_frames = []

for (seed, client), group in dwfa.groupby(
    ["seed", "client"]
):
    group = group.sort_values(
        "round"
    ).copy()

    group["delta_weight"] = (
        group["weight"].diff()
    )

    group["delta_gap"] = (
        group[
            "val_eo_gap_local"
        ].diff()
    )

    group["transition_to_round"] = (
        group["round"]
    )

    group["transition_phase"] = np.where(
        group[
            "transition_to_round"
        ]
        <= group[
            "selected_round"
        ],
        "model_forming_transition",
        "post_selection_transition",
    )

    transition_frames.append(
        group[
            [
                "seed",
                "client",
                "transition_to_round",
                "selected_round",
                "transition_phase",
                "delta_weight",
                "delta_gap",
            ]
        ]
    )

transitions = pd.concat(
    transition_frames,
    ignore_index=True,
).dropna(
    subset=[
        "delta_weight",
        "delta_gap",
    ]
)

transitions.to_csv(
    OUTPUTS / "run4_dwfa_round_transitions.csv",
    index=False,
)

trajectory_rows = []

for (seed, client), group in transitions.groupby(
    ["seed", "client"]
):
    trajectory_rows.append(
        {
            "seed": int(seed),
            "client": client,
            "n_transitions": int(len(group)),
            "pearson_r": pearson_correlation(
                group["delta_gap"],
                group["delta_weight"],
            ),
            "spearman_rho": spearman_correlation(
                group["delta_gap"],
                group["delta_weight"],
            ),
        }
    )

trajectory_correlations = pd.DataFrame(
    trajectory_rows
)

trajectory_correlations.to_csv(
    OUTPUTS
    / "run4_dwfa_correlations_per_seed_client.csv",
    index=False,
)

seed_rows = []

for seed, group in transitions.groupby("seed"):
    seed_rows.append(
        {
            "seed": int(seed),
            "n_transitions": int(len(group)),
            "pearson_r": pearson_correlation(
                group["delta_gap"],
                group["delta_weight"],
            ),
            "spearman_rho": spearman_correlation(
                group["delta_gap"],
                group["delta_weight"],
            ),
        }
    )

seed_correlations = pd.DataFrame(
    seed_rows
)

seed_correlations.to_csv(
    OUTPUTS / "run4_dwfa_correlations_per_seed.csv",
    index=False,
)

stage_rows = []

for stage, group in transitions.groupby(
    "transition_phase"
):
    stage_rows.append(
        {
            "transition_phase": stage,
            "n_transitions": int(len(group)),
            "pearson_r": pearson_correlation(
                group["delta_gap"],
                group["delta_weight"],
            ),
            "spearman_rho": spearman_correlation(
                group["delta_gap"],
                group["delta_weight"],
            ),
        }
    )

stage_correlations = pd.DataFrame(
    stage_rows
)

pooled_correlation = pd.DataFrame(
    [
        {
            "analysis": (
                "pooled_descriptive_only"
            ),
            "n_transitions": int(
                len(transitions)
            ),
            "pearson_r": pearson_correlation(
                transitions["delta_gap"],
                transitions["delta_weight"],
            ),
            "spearman_rho": spearman_correlation(
                transitions["delta_gap"],
                transitions["delta_weight"],
            ),
            "p_value_reported": False,
            "interpretation": (
                "Implementation-behaviour check only; "
                "transitions are nested within clients, "
                "seeds, and communication rounds."
            ),
        }
    ]
)

pooled_correlation.to_csv(
    OUTPUTS / "run4_dwfa_pooled_correlation_descriptive.csv",
    index=False,
)

stage_correlations.to_csv(
    OUTPUTS / "run4_dwfa_correlations_by_phase.csv",
    index=False,
)

# Corrected: no groupby([]) call.
trajectory_distribution_summary = summarize_numeric(
    trajectory_correlations,
    grouping_columns=None,
    value_columns=[
        "pearson_r",
        "spearman_rho",
    ],
)

trajectory_distribution_summary.to_csv(
    OUTPUTS
    / "run4_dwfa_trajectory_correlation_summary.csv",
    index=False,
)

print("\nPooled descriptive association — no independence-based p-value:")
print(
    pooled_correlation.to_string(
        index=False,
        float_format=lambda value: f"{value:.6f}",
    )
)

print("\nAssociation by training phase:")
print(
    stage_correlations.to_string(
        index=False,
        float_format=lambda value: f"{value:.6f}",
    )
)

print("\nDistribution across the 20 seed–client trajectories:")
print(
    trajectory_distribution_summary.to_string(
        index=False,
        float_format=lambda value: f"{value:.6f}",
    )
)

print("\nPer-seed/client trajectory correlations:")
print(
    trajectory_correlations.to_string(
        index=False,
        float_format=lambda value: f"{value:.6f}",
    )
)


# ===========================================================================
# PART F: AVAILABLE CADR/FAIRFED-STYLE LOG EVIDENCE
# ===========================================================================
print("\n" + "=" * 96)
print("PART F: AVAILABLE CADR LOG EVIDENCE")
print("=" * 96)

cadr_availability_rows = []

if CADR_PARTIAL_LOG.exists():
    cadr = pd.read_csv(
        local_paths[
            CADR_PARTIAL_LOG.name
        ],
        low_memory=False,
    )

    required_cadr = {
        "beta",
        "seed",
        "round",
        "client",
        "omega_bar_raw",
        "omega_bar",
        "omega",
        "fedavg_prior",
    }

    missing_cadr = (
        required_cadr
        - set(cadr.columns)
    )

    if missing_cadr:
        cadr_availability_rows.append(
            {
                "file": CADR_PARTIAL_LOG.name,
                "usable": False,
                "reason": (
                    "Missing columns: "
                    + ",".join(
                        sorted(
                            missing_cadr
                        )
                    )
                ),
            }
        )

    else:
        for (
            beta,
            seed,
            client,
        ), group in cadr.groupby(
            [
                "beta",
                "seed",
                "client",
            ]
        ):
            group = group.sort_values(
                "round"
            )

            cadr_availability_rows.append(
                {
                    "file": CADR_PARTIAL_LOG.name,
                    "usable": True,
                    "beta": float(beta),
                    "seed": int(seed),
                    "client": client,
                    "first_round": int(
                        group["round"].min()
                    ),
                    "last_round": int(
                        group["round"].max()
                    ),
                    "n_logged_rounds": int(
                        group["round"].nunique()
                    ),
                    "fedavg_prior": float(
                        group[
                            "fedavg_prior"
                        ].iloc[0]
                    ),
                    "minimum_normalized_weight": float(
                        group["omega"].min()
                    ),
                    "final_normalized_weight": float(
                        group["omega"].iloc[-1]
                    ),
                    "minimum_raw_bar": float(
                        group[
                            "omega_bar_raw"
                        ].min()
                    ),
                    "final_raw_bar": float(
                        group[
                            "omega_bar_raw"
                        ].iloc[-1]
                    ),
                    "raw_bar_below_zero_observed": bool(
                        (
                            group[
                                "omega_bar_raw"
                            ]
                            < 0
                        ).any()
                    ),
                    "complete_30_round_run": bool(
                        group[
                            "round"
                        ].max()
                        >= 30
                    ),
                }
            )

else:
    cadr_availability_rows.append(
        {
            "file": CADR_PARTIAL_LOG.name,
            "usable": False,
            "reason": "File not found",
        }
    )

cadr_log_availability = pd.DataFrame(
    cadr_availability_rows
)

cadr_log_availability.to_csv(
    OUTPUTS / "run4_cadr_log_availability.csv",
    index=False,
)

print(
    cadr_log_availability.to_string(
        index=False,
        float_format=lambda value: f"{value:.6f}",
    )
)


# ===========================================================================
# PART G: DIAGNOSTIC FIGURES
# ===========================================================================
print("\n" + "=" * 96)
print("PART G: DIAGNOSTIC FIGURES")
print("=" * 96)

# ---------------------------------------------------------------------------
# Figure 1: selected checkpoint rounds
# ---------------------------------------------------------------------------
method_order = [
    "FedAvg",
    "FedProx",
    "LPR",
    "DWFA",
]

figure, axis = plt.subplots(
    figsize=(8.5, 5.3)
)

for method_index, method in enumerate(
    method_order
):
    subset = checkpoint_selection[
        checkpoint_selection["method"]
        == method
    ].sort_values("seed")

    if len(subset) == 0:
        continue

    offsets = np.linspace(
        -0.10,
        0.10,
        len(subset),
    )

    axis.scatter(
        np.full(len(subset), method_index)
        + offsets,
        subset["selected_round"],
        s=55,
    )

axis.set_xticks(
    range(len(method_order))
)

axis.set_xticklabels(
    method_order
)

axis.set_ylabel(
    "Selected communication round"
)

axis.set_xlabel(
    "Aggregation procedure"
)

axis.set_ylim(
    0,
    max(
        5,
        checkpoint_selection[
            "selected_round"
        ].max() + 1,
    ),
)

axis.grid(
    axis="y",
    alpha=0.25,
)

figure.tight_layout()

figure.savefig(
    OUTPUTS
    / "run4_selected_checkpoint_rounds.pdf",
    bbox_inches="tight",
)

figure.savefig(
    OUTPUTS
    / "run4_selected_checkpoint_rounds.png",
    dpi=300,
    bbox_inches="tight",
)

plt.close(figure)


# ---------------------------------------------------------------------------
# Figure 2: DWFA weight/prior ratio trajectories
# ---------------------------------------------------------------------------
figure, axes = plt.subplots(
    2,
    2,
    figsize=(11.0, 7.8),
    sharex=True,
)

for axis, client in zip(
    axes.flat,
    CLIENT_ORDER,
):
    subset = dwfa[
        dwfa["client"] == client
    ]

    for seed, group in subset.groupby(
        "seed"
    ):
        group = group.sort_values(
            "round"
        )

        axis.plot(
            group["round"],
            group[
                "weight_ratio_to_prior"
            ],
            linewidth=1,
            alpha=0.65,
        )

        selected_round = int(
            group[
                "selected_round"
            ].iloc[0]
        )

        selected_row = group[
            group["round"]
            == selected_round
        ]

        axis.scatter(
            selected_row["round"],
            selected_row[
                "weight_ratio_to_prior"
            ],
            s=28,
            marker="D",
        )

    mean_trajectory = (
        subset.groupby("round")[
            "weight_ratio_to_prior"
        ]
        .mean()
    )

    axis.plot(
        mean_trajectory.index,
        mean_trajectory.values,
        linewidth=2.2,
    )

    axis.axhline(
        1.0,
        linestyle="--",
        linewidth=1,
    )

    axis.set_title(
        f"Client {client}"
    )

    axis.set_ylabel(
        "DWFA weight / FedAvg prior"
    )

    axis.grid(
        alpha=0.20,
    )

for axis in axes[-1, :]:
    axis.set_xlabel(
        "Communication round"
    )

figure.tight_layout()

figure.savefig(
    OUTPUTS
    / "run4_dwfa_weight_ratio_trajectories.pdf",
    bbox_inches="tight",
)

figure.savefig(
    OUTPUTS
    / "run4_dwfa_weight_ratio_trajectories.png",
    dpi=300,
    bbox_inches="tight",
)

plt.close(figure)


# ---------------------------------------------------------------------------
# Figure 3: model-forming versus post-selection weight change
# ---------------------------------------------------------------------------
stage_plot = weight_summary[
    weight_summary[
        "analysis_period"
    ].isin(
        [
            "model_forming_rounds",
            "post_selection_unused_rounds",
        ]
    )
].copy()

figure, axis = plt.subplots(
    figsize=(8.7, 5.5)
)

periods = [
    "model_forming_rounds",
    "post_selection_unused_rounds",
]

period_offsets = {
    "model_forming_rounds": -0.12,
    "post_selection_unused_rounds": 0.12,
}

period_labels = {
    "model_forming_rounds": (
        "Rounds forming selected checkpoint"
    ),
    "post_selection_unused_rounds": (
        "Later unused rounds"
    ),
}

for period in periods:
    subset = stage_plot[
        stage_plot["analysis_period"]
        == period
    ].sort_values("client")

    x_positions = np.arange(
        len(subset)
    ) + period_offsets[period]

    axis.scatter(
        x_positions,
        subset[
            "mean_weight_ratio_to_prior"
        ],
        s=65,
        label=period_labels[period],
    )

axis.axhline(
    1.0,
    linestyle="--",
    linewidth=1,
)

axis.set_xticks(
    range(len(CLIENT_ORDER))
)

axis.set_xticklabels(
    CLIENT_ORDER
)

axis.set_xlabel("Simulated client")

axis.set_ylabel(
    "Mean DWFA weight / FedAvg prior"
)

axis.grid(
    axis="y",
    alpha=0.25,
)

axis.legend(
    frameon=False
)

figure.tight_layout()

figure.savefig(
    OUTPUTS
    / "run4_dwfa_model_forming_vs_late_weights.pdf",
    bbox_inches="tight",
)

figure.savefig(
    OUTPUTS
    / "run4_dwfa_model_forming_vs_late_weights.png",
    dpi=300,
    bbox_inches="tight",
)

plt.close(figure)


# ---------------------------------------------------------------------------
# Figure 4: trajectory-level correlations
# ---------------------------------------------------------------------------
trajectory_correlations["trajectory"] = (
    "S"
    + trajectory_correlations[
        "seed"
    ].astype(str)
    + "-"
    + trajectory_correlations[
        "client"
    ]
)

trajectory_correlations = (
    trajectory_correlations.sort_values(
        [
            "seed",
            "client",
        ]
    )
)

figure, axis = plt.subplots(
    figsize=(10.2, 5.8)
)

x_positions = np.arange(
    len(
        trajectory_correlations
    )
)

axis.scatter(
    x_positions - 0.10,
    trajectory_correlations[
        "pearson_r"
    ],
    s=45,
    label="Pearson correlation",
)

axis.scatter(
    x_positions + 0.10,
    trajectory_correlations[
        "spearman_rho"
    ],
    s=45,
    marker="s",
    label="Spearman correlation",
)

axis.axhline(
    0,
    linestyle="--",
    linewidth=1,
)

axis.set_xticks(
    x_positions
)

axis.set_xticklabels(
    trajectory_correlations[
        "trajectory"
    ],
    rotation=60,
    ha="right",
)

axis.set_ylabel(
    "Correlation between Δ local gap and Δ weight"
)

axis.set_xlabel(
    "Seed–client trajectory"
)

axis.grid(
    axis="y",
    alpha=0.25,
)

axis.legend(
    frameon=False
)

figure.tight_layout()

figure.savefig(
    OUTPUTS
    / "run4_dwfa_trajectory_correlations.pdf",
    bbox_inches="tight",
)

figure.savefig(
    OUTPUTS
    / "run4_dwfa_trajectory_correlations.png",
    dpi=300,
    bbox_inches="tight",
)

plt.close(figure)


# ===========================================================================
# FINAL DECISION SUMMARY
# ===========================================================================
print("\n" + "=" * 96)
print("RUN 4 DECISION SUMMARY")
print("=" * 96)

all_logged_claim = weight_summary[
    weight_summary["analysis_period"]
    == "all_30_logged_rounds"
][
    [
        "client",
        "mean_percent_change_from_prior",
    ]
]

model_forming_claim = weight_summary[
    weight_summary["analysis_period"]
    == "model_forming_rounds"
][
    [
        "client",
        "mean_percent_change_from_prior",
    ]
]

selected_claim = weight_summary[
    weight_summary["analysis_period"]
    == "selected_checkpoint_round_only"
][
    [
        "client",
        "mean_percent_change_from_prior",
    ]
]

print("Checkpoint rounds:")
print(
    checkpoint_summary[
        [
            "method",
            "n_seeds",
            "minimum_selected_round",
            "median_selected_round",
            "maximum_selected_round",
            "n_within_first_3",
        ]
    ].to_string(index=False)
)

print("\nAll-30-round DWFA weight change from FedAvg prior:")
print(
    all_logged_claim.to_string(
        index=False,
        float_format=lambda value: f"{value:+.2f}%",
    )
)

print("\nRounds that actually formed the selected DWFA checkpoint:")
print(
    model_forming_claim.to_string(
        index=False,
        float_format=lambda value: f"{value:+.2f}%",
    )
)

print("\nSelected checkpoint round only:")
print(
    selected_claim.to_string(
        index=False,
        float_format=lambda value: f"{value:+.2f}%",
    )
)

print(
    "\nBound violations:",
    int(
        dwfa[
            "lower_bound_violation"
        ].sum()
        + dwfa[
            "upper_bound_violation"
        ].sum()
    ),
)

print(
    "Maximum reconstructed-weight error:",
    f"{dwfa['weight_absolute_error'].max():.3e}",
)

print(
    "Pooled descriptive Pearson correlation:",
    f"{pooled_correlation['pearson_r'].iloc[0]:+.4f}",
)

print(
    "Pooled descriptive Spearman correlation:",
    f"{pooled_correlation['spearman_rho'].iloc[0]:+.4f}",
)

print(
    "No independence-based correlation p-value was calculated."
)


# ---------------------------------------------------------------------------
# Save metadata and all audited rows
# ---------------------------------------------------------------------------
dwfa.to_csv(
    OUTPUTS / "run4_dwfa_audited_rows.csv",
    index=False,
)

metadata = {
    "dwfa_constants": {
        "rho": RHO,
        "alpha": ALPHA,
        "kappa": KAPPA,
        "g_ref": G_REF,
        "q_min": Q_MIN,
    },
    "checkpoint_metric": (
        "Highest pooled validation AUROC; "
        "earliest round retained on ties"
    ),
    "model_forming_definition": (
        "All communication rounds from round 1 through "
        "the selected checkpoint round, inclusive"
    ),
    "post_selection_definition": (
        "Logged rounds after the selected checkpoint; "
        "these rounds did not form the reported checkpoint"
    ),
    "correlation_policy": (
        "Descriptive only; no pooled independence-based p-value"
    ),
    "cadr_policy": (
        "Only retained round logs are summarized; no claims are "
        "made about unlogged seeds or the published FairFed method"
    ),
    "bug_fix": (
        "summarize_numeric now handles an empty grouping list by "
        "treating the complete DataFrame as one overall group"
    ),
}

with open(
    OUTPUTS / "run4_metadata.json",
    "w",
    encoding="utf-8",
) as file:
    json.dump(
        metadata,
        file,
        indent=2,
    )

local_zip = Path(
    "/content/revision_run4.zip"
)

with zipfile.ZipFile(
    local_zip,
    "w",
    compression=zipfile.ZIP_DEFLATED,
) as archive:
    for path in sorted(
        OUTPUTS.rglob("*")
    ):
        if path.is_file():
            archive.write(
                path,
                arcname=path.relative_to(
                    OUTPUTS
                ),
            )

drive_zip = (
    RESULTS / "revision_run4.zip"
)

shutil.copy2(
    local_zip,
    drive_zip,
)

print("\n" + "=" * 96)
print("RUN 4 COMPLETE")
print("Local ZIP:", local_zip)
print("Drive ZIP:", drive_zip)
print("=" * 96)

Mounted at /content/drive
RUN 4: CHECKPOINT SELECTION, DWFA WEIGHTS, AND DEPENDENCE AUDIT

PART A: CHECKPOINT-SELECTION AUDIT

Selected checkpoints reconstructed from retained logs:
 method  seed  n_logged_rounds  first_logged_round  final_round  selected_round  selected_validation_auroc  final_validation_auroc  selected_minus_final_auroc  selected_within_first_3_rounds
   DWFA    42               30                   1           30               2                   0.952106                0.944329                    0.007777                            True
   DWFA   123               30                   1           30               4                   0.953496                0.948236                    0.005260                           False
   DWFA   456               30                   1           30               2                   0.953895                0.948914                    0.004980                            True
   DWFA   789               30                   1    

In [ ]:
# ============================================================================
# RUN 5: CALIBRATION, BRIER INFERENCE, AND THRESHOLD ROBUSTNESS
#
# No training.
# No checkpoint inference.
# L4 GPU recommended for patient-clustered Brier calculations.
#
# Output:
#   /content/drive/MyDrive/FairFedCXR/results/revision_run5.zip
# ============================================================================

from google.colab import drive
drive.mount("/content/drive", force_remount=True)

from pathlib import Path
from itertools import combinations, product
import json
import shutil
import zipfile

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from scipy.special import expit
from sklearn.metrics import brier_score_loss


# ---------------------------------------------------------------------------
# Configuration
# ---------------------------------------------------------------------------
BASE = Path("/content/drive/MyDrive/FairFedCXR")
RESULTS = BASE / "results"

LOCAL = Path("/content/revision_run5")
INPUTS = LOCAL / "inputs"
OUTPUTS = LOCAL / "outputs"

if LOCAL.exists():
    shutil.rmtree(LOCAL)

INPUTS.mkdir(parents=True, exist_ok=True)
OUTPUTS.mkdir(parents=True, exist_ok=True)

NIH_CSV = RESULTS / "nih_effusion_cohort.csv"
MAIN_CACHE = RESULTS / "nih_pred_cache.npz"

for path in [NIH_CSV, MAIN_CACHE]:
    if not path.exists():
        raise FileNotFoundError(path)

N_BOOTSTRAP = 2000
BOOTSTRAP_SEED = 20260725

ECE_BINS = 8
RELIABILITY_BINS = 10

THRESHOLDS = np.round(
    np.arange(0.05, 0.951, 0.025),
    3,
)

METHOD_LABELS = {
    "fedavg": "FedAvg",
    "qfedavg": "LPR",
    "fairfed": "CADR",
    "dwfa": "DWFA",
}

METHOD_ORDER = [
    "FedAvg",
    "LPR",
    "CADR",
    "DWFA",
]

CORE_METHODS = [
    "LPR",
    "CADR",
    "DWFA",
]

COMPARISONS = [
    ("DWFA", "LPR"),
    ("DWFA", "CADR"),
    ("LPR", "CADR"),
]

BRIER_METRICS = [
    "brier_score",
    "brier_gap",
]

print("=" * 96)
print("RUN 5: CALIBRATION, BRIER INFERENCE, AND THRESHOLD ROBUSTNESS")
print("=" * 96)


# ---------------------------------------------------------------------------
# Copy inputs locally
# ---------------------------------------------------------------------------
local_nih = INPUTS / NIH_CSV.name
local_cache = INPUTS / MAIN_CACHE.name

shutil.copy2(NIH_CSV, local_nih)
shutil.copy2(MAIN_CACHE, local_cache)


# ---------------------------------------------------------------------------
# Load and validate data
# ---------------------------------------------------------------------------
nih = pd.read_csv(
    local_nih,
    low_memory=False,
)

required_columns = {
    "Patient ID",
    "label",
    "sex_encoded",
}

missing = required_columns - set(nih.columns)

if missing:
    raise KeyError(
        f"NIH CSV missing columns: {sorted(missing)}"
    )

nih = nih.copy()

nih["patient_id"] = (
    nih["Patient ID"]
    .astype(str)
    .str.strip()
)

nih["label"] = (
    nih["label"]
    .astype(np.int8)
)

nih["sex_encoded"] = (
    nih["sex_encoded"]
    .astype(np.int8)
)

patient_code, patient_levels = pd.factorize(
    nih["patient_id"],
    sort=True,
)

patient_code = patient_code.astype(np.int32)

labels = nih["label"].to_numpy(np.int8)
sex = nih["sex_encoded"].to_numpy(np.int8)

n_images = len(nih)
n_patients = len(patient_levels)

cache = np.load(
    local_cache,
    allow_pickle=True,
)

predictions = cache["all_probs"].item()

assert np.array_equal(
    cache["labels"].astype(np.int8),
    labels,
)

assert np.array_equal(
    cache["sex"].astype(np.int8),
    sex,
)

for key, values in predictions.items():
    values = np.asarray(values).reshape(-1)

    assert len(values) == n_images
    assert np.isfinite(values).all()
    assert values.min() >= 0
    assert values.max() <= 1

print(f"Images:                  {n_images:,}")
print(f"Unique patients:         {n_patients:,}")
print(f"Prediction arrays:       {len(predictions):,}")
print(f"Threshold grid points:   {len(THRESHOLDS):,}")


# ---------------------------------------------------------------------------
# Organize individual training runs
# ---------------------------------------------------------------------------
model_probabilities = {}
method_model_ids = {
    method: []
    for method in METHOD_ORDER
}

for (
    cache_method,
    seed,
), probabilities in sorted(
    predictions.items(),
    key=lambda item: (
        item[0][0],
        int(item[0][1]),
    ),
):
    method = METHOD_LABELS[cache_method]
    seed = int(seed)

    model_id = f"{method}|{seed}"

    model_probabilities[model_id] = np.asarray(
        probabilities,
        dtype=np.float64,
    )

    method_model_ids[method].append(
        model_id
    )

for method in method_model_ids:
    method_model_ids[method] = sorted(
        method_model_ids[method],
        key=lambda model_id: int(
            model_id.split("|")[1]
        ),
    )

print("\nAvailable runs:")

for method, model_ids in method_model_ids.items():
    print(method, model_ids)


# ---------------------------------------------------------------------------
# Utility functions
# ---------------------------------------------------------------------------
def expected_calibration_error(
    probabilities,
    outcomes,
    n_bins=ECE_BINS,
):
    probabilities = np.asarray(
        probabilities,
        dtype=float,
    )

    outcomes = np.asarray(
        outcomes,
        dtype=float,
    )

    boundaries = np.linspace(
        0.0,
        1.0,
        n_bins + 1,
    )

    total = len(outcomes)
    result = 0.0

    for bin_index in range(n_bins):
        lower = boundaries[bin_index]
        upper = boundaries[bin_index + 1]

        if bin_index == n_bins - 1:
            selected = (
                (probabilities >= lower)
                & (probabilities <= upper)
            )
        else:
            selected = (
                (probabilities >= lower)
                & (probabilities < upper)
            )

        count = int(selected.sum())

        if count == 0:
            continue

        observed_frequency = float(
            outcomes[selected].mean()
        )

        mean_probability = float(
            probabilities[selected].mean()
        )

        result += (
            count / total
        ) * abs(
            observed_frequency
            - mean_probability
        )

    return float(result)


def calibration_intercept_slope(
    probabilities,
    outcomes,
    maximum_iterations=100,
    tolerance=1e-9,
):
    """
    Logistic calibration model:

        logit(P(Y=1)) = intercept + slope * logit(predicted_probability)

    Ideal calibration:
        intercept = 0
        slope = 1

    Fitted by Newton–Raphson with a very small ridge term for numerical
    stability.
    """
    probabilities = np.asarray(
        probabilities,
        dtype=np.float64,
    )

    outcomes = np.asarray(
        outcomes,
        dtype=np.float64,
    )

    if (
        len(outcomes) < 10
        or np.unique(outcomes).size < 2
    ):
        return np.nan, np.nan, False, 0

    clipped = np.clip(
        probabilities,
        1e-6,
        1.0 - 1e-6,
    )

    logits = np.log(
        clipped / (1.0 - clipped)
    )

    design = np.column_stack(
        [
            np.ones(len(logits)),
            logits,
        ]
    )

    coefficients = np.array(
        [0.0, 1.0],
        dtype=np.float64,
    )

    ridge = np.eye(2) * 1e-8
    converged = False

    for iteration in range(
        1,
        maximum_iterations + 1,
    ):
        linear_predictor = (
            design @ coefficients
        )

        fitted = expit(
            linear_predictor
        )

        weights = np.clip(
            fitted * (1.0 - fitted),
            1e-9,
            None,
        )

        gradient = (
            design.T
            @ (outcomes - fitted)
        )

        information = (
            design.T
            @ (
                design
                * weights[:, None]
            )
            + ridge
        )

        try:
            step = np.linalg.solve(
                information,
                gradient,
            )
        except np.linalg.LinAlgError:
            return np.nan, np.nan, False, iteration

        new_coefficients = (
            coefficients + step
        )

        if np.max(
            np.abs(
                new_coefficients
                - coefficients
            )
        ) < tolerance:
            coefficients = new_coefficients
            converged = True
            break

        coefficients = new_coefficients

    return (
        float(coefficients[0]),
        float(coefficients[1]),
        bool(converged),
        int(iteration),
    )


def reliability_curve_fixed_bins(
    probabilities,
    outcomes,
    n_bins=RELIABILITY_BINS,
):
    probabilities = np.asarray(
        probabilities,
        dtype=float,
    )

    outcomes = np.asarray(
        outcomes,
        dtype=float,
    )

    boundaries = np.linspace(
        0.0,
        1.0,
        n_bins + 1,
    )

    rows = []

    for bin_index in range(n_bins):
        lower = boundaries[bin_index]
        upper = boundaries[bin_index + 1]

        if bin_index == n_bins - 1:
            selected = (
                (probabilities >= lower)
                & (probabilities <= upper)
            )
        else:
            selected = (
                (probabilities >= lower)
                & (probabilities < upper)
            )

        count = int(
            selected.sum()
        )

        rows.append(
            {
                "bin_index": bin_index + 1,
                "bin_lower": lower,
                "bin_upper": upper,
                "n": count,
                "mean_probability": (
                    float(
                        probabilities[
                            selected
                        ].mean()
                    )
                    if count
                    else np.nan
                ),
                "observed_frequency": (
                    float(
                        outcomes[
                            selected
                        ].mean()
                    )
                    if count
                    else np.nan
                ),
            }
        )

    return pd.DataFrame(rows)


def smoothed_rate(
    numerator,
    denominator,
):
    return float(
        (numerator + 1.0)
        / (denominator + 2.0)
    )


def threshold_statistics(
    probabilities,
    threshold,
):
    predicted_positive = (
        probabilities >= threshold
    )

    rows = {}

    for group_name, group_value in [
        ("male", 1),
        ("female", 0),
    ]:
        group = sex == group_value
        positive = group & (labels == 1)
        negative = group & (labels == 0)

        true_positive = int(
            (
                predicted_positive
                & positive
            ).sum()
        )

        false_negative = int(
            (
                (~predicted_positive)
                & positive
            ).sum()
        )

        false_positive = int(
            (
                predicted_positive
                & negative
            ).sum()
        )

        true_negative = int(
            (
                (~predicted_positive)
                & negative
            ).sum()
        )

        n_positive = (
            true_positive
            + false_negative
        )

        n_negative = (
            false_positive
            + true_negative
        )

        rows[
            f"{group_name}_tpr"
        ] = smoothed_rate(
            true_positive,
            n_positive,
        )

        rows[
            f"{group_name}_fpr"
        ] = smoothed_rate(
            false_positive,
            n_negative,
        )

        rows[
            f"{group_name}_selection_rate"
        ] = float(
            predicted_positive[
                group
            ].mean()
        )

    rows["eo_gap"] = abs(
        rows["male_tpr"]
        - rows["female_tpr"]
    )

    rows["fpr_gap"] = abs(
        rows["male_fpr"]
        - rows["female_fpr"]
    )

    rows["selection_rate_gap"] = abs(
        rows[
            "male_selection_rate"
        ]
        - rows[
            "female_selection_rate"
        ]
    )

    rows["overall_sensitivity"] = smoothed_rate(
        int(
            (
                predicted_positive
                & (labels == 1)
            ).sum()
        ),
        int(
            (labels == 1).sum()
        ),
    )

    rows["overall_fpr"] = smoothed_rate(
        int(
            (
                predicted_positive
                & (labels == 0)
            ).sum()
        ),
        int(
            (labels == 0).sum()
        ),
    )

    rows["overall_specificity"] = (
        1.0
        - rows["overall_fpr"]
    )

    return rows


def percentile_interval(values):
    lower, upper = np.percentile(
        values,
        [2.5, 97.5],
    )

    return float(lower), float(upper)


def holm_adjust(p_values):
    p_values = np.asarray(
        p_values,
        dtype=float,
    )

    order = np.argsort(
        p_values
    )

    adjusted = np.empty_like(
        p_values
    )

    running_maximum = 0.0
    number = len(p_values)

    for rank, original_index in enumerate(
        order
    ):
        candidate = min(
            1.0,
            (number - rank)
            * p_values[
                original_index
            ],
        )

        running_maximum = max(
            running_maximum,
            candidate,
        )

        adjusted[
            original_index
        ] = running_maximum

    return adjusted


def exact_independent_permutation_p(
    values_a,
    values_b,
):
    values_a = np.asarray(
        values_a,
        dtype=float,
    )

    values_b = np.asarray(
        values_b,
        dtype=float,
    )

    combined = np.r_[
        values_a,
        values_b,
    ]

    n_a = len(values_a)
    all_indices = np.arange(
        len(combined)
    )

    observed = (
        values_a.mean()
        - values_b.mean()
    )

    statistics = []

    for selected_indices in combinations(
        all_indices,
        n_a,
    ):
        selected_indices = np.asarray(
            selected_indices
        )

        remaining_indices = np.setdiff1d(
            all_indices,
            selected_indices,
            assume_unique=True,
        )

        statistics.append(
            combined[
                selected_indices
            ].mean()
            - combined[
                remaining_indices
            ].mean()
        )

    statistics = np.asarray(
        statistics
    )

    return float(
        np.mean(
            np.abs(statistics)
            >= abs(observed) - 1e-15
        )
    )


def exact_paired_signflip_p(
    differences,
):
    differences = np.asarray(
        differences,
        dtype=float,
    )

    observed = differences.mean()
    statistics = []

    for signs in product(
        [-1.0, 1.0],
        repeat=len(differences),
    ):
        statistics.append(
            np.mean(
                np.asarray(signs)
                * differences
            )
        )

    statistics = np.asarray(
        statistics
    )

    return float(
        np.mean(
            np.abs(statistics)
            >= abs(observed) - 1e-15
        )
    )


# ===========================================================================
# PART A: PER-RUN CALIBRATION METRICS
# ===========================================================================
print("\n" + "=" * 96)
print("PART A: PER-RUN CALIBRATION METRICS")
print("=" * 96)

GROUPS = {
    "all": np.ones(
        n_images,
        dtype=bool,
    ),
    "male": sex == 1,
    "female": sex == 0,
}

calibration_rows = []
reliability_frames = []

for model_id, probabilities in (
    model_probabilities.items()
):
    method, seed_text = model_id.split("|")
    seed_value = int(seed_text)

    for group_name, group_mask in (
        GROUPS.items()
    ):
        group_probabilities = (
            probabilities[group_mask]
        )

        group_outcomes = (
            labels[group_mask]
        )

        intercept, slope, converged, iterations = (
            calibration_intercept_slope(
                group_probabilities,
                group_outcomes,
            )
        )

        calibration_rows.append(
            {
                "model_id": model_id,
                "method": method,
                "seed": seed_value,
                "group": group_name,
                "n_images": int(
                    group_mask.sum()
                ),
                "n_patients": int(
                    np.unique(
                        patient_code[
                            group_mask
                        ]
                    ).size
                ),
                "prevalence": float(
                    group_outcomes.mean()
                ),
                "mean_probability": float(
                    group_probabilities.mean()
                ),
                "brier_score": float(
                    brier_score_loss(
                        group_outcomes,
                        group_probabilities,
                    )
                ),
                "ece_8bin": (
                    expected_calibration_error(
                        group_probabilities,
                        group_outcomes,
                    )
                ),
                "calibration_intercept": (
                    intercept
                ),
                "calibration_slope": slope,
                "calibration_fit_converged": (
                    converged
                ),
                "calibration_fit_iterations": (
                    iterations
                ),
            }
        )

        curve = reliability_curve_fixed_bins(
            group_probabilities,
            group_outcomes,
        )

        curve["model_id"] = model_id
        curve["method"] = method
        curve["seed"] = seed_value
        curve["group"] = group_name

        reliability_frames.append(
            curve
        )

calibration_per_run = pd.DataFrame(
    calibration_rows
)

reliability_per_run = pd.concat(
    reliability_frames,
    ignore_index=True,
)

calibration_per_run.to_csv(
    OUTPUTS
    / "run5_calibration_metrics_per_run.csv",
    index=False,
)

reliability_per_run.to_csv(
    OUTPUTS
    / "run5_reliability_bins_per_run.csv",
    index=False,
)

nonconverged = calibration_per_run[
    ~calibration_per_run[
        "calibration_fit_converged"
    ]
]

if len(nonconverged):
    print(
        "WARNING: Some calibration fits did not converge:"
    )

    print(
        nonconverged[
            [
                "model_id",
                "group",
                "calibration_fit_iterations",
            ]
        ].to_string(index=False)
    )

calibration_summary_rows = []

summary_metrics = [
    "mean_probability",
    "brier_score",
    "ece_8bin",
    "calibration_intercept",
    "calibration_slope",
]

for (
    method,
    group_name,
), group in calibration_per_run.groupby(
    ["method", "group"]
):
    for metric in summary_metrics:
        values = (
            group[metric]
            .dropna()
            .to_numpy(dtype=float)
        )

        calibration_summary_rows.append(
            {
                "method": method,
                "group": group_name,
                "metric": metric,
                "n_runs": int(
                    len(values)
                ),
                "mean": float(
                    values.mean()
                ),
                "sd": (
                    float(
                        values.std(ddof=1)
                    )
                    if len(values) > 1
                    else np.nan
                ),
                "minimum": float(
                    values.min()
                ),
                "maximum": float(
                    values.max()
                ),
            }
        )

calibration_summary = pd.DataFrame(
    calibration_summary_rows
)

calibration_summary.to_csv(
    OUTPUTS
    / "run5_calibration_summary_across_runs.csv",
    index=False,
)

print("\nOverall calibration summary:")

print(
    calibration_summary[
        (
            calibration_summary["group"]
            == "all"
        )
        & calibration_summary[
            "metric"
        ].isin(
            [
                "brier_score",
                "ece_8bin",
                "calibration_intercept",
                "calibration_slope",
            ]
        )
    ].to_string(
        index=False,
        float_format=lambda value: (
            f"{value:.6f}"
        ),
    )
)


# ===========================================================================
# PART B: PATIENT-CLUSTERED BRIER BOOTSTRAP
# ===========================================================================
print("\n" + "=" * 96)
print("PART B: PATIENT-CLUSTERED BRIER INFERENCE")
print("=" * 96)

rng_counts = np.random.default_rng(
    BOOTSTRAP_SEED
)

patient_counts = np.empty(
    (
        N_BOOTSTRAP,
        n_patients,
    ),
    dtype=np.uint16,
)

for bootstrap_index in range(
    N_BOOTSTRAP
):
    sampled_patients = rng_counts.integers(
        0,
        n_patients,
        size=n_patients,
        dtype=np.int32,
    )

    patient_counts[
        bootstrap_index
    ] = np.bincount(
        sampled_patients,
        minlength=n_patients,
    ).astype(np.uint16)

print(
    "Patient bootstrap matrix:",
    f"{patient_counts.nbytes / 1024**2:.1f} MB",
)

USE_GPU = False

try:
    import cupy as cp

    if cp.cuda.runtime.getDeviceCount() > 0:
        USE_GPU = True
        print("Brier backend: CuPy GPU")

except Exception as error:
    print(
        "CuPy unavailable; NumPy CPU fallback:",
        repr(error),
    )


def matrix_multiply_counts(
    counts,
    statistics,
):
    if USE_GPU:
        counts_gpu = cp.asarray(
            counts,
            dtype=cp.float32,
        )

        statistics_gpu = cp.asarray(
            statistics,
            dtype=cp.float32,
        )

        result = cp.asnumpy(
            counts_gpu
            @ statistics_gpu
        )

        del counts_gpu
        del statistics_gpu

        cp.get_default_memory_pool().free_all_blocks()

        return result.astype(
            np.float64
        )

    return (
        counts.astype(np.float64)
        @ statistics.astype(np.float64)
    )


patient_image_count = np.bincount(
    patient_code,
    minlength=n_patients,
).astype(np.float64)

male_mask = sex == 1
female_mask = sex == 0

patient_male_image_count = np.bincount(
    patient_code[male_mask],
    minlength=n_patients,
).astype(np.float64)

patient_female_image_count = np.bincount(
    patient_code[female_mask],
    minlength=n_patients,
).astype(np.float64)

brier_bootstrap = {}
brier_full_rows = []

for model_index, (
    model_id,
    probabilities,
) in enumerate(
    model_probabilities.items(),
    start=1,
):
    print(
        f"[{model_index:02d}/"
        f"{len(model_probabilities)}] "
        f"{model_id}"
    )

    squared_error = (
        probabilities
        - labels
    ) ** 2

    patient_sse = np.bincount(
        patient_code,
        weights=squared_error,
        minlength=n_patients,
    ).astype(np.float64)

    patient_male_sse = np.bincount(
        patient_code[male_mask],
        weights=squared_error[
            male_mask
        ],
        minlength=n_patients,
    ).astype(np.float64)

    patient_female_sse = np.bincount(
        patient_code[female_mask],
        weights=squared_error[
            female_mask
        ],
        minlength=n_patients,
    ).astype(np.float64)

    patient_statistics = np.column_stack(
        [
            patient_sse,
            patient_image_count,
            patient_male_sse,
            patient_male_image_count,
            patient_female_sse,
            patient_female_image_count,
        ]
    )

    bootstrap_totals = matrix_multiply_counts(
        patient_counts,
        patient_statistics,
    )

    overall_brier = (
        bootstrap_totals[:, 0]
        / bootstrap_totals[:, 1]
    )

    male_brier = (
        bootstrap_totals[:, 2]
        / bootstrap_totals[:, 3]
    )

    female_brier = (
        bootstrap_totals[:, 4]
        / bootstrap_totals[:, 5]
    )

    brier_bootstrap[model_id] = {
        "brier_score": overall_brier,
        "male_brier": male_brier,
        "female_brier": female_brier,
        "brier_gap": np.abs(
            male_brier
            - female_brier
        ),
    }

    full_overall = float(
        squared_error.mean()
    )

    full_male = float(
        squared_error[
            male_mask
        ].mean()
    )

    full_female = float(
        squared_error[
            female_mask
        ].mean()
    )

    brier_full_rows.append(
        {
            "model_id": model_id,
            "method": model_id.split("|")[0],
            "seed": int(
                model_id.split("|")[1]
            ),
            "brier_score": full_overall,
            "male_brier": full_male,
            "female_brier": full_female,
            "brier_gap": abs(
                full_male
                - full_female
            ),
        }
    )

brier_full = pd.DataFrame(
    brier_full_rows
)

brier_full.to_csv(
    OUTPUTS
    / "run5_brier_full_sample_per_run.csv",
    index=False,
)


# ---------------------------------------------------------------------------
# Run-aware method-level Brier distributions
# ---------------------------------------------------------------------------
rng_runs = np.random.default_rng(
    BOOTSTRAP_SEED + 1
)

run_indices = {
    method: rng_runs.integers(
        0,
        5,
        size=(
            N_BOOTSTRAP,
            5,
        ),
    )
    for method in CORE_METHODS
}

method_observed = {
    method: {}
    for method in CORE_METHODS
}

method_bootstrap = {
    method: {}
    for method in CORE_METHODS
}

for method in CORE_METHODS:
    model_ids = method_model_ids[
        method
    ]

    if len(model_ids) != 5:
        raise RuntimeError(
            f"{method}: expected five runs."
        )

    for metric in BRIER_METRICS:
        matrix = np.column_stack(
            [
                brier_bootstrap[
                    model_id
                ][metric]
                for model_id in model_ids
            ]
        )

        full_values = np.asarray(
            [
                brier_full.loc[
                    brier_full[
                        "model_id"
                    ] == model_id,
                    metric,
                ].iloc[0]
                for model_id in model_ids
            ],
            dtype=float,
        )

        method_observed[
            method
        ][metric] = float(
            full_values.mean()
        )

        method_bootstrap[
            method
        ][metric] = np.take_along_axis(
            matrix,
            run_indices[method],
            axis=1,
        ).mean(axis=1)


# ---------------------------------------------------------------------------
# Simultaneous familywise intervals:
# 3 comparisons × 2 Brier metrics
# ---------------------------------------------------------------------------
identities = []
observed_differences = []
bootstrap_columns = []

for method_a, method_b in COMPARISONS:
    for metric in BRIER_METRICS:
        identities.append(
            (
                method_a,
                method_b,
                metric,
            )
        )

        observed_differences.append(
            method_observed[
                method_a
            ][metric]
            - method_observed[
                method_b
            ][metric]
        )

        bootstrap_columns.append(
            method_bootstrap[
                method_a
            ][metric]
            - method_bootstrap[
                method_b
            ][metric]
        )

observed_differences = np.asarray(
    observed_differences,
    dtype=float,
)

bootstrap_matrix = np.column_stack(
    bootstrap_columns
)

bootstrap_mean = bootstrap_matrix.mean(
    axis=0
)

bootstrap_sd = bootstrap_matrix.std(
    axis=0,
    ddof=1,
)

standardized_centered = (
    bootstrap_matrix
    - bootstrap_mean
) / bootstrap_sd

maximum_absolute = np.max(
    np.abs(
        standardized_centered
    ),
    axis=1,
)

critical_value = float(
    np.quantile(
        maximum_absolute,
        0.95,
    )
)

brier_inference_rows = []

for index, (
    method_a,
    method_b,
    metric,
) in enumerate(identities):
    pointwise_low, pointwise_high = (
        percentile_interval(
            bootstrap_matrix[:, index]
        )
    )

    brier_inference_rows.append(
        {
            "comparison": (
                f"{method_a} minus "
                f"{method_b}"
            ),
            "metric": metric,
            "estimate": (
                observed_differences[
                    index
                ]
            ),
            "pointwise_ci_low": (
                pointwise_low
            ),
            "pointwise_ci_high": (
                pointwise_high
            ),
            "simultaneous_ci_low": (
                observed_differences[
                    index
                ]
                - critical_value
                * bootstrap_sd[index]
            ),
            "simultaneous_ci_high": (
                observed_differences[
                    index
                ]
                + critical_value
                * bootstrap_sd[index]
            ),
            "bootstrap_sd": (
                bootstrap_sd[index]
            ),
            "familywise_critical_value": (
                critical_value
            ),
            "n_runs_each": 5,
            "n_patient_bootstrap": (
                N_BOOTSTRAP
            ),
        }
    )

brier_inference = pd.DataFrame(
    brier_inference_rows
)


# ---------------------------------------------------------------------------
# Exact run-level Brier tests
# ---------------------------------------------------------------------------
exact_rows = []

for method_a, method_b in COMPARISONS:
    for metric in BRIER_METRICS:
        values_a = (
            brier_full[
                brier_full["method"]
                == method_a
            ]
            .sort_values("seed")[
                metric
            ]
            .to_numpy(dtype=float)
        )

        values_b = (
            brier_full[
                brier_full["method"]
                == method_b
            ]
            .sort_values("seed")[
                metric
            ]
            .to_numpy(dtype=float)
        )

        exact_rows.append(
            {
                "comparison": (
                    f"{method_a} minus "
                    f"{method_b}"
                ),
                "metric": metric,
                "independent_exact_p": (
                    exact_independent_permutation_p(
                        values_a,
                        values_b,
                    )
                ),
                "paired_signflip_p": (
                    exact_paired_signflip_p(
                        values_a
                        - values_b
                    )
                ),
            }
        )

exact_tests = pd.DataFrame(
    exact_rows
)

exact_tests[
    "independent_holm_p"
] = holm_adjust(
    exact_tests[
        "independent_exact_p"
    ]
)

exact_tests[
    "paired_holm_p"
] = holm_adjust(
    exact_tests[
        "paired_signflip_p"
    ]
)

brier_inference = brier_inference.merge(
    exact_tests,
    on=[
        "comparison",
        "metric",
    ],
    how="left",
)

brier_inference.to_csv(
    OUTPUTS
    / "run5_brier_run_aware_inference.csv",
    index=False,
)

print("\nRun-aware Brier comparisons:")

print(
    brier_inference.to_string(
        index=False,
        float_format=lambda value: (
            f"{value:+.6f}"
        ),
    )
)


# ===========================================================================
# PART C: THRESHOLD ROBUSTNESS
# ===========================================================================
print("\n" + "=" * 96)
print("PART C: THRESHOLD ROBUSTNESS")
print("=" * 96)

threshold_rows = []

for model_index, (
    model_id,
    probabilities,
) in enumerate(
    model_probabilities.items(),
    start=1,
):
    method, seed_text = model_id.split("|")
    seed_value = int(seed_text)

    for threshold in THRESHOLDS:
        row = {
            "model_id": model_id,
            "method": method,
            "seed": seed_value,
            "threshold": float(
                threshold
            ),
        }

        row.update(
            threshold_statistics(
                probabilities,
                float(threshold),
            )
        )

        threshold_rows.append(row)

threshold_per_run = pd.DataFrame(
    threshold_rows
)

threshold_per_run.to_csv(
    OUTPUTS
    / "run5_threshold_sweep_per_run.csv",
    index=False,
)

threshold_summary_rows = []

threshold_metrics = [
    "male_tpr",
    "female_tpr",
    "eo_gap",
    "male_fpr",
    "female_fpr",
    "fpr_gap",
    "selection_rate_gap",
    "overall_sensitivity",
    "overall_specificity",
]

for (
    method,
    threshold,
), group in threshold_per_run.groupby(
    ["method", "threshold"]
):
    for metric in threshold_metrics:
        values = group[
            metric
        ].to_numpy(dtype=float)

        threshold_summary_rows.append(
            {
                "method": method,
                "threshold": float(
                    threshold
                ),
                "metric": metric,
                "n_runs": int(
                    len(values)
                ),
                "mean": float(
                    values.mean()
                ),
                "sd": (
                    float(
                        values.std(ddof=1)
                    )
                    if len(values) > 1
                    else np.nan
                ),
                "minimum": float(
                    values.min()
                ),
                "maximum": float(
                    values.max()
                ),
            }
        )

threshold_summary = pd.DataFrame(
    threshold_summary_rows
)

threshold_summary.to_csv(
    OUTPUTS
    / "run5_threshold_sweep_summary.csv",
    index=False,
)


# ---------------------------------------------------------------------------
# Threshold-dependent ranking audit
# ---------------------------------------------------------------------------
rank_rows = []

for metric in [
    "eo_gap",
    "fpr_gap",
]:
    metric_summary = threshold_summary[
        (
            threshold_summary[
                "metric"
            ] == metric
        )
        & threshold_summary[
            "method"
        ].isin(
            CORE_METHODS
        )
    ]

    for threshold, group in (
        metric_summary.groupby(
            "threshold"
        )
    ):
        ordered = group.sort_values(
            [
                "mean",
                "method",
            ]
        )

        rank_order = (
            " < ".join(
                ordered["method"]
                .astype(str)
                .tolist()
            )
        )

        best = ordered.iloc[0]

        rank_rows.append(
            {
                "metric": metric,
                "threshold": float(
                    threshold
                ),
                "lowest_gap_method": (
                    best["method"]
                ),
                "lowest_mean_gap": float(
                    best["mean"]
                ),
                "rank_order_low_to_high": (
                    rank_order
                ),
            }
        )

threshold_rankings = pd.DataFrame(
    rank_rows
)

threshold_rankings.to_csv(
    OUTPUTS
    / "run5_threshold_dependent_rankings.csv",
    index=False,
)

rank_summary_rows = []

for metric, group in threshold_rankings.groupby(
    "metric"
):
    group = group.sort_values(
        "threshold"
    )

    winner_counts = (
        group[
            "lowest_gap_method"
        ]
        .value_counts()
        .to_dict()
    )

    rank_order_changes = int(
        (
            group[
                "rank_order_low_to_high"
            ]
            != group[
                "rank_order_low_to_high"
            ].shift(1)
        ).sum()
        - 1
    )

    winner_changes = int(
        (
            group[
                "lowest_gap_method"
            ]
            != group[
                "lowest_gap_method"
            ].shift(1)
        ).sum()
        - 1
    )

    rank_summary_rows.append(
        {
            "metric": metric,
            "n_thresholds": int(
                len(group)
            ),
            "rank_order_changes": (
                rank_order_changes
            ),
            "lowest_gap_method_changes": (
                winner_changes
            ),
            "thresholds_best_LPR": int(
                winner_counts.get(
                    "LPR",
                    0,
                )
            ),
            "thresholds_best_CADR": int(
                winner_counts.get(
                    "CADR",
                    0,
                )
            ),
            "thresholds_best_DWFA": int(
                winner_counts.get(
                    "DWFA",
                    0,
                )
            ),
        }
    )

threshold_rank_summary = pd.DataFrame(
    rank_summary_rows
)

threshold_rank_summary.to_csv(
    OUTPUTS
    / "run5_threshold_ranking_summary.csv",
    index=False,
)


# ---------------------------------------------------------------------------
# Extract threshold 0.5 summary
# ---------------------------------------------------------------------------
threshold_50 = threshold_summary[
    np.isclose(
        threshold_summary[
            "threshold"
        ],
        0.5,
    )
    & threshold_summary[
        "metric"
    ].isin(
        [
            "eo_gap",
            "fpr_gap",
            "overall_sensitivity",
            "overall_specificity",
        ]
    )
].copy()

threshold_50.to_csv(
    OUTPUTS
    / "run5_threshold_050_summary.csv",
    index=False,
)

print("\nThreshold-dependent ranking summary:")

print(
    threshold_rank_summary.to_string(
        index=False
    )
)

print("\nThreshold 0.5 summary:")

print(
    threshold_50.to_string(
        index=False,
        float_format=lambda value: (
            f"{value:.6f}"
        ),
    )
)


# ===========================================================================
# PART D: DIAGNOSTIC FIGURES
# ===========================================================================
print("\n" + "=" * 96)
print("PART D: DIAGNOSTIC FIGURES")
print("=" * 96)


def save_figure(
    figure,
    stem,
):
    figure.tight_layout()

    figure.savefig(
        OUTPUTS / f"{stem}.pdf",
        bbox_inches="tight",
    )

    figure.savefig(
        OUTPUTS / f"{stem}.png",
        dpi=300,
        bbox_inches="tight",
    )

    plt.close(figure)


# ---------------------------------------------------------------------------
# Threshold robustness: one figure per fairness metric
# ---------------------------------------------------------------------------
for metric, y_label, stem in [
    (
        "eo_gap",
        "Equal-opportunity gap",
        "run5_threshold_robustness_eo_gap",
    ),
    (
        "fpr_gap",
        "False-positive-rate gap",
        "run5_threshold_robustness_fpr_gap",
    ),
]:
    figure, axis = plt.subplots(
        figsize=(8.8, 5.6)
    )

    for method in CORE_METHODS:
        subset = threshold_summary[
            (
                threshold_summary[
                    "method"
                ] == method
            )
            & (
                threshold_summary[
                    "metric"
                ] == metric
            )
        ].sort_values(
            "threshold"
        )

        line = axis.plot(
            subset["threshold"],
            subset["mean"],
            label=method,
            linewidth=2,
        )[0]

        axis.fill_between(
            subset["threshold"],
            subset["minimum"],
            subset["maximum"],
            alpha=0.15,
            color=line.get_color(),
        )

    axis.axvline(
        0.5,
        linestyle="--",
        linewidth=1,
    )

    axis.set_xlabel(
        "Decision threshold"
    )

    axis.set_ylabel(
        y_label
    )

    axis.grid(
        alpha=0.25
    )

    axis.legend(
        frameon=False
    )

    save_figure(
        figure,
        stem,
    )


# ---------------------------------------------------------------------------
# Reliability figure: one separate figure per core method
# ---------------------------------------------------------------------------
for method in CORE_METHODS:
    figure, axis = plt.subplots(
        figsize=(6.6, 5.8)
    )

    for group_name in [
        "male",
        "female",
    ]:
        subset = reliability_per_run[
            (
                reliability_per_run[
                    "method"
                ] == method
            )
            & (
                reliability_per_run[
                    "group"
                ] == group_name
            )
        ]

        summarized = (
            subset.groupby(
                "bin_index"
            )
            .agg(
                mean_predicted=(
                    "mean_probability",
                    "mean",
                ),
                mean_observed=(
                    "observed_frequency",
                    "mean",
                ),
                minimum_observed=(
                    "observed_frequency",
                    "min",
                ),
                maximum_observed=(
                    "observed_frequency",
                    "max",
                ),
                total_images=(
                    "n",
                    "sum",
                ),
            )
            .reset_index()
        )

        summarized = summarized[
            summarized[
                "total_images"
            ] > 0
        ]

        line = axis.plot(
            summarized[
                "mean_predicted"
            ],
            summarized[
                "mean_observed"
            ],
            marker="o",
            label=group_name.capitalize(),
        )[0]

        axis.fill_between(
            summarized[
                "mean_predicted"
            ],
            summarized[
                "minimum_observed"
            ],
            summarized[
                "maximum_observed"
            ],
            alpha=0.15,
            color=line.get_color(),
        )

    axis.plot(
        [0, 1],
        [0, 1],
        linestyle="--",
        linewidth=1,
        label="Ideal calibration",
    )

    axis.set_xlim(
        0,
        1,
    )

    axis.set_ylim(
        0,
        1,
    )

    axis.set_xlabel(
        "Mean predicted probability"
    )

    axis.set_ylabel(
        "Observed outcome frequency"
    )

    axis.set_title(
        f"{method}: sex-stratified calibration"
    )

    axis.grid(
        alpha=0.25
    )

    axis.legend(
        frameon=False
    )

    save_figure(
        figure,
        f"run5_calibration_{method.lower()}",
    )


# ---------------------------------------------------------------------------
# Brier forest plot
# ---------------------------------------------------------------------------
plot_data = brier_inference.copy()

plot_data["label"] = (
    plot_data["comparison"]
    + " | "
    + plot_data["metric"].replace(
        {
            "brier_score": "Overall Brier score",
            "brier_gap": "Sex-specific Brier gap",
        }
    )
)

plot_data = (
    plot_data.iloc[::-1]
    .reset_index(drop=True)
)

figure, axis = plt.subplots(
    figsize=(9.4, 6.2)
)

positions = np.arange(
    len(plot_data)
)

for row_index, row in (
    plot_data.iterrows()
):
    axis.errorbar(
        row["estimate"],
        positions[row_index],
        xerr=[
            [
                row["estimate"]
                - row[
                    "pointwise_ci_low"
                ]
            ],
            [
                row[
                    "pointwise_ci_high"
                ]
                - row["estimate"]
            ],
        ],
        fmt="o",
        capsize=4,
    )

    axis.hlines(
        positions[row_index] + 0.12,
        row[
            "simultaneous_ci_low"
        ],
        row[
            "simultaneous_ci_high"
        ],
        linewidth=2,
    )

axis.axvline(
    0,
    linestyle="--",
    linewidth=1,
)

axis.set_yticks(
    positions
)

axis.set_yticklabels(
    plot_data["label"]
)

axis.set_xlabel(
    "Difference "
    "(first method minus second method; "
    "lower favours the first method)"
)

axis.grid(
    axis="x",
    alpha=0.25,
)

save_figure(
    figure,
    "run5_brier_run_aware_forest",
)


# ===========================================================================
# FINAL DECISION SUMMARY
# ===========================================================================
print("\n" + "=" * 96)
print("RUN 5 DECISION SUMMARY")
print("=" * 96)

print("\nCalibration fit failures:")
print(
    f"{len(nonconverged)} of "
    f"{len(calibration_per_run)} fits"
)

print("\nBrier comparisons:")
print(
    brier_inference[
        [
            "comparison",
            "metric",
            "estimate",
            "pointwise_ci_low",
            "pointwise_ci_high",
            "simultaneous_ci_low",
            "simultaneous_ci_high",
            "independent_holm_p",
            "paired_holm_p",
        ]
    ].to_string(
        index=False,
        float_format=lambda value: (
            f"{value:+.6f}"
        ),
    )
)

print("\nThreshold ranking stability:")
print(
    threshold_rank_summary.to_string(
        index=False
    )
)

metadata = {
    "n_images": int(n_images),
    "n_patients": int(n_patients),
    "n_patient_bootstrap": int(
        N_BOOTSTRAP
    ),
    "bootstrap_seed": int(
        BOOTSTRAP_SEED
    ),
    "calibration_metrics": {
        "ece_bins": ECE_BINS,
        "reliability_bins": (
            RELIABILITY_BINS
        ),
        "calibration_model": (
            "logistic calibration intercept "
            "and slope fitted to logit probabilities"
        ),
    },
    "threshold_grid": (
        THRESHOLDS.tolist()
    ),
    "threshold_interpretation": (
        "Descriptive robustness analysis only. "
        "No threshold is selected using NIH outcomes."
    ),
    "brier_inference": (
        "Patient-clustered and independently "
        "run-resampled, with simultaneous max-t "
        "intervals over six comparisons."
    ),
    "gpu_used": bool(USE_GPU),
}

with open(
    OUTPUTS / "run5_metadata.json",
    "w",
    encoding="utf-8",
) as file:
    json.dump(
        metadata,
        file,
        indent=2,
    )

np.savez_compressed(
    OUTPUTS
    / "run5_patient_cluster_brier_bootstrap.npz",
    brier_bootstrap=np.array(
        brier_bootstrap,
        dtype=object,
    ),
    n_bootstrap=np.array(
        N_BOOTSTRAP
    ),
    bootstrap_seed=np.array(
        BOOTSTRAP_SEED
    ),
)

local_zip = Path(
    "/content/revision_run5.zip"
)

with zipfile.ZipFile(
    local_zip,
    "w",
    compression=zipfile.ZIP_DEFLATED,
) as archive:
    for path in sorted(
        OUTPUTS.rglob("*")
    ):
        if path.is_file():
            archive.write(
                path,
                arcname=path.relative_to(
                    OUTPUTS
                ),
            )

drive_zip = (
    RESULTS
    / "revision_run5.zip"
)

shutil.copy2(
    local_zip,
    drive_zip,
)

print("\n" + "=" * 96)
print("RUN 5 COMPLETE")
print("Local ZIP:", local_zip)
print("Drive ZIP:", drive_zip)
print("=" * 96)

Mounted at /content/drive
RUN 5: CALIBRATION, BRIER INFERENCE, AND THRESHOLD ROBUSTNESS
Images:                  112,120
Unique patients:         30,805
Prediction arrays:       18
Threshold grid points:   37

Available runs:
FedAvg ['FedAvg|42', 'FedAvg|123', 'FedAvg|456']
LPR ['LPR|42', 'LPR|123', 'LPR|456', 'LPR|789', 'LPR|1010']
CADR ['CADR|42', 'CADR|123', 'CADR|456', 'CADR|789', 'CADR|1010']
DWFA ['DWFA|42', 'DWFA|123', 'DWFA|456', 'DWFA|789', 'DWFA|1010']

PART A: PER-RUN CALIBRATION METRICS
  model_id  group  calibration_fit_iterations
   DWFA|42    all                         100
   DWFA|42   male                         100
   DWFA|42 female                         100
  DWFA|123    all                         100
  DWFA|123   male                         100
  DWFA|123 female                         100
  DWFA|456    all                         100
  DWFA|456   male                         100
  DWFA|456 female                         100
  DWFA|789    all                   

In [ ]:
# ============================================================================
# RUN 5B: ROBUST CALIBRATION INTERCEPT, SLOPE, AND RELIABILITY CURVES
#
# CPU only.
# No training.
# No checkpoint inference.
#
# Output:
#   /content/drive/MyDrive/FairFedCXR/results/revision_run5b.zip
# ============================================================================

from google.colab import drive
drive.mount("/content/drive", force_remount=True)

from pathlib import Path
import json
import shutil
import zipfile

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from scipy.optimize import minimize
from scipy.special import expit, logit
from sklearn.metrics import brier_score_loss


# ---------------------------------------------------------------------------
# Paths
# ---------------------------------------------------------------------------
BASE = Path("/content/drive/MyDrive/FairFedCXR")
RESULTS = BASE / "results"

NIH_CSV = RESULTS / "nih_effusion_cohort.csv"
PREDICTION_CACHE = RESULTS / "nih_pred_cache.npz"

LOCAL = Path("/content/revision_run5b")
INPUTS = LOCAL / "inputs"
OUTPUTS = LOCAL / "outputs"

if LOCAL.exists():
    shutil.rmtree(LOCAL)

INPUTS.mkdir(parents=True, exist_ok=True)
OUTPUTS.mkdir(parents=True, exist_ok=True)

for path in [NIH_CSV, PREDICTION_CACHE]:
    if not path.exists():
        raise FileNotFoundError(path)

local_nih = INPUTS / NIH_CSV.name
local_cache = INPUTS / PREDICTION_CACHE.name

shutil.copy2(NIH_CSV, local_nih)
shutil.copy2(PREDICTION_CACHE, local_cache)

print("=" * 94)
print("RUN 5B: ROBUST CALIBRATION ANALYSIS")
print("=" * 94)


# ---------------------------------------------------------------------------
# Load data
# ---------------------------------------------------------------------------
nih = pd.read_csv(local_nih, low_memory=False)

required_columns = {
    "Patient ID",
    "label",
    "sex_encoded",
}

missing = required_columns - set(nih.columns)

if missing:
    raise KeyError(
        f"NIH cohort missing columns: {sorted(missing)}"
    )

nih = nih.copy()
nih["patient_id"] = nih["Patient ID"].astype(str).str.strip()
nih["label"] = nih["label"].astype(np.int8)
nih["sex_encoded"] = nih["sex_encoded"].astype(np.int8)

labels = nih["label"].to_numpy(dtype=np.int8)
sex = nih["sex_encoded"].to_numpy(dtype=np.int8)

cache = np.load(local_cache, allow_pickle=True)
predictions = cache["all_probs"].item()

assert np.array_equal(
    cache["labels"].astype(np.int8),
    labels,
)

assert np.array_equal(
    cache["sex"].astype(np.int8),
    sex,
)

METHOD_LABELS = {
    "fedavg": "FedAvg",
    "qfedavg": "LPR",
    "fairfed": "CADR",
    "dwfa": "DWFA",
}

METHOD_ORDER = [
    "FedAvg",
    "LPR",
    "CADR",
    "DWFA",
]

model_probabilities = {}

for (
    cache_method,
    seed,
), probabilities in sorted(
    predictions.items(),
    key=lambda item: (
        item[0][0],
        int(item[0][1]),
    ),
):
    method = METHOD_LABELS[cache_method]
    model_id = f"{method}|{int(seed)}"

    probabilities = np.asarray(
        probabilities,
        dtype=np.float64,
    ).reshape(-1)

    assert len(probabilities) == len(labels)
    assert np.isfinite(probabilities).all()
    assert probabilities.min() >= 0
    assert probabilities.max() <= 1

    model_probabilities[model_id] = probabilities

print(f"Images:             {len(nih):,}")
print(f"Unique patients:    {nih['patient_id'].nunique():,}")
print(f"Prediction arrays:  {len(model_probabilities):,}")


# ---------------------------------------------------------------------------
# Robust logistic calibration fit
# ---------------------------------------------------------------------------
def robust_calibration_fit(
    probabilities,
    outcomes,
    probability_clip=1e-6,
):
    """
    Fits:

        logit[P(Y=1)] = intercept + slope * logit(predicted probability)

    Optimization is performed using standardized logits and a numerically
    stable Bernoulli negative log-likelihood.

    Ideal calibration:
        intercept = 0
        slope = 1
    """
    probabilities = np.asarray(
        probabilities,
        dtype=np.float64,
    )

    outcomes = np.asarray(
        outcomes,
        dtype=np.float64,
    )

    if len(outcomes) < 20:
        return {
            "intercept": np.nan,
            "slope": np.nan,
            "converged": False,
            "reason": "too_few_observations",
            "iterations": 0,
            "gradient_norm": np.nan,
            "objective": np.nan,
            "hit_boundary": False,
        }

    if np.unique(outcomes).size < 2:
        return {
            "intercept": np.nan,
            "slope": np.nan,
            "converged": False,
            "reason": "single_outcome_class",
            "iterations": 0,
            "gradient_norm": np.nan,
            "objective": np.nan,
            "hit_boundary": False,
        }

    clipped = np.clip(
        probabilities,
        probability_clip,
        1.0 - probability_clip,
    )

    original_logit = logit(clipped)

    logit_mean = float(original_logit.mean())
    logit_sd = float(original_logit.std(ddof=0))

    if not np.isfinite(logit_sd) or logit_sd <= 1e-12:
        return {
            "intercept": np.nan,
            "slope": np.nan,
            "converged": False,
            "reason": "constant_prediction_logit",
            "iterations": 0,
            "gradient_norm": np.nan,
            "objective": np.nan,
            "hit_boundary": False,
        }

    standardized_logit = (
        original_logit - logit_mean
    ) / logit_sd

    design = np.column_stack(
        [
            np.ones(len(outcomes)),
            standardized_logit,
        ]
    )

    # Very small regularizer for numerical stability only.
    ridge_strength = 1e-10

    def objective(coefficients):
        linear_predictor = design @ coefficients

        negative_log_likelihood = np.sum(
            np.logaddexp(
                0.0,
                linear_predictor,
            )
            - outcomes * linear_predictor
        )

        ridge = 0.5 * ridge_strength * np.sum(
            coefficients ** 2
        )

        return float(
            negative_log_likelihood + ridge
        )

    def gradient(coefficients):
        linear_predictor = design @ coefficients
        fitted = expit(linear_predictor)

        return (
            design.T @ (fitted - outcomes)
            + ridge_strength * coefficients
        )

    prevalence = float(outcomes.mean())

    prevalence_logit = float(
        logit(
            np.clip(
                prevalence,
                1e-6,
                1.0 - 1e-6,
            )
        )
    )

    # Three sensible initializations.
    starting_values = [
        np.array(
            [prevalence_logit, 0.0],
            dtype=np.float64,
        ),
        np.array(
            [logit_mean, logit_sd],
            dtype=np.float64,
        ),
        np.array(
            [0.0, 1.0],
            dtype=np.float64,
        ),
    ]

    fitted_results = []

    for starting_value in starting_values:
        result = minimize(
            objective,
            x0=starting_value,
            jac=gradient,
            method="L-BFGS-B",
            options={
                "maxiter": 5000,
                "ftol": 1e-12,
                "gtol": 1e-8,
                "maxls": 50,
            },
        )

        fitted_results.append(result)

    successful_results = [
        result
        for result in fitted_results
        if np.isfinite(result.fun)
        and np.isfinite(result.x).all()
    ]

    if not successful_results:
        return {
            "intercept": np.nan,
            "slope": np.nan,
            "converged": False,
            "reason": "all_optimizations_failed",
            "iterations": 0,
            "gradient_norm": np.nan,
            "objective": np.nan,
            "hit_boundary": False,
        }

    best = min(
        successful_results,
        key=lambda result: result.fun,
    )

    standardized_intercept = float(best.x[0])
    standardized_slope = float(best.x[1])

    # Transform coefficients back to the original prediction-logit scale.
    original_slope = (
        standardized_slope / logit_sd
    )

    original_intercept = (
        standardized_intercept
        - standardized_slope
        * logit_mean
        / logit_sd
    )

    gradient_norm = float(
        np.linalg.norm(
            gradient(best.x),
            ord=np.inf,
        )
    )

    converged = bool(
        best.success
        and np.isfinite(original_intercept)
        and np.isfinite(original_slope)
        and gradient_norm < 1e-4
    )

    return {
        "intercept": float(original_intercept),
        "slope": float(original_slope),
        "converged": converged,
        "reason": str(best.message),
        "iterations": int(best.nit),
        "gradient_norm": gradient_norm,
        "objective": float(best.fun),
        "hit_boundary": False,
    }


# ---------------------------------------------------------------------------
# Equal-frequency reliability bins
# ---------------------------------------------------------------------------
def equal_frequency_reliability_curve(
    probabilities,
    outcomes,
    n_bins=10,
):
    probabilities = np.asarray(
        probabilities,
        dtype=np.float64,
    )

    outcomes = np.asarray(
        outcomes,
        dtype=np.float64,
    )

    order = np.argsort(
        probabilities,
        kind="mergesort",
    )

    ordered_indices = np.array_split(
        order,
        n_bins,
    )

    rows = []

    for bin_number, selected in enumerate(
        ordered_indices,
        start=1,
    ):
        if len(selected) == 0:
            continue

        selected_probabilities = probabilities[
            selected
        ]

        selected_outcomes = outcomes[
            selected
        ]

        rows.append(
            {
                "bin": bin_number,
                "n": int(len(selected)),
                "probability_min": float(
                    selected_probabilities.min()
                ),
                "probability_max": float(
                    selected_probabilities.max()
                ),
                "mean_probability": float(
                    selected_probabilities.mean()
                ),
                "observed_frequency": float(
                    selected_outcomes.mean()
                ),
            }
        )

    return pd.DataFrame(rows)


# ---------------------------------------------------------------------------
# Fit every run and group
# ---------------------------------------------------------------------------
GROUPS = {
    "all": np.ones(
        len(labels),
        dtype=bool,
    ),
    "male": sex == 1,
    "female": sex == 0,
}

fit_rows = []
curve_frames = []

for model_number, (
    model_id,
    probabilities,
) in enumerate(
    model_probabilities.items(),
    start=1,
):
    method, seed_text = model_id.split("|")
    seed = int(seed_text)

    print(
        f"[{model_number:02d}/"
        f"{len(model_probabilities)}] "
        f"{model_id}"
    )

    for group_name, group_mask in GROUPS.items():
        group_probabilities = probabilities[
            group_mask
        ]

        group_outcomes = labels[
            group_mask
        ]

        fit = robust_calibration_fit(
            group_probabilities,
            group_outcomes,
        )

        fit_rows.append(
            {
                "model_id": model_id,
                "method": method,
                "seed": seed,
                "group": group_name,
                "n_images": int(
                    group_mask.sum()
                ),
                "n_positive": int(
                    group_outcomes.sum()
                ),
                "prevalence": float(
                    group_outcomes.mean()
                ),
                "mean_probability": float(
                    group_probabilities.mean()
                ),
                "brier_score": float(
                    brier_score_loss(
                        group_outcomes,
                        group_probabilities,
                    )
                ),
                "calibration_intercept": (
                    fit["intercept"]
                ),
                "calibration_slope": (
                    fit["slope"]
                ),
                "converged": fit["converged"],
                "optimizer_message": fit["reason"],
                "iterations": fit["iterations"],
                "gradient_norm": fit[
                    "gradient_norm"
                ],
                "objective": fit["objective"],
            }
        )

        curve = equal_frequency_reliability_curve(
            group_probabilities,
            group_outcomes,
            n_bins=10,
        )

        curve["model_id"] = model_id
        curve["method"] = method
        curve["seed"] = seed
        curve["group"] = group_name

        curve_frames.append(curve)

calibration_fits = pd.DataFrame(
    fit_rows
)

reliability_curves = pd.concat(
    curve_frames,
    ignore_index=True,
)

calibration_fits.to_csv(
    OUTPUTS
    / "run5b_calibration_intercept_slope_per_run.csv",
    index=False,
)

reliability_curves.to_csv(
    OUTPUTS
    / "run5b_equal_frequency_reliability_bins.csv",
    index=False,
)


# ---------------------------------------------------------------------------
# Summary across training runs
# ---------------------------------------------------------------------------
summary_rows = []

for (
    method,
    group_name,
), group in calibration_fits.groupby(
    ["method", "group"]
):
    for metric in [
        "calibration_intercept",
        "calibration_slope",
        "brier_score",
    ]:
        values = (
            group.loc[
                group["converged"],
                metric,
            ]
            .dropna()
            .to_numpy(dtype=float)
        )

        if len(values) == 0:
            continue

        summary_rows.append(
            {
                "method": method,
                "group": group_name,
                "metric": metric,
                "n_converged_runs": int(
                    len(values)
                ),
                "mean": float(
                    values.mean()
                ),
                "sd": (
                    float(
                        values.std(ddof=1)
                    )
                    if len(values) > 1
                    else np.nan
                ),
                "minimum": float(
                    values.min()
                ),
                "median": float(
                    np.median(values)
                ),
                "maximum": float(
                    values.max()
                ),
            }
        )

calibration_summary = pd.DataFrame(
    summary_rows
)

calibration_summary.to_csv(
    OUTPUTS
    / "run5b_calibration_summary_across_runs.csv",
    index=False,
)


# ---------------------------------------------------------------------------
# Create corrected equal-frequency calibration figures
# ---------------------------------------------------------------------------
def save_figure(figure, stem):
    figure.tight_layout()

    figure.savefig(
        OUTPUTS / f"{stem}.pdf",
        bbox_inches="tight",
    )

    figure.savefig(
        OUTPUTS / f"{stem}.png",
        dpi=300,
        bbox_inches="tight",
    )

    plt.close(figure)


for method in METHOD_ORDER:
    if method not in set(
        reliability_curves["method"]
    ):
        continue

    figure, axis = plt.subplots(
        figsize=(6.8, 5.8)
    )

    for group_name in [
        "male",
        "female",
    ]:
        subset = reliability_curves[
            (
                reliability_curves["method"]
                == method
            )
            & (
                reliability_curves["group"]
                == group_name
            )
        ]

        # Show each training run as a thin line.
        for seed, seed_frame in subset.groupby(
            "seed"
        ):
            seed_frame = seed_frame.sort_values(
                "bin"
            )

            axis.plot(
                seed_frame["mean_probability"],
                seed_frame["observed_frequency"],
                linewidth=0.8,
                alpha=0.25,
            )

        # Across-run mean calibration curve.
        mean_curve = (
            subset.groupby("bin")
            .agg(
                mean_probability=(
                    "mean_probability",
                    "mean",
                ),
                mean_observed=(
                    "observed_frequency",
                    "mean",
                ),
                minimum_observed=(
                    "observed_frequency",
                    "min",
                ),
                maximum_observed=(
                    "observed_frequency",
                    "max",
                ),
            )
            .reset_index()
            .sort_values("bin")
        )

        line = axis.plot(
            mean_curve["mean_probability"],
            mean_curve["mean_observed"],
            marker="o",
            linewidth=2,
            label=group_name.capitalize(),
        )[0]

        axis.fill_between(
            mean_curve["mean_probability"],
            mean_curve["minimum_observed"],
            mean_curve["maximum_observed"],
            alpha=0.12,
            color=line.get_color(),
        )

    axis.plot(
        [0, 1],
        [0, 1],
        linestyle="--",
        linewidth=1,
        label="Ideal calibration",
    )

    axis.set_xlim(0, 1)
    axis.set_ylim(0, 1)

    axis.set_xlabel(
        "Mean predicted probability"
    )

    axis.set_ylabel(
        "Observed outcome frequency"
    )

    axis.set_title(
        f"{method}: external-test calibration"
    )

    axis.grid(alpha=0.25)
    axis.legend(frameon=False)

    save_figure(
        figure,
        f"run5b_calibration_{method.lower()}",
    )


# ---------------------------------------------------------------------------
# Print decision summary
# ---------------------------------------------------------------------------
n_total = len(calibration_fits)
n_converged = int(
    calibration_fits["converged"].sum()
)

print("\n" + "=" * 94)
print("RUN 5B DECISION SUMMARY")
print("=" * 94)

print(
    f"Calibration fits converged: "
    f"{n_converged}/{n_total}"
)

failed = calibration_fits[
    ~calibration_fits["converged"]
]

if len(failed):
    print("\nFailed fits:")

    print(
        failed[
            [
                "model_id",
                "group",
                "optimizer_message",
                "gradient_norm",
            ]
        ].to_string(
            index=False
        )
    )

print("\nOverall calibration summary:")

print(
    calibration_summary[
        (
            calibration_summary["group"]
            == "all"
        )
        & calibration_summary[
            "metric"
        ].isin(
            [
                "calibration_intercept",
                "calibration_slope",
                "brier_score",
            ]
        )
    ].to_string(
        index=False,
        float_format=lambda value: (
            f"{value:.6f}"
        ),
    )
)

print("\nSex-stratified calibration summary:")

print(
    calibration_summary[
        calibration_summary["group"].isin(
            ["male", "female"]
        )
        & calibration_summary[
            "metric"
        ].isin(
            [
                "calibration_intercept",
                "calibration_slope",
            ]
        )
    ].to_string(
        index=False,
        float_format=lambda value: (
            f"{value:.6f}"
        ),
    )
)


# ---------------------------------------------------------------------------
# Metadata and ZIP
# ---------------------------------------------------------------------------
metadata = {
    "calibration_model": (
        "Outcome regressed on the logit of predicted probability"
    ),
    "ideal_intercept": 0.0,
    "ideal_slope": 1.0,
    "optimizer": (
        "L-BFGS-B with standardized prediction logits, "
        "stable logaddexp likelihood, analytic gradient, "
        "and multiple starting values"
    ),
    "reliability_bins": (
        "10 equal-frequency bins per training run and sex group"
    ),
    "inference_status": (
        "Calibration intercept and slope are descriptive across runs; "
        "Brier-score inference remains in Run 5."
    ),
}

with open(
    OUTPUTS / "run5b_metadata.json",
    "w",
    encoding="utf-8",
) as file:
    json.dump(
        metadata,
        file,
        indent=2,
    )

local_zip = Path(
    "/content/revision_run5b.zip"
)

with zipfile.ZipFile(
    local_zip,
    "w",
    compression=zipfile.ZIP_DEFLATED,
) as archive:
    for output_path in sorted(
        OUTPUTS.rglob("*")
    ):
        if output_path.is_file():
            archive.write(
                output_path,
                arcname=output_path.relative_to(
                    OUTPUTS
                ),
            )

drive_zip = RESULTS / "revision_run5b.zip"

shutil.copy2(
    local_zip,
    drive_zip,
)

print("\nRUN 5B COMPLETE")
print("Local ZIP:", local_zip)
print("Drive ZIP:", drive_zip)
print("=" * 94)

Mounted at /content/drive
RUN 5B: ROBUST CALIBRATION ANALYSIS
Images:             112,120
Unique patients:    30,805
Prediction arrays:  18
[01/18] DWFA|42
[02/18] DWFA|123
[03/18] DWFA|456
[04/18] DWFA|789
[05/18] DWFA|1010
[06/18] CADR|42
[07/18] CADR|123
[08/18] CADR|456
[09/18] CADR|789
[10/18] CADR|1010
[11/18] FedAvg|42
[12/18] FedAvg|123
[13/18] FedAvg|456
[14/18] LPR|42
[15/18] LPR|123
[16/18] LPR|456
[17/18] LPR|789
[18/18] LPR|1010

RUN 5B DECISION SUMMARY
Calibration fits converged: 53/54

Failed fits:
 model_id group                                    optimizer_message  gradient_norm
FedAvg|42   all CONVERGENCE: RELATIVE REDUCTION OF F <= FACTR*EPSMCH       0.000217

Overall calibration summary:
method group                metric  n_converged_runs      mean       sd   minimum    median   maximum
  CADR   all calibration_intercept                 5 -1.662782 0.116917 -1.784378 -1.704986 -1.482790
  CADR   all     calibration_slope                 5  0.413565 0.039593  0.3626

In [ ]:
# ============================================================================
# RUN 6: FINAL ROBUSTNESS ANALYSIS
#
# Includes:
#   1. Three independent 5,000-replicate Monte Carlo bootstrap analyses.
#   2. Leave-one-seed-out run-aware inference.
#   3. Patient-equal-weighted NIH sensitivity analysis.
#   4. Simultaneous max-t intervals across six primary comparisons.
#   5. Journal-ready diagnostic figures and machine-readable tables.
#
# No training.
# No checkpoint inference.
# L4 GPU strongly recommended.
#
# Final output:
#   /content/drive/MyDrive/FairFedCXR/results/revision_run6.zip
# ============================================================================

from google.colab import drive
drive.mount("/content/drive", force_remount=True)

from pathlib import Path
import gc
import json
import shutil
import time
import zipfile

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd


# ============================================================================
# CONFIGURATION
# ============================================================================

BASE = Path("/content/drive/MyDrive/FairFedCXR")
RESULTS = BASE / "results"

NIH_CSV = RESULTS / "nih_effusion_cohort.csv"
PREDICTION_CACHE = RESULTS / "nih_pred_cache.npz"

LOCAL = Path("/content/revision_run6")
INPUTS = LOCAL / "inputs"
OUTPUTS = LOCAL / "outputs"
INTERMEDIATE = LOCAL / "intermediate"

if LOCAL.exists():
    shutil.rmtree(LOCAL)

for directory in [INPUTS, OUTPUTS, INTERMEDIATE]:
    directory.mkdir(parents=True, exist_ok=True)

for required_path in [NIH_CSV, PREDICTION_CACHE]:
    if not required_path.exists():
        raise FileNotFoundError(
            f"Required input file is missing: {required_path}"
        )

# Final-analysis settings.
N_BOOTSTRAP = 5000
GPU_BATCH_SIZE = 25

# Three independent bootstrap/random-number seeds for Monte Carlo stability.
MONTE_CARLO_SEEDS = [
    20260726,
    20260817,
    20260911,
]

# Separate bootstrap seed for patient-equal weighting.
PATIENT_EQUAL_BOOTSTRAP_SEED = 20261003

# The five runs common to LPR, CADR, and DWFA.
TRAINING_SEEDS = [
    42,
    123,
    456,
    789,
    1010,
]

METHOD_LABELS = {
    "qfedavg": "LPR",
    "fairfed": "CADR",
    "dwfa": "DWFA",
}

CORE_METHODS = [
    "LPR",
    "CADR",
    "DWFA",
]

PRIMARY_COMPARISONS = [
    ("DWFA", "LPR"),
    ("DWFA", "CADR"),
    ("LPR", "CADR"),
]

PRIMARY_METRICS = [
    "auroc",
    "worst_group_auroc",
]

METRIC_LABELS = {
    "auroc": "Overall AUROC",
    "worst_group_auroc": "Worst-group AUROC",
}

print("=" * 100)
print("RUN 6: FINAL ROBUSTNESS ANALYSIS")
print("=" * 100)
print(f"Bootstrap replicates per analysis: {N_BOOTSTRAP:,}")
print(f"Monte Carlo seeds: {MONTE_CARLO_SEEDS}")
print(f"Training seeds: {TRAINING_SEEDS}")


# ============================================================================
# COPY INPUTS LOCALLY
# ============================================================================

local_nih = INPUTS / NIH_CSV.name
local_cache = INPUTS / PREDICTION_CACHE.name

shutil.copy2(NIH_CSV, local_nih)
shutil.copy2(PREDICTION_CACHE, local_cache)


# ============================================================================
# LOAD AND VALIDATE DATA
# ============================================================================

nih = pd.read_csv(
    local_nih,
    low_memory=False,
)

required_columns = {
    "Patient ID",
    "label",
    "sex_encoded",
}

missing_columns = required_columns - set(nih.columns)

if missing_columns:
    raise KeyError(
        "NIH cohort is missing required columns: "
        f"{sorted(missing_columns)}"
    )

nih = nih.copy()

nih["patient_id"] = (
    nih["Patient ID"]
    .astype(str)
    .str.strip()
)

nih["label"] = (
    nih["label"]
    .astype(np.int8)
)

nih["sex_encoded"] = (
    nih["sex_encoded"]
    .astype(np.int8)
)

labels = nih["label"].to_numpy(
    dtype=np.int8,
)

sex = nih["sex_encoded"].to_numpy(
    dtype=np.int8,
)

patient_code, patient_levels = pd.factorize(
    nih["patient_id"],
    sort=True,
)

patient_code = patient_code.astype(
    np.int32
)

n_images = len(nih)
n_patients = len(patient_levels)

cache = np.load(
    local_cache,
    allow_pickle=True,
)

raw_predictions = cache["all_probs"].item()

assert np.array_equal(
    cache["labels"].astype(np.int8),
    labels,
), "Cached labels do not match the NIH CSV row order."

assert np.array_equal(
    cache["sex"].astype(np.int8),
    sex,
), "Cached sex values do not match the NIH CSV row order."

print(f"NIH radiographs:          {n_images:,}")
print(f"Unique NIH patients:      {n_patients:,}")
print(f"Prediction arrays found:  {len(raw_predictions):,}")


# ============================================================================
# ORGANIZE THE 15 PRIMARY MODELS
# ============================================================================

model_probabilities = {}

method_model_ids = {
    method: []
    for method in CORE_METHODS
}

for (
    cache_method,
    seed,
), probabilities in sorted(
    raw_predictions.items(),
    key=lambda item: (
        str(item[0][0]),
        int(item[0][1]),
    ),
):
    if cache_method not in METHOD_LABELS:
        continue

    method = METHOD_LABELS[cache_method]
    seed = int(seed)

    if seed not in TRAINING_SEEDS:
        continue

    model_id = f"{method}|{seed}"

    probabilities = np.asarray(
        probabilities,
        dtype=np.float32,
    ).reshape(-1)

    assert len(probabilities) == n_images
    assert np.isfinite(probabilities).all()
    assert probabilities.min() >= 0.0
    assert probabilities.max() <= 1.0

    model_probabilities[model_id] = probabilities
    method_model_ids[method].append(model_id)

for method in CORE_METHODS:
    method_model_ids[method] = sorted(
        method_model_ids[method],
        key=lambda model_id: int(
            model_id.split("|")[1]
        ),
    )

    if len(method_model_ids[method]) != 5:
        raise RuntimeError(
            f"{method}: expected 5 runs, "
            f"found {len(method_model_ids[method])}"
        )

print("\nPrimary models:")

for method, model_ids in method_model_ids.items():
    print(f"  {method}: {model_ids}")


# ============================================================================
# GPU BACKEND
# ============================================================================

USE_GPU = False

try:
    import cupy as cp

    if cp.cuda.runtime.getDeviceCount() > 0:
        USE_GPU = True

        device_properties = cp.cuda.runtime.getDeviceProperties(0)
        device_name = device_properties["name"]

        if isinstance(device_name, bytes):
            device_name = device_name.decode()

        print(f"\nWeighted-AUROC backend: CuPy GPU ({device_name})")

except Exception as error:
    print("\nCuPy GPU backend unavailable:", repr(error))

if not USE_GPU:
    print(
        "WARNING: NumPy CPU mode will be extremely slow for this final run. "
        "An L4 GPU is strongly recommended."
    )


# ============================================================================
# PATIENT AND IMAGE WEIGHTS
# ============================================================================

images_per_patient = np.bincount(
    patient_code,
    minlength=n_patients,
).astype(np.float64)

if np.any(images_per_patient <= 0):
    raise RuntimeError(
        "At least one encoded patient has no image records."
    )

# Primary image-level estimand:
# every radiograph contributes weight 1, while the bootstrap clusters by patient.
image_level_base_weights = np.ones(
    n_images,
    dtype=np.float32,
)

# Patient-equal sensitivity estimand:
# all radiographs belonging to one patient sum to total weight 1.
patient_equal_base_weights = (
    1.0
    / images_per_patient[
        patient_code
    ]
).astype(np.float32)

patient_total_weights = np.bincount(
    patient_code,
    weights=patient_equal_base_weights,
    minlength=n_patients,
)

assert np.allclose(
    patient_total_weights,
    1.0,
    atol=1e-6,
)

print(
    "\nPatient-equal sensitivity weights verified: "
    "each patient contributes total weight 1."
)


# ============================================================================
# PRECOMPUTE AUROC SORTING STRUCTURES
# ============================================================================

GROUP_MASKS = {
    "all": np.ones(
        n_images,
        dtype=bool,
    ),
    "male": sex == 1,
    "female": sex == 0,
}


def prepare_auc_structure(
    probabilities,
    group_mask,
):
    """
    Precompute score sorting and tie-group structure.

    The same structure is reused across every patient bootstrap and both
    weighting estimands.
    """
    original_indices = np.flatnonzero(
        group_mask
    )

    group_probabilities = probabilities[
        original_indices
    ]

    order = np.argsort(
        group_probabilities,
        kind="mergesort",
    )

    sorted_original_indices = original_indices[
        order
    ].astype(np.int32)

    sorted_probabilities = probabilities[
        sorted_original_indices
    ]

    sorted_labels = labels[
        sorted_original_indices
    ].astype(np.float32)

    sorted_patient_codes = patient_code[
        sorted_original_indices
    ].astype(np.int32)

    tie_starts = np.r_[
        0,
        np.flatnonzero(
            np.diff(
                sorted_probabilities
            ) != 0
        ) + 1,
    ].astype(np.int32)

    return {
        "sorted_original_indices": sorted_original_indices,
        "labels": sorted_labels,
        "patient_codes": sorted_patient_codes,
        "tie_starts": tie_starts,
    }


print("\nPrecomputing model score-order structures...")

auc_structures = {}

for model_number, (
    model_id,
    probabilities,
) in enumerate(
    model_probabilities.items(),
    start=1,
):
    print(
        f"  [{model_number:02d}/"
        f"{len(model_probabilities)}] {model_id}"
    )

    auc_structures[model_id] = {}

    for group_name, group_mask in GROUP_MASKS.items():
        auc_structures[
            model_id
        ][group_name] = prepare_auc_structure(
            probabilities,
            group_mask,
        )


# ============================================================================
# PATIENT-CLUSTER BOOTSTRAP COUNTS
# ============================================================================

def generate_patient_bootstrap_counts(
    random_seed,
    n_bootstrap=N_BOOTSTRAP,
):
    """
    Draw n_patients unique patient clusters with replacement in each
    bootstrap replicate.

    Returns patient multiplicities:
        shape = bootstrap replicate × unique patient.
    """
    rng = np.random.default_rng(
        random_seed
    )

    counts = np.empty(
        (
            n_bootstrap,
            n_patients,
        ),
        dtype=np.uint16,
    )

    for bootstrap_index in range(
        n_bootstrap
    ):
        sampled_patient_codes = rng.integers(
            0,
            n_patients,
            size=n_patients,
            dtype=np.int32,
        )

        counts[
            bootstrap_index
        ] = np.bincount(
            sampled_patient_codes,
            minlength=n_patients,
        ).astype(np.uint16)

    assert np.all(
        counts.sum(axis=1)
        == n_patients
    )

    return counts


# ============================================================================
# EXACT WEIGHTED AUROC WITH TIED PREDICTIONS
# ============================================================================

def auc_from_structure_numpy(
    structure,
    patient_counts,
    base_image_weights,
):
    n_replicates = len(
        patient_counts
    )

    output = np.empty(
        n_replicates,
        dtype=np.float64,
    )

    sorted_indices = structure[
        "sorted_original_indices"
    ]

    sorted_labels = structure[
        "labels"
    ].astype(np.float64)

    sorted_patient_codes = structure[
        "patient_codes"
    ]

    tie_starts = structure[
        "tie_starts"
    ]

    sorted_base_weights = base_image_weights[
        sorted_indices
    ].astype(np.float64)

    for batch_start in range(
        0,
        n_replicates,
        GPU_BATCH_SIZE,
    ):
        batch_stop = min(
            batch_start + GPU_BATCH_SIZE,
            n_replicates,
        )

        cluster_multiplicities = patient_counts[
            batch_start:batch_stop,
            sorted_patient_codes,
        ].astype(np.float64)

        image_weights = (
            cluster_multiplicities
            * sorted_base_weights[None, :]
        )

        positive_weights = (
            image_weights
            * sorted_labels[None, :]
        )

        negative_weights = (
            image_weights
            - positive_weights
        )

        positive_by_score = np.add.reduceat(
            positive_weights,
            tie_starts,
            axis=1,
        )

        negative_by_score = np.add.reduceat(
            negative_weights,
            tie_starts,
            axis=1,
        )

        cumulative_negative_before = (
            np.cumsum(
                negative_by_score,
                axis=1,
            )
            - negative_by_score
        )

        numerator = np.sum(
            positive_by_score
            * (
                cumulative_negative_before
                + 0.5
                * negative_by_score
            ),
            axis=1,
        )

        total_positive = positive_by_score.sum(
            axis=1
        )

        total_negative = negative_by_score.sum(
            axis=1
        )

        denominator = (
            total_positive
            * total_negative
        )

        batch_auc = np.divide(
            numerator,
            denominator,
            out=np.full_like(
                numerator,
                np.nan,
            ),
            where=denominator > 0,
        )

        output[
            batch_start:batch_stop
        ] = batch_auc

    return output


def auc_from_structure_gpu(
    structure,
    patient_counts,
    base_image_weights,
):
    n_replicates = len(
        patient_counts
    )

    output = np.empty(
        n_replicates,
        dtype=np.float64,
    )

    sorted_indices = structure[
        "sorted_original_indices"
    ]

    sorted_labels_gpu = cp.asarray(
        structure["labels"],
        dtype=cp.float32,
    )

    sorted_patient_codes_gpu = cp.asarray(
        structure["patient_codes"],
        dtype=cp.int32,
    )

    tie_starts_gpu = cp.asarray(
        structure["tie_starts"],
        dtype=cp.int32,
    )

    sorted_base_weights_gpu = cp.asarray(
        base_image_weights[
            sorted_indices
        ],
        dtype=cp.float32,
    )

    for batch_start in range(
        0,
        n_replicates,
        GPU_BATCH_SIZE,
    ):
        batch_stop = min(
            batch_start + GPU_BATCH_SIZE,
            n_replicates,
        )

        counts_gpu = cp.asarray(
            patient_counts[
                batch_start:batch_stop
            ],
            dtype=cp.float32,
        )

        cluster_multiplicities = counts_gpu[
            :,
            sorted_patient_codes_gpu,
        ]

        image_weights = (
            cluster_multiplicities
            * sorted_base_weights_gpu[
                None,
                :,
            ]
        )

        positive_weights = (
            image_weights
            * sorted_labels_gpu[
                None,
                :,
            ]
        )

        negative_weights = (
            image_weights
            - positive_weights
        )

        positive_by_score = cp.add.reduceat(
            positive_weights,
            tie_starts_gpu,
            axis=1,
        )

        negative_by_score = cp.add.reduceat(
            negative_weights,
            tie_starts_gpu,
            axis=1,
        )

        cumulative_negative_before = (
            cp.cumsum(
                negative_by_score,
                axis=1,
            )
            - negative_by_score
        )

        numerator = cp.sum(
            positive_by_score
            * (
                cumulative_negative_before
                + 0.5
                * negative_by_score
            ),
            axis=1,
        )

        denominator = (
            cp.sum(
                positive_by_score,
                axis=1,
            )
            * cp.sum(
                negative_by_score,
                axis=1,
            )
        )

        batch_auc = cp.where(
            denominator > 0,
            numerator / denominator,
            cp.nan,
        )

        output[
            batch_start:batch_stop
        ] = cp.asnumpy(
            batch_auc
        )

        del (
            counts_gpu,
            cluster_multiplicities,
            image_weights,
            positive_weights,
            negative_weights,
            positive_by_score,
            negative_by_score,
            cumulative_negative_before,
            numerator,
            denominator,
            batch_auc,
        )

    del (
        sorted_labels_gpu,
        sorted_patient_codes_gpu,
        tie_starts_gpu,
        sorted_base_weights_gpu,
    )

    cp.get_default_memory_pool().free_all_blocks()

    return output


def auc_from_structure(
    structure,
    patient_counts,
    base_image_weights,
):
    if USE_GPU:
        return auc_from_structure_gpu(
            structure,
            patient_counts,
            base_image_weights,
        )

    return auc_from_structure_numpy(
        structure,
        patient_counts,
        base_image_weights,
    )


# ============================================================================
# SELF-TEST OF THE WEIGHTED AUROC IMPLEMENTATION
# ============================================================================

toy_probabilities = np.array(
    [0.1, 0.2, 0.2, 0.8, 0.8, 0.7],
    dtype=np.float32,
)

toy_labels = np.array(
    [0, 1, 0, 1, 1, 0],
    dtype=np.int8,
)

toy_patient_codes = np.arange(
    6,
    dtype=np.int32,
)

# Manual independent weighted-AUROC implementation for the toy test.
def direct_weighted_auc(
    y_true,
    scores,
    weights,
):
    numerator = 0.0
    denominator = 0.0

    positive_indices = np.flatnonzero(
        y_true == 1
    )

    negative_indices = np.flatnonzero(
        y_true == 0
    )

    for positive_index in positive_indices:
        for negative_index in negative_indices:
            pair_weight = (
                weights[positive_index]
                * weights[negative_index]
            )

            denominator += pair_weight

            if (
                scores[positive_index]
                > scores[negative_index]
            ):
                numerator += pair_weight

            elif (
                scores[positive_index]
                == scores[negative_index]
            ):
                numerator += (
                    0.5 * pair_weight
                )

    return numerator / denominator


toy_order = np.argsort(
    toy_probabilities,
    kind="mergesort",
)

toy_sorted_scores = toy_probabilities[
    toy_order
]

toy_structure = {
    "sorted_original_indices": toy_order.astype(
        np.int32
    ),
    "labels": toy_labels[
        toy_order
    ].astype(np.float32),
    "patient_codes": toy_patient_codes[
        toy_order
    ].astype(np.int32),
    "tie_starts": np.r_[
        0,
        np.flatnonzero(
            np.diff(
                toy_sorted_scores
            ) != 0
        ) + 1,
    ].astype(np.int32),
}

toy_counts = np.ones(
    (1, 6),
    dtype=np.uint16,
)

toy_weights = np.array(
    [1.0, 0.5, 0.5, 1.0, 2.0, 1.0],
    dtype=np.float32,
)

toy_auc_calculated = auc_from_structure(
    toy_structure,
    toy_counts,
    toy_weights,
)[0]

toy_auc_expected = direct_weighted_auc(
    toy_labels,
    toy_probabilities,
    toy_weights,
)

assert abs(
    toy_auc_calculated
    - toy_auc_expected
) < 2e-6

print("\nWeighted-AUROC implementation self-test: PASS")


# ============================================================================
# COMPUTE ALL MODEL METRICS FOR ONE BOOTSTRAP SET
# ============================================================================

def compute_model_bootstrap_metrics(
    patient_counts,
    base_image_weights,
    analysis_label,
):
    """
    Returns:
      model_bootstrap[model_id][metric] -> 5,000 bootstrap values
      full_sample DataFrame -> one full-cohort result per model
    """
    all_patients_once = np.ones(
        (1, n_patients),
        dtype=np.uint16,
    )

    model_bootstrap = {}
    full_sample_rows = []

    total_models = len(
        model_probabilities
    )

    start_time = time.time()

    for model_number, model_id in enumerate(
        model_probabilities,
        start=1,
    ):
        print(
            f"    [{model_number:02d}/{total_models}] "
            f"{analysis_label}: {model_id}"
        )

        bootstrap_values = {}
        full_values = {}

        for group_name in [
            "all",
            "male",
            "female",
        ]:
            structure = auc_structures[
                model_id
            ][group_name]

            full_auc = auc_from_structure(
                structure,
                all_patients_once,
                base_image_weights,
            )[0]

            bootstrap_auc = auc_from_structure(
                structure,
                patient_counts,
                base_image_weights,
            )

            metric_name = (
                "auroc"
                if group_name == "all"
                else f"{group_name}_auroc"
            )

            full_values[
                metric_name
            ] = float(full_auc)

            bootstrap_values[
                metric_name
            ] = bootstrap_auc

        full_values[
            "worst_group_auroc"
        ] = min(
            full_values["male_auroc"],
            full_values["female_auroc"],
        )

        bootstrap_values[
            "worst_group_auroc"
        ] = np.minimum(
            bootstrap_values[
                "male_auroc"
            ],
            bootstrap_values[
                "female_auroc"
            ],
        )

        method, seed_text = model_id.split("|")

        full_sample_rows.append(
            {
                "analysis": analysis_label,
                "model_id": model_id,
                "method": method,
                "seed": int(seed_text),
                **full_values,
            }
        )

        model_bootstrap[
            model_id
        ] = bootstrap_values

    elapsed_minutes = (
        time.time() - start_time
    ) / 60.0

    print(
        f"    Completed {analysis_label} model metrics "
        f"in {elapsed_minutes:.1f} minutes."
    )

    return (
        model_bootstrap,
        pd.DataFrame(full_sample_rows),
    )


# ============================================================================
# FAMILYWISE RUN-AWARE INFERENCE
# ============================================================================

def percentile_interval(
    values,
):
    lower, upper = np.percentile(
        values,
        [2.5, 97.5],
    )

    return float(lower), float(upper)


def interval_excludes_zero(
    lower,
    upper,
):
    return bool(
        lower > 0
        or upper < 0
    )


def build_run_aware_family(
    model_bootstrap,
    full_sample_metrics,
    run_resampling_seed,
    analysis_name,
    omitted_seed=None,
):
    """
    Independent run resampling across methods.

    The same run-index matrix is reused across AUROC and worst-group AUROC
    within each method, preserving cross-metric dependence for max-t
    simultaneous inference.
    """
    full_lookup = full_sample_metrics.set_index(
        "model_id"
    )

    method_ids = {}

    for method in CORE_METHODS:
        ids = method_model_ids[
            method
        ]

        if omitted_seed is not None:
            ids = [
                model_id
                for model_id in ids
                if int(
                    model_id.split("|")[1]
                ) != int(omitted_seed)
            ]

        method_ids[method] = ids

    n_runs = {
        method: len(ids)
        for method, ids in method_ids.items()
    }

    if len(set(n_runs.values())) != 1:
        raise RuntimeError(
            f"Methods have different run counts: {n_runs}"
        )

    number_of_runs = next(
        iter(n_runs.values())
    )

    number_of_bootstrap_replicates = len(
        next(
            iter(
                model_bootstrap.values()
            )
        )["auroc"]
    )

    rng = np.random.default_rng(
        run_resampling_seed
    )

    # One matrix per method; reused for every metric in that method.
    run_indices = {
        method: rng.integers(
            0,
            number_of_runs,
            size=(
                number_of_bootstrap_replicates,
                number_of_runs,
            ),
        )
        for method in CORE_METHODS
    }

    observed_method_metrics = {
        method: {}
        for method in CORE_METHODS
    }

    bootstrap_method_metrics = {
        method: {}
        for method in CORE_METHODS
    }

    for method in CORE_METHODS:
        for metric in PRIMARY_METRICS:
            ids = method_ids[method]

            bootstrap_matrix = np.column_stack(
                [
                    model_bootstrap[
                        model_id
                    ][metric]
                    for model_id in ids
                ]
            )

            full_values = np.array(
                [
                    full_lookup.loc[
                        model_id,
                        metric,
                    ]
                    for model_id in ids
                ],
                dtype=np.float64,
            )

            observed_method_metrics[
                method
            ][metric] = float(
                full_values.mean()
            )

            bootstrap_method_metrics[
                method
            ][metric] = np.take_along_axis(
                bootstrap_matrix,
                run_indices[method],
                axis=1,
            ).mean(axis=1)

    identities = []
    observed_differences = []
    bootstrap_columns = []

    for method_a, method_b in PRIMARY_COMPARISONS:
        for metric in PRIMARY_METRICS:
            identities.append(
                (
                    method_a,
                    method_b,
                    metric,
                )
            )

            observed_differences.append(
                observed_method_metrics[
                    method_a
                ][metric]
                - observed_method_metrics[
                    method_b
                ][metric]
            )

            bootstrap_columns.append(
                bootstrap_method_metrics[
                    method_a
                ][metric]
                - bootstrap_method_metrics[
                    method_b
                ][metric]
            )

    observed_differences = np.asarray(
        observed_differences,
        dtype=np.float64,
    )

    bootstrap_matrix = np.column_stack(
        bootstrap_columns
    )

    bootstrap_mean = bootstrap_matrix.mean(
        axis=0
    )

    bootstrap_sd = bootstrap_matrix.std(
        axis=0,
        ddof=1,
    )

    if np.any(
        bootstrap_sd <= 0
    ):
        raise RuntimeError(
            "At least one bootstrap comparison has zero variance."
        )

    standardized_centered = (
        bootstrap_matrix
        - bootstrap_mean
    ) / bootstrap_sd

    maximum_absolute_statistic = np.max(
        np.abs(
            standardized_centered
        ),
        axis=1,
    )

    familywise_critical_value = float(
        np.quantile(
            maximum_absolute_statistic,
            0.95,
        )
    )

    rows = []

    for index, (
        method_a,
        method_b,
        metric,
    ) in enumerate(identities):
        pointwise_low, pointwise_high = percentile_interval(
            bootstrap_matrix[
                :,
                index,
            ]
        )

        simultaneous_low = (
            observed_differences[index]
            - familywise_critical_value
            * bootstrap_sd[index]
        )

        simultaneous_high = (
            observed_differences[index]
            + familywise_critical_value
            * bootstrap_sd[index]
        )

        rows.append(
            {
                "analysis": analysis_name,
                "omitted_seed": (
                    omitted_seed
                    if omitted_seed is not None
                    else np.nan
                ),
                "method_a": method_a,
                "method_b": method_b,
                "comparison": (
                    f"{method_a} minus {method_b}"
                ),
                "metric": metric,
                "estimate": (
                    observed_differences[index]
                ),
                "pointwise_ci_low": (
                    pointwise_low
                ),
                "pointwise_ci_high": (
                    pointwise_high
                ),
                "simultaneous_ci_low": (
                    simultaneous_low
                ),
                "simultaneous_ci_high": (
                    simultaneous_high
                ),
                "pointwise_excludes_zero": (
                    interval_excludes_zero(
                        pointwise_low,
                        pointwise_high,
                    )
                ),
                "simultaneous_excludes_zero": (
                    interval_excludes_zero(
                        simultaneous_low,
                        simultaneous_high,
                    )
                ),
                "bootstrap_sd": (
                    bootstrap_sd[index]
                ),
                "familywise_critical_value": (
                    familywise_critical_value
                ),
                "n_runs_each": number_of_runs,
                "n_patient_bootstrap": (
                    number_of_bootstrap_replicates
                ),
                "run_resampling_seed": (
                    run_resampling_seed
                ),
            }
        )

    return pd.DataFrame(rows)


# ============================================================================
# PART A: THREE MONTE CARLO REPLICATIONS
# ============================================================================

print("\n" + "=" * 100)
print("PART A: THREE 5,000-REPLICATE MONTE CARLO ANALYSES")
print("=" * 100)

monte_carlo_result_frames = []
monte_carlo_full_metric_frames = []

# The first Monte Carlo model-level bootstrap is retained for LOO analysis.
first_model_bootstrap = None
first_full_sample_metrics = None

# Save compact intermediate model metrics for reproducibility.
saved_model_bootstraps = {}

for monte_carlo_index, bootstrap_seed in enumerate(
    MONTE_CARLO_SEEDS,
    start=1,
):
    print(
        f"\nMonte Carlo analysis {monte_carlo_index}/"
        f"{len(MONTE_CARLO_SEEDS)}"
    )

    print(
        f"  Patient bootstrap seed: {bootstrap_seed}"
    )

    patient_counts = generate_patient_bootstrap_counts(
        bootstrap_seed
    )

    print(
        "  Patient multiplicity matrix:",
        f"{patient_counts.nbytes / 1024**2:.1f} MB",
    )

    model_bootstrap, full_sample_metrics = (
        compute_model_bootstrap_metrics(
            patient_counts=patient_counts,
            base_image_weights=image_level_base_weights,
            analysis_label=(
                f"image_level_mc_{monte_carlo_index}"
            ),
        )
    )

    run_resampling_seed = (
        bootstrap_seed + 100_000
    )

    family_results = build_run_aware_family(
        model_bootstrap=model_bootstrap,
        full_sample_metrics=full_sample_metrics,
        run_resampling_seed=run_resampling_seed,
        analysis_name=(
            f"image_level_mc_{monte_carlo_index}"
        ),
    )

    family_results[
        "monte_carlo_index"
    ] = monte_carlo_index

    family_results[
        "patient_bootstrap_seed"
    ] = bootstrap_seed

    monte_carlo_result_frames.append(
        family_results
    )

    monte_carlo_full_metric_frames.append(
        full_sample_metrics
    )

    # Retain the first analysis for leave-one-seed-out calculations.
    if monte_carlo_index == 1:
        first_model_bootstrap = model_bootstrap
        first_full_sample_metrics = full_sample_metrics

    # Save every model bootstrap in compact float32 form.
    compact_bootstrap = {}

    for model_id, metric_dictionary in (
        model_bootstrap.items()
    ):
        compact_bootstrap[model_id] = {
            metric: values.astype(
                np.float32
            )
            for metric, values in (
                metric_dictionary.items()
            )
        }

    saved_model_bootstraps[
        f"image_level_mc_{monte_carlo_index}"
    ] = compact_bootstrap

    # Save each completed Monte Carlo result immediately.
    family_results.to_csv(
        INTERMEDIATE
        / (
            f"run6_mc_{monte_carlo_index}_"
            "primary_inference.csv"
        ),
        index=False,
    )

    del patient_counts
    del model_bootstrap
    del compact_bootstrap

    gc.collect()

    if USE_GPU:
        cp.get_default_memory_pool().free_all_blocks()


monte_carlo_results = pd.concat(
    monte_carlo_result_frames,
    ignore_index=True,
)

monte_carlo_full_metrics = pd.concat(
    monte_carlo_full_metric_frames,
    ignore_index=True,
)

monte_carlo_results.to_csv(
    OUTPUTS
    / "run6_monte_carlo_primary_inference.csv",
    index=False,
)

monte_carlo_full_metrics.to_csv(
    OUTPUTS
    / "run6_monte_carlo_full_sample_metrics.csv",
    index=False,
)


# ============================================================================
# MONTE CARLO ENDPOINT STABILITY SUMMARY
# ============================================================================

monte_carlo_stability = (
    monte_carlo_results.groupby(
        [
            "comparison",
            "metric",
        ]
    )
    .agg(
        n_monte_carlo_runs=(
            "monte_carlo_index",
            "nunique",
        ),
        estimate=(
            "estimate",
            "first",
        ),
        pointwise_ci_low_min=(
            "pointwise_ci_low",
            "min",
        ),
        pointwise_ci_low_max=(
            "pointwise_ci_low",
            "max",
        ),
        pointwise_ci_high_min=(
            "pointwise_ci_high",
            "min",
        ),
        pointwise_ci_high_max=(
            "pointwise_ci_high",
            "max",
        ),
        simultaneous_ci_low_min=(
            "simultaneous_ci_low",
            "min",
        ),
        simultaneous_ci_low_max=(
            "simultaneous_ci_low",
            "max",
        ),
        simultaneous_ci_high_min=(
            "simultaneous_ci_high",
            "min",
        ),
        simultaneous_ci_high_max=(
            "simultaneous_ci_high",
            "max",
        ),
        n_pointwise_resolved=(
            "pointwise_excludes_zero",
            "sum",
        ),
        n_simultaneous_resolved=(
            "simultaneous_excludes_zero",
            "sum",
        ),
    )
    .reset_index()
)

monte_carlo_stability[
    "maximum_pointwise_low_variation"
] = (
    monte_carlo_stability[
        "pointwise_ci_low_max"
    ]
    - monte_carlo_stability[
        "pointwise_ci_low_min"
    ]
)

monte_carlo_stability[
    "maximum_pointwise_high_variation"
] = (
    monte_carlo_stability[
        "pointwise_ci_high_max"
    ]
    - monte_carlo_stability[
        "pointwise_ci_high_min"
    ]
)

monte_carlo_stability.to_csv(
    OUTPUTS
    / "run6_monte_carlo_endpoint_stability.csv",
    index=False,
)


# ============================================================================
# PART B: LEAVE-ONE-SEED-OUT ANALYSIS
# ============================================================================

print("\n" + "=" * 100)
print("PART B: LEAVE-ONE-SEED-OUT STABILITY")
print("=" * 100)

leave_one_out_frames = []

for omitted_seed in TRAINING_SEEDS:
    print(
        f"  Omitting seed {omitted_seed}"
    )

    leave_one_out_result = build_run_aware_family(
        model_bootstrap=first_model_bootstrap,
        full_sample_metrics=first_full_sample_metrics,
        run_resampling_seed=(
            20262000 + omitted_seed
        ),
        analysis_name=(
            f"leave_seed_{omitted_seed}_out"
        ),
        omitted_seed=omitted_seed,
    )

    leave_one_out_frames.append(
        leave_one_out_result
    )

leave_one_out_results = pd.concat(
    leave_one_out_frames,
    ignore_index=True,
)

leave_one_out_results.to_csv(
    OUTPUTS
    / "run6_leave_one_seed_out_inference.csv",
    index=False,
)

leave_one_out_summary = (
    leave_one_out_results.groupby(
        "omitted_seed"
    )
    .agg(
        n_primary_comparisons=(
            "comparison",
            "size",
        ),
        n_pointwise_intervals_excluding_zero=(
            "pointwise_excludes_zero",
            "sum",
        ),
        n_simultaneous_intervals_excluding_zero=(
            "simultaneous_excludes_zero",
            "sum",
        ),
        minimum_estimate=(
            "estimate",
            "min",
        ),
        maximum_estimate=(
            "estimate",
            "max",
        ),
    )
    .reset_index()
)

leave_one_out_summary.to_csv(
    OUTPUTS
    / "run6_leave_one_seed_out_summary.csv",
    index=False,
)


# ============================================================================
# PART C: PATIENT-EQUAL-WEIGHTED SENSITIVITY
# ============================================================================

print("\n" + "=" * 100)
print("PART C: PATIENT-EQUAL-WEIGHTED SENSITIVITY")
print("=" * 100)

patient_equal_counts = generate_patient_bootstrap_counts(
    PATIENT_EQUAL_BOOTSTRAP_SEED
)

print(
    "Patient-equal bootstrap matrix:",
    f"{patient_equal_counts.nbytes / 1024**2:.1f} MB",
)

patient_equal_model_bootstrap, patient_equal_full_metrics = (
    compute_model_bootstrap_metrics(
        patient_counts=patient_equal_counts,
        base_image_weights=patient_equal_base_weights,
        analysis_label="patient_equal_weighted",
    )
)

patient_equal_results = build_run_aware_family(
    model_bootstrap=patient_equal_model_bootstrap,
    full_sample_metrics=patient_equal_full_metrics,
    run_resampling_seed=(
        PATIENT_EQUAL_BOOTSTRAP_SEED
        + 100_000
    ),
    analysis_name="patient_equal_weighted",
)

patient_equal_results[
    "patient_bootstrap_seed"
] = PATIENT_EQUAL_BOOTSTRAP_SEED

patient_equal_results.to_csv(
    OUTPUTS
    / "run6_patient_equal_weighted_inference.csv",
    index=False,
)

patient_equal_full_metrics.to_csv(
    OUTPUTS
    / "run6_patient_equal_weighted_full_metrics.csv",
    index=False,
)

saved_model_bootstraps[
    "patient_equal_weighted"
] = {
        model_id: {
            metric: values.astype(
                np.float32
            )
            for metric, values in (
                metric_dictionary.items()
            )
        }
        for model_id, metric_dictionary in (
            patient_equal_model_bootstrap.items()
        )
}

del patient_equal_counts
gc.collect()

if USE_GPU:
    cp.get_default_memory_pool().free_all_blocks()


# ============================================================================
# IMAGE-LEVEL VERSUS PATIENT-EQUAL COMPARISON
# ============================================================================

first_mc_results = monte_carlo_results[
    monte_carlo_results[
        "monte_carlo_index"
    ] == 1
].copy()

weighting_comparison = first_mc_results[
    [
        "comparison",
        "metric",
        "estimate",
        "pointwise_ci_low",
        "pointwise_ci_high",
        "simultaneous_ci_low",
        "simultaneous_ci_high",
    ]
].rename(
    columns={
        "estimate": (
            "image_level_estimate"
        ),
        "pointwise_ci_low": (
            "image_level_pointwise_ci_low"
        ),
        "pointwise_ci_high": (
            "image_level_pointwise_ci_high"
        ),
        "simultaneous_ci_low": (
            "image_level_simultaneous_ci_low"
        ),
        "simultaneous_ci_high": (
            "image_level_simultaneous_ci_high"
        ),
    }
)

weighting_comparison = weighting_comparison.merge(
    patient_equal_results[
        [
            "comparison",
            "metric",
            "estimate",
            "pointwise_ci_low",
            "pointwise_ci_high",
            "simultaneous_ci_low",
            "simultaneous_ci_high",
            "pointwise_excludes_zero",
            "simultaneous_excludes_zero",
        ]
    ].rename(
        columns={
            "estimate": (
                "patient_equal_estimate"
            ),
            "pointwise_ci_low": (
                "patient_equal_pointwise_ci_low"
            ),
            "pointwise_ci_high": (
                "patient_equal_pointwise_ci_high"
            ),
            "simultaneous_ci_low": (
                "patient_equal_simultaneous_ci_low"
            ),
            "simultaneous_ci_high": (
                "patient_equal_simultaneous_ci_high"
            ),
            "pointwise_excludes_zero": (
                "patient_equal_pointwise_excludes_zero"
            ),
            "simultaneous_excludes_zero": (
                "patient_equal_simultaneous_excludes_zero"
            ),
        }
    ),
    on=[
        "comparison",
        "metric",
    ],
    how="left",
)

weighting_comparison[
    "estimate_change_patient_equal_minus_image"
] = (
    weighting_comparison[
        "patient_equal_estimate"
    ]
    - weighting_comparison[
        "image_level_estimate"
    ]
)

weighting_comparison[
    "estimate_sign_changed"
] = (
    np.sign(
        weighting_comparison[
            "patient_equal_estimate"
        ]
    )
    != np.sign(
        weighting_comparison[
            "image_level_estimate"
        ]
    )
)

weighting_comparison.to_csv(
    OUTPUTS
    / "run6_image_vs_patient_equal_comparison.csv",
    index=False,
)


# ============================================================================
# SAVE MODEL-LEVEL BOOTSTRAP ARRAYS
# ============================================================================

np.savez_compressed(
    OUTPUTS
    / "run6_model_level_bootstrap_metrics.npz",
    bootstrap_metrics=np.array(
        saved_model_bootstraps,
        dtype=object,
    ),
    monte_carlo_seeds=np.asarray(
        MONTE_CARLO_SEEDS,
        dtype=np.int64,
    ),
    patient_equal_bootstrap_seed=np.asarray(
        PATIENT_EQUAL_BOOTSTRAP_SEED,
        dtype=np.int64,
    ),
    n_bootstrap=np.asarray(
        N_BOOTSTRAP,
        dtype=np.int64,
    ),
)


# ============================================================================
# FIGURES
# ============================================================================

print("\n" + "=" * 100)
print("PART D: FINAL ROBUSTNESS FIGURES")
print("=" * 100)


def save_figure(
    figure,
    filename_stem,
):
    figure.tight_layout()

    figure.savefig(
        OUTPUTS
        / f"{filename_stem}.pdf",
        bbox_inches="tight",
    )

    figure.savefig(
        OUTPUTS
        / f"{filename_stem}.png",
        dpi=300,
        bbox_inches="tight",
    )

    plt.close(figure)


# ----------------------------------------------------------------------------
# Figure 1: Monte Carlo interval stability
# ----------------------------------------------------------------------------

plot_data = monte_carlo_results.copy()

plot_data["display_label"] = (
    plot_data["comparison"]
    + " | "
    + plot_data["metric"].map(
        METRIC_LABELS
    )
)

display_labels = list(
    dict.fromkeys(
        plot_data[
            "display_label"
        ].tolist()
    )
)

figure, axis = plt.subplots(
    figsize=(10.5, 7.0)
)

offsets = np.linspace(
    -0.18,
    0.18,
    len(MONTE_CARLO_SEEDS),
)

for monte_carlo_index, offset in zip(
    range(
        1,
        len(MONTE_CARLO_SEEDS) + 1,
    ),
    offsets,
):
    subset = plot_data[
        plot_data[
            "monte_carlo_index"
        ] == monte_carlo_index
    ].copy()

    y_values = np.array(
        [
            display_labels.index(label)
            for label in subset[
                "display_label"
            ]
        ],
        dtype=float,
    ) + offset

    estimates = subset[
        "estimate"
    ].to_numpy()

    lower = subset[
        "pointwise_ci_low"
    ].to_numpy()

    upper = subset[
        "pointwise_ci_high"
    ].to_numpy()

    axis.errorbar(
        estimates,
        y_values,
        xerr=[
            estimates - lower,
            upper - estimates,
        ],
        fmt="o",
        capsize=3,
        label=(
            f"Monte Carlo seed "
            f"{MONTE_CARLO_SEEDS[monte_carlo_index - 1]}"
        ),
    )

axis.axvline(
    0,
    linestyle="--",
    linewidth=1,
)

axis.set_yticks(
    range(len(display_labels))
)

axis.set_yticklabels(
    display_labels
)

axis.set_xlabel(
    "Run-aware difference in AUROC "
    "(first method minus second method)"
)

axis.grid(
    axis="x",
    alpha=0.25,
)

axis.legend(
    frameon=False,
    fontsize=8,
)

save_figure(
    figure,
    "run6_monte_carlo_interval_stability",
)


# ----------------------------------------------------------------------------
# Figure 2: Image-level versus patient-equal weighting
# ----------------------------------------------------------------------------

comparison_plot = weighting_comparison.copy()

comparison_plot[
    "display_label"
] = (
    comparison_plot["comparison"]
    + " | "
    + comparison_plot["metric"].map(
        METRIC_LABELS
    )
)

comparison_plot = (
    comparison_plot.iloc[::-1]
    .reset_index(drop=True)
)

figure, axis = plt.subplots(
    figsize=(10.0, 7.0)
)

y_positions = np.arange(
    len(comparison_plot)
)

for label, offset, marker in [
    (
        "Image-level estimand",
        -0.12,
        "o",
    ),
    (
        "Patient-equal-weighted estimand",
        0.12,
        "s",
    ),
]:
    if label == "Image-level estimand":
        estimates = comparison_plot[
            "image_level_estimate"
        ].to_numpy()

        lower = comparison_plot[
            "image_level_simultaneous_ci_low"
        ].to_numpy()

        upper = comparison_plot[
            "image_level_simultaneous_ci_high"
        ].to_numpy()

    else:
        estimates = comparison_plot[
            "patient_equal_estimate"
        ].to_numpy()

        lower = comparison_plot[
            "patient_equal_simultaneous_ci_low"
        ].to_numpy()

        upper = comparison_plot[
            "patient_equal_simultaneous_ci_high"
        ].to_numpy()

    axis.errorbar(
        estimates,
        y_positions + offset,
        xerr=[
            estimates - lower,
            upper - estimates,
        ],
        fmt=marker,
        capsize=4,
        label=label,
    )

axis.axvline(
    0,
    linestyle="--",
    linewidth=1,
)

axis.set_yticks(
    y_positions
)

axis.set_yticklabels(
    comparison_plot[
        "display_label"
    ]
)

axis.set_xlabel(
    "Difference in AUROC "
    "(simultaneous familywise 95% interval)"
)

axis.grid(
    axis="x",
    alpha=0.25,
)

axis.legend(
    frameon=False,
)

save_figure(
    figure,
    "run6_image_vs_patient_equal_forest",
)


# ----------------------------------------------------------------------------
# Figure 3: Leave-one-seed-out simultaneous intervals
# ----------------------------------------------------------------------------

leave_plot = leave_one_out_results.copy()

leave_plot[
    "display_label"
] = (
    leave_plot["comparison"]
    + " | "
    + leave_plot["metric"].map(
        METRIC_LABELS
    )
)

leave_display_labels = list(
    dict.fromkeys(
        leave_plot[
            "display_label"
        ].tolist()
    )
)

figure, axis = plt.subplots(
    figsize=(10.7, 7.2)
)

leave_offsets = np.linspace(
    -0.24,
    0.24,
    len(TRAINING_SEEDS),
)

for omitted_seed, offset in zip(
    TRAINING_SEEDS,
    leave_offsets,
):
    subset = leave_plot[
        leave_plot[
            "omitted_seed"
        ] == omitted_seed
    ]

    y_values = np.array(
        [
            leave_display_labels.index(label)
            for label in subset[
                "display_label"
            ]
        ],
        dtype=float,
    ) + offset

    estimates = subset[
        "estimate"
    ].to_numpy()

    lower = subset[
        "simultaneous_ci_low"
    ].to_numpy()

    upper = subset[
        "simultaneous_ci_high"
    ].to_numpy()

    axis.errorbar(
        estimates,
        y_values,
        xerr=[
            estimates - lower,
            upper - estimates,
        ],
        fmt="o",
        capsize=3,
        label=f"Omit seed {int(omitted_seed)}",
    )

axis.axvline(
    0,
    linestyle="--",
    linewidth=1,
)

axis.set_yticks(
    range(len(leave_display_labels))
)

axis.set_yticklabels(
    leave_display_labels
)

axis.set_xlabel(
    "Difference in AUROC "
    "(leave-one-seed-out simultaneous 95% interval)"
)

axis.grid(
    axis="x",
    alpha=0.25,
)

axis.legend(
    frameon=False,
    fontsize=8,
)

save_figure(
    figure,
    "run6_leave_one_seed_out_forest",
)


# ============================================================================
# DECISION SUMMARY
# ============================================================================

monte_carlo_decision_rows = (
    monte_carlo_results.groupby(
        [
            "monte_carlo_index",
            "patient_bootstrap_seed",
        ]
    )
    .agg(
        n_comparisons=(
            "comparison",
            "size",
        ),
        n_pointwise_intervals_excluding_zero=(
            "pointwise_excludes_zero",
            "sum",
        ),
        n_simultaneous_intervals_excluding_zero=(
            "simultaneous_excludes_zero",
            "sum",
        ),
    )
    .reset_index()
)

patient_equal_decision = pd.DataFrame(
    [
        {
            "n_comparisons": int(
                len(patient_equal_results)
            ),
            "n_pointwise_intervals_excluding_zero": int(
                patient_equal_results[
                    "pointwise_excludes_zero"
                ].sum()
            ),
            "n_simultaneous_intervals_excluding_zero": int(
                patient_equal_results[
                    "simultaneous_excludes_zero"
                ].sum()
            ),
            "n_estimate_sign_changes_vs_image_level": int(
                weighting_comparison[
                    "estimate_sign_changed"
                ].sum()
            ),
        }
    ]
)

monte_carlo_decision_rows.to_csv(
    OUTPUTS
    / "run6_monte_carlo_decision_summary.csv",
    index=False,
)

patient_equal_decision.to_csv(
    OUTPUTS
    / "run6_patient_equal_decision_summary.csv",
    index=False,
)

print("\n" + "=" * 100)
print("RUN 6 DECISION SUMMARY")
print("=" * 100)

print("\nMonte Carlo stability:")

print(
    monte_carlo_decision_rows.to_string(
        index=False
    )
)

print("\nMonte Carlo interval endpoint ranges:")

print(
    monte_carlo_stability[
        [
            "comparison",
            "metric",
            "estimate",
            "pointwise_ci_low_min",
            "pointwise_ci_low_max",
            "pointwise_ci_high_min",
            "pointwise_ci_high_max",
            "n_pointwise_resolved",
            "n_simultaneous_resolved",
        ]
    ].to_string(
        index=False,
        float_format=lambda value: (
            f"{value:+.6f}"
        ),
    )
)

print("\nLeave-one-seed-out stability:")

print(
    leave_one_out_summary.to_string(
        index=False
    )
)

print("\nPatient-equal-weighted sensitivity:")

print(
    patient_equal_results[
        [
            "comparison",
            "metric",
            "estimate",
            "pointwise_ci_low",
            "pointwise_ci_high",
            "simultaneous_ci_low",
            "simultaneous_ci_high",
            "pointwise_excludes_zero",
            "simultaneous_excludes_zero",
        ]
    ].to_string(
        index=False,
        float_format=lambda value: (
            f"{value:+.6f}"
        ),
    )
)

print("\nImage-level versus patient-equal estimates:")

print(
    weighting_comparison[
        [
            "comparison",
            "metric",
            "image_level_estimate",
            "patient_equal_estimate",
            "estimate_change_patient_equal_minus_image",
            "estimate_sign_changed",
        ]
    ].to_string(
        index=False,
        float_format=lambda value: (
            f"{value:+.6f}"
        ),
    )
)


# ============================================================================
# METADATA
# ============================================================================

metadata = {
    "n_images": int(n_images),
    "n_unique_patients": int(n_patients),
    "n_bootstrap_per_analysis": int(
        N_BOOTSTRAP
    ),
    "monte_carlo_seeds": (
        MONTE_CARLO_SEEDS
    ),
    "patient_equal_bootstrap_seed": int(
        PATIENT_EQUAL_BOOTSTRAP_SEED
    ),
    "training_seeds": (
        TRAINING_SEEDS
    ),
    "primary_methods": (
        CORE_METHODS
    ),
    "primary_comparisons": [
        f"{method_a} minus {method_b}"
        for method_a, method_b in (
            PRIMARY_COMPARISONS
        )
    ],
    "primary_metrics": (
        PRIMARY_METRICS
    ),
    "primary_estimand": (
        "Image-level AUROC with uncertainty clustered by unique NIH "
        "patient and independent resampling of training runs across methods"
    ),
    "patient_equal_sensitivity_estimand": (
        "Each NIH patient has total weight one, divided equally among "
        "that patient's radiographs; uncertainty remains patient-clustered"
    ),
    "leave_one_seed_out": (
        "Each numeric training seed is omitted jointly from LPR, CADR, "
        "and DWFA, leaving four runs per method"
    ),
    "multiplicity": (
        "Single-step max-t simultaneous 95% intervals over three "
        "comparisons times two discrimination metrics"
    ),
    "run_resampling": (
        "Independent across methods; the same run-index matrix is reused "
        "across metrics within each method"
    ),
    "gpu_used": bool(USE_GPU),
    "gpu_batch_size": int(
        GPU_BATCH_SIZE
    ),
}

with open(
    OUTPUTS / "run6_metadata.json",
    "w",
    encoding="utf-8",
) as file:
    json.dump(
        metadata,
        file,
        indent=2,
    )


# ============================================================================
# FINAL ZIP
# ============================================================================

local_zip = Path(
    "/content/revision_run6.zip"
)

with zipfile.ZipFile(
    local_zip,
    "w",
    compression=zipfile.ZIP_DEFLATED,
) as archive:
    for output_path in sorted(
        OUTPUTS.rglob("*")
    ):
        if output_path.is_file():
            archive.write(
                output_path,
                arcname=output_path.relative_to(
                    OUTPUTS
                ),
            )

drive_zip = RESULTS / "revision_run6.zip"

shutil.copy2(
    local_zip,
    drive_zip,
)

print("\n" + "=" * 100)
print("RUN 6 COMPLETE")
print("=" * 100)
print("Local ZIP:", local_zip)
print("Drive ZIP:", drive_zip)
print(
    "\nUpload revision_run6.zip and a screenshot of the "
    "RUN 6 DECISION SUMMARY."
)

Mounted at /content/drive
RUN 6: FINAL ROBUSTNESS ANALYSIS
Bootstrap replicates per analysis: 5,000
Monte Carlo seeds: [20260726, 20260817, 20260911]
Training seeds: [42, 123, 456, 789, 1010]
NIH radiographs:          112,120
Unique NIH patients:      30,805
Prediction arrays found:  18

Primary models:
  LPR: ['LPR|42', 'LPR|123', 'LPR|456', 'LPR|789', 'LPR|1010']
  CADR: ['CADR|42', 'CADR|123', 'CADR|456', 'CADR|789', 'CADR|1010']
  DWFA: ['DWFA|42', 'DWFA|123', 'DWFA|456', 'DWFA|789', 'DWFA|1010']

Weighted-AUROC backend: CuPy GPU (Tesla T4)

Patient-equal sensitivity weights verified: each patient contributes total weight 1.

Precomputing model score-order structures...
  [01/15] DWFA|42
  [02/15] DWFA|123
  [03/15] DWFA|456
  [04/15] DWFA|789
  [05/15] DWFA|1010
  [06/15] CADR|42
  [07/15] CADR|123
  [08/15] CADR|456
  [09/15] CADR|789
  [10/15] CADR|1010
  [11/15] LPR|42
  [12/15] LPR|123
  [13/15] LPR|456
  [14/15] LPR|789
  [15/15] LPR|1010

Weighted-AUROC implementation self-t

In [ ]:
#CELL 1: GPU INFERENCE-ONLY REVISION CELL

# ============================================================================
# NEW REVISION GPU CELL — INFERENCE-ONLY EXTERNAL BASELINE EXPANSION
#
# PURPOSE
#   Use EXISTING trained checkpoints only. NO RETRAINING.
#   Spend the limited T4 time on missing NIH external predictions for:
#     1) FedProx
#     2) Centralized training
#     3) Local-only Client A/B/C/D models
#
# BEFORE RUNNING THIS NEW CELL
#   REQUIRED OLD CELL IN THIS SESSION:
#     "# external validation"
#     "#NIH CELL 1 — Download + unzip NIH ChestX-ray14 (224x224 resized mirror)"
#       -> this stages NIH images on /content and defines NIH_IMG.
#
#   ONLY IF results/nih_effusion_cohort.csv IS MISSING, ALSO RUN:
#     "# NIH CELL 2 — Build the effusion cohort CSV in YOUR schema (CPU only, ~0 units)"
#
#   DO NOT RUN ANY OLD TRAINING CELL.
#   You do NOT need to rerun FedProx, centralized, or local-only training.
#
# SAFE / RESUMABLE
#   Predictions are saved after EVERY checkpoint. If Colab disconnects or the
#   time budget is reached, rerun this same cell on another T4 session.
#
# OUTPUTS
#   results/nih_extended_baselines_cache.npz
#   results/nih_extended_baselines_gpu_progress.csv
# ============================================================================

from google.colab import drive
drive.mount('/content/drive', force_remount=False)

from pathlib import Path
import os, time, gc, shutil
import numpy as np
import pandas as pd
import torch
import torchvision.models as models
from torchvision import transforms
from torch.utils.data import Dataset, DataLoader
from PIL import Image

# ----------------------------- configuration ---------------------------------
BASE = Path('/content/drive/MyDrive/FairFedCXR')
RESULTS = BASE / 'results'
CKPTS = BASE / 'checkpoints'
NIH_CSV = RESULTS / 'nih_effusion_cohort.csv'

OUT_CACHE = RESULTS / 'nih_extended_baselines_cache.npz'
OUT_PROGRESS = RESULTS / 'nih_extended_baselines_gpu_progress.csv'

GPU_HOUR_BUDGET = 2.75       # hard stop between models; safe to rerun tomorrow
INFER_BATCH_SIZE = 128       # safe starting point for a 16-GB T4
NUM_WORKERS = 4
INCLUDE_LOCAL_ONLY = True    # set False if you only want FedProx + Centralized first

if not torch.cuda.is_available():
    raise RuntimeError('GPU not detected. Switch Colab runtime to T4 GPU before running this cell.')

device = torch.device('cuda')
print('GPU:', torch.cuda.get_device_name(0))
print('Time budget:', GPU_HOUR_BUDGET, 'hours')

if not NIH_CSV.exists():
    raise FileNotFoundError(
        f'{NIH_CSV}\n'
        'Run the old cell named:\n'
        '"# NIH CELL 2 — Build the effusion cohort CSV in YOUR schema (CPU only, ~0 units)"'
    )

# NIH_IMG should have been defined by the old NIH CELL 1.
if 'NIH_IMG' not in globals() or not os.path.isdir(str(NIH_IMG)):
    raise RuntimeError(
        'NIH_IMG is not available in this session.\n'
        'Run the old cell named:\n'
        '"#NIH CELL 1 — Download + unzip NIH ChestX-ray14 (224x224 resized mirror)"'
    )

NIH_IMG = str(NIH_IMG)
cohort = pd.read_csv(NIH_CSV, low_memory=False).reset_index(drop=True)

required = {'Path', 'label', 'sex_encoded'}
missing = required - set(cohort.columns)
if missing:
    raise KeyError(f'NIH cohort missing columns: {sorted(missing)}')

cohort['label'] = cohort['label'].astype(np.int8)
cohort['sex_encoded'] = cohort['sex_encoded'].astype(np.int8)

probe_paths = [os.path.join(NIH_IMG, str(p)) for p in cohort['Path'].head(10)]
if not any(os.path.exists(p) for p in probe_paths):
    raise FileNotFoundError(
        'NIH_IMG does not point to the folder containing the cohort images.\n'
        f'Current NIH_IMG = {NIH_IMG}\n'
        'Rerun old NIH CELL 1 and inspect its printed NIH_IMG path.'
    )

# ------------------------------- data loader ---------------------------------
eval_tf_revision = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406],
                         [0.229, 0.224, 0.225]),
])

class RevisionNihDataset(Dataset):
    def __init__(self, frame, image_root, transform):
        self.df = frame.reset_index(drop=True)
        self.root = image_root
        self.tf = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        path = os.path.join(self.root, str(row['Path']))
        img = Image.open(path).convert('RGB')
        return (
            self.tf(img),
            torch.tensor(row['label'], dtype=torch.float32),
            torch.tensor(row['sex_encoded'], dtype=torch.long),
        )

nih_loader = DataLoader(
    RevisionNihDataset(cohort, NIH_IMG, eval_tf_revision),
    batch_size=INFER_BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=True,
    persistent_workers=True,
)

def build_revision_model():
    model = models.densenet121(weights=None)
    model.classifier = torch.nn.Linear(model.classifier.in_features, 1)
    return model

def extract_state_dict(obj):
    if isinstance(obj, dict):
        for key in ['state_dict', 'model_state_dict', 'global_state', 'gs', 'model']:
            if key in obj and isinstance(obj[key], dict):
                obj = obj[key]
                break
    if not isinstance(obj, dict):
        raise TypeError('Checkpoint does not contain a recognizable state_dict.')
    if any(str(k).startswith('module.') for k in obj.keys()):
        obj = {str(k).replace('module.', '', 1): v for k, v in obj.items()}
    return obj

@torch.inference_mode()
def infer_one_checkpoint(ckpt_path):
    model = build_revision_model()
    raw = torch.load(ckpt_path, map_location='cpu', weights_only=False)
    state = extract_state_dict(raw)
    missing_keys, unexpected_keys = model.load_state_dict(state, strict=False)

    if len(missing_keys) > 2 or len(unexpected_keys) > 2:
        raise RuntimeError(
            f'Checkpoint mismatch for {ckpt_path.name}: '
            f'missing={missing_keys[:8]}, unexpected={unexpected_keys[:8]}'
        )

    model = model.to(device).eval()

    probs, labels, sex = [], [], []
    for images, y, s in nih_loader:
        images = images.to(device, non_blocking=True)
        with torch.amp.autocast(device_type='cuda', dtype=torch.float16):
            logits = model(images).squeeze(1)
        probs.append(torch.sigmoid(logits.float()).cpu().numpy().astype(np.float32))
        labels.append(y.numpy().astype(np.int8))
        sex.append(s.numpy().astype(np.int8))

    del model, raw, state
    gc.collect()
    torch.cuda.empty_cache()

    return np.concatenate(probs), np.concatenate(labels), np.concatenate(sex)

# ----------------------- load resumable prediction cache ----------------------
if OUT_CACHE.exists():
    old = np.load(OUT_CACHE, allow_pickle=True)
    all_probs = old['all_probs'].item()
    saved_labels = old['labels'].astype(np.int8)
    saved_sex = old['sex'].astype(np.int8)
    print(f'Resuming cache with {len(all_probs)} completed checkpoint predictions.')
else:
    all_probs = {}
    saved_labels = None
    saved_sex = None

def save_cache():
    local_tmp = Path('/content/nih_extended_baselines_cache.npz')
    np.savez_compressed(
        local_tmp,
        all_probs=all_probs,
        labels=saved_labels,
        sex=saved_sex,
    )
    shutil.copy2(local_tmp, OUT_CACHE)

# --------------------------- checkpoint priority ------------------------------
jobs = []

for seed in [42, 123, 456]:
    jobs.append(('fedprox', seed, CKPTS / f'fedprox_seed{seed}.pt'))
for seed in [42, 123, 456]:
    jobs.append(('centralized', seed, CKPTS / f'centralized_seed{seed}.pt'))

if INCLUDE_LOCAL_ONLY:
    for client in ['A', 'B', 'C', 'D']:
        for seed in [42, 123, 456]:
            jobs.append((f'local_{client}', seed, CKPTS / f'local_{client}_seed{seed}.pt'))

print('\nPlanned inference jobs:')
for method, seed, path in jobs:
    status = 'cached' if (method, seed) in all_probs else ('ready' if path.exists() else 'MISSING CHECKPOINT')
    print(f'  {method:12s} seed {seed:<4} -> {status}')

# ------------------------------- run inference --------------------------------
start = time.time()
progress_rows = []

for job_index, (method, seed, ckpt) in enumerate(jobs, start=1):
    elapsed_hr = (time.time() - start) / 3600.0

    if elapsed_hr >= GPU_HOUR_BUDGET:
        print(f'\nTime budget reached ({elapsed_hr:.2f} h). Stopping safely between models.')
        break

    if (method, seed) in all_probs:
        continue

    if not ckpt.exists():
        progress_rows.append({
            'method': method, 'seed': seed, 'status': 'missing_checkpoint',
            'minutes': np.nan, 'checkpoint': str(ckpt)
        })
        continue

    print(f'\n[{job_index}/{len(jobs)}] {method} seed {seed}')
    print('checkpoint:', ckpt.name)

    t0 = time.time()
    p, y, s = infer_one_checkpoint(ckpt)
    minutes = (time.time() - t0) / 60.0

    if saved_labels is None:
        saved_labels = y.copy()
        saved_sex = s.copy()
    else:
        if not np.array_equal(saved_labels, y):
            raise AssertionError(f'Label-order mismatch at {method} seed {seed}.')
        if not np.array_equal(saved_sex, s):
            raise AssertionError(f'Sex-order mismatch at {method} seed {seed}.')

    if len(p) != len(cohort):
        raise AssertionError(f'Prediction length mismatch at {method} seed {seed}.')

    all_probs[(method, int(seed))] = p
    save_cache()

    progress_rows.append({
        'method': method, 'seed': int(seed), 'status': 'completed',
        'minutes': minutes, 'checkpoint': str(ckpt)
    })
    pd.DataFrame(progress_rows).to_csv(OUT_PROGRESS, index=False)

    print(f'completed in {minutes:.1f} min | cached models now: {len(all_probs)}')
    print(f'elapsed session time: {(time.time()-start)/3600:.2f} h')

save_cache()
if progress_rows:
    pd.DataFrame(progress_rows).to_csv(OUT_PROGRESS, index=False)

print('\n' + '='*80)
print('GPU INFERENCE CELL FINISHED / PAUSED SAFELY')
print('='*80)
print('Cache:', OUT_CACHE)
print('Progress:', OUT_PROGRESS)
print('Cached keys:')
for key in sorted(all_probs.keys(), key=lambda x: (x[0], x[1])):
    print(' ', key)

remaining = [(m,s,str(p)) for m,s,p in jobs if (m,s) not in all_probs and p.exists()]
if remaining:
    print(f'\n{len(remaining)} existing checkpoints remain. Rerun THIS SAME NEW GPU CELL next T4 session.')
else:
    print('\nAll available requested checkpoints are cached. No more GPU inference is needed.')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
GPU: Tesla T4
Time budget: 2.75 hours

Planned inference jobs:
  fedprox      seed 42   -> ready
  fedprox      seed 123  -> ready
  fedprox      seed 456  -> ready
  centralized  seed 42   -> ready
  centralized  seed 123  -> ready
  centralized  seed 456  -> ready
  local_A      seed 42   -> ready
  local_A      seed 123  -> ready
  local_A      seed 456  -> ready
  local_B      seed 42   -> ready
  local_B      seed 123  -> ready
  local_B      seed 456  -> ready
  local_C      seed 42   -> ready
  local_C      seed 123  -> ready
  local_C      seed 456  -> ready
  local_D      seed 42   -> ready
  local_D      seed 123  -> ready
  local_D      seed 456  -> ready

[1/18] fedprox seed 42
checkpoint: fedprox_seed42.pt


/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:424: UserWarning: This DataLoader will create 4 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_rationality()
/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:432: UserWarning: This DataLoader will create 4 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_rationality()


completed in 6.0 min | cached models now: 1
elapsed session time: 0.10 h

[2/18] fedprox seed 123
checkpoint: fedprox_seed123.pt
completed in 4.7 min | cached models now: 2
elapsed session time: 0.18 h

[3/18] fedprox seed 456
checkpoint: fedprox_seed456.pt
completed in 4.4 min | cached models now: 3
elapsed session time: 0.25 h

[4/18] centralized seed 42
checkpoint: centralized_seed42.pt
completed in 4.4 min | cached models now: 4
elapsed session time: 0.33 h

[5/18] centralized seed 123
checkpoint: centralized_seed123.pt
completed in 4.6 min | cached models now: 5
elapsed session time: 0.40 h

[6/18] centralized seed 456
checkpoint: centralized_seed456.pt
completed in 4.5 min | cached models now: 6
elapsed session time: 0.48 h

[7/18] local_A seed 42
checkpoint: local_A_seed42.pt
completed in 4.5 min | cached models now: 7
elapsed session time: 0.55 h

[8/18] local_A seed 123
checkpoint: local_A_seed123.pt
completed in 4.5 min | cached models now: 8
elapsed session time: 0.63 h

[9/

In [ ]:
#CELL 2: CPU REVIEWER-POLISH CELL

# ============================================================================
# NEW REVISION CPU CELL — REVIEWER-FOCUSED NO-RETRAINING ANALYSES
#
# PURPOSE
#   Do every high-value analysis that is valid WITHOUT retraining:
#     A) extended NIH baseline metrics from existing checkpoints/predictions
#     B) post-hoc DWFA WEIGHT-RESPONSE parameter sensitivity
#     C) counterfactual aggregation-component diagnostics
#     D) exhaustive 3/4/5-seed subset stability analysis
#     E) exploratory run-count precision analysis
#     F) 37-threshold fairness summary
#     G) claim-to-evidence table
#
# BEFORE RUNNING THIS NEW CELL
#   NO OLD TRAINING CELL NEEDS TO BE RERUN.
#
#   This cell expects files already produced by your completed notebook:
#     - results/dwfa_round_log.csv
#         produced by old cell "# DWFA day 8,7"
#     - results/nih_pred_cache.npz
#         produced/updated by the old paired NIH cache and later by:
#         "# COMPLETE POST-TRAINING EVALUATION — run this ONE cell after FairFed's full..."
#     - clients/client_[A-D]_val.csv
#
#   OPTIONAL BUT STRONGLY RECOMMENDED:
#     Run the NEW REVISION GPU CELL above first. If
#     results/nih_extended_baselines_cache.npz exists, this CPU cell will add
#     FedProx, Centralized, and available Local-only external results.
#
# IMPORTANT INTERPRETATION
#   The parameter/component analyses below are WEIGHT-LEVEL sensitivity
#   CONDITIONAL ON THE OBSERVED TRAINING TRAJECTORY. They DO NOT estimate how
#   AUROC/fairness would change after retraining with alternative weights.
#
# OUTPUT
#   results/reviewer_polish_cpu.zip
#   plus the individual CSV/PNG/TXT files in results/reviewer_polish_cpu/
# ============================================================================

from google.colab import drive
drive.mount('/content/drive', force_remount=False)

from pathlib import Path
from itertools import combinations
import os, math, shutil, zipfile
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from scipy.special import logit
from scipy.optimize import minimize
from scipy.stats import t as student_t
from sklearn.metrics import roc_auc_score, average_precision_score, brier_score_loss

BASE = Path('/content/drive/MyDrive/FairFedCXR')
RESULTS = BASE / 'results'
CLIENTS = BASE / 'clients'

DWFA_LOG = RESULTS / 'dwfa_round_log.csv'
MAIN_CACHE = RESULTS / 'nih_pred_cache.npz'
EXT_CACHE = RESULTS / 'nih_extended_baselines_cache.npz'
NIH_CSV = RESULTS / 'nih_effusion_cohort.csv'

OUTDIR = RESULTS / 'reviewer_polish_cpu'
ZIP_PATH = RESULTS / 'reviewer_polish_cpu.zip'

if OUTDIR.exists():
    shutil.rmtree(OUTDIR)
OUTDIR.mkdir(parents=True, exist_ok=True)

required = [DWFA_LOG, MAIN_CACHE, NIH_CSV]
missing = [str(p) for p in required if not p.exists()]
if missing:
    raise FileNotFoundError(
        'Missing required files:\n  ' + '\n  '.join(missing) +
        '\nDo NOT retrain. Restore the saved result files from Drive.'
    )

EPS = 1.0
THRESHOLDS = np.round(np.arange(0.05, 0.951, 0.025), 3)

def safe_auroc(y, p):
    y = np.asarray(y); p = np.asarray(p)
    if len(np.unique(y)) < 2:
        return np.nan
    return float(roc_auc_score(y, p))

def safe_auprc(y, p):
    y = np.asarray(y); p = np.asarray(p)
    if y.sum() == 0:
        return np.nan
    return float(average_precision_score(y, p))

def smoothed_tpr(p, y, thr):
    pred = (p >= thr).astype(np.int8)
    pos = (y == 1)
    return float(((pred == 1) & pos).sum() + EPS) / float(pos.sum() + 2*EPS)

def smoothed_fpr(p, y, thr):
    pred = (p >= thr).astype(np.int8)
    neg = (y == 0)
    return float(((pred == 1) & neg).sum() + EPS) / float(neg.sum() + 2*EPS)

def eo_gap(p, y, sex, thr=0.5):
    m = (sex == 1); f = (sex == 0)
    return abs(smoothed_tpr(p[m], y[m], thr) - smoothed_tpr(p[f], y[f], thr))

def fpr_gap(p, y, sex, thr=0.5):
    m = (sex == 1); f = (sex == 0)
    return abs(smoothed_fpr(p[m], y[m], thr) - smoothed_fpr(p[f], y[f], thr))

def ece_equal_width(p, y, bins=8):
    p = np.asarray(p); y = np.asarray(y)
    edges = np.linspace(0.0, 1.0, bins + 1)
    out = 0.0
    for i, (lo, hi) in enumerate(zip(edges[:-1], edges[1:])):
        if i == bins - 1:
            mask = (p >= lo) & (p <= hi)
        else:
            mask = (p >= lo) & (p < hi)
        if mask.sum() == 0:
            continue
        out += mask.mean() * abs(y[mask].mean() - p[mask].mean())
    return float(out)

def calibration_intercept_slope(p, y):
    p = np.clip(np.asarray(p, dtype=float), 1e-6, 1 - 1e-6)
    y = np.asarray(y, dtype=float)
    x = logit(p)
    xm, xs = x.mean(), x.std()
    if xs <= 0:
        return np.nan, np.nan, False
    z = (x - xm) / xs

    def objective(theta):
        a, b = theta
        eta = np.clip(a + b*z, -35, 35)
        nll = np.logaddexp(0.0, eta).sum() - (y * eta).sum()
        return float(nll + 1e-10*(a*a + b*b))

    best = None
    for start in [(0.0, 1.0), (0.0, 0.5), (-1.0, 1.0)]:
        fit = minimize(objective, x0=np.array(start), method='L-BFGS-B',
                       options={'maxiter': 5000})
        if best is None or fit.fun < best.fun:
            best = fit

    a_z, b_z = best.x
    slope = float(b_z / xs)
    intercept = float(a_z - b_z*xm/xs)
    return intercept, slope, bool(best.success)

def full_metrics(p, y, sex):
    m = (sex == 1); f = (sex == 0)
    male_auc = safe_auroc(y[m], p[m])
    female_auc = safe_auroc(y[f], p[f])
    male_ap = safe_auprc(y[m], p[m])
    female_ap = safe_auprc(y[f], p[f])
    cal_i, cal_s, cal_ok = calibration_intercept_slope(p, y)
    return {
        'auroc': safe_auroc(y, p),
        'male_auroc': male_auc,
        'female_auroc': female_auc,
        'worst_group_auroc': float(np.nanmin([male_auc, female_auc])),
        'auprc': safe_auprc(y, p),
        'male_auprc': male_ap,
        'female_auprc': female_ap,
        'worst_group_auprc': float(np.nanmin([male_ap, female_ap])),
        'eo_gap_0.5': eo_gap(p, y, sex, 0.5),
        'fpr_gap_0.5': fpr_gap(p, y, sex, 0.5),
        'brier': float(brier_score_loss(y, p)),
        'ece_8bin': ece_equal_width(p, y, 8),
        'calibration_intercept': cal_i,
        'calibration_slope': cal_s,
        'calibration_converged': cal_ok,
    }

# ============================================================================
# A. EXTENDED NIH BASELINE TABLE
# ============================================================================
nih = pd.read_csv(NIH_CSV, low_memory=False).reset_index(drop=True)
y = nih['label'].to_numpy(dtype=np.int8)
sex = nih['sex_encoded'].to_numpy(dtype=np.int8)

main = np.load(MAIN_CACHE, allow_pickle=True)
main_probs = main['all_probs'].item()

if not np.array_equal(main['labels'].astype(np.int8), y):
    raise AssertionError('Main NIH cache label order does not match nih_effusion_cohort.csv.')
if not np.array_equal(main['sex'].astype(np.int8), sex):
    raise AssertionError('Main NIH cache sex order does not match nih_effusion_cohort.csv.')

all_probs = dict(main_probs)

if EXT_CACHE.exists():
    ext = np.load(EXT_CACHE, allow_pickle=True)
    if not np.array_equal(ext['labels'].astype(np.int8), y):
        raise AssertionError('Extended cache labels do not match NIH cohort.')
    if not np.array_equal(ext['sex'].astype(np.int8), sex):
        raise AssertionError('Extended cache sex does not match NIH cohort.')
    all_probs.update(ext['all_probs'].item())
    print(f'Loaded optional extended GPU cache: {EXT_CACHE.name}')
else:
    print('Extended GPU cache not found. Continuing with the existing main NIH cache only.')

METHOD_LABEL = {
    'fedavg': 'FedAvg',
    'fedprox': 'FedProx',
    'centralized': 'Centralized',
    'qfedavg': 'LPR',
    'fairfed': 'CADR',
    'dwfa': 'DWFA',
    'local_A': 'Local-only A',
    'local_B': 'Local-only B',
    'local_C': 'Local-only C',
    'local_D': 'Local-only D',
}

external_rows = []
for key, p in sorted(all_probs.items(), key=lambda kv: (str(kv[0][0]), int(kv[0][1]))):
    method, seed = key
    if method not in METHOD_LABEL:
        continue
    p = np.asarray(p, dtype=np.float32)
    if len(p) != len(y):
        print('Skipping length-mismatched cache entry:', key)
        continue
    row = {'cache_method': method, 'method': METHOD_LABEL[method], 'seed': int(seed)}
    row.update(full_metrics(p, y, sex))
    external_rows.append(row)

external_per_run = pd.DataFrame(external_rows)
external_per_run.to_csv(OUTDIR / 'extended_external_metrics_per_run.csv', index=False)

numeric_cols = [c for c in external_per_run.columns
                if c not in ['cache_method', 'method', 'seed', 'calibration_converged']]
external_summary = external_per_run.groupby('method')[numeric_cols].agg(['mean', 'std'])
external_summary.columns = [f'{a}_{b}' for a, b in external_summary.columns]
external_summary = external_summary.reset_index()
external_summary.to_csv(OUTDIR / 'extended_external_metrics_summary.csv', index=False)

# ============================================================================
# B. DWFA POST-HOC WEIGHT-RESPONSE PARAMETER SENSITIVITY
# ============================================================================
dwfa_log = pd.read_csv(DWFA_LOG)
needed_cols = {'seed', 'round', 'client', 'val_eo_gap_local', 'fedavg_weight', 'weight'}
missing_cols = needed_cols - set(dwfa_log.columns)
if missing_cols:
    raise KeyError(f'dwfa_round_log.csv missing columns: {sorted(missing_cols)}')

val_support = {}
for client in ['A', 'B', 'C', 'D']:
    dfv = pd.read_csv(CLIENTS / f'client_{client}_val.csv')
    m_pos = int(((dfv['sex_encoded'] == 1) & (dfv['label'] == 1)).sum())
    f_pos = int(((dfv['sex_encoded'] == 0) & (dfv['label'] == 1)).sum())
    val_support[client] = {'male_pos': m_pos, 'female_pos': f_pos,
                           'minority_pos': min(m_pos, f_pos)}

pd.DataFrame([
    {'client': c, **v} for c, v in val_support.items()
]).to_csv(OUTDIR / 'dwfa_validation_subgroup_support.csv', index=False)

def reconstruct_counterfactual_weights(g_ref=0.05, rho=0.10, tau_ref=200,
                                       kappa=0.60, mode='full'):
    """
    Recompute weights from the SAVED observed local-gap trajectory.

    mode:
      full          = original confidence-gated historical rule
      current_only  = use current raw equity only (r=1 behavior)
      history_only  = after round 1 use prior raw-equity mean only
      fedavg        = no adaptive multiplier; exact FedAvg normalized prior

    This is NOT a retrained performance counterfactual.
    """
    alpha = 1.0 - float(kappa)
    x = dwfa_log[['seed', 'round', 'client', 'val_eo_gap_local',
                  'fedavg_weight', 'weight']].copy()
    x = x.sort_values(['seed', 'client', 'round']).reset_index(drop=True)

    out_parts = []

    for (seed, client), g in x.groupby(['seed', 'client'], sort=False):
        hist = []
        minority_pos = val_support[str(client)]['minority_pos']
        r = float(np.clip(minority_pos / float(tau_ref), rho, 1.0))

        gg = g.copy()
        e_list, et_list, r_list = [], [], []

        for _, row in gg.iterrows():
            gap = float(row['val_eo_gap_local'])
            e_cur = float(np.clip(1.0 - gap/float(g_ref), rho, 1.0))

            if mode == 'fedavg':
                etilde = 1.0
            elif mode == 'current_only':
                etilde = e_cur
            elif mode == 'history_only':
                etilde = e_cur if len(hist) == 0 else float(np.mean(hist))
            elif mode == 'full':
                ebar_prior = e_cur if len(hist) == 0 else float(np.mean(hist))
                etilde = float((1.0-r)*ebar_prior + r*e_cur)
            else:
                raise ValueError(mode)

            hist.append(e_cur)
            e_list.append(e_cur)
            et_list.append(etilde)
            r_list.append(r)

        gg['e_counterfactual'] = e_list
        gg['etilde_counterfactual'] = et_list
        gg['r_counterfactual'] = r_list
        out_parts.append(gg)

    z = pd.concat(out_parts, ignore_index=True)

    if mode == 'fedavg':
        z['q_counterfactual'] = 1.0
    else:
        z['q_counterfactual'] = alpha + float(kappa)*z['etilde_counterfactual']

    z['score_counterfactual'] = z['fedavg_weight'] * z['q_counterfactual']
    denom = z.groupby(['seed', 'round'])['score_counterfactual'].transform('sum')
    z['weight_counterfactual'] = z['score_counterfactual'] / denom
    z['weight_ratio_to_prior'] = z['weight_counterfactual'] / z['fedavg_weight']
    z['abs_delta_from_prior'] = (z['weight_counterfactual'] - z['fedavg_weight']).abs()
    return z

default_recon = reconstruct_counterfactual_weights()
recon_error = float(np.max(np.abs(default_recon['weight_counterfactual'] -
                                  default_recon['weight'])))
print(f'DWFA default weight reconstruction max abs error: {recon_error:.3e}')
if recon_error > 1e-8:
    raise AssertionError(
        'Default weight reconstruction does not match the real log. '
        'Stop here before interpreting sensitivity results.'
    )

GREF_GRID = [0.025, 0.05, 0.075, 0.10]
RHO_GRID = [0.05, 0.10, 0.20]
TAU_GRID = [100, 200, 400]
KAPPA_GRID = [0.30, 0.60]

sens_summary = []
sens_client = []

for gref in GREF_GRID:
    for rho in RHO_GRID:
        for tau in TAU_GRID:
            for kappa in KAPPA_GRID:
                z = reconstruct_counterfactual_weights(
                    g_ref=gref, rho=rho, tau_ref=tau, kappa=kappa, mode='full'
                )
                eff = z.groupby(['seed', 'round'])['weight_counterfactual'].apply(
                    lambda s: 1.0 / np.square(s.to_numpy()).sum()
                ).to_numpy()

                tag = f'G{gref}_rho{rho}_tau{tau}_k{kappa}'
                sens_summary.append({
                    'setting': tag,
                    'g_ref': gref, 'rho': rho, 'tau_ref': tau,
                    'kappa': kappa, 'alpha': 1-kappa,
                    'mean_abs_weight_delta_from_fedavg': z['abs_delta_from_prior'].mean(),
                    'max_abs_weight_delta_from_fedavg': z['abs_delta_from_prior'].max(),
                    'mean_effective_clients': np.mean(eff),
                    'min_effective_clients': np.min(eff),
                    'min_weight_to_prior_ratio': z['weight_ratio_to_prior'].min(),
                    'max_weight_to_prior_ratio': z['weight_ratio_to_prior'].max(),
                    'equity_floor_rate': np.mean(np.isclose(z['e_counterfactual'], rho)),
                    'reliability_floor_rate': np.mean(np.isclose(z['r_counterfactual'], rho)),
                })

                for client, d in z.groupby('client'):
                    sens_client.append({
                        'setting': tag, 'g_ref': gref, 'rho': rho,
                        'tau_ref': tau, 'kappa': kappa,
                        'client': client,
                        'mean_weight_to_prior_ratio': d['weight_ratio_to_prior'].mean(),
                        'min_weight_to_prior_ratio': d['weight_ratio_to_prior'].min(),
                        'max_weight_to_prior_ratio': d['weight_ratio_to_prior'].max(),
                        'mean_abs_weight_delta_from_prior': d['abs_delta_from_prior'].mean(),
                    })

sens_summary = pd.DataFrame(sens_summary)
sens_client = pd.DataFrame(sens_client)
sens_summary.to_csv(OUTDIR / 'weight_parameter_sensitivity_summary.csv', index=False)
sens_client.to_csv(OUTDIR / 'weight_parameter_sensitivity_by_client.csv', index=False)

# ============================================================================
# C. COUNTERFACTUAL AGGREGATION-COMPONENT DIAGNOSTICS
# ============================================================================
variants = [
    ('Full DWFA', dict(mode='full', g_ref=0.05, rho=0.10, tau_ref=200, kappa=0.60)),
    ('Current signal only', dict(mode='current_only', g_ref=0.05, rho=0.10, tau_ref=200, kappa=0.60)),
    ('History only after first round', dict(mode='history_only', g_ref=0.05, rho=0.10, tau_ref=200, kappa=0.60)),
    ('FedAvg-equivalent no adaptation', dict(mode='fedavg', g_ref=0.05, rho=0.10, tau_ref=200, kappa=0.0)),
    ('No equity/reliability floor (rho=0)', dict(mode='full', g_ref=0.05, rho=0.0, tau_ref=200, kappa=0.60)),
]

variant_summary = []
variant_client = []

for name, kwargs in variants:
    z = reconstruct_counterfactual_weights(**kwargs)
    eff = z.groupby(['seed', 'round'])['weight_counterfactual'].apply(
        lambda s: 1.0 / np.square(s.to_numpy()).sum()
    ).to_numpy()

    variant_summary.append({
        'variant': name,
        'mean_abs_weight_delta_from_fedavg': z['abs_delta_from_prior'].mean(),
        'max_abs_weight_delta_from_fedavg': z['abs_delta_from_prior'].max(),
        'mean_effective_clients': np.mean(eff),
        'min_effective_clients': np.min(eff),
        'min_weight_to_prior_ratio': z['weight_ratio_to_prior'].min(),
        'max_weight_to_prior_ratio': z['weight_ratio_to_prior'].max(),
    })

    for client, d in z.groupby('client'):
        variant_client.append({
            'variant': name,
            'client': client,
            'mean_weight_to_prior_ratio': d['weight_ratio_to_prior'].mean(),
            'min_weight_to_prior_ratio': d['weight_ratio_to_prior'].min(),
            'max_weight_to_prior_ratio': d['weight_ratio_to_prior'].max(),
        })

pd.DataFrame(variant_summary).to_csv(
    OUTDIR / 'weight_component_diagnostics_summary.csv', index=False
)
pd.DataFrame(variant_client).to_csv(
    OUTDIR / 'weight_component_diagnostics_by_client.csv', index=False
)

# ============================================================================
# D. EXHAUSTIVE 3/4/5-SEED SUBSET STABILITY
# ============================================================================
CORE_CACHE = {'LPR': 'qfedavg', 'CADR': 'fairfed', 'DWFA': 'dwfa'}

available = {
    label: sorted(int(s) for (m, s) in main_probs.keys() if m == cache_name)
    for label, cache_name in CORE_CACHE.items()
}
common_seeds = sorted(set(available['LPR']) & set(available['CADR']) & set(available['DWFA']))

if len(common_seeds) < 5:
    print('WARNING: fewer than five common LPR/CADR/DWFA seeds found:', common_seeds)

def model_metric(method_label, seed, metric):
    p = np.asarray(main_probs[(CORE_CACHE[method_label], seed)])
    if metric == 'auroc':
        return safe_auroc(y, p)
    if metric == 'worst_group_auroc':
        return min(safe_auroc(y[sex==1], p[sex==1]),
                   safe_auroc(y[sex==0], p[sex==0]))
    raise ValueError(metric)

subset_rows = []
for k in [3, 4, 5]:
    if len(common_seeds) < k:
        continue
    for subset in combinations(common_seeds, k):
        subset = tuple(int(s) for s in subset)
        for metric in ['auroc', 'worst_group_auroc']:
            method_values = {}
            ensemble_values = {}

            for method in ['LPR', 'CADR', 'DWFA']:
                vals = [model_metric(method, seed, metric) for seed in subset]
                method_values[method] = float(np.mean(vals))

                pp = np.mean(
                    [np.asarray(main_probs[(CORE_CACHE[method], seed)], dtype=float)
                     for seed in subset],
                    axis=0
                )
                if metric == 'auroc':
                    ensemble_values[method] = safe_auroc(y, pp)
                else:
                    ensemble_values[method] = min(
                        safe_auroc(y[sex==1], pp[sex==1]),
                        safe_auroc(y[sex==0], pp[sex==0])
                    )

            for a, b in [('DWFA', 'LPR'), ('DWFA', 'CADR'), ('LPR', 'CADR')]:
                subset_rows.append({
                    'n_seeds': k,
                    'seeds': ','.join(map(str, subset)),
                    'metric': metric,
                    'comparison': f'{a} - {b}',
                    'procedure_mean_difference': method_values[a] - method_values[b],
                    'fixed_ensemble_difference': ensemble_values[a] - ensemble_values[b],
                })

subset_df = pd.DataFrame(subset_rows)
subset_df.to_csv(OUTDIR / 'seed_subset_stability_3_4_5.csv', index=False)

if not subset_df.empty:
    seed_summary = subset_df.groupby(
        ['n_seeds', 'metric', 'comparison']
    ).agg(
        n_subsets=('procedure_mean_difference', 'size'),
        mean_difference=('procedure_mean_difference', 'mean'),
        min_difference=('procedure_mean_difference', 'min'),
        max_difference=('procedure_mean_difference', 'max'),
        positive_fraction=('procedure_mean_difference', lambda x: float(np.mean(np.asarray(x) > 0))),
        ensemble_positive_fraction=('fixed_ensemble_difference', lambda x: float(np.mean(np.asarray(x) > 0))),
    ).reset_index()
else:
    seed_summary = pd.DataFrame()

seed_summary.to_csv(OUTDIR / 'seed_subset_stability_summary.csv', index=False)

# ============================================================================
# E. EXPLORATORY RUN-COUNT PRECISION ANALYSIS
# ============================================================================
precision_rows = []

for metric in ['auroc', 'worst_group_auroc']:
    for a, b in [('DWFA', 'LPR'), ('DWFA', 'CADR'), ('LPR', 'CADR')]:
        paired = np.array([
            model_metric(a, s, metric) - model_metric(b, s, metric)
            for s in common_seeds
        ], dtype=float)

        observed_mean = float(np.mean(paired))
        observed_sd = float(np.std(paired, ddof=1))

        for n in [5, 10, 15, 20]:
            crit = float(student_t.ppf(0.975, df=n-1))
            half_width = crit * observed_sd / math.sqrt(n)
            precision_rows.append({
                'metric': metric,
                'comparison': f'{a} - {b}',
                'observed_common_seed_mean_diff': observed_mean,
                'observed_paired_seed_sd': observed_sd,
                'planned_n_runs': n,
                'approx_95pct_half_width': half_width,
                'note': 'Exploratory precision projection from the observed five-seed paired SD; not a formal power guarantee.'
            })

precision_df = pd.DataFrame(precision_rows)
precision_df.to_csv(OUTDIR / 'exploratory_run_count_precision.csv', index=False)

# ============================================================================
# F. 37-THRESHOLD FAIRNESS SUMMARY
# ============================================================================
threshold_rows = []

for method_label, cache_method in CORE_CACHE.items():
    method_seeds = sorted(int(s) for (m, s) in main_probs.keys() if m == cache_method)
    for thr in THRESHOLDS:
        eo_vals, fpr_vals = [], []
        for seed in method_seeds:
            p = np.asarray(main_probs[(cache_method, seed)])
            eo_vals.append(eo_gap(p, y, sex, float(thr)))
            fpr_vals.append(fpr_gap(p, y, sex, float(thr)))
        threshold_rows.append({
            'method': method_label,
            'threshold': float(thr),
            'mean_eo_gap': float(np.mean(eo_vals)),
            'min_eo_gap_across_runs': float(np.min(eo_vals)),
            'max_eo_gap_across_runs': float(np.max(eo_vals)),
            'mean_fpr_gap': float(np.mean(fpr_vals)),
            'min_fpr_gap_across_runs': float(np.min(fpr_vals)),
            'max_fpr_gap_across_runs': float(np.max(fpr_vals)),
        })

threshold_df = pd.DataFrame(threshold_rows)
threshold_df.to_csv(OUTDIR / 'threshold_grid_37_points.csv', index=False)

eo_winners = (
    threshold_df.loc[threshold_df.groupby('threshold')['mean_eo_gap'].idxmin(), 'method']
    .value_counts()
)
fpr_winners = (
    threshold_df.loc[threshold_df.groupby('threshold')['mean_fpr_gap'].idxmin(), 'method']
    .value_counts()
)

threshold_summary = []
for method in ['LPR', 'CADR', 'DWFA']:
    d = threshold_df[threshold_df['method'] == method]
    threshold_summary.append({
        'method': method,
        'mean_of_mean_eo_gap_over_37_thresholds': d['mean_eo_gap'].mean(),
        'median_mean_eo_gap_over_37_thresholds': d['mean_eo_gap'].median(),
        'eo_lowest_gap_threshold_count': int(eo_winners.get(method, 0)),
        'mean_of_mean_fpr_gap_over_37_thresholds': d['mean_fpr_gap'].mean(),
        'median_mean_fpr_gap_over_37_thresholds': d['mean_fpr_gap'].median(),
        'fpr_lowest_gap_threshold_count': int(fpr_winners.get(method, 0)),
    })

pd.DataFrame(threshold_summary).to_csv(
    OUTDIR / 'threshold_grid_summary.csv', index=False
)

# ============================================================================
# G. CLAIM-TO-EVIDENCE TABLE FOR THE REVISION
# ============================================================================
claim_table = pd.DataFrame([
    ['DWFA weights are positive and normalized', 'Supported mathematically and empirically',
     'Proposition 2 plus exact 600-weight reconstruction audit'],
    ['DWFA reduces to FedAvg under equal quality multipliers', 'Supported mathematically',
     'Proposition 1'],
    ['A larger well-supported local disparity reduces client influence under stated conditions',
     'Supported mathematically', 'Proposition 3'],
    ['DWFA has bounded per-round deviation from FedAvg', 'Supported mathematically',
     'Proposition 4'],
    ['DWFA achieved the highest descriptive mean on several NIH discrimination metrics',
     'Supported descriptively', 'Five-run external NIH summaries'],
    ['DWFA is stably superior to LPR or CADR', 'Not established',
     'Run-aware simultaneous and patient-equal intervals include zero'],
    ['Correct fairness-signal routing causes better external performance', 'Not established',
     'Existing scrambled-routing experiment is a non-matched mechanism-disruption diagnostic'],
    ['DWFA directly optimizes global equal opportunity', 'Not claimed / not established',
     'DWFA uses local EO disparity as a reliability input rather than a global parity objective'],
    ['Alternative hyperparameters preserve similar model performance', 'Not established without retraining',
     'New analysis evaluates only weight-response sensitivity conditional on the observed trajectory'],
], columns=['claim', 'status', 'evidence_or_limit'])
claim_table.to_csv(OUTDIR / 'claim_to_evidence_table.csv', index=False)

# ============================================================================
# H. SIMPLE JOURNAL-READY DIAGNOSTIC FIGURES
# ============================================================================
fig, ax = plt.subplots(figsize=(8.2, 4.8))
plot_df = sens_summary[(sens_summary['rho'] == 0.10) &
                       (sens_summary['tau_ref'] == 200) &
                       (sens_summary['kappa'] == 0.60)].sort_values('g_ref')
ax.plot(plot_df['g_ref'], plot_df['mean_effective_clients'], marker='o')
ax.set_xlabel(r'$G_{\mathrm{ref}}$')
ax.set_ylabel('Mean effective number of clients')
ax.spines[['top', 'right']].set_visible(False)
ax.grid(axis='y', alpha=0.25)
fig.tight_layout()
fig.savefig(OUTDIR / 'weight_sensitivity_effective_clients.png', dpi=300)
plt.close(fig)

if not subset_df.empty:
    fig, ax = plt.subplots(figsize=(8.2, 4.8))
    q = subset_df[(subset_df['comparison'] == 'DWFA - LPR') &
                  (subset_df['metric'] == 'auroc')].copy()
    for n, d in q.groupby('n_seeds'):
        x = np.full(len(d), n, dtype=float) + np.linspace(-0.08, 0.08, len(d))
        ax.scatter(x, d['procedure_mean_difference']*1000, label=f'{n} seeds', alpha=0.8)
    ax.axhline(0, color='black', linestyle='--', linewidth=1)
    ax.set_xticks(sorted(q['n_seeds'].unique()))
    ax.set_xlabel('Number of common seeds retained')
    ax.set_ylabel(r'DWFA - LPR overall AUROC ($\times 10^{-3}$)')
    ax.spines[['top', 'right']].set_visible(False)
    ax.grid(axis='y', alpha=0.25)
    fig.tight_layout()
    fig.savefig(OUTDIR / 'seed_subset_stability_dwfa_lpr.png', dpi=300)
    plt.close(fig)

readme = f"""
REVIEWER-FOCUSED NO-RETRAINING ANALYSES

Default DWFA weight reconstruction max absolute error:
{recon_error:.6e}

IMPORTANT:
1. Parameter sensitivity and component diagnostics are POST-HOC WEIGHT-RESPONSE
   analyses conditional on the observed local-gap trajectory.
2. They DO NOT estimate the AUROC, AUPRC, fairness, or calibration that would
   result after retraining under alternative hyperparameters.
3. The 3/4/5-seed subset analysis characterizes sensitivity of the EXISTING
   five runs. It does not create new independent training runs.
4. The run-count precision table is exploratory and based on observed paired
   seed variation. It is not a formal power guarantee.
5. Local-only A/B/C/D checkpoints are reported separately. They are not merged
   into a single 'local-only' global training procedure.
6. Patient-level CheXpert reconstruction remains unresolved without retraining.

Optional extended GPU cache loaded: {EXT_CACHE.exists()}
Common LPR/CADR/DWFA seeds found: {common_seeds}
"""
(OUTDIR / 'README_interpretation.txt').write_text(readme, encoding='utf-8')

if ZIP_PATH.exists():
    ZIP_PATH.unlink()

with zipfile.ZipFile(ZIP_PATH, 'w', compression=zipfile.ZIP_DEFLATED) as zf:
    for path in sorted(OUTDIR.iterdir()):
        if path.is_file():
            zf.write(path, arcname=path.name)

print('\n' + '='*88)
print('CPU REVIEWER-POLISH CELL COMPLETE')
print('='*88)
print('Output folder:', OUTDIR)
print('ZIP:', ZIP_PATH)
print('\nFiles created:')
for path in sorted(OUTDIR.iterdir()):
    print(' ', path.name)

print('\nKEY INTERPRETATION:')
print('  Weight sensitivity = valid no-retraining mechanism audit.')
print('  It is NOT a performance sensitivity experiment.')
print('  Seed subsets = stability audit of the five observed runs, not extra seeds.')
print('  Patient-disjoint CheXpert reconstruction still requires retraining.')


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Loaded optional extended GPU cache: nih_extended_baselines_cache.npz
DWFA default weight reconstruction max abs error: 4.996e-16

CPU REVIEWER-POLISH CELL COMPLETE
Output folder: /content/drive/MyDrive/FairFedCXR/results/reviewer_polish_cpu
ZIP: /content/drive/MyDrive/FairFedCXR/results/reviewer_polish_cpu.zip

Files created:
  README_interpretation.txt
  claim_to_evidence_table.csv
  dwfa_validation_subgroup_support.csv
  exploratory_run_count_precision.csv
  extended_external_metrics_per_run.csv
  extended_external_metrics_summary.csv
  seed_subset_stability_3_4_5.csv
  seed_subset_stability_dwfa_lpr.png
  seed_subset_stability_summary.csv
  threshold_grid_37_points.csv
  threshold_grid_summary.csv
  weight_component_diagnostics_by_client.csv
  weight_component_diagnostics_summary.csv
  weight_parameter_sensitivity_by_client.csv
  weight_parameter_sensitivi

In [ ]:
# ============================================================================
# FINAL CPU CELL — TWO REMAINING CONCERNS, NO RETRAINING
#
# PURPOSE
#   Concern 1: quantify the CheXpert patient-overlap issue using a STRICT
#              patient-isolated retained test subset from EXISTING predictions.
#   Concern 2: formalize the strongest possible NO-RETRAINING routing analysis
#              using matched seeds, the primary DWFA round log, the cached
#              scrambled predictions, and the already-computed patient-cluster
#              run-aware results when revision_run3.zip is available.
#
# BEFORE RUNNING THIS CELL
#   NO OLD NOTEBOOK CELL NEEDS TO BE RUN IN THE CURRENT SESSION.
#   This cell mounts Drive itself and reads saved artifacts only.
#
# REQUIRED SAVED FILES ON DRIVE
#   /content/drive/MyDrive/FairFedCXR/clients/client_[A-D]_[train|val|test].csv
#   /content/drive/MyDrive/FairFedCXR/results/chexpert_pred_cache.npz
#   /content/drive/MyDrive/FairFedCXR/results/nih_pred_cache.npz
#   /content/drive/MyDrive/FairFedCXR/results/nih_scrambled_pred_cache.npz
#   /content/drive/MyDrive/FairFedCXR/results/nih_effusion_cohort.csv
#   /content/drive/MyDrive/FairFedCXR/results/dwfa_round_log.csv
#
# STRONGLY RECOMMENDED SAVED FILE
#   /content/drive/MyDrive/FairFedCXR/results/revision_run3.zip
#   This was produced by the old CPU cell:
#     "# RUN 3: PATIENT-CLUSTERED, RUN-AWARE NIH INFERENCE"
#
# ONLY IF nih_scrambled_pred_cache.npz IS MISSING
#   Run the old cell:
#     "# PLACEBO, MATCHED SEEDS: real DWFA (3 seeds) vs scrambled DWFA (3 seeds)"
#   It should load the existing scrambled checkpoints. Do NOT retrain anything.
#
# IMPORTANT SCIENTIFIC LIMITS
#   1. The strict CheXpert reanalysis can remove overlapped patients from the
#      RETAINED TEST EVALUATION. It cannot reconstruct a patient-disjoint
#      training trajectory or checkpoint-selection history.
#   2. The routing analysis can show whether the existing 20-round scrambled
#      arm is selection-window-equivalent to the reported primary DWFA models
#      for seeds 42/123/456. It does not create new models.
#   3. FedAvg is used only as a mathematical no-adaptation reference because
#      kappa=0 reduces the DWFA aggregation weights to the FedAvg prior.
#
# OUTPUT
#   results/final_two_concerns_cpu/
#   results/final_two_concerns_cpu.zip
# ============================================================================

from google.colab import drive
drive.mount('/content/drive', force_remount=False)

from pathlib import Path
from io import BytesIO
import os, re, json, zipfile, shutil, math
import numpy as np
import pandas as pd
from sklearn.metrics import roc_auc_score, average_precision_score

BASE = Path('/content/drive/MyDrive/FairFedCXR')
CLIENTS = BASE / 'clients'
RESULTS = BASE / 'results'

CX_CACHE = RESULTS / 'chexpert_pred_cache.npz'
NIH_CACHE = RESULTS / 'nih_pred_cache.npz'
SCR_CACHE = RESULTS / 'nih_scrambled_pred_cache.npz'
NIH_CSV = RESULTS / 'nih_effusion_cohort.csv'
DWFA_LOG = RESULTS / 'dwfa_round_log.csv'
RUN3_ZIP = RESULTS / 'revision_run3.zip'

OUT = RESULTS / 'final_two_concerns_cpu'
ZIP_OUT = RESULTS / 'final_two_concerns_cpu.zip'

if OUT.exists():
    shutil.rmtree(OUT)
OUT.mkdir(parents=True, exist_ok=True)

required = [CX_CACHE, NIH_CACHE, SCR_CACHE, NIH_CSV, DWFA_LOG]
missing = [str(p) for p in required if not p.exists()]
if missing:
    print('\nMISSING REQUIRED SAVED FILES:')
    for p in missing:
        print('  ', p)
    if SCR_CACHE in [Path(p) for p in missing]:
        print('\nIf only nih_scrambled_pred_cache.npz is missing, run the old cell:')
        print('"# PLACEBO, MATCHED SEEDS: real DWFA (3 seeds) vs scrambled DWFA (3 seeds)"')
        print('Do NOT retrain anything.')
    raise FileNotFoundError('Required saved artifacts are missing. See messages above.')

CORE_LABEL = {
    'fedavg': 'FedAvg',
    'fedprox': 'FedProx',
    'centralized': 'Centralized',
    'qfedavg': 'LPR',
    'fairfed': 'CADR',
    'dwfa': 'DWFA',
}

S3 = [42, 123, 456]
EPS = 1.0
BOOT_CX = 1500
BOOT_SEED = 20260817

# ============================================================================
# COMMON METRIC HELPERS
# ============================================================================

def safe_auc(y, p, sample_weight=None):
    y = np.asarray(y, dtype=int)
    p = np.asarray(p, dtype=float)
    if len(y) == 0 or np.unique(y).size < 2:
        return np.nan
    try:
        return float(roc_auc_score(y, p, sample_weight=sample_weight))
    except Exception:
        return np.nan

def safe_ap(y, p, sample_weight=None):
    y = np.asarray(y, dtype=int)
    p = np.asarray(p, dtype=float)
    if len(y) == 0 or np.sum(y == 1) == 0:
        return np.nan
    try:
        return float(average_precision_score(y, p, sample_weight=sample_weight))
    except Exception:
        return np.nan

def smoothed_rate(y, pred, target_label, predicted_positive=True, sample_weight=None):
    y = np.asarray(y, dtype=int)
    pred = np.asarray(pred, dtype=int)
    mask = (y == target_label)
    if sample_weight is None:
        w = np.ones(len(y), dtype=float)
    else:
        w = np.asarray(sample_weight, dtype=float)

    den = w[mask].sum()
    if predicted_positive:
        num = w[mask & (pred == 1)].sum()
    else:
        num = w[mask & (pred == 0)].sum()

    # Preserve the paper's Laplace-style +1/+2 smoothing for image-weighted
    # threshold summaries. For non-unit patient-equal weights, use the weighted
    # empirical rate without pseudo-counts.
    if sample_weight is None:
        return float((num + EPS) / (den + 2 * EPS))
    return float(num / den) if den > 0 else np.nan

def metric_bundle(p, y, sex, patient_id=None, threshold=0.5):
    p = np.asarray(p, dtype=float)
    y = np.asarray(y, dtype=int)
    sex = np.asarray(sex, dtype=int)
    m = (sex == 1)
    f = (sex == 0)
    pred = (p >= threshold).astype(int)

    male_auc = safe_auc(y[m], p[m])
    female_auc = safe_auc(y[f], p[f])
    male_ap = safe_ap(y[m], p[m])
    female_ap = safe_ap(y[f], p[f])

    tpr_m = smoothed_rate(y[m], pred[m], 1, True)
    tpr_f = smoothed_rate(y[f], pred[f], 1, True)
    fpr_m = smoothed_rate(1 - y[m], pred[m], 1, True)
    fpr_f = smoothed_rate(1 - y[f], pred[f], 1, True)

    out = {
        'n_images': int(len(y)),
        'prevalence': float(np.mean(y)) if len(y) else np.nan,
        'auroc': safe_auc(y, p),
        'male_auroc': male_auc,
        'female_auroc': female_auc,
        'worst_group_auroc': float(np.nanmin([male_auc, female_auc])),
        'auprc': safe_ap(y, p),
        'male_auprc': male_ap,
        'female_auprc': female_ap,
        'worst_group_auprc': float(np.nanmin([male_ap, female_ap])),
        'eo_gap_0.5': abs(tpr_m - tpr_f),
        'fpr_gap_0.5': abs(fpr_m - fpr_f),
        'brier': float(np.mean((p - y) ** 2)),
    }

    if patient_id is not None:
        patient_id = np.asarray(patient_id)
        patient_counts = pd.Series(patient_id).value_counts()
        w = np.array([1.0 / patient_counts[x] for x in patient_id], dtype=float)

        male_auc_pe = safe_auc(y[m], p[m], w[m])
        female_auc_pe = safe_auc(y[f], p[f], w[f])
        male_ap_pe = safe_ap(y[m], p[m], w[m])
        female_ap_pe = safe_ap(y[f], p[f], w[f])

        out.update({
            'n_patients': int(pd.Series(patient_id).nunique()),
            'patient_equal_auroc': safe_auc(y, p, w),
            'patient_equal_worst_group_auroc': float(np.nanmin([male_auc_pe, female_auc_pe])),
            'patient_equal_auprc': safe_ap(y, p, w),
            'patient_equal_worst_group_auprc': float(np.nanmin([male_ap_pe, female_ap_pe])),
            'patient_equal_brier': float(np.average((p - y) ** 2, weights=w)),
        })
    return out

def extract_patient_id(path_value):
    m = re.search(r'(patient\d+)', str(path_value), flags=re.IGNORECASE)
    return m.group(1).lower() if m else None

# ============================================================================
# PART 1 — EXACT CHEXPERT PATIENT-OVERLAP INVENTORY
# ============================================================================

print('\n' + '=' * 96)
print('PART 1 — CHEXPERT PATIENT-OVERLAP INVENTORY')
print('=' * 96)

assignment_frames = []
test_frames = []

for client in ['A', 'B', 'C', 'D']:
    for split in ['train', 'val', 'test']:
        path = CLIENTS / f'client_{client}_{split}.csv'
        if not path.exists():
            raise FileNotFoundError(path)

        df = pd.read_csv(path, low_memory=False).copy()
        if 'Path' not in df.columns or 'label' not in df.columns or 'sex_encoded' not in df.columns:
            raise KeyError(f'{path.name} does not contain Path, label, and sex_encoded.')

        df['client'] = client
        df['split'] = split
        df['patient_id'] = df['Path'].map(extract_patient_id)

        if df['patient_id'].isna().any():
            bad = int(df['patient_id'].isna().sum())
            raise ValueError(f'Could not extract patient ID from {bad} rows in {path.name}.')

        assignment_frames.append(df)
        if split == 'test':
            test_frames.append(df)

assignments = pd.concat(assignment_frames, ignore_index=True)
pooled_test = pd.concat(test_frames, ignore_index=True).reset_index(drop=True)

patient_inventory = (
    assignments.groupby('patient_id')
    .agg(
        n_images=('Path', 'size'),
        n_clients=('client', 'nunique'),
        n_splits=('split', 'nunique'),
        clients=('client', lambda x: '|'.join(sorted(set(x)))),
        splits=('split', lambda x: '|'.join(sorted(set(x)))),
    )
    .reset_index()
)
patient_inventory['cross_client'] = patient_inventory['n_clients'] > 1
patient_inventory['cross_split'] = patient_inventory['n_splits'] > 1
patient_inventory['appears_in_train'] = patient_inventory['splits'].str.contains('train')
patient_inventory['appears_in_val'] = patient_inventory['splits'].str.contains('val')
patient_inventory['appears_in_test'] = patient_inventory['splits'].str.contains('test')
patient_inventory.to_csv(OUT / 'chexpert_overlap_patient_inventory.csv', index=False)

profile = patient_inventory.set_index('patient_id')
test_pid = pooled_test['patient_id']

# Less strict audit: test patient never appears in any train or validation set.
no_trainval_mask = ~test_pid.map(
    profile['appears_in_train'] | profile['appears_in_val']
).to_numpy(dtype=bool)

# Strict audit: test patient is absent from train/val AND belongs to only one
# simulated client anywhere in the constructed development federation.
single_client_mask = test_pid.map(profile['n_clients']).to_numpy() == 1
strict_mask = no_trainval_mask & single_client_mask

pooled_test['audit_no_trainval'] = no_trainval_mask
pooled_test['audit_strict_patient_isolated'] = strict_mask
pooled_test['cache_row'] = np.arange(len(pooled_test))
strict_test = pooled_test.loc[strict_mask].copy().reset_index(drop=True)
strict_test.to_csv(OUT / 'chexpert_strict_patient_isolated_test_manifest.csv', index=False)

overlap_summary = pd.DataFrame([
    {
        'quantity': 'all_constructed_images',
        'value': len(assignments),
    },
    {
        'quantity': 'all_constructed_unique_patients',
        'value': assignments['patient_id'].nunique(),
    },
    {
        'quantity': 'patients_crossing_clients',
        'value': int(patient_inventory['cross_client'].sum()),
    },
    {
        'quantity': 'patients_crossing_splits',
        'value': int(patient_inventory['cross_split'].sum()),
    },
    {
        'quantity': 'pooled_test_images_full',
        'value': len(pooled_test),
    },
    {
        'quantity': 'pooled_test_patients_full',
        'value': pooled_test['patient_id'].nunique(),
    },
    {
        'quantity': 'test_images_no_train_or_val_patient',
        'value': int(no_trainval_mask.sum()),
    },
    {
        'quantity': 'test_patients_no_train_or_val_patient',
        'value': pooled_test.loc[no_trainval_mask, 'patient_id'].nunique(),
    },
    {
        'quantity': 'strict_patient_isolated_test_images',
        'value': int(strict_mask.sum()),
    },
    {
        'quantity': 'strict_patient_isolated_test_patients',
        'value': strict_test['patient_id'].nunique(),
    },
])
overlap_summary.to_csv(OUT / 'chexpert_overlap_summary.csv', index=False)

strict_counts = (
    pooled_test.assign(
        subset=np.where(
            strict_mask,
            'strict_patient_isolated',
            'excluded_from_strict'
        )
    )
    .groupby(['subset', 'client', 'sex_encoded', 'label'])
    .size()
    .rename('n_images')
    .reset_index()
)
strict_counts.to_csv(OUT / 'chexpert_strict_test_counts.csv', index=False)

print(overlap_summary.to_string(index=False))
print('\nStrict test counts by client/sex/label:')
print(strict_counts.to_string(index=False))

# ============================================================================
# PART 2 — STRICT PATIENT-ISOLATED CHEXPERT REANALYSIS FROM CACHED PREDICTIONS
# ============================================================================

print('\n' + '=' * 96)
print('PART 2 — STRICT PATIENT-ISOLATED CHEXPERT REANALYSIS')
print('=' * 96)

cx = np.load(CX_CACHE, allow_pickle=True)
cx_probs = cx['all_probs'].item()
cx_labels = cx['labels'].astype(int)
cx_sex = cx['sex'].astype(int)

if len(cx_labels) != len(pooled_test):
    raise AssertionError(
        f'CheXpert cache has {len(cx_labels)} rows but pooled test has {len(pooled_test)}.'
    )
if not np.array_equal(cx_labels, pooled_test['label'].to_numpy(dtype=int)):
    raise AssertionError('CheXpert cached label order does not match pooled A/B/C/D test order.')
if not np.array_equal(cx_sex, pooled_test['sex_encoded'].to_numpy(dtype=int)):
    raise AssertionError('CheXpert cached sex order does not match pooled A/B/C/D test order.')

subsets = {
    'full_pooled_test': np.ones(len(pooled_test), dtype=bool),
    'test_patients_absent_from_train_val': no_trainval_mask,
    'strict_patient_isolated_test': strict_mask,
}

per_run_rows = []
for (cache_method, seed), p in sorted(cx_probs.items(), key=lambda kv: (str(kv[0][0]), int(kv[0][1]))):
    p = np.asarray(p, dtype=float)
    if len(p) != len(pooled_test):
        continue

    for subset_name, mask in subsets.items():
        row = {
            'cache_method': cache_method,
            'method': CORE_LABEL.get(cache_method, cache_method),
            'seed': int(seed),
            'subset': subset_name,
        }
        row.update(
            metric_bundle(
                p[mask],
                cx_labels[mask],
                cx_sex[mask],
                pooled_test.loc[mask, 'patient_id'].to_numpy(),
            )
        )
        per_run_rows.append(row)

cx_per_run = pd.DataFrame(per_run_rows)
cx_per_run.to_csv(OUT / 'chexpert_overlap_sensitivity_metrics_per_run.csv', index=False)

numeric_cols = [
    c for c in cx_per_run.columns
    if c not in ['cache_method', 'method', 'seed', 'subset']
]
cx_summary = (
    cx_per_run.groupby(['method', 'subset'])[numeric_cols]
    .agg(['mean', 'std'])
)
cx_summary.columns = [f'{a}_{b}' for a, b in cx_summary.columns]
cx_summary = cx_summary.reset_index()
cx_summary.to_csv(OUT / 'chexpert_overlap_sensitivity_metrics_summary.csv', index=False)

# Fixed probability ensembles for each available method.
ensemble_rows = []
ensemble_probs = {}
for cache_method in sorted(set(k[0] for k in cx_probs.keys())):
    arrs = [
        np.asarray(p, dtype=float)
        for (m, s), p in cx_probs.items()
        if m == cache_method and len(p) == len(pooled_test)
    ]
    if not arrs:
        continue
    ensemble_probs[cache_method] = np.mean(np.stack(arrs), axis=0)

    for subset_name, mask in subsets.items():
        row = {
            'cache_method': cache_method,
            'method': CORE_LABEL.get(cache_method, cache_method),
            'n_seed_models_in_ensemble': len(arrs),
            'subset': subset_name,
        }
        row.update(
            metric_bundle(
                ensemble_probs[cache_method][mask],
                cx_labels[mask],
                cx_sex[mask],
                pooled_test.loc[mask, 'patient_id'].to_numpy(),
            )
        )
        ensemble_rows.append(row)

cx_ensemble = pd.DataFrame(ensemble_rows)
cx_ensemble.to_csv(OUT / 'chexpert_overlap_sensitivity_fixed_ensembles.csv', index=False)

# Core strict-vs-full contrast table.
core_methods = [('dwfa', 'DWFA'), ('qfedavg', 'LPR'), ('fairfed', 'CADR'), ('fedavg', 'FedAvg')]
contrast_rows = []
for cache_method, label in core_methods:
    if cache_method not in ensemble_probs:
        continue
    d = cx_ensemble[cx_ensemble['cache_method'] == cache_method].set_index('subset')
    if 'full_pooled_test' not in d.index or 'strict_patient_isolated_test' not in d.index:
        continue
    for metric in ['auroc', 'worst_group_auroc', 'auprc', 'worst_group_auprc',
                   'eo_gap_0.5', 'fpr_gap_0.5', 'brier',
                   'patient_equal_auroc', 'patient_equal_worst_group_auroc']:
        contrast_rows.append({
            'method': label,
            'metric': metric,
            'full_pooled_test': float(d.loc['full_pooled_test', metric]),
            'strict_patient_isolated_test': float(d.loc['strict_patient_isolated_test', metric]),
            'strict_minus_full': float(
                d.loc['strict_patient_isolated_test', metric]
                - d.loc['full_pooled_test', metric]
            ),
        })

pd.DataFrame(contrast_rows).to_csv(
    OUT / 'chexpert_strict_minus_full_fixed_ensemble.csv',
    index=False
)

# Fixed-ensemble patient-cluster bootstrap on the STRICT subset only.
# This is conditional on the already-trained models. It does not add
# training-run uncertainty and does not repair the development trajectory.
strict_core = {
    label: ensemble_probs[cache_method][strict_mask]
    for cache_method, label in core_methods
    if cache_method in ensemble_probs
}

strict_y = cx_labels[strict_mask]
strict_s = cx_sex[strict_mask]
strict_pid = strict_test['patient_id'].to_numpy()

if len(strict_y) > 0 and len(strict_core) >= 2:
    unique_patients, patient_code = np.unique(strict_pid, return_inverse=True)
    n_patients = len(unique_patients)
    rng = np.random.default_rng(BOOT_SEED)

    comparisons = [
        ('DWFA', 'LPR'),
        ('DWFA', 'CADR'),
        ('LPR', 'CADR'),
    ]
    comparisons = [(a, b) for a, b in comparisons if a in strict_core and b in strict_core]

    point_lookup = {}
    for method, p in strict_core.items():
        point_lookup[(method, 'auroc')] = safe_auc(strict_y, p)
        m = strict_s == 1
        f = strict_s == 0
        point_lookup[(method, 'worst_group_auroc')] = min(
            safe_auc(strict_y[m], p[m]),
            safe_auc(strict_y[f], p[f]),
        )

    boot_store = {
        (a, b, metric): []
        for a, b in comparisons
        for metric in ['auroc', 'worst_group_auroc']
    }

    for _ in range(BOOT_CX):
        sampled = rng.integers(0, n_patients, size=n_patients)
        counts = np.bincount(sampled, minlength=n_patients).astype(float)
        w = counts[patient_code]

        for a, b in comparisons:
            for metric in ['auroc', 'worst_group_auroc']:
                if metric == 'auroc':
                    va = safe_auc(strict_y, strict_core[a], w)
                    vb = safe_auc(strict_y, strict_core[b], w)
                else:
                    m = strict_s == 1
                    f = strict_s == 0
                    va = min(
                        safe_auc(strict_y[m], strict_core[a][m], w[m]),
                        safe_auc(strict_y[f], strict_core[a][f], w[f]),
                    )
                    vb = min(
                        safe_auc(strict_y[m], strict_core[b][m], w[m]),
                        safe_auc(strict_y[f], strict_core[b][f], w[f]),
                    )
                boot_store[(a, b, metric)].append(va - vb)

    boot_rows = []
    for (a, b, metric), vals in boot_store.items():
        vals = np.asarray(vals, dtype=float)
        vals = vals[np.isfinite(vals)]
        point = point_lookup[(a, metric)] - point_lookup[(b, metric)]
        lo, hi = np.percentile(vals, [2.5, 97.5])
        boot_rows.append({
            'analysis': 'strict_test_fixed_ensemble_patient_cluster_only',
            'comparison': f'{a} - {b}',
            'metric': metric,
            'estimate': point,
            'ci_low': lo,
            'ci_high': hi,
            'n_strict_patients': n_patients,
            'n_bootstrap': BOOT_CX,
        })
    strict_boot = pd.DataFrame(boot_rows)
else:
    strict_boot = pd.DataFrame()

strict_boot.to_csv(
    OUT / 'chexpert_strict_fixed_ensemble_patient_cluster_bootstrap.csv',
    index=False
)

print('\nStrict patient-isolated fixed-ensemble metrics:')
show_cols = ['method', 'n_seed_models_in_ensemble', 'subset',
             'n_images', 'n_patients', 'auroc', 'worst_group_auroc',
             'auprc', 'eo_gap_0.5', 'fpr_gap_0.5']
print(
    cx_ensemble[cx_ensemble['subset'] == 'strict_patient_isolated_test']
    [show_cols]
    .to_string(index=False)
)

if not strict_boot.empty:
    print('\nStrict-subset fixed-ensemble patient-cluster comparisons:')
    print(strict_boot.to_string(index=False))

# ============================================================================
# PART 3 — VERIFY 20-ROUND VS 30-ROUND SELECTION-WINDOW EQUIVALENCE
# ============================================================================

print('\n' + '=' * 96)
print('PART 3 — PRIMARY DWFA SELECTION-WINDOW EQUIVALENCE')
print('=' * 96)

rlog = pd.read_csv(DWFA_LOG, low_memory=False)
if 'method' in rlog.columns:
    rlog = rlog[rlog['method'].astype(str).str.lower() == 'dwfa'].copy()

need_cols = {'seed', 'round', 'pooled_val_auroc'}
missing_cols = need_cols - set(rlog.columns)
if missing_cols:
    raise KeyError(f'dwfa_round_log.csv missing: {sorted(missing_cols)}')

round_level = (
    rlog[rlog['seed'].isin(S3)]
    .groupby(['seed', 'round'], as_index=False)
    .agg(pooled_val_auroc=('pooled_val_auroc', 'first'))
)

selection_rows = []
for seed in S3:
    d = round_level[round_level['seed'] == seed].sort_values('round')
    d20 = d[d['round'] <= 20]
    d30 = d[d['round'] <= 30]

    if d20.empty or d30.empty:
        raise ValueError(f'Insufficient DWFA round log for seed {seed}.')

    best20 = d20.loc[d20['pooled_val_auroc'].idxmax()]
    best30 = d30.loc[d30['pooled_val_auroc'].idxmax()]

    selection_rows.append({
        'seed': seed,
        'best_round_within_first_20': int(best20['round']),
        'best_val_auroc_within_first_20': float(best20['pooled_val_auroc']),
        'best_round_within_first_30': int(best30['round']),
        'best_val_auroc_within_first_30': float(best30['pooled_val_auroc']),
        'same_selected_round': int(best20['round']) == int(best30['round']),
        'same_selected_val_auroc': bool(np.isclose(
            best20['pooled_val_auroc'], best30['pooled_val_auroc'], atol=1e-12
        )),
    })

selection_equivalence = pd.DataFrame(selection_rows)
selection_equivalence['selection_window_equivalent'] = (
    selection_equivalence['same_selected_round']
    & selection_equivalence['same_selected_val_auroc']
)
selection_equivalence.to_csv(
    OUT / 'routing_selection_window_equivalence.csv',
    index=False
)

ALL_SELECTION_EQUIVALENT = bool(selection_equivalence['selection_window_equivalent'].all())

print(selection_equivalence.to_string(index=False))
print('\nAll three primary DWFA seeds selection-window equivalent (20 vs 30):',
      ALL_SELECTION_EQUIVALENT)

# ============================================================================
# PART 4 — MATCHED-SEED ROUTING POINT METRICS FROM EXISTING NIH CACHES
# ============================================================================

print('\n' + '=' * 96)
print('PART 4 — MATCHED-SEED ROUTING ANALYSIS FROM EXISTING NIH PREDICTIONS')
print('=' * 96)

nih = pd.read_csv(NIH_CSV, low_memory=False).reset_index(drop=True)
if 'Patient ID' not in nih.columns:
    raise KeyError('nih_effusion_cohort.csv must contain Patient ID.')

main = np.load(NIH_CACHE, allow_pickle=True)
P = main['all_probs'].item()
nih_y = main['labels'].astype(int)
nih_s = main['sex'].astype(int)

if not np.array_equal(nih_y, nih['label'].to_numpy(dtype=int)):
    raise AssertionError('NIH cache label order does not match NIH CSV.')
if not np.array_equal(nih_s, nih['sex_encoded'].to_numpy(dtype=int)):
    raise AssertionError('NIH cache sex order does not match NIH CSV.')

scr_npz = np.load(SCR_CACHE, allow_pickle=True)
if 'probs' not in scr_npz.files:
    raise KeyError('nih_scrambled_pred_cache.npz has no "probs" field.')
SCR = scr_npz['probs'].item()

for seed in S3:
    if ('dwfa', seed) not in P:
        raise KeyError(f'Missing real DWFA seed {seed} in NIH cache.')
    if ('fedavg', seed) not in P:
        raise KeyError(f'Missing FedAvg seed {seed} in NIH cache.')
    if seed not in SCR:
        raise KeyError(f'Missing scrambled DWFA seed {seed} in scrambled cache.')

def nih_small_metrics(p):
    p = np.asarray(p, dtype=float)
    m = nih_s == 1
    f = nih_s == 0
    pred = (p >= 0.5).astype(int)

    tpr_m = smoothed_rate(nih_y[m], pred[m], 1, True)
    tpr_f = smoothed_rate(nih_y[f], pred[f], 1, True)
    fpr_m = smoothed_rate(1 - nih_y[m], pred[m], 1, True)
    fpr_f = smoothed_rate(1 - nih_y[f], pred[f], 1, True)

    return {
        'auroc': safe_auc(nih_y, p),
        'worst_group_auroc': min(
            safe_auc(nih_y[m], p[m]),
            safe_auc(nih_y[f], p[f]),
        ),
        'eo_gap_smoothed_0.5': abs(tpr_m - tpr_f),
        'fpr_gap_smoothed_0.5': abs(fpr_m - fpr_f),
    }

routing_rows = []
for seed in S3:
    arms = {
        'Correct DWFA': np.asarray(P[('dwfa', seed)]),
        'Scrambled DWFA': np.asarray(SCR[seed]),
        'FedAvg no-adaptation reference': np.asarray(P[('fedavg', seed)]),
    }
    for arm, probs in arms.items():
        row = {'seed': seed, 'arm': arm}
        row.update(nih_small_metrics(probs))
        routing_rows.append(row)

routing_per_seed = pd.DataFrame(routing_rows)
routing_per_seed.to_csv(OUT / 'routing_three_seed_point_metrics.csv', index=False)

routing_ensemble_rows = []
for arm, arrays in {
    'Correct DWFA': [P[('dwfa', s)] for s in S3],
    'Scrambled DWFA': [SCR[s] for s in S3],
    'FedAvg no-adaptation reference': [P[('fedavg', s)] for s in S3],
}.items():
    probs = np.mean(np.stack(arrays), axis=0)
    row = {'arm': arm, 'n_common_seeds': 3}
    row.update(nih_small_metrics(probs))
    routing_ensemble_rows.append(row)

routing_ensemble = pd.DataFrame(routing_ensemble_rows)
routing_ensemble.to_csv(OUT / 'routing_three_seed_fixed_ensemble_metrics.csv', index=False)

# Direct same-seed differences.
pair_rows = []
for seed in S3:
    real = routing_per_seed[(routing_per_seed['seed'] == seed)
                            & (routing_per_seed['arm'] == 'Correct DWFA')].iloc[0]
    scrambled = routing_per_seed[(routing_per_seed['seed'] == seed)
                                 & (routing_per_seed['arm'] == 'Scrambled DWFA')].iloc[0]
    for metric in ['auroc', 'worst_group_auroc',
                   'eo_gap_smoothed_0.5', 'fpr_gap_smoothed_0.5']:
        pair_rows.append({
            'seed': seed,
            'comparison': 'Correct DWFA - Scrambled DWFA',
            'metric': metric,
            'difference': float(real[metric] - scrambled[metric]),
        })

routing_paired = pd.DataFrame(pair_rows)
routing_paired.to_csv(OUT / 'routing_same_seed_differences.csv', index=False)

print('\nThree-seed fixed ensembles:')
print(routing_ensemble.to_string(index=False))
print('\nSame-seed correct minus scrambled differences:')
print(routing_paired.to_string(index=False))

# ============================================================================
# PART 5 — IMPORT THE ALREADY-COMPUTED PATIENT-CLUSTER RUN-AWARE ROUTING RESULTS
# ============================================================================

print('\n' + '=' * 96)
print('PART 5 — PATIENT-CLUSTER, RUN-AWARE ROUTING INFERENCE')
print('=' * 96)

run3_placebo = pd.DataFrame()
run3_fedavg = pd.DataFrame()

if RUN3_ZIP.exists():
    with zipfile.ZipFile(RUN3_ZIP, 'r') as zf:
        names = set(zf.namelist())

        if 'run3_placebo_inference.csv' in names:
            run3_placebo = pd.read_csv(
                BytesIO(zf.read('run3_placebo_inference.csv'))
            )
            run3_placebo.to_csv(
                OUT / 'routing_run3_patient_cluster_runaware.csv',
                index=False
            )

        if 'run3_fedavg_three_seed_exploratory.csv' in names:
            run3_fedavg = pd.read_csv(
                BytesIO(zf.read('run3_fedavg_three_seed_exploratory.csv'))
            )
            run3_fedavg.to_csv(
                OUT / 'routing_fedavg_no_adaptation_reference_run3.csv',
                index=False
            )

    print('Loaded existing patient-cluster/run-aware results from revision_run3.zip.')

    if not run3_placebo.empty:
        focus = run3_placebo[
            run3_placebo['analysis'].isin([
                'matched_seed_resampling_sensitivity',
                'observed_runs_fixed_patient_cluster_only',
                'independent_run_resampling_primary',
            ])
        ].copy()
        print('\nRouting inference from RUN 3:')
        print(focus.to_string(index=False))
else:
    print('WARNING: revision_run3.zip is not present.')
    print('No new model inference or training is required, but the strongest')
    print('patient-cluster/run-aware routing CI table cannot be imported.')
    print('If needed, run the old CPU cell:')
    print('"# RUN 3: PATIENT-CLUSTERED, RUN-AWARE NIH INFERENCE"')

# ============================================================================
# PART 6 — DECISION SUMMARY FOR MANUSCRIPT USE
# ============================================================================

strict_n_patients = int(strict_test['patient_id'].nunique())
strict_n_images = int(len(strict_test))
full_test_patients = int(pooled_test['patient_id'].nunique())

# Check whether the strict result preserves the sign of core ensemble contrasts.
sign_rows = []
for metric in ['auroc', 'worst_group_auroc']:
    look = cx_ensemble.pivot_table(
        index='method', columns='subset', values=metric, aggfunc='first'
    )
    for a, b in [('DWFA', 'LPR'), ('DWFA', 'CADR'), ('LPR', 'CADR')]:
        if a not in look.index or b not in look.index:
            continue
        full_diff = look.loc[a, 'full_pooled_test'] - look.loc[b, 'full_pooled_test']
        strict_diff = (
            look.loc[a, 'strict_patient_isolated_test']
            - look.loc[b, 'strict_patient_isolated_test']
        )
        sign_rows.append({
            'comparison': f'{a} - {b}',
            'metric': metric,
            'full_difference': full_diff,
            'strict_difference': strict_diff,
            'same_direction': (
                np.sign(full_diff) == np.sign(strict_diff)
                if full_diff != 0 and strict_diff != 0
                else False
            ),
        })

sign_check = pd.DataFrame(sign_rows)
sign_check.to_csv(OUT / 'chexpert_strict_direction_check.csv', index=False)

matched_route_available = not run3_placebo.empty

readme = f"""
FINAL TWO-CONCERN NO-RETRAINING AUDIT
=====================================

CONCERN 1: PATIENT-LEVEL CHEXPERT CONSTRUCTION
----------------------------------------------
Full pooled CheXpert test:
  images   = {len(pooled_test)}
  patients = {full_test_patients}

Strict patient-isolated retained test subset:
  images   = {strict_n_images}
  patients = {strict_n_patients}

Definition of strict subset:
  A test patient is retained only if that patient is absent from every
  constructed training and validation split AND belongs to only one simulated
  client anywhere in the constructed federation.

What this DOES address:
  It removes direct patient overlap between the retained internal TEST
  evaluation and all constructed train/validation sets.
  It also removes test patients assigned to multiple simulated clients.

What this DOES NOT address:
  It does not reconstruct patient-disjoint client training.
  It does not reconstruct patient-disjoint validation.
  It does not reconstruct an alternative checkpoint-selection trajectory.
  Therefore this is a robustness audit of retained test predictions, not a
  corrected retraining experiment.

CONCERN 2: MATCHED ROUTING MECHANISM
------------------------------------
Primary seeds examined: {S3}

20-round versus 30-round primary DWFA checkpoint-selection equivalence:
  all three seeds equivalent = {ALL_SELECTION_EQUIVALENT}

If True:
  For seeds 42, 123, and 456, the best primary DWFA checkpoint selected from
  rounds 1-30 is the same checkpoint that would have been selected from
  rounds 1-20. This removes the selection-window difference for the reported
  correct-DWFA models when comparing them with the existing 20-round
  scrambled-routing arm.

Routing cache:
  {SCR_CACHE}

Patient-cluster/run-aware RUN 3 results imported:
  {matched_route_available}

Important provenance note:
  The notebook cell that created dwfa_scrambled_ckpt_seed42/123/456 used the
  primary DWFA constants loaded earlier in the notebook
  (G_ref=0.05, tau_ref=200, rho=0.10, alpha=0.40, kappa=0.60), a fixed
  derangement, the same client files, and seeds 42/123/456.
  The NPZ prediction cache itself does not encode these hyperparameters, so
  this constant-matching statement is a code-provenance claim, not metadata
  stored inside the NPZ file.

FedAvg reference:
  Setting kappa=0 makes the DWFA quality multiplier constant across clients,
  so normalized DWFA aggregation weights reduce mathematically to the FedAvg
  sample-size prior. The saved FedAvg models are therefore reported as a
  no-adaptation reference, not as a newly trained DWFA ablation.

SAFE MANUSCRIPT INTERPRETATION
------------------------------
1. Do not say the patient-level development problem was "fixed".
   Say a strict patient-isolated retained-test sensitivity analysis was added.

2. If ALL_SELECTION_EQUIVALENT is True, it is defensible to describe the
   correct-vs-scrambled comparison as matched in seeds, primary constants, and
   effective 20-round checkpoint-selection window for these three seeds.

3. Even then, do not claim that routing causes a fairness improvement unless
   the patient-cluster/run-aware intervals support it. If they include zero,
   state that a reproducible routing advantage was not established.

4. No new training was performed in this cell.
"""

(OUT / 'README_INTERPRETATION.txt').write_text(readme, encoding='utf-8')

# Small machine-readable decision table.
decision_table = pd.DataFrame([
    {
        'concern': 'Patient-level CheXpert reconstruction',
        'no_retrain_status': 'partially mitigated, not fully solved',
        'new_evidence': 'strict patient-isolated retained-test reanalysis',
        'remaining_limit': 'training/validation/checkpoint trajectory remains from original construction',
    },
    {
        'concern': 'Matched routing mechanism',
        'no_retrain_status': (
            'strongly addressed if selection_window_equivalent=True and RUN 3 cache is available'
        ),
        'new_evidence': (
            'matched seeds + primary-constant scrambled cache + 20-vs-30 selection equivalence '
            '+ patient-cluster/run-aware routing inference'
        ),
        'remaining_limit': (
            'no new causal claim if routing confidence intervals remain unresolved'
        ),
    },
])
decision_table.to_csv(OUT / 'two_concerns_decision_table.csv', index=False)

# ============================================================================
# ZIP EVERYTHING
# ============================================================================

if ZIP_OUT.exists():
    ZIP_OUT.unlink()

with zipfile.ZipFile(ZIP_OUT, 'w', compression=zipfile.ZIP_DEFLATED) as zf:
    for path in sorted(OUT.iterdir()):
        if path.is_file():
            zf.write(path, arcname=path.name)

print('\n' + '=' * 96)
print('FINAL CPU CELL COMPLETE')
print('=' * 96)
print('Output folder:', OUT)
print('ZIP:', ZIP_OUT)
print('\nFiles created:')
for p in sorted(OUT.iterdir()):
    print(' ', p.name)

print('\nLOCKED INTERPRETATION:')
print('  Patient concern: strict-test leakage sensitivity added; training history NOT repaired.')
print('  Routing concern: use 20-vs-30 equivalence only if the table says True for all 3 seeds.')
print('  No retraining and no new model inference were performed.')


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).

PART 1 — CHEXPERT PATIENT-OVERLAP INVENTORY
                             quantity  value
               all_constructed_images  35899
      all_constructed_unique_patients  24709
            patients_crossing_clients   4736
             patients_crossing_splits   3634
              pooled_test_images_full   5385
            pooled_test_patients_full   4966
  test_images_no_train_or_val_patient   2902
test_patients_no_train_or_val_patient   2804
  strict_patient_isolated_test_images   2785
strict_patient_isolated_test_patients   2748

Strict test counts by client/sex/label:
                 subset client  sex_encoded  label  n_images
   excluded_from_strict      A            0      0       239
   excluded_from_strict      A            0      1       287
   excluded_from_strict      A            1      0       312
   excluded_from_strict      A            1   

In [ ]:
# ============================================================================
# CPU CELL — PATIENT-OVERLAP COUNTERFACTUAL AND SIGNAL-CONTAMINATION AUDIT
#
# PURPOSE
#   Address the "rebuild the federation at patient level" reviewer demand as far
#   as it can be addressed WITHOUT RETRAINING, by separating the parts of the
#   leakage effect that are recomputable from the parts that are not.
#
#   A. Contamination map of the DWFA input signal. The fairness scalar G_i^t and
#      the reliability term r_i are computed on each client's LOCAL VALIDATION
#      set. This quantifies how much of each validation set was seen in training.
#   B. Patient-disjoint repartition of the SAME candidate images under two
#      deterministic repair rules, with realized composition vs the reported one.
#   C. Counterfactual DWFA weight replay. p_i and r_i are pure count quantities,
#      so they ARE recomputable under a clean partition. Replays Eqs. (3)-(11)
#      with clean p_i and r_i while holding the LOGGED G_i^t fixed.
#   D. Contamination-stratified method contrasts on the retained CheXpert test,
#      with a patient-clustered difference-in-differences interval.
#   E. Prevalence-reweighted strict subset, removing the composition confound
#      between the full pooled test (prev 0.403) and the strict subset (0.320).
#
# BEFORE RUNNING THIS CELL
#   NO OLD NOTEBOOK CELL NEEDS TO BE RUN IN THE CURRENT SESSION.
#   This cell mounts Drive itself and reads saved artifacts only.
#
# REQUIRED SAVED FILES ON DRIVE
#   clients/client_[A-D]_[train|val|test].csv
#   results/dwfa_round_log.csv
#   results/chexpert_pred_cache.npz
#
# SCIENTIFIC LIMITS — STATE THESE IN THE PAPER
#   1. Part C holds the observed disparity trajectory G_i^t fixed. Under a truly
#      patient-disjoint federation the local models and validation sets would
#      differ, so G_i^t would also change. Part C therefore isolates ONLY the
#      count-driven channel of the leakage effect. It is not a retraining
#      substitute and must not be reported as one.
#   2. Parts D and E condition on the retained fixed ensembles. They add no
#      training-run uncertainty and do not repair checkpoint selection.
#   3. Part B is a composition comparison. No model is trained on the repartition.
#
# OUTPUT
#   results/overlap_counterfactual_cpu/
#   results/overlap_counterfactual_cpu.zip
# ============================================================================

from google.colab import drive
drive.mount('/content/drive', force_remount=False)

from pathlib import Path
import os, re, json, shutil, zipfile
import numpy as np
import pandas as pd
from sklearn.metrics import roc_auc_score, average_precision_score

BASE = Path('/content/drive/MyDrive/FairFedCXR')
CLIENTS = BASE / 'clients'
RESULTS = BASE / 'results'

DWFA_LOG = RESULTS / 'dwfa_round_log.csv'
CX_CACHE = RESULTS / 'chexpert_pred_cache.npz'

OUT = RESULTS / 'overlap_counterfactual_cpu'
ZIP_OUT = RESULTS / 'overlap_counterfactual_cpu.zip'

# Primary DWFA constants (Section V-A of the manuscript).
G_REF = 0.05
RHO = 0.10
TAU_REF = 200
ALPHA = 0.40
KAPPA = 0.60

CLIENT_LIST = ['A', 'B', 'C', 'D']
CORE_LABEL = {'fedavg': 'FedAvg', 'fedprox': 'FedProx', 'centralized': 'Centralized',
              'qfedavg': 'LPR', 'fairfed': 'CADR', 'dwfa': 'DWFA'}
CONTRASTS = [('DWFA', 'LPR'), ('DWFA', 'CADR'), ('LPR', 'CADR')]
BOOT = 2000
BOOT_SEED = 20260827

if OUT.exists():
    shutil.rmtree(OUT)
OUT.mkdir(parents=True, exist_ok=True)

required = [DWFA_LOG, CX_CACHE]
missing = [str(p) for p in required if not p.exists()]
for c in CLIENT_LIST:
    for s in ['train', 'val', 'test']:
        p = CLIENTS / f'client_{c}_{s}.csv'
        if not p.exists():
            missing.append(str(p))
if missing:
    print('\nMISSING REQUIRED SAVED FILES:')
    for p in missing:
        print('  ', p)
    raise FileNotFoundError('Required saved artifacts are missing. See list above.')


def extract_patient_id(path_value):
    m = re.search(r'(patient\d+)', str(path_value), flags=re.IGNORECASE)
    return m.group(1).lower() if m else None


def safe_auc(y, p, w=None):
    y = np.asarray(y, dtype=int)
    p = np.asarray(p, dtype=float)
    if len(y) == 0 or np.unique(y).size < 2:
        return np.nan
    try:
        return float(roc_auc_score(y, p, sample_weight=w))
    except Exception:
        return np.nan


def safe_ap(y, p, w=None):
    y = np.asarray(y, dtype=int)
    p = np.asarray(p, dtype=float)
    if len(y) == 0 or (y == 1).sum() == 0:
        return np.nan
    try:
        return float(average_precision_score(y, p, sample_weight=w))
    except Exception:
        return np.nan


def wg_auc(y, p, sex, w=None):
    m = (sex == 1)
    f = (sex == 0)
    a = safe_auc(y[m], p[m], None if w is None else w[m])
    b = safe_auc(y[f], p[f], None if w is None else w[f])
    if not np.isfinite(a) or not np.isfinite(b):
        return np.nan
    return float(min(a, b))


def wg_ap(y, p, sex, w=None):
    m = (sex == 1)
    f = (sex == 0)
    a = safe_ap(y[m], p[m], None if w is None else w[m])
    b = safe_ap(y[f], p[f], None if w is None else w[f])
    if not np.isfinite(a) or not np.isfinite(b):
        return np.nan
    return float(min(a, b))


def _rate(y, pred, target, w):
    mask = (y == target)
    den = w[mask].sum()
    if den <= 0:
        return np.nan
    return float(w[mask & (pred == 1)].sum() / den)


def gap_at(y, p, sex, w, kind, thr=0.5):
    pred = (np.asarray(p, float) >= thr).astype(int)
    m = (sex == 1)
    f = (sex == 0)
    tgt = 1 if kind == 'eo' else 0
    a = _rate(y[m], pred[m], tgt, w[m])
    b = _rate(y[f], pred[f], tgt, w[f])
    if not np.isfinite(a) or not np.isfinite(b):
        return np.nan
    return float(abs(a - b))


# ============================================================================
# LOAD THE CONSTRUCTED FEDERATION
# ============================================================================

frames = []
for c in CLIENT_LIST:
    for s in ['train', 'val', 'test']:
        df = pd.read_csv(CLIENTS / f'client_{c}_{s}.csv', low_memory=False).copy()
        for col in ['Path', 'label', 'sex_encoded']:
            if col not in df.columns:
                raise KeyError(f'client_{c}_{s}.csv is missing column {col}')
        df['client'] = c
        df['split'] = s
        df['patient_id'] = df['Path'].map(extract_patient_id)
        if df['patient_id'].isna().any():
            raise ValueError(f'Could not parse patient IDs in client_{c}_{s}.csv')
        frames.append(df)

alloc = pd.concat(frames, ignore_index=True)
HAS_AGE = 'Age' in alloc.columns

pooled_test = alloc[alloc['split'] == 'test'].copy().reset_index(drop=True)

train_pids_all = set(alloc.loc[alloc['split'] == 'train', 'patient_id'])
train_pids_by_client = {c: set(alloc[(alloc['split'] == 'train') & (alloc['client'] == c)]['patient_id'])
                        for c in CLIENT_LIST}
pid_client_count = alloc.groupby('patient_id')['client'].nunique()
pid_split_count = alloc.groupby('patient_id')['split'].nunique()
multi_client_pids = set(pid_client_count[pid_client_count > 1].index)

# ============================================================================
# PART A — CONTAMINATION MAP OF THE DWFA INPUT SIGNAL
# ============================================================================

print('\n' + '=' * 96)
print('PART A — VALIDATION-SET CONTAMINATION (THE SOURCE OF G_i^t AND r_i)')
print('=' * 96)


def val_counts(df):
    m_pos = int(((df['sex_encoded'] == 1) & (df['label'] == 1)).sum())
    f_pos = int(((df['sex_encoded'] == 0) & (df['label'] == 1)).sum())
    return m_pos, f_pos


def r_from_counts(m_pos, f_pos, tau=TAU_REF, rho=RHO):
    return float(np.clip(min(m_pos, f_pos) / tau, rho, 1.0))


rows_a = []
clean_val_index = {}
for c in CLIENT_LIST:
    v = alloc[(alloc['client'] == c) & (alloc['split'] == 'val')].copy()
    own_train = v['patient_id'].isin(train_pids_by_client[c])
    any_train = v['patient_id'].isin(train_pids_all)
    other_train = any_train & (~own_train)
    cross_client = v['patient_id'].isin(multi_client_pids)

    keep = (~any_train) & (~cross_client)
    clean_val_index[c] = v.loc[keep].copy()

    m_pos, f_pos = val_counts(v)
    m_pos_c, f_pos_c = val_counts(clean_val_index[c])

    rows_a.append({
        'client': c,
        'val_images': int(len(v)),
        'val_patients': int(v['patient_id'].nunique()),
        'val_images_seen_in_own_train': int(own_train.sum()),
        'val_images_seen_in_other_client_train': int(other_train.sum()),
        'val_images_seen_in_any_train': int(any_train.sum()),
        'val_images_patient_crosses_clients': int(cross_client.sum()),
        'val_images_clean': int(keep.sum()),
        'frac_val_images_contaminated': float(1.0 - keep.mean()) if len(v) else np.nan,
        'val_m_pos_reported': m_pos,
        'val_f_pos_reported': f_pos,
        'r_i_reported': r_from_counts(m_pos, f_pos),
        'val_m_pos_clean': m_pos_c,
        'val_f_pos_clean': f_pos_c,
        'r_i_clean': r_from_counts(m_pos_c, f_pos_c),
    })

signal_contamination = pd.DataFrame(rows_a)
signal_contamination['r_i_change'] = (signal_contamination['r_i_clean']
                                      - signal_contamination['r_i_reported'])
signal_contamination.to_csv(OUT / 'A_validation_signal_contamination.csv', index=False)
print(signal_contamination.to_string(index=False))

# ============================================================================
# PART B — PATIENT-DISJOINT REPARTITION UNDER TWO DETERMINISTIC RULES
# ============================================================================

print('\n' + '=' * 96)
print('PART B — PATIENT-DISJOINT REPARTITION OF THE SAME IMAGES')
print('=' * 96)

SPLIT_PRIORITY = {'train': 0, 'val': 1, 'test': 2}

cc = (alloc.groupby(['patient_id', 'client']).size().rename('n').reset_index()
      .sort_values(['patient_id', 'n', 'client'], ascending=[True, False, True]))
patient_to_client = cc.drop_duplicates('patient_id').set_index('patient_id')['client']

sc = (alloc.groupby(['patient_id', 'split']).size().rename('n').reset_index())
sc['prio'] = sc['split'].map(SPLIT_PRIORITY)
sc = sc.sort_values(['patient_id', 'n', 'prio'], ascending=[True, False, True])
patient_to_split = sc.drop_duplicates('patient_id').set_index('patient_id')['split']

repair = alloc.copy()
repair['client'] = repair['patient_id'].map(patient_to_client)
repair['split'] = repair['patient_id'].map(patient_to_split)

contaminated_pids = set(pid_client_count[pid_client_count > 1].index) | \
                    set(pid_split_count[pid_split_count > 1].index)
drop_rule = alloc[~alloc['patient_id'].isin(contaminated_pids)].copy()

for nm, d in [('majority_repair', repair), ('drop_contaminated', drop_rule)]:
    bad_c = (d.groupby('patient_id')['client'].nunique() > 1).sum()
    bad_s = (d.groupby('patient_id')['split'].nunique() > 1).sum()
    assert bad_c == 0 and bad_s == 0, f'{nm} did not produce a disjoint partition'
    for c in sorted(d['client'].unique()):
        sub = d[d['client'] == c]
        for s in ['train', 'val', 'test']:
            if (sub['split'] == s).sum() == 0:
                print(f'WARNING [{nm}]: client {c} has an empty {s} split.')
        v = sub[sub['split'] == 'val']
        mp, fp = val_counts(v)
        if min(mp, fp) == 0:
            print(f'WARNING [{nm}]: client {c} has zero positives in one validation '
                  f'sex subgroup (m_pos={mp}, f_pos={fp}); r_i falls to the floor.')


def composition(d, tag):
    out = []
    for c in sorted(d['client'].unique()):
        sub = d[d['client'] == c]
        v = sub[sub['split'] == 'val']
        m_pos, f_pos = val_counts(v)
        row = {
            'partition': tag,
            'client': c,
            'images': int(len(sub)),
            'patients': int(sub['patient_id'].nunique()),
            'train': int((sub['split'] == 'train').sum()),
            'val': int((sub['split'] == 'val').sum()),
            'test': int((sub['split'] == 'test').sum()),
            'male_fraction': float((sub['sex_encoded'] == 1).mean()),
            'prevalence': float(sub['label'].mean()),
            'val_m_pos': m_pos,
            'val_f_pos': f_pos,
            'r_i': r_from_counts(m_pos, f_pos),
        }
        row['mean_age'] = float(sub['Age'].mean()) if HAS_AGE else np.nan
        out.append(row)
    df = pd.DataFrame(out)
    df['p_i'] = df['train'] / df['train'].sum()
    return df


comp = pd.concat([
    composition(alloc, 'reported_construction'),
    composition(repair, 'majority_repair'),
    composition(drop_rule, 'drop_contaminated'),
], ignore_index=True)
comp.to_csv(OUT / 'B_repartition_composition.csv', index=False)
print(comp.to_string(index=False))

# ============================================================================
# PART C — COUNTERFACTUAL DWFA WEIGHT REPLAY UNDER CLEAN COUNTS
# ============================================================================

print('\n' + '=' * 96)
print('PART C — COUNTERFACTUAL DWFA WEIGHT REPLAY (G_i^t HELD FIXED)')
print('=' * 96)

rlog = pd.read_csv(DWFA_LOG, low_memory=False)
if 'method' in rlog.columns:
    rlog = rlog[rlog['method'].astype(str).str.lower() == 'dwfa'].copy()
need = {'seed', 'round', 'client', 'val_eo_gap_local', 'r_conf', 'weight', 'fedavg_weight'}
if need - set(rlog.columns):
    raise KeyError(f'dwfa_round_log.csv missing: {sorted(need - set(rlog.columns))}')
rlog = rlog.drop_duplicates(['seed', 'round', 'client'], keep='last')

p_reported = {c: float(rlog[rlog['client'] == c]['fedavg_weight'].iloc[0]) for c in CLIENT_LIST}
r_reported = {c: float(rlog[rlog['client'] == c]['r_conf'].iloc[0]) for c in CLIENT_LIST}
r_clean = dict(zip(signal_contamination['client'], signal_contamination['r_i_clean']))

rep_comp = comp[comp['partition'] == 'majority_repair'].set_index('client')
p_clean = {c: float(rep_comp.loc[c, 'p_i']) for c in CLIENT_LIST if c in rep_comp.index}
r_repart = {c: float(rep_comp.loc[c, 'r_i']) for c in CLIENT_LIST if c in rep_comp.index}
for c in CLIENT_LIST:
    p_clean.setdefault(c, p_reported[c])
    r_repart.setdefault(c, r_reported[c])

SCENARIOS = {
    'reported': (p_reported, r_reported),
    'clean_val_counts_only': (p_reported, r_clean),
    'majority_repair_counts': (p_clean, r_repart),
}


def replay(p_map, r_map):
    out = []
    for seed, g in rlog.groupby('seed'):
        hist = {c: [] for c in CLIENT_LIST}
        for rnd in sorted(g['round'].unique()):
            d = g[g['round'] == rnd].set_index('client')
            cs = [c for c in CLIENT_LIST if c in d.index]
            if not cs:
                continue
            e_cur, etilde = {}, {}
            for c in cs:
                G = float(d.loc[c, 'val_eo_gap_local'])
                e = float(np.clip(1.0 - G / G_REF, RHO, 1.0))
                ebar = np.mean(hist[c]) if hist[c] else e
                e_cur[c] = e
                etilde[c] = (1.0 - r_map[c]) * ebar + r_map[c] * e
            q = np.array([ALPHA + KAPPA * etilde[c] for c in cs], dtype=float)
            s = np.array([p_map[c] for c in cs], dtype=float) * q
            a = s / s.sum()
            for i, c in enumerate(cs):
                out.append({'seed': int(seed), 'round': int(rnd), 'client': c,
                            'e_cur': e_cur[c], 'etilde': etilde[c],
                            'p_i': p_map[c], 'r_i': r_map[c],
                            'weight': float(a[i]),
                            'logged_weight': float(d.loc[c, 'weight'])})
            for c in cs:
                hist[c].append(e_cur[c])
    return pd.DataFrame(out)


replays = {}
for name, (pm, rm) in SCENARIOS.items():
    replays[name] = replay(pm, rm)
    replays[name]['scenario'] = name

fid = replays['reported']
max_err = float(np.max(np.abs(fid['weight'] - fid['logged_weight'])))
print(f'Replay self-validation: max |replayed - logged| DWFA weight = {max_err:.3e}')
print('(This must be at machine-precision scale. If not, the constants or the '
      'ebar recursion do not match the training code and Part C is invalid.)')

replay_all = pd.concat(replays.values(), ignore_index=True)
replay_all.to_csv(OUT / 'C_weight_replay_per_round.csv', index=False)


def wsummary(df, tag):
    rows = []
    for (seed,), g in df.groupby(['seed']):
        dev, eff, ratio = [], [], []
        for rnd, gg in g.groupby('round'):
            a = gg['weight'].to_numpy()
            p = gg['p_i'].to_numpy()
            dev.append(np.mean(np.abs(a - p)))
            eff.append(1.0 / np.sum(a ** 2))
            ratio.append(np.max(a / p))
        rows.append({'scenario': tag, 'seed': int(seed),
                     'mean_abs_dev_from_prior': float(np.mean(dev)),
                     'mean_effective_clients': float(np.mean(eff)),
                     'max_weight_to_prior_ratio': float(np.max(ratio)),
                     'equity_floor_activation_rate': float(np.mean(g['e_cur'] <= RHO + 1e-12))})
    return pd.DataFrame(rows)


wsum = pd.concat([wsummary(replays[n], n) for n in SCENARIOS], ignore_index=True)
wsum_mean = wsum.groupby('scenario').mean(numeric_only=True).drop(columns=['seed']).reset_index()
wsum.to_csv(OUT / 'C_weight_response_per_seed.csv', index=False)
wsum_mean.to_csv(OUT / 'C_weight_response_summary.csv', index=False)
print('\nAggregation response by scenario (mean over seeds):')
print(wsum_mean.to_string(index=False))

base = replays['reported'][['seed', 'round', 'client', 'weight']].rename(
    columns={'weight': 'w_reported'})
shift_rows = []
for name in ['clean_val_counts_only', 'majority_repair_counts']:
    m = replays[name][['seed', 'round', 'client', 'weight']].merge(base, on=['seed', 'round', 'client'])
    m['delta'] = m['weight'] - m['w_reported']
    for c, gg in m.groupby('client'):
        shift_rows.append({'scenario': name, 'client': c,
                           'mean_weight_reported': float(gg['w_reported'].mean()),
                           'mean_weight_counterfactual': float(gg['weight'].mean()),
                           'mean_abs_weight_change': float(gg['delta'].abs().mean()),
                           'max_abs_weight_change': float(gg['delta'].abs().max())})
weight_shift = pd.DataFrame(shift_rows)
weight_shift.to_csv(OUT / 'C_weight_shift_by_client.csv', index=False)
print('\nPer-client weight shift under counterfactual counts:')
print(weight_shift.to_string(index=False))

# ============================================================================
# PART D — CONTAMINATION-STRATIFIED CONTRASTS AND DIFFERENCE IN DIFFERENCES
# ============================================================================

print('\n' + '=' * 96)
print('PART D — CONTAMINATION-STRATIFIED METHOD CONTRASTS ON RETAINED TEST')
print('=' * 96)

cx = np.load(CX_CACHE, allow_pickle=True)
cx_probs = cx['all_probs'].item()
y = cx['labels'].astype(int)
sex = cx['sex'].astype(int)

if len(y) != len(pooled_test):
    raise AssertionError(f'cache rows {len(y)} != pooled test rows {len(pooled_test)}')
if not np.array_equal(y, pooled_test['label'].to_numpy(dtype=int)):
    raise AssertionError('cache label order does not match pooled A/B/C/D test order')
if not np.array_equal(sex, pooled_test['sex_encoded'].to_numpy(dtype=int)):
    raise AssertionError('cache sex order does not match pooled A/B/C/D test order')

tp = pooled_test['patient_id']
seen_train_val = tp.isin(set(alloc.loc[alloc['split'].isin(['train', 'val']), 'patient_id']))
multi_client = tp.isin(multi_client_pids)
strict = (~seen_train_val) & (~multi_client)
strict = strict.to_numpy()

ens = {}
for cm in sorted({k[0] for k in cx_probs}):
    arrs = [np.asarray(p, float) for (m, s), p in cx_probs.items()
            if m == cm and len(p) == len(y)]
    if arrs:
        ens[CORE_LABEL.get(cm, cm)] = np.mean(np.stack(arrs), axis=0)

avail = [c for c in CONTRASTS if c[0] in ens and c[1] in ens]
pid = pooled_test['patient_id'].to_numpy()

strata = {'clean': strict, 'contaminated': ~strict}
print('Stratum sizes:', {k: (int(v.sum()), int(pd.Series(pid[v]).nunique())) for k, v in strata.items()})


def contrast(mask, a, b, metric, w=None):
    ya, pa, pb, sa = y[mask], ens[a][mask], ens[b][mask], sex[mask]
    if metric == 'auroc':
        return safe_auc(ya, pa, w) - safe_auc(ya, pb, w)
    return wg_auc(ya, pa, sa, w) - wg_auc(ya, pb, sa, w)


rng = np.random.default_rng(BOOT_SEED)
codes, counts_map = {}, {}
for k, m in strata.items():
    u, inv = np.unique(pid[m], return_inverse=True)
    codes[k] = (u, inv)

did_rows = []
for a, b in avail:
    for metric in ['auroc', 'worst_group_auroc']:
        pt = {k: contrast(strata[k], a, b, metric) for k in strata}
        boots = {k: [] for k in strata}
        inter = []
        for _ in range(BOOT):
            vals = {}
            for k, m in strata.items():
                u, inv = codes[k]
                samp = rng.integers(0, len(u), size=len(u))
                cnt = np.bincount(samp, minlength=len(u)).astype(float)
                vals[k] = contrast(m, a, b, metric, cnt[inv])
                boots[k].append(vals[k])
            inter.append(vals['clean'] - vals['contaminated'])
        row = {'comparison': f'{a} - {b}', 'metric': metric,
               'estimate_clean': pt['clean'], 'estimate_contaminated': pt['contaminated'],
               'interaction_clean_minus_contaminated': pt['clean'] - pt['contaminated']}
        for k in strata:
            v = np.asarray(boots[k], float)
            v = v[np.isfinite(v)]
            row[f'ci_low_{k}'], row[f'ci_high_{k}'] = np.percentile(v, [2.5, 97.5])
        iv = np.asarray(inter, float)
        iv = iv[np.isfinite(iv)]
        row['interaction_ci_low'], row['interaction_ci_high'] = np.percentile(iv, [2.5, 97.5])
        row['interaction_includes_zero'] = bool(row['interaction_ci_low'] <= 0 <= row['interaction_ci_high'])
        did_rows.append(row)

did = pd.DataFrame(did_rows)
did.to_csv(OUT / 'D_contamination_stratified_did.csv', index=False)
print(did.to_string(index=False))

# ============================================================================
# PART E — PREVALENCE-REWEIGHTED STRICT SUBSET
# ============================================================================

print('\n' + '=' * 96)
print('PART E — PREVALENCE CONFOUND IN THE STRICT SUBSET')
print('=' * 96)

prev_full = float(y.mean())
ys, ss = y[strict], sex[strict]
P, N = int((ys == 1).sum()), int((ys == 0).sum())
prev_strict = float(ys.mean())
print(f'full pooled prevalence {prev_full:.4f} | strict subset prevalence {prev_strict:.4f}')

# AUROC and worst-group AUROC are rank statistics over positive-negative pairs, so
# scaling every positive by a constant cancels. EO gap uses positives only and FPR
# gap uses negatives only, so a within-class constant cancels there too. Those four
# outcomes CANNOT be confounded by the prevalence difference. AUPRC and Brier mix
# the classes and are the only reported outcomes that move.
INVARIANT_METRICS = ['auroc', 'worst_group_auroc', 'eo_gap_0.5', 'fpr_gap_0.5']
PREV_METRICS = ['auprc', 'worst_group_auprc', 'brier']


def metric_val(mask, name, method, w):
    ya, pa, sa = y[mask], ens[method][mask], sex[mask]
    if name == 'auroc':
        return safe_auc(ya, pa, w)
    if name == 'worst_group_auroc':
        return wg_auc(ya, pa, sa, w)
    if name == 'auprc':
        return safe_ap(ya, pa, w)
    if name == 'worst_group_auprc':
        return wg_ap(ya, pa, sa, w)
    if name == 'eo_gap_0.5':
        return gap_at(ya, pa, sa, w, 'eo')
    if name == 'fpr_gap_0.5':
        return gap_at(ya, pa, sa, w, 'fpr')
    if name == 'brier':
        return float(np.average((pa - ya) ** 2, weights=w))
    raise ValueError(name)


if P == 0 or N == 0:
    print('Strict subset has a single class. Part E skipped.')
    prev_matched = pd.DataFrame()
    invariance = pd.DataFrame()
else:
    a_pos = prev_full * N / ((1.0 - prev_full) * P)
    w_flat = np.ones(int(strict.sum()), dtype=float)
    w_prev = np.where(ys == 1, a_pos, 1.0).astype(float)
    print(f'positive reweight factor to restore full-test prevalence: {a_pos:.4f}')

    # Demonstrated numerically rather than asserted.
    inv_rows = []
    for mth in ens:
        for name in INVARIANT_METRICS:
            v0 = metric_val(strict, name, mth, w_flat)
            v1 = metric_val(strict, name, mth, w_prev)
            inv_rows.append({'method': mth, 'metric': name,
                             'unweighted': v0, 'prevalence_reweighted': v1,
                             'abs_difference': abs(v1 - v0)})
    invariance = pd.DataFrame(inv_rows)
    invariance.to_csv(OUT / 'E_prevalence_invariance_check.csv', index=False)
    print('\nInvariance check: these outcomes cannot be confounded by the '
          'prevalence difference.')
    print(f"Max abs difference = {invariance['abs_difference'].max():.3e}")
    print(invariance.to_string(index=False))

    u, inv = np.unique(pid[strict], return_inverse=True)
    rng2 = np.random.default_rng(BOOT_SEED + 1)
    prev_rows = []
    for a, b in avail:
        for name in PREV_METRICS:
            pt_raw = metric_val(strict, name, a, w_flat) - metric_val(strict, name, b, w_flat)
            pt_adj = metric_val(strict, name, a, w_prev) - metric_val(strict, name, b, w_prev)
            vals = []
            for _ in range(BOOT):
                samp = rng2.integers(0, len(u), size=len(u))
                cnt = np.bincount(samp, minlength=len(u)).astype(float)
                w = cnt[inv] * w_prev
                if w.sum() <= 0:
                    continue
                vals.append(metric_val(strict, name, a, w) - metric_val(strict, name, b, w))
            v = np.asarray(vals, float)
            v = v[np.isfinite(v)]
            lo, hi = (np.percentile(v, [2.5, 97.5]) if len(v) else (np.nan, np.nan))
            prev_rows.append({'comparison': f'{a} - {b}', 'metric': name,
                              'strict_unadjusted': pt_raw,
                              'strict_prevalence_reweighted': pt_adj,
                              'ci_low': lo, 'ci_high': hi,
                              'includes_zero': bool(lo <= 0 <= hi) if np.isfinite(lo) else None})

    prev_matched = pd.DataFrame(prev_rows)
    prev_matched.to_csv(OUT / 'E_prevalence_reweighted_strict.csv', index=False)
    print('\nPrevalence-sensitive contrasts on the strict subset:')
    print(prev_matched.to_string(index=False))

# ============================================================================
# PACKAGE
# ============================================================================

meta = {
    'constants': {'G_ref': G_REF, 'rho': RHO, 'tau_ref': TAU_REF,
                  'alpha': ALPHA, 'kappa': KAPPA},
    'replay_max_abs_error_vs_logged_weights': max_err,
    'bootstrap_replicates': BOOT,
    'strict_test_images': int(strict.sum()),
    'strict_test_patients': int(pd.Series(pid[strict]).nunique()),
    'full_test_images': int(len(y)),
    'full_test_patients': int(pd.Series(pid).nunique()),
    'limits': [
        'Part C holds logged G_i^t fixed and isolates only the count-driven channel.',
        'Parts D and E condition on retained fixed ensembles and add no run uncertainty.',
        'No model was retrained or re-inferred in this cell.',
    ],
}
(OUT / 'run_metadata.json').write_text(json.dumps(meta, indent=2))

with zipfile.ZipFile(ZIP_OUT, 'w', zipfile.ZIP_DEFLATED) as z:
    for f in sorted(OUT.glob('*')):
        z.write(f, f.name)

print('\n' + '=' * 96)
print(f'Saved -> {OUT}')
print(f'Zip   -> {ZIP_OUT}')
print('=' * 96)

Mounted at /content/drive

PART A — VALIDATION-SET CONTAMINATION (THE SOURCE OF G_i^t AND r_i)
client  val_images  val_patients  val_images_seen_in_own_train  val_images_seen_in_other_client_train  val_images_seen_in_any_train  val_images_patient_crosses_clients  val_images_clean  frac_val_images_contaminated  val_m_pos_reported  val_f_pos_reported  r_i_reported  val_m_pos_clean  val_f_pos_clean  r_i_clean  r_i_change
     A        2250          2169                           533                                    370                           903                                 825              1248                      0.445333                 433                 467         1.000              177              205      0.885      -0.115
     B        1200          1163                           232                                    290                           522                                 513               612                      0.490000                 447                

In [ ]:
# ============================================================================
# FINAL ANALYSIS AND FIGURE CELL — FEDAVG AT FIVE SEEDS IN THE PRIMARY FAMILY
#
# WHAT THIS DOES
#   PART 0  Adds ('fedavg', 789) and ('fedavg', 1010) to nih_pred_cache.npz.
#           Needs GPU ONLY if those keys are missing. If they are already
#           cached, the whole cell runs on CPU.
#   PART 1  Per-seed NIH metrics for FedAvg, LPR, CADR, DWFA at five seeds.
#   PART 2  Patient-clustered bootstrap: fixed-ensemble, run-aware pointwise,
#           and single-step simultaneous max-t intervals.
#   PART 3  Threshold sweep, 37 operating points, EO gap and FPR gap.
#   PART 4  Leave-one-seed-out simultaneous intervals.
#   PART 5  Regenerates Figures 4, 5, 6, and 8 in the manuscript's style and
#           saves PNG (300 dpi) and PDF to Drive.
#
# PRIMARY FAMILY (declared before looking at any result)
#   DWFA - FedAvg, LPR - FedAvg, CADR - FedAvg
#   on overall AUROC and worst-group AUROC = SIX contrasts.
#   The three fairness-vs-fairness pairs are kept as SECONDARY.
#
# BEFORE RUNNING
#   If ('fedavg', 789) is already in nih_pred_cache.npz: no old cell needed.
#   If it is NOT: run these first, in this order, on a GPU runtime:
#     "# external validation"   (NIH CELL 1, stages images, defines NIH_IMG)
#     "# NIH CELL 2 — Build the effusion cohort CSV in YOUR schema"
#   Do NOT run "# PAIRED SIGNIFICANCE TEST on NIH" or the CheXpert twin.
#   Both take a load-branch when the cache exists and will not add seeds.
#
# OUTPUT
#   results/final_figures/  (figures + every underlying CSV)
#   results/final_figures.zip
# ============================================================================

from google.colab import drive
drive.mount('/content/drive', force_remount=False)

from pathlib import Path
import os, gc, json, shutil, zipfile
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from sklearn.metrics import roc_auc_score

BASE = Path('/content/drive/MyDrive/FairFedCXR')
RESULTS = BASE / 'results'
CKPTS = BASE / 'checkpoints'
NIH_CSV = RESULTS / 'nih_effusion_cohort.csv'
CACHE = RESULTS / 'nih_pred_cache.npz'

OUT = RESULTS / 'final_figures'
ZIP_OUT = RESULTS / 'final_figures.zip'

METHOD_LABELS = {'fedavg': 'FedAvg', 'qfedavg': 'LPR',
                 'fairfed': 'CADR', 'dwfa': 'DWFA'}
METHODS = ['FedAvg', 'LPR', 'CADR', 'DWFA']
SEEDS = [42, 123, 456, 789, 1010]

PRIMARY_PAIRS = [('DWFA', 'FedAvg'), ('LPR', 'FedAvg'), ('CADR', 'FedAvg')]
SECONDARY_PAIRS = [('DWFA', 'LPR'), ('DWFA', 'CADR'), ('LPR', 'CADR')]
PRIMARY_METRICS = ['auroc', 'worst_group_auroc']

N_BOOT = 2000
BOOT_SEED = 20260723
THRESHOLDS = np.round(np.arange(0.05, 0.951, 0.025), 4)

COLORS = {'FedAvg': '#7f7f7f', 'LPR': '#1f77b4',
          'CADR': '#ff7f0e', 'DWFA': '#2ca02c'}

if OUT.exists():
    shutil.rmtree(OUT)
OUT.mkdir(parents=True, exist_ok=True)

if not CACHE.exists():
    raise FileNotFoundError(f'{CACHE} not found.')
if not NIH_CSV.exists():
    raise FileNotFoundError(f'{NIH_CSV} not found. Run NIH CELL 2 first.')


# ============================================================================
# PART 0 — ENSURE FIVE FEDAVG SEEDS ARE CACHED
# ============================================================================

print('=' * 92)
print('PART 0 — PREDICTION CACHE')
print('=' * 92)

_c = np.load(CACHE, allow_pickle=True)
all_probs = _c['all_probs'].item()
labels = _c['labels'].astype(np.int8)
sex = _c['sex'].astype(np.int8)

need = [(m, s) for m in ['fedavg'] for s in SEEDS if (m, s) not in all_probs]
print(f'Cached keys: {len(all_probs)}')
print(f'FedAvg seeds present: {sorted(s for m, s in all_probs if m == "fedavg")}')

if need:
    print(f'\nMissing and will be inferred on GPU: {need}')
    import torch
    import torchvision.models as tvm
    from torchvision import transforms
    from torch.utils.data import Dataset, DataLoader
    from PIL import Image

    if not torch.cuda.is_available():
        raise RuntimeError('GPU required to add the missing FedAvg seeds. '
                           'Switch to a GPU runtime and rerun.')
    if 'NIH_IMG' not in globals() or not os.path.isdir(str(NIH_IMG)):
        raise RuntimeError('NIH_IMG not in session. Run "# external validation" '
                           '(NIH CELL 1) first.')

    dev = torch.device('cuda')
    _cohort = pd.read_csv(NIH_CSV, low_memory=False).reset_index(drop=True)
    _tf = transforms.Compose([
        transforms.Resize((224, 224)), transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])])

    class _DS(Dataset):
        def __init__(s, df, root, tf):
            s.df, s.root, s.tf = df.reset_index(drop=True), root, tf
        def __len__(s):
            return len(s.df)
        def __getitem__(s, i):
            r = s.df.iloc[i]
            img = Image.open(os.path.join(s.root, str(r['Path']))).convert('RGB')
            return s.tf(img), np.int8(r['label']), np.int8(r['sex_encoded'])

    _loader = DataLoader(_DS(_cohort, str(NIH_IMG), _tf), batch_size=128,
                         shuffle=False, num_workers=4, pin_memory=True)

    def _state(o):
        if isinstance(o, dict):
            for k in ['state_dict', 'model_state_dict', 'global_state', 'model']:
                if k in o and isinstance(o[k], dict):
                    return o[k]
        return o

    for m, sd in need:
        ck = CKPTS / f'{m}_seed{sd}.pt'
        if not ck.exists():
            raise FileNotFoundError(f'{ck} missing. Finish training seed {sd} first.')
        net = tvm.densenet121(weights=None)
        net.classifier = torch.nn.Linear(net.classifier.in_features, 1)
        net.load_state_dict(_state(torch.load(ck, map_location='cpu',
                                              weights_only=False)), strict=False)
        net = net.to(dev).eval()
        P = []
        with torch.inference_mode():
            for imgs, _, _ in _loader:
                with torch.amp.autocast('cuda', dtype=torch.float16):
                    lg = net(imgs.to(dev, non_blocking=True)).squeeze(1)
                P.append(torch.sigmoid(lg.float()).cpu().numpy().astype(np.float32))
        all_probs[(m, sd)] = np.concatenate(P)
        print(f'  cached {m} s{sd}  (n={len(all_probs[(m, sd)])})')
        del net
        gc.collect()
        torch.cuda.empty_cache()

    tmp = Path('/content/_nih_cache.npz')
    np.savez(tmp, all_probs=all_probs, labels=labels, sex=sex)
    shutil.copy2(tmp, CACHE)
    print(f'Cache updated -> {CACHE}')
else:
    print('\nAll five FedAvg seeds already cached. Running CPU-only.')

nih = pd.read_csv(NIH_CSV, low_memory=False).reset_index(drop=True)
if len(nih) != len(labels):
    raise AssertionError(f'cohort rows {len(nih)} != cache rows {len(labels)}')
if not np.array_equal(nih['label'].to_numpy(np.int8), labels):
    raise AssertionError('cohort label order does not match the cache')

patient_code, patient_levels = pd.factorize(
    nih['Patient ID'].astype(str).str.strip(), sort=True)
patient_code = patient_code.astype(np.int32)
n_patients = len(patient_levels)
print(f'\nCohort: {len(nih):,} images | {n_patients:,} patients | '
      f'prevalence {labels.mean():.4f}')

probs = {}
for (cm, sd), p in all_probs.items():
    lab = METHOD_LABELS.get(cm)
    if lab in METHODS and sd in SEEDS and len(p) == len(labels):
        probs[(lab, sd)] = np.asarray(p, np.float32)

for m in METHODS:
    got = sorted(s for (mm, s) in probs if mm == m)
    print(f'  {m:<8} seeds {got}')
    if got != SEEDS:
        raise AssertionError(f'{m} does not have all five seeds; got {got}')


# ============================================================================
# METRIC HELPERS
# ============================================================================

# ============================================================================
# FAST WEIGHTED AUC
#   roc_auc_score with sample_weight is far too slow inside a 2000-replicate
#   bootstrap. This computes the same quantity from a one-time sort plus a
#   bincount per replicate, with exact mid-rank handling of tied scores.
# ============================================================================

M_MASK = (sex == 1)
F_MASK = (sex == 0)
SUBSETS = {'all': np.ones(len(labels), bool), 'male': M_MASK, 'female': F_MASK}


def build_struct(p, submask):
    rows = np.flatnonzero(submask)
    if len(rows) == 0 or np.unique(labels[rows]).size < 2:
        return None
    order = rows[np.argsort(p[rows], kind='mergesort')]
    _, gidx = np.unique(p[order], return_inverse=True)
    y = labels[order]
    return {'rows': order, 'gidx': gidx.astype(np.int32),
            'G': int(gidx.max()) + 1,
            'pos': (y == 1).astype(np.float64),
            'neg': (y == 0).astype(np.float64)}


def wauc(st, w=None):
    if st is None:
        return np.nan
    ws = np.ones(len(st['rows'])) if w is None else w[st['rows']]
    wp, wn = ws * st['pos'], ws * st['neg']
    Wp, Wn = wp.sum(), wn.sum()
    if Wp <= 0 or Wn <= 0:
        return np.nan
    npg = np.bincount(st['gidx'], weights=wn, minlength=st['G'])
    ppg = np.bincount(st['gidx'], weights=wp, minlength=st['G'])
    below = np.concatenate(([0.0], np.cumsum(npg)[:-1]))
    return float((ppg * (below + 0.5 * npg)).sum() / (Wp * Wn))


def struct_set(p):
    return {k: build_struct(p, m) for k, m in SUBSETS.items()}


def metrics_from(sts, w=None):
    a = wauc(sts['all'], w)
    am, af = wauc(sts['male'], w), wauc(sts['female'], w)
    wg = np.nan if not (np.isfinite(am) and np.isfinite(af)) else min(am, af)
    return {'auroc': a, 'male_auroc': am, 'female_auroc': af,
            'worst_group_auroc': wg}


def metrics(p, w=None):
    return metrics_from(struct_set(p), w)


def _rate(mask, pred, w):
    d = w[mask].sum()
    return np.nan if d <= 0 else float(w[mask & (pred == 1)].sum() / d)


def gaps(p, thr, w=None):
    w = np.ones(len(p)) if w is None else w
    pred = (p >= thr).astype(np.int8)
    pos, neg = (labels == 1), (labels == 0)
    eo = abs(_rate(M_MASK & pos, pred, w) - _rate(F_MASK & pos, pred, w))
    fp = abs(_rate(M_MASK & neg, pred, w) - _rate(F_MASK & neg, pred, w))
    return eo, fp


# ============================================================================
# PART 1 — PER-SEED METRICS
# ============================================================================

print('\n' + '=' * 92)
print('PART 1 — PER-SEED NIH METRICS')
print('=' * 92)

rows = []
for m in METHODS:
    for sd in SEEDS:
        r = metrics(probs[(m, sd)])
        eo, fp = gaps(probs[(m, sd)], 0.5)
        r.update({'method': m, 'seed': sd, 'eo_gap_50': eo, 'fpr_gap_50': fp})
        rows.append(r)
per_seed = pd.DataFrame(rows)
per_seed.to_csv(OUT / 'per_seed_metrics.csv', index=False)

summary = (per_seed.groupby('method')[['auroc', 'worst_group_auroc',
                                       'eo_gap_50', 'fpr_gap_50']]
           .agg(['mean', 'std']).round(4).loc[METHODS])
summary.to_csv(OUT / 'method_summary_mean_sd.csv')
print(summary.to_string())


# ============================================================================
# PART 2 — PATIENT-CLUSTERED BOOTSTRAP
# ============================================================================

print('\n' + '=' * 92)
print('PART 2 — PATIENT-CLUSTERED INFERENCE')
print('=' * 92)

ens = {m: np.mean([probs[(m, s)] for s in SEEDS], axis=0) for m in METHODS}
ALL_PAIRS = PRIMARY_PAIRS + SECONDARY_PAIRS

ST_RUN = {(m, s): struct_set(probs[(m, s)]) for m in METHODS for s in SEEDS}
ST_ENS = {m: struct_set(ens[m]) for m in METHODS}

rng = np.random.default_rng(BOOT_SEED)
boot_w = np.empty((N_BOOT, len(labels)), np.float32)
for b in range(N_BOOT):
    samp = rng.integers(0, n_patients, size=n_patients)
    boot_w[b] = np.bincount(samp, minlength=n_patients).astype(np.float32)[patient_code]

# A[method][metric] -> (seeds x N_BOOT); computed ONCE and reused by the
# run-aware, simultaneous, and leave-one-seed-out analyses.
print(f'Bootstrapping {N_BOOT} patient-cluster replicates ...')
A = {m: {k: np.empty((len(SEEDS), N_BOOT)) for k in PRIMARY_METRICS}
     for m in METHODS}
E = {m: {k: np.empty(N_BOOT) for k in PRIMARY_METRICS} for m in METHODS}
for b in range(N_BOOT):
    w = boot_w[b]
    for m in METHODS:
        for si, s in enumerate(SEEDS):
            r = metrics_from(ST_RUN[(m, s)], w)
            for k in PRIMARY_METRICS:
                A[m][k][si, b] = r[k]
        r = metrics_from(ST_ENS[m], w)
        for k in PRIMARY_METRICS:
            E[m][k][b] = r[k]

rng_run = np.random.default_rng(BOOT_SEED + 1)
run_draw = {m: rng_run.integers(0, len(SEEDS), size=(N_BOOT, len(SEEDS)))
            for m in METHODS}
run_boot = {(m, k): np.array([np.nanmean(A[m][k][run_draw[m][b], b])
                              for b in range(N_BOOT)])
            for m in METHODS for k in PRIMARY_METRICS}
fix_boot = {(m, k): E[m][k] for m in METHODS for k in PRIMARY_METRICS}

POINT_FIX = {m: metrics_from(ST_ENS[m]) for m in METHODS}


def _pt(a, b, met, kind):
    if kind == 'fixed':
        return POINT_FIX[a][met] - POINT_FIX[b][met]
    return (per_seed.loc[per_seed.method == a, met].mean()
            - per_seed.loc[per_seed.method == b, met].mean())


results, std_mat, keys = [], [], []
for a, b in ALL_PAIRS:
    for met in PRIMARY_METRICS:
        fx = fix_boot[(a, met)] - fix_boot[(b, met)]
        rn = run_boot[(a, met)] - run_boot[(b, met)]
        row = {'comparison': f'{a} - {b}', 'metric': met,
               'family': 'primary' if (a, b) in PRIMARY_PAIRS else 'secondary',
               'fixed_estimate': _pt(a, b, met, 'fixed'),
               'fixed_lo': np.nanpercentile(fx, 2.5),
               'fixed_hi': np.nanpercentile(fx, 97.5),
               'run_estimate': _pt(a, b, met, 'run'),
               'run_lo': np.nanpercentile(rn, 2.5),
               'run_hi': np.nanpercentile(rn, 97.5)}
        results.append(row)
        if row['family'] == 'primary':
            std_mat.append(rn)
            keys.append((f'{a} - {b}', met))

std_mat = np.vstack(std_mat)
centred = std_mat - np.nanmean(std_mat, axis=1, keepdims=True)
sds = np.nanstd(std_mat, axis=1, keepdims=True)
sds[sds == 0] = np.nan
crit = float(np.nanpercentile(np.nanmax(np.abs(centred / sds), axis=0), 95))
print(f'Single-step max-t critical value over six primary contrasts: {crit:.3f}')

res = pd.DataFrame(results)
res['sim_lo'] = np.nan
res['sim_hi'] = np.nan
for i, (cmp_, met) in enumerate(keys):
    j = res.index[(res.comparison == cmp_) & (res.metric == met)][0]
    sd_i = float(np.nanstd(std_mat[i]))
    res.at[j, 'sim_lo'] = res.at[j, 'run_estimate'] - crit * sd_i
    res.at[j, 'sim_hi'] = res.at[j, 'run_estimate'] + crit * sd_i
for c in ['fixed', 'run', 'sim']:
    lo, hi = res[f'{c}_lo'], res[f'{c}_hi']
    ok = lo.notna() & hi.notna()
    res[f'{c}_excludes_zero'] = np.where(ok, ~((lo <= 0) & (0 <= hi)), np.nan)
    res[f'{c}_excludes_zero'] = res[f'{c}_excludes_zero'].astype('object')
    res.loc[~ok, f'{c}_excludes_zero'] = 'n/a (simultaneous control covers the primary family only)'
res.to_csv(OUT / 'primary_contrasts.csv', index=False)
print(res.round(6).to_string(index=False))


# ============================================================================
# PART 3 — THRESHOLD SWEEP
# ============================================================================

print('\n' + '=' * 92)
print('PART 3 — THRESHOLD SWEEP, 37 OPERATING POINTS')
print('=' * 92)

trow = []
for m in METHODS:
    for sd in SEEDS:
        for t in THRESHOLDS:
            eo, fp = gaps(probs[(m, sd)], t)
            trow.append({'method': m, 'seed': sd, 'threshold': t,
                         'eo_gap': eo, 'fpr_gap': fp})
thr_df = pd.DataFrame(trow)
thr_df.to_csv(OUT / 'threshold_sweep.csv', index=False)

thr_mean = thr_df.groupby(['method', 'threshold'])[['eo_gap', 'fpr_gap']].mean()
win_eo = (thr_mean['eo_gap'].unstack(0)[METHODS].idxmin(axis=1).value_counts())
win_fp = (thr_mean['fpr_gap'].unstack(0)[METHODS].idxmin(axis=1).value_counts())
print('Lowest mean EO gap  count by method:', win_eo.to_dict())
print('Lowest mean FPR gap count by method:', win_fp.to_dict())


# ============================================================================
# PART 4 — LEAVE ONE SEED OUT
# ============================================================================

print('\n' + '=' * 92)
print('PART 4 — LEAVE-ONE-SEED-OUT SIMULTANEOUS INTERVALS')
print('=' * 92)

loo = []
for drop in SEEDS:
    keep = [i for i, s in enumerate(SEEDS) if s != drop]
    kb, kk = [], []
    for a, b in PRIMARY_PAIRS:
        for met in PRIMARY_METRICS:
            kb.append(np.nanmean(A[a][met][keep], axis=0)
                      - np.nanmean(A[b][met][keep], axis=0))
            kk.append((f'{a} - {b}', met))
    kb = np.vstack(kb)
    c = kb - np.nanmean(kb, axis=1, keepdims=True)
    s_ = np.nanstd(kb, axis=1, keepdims=True)
    s_[s_ == 0] = np.nan
    cv = float(np.nanpercentile(np.nanmax(np.abs(c / s_), axis=0), 95))
    for i, (cmp_, met) in enumerate(kk):
        est = float(np.nanmean(kb[i]))
        sd_i = float(np.nanstd(kb[i]))
        loo.append({'omitted_seed': drop, 'comparison': cmp_, 'metric': met,
                    'estimate': est, 'sim_lo': est - cv * sd_i,
                    'sim_hi': est + cv * sd_i})
loo_df = pd.DataFrame(loo)
loo_df['excludes_zero'] = ~((loo_df.sim_lo <= 0) & (0 <= loo_df.sim_hi))
loo_df.to_csv(OUT / 'leave_one_seed_out.csv', index=False)
print(f'Intervals excluding zero: {int(loo_df.excludes_zero.sum())} of {len(loo_df)}')


# ============================================================================
# PART 5 — FIGURES
# ============================================================================

print('\n' + '=' * 92)
print('PART 5 — FIGURES')
print('=' * 92)

plt.rcParams.update({'font.size': 9, 'axes.linewidth': 0.8,
                     'font.family': 'DejaVu Sans'})


def save(fig, stem):
    fig.savefig(OUT / f'{stem}.png', dpi=300, bbox_inches='tight')
    fig.savefig(OUT / f'{stem}.pdf', bbox_inches='tight')
    plt.close(fig)
    print(f'  saved {stem}.png / .pdf')


# ---- Figure 4 -------------------------------------------------------------
fig = plt.figure(figsize=(11, 3.6))
gs = fig.add_gridspec(1, 3, wspace=0.32)
for k, (met, ttl) in enumerate([('auroc', 'Overall AUROC'),
                                ('worst_group_auroc', 'Worst-group AUROC')]):
    ax = fig.add_subplot(gs[0, k])
    for xi, m in enumerate(METHODS):
        v = per_seed.loc[per_seed.method == m, met].to_numpy()
        ax.scatter(np.full(len(v), xi) + np.linspace(-.12, .12, len(v)), v,
                   s=22, color=COLORS[m], zorder=3)
        ax.errorbar(xi, v.mean(), yerr=v.std(ddof=1), fmt='D', color='black',
                    ms=5, capsize=3, lw=1.1, zorder=4)
    ax.set_xticks(range(len(METHODS)))
    ax.set_xticklabels(METHODS, rotation=30, ha='right')
    ax.set_ylabel(ttl)
    ax.set_title(f'{"AB"[k]}   {ttl}', loc='left', fontweight='bold')
    ax.grid(axis='y', ls=':', lw=.5, alpha=.6)

ax = fig.add_subplot(gs[0, 2])
for met, lab, cl, mk in [('auroc', 'Overall AUROC', '#2ca02c', 'o'),
                         ('worst_group_auroc', 'Worst-group AUROC', '#1f77b4', 's')]:
    d = [per_seed.loc[(per_seed.method == 'DWFA') & (per_seed.seed == s), met].item()
         - per_seed.loc[(per_seed.method == 'FedAvg') & (per_seed.seed == s), met].item()
         for s in SEEDS]
    ax.plot(range(len(SEEDS)), np.array(d) * 1e3, marker=mk, color=cl, label=lab, lw=1.3)
ax.axhline(0, color='black', ls='--', lw=.9)
ax.set_xticks(range(len(SEEDS)))
ax.set_xticklabels(SEEDS)
ax.set_xlabel('Numeric seed')
ax.set_ylabel(r'DWFA $-$ FedAvg ($\times 10^{-3}$)')
ax.set_title('C   Same-seed differences', loc='left', fontweight='bold')
ax.legend(frameon=False, fontsize=7.5)
ax.grid(axis='y', ls=':', lw=.5, alpha=.6)
save(fig, 'figure4_external_nih_across_runs')

# ---- Figure 5 -------------------------------------------------------------
prim = res[res.family == 'primary'].reset_index(drop=True)
lab = [f'{r.comparison} | '
       f'{"Overall" if r.metric == "auroc" else "Worst-group"}'
       for r in prim.itertuples()]
y = np.arange(len(prim))[::-1]
fig, ax = plt.subplots(figsize=(7.6, 4.4))
for off, (lo, hi, est, cl, mk, nm) in enumerate([
        ('fixed_lo', 'fixed_hi', 'fixed_estimate', '#1f77b4', 'o',
         'Fixed seed ensemble: pointwise 95% CI'),
        ('run_lo', 'run_hi', 'run_estimate', '#ff7f0e', 's',
         'Run-aware: pointwise 95% CI'),
        ('sim_lo', 'sim_hi', 'run_estimate', '#d62728', 'D',
         'Run-aware: simultaneous 95% CI')]):
    dy = (1 - off) * 0.22
    ax.errorbar(prim[est] * 1e3, y + dy,
                xerr=[(prim[est] - prim[lo]) * 1e3, (prim[hi] - prim[est]) * 1e3],
                fmt=mk, color=cl, ms=5, lw=1.2, capsize=2.5, label=nm)
ax.axvline(0, color='black', ls='--', lw=1)
ax.set_yticks(y)
ax.set_yticklabels(lab)
ax.set_xlabel(r'AUROC difference ($\times 10^{-3}$; first method minus second)')
ax.legend(frameon=False, fontsize=7.5, loc='upper center',
          bbox_to_anchor=(0.5, -0.16), ncol=1)
ax.grid(axis='x', ls=':', lw=.5, alpha=.6)
save(fig, 'figure5_primary_discrimination_contrasts')

# ---- Figure 6 -------------------------------------------------------------
fig, axes = plt.subplots(1, 2, figsize=(10, 3.8))
for ax, col, ttl in [(axes[0], 'eo_gap', 'Equal-opportunity gap'),
                     (axes[1], 'fpr_gap', 'False-positive-rate gap')]:
    for m in METHODS:
        d = thr_df[thr_df.method == m]
        g = d.groupby('threshold')[col]
        ax.plot(THRESHOLDS, g.mean().to_numpy(), color=COLORS[m], lw=1.4, label=m)
        ax.fill_between(THRESHOLDS, g.min().to_numpy(), g.max().to_numpy(),
                        color=COLORS[m], alpha=.13, lw=0)
    ax.axvline(0.5, color='grey', ls='--', lw=.9)
    ax.set_xlabel('Decision threshold')
    ax.set_ylabel('Gap')
    ax.set_title(ttl)
    ax.legend(frameon=False, fontsize=7.5)
    ax.grid(ls=':', lw=.5, alpha=.6)
save(fig, 'figure6_threshold_sensitivity')

# ---- Figure 8 -------------------------------------------------------------
combos = [(f'{a} - {b}', m) for a, b in PRIMARY_PAIRS for m in PRIMARY_METRICS]
y = np.arange(len(combos))[::-1]
cmap = plt.get_cmap('tab10')
fig, ax = plt.subplots(figsize=(8, 4.8))
for si, drop in enumerate(SEEDS):
    d = loo_df[loo_df.omitted_seed == drop]
    xs, los, his = [], [], []
    for cmp_, met in combos:
        r = d[(d.comparison == cmp_) & (d.metric == met)].iloc[0]
        xs.append(r.estimate * 1e3)
        los.append((r.estimate - r.sim_lo) * 1e3)
        his.append((r.sim_hi - r.estimate) * 1e3)
    ax.errorbar(xs, y + (si - 2) * 0.14, xerr=[los, his], fmt='o', ms=3.6,
                lw=1.0, capsize=2, color=cmap(si), label=f'Omit {drop}')
ax.axvline(0, color='black', ls='--', lw=1)
ax.set_yticks(y)
ax.set_yticklabels([f'{c} | {"Overall" if m == "auroc" else "Worst-group"}'
                    for c, m in combos])
ax.set_xlabel(r'AUROC difference ($\times 10^{-3}$; simultaneous 95% CI)')
ax.legend(frameon=False, fontsize=7.5, ncol=5, loc='upper center',
          bbox_to_anchor=(0.5, 1.13))
ax.grid(axis='x', ls=':', lw=.5, alpha=.6)
save(fig, 'figure8_leave_one_seed_out')


# ============================================================================
# PACKAGE
# ============================================================================

(OUT / 'run_metadata.json').write_text(json.dumps({
    'methods': METHODS, 'seeds': SEEDS,
    'primary_pairs': [f'{a} - {b}' for a, b in PRIMARY_PAIRS],
    'secondary_pairs': [f'{a} - {b}' for a, b in SECONDARY_PAIRS],
    'primary_metrics': PRIMARY_METRICS,
    'bootstrap_replicates': N_BOOT, 'bootstrap_seed': BOOT_SEED,
    'maxt_critical_value': crit,
    'n_images': int(len(labels)), 'n_patients': int(n_patients),
}, indent=2))

with zipfile.ZipFile(ZIP_OUT, 'w', zipfile.ZIP_DEFLATED) as z:
    for f in sorted(OUT.glob('*')):
        z.write(f, f.name)

n_fix = int(res[res.family == 'primary'].fixed_excludes_zero.sum())
n_run = int(res[res.family == 'primary'].run_excludes_zero.sum())
n_sim = int(res[res.family == 'primary'].sim_excludes_zero.sum())
print('\n' + '=' * 92)
print(f'Primary family, six contrasts:')
print(f'  fixed-ensemble intervals excluding zero      : {n_fix} of 6')
print(f'  run-aware pointwise intervals excluding zero : {n_run} of 6')
print(f'  run-aware simultaneous excluding zero        : {n_sim} of 6')
print(f'\nSaved -> {OUT}')
print(f'Zip   -> {ZIP_OUT}')
print('=' * 92)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
PART 0 — PREDICTION CACHE
Cached keys: 18
FedAvg seeds present: [42, 123, 456]

Missing and will be inferred on GPU: [('fedavg', 789), ('fedavg', 1010)]


/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py:424: UserWarning: This DataLoader will create 4 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_rationality()
/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py:432: UserWarning: This DataLoader will create 4 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_rationality()


  cached fedavg s789  (n=112120)
  cached fedavg s1010  (n=112120)
Cache updated -> /content/drive/MyDrive/FairFedCXR/results/nih_pred_cache.npz

Cohort: 112,120 images | 30,805 patients | prevalence 0.1187
  FedAvg   seeds [42, 123, 456, 789, 1010]
  LPR      seeds [42, 123, 456, 789, 1010]
  CADR     seeds [42, 123, 456, 789, 1010]
  DWFA     seeds [42, 123, 456, 789, 1010]

PART 1 — PER-SEED NIH METRICS
         auroc         worst_group_auroc         eo_gap_50         fpr_gap_50        
          mean     std              mean     std      mean     std       mean     std
method                                                                               
FedAvg  0.8572  0.0023            0.8557  0.0017    0.0270  0.0050     0.0191  0.0022
LPR     0.8554  0.0016            0.8537  0.0018    0.0285  0.0062     0.0185  0.0042
CADR    0.8565  0.0024            0.8548  0.0017    0.0231  0.0069     0.0161  0.0051
DWFA    0.8578  0.0022            0.8557  0.0019    0.0253  0.0071     0.0

In [ ]:
# ============================================================================
# FINAL ANALYSIS AND FIGURE CELL — FEDAVG AT FIVE SEEDS IN THE PRIMARY FAMILY
#
# WHAT THIS DOES
#   PART 0  Adds ('fedavg', 789) and ('fedavg', 1010) to nih_pred_cache.npz.
#           Needs GPU ONLY if those keys are missing. If they are already
#           cached, the whole cell runs on CPU.
#   PART 1  Per-seed NIH metrics for FedAvg, LPR, CADR, DWFA at five seeds.
#   PART 2  Patient-clustered bootstrap: fixed-ensemble, run-aware pointwise,
#           and single-step simultaneous max-t intervals.
#   PART 3  Threshold sweep, 37 operating points, EO gap and FPR gap.
#   PART 4  Leave-one-seed-out simultaneous intervals.
#   PART 5  Regenerates Figures 4, 5, 6, and 8 in the manuscript's style and
#           saves PNG (300 dpi) and PDF to Drive.
#
# PRIMARY FAMILY (declared before looking at any result)
#   DWFA - FedAvg, LPR - FedAvg, CADR - FedAvg
#   on overall AUROC and worst-group AUROC = SIX contrasts.
#   The three fairness-vs-fairness pairs are kept as SECONDARY.
#
# BEFORE RUNNING
#   If ('fedavg', 789) is already in nih_pred_cache.npz: no old cell needed.
#   If it is NOT: run these first, in this order, on a GPU runtime:
#     "# external validation"   (NIH CELL 1, stages images, defines NIH_IMG)
#     "# NIH CELL 2 — Build the effusion cohort CSV in YOUR schema"
#   Do NOT run "# PAIRED SIGNIFICANCE TEST on NIH" or the CheXpert twin.
#   Both take a load-branch when the cache exists and will not add seeds.
#
# OUTPUT
#   results/final_figures/  (figures + every underlying CSV)
#   results/final_figures.zip
# ============================================================================

from google.colab import drive
drive.mount('/content/drive', force_remount=False)

from pathlib import Path
import os, gc, json, shutil, zipfile
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from sklearn.metrics import roc_auc_score

BASE = Path('/content/drive/MyDrive/FairFedCXR')
RESULTS = BASE / 'results'
CKPTS = BASE / 'checkpoints'
NIH_CSV = RESULTS / 'nih_effusion_cohort.csv'
CACHE = RESULTS / 'nih_pred_cache.npz'

OUT = RESULTS / 'final_figures'
ZIP_OUT = RESULTS / 'final_figures.zip'

METHOD_LABELS = {'fedavg': 'FedAvg', 'qfedavg': 'LPR',
                 'fairfed': 'CADR', 'dwfa': 'DWFA'}
METHODS = ['FedAvg', 'LPR', 'CADR', 'DWFA']
SEEDS = [42, 123, 456, 789, 1010]

PRIMARY_PAIRS = [('DWFA', 'FedAvg'), ('LPR', 'FedAvg'), ('CADR', 'FedAvg')]
SECONDARY_PAIRS = [('DWFA', 'LPR'), ('DWFA', 'CADR'), ('LPR', 'CADR')]
PRIMARY_METRICS = ['auroc', 'worst_group_auroc']

N_BOOT = 2000
BOOT_SEED = 20260723
THRESHOLDS = np.round(np.arange(0.05, 0.951, 0.025), 4)

COLORS = {'FedAvg': '#7f7f7f', 'LPR': '#1f77b4',
          'CADR': '#ff7f0e', 'DWFA': '#2ca02c'}

if OUT.exists():
    shutil.rmtree(OUT)
OUT.mkdir(parents=True, exist_ok=True)

if not CACHE.exists():
    raise FileNotFoundError(f'{CACHE} not found.')
if not NIH_CSV.exists():
    raise FileNotFoundError(f'{NIH_CSV} not found. Run NIH CELL 2 first.')


# ============================================================================
# PART 0 — ENSURE FIVE FEDAVG SEEDS ARE CACHED
# ============================================================================

print('=' * 92)
print('PART 0 — PREDICTION CACHE')
print('=' * 92)

_c = np.load(CACHE, allow_pickle=True)
all_probs = _c['all_probs'].item()
labels = _c['labels'].astype(np.int8)
sex = _c['sex'].astype(np.int8)

need = [(m, s) for m in ['fedavg'] for s in SEEDS if (m, s) not in all_probs]
print(f'Cached keys: {len(all_probs)}')
print(f'FedAvg seeds present: {sorted(s for m, s in all_probs if m == "fedavg")}')

if need:
    print(f'\nMissing and will be inferred on GPU: {need}')
    import torch
    import torchvision.models as tvm
    from torchvision import transforms
    from torch.utils.data import Dataset, DataLoader
    from PIL import Image

    if not torch.cuda.is_available():
        raise RuntimeError('GPU required to add the missing FedAvg seeds. '
                           'Switch to a GPU runtime and rerun.')
    if 'NIH_IMG' not in globals() or not os.path.isdir(str(NIH_IMG)):
        raise RuntimeError('NIH_IMG not in session. Run "# external validation" '
                           '(NIH CELL 1) first.')

    dev = torch.device('cuda')
    _cohort = pd.read_csv(NIH_CSV, low_memory=False).reset_index(drop=True)
    _tf = transforms.Compose([
        transforms.Resize((224, 224)), transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])])

    class _DS(Dataset):
        def __init__(s, df, root, tf):
            s.df, s.root, s.tf = df.reset_index(drop=True), root, tf
        def __len__(s):
            return len(s.df)
        def __getitem__(s, i):
            r = s.df.iloc[i]
            img = Image.open(os.path.join(s.root, str(r['Path']))).convert('RGB')
            return s.tf(img), np.int8(r['label']), np.int8(r['sex_encoded'])

    _loader = DataLoader(_DS(_cohort, str(NIH_IMG), _tf), batch_size=128,
                         shuffle=False, num_workers=4, pin_memory=True)

    def _state(o):
        if isinstance(o, dict):
            for k in ['state_dict', 'model_state_dict', 'global_state', 'model']:
                if k in o and isinstance(o[k], dict):
                    return o[k]
        return o

    for m, sd in need:
        ck = CKPTS / f'{m}_seed{sd}.pt'
        if not ck.exists():
            raise FileNotFoundError(f'{ck} missing. Finish training seed {sd} first.')
        net = tvm.densenet121(weights=None)
        net.classifier = torch.nn.Linear(net.classifier.in_features, 1)
        net.load_state_dict(_state(torch.load(ck, map_location='cpu',
                                              weights_only=False)), strict=False)
        net = net.to(dev).eval()
        P = []
        with torch.inference_mode():
            for imgs, _, _ in _loader:
                with torch.amp.autocast('cuda', dtype=torch.float16):
                    lg = net(imgs.to(dev, non_blocking=True)).squeeze(1)
                P.append(torch.sigmoid(lg.float()).cpu().numpy().astype(np.float32))
        all_probs[(m, sd)] = np.concatenate(P)
        print(f'  cached {m} s{sd}  (n={len(all_probs[(m, sd)])})')
        del net
        gc.collect()
        torch.cuda.empty_cache()

    tmp = Path('/content/_nih_cache.npz')
    np.savez(tmp, all_probs=all_probs, labels=labels, sex=sex)
    shutil.copy2(tmp, CACHE)
    print(f'Cache updated -> {CACHE}')
else:
    print('\nAll five FedAvg seeds already cached. Running CPU-only.')

nih = pd.read_csv(NIH_CSV, low_memory=False).reset_index(drop=True)
if len(nih) != len(labels):
    raise AssertionError(f'cohort rows {len(nih)} != cache rows {len(labels)}')
if not np.array_equal(nih['label'].to_numpy(np.int8), labels):
    raise AssertionError('cohort label order does not match the cache')

patient_code, patient_levels = pd.factorize(
    nih['Patient ID'].astype(str).str.strip(), sort=True)
patient_code = patient_code.astype(np.int32)
n_patients = len(patient_levels)
print(f'\nCohort: {len(nih):,} images | {n_patients:,} patients | '
      f'prevalence {labels.mean():.4f}')

probs = {}
for (cm, sd), p in all_probs.items():
    lab = METHOD_LABELS.get(cm)
    if lab in METHODS and sd in SEEDS and len(p) == len(labels):
        probs[(lab, sd)] = np.asarray(p, np.float32)

for m in METHODS:
    got = sorted(s for (mm, s) in probs if mm == m)
    print(f'  {m:<8} seeds {got}')
    if got != SEEDS:
        raise AssertionError(f'{m} does not have all five seeds; got {got}')


# ============================================================================
# METRIC HELPERS
# ============================================================================

# ============================================================================
# FAST WEIGHTED AUC
#   roc_auc_score with sample_weight is far too slow inside a 2000-replicate
#   bootstrap. This computes the same quantity from a one-time sort plus a
#   bincount per replicate, with exact mid-rank handling of tied scores.
# ============================================================================

M_MASK = (sex == 1)
F_MASK = (sex == 0)
SUBSETS = {'all': np.ones(len(labels), bool), 'male': M_MASK, 'female': F_MASK}


def build_struct(p, submask):
    rows = np.flatnonzero(submask)
    if len(rows) == 0 or np.unique(labels[rows]).size < 2:
        return None
    order = rows[np.argsort(p[rows], kind='mergesort')]
    _, gidx = np.unique(p[order], return_inverse=True)
    y = labels[order]
    return {'rows': order, 'gidx': gidx.astype(np.int32),
            'G': int(gidx.max()) + 1,
            'pos': (y == 1).astype(np.float64),
            'neg': (y == 0).astype(np.float64)}


def wauc(st, w=None):
    if st is None:
        return np.nan
    ws = np.ones(len(st['rows'])) if w is None else w[st['rows']]
    wp, wn = ws * st['pos'], ws * st['neg']
    Wp, Wn = wp.sum(), wn.sum()
    if Wp <= 0 or Wn <= 0:
        return np.nan
    npg = np.bincount(st['gidx'], weights=wn, minlength=st['G'])
    ppg = np.bincount(st['gidx'], weights=wp, minlength=st['G'])
    below = np.concatenate(([0.0], np.cumsum(npg)[:-1]))
    return float((ppg * (below + 0.5 * npg)).sum() / (Wp * Wn))


def struct_set(p):
    return {k: build_struct(p, m) for k, m in SUBSETS.items()}


def metrics_from(sts, w=None):
    a = wauc(sts['all'], w)
    am, af = wauc(sts['male'], w), wauc(sts['female'], w)
    wg = np.nan if not (np.isfinite(am) and np.isfinite(af)) else min(am, af)
    return {'auroc': a, 'male_auroc': am, 'female_auroc': af,
            'worst_group_auroc': wg}


def metrics(p, w=None):
    return metrics_from(struct_set(p), w)


def _rate(mask, pred, w):
    d = w[mask].sum()
    return np.nan if d <= 0 else float(w[mask & (pred == 1)].sum() / d)


def gaps(p, thr, w=None):
    w = np.ones(len(p)) if w is None else w
    pred = (p >= thr).astype(np.int8)
    pos, neg = (labels == 1), (labels == 0)
    eo = abs(_rate(M_MASK & pos, pred, w) - _rate(F_MASK & pos, pred, w))
    fp = abs(_rate(M_MASK & neg, pred, w) - _rate(F_MASK & neg, pred, w))
    return eo, fp


# ============================================================================
# PART 1 — PER-SEED METRICS
# ============================================================================

print('\n' + '=' * 92)
print('PART 1 — PER-SEED NIH METRICS')
print('=' * 92)

rows = []
for m in METHODS:
    for sd in SEEDS:
        r = metrics(probs[(m, sd)])
        eo, fp = gaps(probs[(m, sd)], 0.5)
        r.update({'method': m, 'seed': sd, 'eo_gap_50': eo, 'fpr_gap_50': fp})
        rows.append(r)
per_seed = pd.DataFrame(rows)
per_seed.to_csv(OUT / 'per_seed_metrics.csv', index=False)

summary = (per_seed.groupby('method')[['auroc', 'worst_group_auroc',
                                       'eo_gap_50', 'fpr_gap_50']]
           .agg(['mean', 'std']).round(4).loc[METHODS])
summary.to_csv(OUT / 'method_summary_mean_sd.csv')
print(summary.to_string())


# ============================================================================
# PART 2 — PATIENT-CLUSTERED BOOTSTRAP
# ============================================================================

print('\n' + '=' * 92)
print('PART 2 — PATIENT-CLUSTERED INFERENCE')
print('=' * 92)

ens = {m: np.mean([probs[(m, s)] for s in SEEDS], axis=0) for m in METHODS}
ALL_PAIRS = PRIMARY_PAIRS + SECONDARY_PAIRS

ST_RUN = {(m, s): struct_set(probs[(m, s)]) for m in METHODS for s in SEEDS}
ST_ENS = {m: struct_set(ens[m]) for m in METHODS}

rng = np.random.default_rng(BOOT_SEED)
boot_w = np.empty((N_BOOT, len(labels)), np.float32)
for b in range(N_BOOT):
    samp = rng.integers(0, n_patients, size=n_patients)
    boot_w[b] = np.bincount(samp, minlength=n_patients).astype(np.float32)[patient_code]

# A[method][metric] -> (seeds x N_BOOT); computed ONCE and reused by the
# run-aware, simultaneous, and leave-one-seed-out analyses.
print(f'Bootstrapping {N_BOOT} patient-cluster replicates ...')
A = {m: {k: np.empty((len(SEEDS), N_BOOT)) for k in PRIMARY_METRICS}
     for m in METHODS}
E = {m: {k: np.empty(N_BOOT) for k in PRIMARY_METRICS} for m in METHODS}
for b in range(N_BOOT):
    w = boot_w[b]
    for m in METHODS:
        for si, s in enumerate(SEEDS):
            r = metrics_from(ST_RUN[(m, s)], w)
            for k in PRIMARY_METRICS:
                A[m][k][si, b] = r[k]
        r = metrics_from(ST_ENS[m], w)
        for k in PRIMARY_METRICS:
            E[m][k][b] = r[k]

rng_run = np.random.default_rng(BOOT_SEED + 1)
run_draw = {m: rng_run.integers(0, len(SEEDS), size=(N_BOOT, len(SEEDS)))
            for m in METHODS}
run_boot = {(m, k): np.array([np.nanmean(A[m][k][run_draw[m][b], b])
                              for b in range(N_BOOT)])
            for m in METHODS for k in PRIMARY_METRICS}
fix_boot = {(m, k): E[m][k] for m in METHODS for k in PRIMARY_METRICS}

POINT_FIX = {m: metrics_from(ST_ENS[m]) for m in METHODS}


def _pt(a, b, met, kind):
    if kind == 'fixed':
        return POINT_FIX[a][met] - POINT_FIX[b][met]
    return (per_seed.loc[per_seed.method == a, met].mean()
            - per_seed.loc[per_seed.method == b, met].mean())


results, std_mat, keys = [], [], []
for a, b in ALL_PAIRS:
    for met in PRIMARY_METRICS:
        fx = fix_boot[(a, met)] - fix_boot[(b, met)]
        rn = run_boot[(a, met)] - run_boot[(b, met)]
        row = {'comparison': f'{a} - {b}', 'metric': met,
               'family': 'primary' if (a, b) in PRIMARY_PAIRS else 'secondary',
               'fixed_estimate': _pt(a, b, met, 'fixed'),
               'fixed_lo': np.nanpercentile(fx, 2.5),
               'fixed_hi': np.nanpercentile(fx, 97.5),
               'run_estimate': _pt(a, b, met, 'run'),
               'run_lo': np.nanpercentile(rn, 2.5),
               'run_hi': np.nanpercentile(rn, 97.5)}
        results.append(row)
        if row['family'] == 'primary':
            std_mat.append(rn)
            keys.append((f'{a} - {b}', met))

std_mat = np.vstack(std_mat)
centred = std_mat - np.nanmean(std_mat, axis=1, keepdims=True)
sds = np.nanstd(std_mat, axis=1, keepdims=True)
sds[sds == 0] = np.nan
crit = float(np.nanpercentile(np.nanmax(np.abs(centred / sds), axis=0), 95))
print(f'Single-step max-t critical value over six primary contrasts: {crit:.3f}')

res = pd.DataFrame(results)
res['sim_lo'] = np.nan
res['sim_hi'] = np.nan
for i, (cmp_, met) in enumerate(keys):
    j = res.index[(res.comparison == cmp_) & (res.metric == met)][0]
    sd_i = float(np.nanstd(std_mat[i]))
    res.at[j, 'sim_lo'] = res.at[j, 'run_estimate'] - crit * sd_i
    res.at[j, 'sim_hi'] = res.at[j, 'run_estimate'] + crit * sd_i
for c in ['fixed', 'run', 'sim']:
    lo, hi = res[f'{c}_lo'], res[f'{c}_hi']
    ok = lo.notna() & hi.notna()
    res[f'{c}_excludes_zero'] = np.where(ok, ~((lo <= 0) & (0 <= hi)), np.nan)
    res[f'{c}_excludes_zero'] = res[f'{c}_excludes_zero'].astype('object')
    res.loc[~ok, f'{c}_excludes_zero'] = 'n/a (simultaneous control covers the primary family only)'
res.to_csv(OUT / 'primary_contrasts.csv', index=False)
print(res.round(6).to_string(index=False))


# ============================================================================
# PART 3 — THRESHOLD SWEEP
# ============================================================================

print('\n' + '=' * 92)
print('PART 3 — THRESHOLD SWEEP, 37 OPERATING POINTS')
print('=' * 92)

trow = []
for m in METHODS:
    for sd in SEEDS:
        for t in THRESHOLDS:
            eo, fp = gaps(probs[(m, sd)], t)
            trow.append({'method': m, 'seed': sd, 'threshold': t,
                         'eo_gap': eo, 'fpr_gap': fp})
thr_df = pd.DataFrame(trow)
thr_df.to_csv(OUT / 'threshold_sweep.csv', index=False)

thr_mean = thr_df.groupby(['method', 'threshold'])[['eo_gap', 'fpr_gap']].mean()
win_eo = (thr_mean['eo_gap'].unstack(0)[METHODS].idxmin(axis=1).value_counts())
win_fp = (thr_mean['fpr_gap'].unstack(0)[METHODS].idxmin(axis=1).value_counts())
print('Lowest mean EO gap  count by method:', win_eo.to_dict())
print('Lowest mean FPR gap count by method:', win_fp.to_dict())


# ============================================================================
# PART 4 — LEAVE ONE SEED OUT
# ============================================================================

print('\n' + '=' * 92)
print('PART 4 — LEAVE-ONE-SEED-OUT SIMULTANEOUS INTERVALS')
print('=' * 92)

loo = []
for drop in SEEDS:
    keep = np.array([i for i, s in enumerate(SEEDS) if s != drop])
    kept_seeds = [s for s in SEEDS if s != drop]

    # Resample the KEPT runs with replacement, exactly as Part 2 does for the
    # full five. Averaging them deterministically would drop training-run
    # variance and give intervals that are far too narrow.
    rng_loo = np.random.default_rng(BOOT_SEED + 100 + drop)
    draw = {m: keep[rng_loo.integers(0, len(keep), size=(N_BOOT, len(keep)))]
            for m in METHODS}
    bo = {(m, k): np.array([np.nanmean(A[m][k][draw[m][b], b])
                            for b in range(N_BOOT)])
          for m in METHODS for k in PRIMARY_METRICS}

    kb, kk, pts = [], [], []
    for a, b in PRIMARY_PAIRS:
        for met in PRIMARY_METRICS:
            kb.append(bo[(a, met)] - bo[(b, met)])
            kk.append((f'{a} - {b}', met))
            sel = per_seed[per_seed.seed.isin(kept_seeds)]
            pts.append(sel.loc[sel.method == a, met].mean()
                       - sel.loc[sel.method == b, met].mean())

    kb = np.vstack(kb)
    c = kb - np.nanmean(kb, axis=1, keepdims=True)
    s_ = np.nanstd(kb, axis=1, keepdims=True)
    s_[s_ == 0] = np.nan
    cv = float(np.nanpercentile(np.nanmax(np.abs(c / s_), axis=0), 95))
    for i, (cmp_, met) in enumerate(kk):
        est = float(pts[i])
        sd_i = float(np.nanstd(kb[i]))
        loo.append({'omitted_seed': drop, 'comparison': cmp_, 'metric': met,
                    'estimate': est, 'sim_lo': est - cv * sd_i,
                    'sim_hi': est + cv * sd_i})
loo_df = pd.DataFrame(loo)
loo_df['excludes_zero'] = ~((loo_df.sim_lo <= 0) & (0 <= loo_df.sim_hi))
loo_df.to_csv(OUT / 'leave_one_seed_out.csv', index=False)
print(f'Intervals excluding zero: {int(loo_df.excludes_zero.sum())} of {len(loo_df)}')


# ============================================================================
# PART 5 — FIGURES
# ============================================================================

print('\n' + '=' * 92)
print('PART 5 — FIGURES')
print('=' * 92)

plt.rcParams.update({'font.size': 9, 'axes.linewidth': 0.8,
                     'font.family': 'DejaVu Sans'})


def save(fig, stem):
    fig.savefig(OUT / f'{stem}.png', dpi=300, bbox_inches='tight')
    fig.savefig(OUT / f'{stem}.pdf', bbox_inches='tight')
    plt.close(fig)
    print(f'  saved {stem}.png / .pdf')


# ---- Figure 4 -------------------------------------------------------------
fig = plt.figure(figsize=(11, 3.6))
gs = fig.add_gridspec(1, 3, wspace=0.32)
for k, (met, ttl) in enumerate([('auroc', 'Overall AUROC'),
                                ('worst_group_auroc', 'Worst-group AUROC')]):
    ax = fig.add_subplot(gs[0, k])
    for xi, m in enumerate(METHODS):
        v = per_seed.loc[per_seed.method == m, met].to_numpy()
        ax.scatter(np.full(len(v), xi) + np.linspace(-.12, .12, len(v)), v,
                   s=22, color=COLORS[m], zorder=3)
        ax.errorbar(xi, v.mean(), yerr=v.std(ddof=1), fmt='D', color='black',
                    ms=5, capsize=3, lw=1.1, zorder=4)
    ax.set_xticks(range(len(METHODS)))
    ax.set_xticklabels(METHODS, rotation=30, ha='right')
    ax.set_ylabel(ttl)
    ax.set_title(f'{"AB"[k]}   {ttl}', loc='left', fontweight='bold')
    ax.grid(axis='y', ls=':', lw=.5, alpha=.6)

ax = fig.add_subplot(gs[0, 2])
for met, lab, cl, mk in [('auroc', 'Overall AUROC', '#2ca02c', 'o'),
                         ('worst_group_auroc', 'Worst-group AUROC', '#1f77b4', 's')]:
    d = [per_seed.loc[(per_seed.method == 'DWFA') & (per_seed.seed == s), met].item()
         - per_seed.loc[(per_seed.method == 'FedAvg') & (per_seed.seed == s), met].item()
         for s in SEEDS]
    ax.plot(range(len(SEEDS)), np.array(d) * 1e3, marker=mk, color=cl, label=lab, lw=1.3)
ax.axhline(0, color='black', ls='--', lw=.9)
ax.set_xticks(range(len(SEEDS)))
ax.set_xticklabels(SEEDS)
ax.set_xlabel('Numeric seed')
ax.set_ylabel(r'DWFA $-$ FedAvg ($\times 10^{-3}$)')
ax.set_title('C   Same-seed differences', loc='left', fontweight='bold')
ax.legend(frameon=False, fontsize=7.5)
ax.grid(axis='y', ls=':', lw=.5, alpha=.6)
save(fig, 'figure4_external_nih_across_runs')

# ---- Figure 5 -------------------------------------------------------------
prim = res[res.family == 'primary'].reset_index(drop=True)
lab = [f'{r.comparison} | '
       f'{"Overall" if r.metric == "auroc" else "Worst-group"}'
       for r in prim.itertuples()]
y = np.arange(len(prim))[::-1]
fig, ax = plt.subplots(figsize=(7.6, 4.4))
for off, (lo, hi, est, cl, mk, nm) in enumerate([
        ('fixed_lo', 'fixed_hi', 'fixed_estimate', '#1f77b4', 'o',
         'Fixed seed ensemble: pointwise 95% CI'),
        ('run_lo', 'run_hi', 'run_estimate', '#ff7f0e', 's',
         'Run-aware: pointwise 95% CI'),
        ('sim_lo', 'sim_hi', 'run_estimate', '#d62728', 'D',
         'Run-aware: simultaneous 95% CI')]):
    dy = (1 - off) * 0.22
    ax.errorbar(prim[est] * 1e3, y + dy,
                xerr=[(prim[est] - prim[lo]) * 1e3, (prim[hi] - prim[est]) * 1e3],
                fmt=mk, color=cl, ms=5, lw=1.2, capsize=2.5, label=nm)
ax.axvline(0, color='black', ls='--', lw=1)
ax.set_yticks(y)
ax.set_yticklabels(lab)
ax.set_xlabel(r'AUROC difference ($\times 10^{-3}$; first method minus second)')
ax.legend(frameon=False, fontsize=7.5, loc='upper center',
          bbox_to_anchor=(0.5, -0.16), ncol=1)
ax.grid(axis='x', ls=':', lw=.5, alpha=.6)
save(fig, 'figure5_primary_discrimination_contrasts')

# ---- Figure 6 -------------------------------------------------------------
fig, axes = plt.subplots(1, 2, figsize=(10, 3.8))
for ax, col, ttl in [(axes[0], 'eo_gap', 'Equal-opportunity gap'),
                     (axes[1], 'fpr_gap', 'False-positive-rate gap')]:
    for m in METHODS:
        d = thr_df[thr_df.method == m]
        g = d.groupby('threshold')[col]
        ax.plot(THRESHOLDS, g.mean().to_numpy(), color=COLORS[m], lw=1.4, label=m)
        ax.fill_between(THRESHOLDS, g.min().to_numpy(), g.max().to_numpy(),
                        color=COLORS[m], alpha=.13, lw=0)
    ax.axvline(0.5, color='grey', ls='--', lw=.9)
    ax.set_xlabel('Decision threshold')
    ax.set_ylabel('Gap')
    ax.set_title(ttl)
    ax.legend(frameon=False, fontsize=7.5)
    ax.grid(ls=':', lw=.5, alpha=.6)
save(fig, 'figure6_threshold_sensitivity')

# ---- Figure 8 -------------------------------------------------------------
combos = [(f'{a} - {b}', m) for a, b in PRIMARY_PAIRS for m in PRIMARY_METRICS]
y = np.arange(len(combos))[::-1]
cmap = plt.get_cmap('tab10')
fig, ax = plt.subplots(figsize=(8, 4.8))
for si, drop in enumerate(SEEDS):
    d = loo_df[loo_df.omitted_seed == drop]
    xs, los, his = [], [], []
    for cmp_, met in combos:
        r = d[(d.comparison == cmp_) & (d.metric == met)].iloc[0]
        xs.append(r.estimate * 1e3)
        los.append((r.estimate - r.sim_lo) * 1e3)
        his.append((r.sim_hi - r.estimate) * 1e3)
    ax.errorbar(xs, y + (si - 2) * 0.14, xerr=[los, his], fmt='o', ms=3.6,
                lw=1.0, capsize=2, color=cmap(si), label=f'Omit {drop}')
ax.axvline(0, color='black', ls='--', lw=1)
ax.set_yticks(y)
ax.set_yticklabels([f'{c} | {"Overall" if m == "auroc" else "Worst-group"}'
                    for c, m in combos])
ax.set_xlabel(r'AUROC difference ($\times 10^{-3}$; simultaneous 95% CI)')
ax.legend(frameon=False, fontsize=7.5, ncol=5, loc='upper center',
          bbox_to_anchor=(0.5, 1.13))
ax.grid(axis='x', ls=':', lw=.5, alpha=.6)
save(fig, 'figure8_leave_one_seed_out')


# ============================================================================
# PACKAGE
# ============================================================================

(OUT / 'run_metadata.json').write_text(json.dumps({
    'methods': METHODS, 'seeds': SEEDS,
    'primary_pairs': [f'{a} - {b}' for a, b in PRIMARY_PAIRS],
    'secondary_pairs': [f'{a} - {b}' for a, b in SECONDARY_PAIRS],
    'primary_metrics': PRIMARY_METRICS,
    'bootstrap_replicates': N_BOOT, 'bootstrap_seed': BOOT_SEED,
    'maxt_critical_value': crit,
    'n_images': int(len(labels)), 'n_patients': int(n_patients),
}, indent=2))

with zipfile.ZipFile(ZIP_OUT, 'w', zipfile.ZIP_DEFLATED) as z:
    for f in sorted(OUT.glob('*')):
        z.write(f, f.name)

n_fix = int(res[res.family == 'primary'].fixed_excludes_zero.sum())
n_run = int(res[res.family == 'primary'].run_excludes_zero.sum())
n_sim = int(res[res.family == 'primary'].sim_excludes_zero.sum())
print('\n' + '=' * 92)
print(f'Primary family, six contrasts:')
print(f'  fixed-ensemble intervals excluding zero      : {n_fix} of 6')
print(f'  run-aware pointwise intervals excluding zero : {n_run} of 6')
print(f'  run-aware simultaneous excluding zero        : {n_sim} of 6')
print(f'\nSaved -> {OUT}')
print(f'Zip   -> {ZIP_OUT}')
print('=' * 92)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
PART 0 — PREDICTION CACHE
Cached keys: 20
FedAvg seeds present: [42, 123, 456, 789, 1010]

All five FedAvg seeds already cached. Running CPU-only.

Cohort: 112,120 images | 30,805 patients | prevalence 0.1187
  FedAvg   seeds [42, 123, 456, 789, 1010]
  LPR      seeds [42, 123, 456, 789, 1010]
  CADR     seeds [42, 123, 456, 789, 1010]
  DWFA     seeds [42, 123, 456, 789, 1010]

PART 1 — PER-SEED NIH METRICS
         auroc         worst_group_auroc         eo_gap_50         fpr_gap_50        
          mean     std              mean     std      mean     std       mean     std
method                                                                               
FedAvg  0.8572  0.0023            0.8557  0.0017    0.0270  0.0050     0.0191  0.0022
LPR     0.8554  0.0016            0.8537  0.0018    0.0285  0.0062     0.0185  0.0042
CADR    0.8565  0.0024       

In [ ]:
# ============================================================================
# FINAL ANALYSIS AND FIGURE CELL — FEDAVG AT FIVE SEEDS IN THE PRIMARY FAMILY
#
# WHAT THIS DOES
#   PART 0  Adds ('fedavg', 789) and ('fedavg', 1010) to nih_pred_cache.npz.
#           Needs GPU ONLY if those keys are missing. If they are already
#           cached, the whole cell runs on CPU.
#   PART 1  Per-seed NIH metrics for FedAvg, LPR, CADR, DWFA at five seeds.
#   PART 2  Patient-clustered bootstrap: fixed-ensemble, run-aware pointwise,
#           and single-step simultaneous max-t intervals.
#   PART 3  Threshold sweep, 37 operating points, EO gap and FPR gap.
#   PART 4  Leave-one-seed-out simultaneous intervals.
#   PART 5  Regenerates Figures 4, 5, 6, and 8 in the manuscript's style and
#           saves PNG (300 dpi) and PDF to Drive.
#
# PRIMARY FAMILY (declared before looking at any result)
#   DWFA - FedAvg, LPR - FedAvg, CADR - FedAvg
#   on overall AUROC and worst-group AUROC = SIX contrasts.
#   The three fairness-vs-fairness pairs are kept as SECONDARY.
#
# BEFORE RUNNING
#   If ('fedavg', 789) is already in nih_pred_cache.npz: no old cell needed.
#   If it is NOT: run these first, in this order, on a GPU runtime:
#     "# external validation"   (NIH CELL 1, stages images, defines NIH_IMG)
#     "# NIH CELL 2 — Build the effusion cohort CSV in YOUR schema"
#   Do NOT run "# PAIRED SIGNIFICANCE TEST on NIH" or the CheXpert twin.
#   Both take a load-branch when the cache exists and will not add seeds.
#
# OUTPUT
#   results/final_figures/  (figures + every underlying CSV)
#   results/final_figures.zip
# ============================================================================

from google.colab import drive
drive.mount('/content/drive', force_remount=False)

from pathlib import Path
import os, gc, json, shutil, zipfile
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from sklearn.metrics import roc_auc_score

BASE = Path('/content/drive/MyDrive/FairFedCXR')
RESULTS = BASE / 'results'
CKPTS = BASE / 'checkpoints'
NIH_CSV = RESULTS / 'nih_effusion_cohort.csv'
CACHE = RESULTS / 'nih_pred_cache.npz'
EXT_CACHE = RESULTS / 'nih_extended_baselines_cache.npz'

OUT = RESULTS / 'final_figures'
ZIP_OUT = RESULTS / 'final_figures.zip'

METHOD_LABELS = {'fedavg': 'FedAvg', 'qfedavg': 'LPR',
                 'fairfed': 'CADR', 'dwfa': 'DWFA'}
METHODS = ['FedAvg', 'LPR', 'CADR', 'DWFA']
SEEDS = [42, 123, 456, 789, 1010]

# Descriptive-only references. Fewer than five runs, so they are reported in
# the summary table but never enter the inferential family or the figures.
DESCRIPTIVE_LABELS = {'centralized': 'Centralized reference', 'fedprox': 'FedProx'}
TABLE_ORDER = ['Centralized reference', 'FedAvg', 'FedProx', 'LPR', 'CADR', 'DWFA']

PRIMARY_PAIRS = [('DWFA', 'FedAvg'), ('LPR', 'FedAvg'), ('CADR', 'FedAvg')]
SECONDARY_PAIRS = [('DWFA', 'LPR'), ('DWFA', 'CADR'), ('LPR', 'CADR')]
PRIMARY_METRICS = ['auroc', 'worst_group_auroc']

N_BOOT = 2000
BOOT_SEED = 20260723
THRESHOLDS = np.round(np.arange(0.05, 0.951, 0.025), 4)

COLORS = {'FedAvg': '#7f7f7f', 'LPR': '#1f77b4',
          'CADR': '#ff7f0e', 'DWFA': '#2ca02c'}

if OUT.exists():
    shutil.rmtree(OUT)
OUT.mkdir(parents=True, exist_ok=True)

if not CACHE.exists():
    raise FileNotFoundError(f'{CACHE} not found.')
if not NIH_CSV.exists():
    raise FileNotFoundError(f'{NIH_CSV} not found. Run NIH CELL 2 first.')


# ============================================================================
# PART 0 — ENSURE FIVE FEDAVG SEEDS ARE CACHED
# ============================================================================

print('=' * 92)
print('PART 0 — PREDICTION CACHE')
print('=' * 92)

_c = np.load(CACHE, allow_pickle=True)
all_probs = _c['all_probs'].item()
labels = _c['labels'].astype(np.int8)
sex = _c['sex'].astype(np.int8)

need = [(m, s) for m in ['fedavg'] for s in SEEDS if (m, s) not in all_probs]
print(f'Cached keys: {len(all_probs)}')
print(f'FedAvg seeds present: {sorted(s for m, s in all_probs if m == "fedavg")}')

if need:
    print(f'\nMissing and will be inferred on GPU: {need}')
    import torch
    import torchvision.models as tvm
    from torchvision import transforms
    from torch.utils.data import Dataset, DataLoader
    from PIL import Image

    if not torch.cuda.is_available():
        raise RuntimeError('GPU required to add the missing FedAvg seeds. '
                           'Switch to a GPU runtime and rerun.')
    if 'NIH_IMG' not in globals() or not os.path.isdir(str(NIH_IMG)):
        raise RuntimeError('NIH_IMG not in session. Run "# external validation" '
                           '(NIH CELL 1) first.')

    dev = torch.device('cuda')
    _cohort = pd.read_csv(NIH_CSV, low_memory=False).reset_index(drop=True)
    _tf = transforms.Compose([
        transforms.Resize((224, 224)), transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])])

    class _DS(Dataset):
        def __init__(s, df, root, tf):
            s.df, s.root, s.tf = df.reset_index(drop=True), root, tf
        def __len__(s):
            return len(s.df)
        def __getitem__(s, i):
            r = s.df.iloc[i]
            img = Image.open(os.path.join(s.root, str(r['Path']))).convert('RGB')
            return s.tf(img), np.int8(r['label']), np.int8(r['sex_encoded'])

    _loader = DataLoader(_DS(_cohort, str(NIH_IMG), _tf), batch_size=128,
                         shuffle=False, num_workers=4, pin_memory=True)

    def _state(o):
        if isinstance(o, dict):
            for k in ['state_dict', 'model_state_dict', 'global_state', 'model']:
                if k in o and isinstance(o[k], dict):
                    return o[k]
        return o

    for m, sd in need:
        ck = CKPTS / f'{m}_seed{sd}.pt'
        if not ck.exists():
            raise FileNotFoundError(f'{ck} missing. Finish training seed {sd} first.')
        net = tvm.densenet121(weights=None)
        net.classifier = torch.nn.Linear(net.classifier.in_features, 1)
        net.load_state_dict(_state(torch.load(ck, map_location='cpu',
                                              weights_only=False)), strict=False)
        net = net.to(dev).eval()
        P = []
        with torch.inference_mode():
            for imgs, _, _ in _loader:
                with torch.amp.autocast('cuda', dtype=torch.float16):
                    lg = net(imgs.to(dev, non_blocking=True)).squeeze(1)
                P.append(torch.sigmoid(lg.float()).cpu().numpy().astype(np.float32))
        all_probs[(m, sd)] = np.concatenate(P)
        print(f'  cached {m} s{sd}  (n={len(all_probs[(m, sd)])})')
        del net
        gc.collect()
        torch.cuda.empty_cache()

    tmp = Path('/content/_nih_cache.npz')
    np.savez(tmp, all_probs=all_probs, labels=labels, sex=sex)
    shutil.copy2(tmp, CACHE)
    print(f'Cache updated -> {CACHE}')
else:
    print('\nAll five FedAvg seeds already cached. Running CPU-only.')

nih = pd.read_csv(NIH_CSV, low_memory=False).reset_index(drop=True)
if len(nih) != len(labels):
    raise AssertionError(f'cohort rows {len(nih)} != cache rows {len(labels)}')
if not np.array_equal(nih['label'].to_numpy(np.int8), labels):
    raise AssertionError('cohort label order does not match the cache')

patient_code, patient_levels = pd.factorize(
    nih['Patient ID'].astype(str).str.strip(), sort=True)
patient_code = patient_code.astype(np.int32)
n_patients = len(patient_levels)
print(f'\nCohort: {len(nih):,} images | {n_patients:,} patients | '
      f'prevalence {labels.mean():.4f}')

probs = {}
for (cm, sd), p in all_probs.items():
    lab = METHOD_LABELS.get(cm)
    if lab in METHODS and sd in SEEDS and len(p) == len(labels):
        probs[(lab, sd)] = np.asarray(p, np.float32)

for m in METHODS:
    got = sorted(s for (mm, s) in probs if mm == m)
    print(f'  {m:<8} seeds {got}')
    if got != SEEDS:
        raise AssertionError(f'{m} does not have all five seeds; got {got}')

# ---- descriptive references, from either cache -----------------------------
desc_probs = {}
_sources = [all_probs]
if EXT_CACHE.exists():
    _e = np.load(EXT_CACHE, allow_pickle=True)
    if len(_e['labels']) == len(labels) and np.array_equal(
            _e['labels'].astype(np.int8), labels):
        _sources.append(_e['all_probs'].item())
        print(f'Merged descriptive references from {EXT_CACHE.name}')
    else:
        print(f'WARNING: {EXT_CACHE.name} row order does not match; skipped.')

for srcmap in _sources:
    for (cm, sd), p in srcmap.items():
        lab = DESCRIPTIVE_LABELS.get(cm)
        if lab is not None and len(p) == len(labels):
            desc_probs[(lab, sd)] = np.asarray(p, np.float32)

DESCRIPTIVE_METHODS = sorted({m for m, _ in desc_probs},
                             key=lambda x: TABLE_ORDER.index(x))
for m in DESCRIPTIVE_METHODS:
    print(f'  {m:<22} seeds {sorted(s for (mm, s) in desc_probs if mm == m)} '
          f'(descriptive only)')
if not DESCRIPTIVE_METHODS:
    print('  no descriptive references found in either cache')


# ============================================================================
# METRIC HELPERS
# ============================================================================

# ============================================================================
# FAST WEIGHTED AUC
#   roc_auc_score with sample_weight is far too slow inside a 2000-replicate
#   bootstrap. This computes the same quantity from a one-time sort plus a
#   bincount per replicate, with exact mid-rank handling of tied scores.
# ============================================================================

M_MASK = (sex == 1)
F_MASK = (sex == 0)
SUBSETS = {'all': np.ones(len(labels), bool), 'male': M_MASK, 'female': F_MASK}


def build_struct(p, submask):
    rows = np.flatnonzero(submask)
    if len(rows) == 0 or np.unique(labels[rows]).size < 2:
        return None
    order = rows[np.argsort(p[rows], kind='mergesort')]
    _, gidx = np.unique(p[order], return_inverse=True)
    y = labels[order]
    return {'rows': order, 'gidx': gidx.astype(np.int32),
            'G': int(gidx.max()) + 1,
            'pos': (y == 1).astype(np.float64),
            'neg': (y == 0).astype(np.float64)}


def wauc(st, w=None):
    if st is None:
        return np.nan
    ws = np.ones(len(st['rows'])) if w is None else w[st['rows']]
    wp, wn = ws * st['pos'], ws * st['neg']
    Wp, Wn = wp.sum(), wn.sum()
    if Wp <= 0 or Wn <= 0:
        return np.nan
    npg = np.bincount(st['gidx'], weights=wn, minlength=st['G'])
    ppg = np.bincount(st['gidx'], weights=wp, minlength=st['G'])
    below = np.concatenate(([0.0], np.cumsum(npg)[:-1]))
    return float((ppg * (below + 0.5 * npg)).sum() / (Wp * Wn))


def struct_set(p):
    return {k: build_struct(p, m) for k, m in SUBSETS.items()}


def metrics_from(sts, w=None):
    a = wauc(sts['all'], w)
    am, af = wauc(sts['male'], w), wauc(sts['female'], w)
    wg = np.nan if not (np.isfinite(am) and np.isfinite(af)) else min(am, af)
    return {'auroc': a, 'male_auroc': am, 'female_auroc': af,
            'worst_group_auroc': wg}


def metrics(p, w=None):
    return metrics_from(struct_set(p), w)


def _rate(mask, pred, w):
    d = w[mask].sum()
    return np.nan if d <= 0 else float(w[mask & (pred == 1)].sum() / d)


def gaps(p, thr, w=None):
    w = np.ones(len(p)) if w is None else w
    pred = (p >= thr).astype(np.int8)
    pos, neg = (labels == 1), (labels == 0)
    eo = abs(_rate(M_MASK & pos, pred, w) - _rate(F_MASK & pos, pred, w))
    fp = abs(_rate(M_MASK & neg, pred, w) - _rate(F_MASK & neg, pred, w))
    return eo, fp


# ============================================================================
# PART 1 — PER-SEED METRICS
# ============================================================================

print('\n' + '=' * 92)
print('PART 1 — PER-SEED NIH METRICS')
print('=' * 92)

rows = []
for m in METHODS:
    for sd in SEEDS:
        r = metrics(probs[(m, sd)])
        eo, fp = gaps(probs[(m, sd)], 0.5)
        r.update({'method': m, 'seed': sd, 'eo_gap_50': eo, 'fpr_gap_50': fp})
        rows.append(r)
per_seed = pd.DataFrame(rows)
per_seed.to_csv(OUT / 'per_seed_metrics.csv', index=False)

desc_rows = []
for (m, sd), p in desc_probs.items():
    r = metrics(p)
    eo, fp = gaps(p, 0.5)
    r.update({'method': m, 'seed': sd, 'eo_gap_50': eo, 'fpr_gap_50': fp})
    desc_rows.append(r)
per_seed_desc = pd.DataFrame(desc_rows)
if not per_seed_desc.empty:
    per_seed_desc.to_csv(OUT / 'per_seed_metrics_descriptive.csv', index=False)

COLS = ['auroc', 'worst_group_auroc', 'eo_gap_50', 'fpr_gap_50']
combined = pd.concat([per_seed.assign(family='inferential'),
                      per_seed_desc.assign(family='descriptive')],
                     ignore_index=True) if not per_seed_desc.empty else \
    per_seed.assign(family='inferential')

tab = []
for m in [x for x in TABLE_ORDER if x in set(combined.method)]:
    d = combined[combined.method == m]
    row = {'method': m, 'runs': int(d.seed.nunique()),
           'family': d.family.iloc[0]}
    for c in COLS:
        row[c] = f'{d[c].mean():.4f} +/- {d[c].std(ddof=1):.4f}'
    tab.append(row)
table5 = pd.DataFrame(tab)
table5.to_csv(OUT / 'table5_nih_metrics_mean_sd.csv', index=False)

combined.to_csv(OUT / 'per_seed_metrics_all_methods.csv', index=False)
print(table5.to_string(index=False))
print('\nProcedures with fewer than five runs are descriptive references only. '
      'They are not in the inferential family and not in Figures 4, 5, 6, or 8.')


# ============================================================================
# PART 2 — PATIENT-CLUSTERED BOOTSTRAP
# ============================================================================

print('\n' + '=' * 92)
print('PART 2 — PATIENT-CLUSTERED INFERENCE')
print('=' * 92)

ens = {m: np.mean([probs[(m, s)] for s in SEEDS], axis=0) for m in METHODS}
ALL_PAIRS = PRIMARY_PAIRS + SECONDARY_PAIRS

ST_RUN = {(m, s): struct_set(probs[(m, s)]) for m in METHODS for s in SEEDS}
ST_ENS = {m: struct_set(ens[m]) for m in METHODS}

rng = np.random.default_rng(BOOT_SEED)
boot_w = np.empty((N_BOOT, len(labels)), np.float32)
for b in range(N_BOOT):
    samp = rng.integers(0, n_patients, size=n_patients)
    boot_w[b] = np.bincount(samp, minlength=n_patients).astype(np.float32)[patient_code]

# A[method][metric] -> (seeds x N_BOOT); computed ONCE and reused by the
# run-aware, simultaneous, and leave-one-seed-out analyses.
print(f'Bootstrapping {N_BOOT} patient-cluster replicates ...')
A = {m: {k: np.empty((len(SEEDS), N_BOOT)) for k in PRIMARY_METRICS}
     for m in METHODS}
E = {m: {k: np.empty(N_BOOT) for k in PRIMARY_METRICS} for m in METHODS}
for b in range(N_BOOT):
    w = boot_w[b]
    for m in METHODS:
        for si, s in enumerate(SEEDS):
            r = metrics_from(ST_RUN[(m, s)], w)
            for k in PRIMARY_METRICS:
                A[m][k][si, b] = r[k]
        r = metrics_from(ST_ENS[m], w)
        for k in PRIMARY_METRICS:
            E[m][k][b] = r[k]

rng_run = np.random.default_rng(BOOT_SEED + 1)
run_draw = {m: rng_run.integers(0, len(SEEDS), size=(N_BOOT, len(SEEDS)))
            for m in METHODS}
run_boot = {(m, k): np.array([np.nanmean(A[m][k][run_draw[m][b], b])
                              for b in range(N_BOOT)])
            for m in METHODS for k in PRIMARY_METRICS}
fix_boot = {(m, k): E[m][k] for m in METHODS for k in PRIMARY_METRICS}

POINT_FIX = {m: metrics_from(ST_ENS[m]) for m in METHODS}


def _pt(a, b, met, kind):
    if kind == 'fixed':
        return POINT_FIX[a][met] - POINT_FIX[b][met]
    return (per_seed.loc[per_seed.method == a, met].mean()
            - per_seed.loc[per_seed.method == b, met].mean())


results, std_mat, keys = [], [], []
for a, b in ALL_PAIRS:
    for met in PRIMARY_METRICS:
        fx = fix_boot[(a, met)] - fix_boot[(b, met)]
        rn = run_boot[(a, met)] - run_boot[(b, met)]
        row = {'comparison': f'{a} - {b}', 'metric': met,
               'family': 'primary' if (a, b) in PRIMARY_PAIRS else 'secondary',
               'fixed_estimate': _pt(a, b, met, 'fixed'),
               'fixed_lo': np.nanpercentile(fx, 2.5),
               'fixed_hi': np.nanpercentile(fx, 97.5),
               'run_estimate': _pt(a, b, met, 'run'),
               'run_lo': np.nanpercentile(rn, 2.5),
               'run_hi': np.nanpercentile(rn, 97.5)}
        results.append(row)
        if row['family'] == 'primary':
            std_mat.append(rn)
            keys.append((f'{a} - {b}', met))

std_mat = np.vstack(std_mat)
centred = std_mat - np.nanmean(std_mat, axis=1, keepdims=True)
sds = np.nanstd(std_mat, axis=1, keepdims=True)
sds[sds == 0] = np.nan
crit = float(np.nanpercentile(np.nanmax(np.abs(centred / sds), axis=0), 95))
print(f'Single-step max-t critical value over six primary contrasts: {crit:.3f}')

res = pd.DataFrame(results)
res['sim_lo'] = np.nan
res['sim_hi'] = np.nan
for i, (cmp_, met) in enumerate(keys):
    j = res.index[(res.comparison == cmp_) & (res.metric == met)][0]
    sd_i = float(np.nanstd(std_mat[i]))
    res.at[j, 'sim_lo'] = res.at[j, 'run_estimate'] - crit * sd_i
    res.at[j, 'sim_hi'] = res.at[j, 'run_estimate'] + crit * sd_i
for c in ['fixed', 'run', 'sim']:
    lo, hi = res[f'{c}_lo'], res[f'{c}_hi']
    ok = lo.notna() & hi.notna()
    res[f'{c}_excludes_zero'] = np.where(ok, ~((lo <= 0) & (0 <= hi)), np.nan)
    res[f'{c}_excludes_zero'] = res[f'{c}_excludes_zero'].astype('object')
    res.loc[~ok, f'{c}_excludes_zero'] = 'n/a (simultaneous control covers the primary family only)'
res.to_csv(OUT / 'primary_contrasts.csv', index=False)
print(res.round(6).to_string(index=False))


# ============================================================================
# PART 3 — THRESHOLD SWEEP
# ============================================================================

print('\n' + '=' * 92)
print('PART 3 — THRESHOLD SWEEP, 37 OPERATING POINTS')
print('=' * 92)

trow = []
for m in METHODS:
    for sd in SEEDS:
        for t in THRESHOLDS:
            eo, fp = gaps(probs[(m, sd)], t)
            trow.append({'method': m, 'seed': sd, 'threshold': t,
                         'eo_gap': eo, 'fpr_gap': fp})
thr_df = pd.DataFrame(trow)
thr_df.to_csv(OUT / 'threshold_sweep.csv', index=False)

thr_mean = thr_df.groupby(['method', 'threshold'])[['eo_gap', 'fpr_gap']].mean()
win_eo = (thr_mean['eo_gap'].unstack(0)[METHODS].idxmin(axis=1).value_counts())
win_fp = (thr_mean['fpr_gap'].unstack(0)[METHODS].idxmin(axis=1).value_counts())
print('Lowest mean EO gap  count by method:', win_eo.to_dict())
print('Lowest mean FPR gap count by method:', win_fp.to_dict())


# ============================================================================
# PART 4 — LEAVE ONE SEED OUT
# ============================================================================

print('\n' + '=' * 92)
print('PART 4 — LEAVE-ONE-SEED-OUT SIMULTANEOUS INTERVALS')
print('=' * 92)

loo = []
for drop in SEEDS:
    keep = np.array([i for i, s in enumerate(SEEDS) if s != drop])
    kept_seeds = [s for s in SEEDS if s != drop]

    # Resample the KEPT runs with replacement, exactly as Part 2 does for the
    # full five. Averaging them deterministically would drop training-run
    # variance and give intervals that are far too narrow.
    rng_loo = np.random.default_rng(BOOT_SEED + 100 + drop)
    draw = {m: keep[rng_loo.integers(0, len(keep), size=(N_BOOT, len(keep)))]
            for m in METHODS}
    bo = {(m, k): np.array([np.nanmean(A[m][k][draw[m][b], b])
                            for b in range(N_BOOT)])
          for m in METHODS for k in PRIMARY_METRICS}

    kb, kk, pts = [], [], []
    for a, b in PRIMARY_PAIRS:
        for met in PRIMARY_METRICS:
            kb.append(bo[(a, met)] - bo[(b, met)])
            kk.append((f'{a} - {b}', met))
            sel = per_seed[per_seed.seed.isin(kept_seeds)]
            pts.append(sel.loc[sel.method == a, met].mean()
                       - sel.loc[sel.method == b, met].mean())

    kb = np.vstack(kb)
    c = kb - np.nanmean(kb, axis=1, keepdims=True)
    s_ = np.nanstd(kb, axis=1, keepdims=True)
    s_[s_ == 0] = np.nan
    cv = float(np.nanpercentile(np.nanmax(np.abs(c / s_), axis=0), 95))
    for i, (cmp_, met) in enumerate(kk):
        est = float(pts[i])
        sd_i = float(np.nanstd(kb[i]))
        loo.append({'omitted_seed': drop, 'comparison': cmp_, 'metric': met,
                    'estimate': est, 'sim_lo': est - cv * sd_i,
                    'sim_hi': est + cv * sd_i})
loo_df = pd.DataFrame(loo)
loo_df['excludes_zero'] = ~((loo_df.sim_lo <= 0) & (0 <= loo_df.sim_hi))
loo_df.to_csv(OUT / 'leave_one_seed_out.csv', index=False)
print(f'Intervals excluding zero: {int(loo_df.excludes_zero.sum())} of {len(loo_df)}')


# ============================================================================
# PART 5 — FIGURES
# ============================================================================

print('\n' + '=' * 92)
print('PART 5 — FIGURES')
print('=' * 92)

plt.rcParams.update({'font.size': 9, 'axes.linewidth': 0.8,
                     'font.family': 'DejaVu Sans'})


def save(fig, stem):
    fig.savefig(OUT / f'{stem}.png', dpi=300, bbox_inches='tight')
    fig.savefig(OUT / f'{stem}.pdf', bbox_inches='tight')
    plt.close(fig)
    print(f'  saved {stem}.png / .pdf')


# ---- Figure 4 -------------------------------------------------------------
fig = plt.figure(figsize=(11, 3.6))
gs = fig.add_gridspec(1, 3, wspace=0.32)
for k, (met, ttl) in enumerate([('auroc', 'Overall AUROC'),
                                ('worst_group_auroc', 'Worst-group AUROC')]):
    ax = fig.add_subplot(gs[0, k])
    for xi, m in enumerate(METHODS):
        v = per_seed.loc[per_seed.method == m, met].to_numpy()
        ax.scatter(np.full(len(v), xi) + np.linspace(-.12, .12, len(v)), v,
                   s=22, color=COLORS[m], zorder=3)
        ax.errorbar(xi, v.mean(), yerr=v.std(ddof=1), fmt='D', color='black',
                    ms=5, capsize=3, lw=1.1, zorder=4)
    ax.set_xticks(range(len(METHODS)))
    ax.set_xticklabels(METHODS, rotation=30, ha='right')
    ax.set_ylabel(ttl)
    ax.set_title(f'{"AB"[k]}   {ttl}', loc='left', fontweight='bold')
    ax.grid(axis='y', ls=':', lw=.5, alpha=.6)

ax = fig.add_subplot(gs[0, 2])
for met, lab, cl, mk in [('auroc', 'Overall AUROC', '#2ca02c', 'o'),
                         ('worst_group_auroc', 'Worst-group AUROC', '#1f77b4', 's')]:
    d = [per_seed.loc[(per_seed.method == 'DWFA') & (per_seed.seed == s), met].item()
         - per_seed.loc[(per_seed.method == 'FedAvg') & (per_seed.seed == s), met].item()
         for s in SEEDS]
    ax.plot(range(len(SEEDS)), np.array(d) * 1e3, marker=mk, color=cl, label=lab, lw=1.3)
ax.axhline(0, color='black', ls='--', lw=.9)
ax.set_xticks(range(len(SEEDS)))
ax.set_xticklabels(SEEDS)
ax.set_xlabel('Numeric seed')
ax.set_ylabel(r'DWFA $-$ FedAvg ($\times 10^{-3}$)')
ax.set_title('C   Same-seed differences', loc='left', fontweight='bold')
ax.legend(frameon=False, fontsize=7.5)
ax.grid(axis='y', ls=':', lw=.5, alpha=.6)
save(fig, 'figure4_external_nih_across_runs')

# ---- Figure 5 -------------------------------------------------------------
prim = res[res.family == 'primary'].reset_index(drop=True)
lab = [f'{r.comparison} | '
       f'{"Overall" if r.metric == "auroc" else "Worst-group"}'
       for r in prim.itertuples()]
y = np.arange(len(prim))[::-1]
fig, ax = plt.subplots(figsize=(7.6, 4.4))
for off, (lo, hi, est, cl, mk, nm) in enumerate([
        ('fixed_lo', 'fixed_hi', 'fixed_estimate', '#1f77b4', 'o',
         'Fixed seed ensemble: pointwise 95% CI'),
        ('run_lo', 'run_hi', 'run_estimate', '#ff7f0e', 's',
         'Run-aware: pointwise 95% CI'),
        ('sim_lo', 'sim_hi', 'run_estimate', '#d62728', 'D',
         'Run-aware: simultaneous 95% CI')]):
    dy = (1 - off) * 0.22
    ax.errorbar(prim[est] * 1e3, y + dy,
                xerr=[(prim[est] - prim[lo]) * 1e3, (prim[hi] - prim[est]) * 1e3],
                fmt=mk, color=cl, ms=5, lw=1.2, capsize=2.5, label=nm)
ax.axvline(0, color='black', ls='--', lw=1)
ax.set_yticks(y)
ax.set_yticklabels(lab)
ax.set_xlabel(r'AUROC difference ($\times 10^{-3}$; first method minus second)')
ax.legend(frameon=False, fontsize=7.5, loc='upper center',
          bbox_to_anchor=(0.5, -0.16), ncol=1)
ax.grid(axis='x', ls=':', lw=.5, alpha=.6)
save(fig, 'figure5_primary_discrimination_contrasts')

# ---- Figure 6 -------------------------------------------------------------
fig, axes = plt.subplots(1, 2, figsize=(10, 3.8))
for ax, col, ttl in [(axes[0], 'eo_gap', 'Equal-opportunity gap'),
                     (axes[1], 'fpr_gap', 'False-positive-rate gap')]:
    for m in METHODS:
        d = thr_df[thr_df.method == m]
        g = d.groupby('threshold')[col]
        ax.plot(THRESHOLDS, g.mean().to_numpy(), color=COLORS[m], lw=1.4, label=m)
        ax.fill_between(THRESHOLDS, g.min().to_numpy(), g.max().to_numpy(),
                        color=COLORS[m], alpha=.13, lw=0)
    ax.axvline(0.5, color='grey', ls='--', lw=.9)
    ax.set_xlabel('Decision threshold')
    ax.set_ylabel('Gap')
    ax.set_title(ttl)
    ax.legend(frameon=False, fontsize=7.5)
    ax.grid(ls=':', lw=.5, alpha=.6)
save(fig, 'figure6_threshold_sensitivity')

# ---- Figure 8 -------------------------------------------------------------
combos = [(f'{a} - {b}', m) for a, b in PRIMARY_PAIRS for m in PRIMARY_METRICS]
y = np.arange(len(combos))[::-1]
cmap = plt.get_cmap('tab10')
fig, ax = plt.subplots(figsize=(8, 4.8))
for si, drop in enumerate(SEEDS):
    d = loo_df[loo_df.omitted_seed == drop]
    xs, los, his = [], [], []
    for cmp_, met in combos:
        r = d[(d.comparison == cmp_) & (d.metric == met)].iloc[0]
        xs.append(r.estimate * 1e3)
        los.append((r.estimate - r.sim_lo) * 1e3)
        his.append((r.sim_hi - r.estimate) * 1e3)
    ax.errorbar(xs, y + (si - 2) * 0.14, xerr=[los, his], fmt='o', ms=3.6,
                lw=1.0, capsize=2, color=cmap(si), label=f'Omit {drop}')
ax.axvline(0, color='black', ls='--', lw=1)
ax.set_yticks(y)
ax.set_yticklabels([f'{c} | {"Overall" if m == "auroc" else "Worst-group"}'
                    for c, m in combos])
ax.set_xlabel(r'AUROC difference ($\times 10^{-3}$; simultaneous 95% CI)')
ax.legend(frameon=False, fontsize=7.5, ncol=5, loc='upper center',
          bbox_to_anchor=(0.5, 1.13))
ax.grid(axis='x', ls=':', lw=.5, alpha=.6)
save(fig, 'figure8_leave_one_seed_out')


# ============================================================================
# PACKAGE
# ============================================================================

(OUT / 'run_metadata.json').write_text(json.dumps({
    'methods': METHODS, 'seeds': SEEDS,
    'primary_pairs': [f'{a} - {b}' for a, b in PRIMARY_PAIRS],
    'secondary_pairs': [f'{a} - {b}' for a, b in SECONDARY_PAIRS],
    'primary_metrics': PRIMARY_METRICS,
    'bootstrap_replicates': N_BOOT, 'bootstrap_seed': BOOT_SEED,
    'maxt_critical_value': crit,
    'n_images': int(len(labels)), 'n_patients': int(n_patients),
}, indent=2))

with zipfile.ZipFile(ZIP_OUT, 'w', zipfile.ZIP_DEFLATED) as z:
    for f in sorted(OUT.glob('*')):
        z.write(f, f.name)

n_fix = int(res[res.family == 'primary'].fixed_excludes_zero.sum())
n_run = int(res[res.family == 'primary'].run_excludes_zero.sum())
n_sim = int(res[res.family == 'primary'].sim_excludes_zero.sum())
print('\n' + '=' * 92)
print(f'Primary family, six contrasts:')
print(f'  fixed-ensemble intervals excluding zero      : {n_fix} of 6')
print(f'  run-aware pointwise intervals excluding zero : {n_run} of 6')
print(f'  run-aware simultaneous excluding zero        : {n_sim} of 6')
print(f'\nSaved -> {OUT}')
print(f'Zip   -> {ZIP_OUT}')
print('=' * 92)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
PART 0 — PREDICTION CACHE
Cached keys: 20
FedAvg seeds present: [42, 123, 456, 789, 1010]

All five FedAvg seeds already cached. Running CPU-only.

Cohort: 112,120 images | 30,805 patients | prevalence 0.1187
  FedAvg   seeds [42, 123, 456, 789, 1010]
  LPR      seeds [42, 123, 456, 789, 1010]
  CADR     seeds [42, 123, 456, 789, 1010]
  DWFA     seeds [42, 123, 456, 789, 1010]
Merged descriptive references from nih_extended_baselines_cache.npz
  Centralized reference  seeds [42, 123, 456] (descriptive only)
  FedProx                seeds [42, 123, 456] (descriptive only)

PART 1 — PER-SEED NIH METRICS
               method  runs      family             auroc worst_group_auroc         eo_gap_50        fpr_gap_50
Centralized reference     3 descriptive 0.8586 +/- 0.0020 0.8557 +/- 0.0016 0.0184 +/- 0.0079 0.0183 +/- 0.0133
               FedAvg     5 inferenti

In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=False)

import numpy as np
from sklearn.metrics import average_precision_score

RESULTS = '/content/drive/MyDrive/FairFedCXR/results'

c  = np.load(f'{RESULTS}/nih_pred_cache.npz', allow_pickle=True)
P  = c['all_probs'].item()
y  = c['labels'].astype(int)
sx = c['sex'].astype(int)

seeds = [42, 123, 456, 789, 1010]
missing = [s for s in seeds if ('fedavg', s) not in P]
if missing:
    raise SystemExit(f'Missing FedAvg seeds in cache: {missing}')

ap, wg, br = [], [], []
for s in seeds:
    p = np.asarray(P[('fedavg', s)], float)
    ap.append(average_precision_score(y, p))
    wg.append(min(average_precision_score(y[sx == 1], p[sx == 1]),
                  average_precision_score(y[sx == 0], p[sx == 0])))
    br.append(np.mean((p - y) ** 2))

print('FedAvg, five common seeds:\n')
for name, v in [('AUPRC', ap), ('WG-AP', wg), ('Brier', br)]:
    print(f'{name}: {np.mean(v):.4f} +/- {np.std(v, ddof=1):.4f}')

Mounted at /content/drive
FedAvg, five common seeds:

AUPRC: 0.4502 +/- 0.0095
WG-AP: 0.4483 +/- 0.0111
Brier: 0.1556 +/- 0.0151


In [ ]:
# ============================================================================
# CALIBRATION FIT AUDIT — READ ONLY
#
# PURPOSE
#   Recover the four items the reviewer asks for in the calibration comment:
#     1. the value of the fit that failed the convergence criterion,
#     2. the optimizer termination state for that fit,
#     3. alternative-solver sensitivity for it,
#     4. whether excluding it changes any comparative calibration conclusion.
#
# POLLUTION SAFETY
#   This cell OPENS the prediction caches read-only and writes NOTHING except a
#   new folder, results/calibration_audit/. It does not touch, overwrite, or
#   re-save any existing CSV, NPZ, or checkpoint. No model is retrained and no
#   inference is run. Safe to run at any point without disturbing other results.
#
# BEFORE RUNNING
#   No old notebook cell is required. This cell mounts Drive itself.
#
# METHOD
#   Refits the recalibration model reported in the paper,
#     logit Pr(Y=1) = beta0 + beta1 * logit(phat),
#   with the same settings: probabilities clipped to [1e-6, 1-1e-6], logits
#   standardised before fitting, L-BFGS-B, three starting values, a 1e-10 ridge
#   term, iteration limit 5000, and a convergence rule of infinity-norm gradient
#   below 1e-4. Coefficients are transformed back to the original logit scale.
# ============================================================================

from google.colab import drive
drive.mount('/content/drive', force_remount=False)

from pathlib import Path
import json
import numpy as np
import pandas as pd
from scipy.optimize import minimize
from scipy.special import expit

BASE = Path('/content/drive/MyDrive/FairFedCXR')
RESULTS = BASE / 'results'
CACHES = [RESULTS / 'nih_pred_cache.npz',
          RESULTS / 'nih_extended_baselines_cache.npz']

OUT = RESULTS / 'calibration_audit'
OUT.mkdir(parents=True, exist_ok=True)

LABELS = {'centralized': 'Centralized reference', 'fedavg': 'FedAvg',
          'fedprox': 'FedProx', 'qfedavg': 'LPR', 'fairfed': 'CADR',
          'dwfa': 'DWFA'}
ORDER = ['Centralized reference', 'FedAvg', 'FedProx', 'LPR', 'CADR', 'DWFA']

EPS = 1e-6
RIDGE = 1e-10
MAXITER = 5000
GRAD_TOL = 1e-4
STARTS = [(0.0, 1.0), (0.0, 0.5), (-1.0, 1.0)]

# ---------------------------------------------------------------- load caches

probs, labels, sex = {}, None, None
for c in CACHES:
    if not c.exists():
        print(f'skip (absent): {c.name}')
        continue
    z = np.load(c, allow_pickle=True)
    y, s = z['labels'].astype(np.int8), z['sex'].astype(np.int8)
    if labels is None:
        labels, sex = y, s
    elif not (len(y) == len(labels) and np.array_equal(y, labels)):
        print(f'skip (row order mismatch): {c.name}')
        continue
    for (m, sd), p in z['all_probs'].item().items():
        lab = LABELS.get(m)
        if lab is not None and len(p) == len(labels):
            probs[(lab, int(sd))] = np.asarray(p, np.float64)
    print(f'loaded {c.name}')

if not probs:
    raise SystemExit('No usable prediction arrays found.')

print(f'\nrows: {len(labels):,} | positives: {int(labels.sum()):,}')
for m in ORDER:
    got = sorted(sd for (mm, sd) in probs if mm == m)
    if got:
        print(f'  {m:<22} seeds {got}')

GROUPS = {'overall': np.ones(len(labels), bool),
          'male': sex == 1, 'female': sex == 0}

# ------------------------------------------------------------------- fitting

def _nll_grad(b, x, y):
    z = b[0] + b[1] * x
    # np.logaddexp(0, z) is a stable log(1 + exp(z)); expit is a stable sigmoid
    nll = float(np.sum(np.logaddexp(0.0, z) - y * z)
                + RIDGE * (b[0] ** 2 + b[1] ** 2))
    r = expit(z) - y
    g = np.array([r.sum() + 2 * RIDGE * b[0], float(r @ x) + 2 * RIDGE * b[1]])
    return nll, g


def fit_one(p, y, method='L-BFGS-B'):
    p = np.clip(p, EPS, 1 - EPS)
    x = np.log(p / (1 - p))
    m, s = x.mean(), x.std()
    s = s if s > 0 else 1.0
    xs = (x - m) / s
    best = None
    for b0 in STARTS:
        opts = {'maxiter': MAXITER}
        # Default L-BFGS-B stopping is looser than the reported gradient rule,
        # so tighten it. Otherwise the optimizer halts above the tolerance and
        # a converged fit is wrongly recorded as a failure.
        if method == 'L-BFGS-B':
            opts.update({'ftol': 1e-15, 'gtol': 1e-12})
        elif method in ('BFGS', 'Newton-CG'):
            opts.update({'gtol': 1e-12})
        elif method == 'SLSQP':
            opts.update({'ftol': 1e-14})
        kw = dict(fun=lambda b: _nll_grad(b, xs, y)[0],
                  jac=lambda b: _nll_grad(b, xs, y)[1],
                  x0=np.array(b0, float), method=method, options=opts)
        try:
            r = minimize(**kw)
        except Exception as e:
            print(f'   solver {method} raised: {e}')
            continue
        f = float(r.fun)
        if best is None or f < best[0]:
            best = (f, r)
    if best is None:
        return None
    f, r = best
    b = r.x
    gnorm = float(np.max(np.abs(_nll_grad(b, xs, y)[1])))
    return {'intercept': float(b[0] - b[1] * m / s),
            'slope': float(b[1] / s),
            'grad_inf_norm': gnorm,
            'converged': bool(gnorm < GRAD_TOL),
            'solver_success': bool(r.success),
            'termination': str(r.message),
            'n_iter': int(getattr(r, 'nit', -1)),
            'nll': f}


rows = []
for (m, sd), p in sorted(probs.items()):
    for gname, gmask in GROUPS.items():
        yg = labels[gmask].astype(np.float64)
        if np.unique(yg).size < 2:
            continue
        r = fit_one(p[gmask], yg)
        if r is None:
            continue
        r.update({'method': m, 'seed': sd, 'group': gname,
                  'n': int(gmask.sum())})
        rows.append(r)

fits = pd.DataFrame(rows)
cols = ['method', 'seed', 'group', 'n', 'intercept', 'slope',
        'grad_inf_norm', 'converged', 'solver_success', 'n_iter',
        'nll', 'termination']
fits = fits[cols].sort_values(['method', 'seed', 'group'])
fits.to_csv(OUT / 'calibration_fits.csv', index=False)

n_tot = len(fits)
n_ok = int(fits.converged.sum())
print('\n' + '=' * 88)
print(f'FITS: {n_ok} of {n_tot} met the infinity-norm gradient criterion '
      f'of {GRAD_TOL:g}')
print('=' * 88)

failed = fits[~fits.converged]
if failed.empty:
    print('No fit failed the criterion in this recomputation.')
else:
    print('\nFits that did not meet the criterion:')
    print(failed[['method', 'seed', 'group', 'intercept', 'slope',
                  'grad_inf_norm', 'n_iter', 'termination']]
          .to_string(index=False))

# --------------------------------------------- alternative-solver sensitivity

alt_rows = []
targets = list(failed.itertuples(index=False))
fa = fits[(fits.method == 'FedAvg') & (fits.seed == 42) & (fits.group == 'overall')]
if not fa.empty and fa.iloc[0].converged:
    targets.append(fa.itertuples(index=False).__next__())

seen = set()
for t in targets:
    key = (t.method, t.seed, t.group)
    if key in seen:
        continue
    seen.add(key)
    p = probs.get((t.method, int(t.seed)))
    if p is None:
        continue
    gm = GROUPS[t.group]
    for solver in ['L-BFGS-B', 'BFGS', 'Newton-CG', 'SLSQP']:
        r = fit_one(p[gm], labels[gm].astype(np.float64), method=solver)
        if r is None:
            continue
        r.update({'method': t.method, 'seed': int(t.seed),
                  'group': t.group, 'solver': solver})
        alt_rows.append(r)

alt = pd.DataFrame(alt_rows)
if not alt.empty:
    alt = alt[['method', 'seed', 'group', 'solver', 'intercept', 'slope',
               'grad_inf_norm', 'converged', 'n_iter', 'termination']]
    alt.to_csv(OUT / 'calibration_alt_solvers.csv', index=False)
    print('\n' + '=' * 88)
    print('ALTERNATIVE-SOLVER SENSITIVITY')
    print('=' * 88)
    print(alt.to_string(index=False))
    for key, g in alt.groupby(['method', 'seed', 'group']):
        print(f'\n{key}: intercept spread '
              f'{g.intercept.max() - g.intercept.min():.3e} | '
              f'slope spread {g.slope.max() - g.slope.min():.3e}')

# ------------------------------- does excluding the flagged fit change ranges

print('\n' + '=' * 88)
print('PER-METHOD OVERALL-GROUP SUMMARY, WITH AND WITHOUT NON-CONVERGED FITS')
print('=' * 88)

ov = fits[fits.group == 'overall']
summ = []
for m in ORDER:
    d = ov[ov.method == m]
    if d.empty:
        continue
    dk = d[d.converged]
    summ.append({'method': m, 'n_fits': len(d), 'n_converged': len(dk),
                 'mean_intercept_all': d.intercept.mean(),
                 'mean_intercept_conv': dk.intercept.mean() if len(dk) else np.nan,
                 'mean_slope_all': d.slope.mean(),
                 'mean_slope_conv': dk.slope.mean() if len(dk) else np.nan})
summ = pd.DataFrame(summ)
summ['d_intercept'] = (summ.mean_intercept_all - summ.mean_intercept_conv).abs()
summ['d_slope'] = (summ.mean_slope_all - summ.mean_slope_conv).abs()
summ.to_csv(OUT / 'calibration_summary_by_method.csv', index=False)
print(summ.round(4).to_string(index=False))

def rng(v):
    v = v.dropna()
    return (np.nan, np.nan) if v.empty else (v.min(), v.max())

ia = rng(summ.mean_intercept_all); ic = rng(summ.mean_intercept_conv)
sa = rng(summ.mean_slope_all);     sc = rng(summ.mean_slope_conv)
print(f'\nMean intercept range, all fits          : {ia[0]:.4f} to {ia[1]:.4f}')
print(f'Mean intercept range, converged only    : {ic[0]:.4f} to {ic[1]:.4f}')
print(f'Mean slope range, all fits              : {sa[0]:.4f} to {sa[1]:.4f}')
print(f'Mean slope range, converged only        : {sc[0]:.4f} to {sc[1]:.4f}')

rank_all = list(summ.sort_values('mean_slope_all').method)
rank_conv = list(summ.sort_values('mean_slope_conv').method)
print(f'\nMethod order by mean slope, all fits       : {rank_all}')
print(f'Method order by mean slope, converged only : {rank_conv}')
print(f'ORDER UNCHANGED: {rank_all == rank_conv}')
print(f'All slopes below 0.5: '
      f'{bool((summ.mean_slope_all < 0.5).all())} (all fits), '
      f'{bool((summ.mean_slope_conv.dropna() < 0.5).all())} (converged only)')
print(f'All intercepts negative: '
      f'{bool((summ.mean_intercept_all < 0).all())} (all fits), '
      f'{bool((summ.mean_intercept_conv.dropna() < 0).all())} (converged only)')

(OUT / 'run_metadata.json').write_text(json.dumps({
    'n_fits': int(n_tot), 'n_converged': int(n_ok),
    'grad_tol': GRAD_TOL, 'ridge': RIDGE, 'maxiter': MAXITER,
    'eps_clip': EPS, 'n_starts': len(STARTS),
    'solvers_tested': ['L-BFGS-B', 'BFGS', 'Newton-CG', 'SLSQP'],
    'wrote_only': str(OUT), 'modified_existing_files': False,
}, indent=2))

print(f'\nWrote {OUT} (new folder only; no existing file was modified).')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
loaded nih_pred_cache.npz
loaded nih_extended_baselines_cache.npz

rows: 112,120 | positives: 13,307
  Centralized reference  seeds [42, 123, 456]
  FedAvg                 seeds [42, 123, 456, 789, 1010]
  FedProx                seeds [42, 123, 456]
  LPR                    seeds [42, 123, 456, 789, 1010]
  CADR                   seeds [42, 123, 456, 789, 1010]
  DWFA                   seeds [42, 123, 456, 789, 1010]

FITS: 75 of 78 met the infinity-norm gradient criterion of 0.0001

Fits that did not meet the criterion:
               method  seed   group  intercept    slope  grad_inf_norm  n_iter                                          termination
Centralized reference    42 overall  -1.494879 0.421288       0.000110      11 CONVERGENCE: RELATIVE REDUCTION OF F <= FACTR*EPSMCH
                 DWFA  1010 overall  -1.598071 0.322003       0.000225      11 C

/tmp/ipykernel_1000/3217867019.py:124: OptimizeWarning: Unknown solver options: gtol
  r = minimize(**kw)
/tmp/ipykernel_1000/3217867019.py:124: OptimizeWarning: Unknown solver options: gtol
  r = minimize(**kw)
/tmp/ipykernel_1000/3217867019.py:124: OptimizeWarning: Unknown solver options: gtol
  r = minimize(**kw)
/tmp/ipykernel_1000/3217867019.py:124: OptimizeWarning: Unknown solver options: gtol
  r = minimize(**kw)
/tmp/ipykernel_1000/3217867019.py:124: OptimizeWarning: Unknown solver options: gtol
  r = minimize(**kw)



ALTERNATIVE-SOLVER SENSITIVITY
               method  seed   group    solver  intercept    slope  grad_inf_norm  converged  n_iter                                                   termination
Centralized reference    42 overall  L-BFGS-B  -1.494879 0.421288   1.096437e-04      False      11          CONVERGENCE: RELATIVE REDUCTION OF F <= FACTR*EPSMCH
Centralized reference    42 overall      BFGS  -1.494879 0.421288   5.448798e-10       True      15 Desired error not necessarily achieved due to precision loss.
Centralized reference    42 overall Newton-CG  -1.494879 0.421288   2.149454e-08       True       7                         Optimization terminated successfully.
Centralized reference    42 overall     SLSQP  -1.494879 0.421288   5.729822e-05       True      11                          Optimization terminated successfully
                 DWFA  1010 overall  L-BFGS-B  -1.598071 0.322003   2.250016e-04      False      11          CONVERGENCE: RELATIVE REDUCTION OF F <= FACTR*EPS